In [ ]:
# ============================================================
# SurfaceTarget-CRC
# Phase 1: Environment and Project Initialization
#
# Project:
# Proteogenomic Prioritization of Cell-Surface Therapeutic
# Targets in Microsatellite-Stable Colorectal Cancer
# ============================================================

from pathlib import Path
import sys
import platform
import datetime
import json
import shutil

# ------------------------------------------------------------
# 1. Define local Colab project directory
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/SurfaceTarget-CRC")

# ------------------------------------------------------------
# 2. Create project directory structure
# ------------------------------------------------------------

directories = [
    "data/raw",
    "data/interim",
    "data/processed",
    "data/metadata",
    "notebooks",
    "src",
    "results/tables",
    "results/figures",
    "reports",
    "logs"
]

for directory in directories:
    (PROJECT_ROOT / directory).mkdir(
        parents=True,
        exist_ok=True
    )

# ------------------------------------------------------------
# 3. Collect environment information
# ------------------------------------------------------------

environment_info = {
    "project": "SurfaceTarget-CRC",

    "full_title": (
        "Proteogenomic Prioritization of Cell-Surface "
        "Therapeutic Targets in Microsatellite-Stable "
        "Colorectal Cancer"
    ),

    "python_version": sys.version,

    "platform": platform.platform(),

    "setup_timestamp_utc": datetime.datetime.now(
        datetime.timezone.utc
    ).isoformat(),

    "project_root": str(PROJECT_ROOT),

    "storage_type": "Google Colab local temporary storage"
}

# ------------------------------------------------------------
# 4. Save environment metadata
# ------------------------------------------------------------

metadata_path = (
    PROJECT_ROOT /
    "data/metadata/environment.json"
)

with open(metadata_path, "w") as f:
    json.dump(
        environment_info,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 5. Check available disk space
# ------------------------------------------------------------

disk = shutil.disk_usage("/content")

total_gb = disk.total / (1024 ** 3)
used_gb = disk.used / (1024 ** 3)
free_gb = disk.free / (1024 ** 3)

# ------------------------------------------------------------
# 6. Verify project initialization
# ------------------------------------------------------------

print("=" * 65)
print("SURFACETARGET-CRC")
print("PROJECT INITIALIZATION")
print("=" * 65)

print(f"\nPython Version: {platform.python_version()}")

print(f"\nProject Root: {PROJECT_ROOT}")

print("\nDirectory Structure:")

for directory in directories:

    path = PROJECT_ROOT / directory

    status = "OK" if path.exists() else "FAILED"

    print(f"[{status}] {directory}")

print("\n" + "-" * 65)
print("COLAB STORAGE INFORMATION")
print("-" * 65)

print(f"Total Disk Space: {total_gb:.2f} GB")
print(f"Used Disk Space:  {used_gb:.2f} GB")
print(f"Free Disk Space:  {free_gb:.2f} GB")

print("\n" + "-" * 65)

print("Environment metadata saved successfully.")

print("\nPROJECT INITIALIZATION COMPLETE")

print("=" * 65)

In [ ]:
# ============================================================
# SurfaceTarget-CRC
# Phase 1 — Patch 1.2
# TCGA Dataset Discovery and Metadata Retrieval
# ============================================================

from pathlib import Path
import requests
import json
from datetime import datetime, timezone

# ------------------------------------------------------------
# 1. Project configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/SurfaceTarget-CRC")

METADATA_DIR = PROJECT_ROOT / "data" / "metadata"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

GDC_API = "https://api.gdc.cancer.gov"

PROJECTS = [
    "TCGA-COAD",
    "TCGA-READ"
]

# ------------------------------------------------------------
# 2. Retrieve project metadata
# ------------------------------------------------------------

project_inventory = {
    "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
    "source": "NCI Genomic Data Commons",
    "api": GDC_API,
    "projects": {}
}

print("=" * 65)
print("SURFACETARGET-CRC")
print("TCGA DATASET DISCOVERY")
print("=" * 65)

for project_id in PROJECTS:

    print(f"\nInvestigating: {project_id}")
    print("-" * 50)

    endpoint = f"{GDC_API}/projects/{project_id}"

    try:

        response = requests.get(
            endpoint,
            params={"expand": "summary,summary.data_categories"},
            timeout=30
        )

        response.raise_for_status()

        data = response.json()["data"]

        summary = data.get("summary", {})

        case_count = summary.get("case_count", "Unknown")

        categories = summary.get("data_categories", [])

        project_inventory["projects"][project_id] = {
            "name": data.get("name"),
            "primary_site": data.get("primary_site"),
            "case_count_all": case_count,
            "data_categories": categories,
            "status": "metadata_retrieved"
        }

        print(f"Project: {data.get('name')}")
        print(f"Primary Site: {data.get('primary_site')}")
        print(f"Total Registered Cases: {case_count}")

        print("\nAvailable Data Categories:")

        if categories:

            for category in categories:

                name = category.get("data_category", "Unknown")
                count = category.get("case_count", "Unknown")

                print(f"  - {name}: {count} cases")

        else:

            print("  No category summary returned.")

        print("\nStatus: SUCCESS")

    except Exception as e:

        project_inventory["projects"][project_id] = {
            "status": "error",
            "error": str(e)
        }

        print(f"Status: ERROR — {e}")

# ------------------------------------------------------------
# 3. Save metadata
# ------------------------------------------------------------

output_path = METADATA_DIR / "tcga_project_inventory.json"

with open(output_path, "w") as f:

    json.dump(
        project_inventory,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 4. Summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("DATASET DISCOVERY SUMMARY")
print("=" * 65)

for project_id, info in project_inventory["projects"].items():

    print(
        f"{project_id}: "
        f"{info.get('case_count_all', 'N/A')} cases "
        f"({info['status']})"
    )

print("\nMetadata saved to:")
print(output_path)

print("\nNOTE:")
print("These counts include all registered project cases.")
print("They are NOT MSS-specific cohort counts.")
print("No molecular datasets have been downloaded.")

print("\nPATCH 1.2 COMPLETE")

In [ ]:
# ============================================================
# SurfaceTarget-CRC
# Phase 1 — Patch 1.3
#
# TCGA RNA-seq File Inventory
# Gene Expression Quantification — STAR Counts
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import requests
import pandas as pd
import json

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/SurfaceTarget-CRC")

METADATA_DIR = PROJECT_ROOT / "data" / "metadata"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

GDC_API = "https://api.gdc.cancer.gov"

PROJECTS = [
    "TCGA-COAD",
    "TCGA-READ"
]

WORKFLOW = "STAR - Counts"

# ------------------------------------------------------------
# 2. Construct GDC API filters
# ------------------------------------------------------------

filters = {
    "op": "and",
    "content": [
        {
            "op": "in",
            "content": {
                "field": "cases.project.project_id",
                "value": PROJECTS
            }
        },
        {
            "op": "in",
            "content": {
                "field": "files.data_type",
                "value": ["Gene Expression Quantification"]
            }
        },
        {
            "op": "in",
            "content": {
                "field": "files.analysis.workflow_type",
                "value": [WORKFLOW]
            }
        }
    ]
}

# ------------------------------------------------------------
# 3. Define metadata fields
# ------------------------------------------------------------

fields = [
    "file_id",
    "file_name",
    "file_size",
    "md5sum",
    "data_type",
    "access",
    "analysis.workflow_type",
    "cases.submitter_id",
    "cases.project.project_id",
    "cases.samples.submitter_id",
    "cases.samples.sample_type"
]

# ------------------------------------------------------------
# 4. Retrieve file metadata with pagination
# ------------------------------------------------------------

print("=" * 65)
print("SURFACETARGET-CRC")
print("TCGA RNA-SEQ FILE INVENTORY")
print("=" * 65)

all_files = []

offset = 0
page_size = 1000

while True:

    params = {
        "filters": json.dumps(filters),
        "fields": ",".join(fields),
        "format": "JSON",
        "size": page_size,
        "from": offset
    }

    response = requests.get(
        f"{GDC_API}/files",
        params=params,
        timeout=90
    )

    response.raise_for_status()

    payload = response.json()["data"]

    hits = payload.get("hits", [])

    total = payload["pagination"]["total"]

    all_files.extend(hits)

    print(
        f"Retrieved {len(all_files)} "
        f"of {total} files"
    )

    if not hits or len(all_files) >= total:
        break

    offset += len(hits)

print("\nMetadata retrieval completed.")

# ------------------------------------------------------------
# 5. Extract file and sample information
# ------------------------------------------------------------

records = []

for item in all_files:

    file_id = item.get("file_id")
    file_name = item.get("file_name")
    file_size = item.get("file_size")

    workflow = (
        item.get("analysis") or {}
    ).get("workflow_type")

    cases = item.get("cases") or []

    for case in cases:

        patient_id = case.get("submitter_id")

        project_id = (
            (case.get("project") or {})
            .get("project_id")
        )

        samples = case.get("samples") or []

        if not samples:

            records.append({
                "project": project_id,
                "patient_id": patient_id,
                "sample_id": None,
                "sample_type": "Unknown",
                "file_id": file_id,
                "file_name": file_name,
                "file_size_bytes": file_size,
                "workflow": workflow,
                "access": item.get("access"),
                "md5sum": item.get("md5sum")
            })

        else:

            for sample in samples:

                records.append({
                    "project": project_id,
                    "patient_id": patient_id,
                    "sample_id": sample.get("submitter_id"),
                    "sample_type": sample.get(
                        "sample_type", "Unknown"
                    ),
                    "file_id": file_id,
                    "file_name": file_name,
                    "file_size_bytes": file_size,
                    "workflow": workflow,
                    "access": item.get("access"),
                    "md5sum": item.get("md5sum")
                })

# ------------------------------------------------------------
# 6. Create metadata dataframe
# ------------------------------------------------------------

df = pd.DataFrame(records)

if df.empty:
    raise RuntimeError(
        "No STAR Counts metadata returned. "
        "Check the GDC query filters."
    )

# Remove exact duplicate metadata records
df = df.drop_duplicates()

print("\n" + "-" * 65)
print("RNA-SEQ METADATA SUMMARY")
print("-" * 65)

print(f"\nUnique RNA-seq files: {df['file_id'].nunique()}")

print(
    f"Unique patients: "
    f"{df['patient_id'].nunique()}"
)

print(
    f"Unique samples: "
    f"{df['sample_id'].nunique()}"
)

# ------------------------------------------------------------
# 7. Sample type distribution
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAMPLE TYPE DISTRIBUTION")
print("-" * 65)

sample_summary = (
    df.groupby(
        ["project", "sample_type"],
        dropna=False
    )
    .agg(
        unique_patients=("patient_id", "nunique"),
        unique_samples=("sample_id", "nunique"),
        unique_files=("file_id", "nunique")
    )
    .reset_index()
)

print(
    sample_summary.to_string(index=False)
)

# ------------------------------------------------------------
# 8. Workflow distribution
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("WORKFLOW DISTRIBUTION")
print("-" * 65)

print(
    df.groupby(
        ["project", "workflow"],
        dropna=False
    )["file_id"]
    .nunique()
    .to_string()
)

# ------------------------------------------------------------
# 9. Save inventory and metadata
# ------------------------------------------------------------

inventory_path = (
    METADATA_DIR / "tcga_rnaseq_file_inventory.csv"
)

summary_path = (
    METADATA_DIR / "tcga_rnaseq_sample_summary.csv"
)

df.to_csv(
    inventory_path,
    index=False
)

sample_summary.to_csv(
    summary_path,
    index=False
)

# Save retrieval information
retrieval_info = {
    "retrieved_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "source": "NCI Genomic Data Commons",
    "endpoint": f"{GDC_API}/files",
    "projects": PROJECTS,
    "workflow": WORKFLOW,
    "data_type": "Gene Expression Quantification",
    "total_files_reported_by_api": total,
    "unique_files_retrieved": int(
        df["file_id"].nunique()
    ),
    "unique_patients": int(
        df["patient_id"].nunique()
    ),
    "unique_samples": int(
        df["sample_id"].nunique()
    ),
    "filters": filters,
    "fields": fields
}

retrieval_path = (
    METADATA_DIR / "tcga_rnaseq_retrieval_metadata.json"
)

with open(retrieval_path, "w") as f:
    json.dump(retrieval_info, f, indent=4)

# ------------------------------------------------------------
# 10. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FILES SAVED")
print("=" * 65)

print(f"\n1. {inventory_path.name}")
print(f"2. {summary_path.name}")
print(f"3. {retrieval_path.name}")

print("\nNo RNA-seq expression matrices downloaded.")

print("\nPATCH 1.3 COMPLETE")

In [ ]:
# ============================================================
# SurfaceTarget-CRC
# Phase 1 — Patch 1.4
#
# RNA-seq Metadata Quality Audit
# and MSI/MSS Clinical Annotation Discovery
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import requests
import json

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/SurfaceTarget-CRC")

METADATA_DIR = PROJECT_ROOT / "data" / "metadata"

INVENTORY_PATH = (
    METADATA_DIR / "tcga_rnaseq_file_inventory.csv"
)

CBIO_API = "https://www.cbioportal.org/api"

STUDY_ID = "coadread_tcga_pan_can_atlas_2018"

# ------------------------------------------------------------
# 2. Load existing RNA-seq inventory
# ------------------------------------------------------------

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.4 — MSI ANNOTATION DISCOVERY")
print("=" * 65)

if not INVENTORY_PATH.exists():
    raise FileNotFoundError(
        "RNA-seq inventory not found. "
        "Please run Patch 1.3 first."
    )

inventory = pd.read_csv(
    INVENTORY_PATH,
    dtype=str,
    keep_default_na=False
)

print("\nRNA-seq inventory loaded successfully.")

print(f"Metadata rows: {len(inventory)}")

print(
    f"Unique files: "
    f"{inventory['file_id'].nunique()}"
)

print(
    f"Unique patients: "
    f"{inventory['patient_id'].nunique()}"
)

# ------------------------------------------------------------
# 3. Audit file-to-sample relationships
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("RNA-SEQ METADATA QUALITY AUDIT")
print("-" * 65)

file_sample_counts = (
    inventory.groupby("file_id")["sample_id"]
    .nunique()
)

sample_file_counts = (
    inventory[
        inventory["sample_id"] != ""
    ]
    .groupby("sample_id")["file_id"]
    .nunique()
)

multi_sample_files = int(
    (file_sample_counts > 1).sum()
)

multi_file_samples = int(
    (sample_file_counts > 1).sum()
)

print(
    "\nFiles associated with multiple sample IDs:",
    multi_sample_files
)

print(
    "Samples associated with multiple files:",
    multi_file_samples
)

print(
    "Maximum files associated with one sample:",
    sample_file_counts.max()
)

print("\nSamples with multiple files (first 10):")

print(
    sample_file_counts[
        sample_file_counts > 1
    ]
    .sort_values(ascending=False)
    .head(10)
    .to_string()
)

# ------------------------------------------------------------
# 4. Connect to cBioPortal
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CBIOPORTAL CLINICAL METADATA")
print("-" * 65)

session = requests.Session()

session.headers.update({
    "Accept": "application/json",
    "User-Agent": "SurfaceTarget-CRC-Research/1.0"
})

attributes_url = (
    f"{CBIO_API}/studies/"
    f"{STUDY_ID}/clinical-attributes"
)

response = session.get(
    attributes_url,
    params={"pageSize": 10000},
    timeout=60
)

response.raise_for_status()

attributes = response.json()

if not isinstance(attributes, list):
    raise RuntimeError(
        "Unexpected clinical attribute API response."
    )

print("\nStudy:", STUDY_ID)

print(
    "Total available clinical attributes:",
    len(attributes)
)

# ------------------------------------------------------------
# 5. Find MSI-related clinical attributes
# ------------------------------------------------------------

keywords = [
    "MSI",
    "MANTIS",
    "MICROSAT",
    "MSISENSOR",
    "MSI SENSOR"
]

msi_attributes = []

for attribute in attributes:

    searchable_text = " ".join([
        str(attribute.get("id", "")),
        str(attribute.get("displayName", "")),
        str(attribute.get("name", ""))
    ]).upper()

    if any(
        keyword in searchable_text
        for keyword in keywords
    ):
        msi_attributes.append(attribute)

print("\nMSI-related clinical attributes found:")

if not msi_attributes:
    print("No MSI-related attributes identified.")

for attribute in msi_attributes:

    print(
        f"\nID: {attribute.get('id')}"
    )

    print(
        "Name:",
        attribute.get(
            "displayName",
            attribute.get("name", "Unknown")
        )
    )

    print(
        "Patient-level attribute:",
        attribute.get("patientAttribute")
    )

# ------------------------------------------------------------
# 6. Retrieve MSI-related clinical measurements
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("RETRIEVING MSI CLINICAL MEASUREMENTS")
print("-" * 65)

all_clinical_records = []

for attribute in msi_attributes:

    attribute_id = attribute.get("id")

    clinical_level = (
        "PATIENT"
        if attribute.get("patientAttribute", False)
        else "SAMPLE"
    )

    endpoint = (
        f"{CBIO_API}/studies/"
        f"{STUDY_ID}/clinical-data"
    )

    params = {
        "clinicalDataType": clinical_level,
        "attributeId": attribute_id,
        "pageSize": 10000
    }

    print(f"\nRetrieving: {attribute_id}")

    try:

        response = session.get(
            endpoint,
            params=params,
            timeout=90
        )

        response.raise_for_status()

        records = response.json()

        if not isinstance(records, list):
            raise ValueError(
                "Unexpected clinical data format."
            )

        all_clinical_records.extend(records)

        print(
            f"Records retrieved: {len(records)}"
        )

    except Exception as e:

        print(
            f"Retrieval failed for {attribute_id}: {e}"
        )

# ------------------------------------------------------------
# 7. Save clinical annotation data
# ------------------------------------------------------------

clinical_df = pd.DataFrame(
    all_clinical_records
)

clinical_path = (
    METADATA_DIR /
    "tcga_pancancer_msi_clinical_raw.csv"
)

clinical_df.to_csv(
    clinical_path,
    index=False
)

# ------------------------------------------------------------
# 8. Inspect MSI measurements
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("MSI CLINICAL DATA SUMMARY")
print("-" * 65)

if clinical_df.empty:

    print(
        "\nNo MSI clinical measurements retrieved."
    )

else:

    print(
        "\nTotal clinical records:",
        len(clinical_df)
    )

    print(
        "\nAvailable columns:",
        clinical_df.columns.tolist()
    )

    if "clinicalAttributeId" in clinical_df.columns:

        for attribute_id, group in clinical_df.groupby(
            "clinicalAttributeId"
        ):

            print(f"\nAttribute: {attribute_id}")

            print(f"Records: {len(group)}")

            if "value" in group.columns:

                values = group["value"].dropna()

                numeric_values = pd.to_numeric(
                    values,
                    errors="coerce"
                )

                numeric_fraction = (
                    numeric_values.notna().mean()
                    if len(values) else 0
                )

                if numeric_fraction > 0.85:

                    print(
                        "Numeric observations:",
                        numeric_values.notna().sum()
                    )

                    print(
                        "Minimum:",
                        numeric_values.min()
                    )

                    print(
                        "Median:",
                        numeric_values.median()
                    )

                    print(
                        "Maximum:",
                        numeric_values.max()
                    )

                else:

                    print(
                        "\nMost common values:"
                    )

                    print(
                        values.value_counts()
                        .head(10)
                        .to_string()
                    )

# ------------------------------------------------------------
# 9. Check patient identifier overlap
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GDC / CBIOPORTAL PATIENT OVERLAP")
print("-" * 65)

gdc_patients = set(
    inventory["patient_id"].dropna()
)

if (
    not clinical_df.empty
    and "patientId" in clinical_df.columns
):

    cbio_patients = set(
        clinical_df["patientId"].dropna()
    )

    overlap = (
        gdc_patients.intersection(
            cbio_patients
        )
    )

    print(
        "\nGDC RNA-seq patients:",
        len(gdc_patients)
    )

    print(
        "Patients with retrieved MSI annotations:",
        len(cbio_patients)
    )

    print(
        "Overlapping patients:",
        len(overlap)
    )

else:

    overlap = set()

    print(
        "\nPatient overlap could not be evaluated."
    )

# ------------------------------------------------------------
# 10. Save provenance metadata
# ------------------------------------------------------------

metadata = {
    "project": "SurfaceTarget-CRC",

    "retrieved_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": "cBioPortal",

    "study_id": STUDY_ID,

    "api": CBIO_API,

    "msi_attributes": [
        a.get("id")
        for a in msi_attributes
    ],

    "gdc_patients": len(gdc_patients),

    "overlapping_patients": len(overlap),

    "files_with_multiple_samples": multi_sample_files,

    "samples_with_multiple_files": multi_file_samples,

    "msi_status_assigned": False,

    "note": (
        "Exploratory clinical annotation retrieval. "
        "No MSI/MSS classification thresholds "
        "have been applied."
    )
}

metadata_path = (
    METADATA_DIR /
    "tcga_msi_annotation_discovery.json"
)

with open(metadata_path, "w") as f:

    json.dump(
        metadata,
        f,
        indent=4
    )

print("\n" + "=" * 65)
print("FILES SAVED")
print("=" * 65)

print("\n1. tcga_pancancer_msi_clinical_raw.csv")
print("2. tcga_msi_annotation_discovery.json")

print("\nNo MSS labels assigned yet.")
print("No RNA-seq expression data downloaded.")

print("\nPATCH 1.4 COMPLETE")

In [ ]:
# ============================================================
# SurfaceTarget-CRC
# Phase 1 — Patch 1.5
#
# Corrected MSI Annotation Retrieval
# MANTIS + MSIsensor
#
# Objectives:
# - Retrieve specific MSI measurements
# - Validate API responses
# - Check patient overlap
# - Identify duplicate/conflicting measurements
# - Preserve data provenance
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import requests
import pandas as pd
import numpy as np
import json

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/SurfaceTarget-CRC")

METADATA_DIR = PROJECT_ROOT / "data" / "metadata"

CBIO_API = "https://www.cbioportal.org/api"

STUDY_ID = "coadread_tcga_pan_can_atlas_2018"

TARGET_ATTRIBUTES = [
    "MSI_SCORE_MANTIS",
    "MSI_SENSOR_SCORE"
]

INVENTORY_PATH = (
    METADATA_DIR / "tcga_rnaseq_file_inventory.csv"
)

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.5 — CORRECTED MSI RETRIEVAL")
print("=" * 65)

# ------------------------------------------------------------
# 2. Load GDC RNA-seq inventory
# ------------------------------------------------------------

inventory = pd.read_csv(
    INVENTORY_PATH,
    dtype=str,
    keep_default_na=False
)

gdc_patients = set(
    inventory["patient_id"].dropna()
)

print("\nGDC RNA-seq patients:", len(gdc_patients))

# ------------------------------------------------------------
# 3. Initialize API session
# ------------------------------------------------------------

session = requests.Session()

session.headers.update({
    "Accept": "application/json",
    "User-Agent": "SurfaceTarget-CRC-Research/1.0"
})

# ------------------------------------------------------------
# 4. Retrieve clinical attribute definitions
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CLINICAL ATTRIBUTE VALIDATION")
print("-" * 65)

url = (
    f"{CBIO_API}/studies/"
    f"{STUDY_ID}/clinical-attributes"
)

response = session.get(
    url,
    params={"pageSize": 10000},
    timeout=60
)

response.raise_for_status()

attributes = response.json()

attribute_lookup = {
    a["clinicalAttributeId"]: a
    for a in attributes
    if a.get("clinicalAttributeId")
}

for attribute_id in TARGET_ATTRIBUTES:

    if attribute_id not in attribute_lookup:
        raise RuntimeError(
            f"Required attribute unavailable: {attribute_id}"
        )

    attribute = attribute_lookup[attribute_id]

    print(f"\nAttribute ID: {attribute_id}")

    print(
        "Display Name:",
        attribute.get("displayName")
    )

    print(
        "Patient-level:",
        attribute.get("patientAttribute")
    )

print("\nBoth MSI attributes verified.")

# ------------------------------------------------------------
# 5. Retrieve MSI measurements
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("RETRIEVING MSI MEASUREMENTS")
print("-" * 65)

all_records = []

PAGE_SIZE = 1000

for attribute_id in TARGET_ATTRIBUTES:

    print(f"\nRetrieving {attribute_id}")

    attribute_records = []
    page_number = 0

    while True:

        endpoint = (
            f"{CBIO_API}/studies/"
            f"{STUDY_ID}/clinical-data"
        )

        params = {
            "clinicalDataType": "SAMPLE",
            "attributeId": attribute_id,
            "pageSize": PAGE_SIZE,
            "pageNumber": page_number
        }

        response = session.get(
            endpoint,
            params=params,
            timeout=90
        )

        response.raise_for_status()

        records = response.json()

        if not isinstance(records, list):
            raise RuntimeError(
                "Unexpected API response format."
            )

        # Critical validation:
        # Check that every returned measurement
        # belongs to the requested attribute.

        unexpected = [
            record.get("clinicalAttributeId")
            for record in records
            if record.get("clinicalAttributeId")
            != attribute_id
        ]

        if unexpected:
            raise RuntimeError(
                f"API returned unrelated clinical attributes "
                f"for {attribute_id}: {set(unexpected)}"
            )

        attribute_records.extend(records)

        if len(records) < PAGE_SIZE:
            break

        page_number += 1

        if page_number > 100:
            raise RuntimeError(
                "Pagination safety limit exceeded."
            )

    print(
        "Records retrieved:",
        len(attribute_records)
    )

    if not attribute_records:
        raise RuntimeError(
            f"No measurements returned for {attribute_id}"
        )

    all_records.extend(attribute_records)

# ------------------------------------------------------------
# 6. Create corrected MSI dataframe
# ------------------------------------------------------------

msi_df = pd.DataFrame(all_records)

required_columns = [
    "patientId",
    "sampleId",
    "clinicalAttributeId",
    "value"
]

missing_columns = [
    col for col in required_columns
    if col not in msi_df.columns
]

if missing_columns:
    raise RuntimeError(
        f"Missing columns: {missing_columns}"
    )

# Ensure only the requested attributes exist.

returned_attributes = set(
    msi_df["clinicalAttributeId"].unique()
)

assert returned_attributes == set(TARGET_ATTRIBUTES), (
    "Unexpected MSI attributes returned."
)

# Remove exact duplicate records.

msi_df = msi_df.drop_duplicates()

# Convert scores to numeric values.

msi_df["numeric_value"] = pd.to_numeric(
    msi_df["value"],
    errors="coerce"
)

non_numeric = msi_df[
    msi_df["numeric_value"].isna()
]

print("\n" + "-" * 65)
print("DATA VALIDATION")
print("-" * 65)

print(
    "\nTotal corrected MSI records:",
    len(msi_df)
)

print(
    "Non-numeric measurements:",
    len(non_numeric)
)

if not non_numeric.empty:
    print("\nWARNING: Non-numeric measurements found.")
    print(non_numeric.head(10).to_string(index=False))

# ------------------------------------------------------------
# 7. Examine individual MSI measurements
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("MSI MEASUREMENT SUMMARY")
print("-" * 65)

for attribute_id in TARGET_ATTRIBUTES:

    subset = msi_df[
        msi_df["clinicalAttributeId"] == attribute_id
    ]

    scores = subset["numeric_value"].dropna()

    print(f"\n{attribute_id}")

    print("Records:", len(subset))

    print(
        "Unique patients:",
        subset["patientId"].nunique()
    )

    print(
        "Unique samples:",
        subset["sampleId"].nunique()
    )

    print("Numeric scores:", len(scores))

    if not scores.empty:

        print(f"Minimum: {scores.min():.4f}")

        print(f"Median:  {scores.median():.4f}")

        print(f"Maximum: {scores.max():.4f}")

# ------------------------------------------------------------
# 8. Identify duplicate or conflicting measurements
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("DUPLICATE / CONFLICT AUDIT")
print("-" * 65)

# Multiple identical values for the same patient
# are different from conflicting values.

patient_measurement_counts = (
    msi_df.groupby(
        ["patientId", "clinicalAttributeId"]
    )["numeric_value"]
    .nunique()
)

conflicting_measurements = (
    patient_measurement_counts[
        patient_measurement_counts > 1
    ]
)

print(
    "\nPatient-attribute combinations with "
    "conflicting numeric scores:",
    len(conflicting_measurements)
)

if not conflicting_measurements.empty:

    print("\nExamples of conflicts:")

    print(
        conflicting_measurements
        .head(10)
        .to_string()
    )

# ------------------------------------------------------------
# 9. Check GDC patient overlap
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GDC / CBIOPORTAL PATIENT MATCHING")
print("-" * 65)

mantis_patients = set(
    msi_df.loc[
        (msi_df["clinicalAttributeId"] == "MSI_SCORE_MANTIS")
        & msi_df["numeric_value"].notna(),
        "patientId"
    ]
)

sensor_patients = set(
    msi_df.loc[
        (msi_df["clinicalAttributeId"] == "MSI_SENSOR_SCORE")
        & msi_df["numeric_value"].notna(),
        "patientId"
    ]
)

both_patients = (
    mantis_patients.intersection(sensor_patients)
)

print("\nGDC RNA-seq patients:", len(gdc_patients))

print(
    "Patients with MANTIS scores:",
    len(mantis_patients)
)

print(
    "Patients with MSIsensor scores:",
    len(sensor_patients)
)

print(
    "Patients with both MSI scores:",
    len(both_patients)
)

print("\nOverlapping with GDC RNA-seq:")

print(
    "MANTIS:",
    len(gdc_patients & mantis_patients)
)

print(
    "MSIsensor:",
    len(gdc_patients & sensor_patients)
)

print(
    "Both measurements:",
    len(gdc_patients & both_patients)
)

# ------------------------------------------------------------
# 10. Project-specific patient overlap
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PROJECT-SPECIFIC OVERLAP")
print("-" * 65)

for project_id in ["TCGA-COAD", "TCGA-READ"]:

    project_patients = set(
        inventory.loc[
            inventory["project"] == project_id,
            "patient_id"
        ]
    )

    print(f"\n{project_id}")

    print(
        "RNA-seq patients:",
        len(project_patients)
    )

    print(
        "With MANTIS:",
        len(project_patients & mantis_patients)
    )

    print(
        "With MSIsensor:",
        len(project_patients & sensor_patients)
    )

    print(
        "With both:",
        len(project_patients & both_patients)
    )

# ------------------------------------------------------------
# 11. Save corrected MSI dataset
# ------------------------------------------------------------

output_path = (
    METADATA_DIR /
    "tcga_pancancer_msi_clinical_corrected.csv"
)

msi_df.to_csv(
    output_path,
    index=False
)

# ------------------------------------------------------------
# 12. Save provenance metadata
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "retrieved_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": "cBioPortal",

    "study_id": STUDY_ID,

    "attributes": TARGET_ATTRIBUTES,

    "corrected_msi_records": len(msi_df),

    "gdc_rnaseq_patients": len(gdc_patients),

    "patients_with_mantis": len(mantis_patients),

    "patients_with_msisensor": len(sensor_patients),

    "patients_with_both": len(both_patients),

    "gdc_overlap_both": len(
        gdc_patients & both_patients
    ),

    "conflicting_patient_attribute_measurements": len(
        conflicting_measurements
    ),

    "msi_classification_performed": False,

    "note": (
        "Corrected retrieval using clinicalAttributeId. "
        "No MSS/MSI-H classification applied. "
        "Patient-level score conflicts must be resolved "
        "before cohort assignment."
    )
}

provenance_path = (
    METADATA_DIR /
    "tcga_msi_corrected_provenance.json"
)

with open(provenance_path, "w") as f:

    json.dump(
        provenance,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 13. Final status
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.5 SUMMARY")
print("=" * 65)

print("\nCorrected MSI dataset saved:")
print(output_path)

print("\nProvenance metadata saved:")
print(provenance_path)

print("\nNo MSS/MSI-H labels assigned.")
print("No molecular datasets downloaded.")

print("\nPATCH 1.5 COMPLETE")

In [ ]:
# ============================================================
# SurfaceTarget-CRC
# Phase 1 — Patch 1.6
#
# Preliminary MSI Cohort Classification
#
# Scientific objective:
# Identify patients with concordant low MSI scores
# using published MANTIS and MSIsensor thresholds.
#
# IMPORTANT:
# Concordant low MSI is NOT equivalent to independently
# verified MSS status.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import json

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/SurfaceTarget-CRC")

METADATA_DIR = PROJECT_ROOT / "data" / "metadata"

RESULTS_DIR = PROJECT_ROOT / "results" / "tables"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MSI_PATH = (
    METADATA_DIR /
    "tcga_pancancer_msi_clinical_corrected.csv"
)

INVENTORY_PATH = (
    METADATA_DIR /
    "tcga_rnaseq_file_inventory.csv"
)

# Published default thresholds

MANTIS_THRESHOLD = 0.4
MSISENSOR_THRESHOLD = 3.5

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.6 — PRELIMINARY MSI COHORT CLASSIFICATION")
print("=" * 65)

# ------------------------------------------------------------
# 2. Load data
# ------------------------------------------------------------

msi = pd.read_csv(MSI_PATH)

inventory = pd.read_csv(
    INVENTORY_PATH,
    dtype=str
)

print("\nMSI records loaded:", len(msi))

print(
    "GDC RNA-seq patients:",
    inventory["patient_id"].nunique()
)

# ------------------------------------------------------------
# 3. Validate MSI data
# ------------------------------------------------------------

required_columns = [
    "patientId",
    "sampleId",
    "clinicalAttributeId",
    "numeric_value"
]

missing = [
    col for col in required_columns
    if col not in msi.columns
]

if missing:
    raise RuntimeError(
        f"Missing required MSI columns: {missing}"
    )

msi["numeric_value"] = pd.to_numeric(
    msi["numeric_value"],
    errors="raise"
)

attributes = {
    "MSI_SCORE_MANTIS",
    "MSI_SENSOR_SCORE"
}

if set(msi["clinicalAttributeId"]) != attributes:
    raise RuntimeError(
        "Unexpected MSI attributes."
    )

# Ensure one measurement per patient per method.

duplicates = msi.duplicated(
    subset=["patientId", "clinicalAttributeId"]
)

if duplicates.any():
    raise RuntimeError(
        "Duplicate patient-method measurements detected. "
        "Resolve these before classification."
    )

print("\nMSI data validation: PASSED")

# ------------------------------------------------------------
# 4. Create patient-level score table
# ------------------------------------------------------------

scores = msi.pivot(
    index="patientId",
    columns="clinicalAttributeId",
    values="numeric_value"
)

scores = scores.rename(columns={
    "MSI_SCORE_MANTIS": "mantis_score",
    "MSI_SENSOR_SCORE": "msisensor_score"
})

# Track the original sample IDs used by each method.

sample_ids = msi.pivot(
    index="patientId",
    columns="clinicalAttributeId",
    values="sampleId"
)

sample_ids = sample_ids.rename(columns={
    "MSI_SCORE_MANTIS": "mantis_sample_id",
    "MSI_SENSOR_SCORE": "msisensor_sample_id"
})

patient_msi = scores.join(sample_ids)

# ------------------------------------------------------------
# 5. Restrict to patients with primary tumor RNA-seq
# ------------------------------------------------------------

primary_inventory = inventory[
    inventory["sample_type"] == "Primary Tumor"
].copy()

primary_patients = (
    primary_inventory[
        ["patient_id", "project"]
    ]
    .drop_duplicates()
)

# Ensure patients are not assigned to multiple projects.

project_conflicts = (
    primary_patients.groupby("patient_id")["project"]
    .nunique()
)

if (project_conflicts > 1).any():
    raise RuntimeError(
        "Patients assigned to multiple TCGA projects."
    )

primary_patients = primary_patients.rename(
    columns={
        "patient_id": "patientId",
        "project": "gdc_project"
    }
)

cohort = primary_patients.merge(
    patient_msi.reset_index(),
    on="patientId",
    how="left",
    validate="one_to_one"
)

print("\nPrimary tumor RNA-seq patients:", len(cohort))

# ------------------------------------------------------------
# 6. Evaluate measurement availability
# ------------------------------------------------------------

has_mantis = cohort["mantis_score"].notna()

has_sensor = cohort["msisensor_score"].notna()

has_both = has_mantis & has_sensor

same_sample = (
    has_both
    & (
        cohort["mantis_sample_id"]
        == cohort["msisensor_sample_id"]
    )
)

print("\nPatients with both scores:", has_both.sum())

print(
    "Both scores from matching cBioPortal samples:",
    same_sample.sum()
)

print(
    "Both scores but different sample IDs:",
    (has_both & ~same_sample).sum()
)

# ------------------------------------------------------------
# 7. Apply published MSI thresholds
# ------------------------------------------------------------

mantis_low = (
    cohort["mantis_score"] < MANTIS_THRESHOLD
)

sensor_low = (
    cohort["msisensor_score"] < MSISENSOR_THRESHOLD
)

mantis_high = (
    cohort["mantis_score"] >= MANTIS_THRESHOLD
)

sensor_high = (
    cohort["msisensor_score"] >= MSISENSOR_THRESHOLD
)

concordant_low = (
    same_sample
    & mantis_low
    & sensor_low
)

concordant_high = (
    same_sample
    & mantis_high
    & sensor_high
)

discordant = (
    same_sample
    & (
        (mantis_low & sensor_high)
        | (mantis_high & sensor_low)
    )
)

sample_mismatch = has_both & ~same_sample

incomplete = ~has_both

cohort["msi_group"] = np.select(
    [
        concordant_low,
        concordant_high,
        discordant,
        sample_mismatch,
        incomplete
    ],
    [
        "Concordant low MSI",
        "Concordant MSI-H",
        "Discordant",
        "Sample ID mismatch",
        "Incomplete"
    ],
    default="Unclassified"
)

# ------------------------------------------------------------
# 8. Display cohort distribution
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PRELIMINARY MSI CLASSIFICATION")
print("-" * 65)

classification_counts = (
    cohort["msi_group"]
    .value_counts(dropna=False)
)

print("\nOverall distribution:")

print(classification_counts.to_string())

# ------------------------------------------------------------
# 9. Project-specific distribution
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PROJECT-SPECIFIC MSI DISTRIBUTION")
print("-" * 65)

project_summary = pd.crosstab(
    cohort["gdc_project"],
    cohort["msi_group"]
)

print("\n")
print(project_summary.to_string())

# ------------------------------------------------------------
# 10. Preliminary research cohort
# ------------------------------------------------------------

preliminary_cohort = cohort[
    cohort["msi_group"] == "Concordant low MSI"
].copy()

print("\n" + "-" * 65)
print("PRELIMINARY LOW-MSI RESEARCH COHORT")
print("-" * 65)

print(
    "\nTotal candidate patients:",
    len(preliminary_cohort)
)

print("\nPatients by project:")

print(
    preliminary_cohort["gdc_project"]
    .value_counts()
    .to_string()
)

print("\nIMPORTANT:")
print(
    "These patients have concordant low MSI scores."
)

print(
    "They are not yet independently verified as MSS."
)

# ------------------------------------------------------------
# 11. Save results
# ------------------------------------------------------------

full_path = (
    METADATA_DIR /
    "tcga_msi_preliminary_patient_classification.csv"
)

cohort.to_csv(
    full_path,
    index=False
)

preliminary_path = (
    METADATA_DIR /
    "tcga_preliminary_low_msi_cohort.csv"
)

preliminary_cohort.to_csv(
    preliminary_path,
    index=False
)

summary_path = (
    RESULTS_DIR /
    "tcga_preliminary_msi_group_summary.csv"
)

project_summary.to_csv(summary_path)

# ------------------------------------------------------------
# 12. Record classification methodology
# ------------------------------------------------------------

methodology = {
    "project": "SurfaceTarget-CRC",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": (
        "cBioPortal TCGA Colorectal "
        "PanCancer Atlas 2018"
    ),

    "classification_type": "Preliminary computational",

    "thresholds": {
        "MANTIS": MANTIS_THRESHOLD,
        "MSIsensor": MSISENSOR_THRESHOLD
    },

    "low_msi_rule": (
        "Both MANTIS < 0.4 and MSIsensor < 3.5, "
        "with matching cBioPortal sample IDs."
    ),

    "primary_tumor_rnaseq_required": True,

    "preliminary_low_msi_patients": int(
        len(preliminary_cohort)
    ),

    "limitations": [
        "Low MSI scores do not distinguish MSS from MSI-L.",
        "Computational classifications are not clinical validation.",
        "Independent MSI annotations remain to be evaluated.",
        "TCGA and cBioPortal cohort differences require documentation."
    ],

    "references": [
        "https://pmc.ncbi.nlm.nih.gov/articles/PMC5352334/",
        "https://pmc.ncbi.nlm.nih.gov/articles/PMC11317526/"
    ]
}

methodology_path = (
    METADATA_DIR /
    "tcga_preliminary_msi_methodology.json"
)

with open(methodology_path, "w") as f:
    json.dump(
        methodology,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 13. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.6 SUMMARY")
print("=" * 65)

print("\nFiles saved:")

print("1. tcga_msi_preliminary_patient_classification.csv")
print("2. tcga_preliminary_low_msi_cohort.csv")
print("3. tcga_preliminary_msi_group_summary.csv")
print("4. tcga_preliminary_msi_methodology.json")

print("\nNo expression matrices downloaded.")

print("\nPATCH 1.6 COMPLETE")

In [ ]:
# ============================================================
# SurfaceTarget-CRC
# Phase 1 — Patch 1.7
#
# Additional MSI Annotation Discovery
# and Research Checkpoint
#
# Objectives:
# 1. Examine related TCGA colorectal studies
# 2. Identify MSI-related clinical attributes
# 3. Identify potential categorical annotations
# 4. Save discovery metadata
# 5. Create a research checkpoint
#
# No additional patient classifications are performed.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
import requests
import json
import re

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/SurfaceTarget-CRC")

METADATA_DIR = PROJECT_ROOT / "data" / "metadata"

CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CBIO_API = "https://www.cbioportal.org/api"

# Candidate TCGA colorectal studies.
# Some may be unavailable or retired.
# Unavailable studies will be recorded and skipped.

STUDY_IDS = [
    "coadread_tcga",
    "coadread_tcga_pub",
    "coadread_tcga_pan_can_atlas_2018",
    "coad_tcga",
    "read_tcga"
]

# Relevant clinical annotation keywords

KEYWORDS = [
    "MSI",
    "MICROSAT",
    "MANTIS",
    "MISMATCH",
    "MMR"
]

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.7 — ADDITIONAL MSI ANNOTATION DISCOVERY")
print("=" * 65)

# ------------------------------------------------------------
# 2. Load preliminary cohort
# ------------------------------------------------------------

COHORT_PATH = (
    METADATA_DIR /
    "tcga_msi_preliminary_patient_classification.csv"
)

cohort = pd.read_csv(COHORT_PATH)

print("\nPreliminary cohort loaded.")

print("Total primary tumor patients:", len(cohort))

print("\nCurrent classification:")

print(
    cohort["msi_group"]
    .value_counts()
    .to_string()
)

# ------------------------------------------------------------
# 3. Initialize API session
# ------------------------------------------------------------

session = requests.Session()

session.headers.update({
    "Accept": "application/json",
    "User-Agent": "SurfaceTarget-CRC-Research/1.0"
})

# ------------------------------------------------------------
# 4. Investigate available studies
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CBIOPORTAL STUDY DISCOVERY")
print("-" * 65)

all_attributes = []

study_status = {}

for study_id in STUDY_IDS:

    print(f"\nInvestigating: {study_id}")

    endpoint = (
        f"{CBIO_API}/studies/"
        f"{study_id}/clinical-attributes"
    )

    try:

        response = session.get(
            endpoint,
            params={"pageSize": 10000},
            timeout=60
        )

        if response.status_code == 404:

            print("Status: Study unavailable")

            study_status[study_id] = "unavailable"

            continue

        response.raise_for_status()

        attributes = response.json()

        if not isinstance(attributes, list):

            raise ValueError(
                "Unexpected clinical attribute response."
            )

        study_status[study_id] = "available"

        print(
            "Total clinical attributes:",
            len(attributes)
        )

        # --------------------------------------------
        # Identify MSI-related attributes
        # --------------------------------------------

        matching_attributes = []

        for attribute in attributes:

            attribute_id = (
                attribute.get("clinicalAttributeId")
                or attribute.get("id")
                or ""
            )

            display_name = (
                attribute.get("displayName")
                or attribute.get("name")
                or ""
            )

            description = (
                attribute.get("description")
                or ""
            )

            searchable_text = " ".join([
                str(attribute_id),
                str(display_name),
                str(description)
            ]).upper()

            if any(
                keyword in searchable_text
                for keyword in KEYWORDS
            ):

                record = {

                    "study_id": study_id,

                    "attribute_id": attribute_id,

                    "display_name": display_name,

                    "description": description,

                    "data_type": attribute.get(
                        "dataType", "Unknown"
                    ),

                    "patient_attribute": attribute.get(
                        "patientAttribute"
                    )
                }

                matching_attributes.append(record)

                all_attributes.append(record)

        print(
            "MSI-related attributes:",
            len(matching_attributes)
        )

        for attribute in matching_attributes:

            print(
                f"  - {attribute['attribute_id']}"
            )

            print(
                f"    Name: {attribute['display_name']}"
            )

            print(
                f"    Data type: {attribute['data_type']}"
            )

            level = (
                "Patient"
                if attribute["patient_attribute"]
                else "Sample"
            )

            print(f"    Level: {level}")

    except Exception as e:

        print(f"Status: ERROR — {e}")

        study_status[study_id] = (
            f"error: {str(e)}"
        )

# ------------------------------------------------------------
# 5. Create discovery dataframe
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("MSI ANNOTATION DISCOVERY SUMMARY")
print("-" * 65)

discovery_df = pd.DataFrame(
    all_attributes,
    columns=[
        "study_id",
        "attribute_id",
        "display_name",
        "description",
        "data_type",
        "patient_attribute"
    ]
)

print(
    "\nTotal MSI-related attribute definitions:",
    len(discovery_df)
)

if not discovery_df.empty:

    print("\nAttributes by study:")

    print(
        discovery_df.groupby("study_id")
        .size()
        .to_string()
    )

# ------------------------------------------------------------
# 6. Identify possible categorical MSI fields
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("POTENTIAL CATEGORICAL MSI ANNOTATIONS")
print("-" * 65)

if not discovery_df.empty:

    # These are only candidate fields.
    # Their actual contents and provenance
    # must be inspected before use.

    categorical_candidates = discovery_df[
        discovery_df["data_type"]
        .astype(str)
        .str.upper()
        .eq("STRING")
    ].copy()

    print(
        "\nPotential categorical fields:",
        len(categorical_candidates)
    )

    if not categorical_candidates.empty:

        print(
            categorical_candidates[
                [
                    "study_id",
                    "attribute_id",
                    "display_name"
                ]
            ].to_string(index=False)
        )

    else:

        print(
            "No STRING-type MSI-related fields identified."
        )

else:

    categorical_candidates = discovery_df.copy()

    print(
        "No MSI-related attributes identified."
    )

# ------------------------------------------------------------
# 7. Save discovery results
# ------------------------------------------------------------

discovery_path = (
    METADATA_DIR /
    "tcga_additional_msi_attribute_discovery.csv"
)

discovery_df.to_csv(
    discovery_path,
    index=False
)

categorical_path = (
    METADATA_DIR /
    "tcga_potential_categorical_msi_attributes.csv"
)

categorical_candidates.to_csv(
    categorical_path,
    index=False
)

# ------------------------------------------------------------
# 8. Save provenance
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "retrieved_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": "cBioPortal",

    "api": CBIO_API,

    "studies_investigated": STUDY_IDS,

    "study_status": study_status,

    "total_matching_attribute_definitions": int(
        len(discovery_df)
    ),

    "potential_categorical_fields": int(
        len(categorical_candidates)
    ),

    "msi_status_assigned": False,

    "note": (
        "Discovery of additional MSI clinical attributes. "
        "No new classifications assigned. "
        "Annotation contents and provenance require "
        "evaluation before inclusion in the study."
    )
}

provenance_path = (
    METADATA_DIR /
    "tcga_additional_msi_discovery_provenance.json"
)

with open(provenance_path, "w") as f:

    json.dump(
        provenance,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 9. Create research checkpoint
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING RESEARCH CHECKPOINT")
print("-" * 65)

timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S")

checkpoint_path = (
    CHECKPOINT_DIR /
    f"SurfaceTarget_CRC_Phase1_Checkpoint_{timestamp}.zip"
)

# Include metadata and small research outputs.
# Exclude raw molecular datasets.

backup_directories = [
    "data/metadata",
    "results/tables",
    "results/figures",
    "src",
    "reports",
    "logs"
]

with ZipFile(
    checkpoint_path,
    "w",
    compression=ZIP_DEFLATED
) as zip_file:

    for directory in backup_directories:

        directory_path = (
            PROJECT_ROOT / directory
        )

        if directory_path.exists():

            for file_path in directory_path.rglob("*"):

                if file_path.is_file():

                    archive_name = (
                        file_path.relative_to(PROJECT_ROOT)
                    )

                    zip_file.write(
                        file_path,
                        arcname=str(archive_name)
                    )

checkpoint_mb = (
    checkpoint_path.stat().st_size
    / (1024 ** 2)
)

print("\nCheckpoint created successfully.")

print(
    f"Checkpoint size: {checkpoint_mb:.2f} MB"
)

print("\nCheckpoint location:")
print(checkpoint_path)

# ------------------------------------------------------------
# 10. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.7 SUMMARY")
print("=" * 65)

print("\nFiles saved:")

print(
    "1. tcga_additional_msi_attribute_discovery.csv"
)

print(
    "2. tcga_potential_categorical_msi_attributes.csv"
)

print(
    "3. tcga_additional_msi_discovery_provenance.json"
)

print("\nResearch checkpoint created.")

print("\nNo new MSS labels assigned.")
print("No molecular expression datasets downloaded.")

print("\nPATCH 1.7 COMPLETE")

In [ ]:
# ============================================================
# SurfaceTarget-CRC
# Phase 1 — Patch 1.8
#
# Published TCGA MSI Classification Retrieval
#
# Source:
# cBioPortal - coadread_tcga_pub
#
# Objectives:
# 1. Retrieve published MSI_STATUS annotations
# 2. Inspect actual categorical values
# 3. Identify patient-level annotation conflicts
# 4. Match annotations to the GDC RNA-seq cohort
# 5. Compare with preliminary computational MSI groups
#
# IMPORTANT:
# No final MSS cohort assignment is performed.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import requests
import json

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/SurfaceTarget-CRC")

METADATA_DIR = PROJECT_ROOT / "data" / "metadata"

RESULTS_DIR = PROJECT_ROOT / "results" / "tables"

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CBIO_API = "https://www.cbioportal.org/api"

STUDY_ID = "coadread_tcga_pub"

ATTRIBUTE_ID = "MSI_STATUS"

COHORT_PATH = (
    METADATA_DIR /
    "tcga_msi_preliminary_patient_classification.csv"
)

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.8 — PUBLISHED MSI CLASSIFICATION")
print("=" * 65)

# ------------------------------------------------------------
# 2. Load preliminary cohort
# ------------------------------------------------------------

cohort = pd.read_csv(
    COHORT_PATH,
    dtype={"patientId": str}
)

if cohort["patientId"].duplicated().any():
    raise RuntimeError(
        "Duplicate patients in preliminary cohort."
    )

print("\nPreliminary cohort loaded.")

print("Total patients:", len(cohort))

print("\nExisting MSI groups:")

print(
    cohort["msi_group"]
    .value_counts()
    .to_string()
)

# ------------------------------------------------------------
# 3. Initialize API session
# ------------------------------------------------------------

session = requests.Session()

session.headers.update({
    "Accept": "application/json",
    "User-Agent": "SurfaceTarget-CRC-Research/1.0"
})

# ------------------------------------------------------------
# 4. Verify published MSI attribute
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PUBLISHED MSI ATTRIBUTE VERIFICATION")
print("-" * 65)

attributes_url = (
    f"{CBIO_API}/studies/"
    f"{STUDY_ID}/clinical-attributes"
)

response = session.get(
    attributes_url,
    params={"pageSize": 10000},
    timeout=60
)

response.raise_for_status()

attributes = response.json()

matches = [
    a for a in attributes
    if a.get("clinicalAttributeId") == ATTRIBUTE_ID
]

if len(matches) != 1:
    raise RuntimeError(
        "MSI_STATUS attribute could not be uniquely verified."
    )

attribute = matches[0]

print("\nStudy:", STUDY_ID)

print("Attribute ID:", ATTRIBUTE_ID)

print(
    "Display name:",
    attribute.get("displayName")
)

print(
    "Data type:",
    attribute.get(
        "datatype",
        attribute.get("dataType", "Not specified")
    )
)

print(
    "Patient-level:",
    attribute.get("patientAttribute")
)

if attribute.get("patientAttribute") is True:
    raise RuntimeError(
        "MSI_STATUS is patient-level; "
        "the retrieval method needs adjustment."
    )

print("\nAttribute verification: PASSED")

# ------------------------------------------------------------
# 5. Retrieve published MSI classifications
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("RETRIEVING PUBLISHED MSI STATUS")
print("-" * 65)

endpoint = (
    f"{CBIO_API}/studies/"
    f"{STUDY_ID}/clinical-data"
)

all_records = []

PAGE_SIZE = 500
page_number = 0

while True:

    params = {
        "clinicalDataType": "SAMPLE",
        "attributeId": ATTRIBUTE_ID,
        "pageSize": PAGE_SIZE,
        "pageNumber": page_number
    }

    response = session.get(
        endpoint,
        params=params,
        timeout=90
    )

    response.raise_for_status()

    records = response.json()

    if not isinstance(records, list):
        raise RuntimeError(
            "Unexpected clinical API response."
        )

    # Ensure the API returned only MSI_STATUS.

    unexpected = {
        record.get("clinicalAttributeId")
        for record in records
        if record.get("clinicalAttributeId") != ATTRIBUTE_ID
    }

    if unexpected:
        raise RuntimeError(
            f"Unexpected attributes returned: {unexpected}"
        )

    all_records.extend(records)

    print(
        f"Page {page_number + 1}: "
        f"{len(records)} records"
    )

    if len(records) < PAGE_SIZE:
        break

    page_number += 1

    if page_number > 100:
        raise RuntimeError(
            "Pagination safety limit exceeded."
        )

if not all_records:
    raise RuntimeError(
        "No published MSI_STATUS records retrieved."
    )

# ------------------------------------------------------------
# 6. Create published MSI dataframe
# ------------------------------------------------------------

published_msi = pd.DataFrame(all_records)

required_columns = [
    "patientId",
    "sampleId",
    "clinicalAttributeId",
    "value"
]

missing = [
    col for col in required_columns
    if col not in published_msi.columns
]

if missing:
    raise RuntimeError(
        f"Required columns missing: {missing}"
    )

published_msi = (
    published_msi
    .drop_duplicates()
    .copy()
)

# Preserve original classifications.

published_msi["original_status"] = (
    published_msi["value"]
    .astype("string")
)

# Normalize formatting only.
# Do not combine MSS, MSI-L, or MSI-H.

published_msi["status_clean"] = (
    published_msi["original_status"]
    .str.strip()
    .str.upper()
)

published_msi.loc[
    published_msi["status_clean"].isin(
        ["", "NA", "N/A", "UNKNOWN", "NOT AVAILABLE"]
    ),
    "status_clean"
] = pd.NA

print("\n" + "-" * 65)
print("PUBLISHED MSI DATA SUMMARY")
print("-" * 65)

print(
    "\nTotal records:",
    len(published_msi)
)

print(
    "Unique patients:",
    published_msi["patientId"].nunique()
)

print(
    "Unique samples:",
    published_msi["sampleId"].nunique()
)

# ------------------------------------------------------------
# 7. Examine actual MSI classifications
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PUBLISHED MSI STATUS DISTRIBUTION")
print("-" * 65)

print("\nOriginal classification values:")

print(
    published_msi["original_status"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nNormalized classification values:")

print(
    published_msi["status_clean"]
    .value_counts(dropna=False)
    .to_string()
)

# ------------------------------------------------------------
# 8. Audit multiple annotations per patient
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PATIENT-LEVEL ANNOTATION AUDIT")
print("-" * 65)

patient_summary = (
    published_msi.groupby("patientId")
    .agg(
        n_samples=("sampleId", "nunique"),
        n_unique_statuses=("status_clean", "nunique"),
        published_status=("status_clean", "first")
    )
    .reset_index()
)

conflicts = patient_summary[
    patient_summary["n_unique_statuses"] > 1
].copy()

print(
    "\nPatients with multiple annotated samples:",
    (patient_summary["n_samples"] > 1).sum()
)

print(
    "Patients with conflicting MSI classifications:",
    len(conflicts)
)

if not conflicts.empty:

    print("\nConflicting patient examples:")

    print(
        conflicts.head(10).to_string(index=False)
    )

# Do not assign a single patient-level status
# when annotations are missing or conflicting.

ambiguous = (
    patient_summary["n_unique_statuses"] != 1
)

patient_summary.loc[
    ambiguous,
    "published_status"
] = pd.NA

# ------------------------------------------------------------
# 9. Match published annotations to GDC cohort
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GDC / PUBLISHED MSI PATIENT MATCHING")
print("-" * 65)

comparison = cohort.merge(
    patient_summary,
    on="patientId",
    how="left",
    validate="one_to_one"
)

has_published_status = (
    comparison["published_status"].notna()
)

print("\nGDC primary tumor RNA-seq patients:", len(cohort))

print(
    "Patients with unambiguous published MSI status:",
    has_published_status.sum()
)

print(
    "Patients without usable published MSI status:",
    (~has_published_status).sum()
)

# ------------------------------------------------------------
# 10. Check sample identifier correspondence
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CROSS-STUDY SAMPLE ID CHECK")
print("-" * 65)

# Compare published study sample IDs with the
# PanCancer Atlas MSI sample identifiers.

published_sample_sets = (
    published_msi.groupby("patientId")["sampleId"]
    .agg(lambda x: set(x.dropna()))
    .to_dict()
)

sample_check = comparison[
    comparison["published_status"].notna()
    & comparison["mantis_sample_id"].notna()
].copy()

sample_check["exact_sample_match"] = (
    sample_check.apply(
        lambda row:
        str(row["mantis_sample_id"])
        in published_sample_sets.get(
            row["patientId"], set()
        ),
        axis=1
    )
)

print(
    "\nPatients eligible for sample ID comparison:",
    len(sample_check)
)

print(
    "Exact cross-study sample ID matches:",
    sample_check["exact_sample_match"].sum()
)

print(
    "Non-matching sample IDs:",
    (~sample_check["exact_sample_match"]).sum()
)

if (~sample_check["exact_sample_match"]).any():

    print("\nExamples requiring sample-level review:")

    examples = sample_check.loc[
        ~sample_check["exact_sample_match"],
        ["patientId", "mantis_sample_id"]
    ].head(5)

    print(examples.to_string(index=False))

print(
    "\nNOTE: Sample IDs may differ in barcode detail "
    "between studies. Non-matches require review."
)

# ------------------------------------------------------------
# 11. Compare published and computational groups
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PUBLISHED VS COMPUTATIONAL MSI")
print("-" * 65)

available = comparison[
    comparison["published_status"].notna()
].copy()

if available.empty:

    print("\nNo overlapping classifications available.")

else:

    cross_tab = pd.crosstab(
        available["msi_group"],
        available["published_status"]
    )

    print("\nClassification comparison:")

    print(cross_tab.to_string())

# ------------------------------------------------------------
# 12. Specifically examine preliminary low-MSI cases
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PRELIMINARY LOW-MSI COHORT EVALUATION")
print("-" * 65)

low_msi = comparison[
    comparison["msi_group"] == "Concordant low MSI"
].copy()

print(
    "\nTotal preliminary low-MSI patients:",
    len(low_msi)
)

print(
    "With published MSI classification:",
    low_msi["published_status"].notna().sum()
)

print("\nPublished status distribution:")

print(
    low_msi["published_status"]
    .fillna("NOT AVAILABLE")
    .value_counts()
    .to_string()
)

print("\nBy TCGA project:")

project_comparison = pd.crosstab(
    low_msi["gdc_project"],
    low_msi["published_status"].fillna("NOT AVAILABLE")
)

print(project_comparison.to_string())

# ------------------------------------------------------------
# 13. Save research outputs
# ------------------------------------------------------------

raw_path = (
    METADATA_DIR /
    "tcga_published_msi_status_raw.csv"
)

patient_path = (
    METADATA_DIR /
    "tcga_published_msi_patient_summary.csv"
)

comparison_path = (
    METADATA_DIR /
    "tcga_computational_vs_published_msi.csv"
)

published_msi.to_csv(
    raw_path,
    index=False
)

patient_summary.to_csv(
    patient_path,
    index=False
)

comparison.to_csv(
    comparison_path,
    index=False
)

# ------------------------------------------------------------
# 14. Save provenance
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "retrieved_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": "cBioPortal",

    "study_id": STUDY_ID,

    "attribute": ATTRIBUTE_ID,

    "total_records": int(len(published_msi)),

    "unique_patients": int(
        published_msi["patientId"].nunique()
    ),

    "gdc_overlap_with_usable_status": int(
        has_published_status.sum()
    ),

    "patients_with_conflicting_annotations": int(
        len(conflicts)
    ),

    "final_mss_cohort_assigned": False,

    "limitations": [
        "Published classifications may cover only a subset of TCGA patients.",
        "Cross-study sample identifiers require verification.",
        "Published MSI and computational scores may not be independent.",
        "No final MSS cohort has been established."
    ]
}

provenance_path = (
    METADATA_DIR /
    "tcga_published_msi_provenance.json"
)

with open(provenance_path, "w") as f:

    json.dump(
        provenance,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 15. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.8 SUMMARY")
print("=" * 65)

print("\nFiles saved:")

print("1. tcga_published_msi_status_raw.csv")

print("2. tcga_published_msi_patient_summary.csv")

print("3. tcga_computational_vs_published_msi.csv")

print("4. tcga_published_msi_provenance.json")

print("\nNo final MSS cohort assigned.")

print("No molecular expression datasets downloaded.")

print("\nPATCH 1.8 COMPLETE")

In [ ]:
# ============================================================
# SurfaceTarget-CRC
# Phase 1 — Patch 1.9 (CORRECTED)
#
# MSS Evidence Tiers and Sample Identifier Validation
#
# Objectives:
# 1. Define preliminary MSS evidence tiers
# 2. Validate GDC / cBioPortal sample correspondence
# 3. Exclude published MSI-L from strict MSS eligibility
# 4. Flag sample mismatches for review
# 5. Save reproducible patient-level cohort metadata
#
# IMPORTANT:
# This is a cohort eligibility audit.
# Final RNA-seq sample selection has not been performed.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import json
import re

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/SurfaceTarget-CRC")

METADATA_DIR = PROJECT_ROOT / "data" / "metadata"

RESULTS_DIR = PROJECT_ROOT / "results" / "tables"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

COMPARISON_PATH = (
    METADATA_DIR /
    "tcga_computational_vs_published_msi.csv"
)

PUBLISHED_PATH = (
    METADATA_DIR /
    "tcga_published_msi_status_raw.csv"
)

INVENTORY_PATH = (
    METADATA_DIR /
    "tcga_rnaseq_file_inventory.csv"
)

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.9 — MSS COHORT EVIDENCE VALIDATION")
print("=" * 65)

# ------------------------------------------------------------
# 2. Load existing data
# ------------------------------------------------------------

required_files = [
    COMPARISON_PATH,
    PUBLISHED_PATH,
    INVENTORY_PATH
]

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found: {path}"
        )

comparison = pd.read_csv(
    COMPARISON_PATH,
    dtype={"patientId": str}
)

published = pd.read_csv(
    PUBLISHED_PATH,
    dtype=str
)

inventory = pd.read_csv(
    INVENTORY_PATH,
    dtype=str
)

print("\nData loaded successfully.")

print("Patient-level records:", len(comparison))

print("Published MSI records:", len(published))

print(
    "GDC RNA-seq files:",
    inventory["file_id"].nunique()
)

# ------------------------------------------------------------
# 3. Validate patient identifiers
# ------------------------------------------------------------

if comparison["patientId"].duplicated().any():
    raise RuntimeError(
        "Duplicate patient identifiers detected."
    )

required_groups = {
    "Concordant low MSI",
    "Concordant MSI-H",
    "Discordant",
    "Incomplete"
}

observed_groups = set(
    comparison["msi_group"].dropna()
)

if not observed_groups.issubset(required_groups):
    raise RuntimeError(
        "Unexpected computational MSI classifications."
    )

print("\nPatient identifier validation: PASSED")

# ------------------------------------------------------------
# 4. Normalize published classifications
# ------------------------------------------------------------

published["status_clean"] = (
    published["status_clean"]
    .astype("string")
    .str.strip()
    .str.upper()
)

allowed_statuses = {
    "MSS",
    "MSI-L",
    "MSI-H",
    "NOT EVALUABLE"
}

unexpected = (
    set(published["status_clean"].dropna())
    - allowed_statuses
)

if unexpected:
    raise RuntimeError(
        f"Unexpected published MSI labels: {unexpected}"
    )

# Require one published annotation per patient.

if published["patientId"].duplicated().any():
    raise RuntimeError(
        "Multiple published annotations per patient. "
        "Manual review is required."
    )

published_subset = published[
    ["patientId", "sampleId", "status_clean"]
].rename(
    columns={
        "sampleId": "published_sample_id",
        "status_clean": "published_msi_status"
    }
)

# ------------------------------------------------------------
# 5. Merge published annotations
# ------------------------------------------------------------

cohort = comparison.drop(
    columns=["published_status"],
    errors="ignore"
).merge(
    published_subset,
    on="patientId",
    how="left",
    validate="one_to_one"
)

print("\nPublished annotation distribution:")

print(
    cohort["published_msi_status"]
    .fillna("NOT AVAILABLE")
    .value_counts()
    .to_string()
)

# ------------------------------------------------------------
# 6. Prepare primary tumor RNA-seq samples
# ------------------------------------------------------------

primary_inventory = inventory[
    inventory["sample_type"] == "Primary Tumor"
].copy()

primary_inventory = primary_inventory[
    [
        "patient_id",
        "sample_id",
        "project",
        "file_id"
    ]
].drop_duplicates()

gdc_samples = (
    primary_inventory
    .groupby("patient_id")["sample_id"]
    .apply(
        lambda values: set(values.dropna())
    )
    .to_dict()
)

print("\nPrimary tumor sample inventory prepared.")

# ------------------------------------------------------------
# 7. TCGA sample barcode normalization
# ------------------------------------------------------------

def tcga_sample_key(sample_id):
    """
    Extract TCGA barcode through sample-type code.

    Example:
    TCGA-XX-XXXX-01A -> TCGA-XX-XXXX-01

    This identifies the same patient and sample-type
    category, but does NOT prove identical aliquots.
    """

    if pd.isna(sample_id):
        return None

    value = str(sample_id).strip().upper()

    match = re.match(
        r"^(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4}-[0-9]{2})",
        value
    )

    return match.group(1) if match else None


def check_gdc_sample(patient_id, sample_id):
    """
    Compare a cBioPortal sample identifier with
    primary tumor RNA-seq samples from GDC.
    """

    if pd.isna(sample_id):
        return "Not available"

    reference_samples = gdc_samples.get(
        patient_id,
        set()
    )

    if not reference_samples:
        return "No GDC primary tumor sample"

    sample_id = str(sample_id).strip().upper()

    normalized_references = {
        str(s).strip().upper()
        for s in reference_samples
    }

    if sample_id in normalized_references:
        return "Exact sample ID match"

    query_key = tcga_sample_key(sample_id)

    reference_keys = {
        tcga_sample_key(s)
        for s in reference_samples
    }

    if (
        query_key is not None
        and query_key in reference_keys
    ):
        return "Sample-type barcode match"

    return "No matching GDC sample"

# ------------------------------------------------------------
# 8. Validate PanCancer Atlas sample IDs
# ------------------------------------------------------------

cohort["pancancer_gdc_match"] = cohort.apply(
    lambda row: check_gdc_sample(
        row["patientId"],
        row["mantis_sample_id"]
    ),
    axis=1
)

# ------------------------------------------------------------
# 9. Validate published MSI sample IDs
# ------------------------------------------------------------

cohort["published_gdc_match"] = cohort.apply(
    lambda row: check_gdc_sample(
        row["patientId"],
        row["published_sample_id"]
    ),
    axis=1
)

# ------------------------------------------------------------
# 10. Compare cBioPortal sample identifiers
# ------------------------------------------------------------

cohort["cbio_sample_match"] = (
    cohort["mantis_sample_id"].notna()
    & cohort["published_sample_id"].notna()
    & (
        cohort["mantis_sample_id"]
        .astype("string")
        .str.strip()
        .str.upper()
        ==
        cohort["published_sample_id"]
        .astype("string")
        .str.strip()
        .str.upper()
    )
).fillna(False).astype(bool)

# ------------------------------------------------------------
# 11. Report sample correspondence
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAMPLE CORRESPONDENCE AUDIT")
print("-" * 65)

print("\nPanCancer Atlas vs GDC:")

print(
    cohort["pancancer_gdc_match"]
    .value_counts()
    .to_string()
)

print("\nPublished study vs GDC:")

print(
    cohort["published_gdc_match"]
    .value_counts()
    .to_string()
)

print("\nExact matches between cBioPortal studies:")

print(
    cohort["cbio_sample_match"]
    .value_counts()
    .to_string()
)

# ------------------------------------------------------------
# 12. Construct preliminary evidence tiers
#
# FIX:
# Explicitly convert all classification conditions
# to standard Boolean arrays without missing values.
# ------------------------------------------------------------

low_msi = (
    cohort["msi_group"]
    .eq("Concordant low MSI")
    .fillna(False)
    .astype(bool)
)

published_mss = (
    cohort["published_msi_status"]
    .eq("MSS")
    .fillna(False)
    .astype(bool)
)

published_msil = (
    cohort["published_msi_status"]
    .eq("MSI-L")
    .fillna(False)
    .astype(bool)
)

published_missing = (
    cohort["published_msi_status"]
    .isna()
    .astype(bool)
)

valid_match_types = {
    "Exact sample ID match",
    "Sample-type barcode match"
}

pancancer_linked = (
    cohort["pancancer_gdc_match"]
    .isin(valid_match_types)
    .fillna(False)
    .astype(bool)
)

published_linked = (
    cohort["published_gdc_match"]
    .isin(valid_match_types)
    .fillna(False)
    .astype(bool)
)

cbio_linked = (
    cohort["cbio_sample_match"]
    .fillna(False)
    .astype(bool)
)

# ------------------------------------------------------------
# 13. Define Tier A
# ------------------------------------------------------------

# Tier A:
# Published MSS + concordant computational low MSI
# + cross-study sample correspondence.

tier_a = (
    low_msi
    & published_mss
    & pancancer_linked
    & published_linked
    & cbio_linked
)

# ------------------------------------------------------------
# 14. Define Tier B
# ------------------------------------------------------------

# Tier B:
# Concordant computational low MSI
# without a published MSI classification.

tier_b = (
    low_msi
    & published_missing
    & pancancer_linked
)

# ------------------------------------------------------------
# 15. Assign explicit evidence categories
#
# FIX:
# Convert every condition to a NumPy Boolean array.
# This prevents the previous np.select TypeError.
# ------------------------------------------------------------

conditions = [
    tier_a,
    tier_b,
    low_msi & published_msil,
    low_msi & published_mss & ~tier_a,
    low_msi & published_missing & ~tier_b
]

conditions = [
    condition.fillna(False).to_numpy(dtype=bool)
    for condition in conditions
]

choices = [
    "Tier A - Published MSS and concordant low MSI",
    "Tier B - Concordant low MSI only",
    "Excluded - Published MSI-L",
    "Review - MSS sample linkage",
    "Review - Low-MSI sample linkage"
]

cohort["evidence_tier"] = np.select(
    conditions,
    choices,
    default="Other - Not eligible for primary MSS cohort"
)

# Validate that every patient has a category.

if cohort["evidence_tier"].isna().any():
    raise RuntimeError(
        "Some patients have missing evidence tiers."
    )

print("\nEvidence tier assignment: PASSED")

# ------------------------------------------------------------
# 16. Display evidence tier results
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("MSS EVIDENCE TIER DISTRIBUTION")
print("-" * 65)

print("\nOverall:")

print(
    cohort["evidence_tier"]
    .value_counts()
    .to_string()
)

print("\nBy TCGA project:")

tier_summary = pd.crosstab(
    cohort["gdc_project"],
    cohort["evidence_tier"]
)

print(tier_summary.to_string())

# ------------------------------------------------------------
# 17. Extract eligible cohorts
# ------------------------------------------------------------

strict_mss = cohort[
    tier_a
].copy()

extended_low_msi = cohort[
    tier_b
].copy()

print("\n" + "-" * 65)
print("ELIGIBLE RESEARCH COHORTS")
print("-" * 65)

print("\nTier A - Published MSS + concordant low MSI")

print("Total:", len(strict_mss))

print(
    strict_mss["gdc_project"]
    .value_counts()
    .to_string()
)

print("\nTier B - Computationally low MSI only")

print("Total:", len(extended_low_msi))

print(
    extended_low_msi["gdc_project"]
    .value_counts()
    .to_string()
)

# ------------------------------------------------------------
# 18. Identify discrepancies for review
# ------------------------------------------------------------

review = cohort[
    cohort["evidence_tier"]
    .str.startswith("Review")
].copy()

print("\n" + "-" * 65)
print("SAMPLE LINKAGE REVIEW")
print("-" * 65)

print("\nPatients requiring sample linkage review:")

print(len(review))

if not review.empty:

    print(
        review[
            [
                "patientId",
                "gdc_project",
                "published_msi_status",
                "pancancer_gdc_match",
                "published_gdc_match",
                "cbio_sample_match"
            ]
        ]
        .head(15)
        .to_string(index=False)
    )

# ------------------------------------------------------------
# 19. Additional cohort validation
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("COHORT VALIDATION CHECKS")
print("-" * 65)

# Check Tier A and Tier B are mutually exclusive.

tier_a_patients = set(
    strict_mss["patientId"]
)

tier_b_patients = set(
    extended_low_msi["patientId"]
)

overlap = tier_a_patients.intersection(
    tier_b_patients
)

if overlap:
    raise RuntimeError(
        "Tier A and Tier B contain overlapping patients."
    )

print("\nTier A / Tier B overlap: 0 patients")

# Check total cohort preservation.

if len(cohort) != len(comparison):
    raise RuntimeError(
        "Patient count changed unexpectedly."
    )

print(
    "Patient count preservation: PASSED"
)

# Check cohort category counts.

if (
    cohort["evidence_tier"]
    .value_counts()
    .sum()
    != len(cohort)
):
    raise RuntimeError(
        "Evidence tier counts do not sum correctly."
    )

print(
    "Evidence tier counts: PASSED"
)

print("\nAll cohort validation checks passed.")

# ------------------------------------------------------------
# 20. Save patient-level outputs
# ------------------------------------------------------------

cohort_path = (
    METADATA_DIR /
    "tcga_mss_evidence_tiers.csv"
)

tier_a_path = (
    METADATA_DIR /
    "tcga_tier_a_published_mss.csv"
)

tier_b_path = (
    METADATA_DIR /
    "tcga_tier_b_computational_low_msi.csv"
)

review_path = (
    METADATA_DIR /
    "tcga_mss_sample_linkage_review.csv"
)

summary_path = (
    RESULTS_DIR /
    "tcga_mss_evidence_tier_summary.csv"
)

cohort.to_csv(
    cohort_path,
    index=False
)

strict_mss.to_csv(
    tier_a_path,
    index=False
)

extended_low_msi.to_csv(
    tier_b_path,
    index=False
)

review.to_csv(
    review_path,
    index=False
)

tier_summary.to_csv(
    summary_path
)

# ------------------------------------------------------------
# 21. Save research methodology
# ------------------------------------------------------------

methodology = {

    "project": "SurfaceTarget-CRC",

    "patch": "1.9",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "sources": [
        "TCGA GDC RNA-seq",
        "cBioPortal PanCancer Atlas 2018",
        "cBioPortal TCGA CRC Published 2012"
    ],

    "tier_a_definition": (
        "Published MSS status with concordant "
        "low MANTIS/MSIsensor scores and "
        "matching sample identifiers."
    ),

    "tier_b_definition": (
        "Concordant low MSI scores without "
        "available published categorical status."
    ),

    "tier_a_patients": int(
        len(strict_mss)
    ),

    "tier_b_patients": int(
        len(extended_low_msi)
    ),

    "sample_barcode_note": (
        "Matching the first 15 characters of "
        "a TCGA barcode establishes patient "
        "and sample-type correspondence, "
        "not identical biospecimen aliquots."
    ),

    "final_expression_samples_selected": False,

    "limitations": [
        "Tier B is not independently verified MSS.",
        "Published MSI-L is excluded from Tier A.",
        "Cross-study samples may represent different aliquots.",
        "Published and computational MSI evidence may not be fully independent.",
        "Selection bias between published and extended cohorts remains possible.",
        "Sample-type barcode matching does not establish identical tissue aliquots."
    ]
}

methodology_path = (
    METADATA_DIR /
    "tcga_mss_evidence_tier_methodology.json"
)

with open(methodology_path, "w") as f:

    json.dump(
        methodology,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 22. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.9 SUMMARY")
print("=" * 65)

print("\nFiles saved:")

print("1. tcga_mss_evidence_tiers.csv")
print("2. tcga_tier_a_published_mss.csv")
print("3. tcga_tier_b_computational_low_msi.csv")
print("4. tcga_mss_sample_linkage_review.csv")
print("5. tcga_mss_evidence_tier_summary.csv")
print("6. tcga_mss_evidence_tier_methodology.json")

print("\nFinal Tier A patients:", len(strict_mss))

print("Final Tier B patients:", len(extended_low_msi))

print("\nNo expression matrices downloaded.")

print("Final RNA-seq sample selection still pending.")

print("\nPATCH 1.9 COMPLETE")
print("=" * 65)

In [ ]:
# ============================================================
# SurfaceTarget-CRC
# Phase 1 — Patch 1.10
#
# RNA-seq Sample and File Selection Audit
#
# Objectives:
# 1. Match Tier A and Tier B patients to GDC RNA-seq files
# 2. Identify patients with multiple primary tumor samples
# 3. Identify samples with multiple RNA-seq files
# 4. Inventory available normal tissue samples
# 5. Estimate storage requirements
# 6. Save reproducible candidate file inventories
#
# IMPORTANT:
# No final RNA-seq file selection is performed.
# No molecular expression data is downloaded.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import json

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/SurfaceTarget-CRC")

METADATA_DIR = PROJECT_ROOT / "data" / "metadata"

RESULTS_DIR = PROJECT_ROOT / "results" / "tables"

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

INVENTORY_PATH = (
    METADATA_DIR /
    "tcga_rnaseq_file_inventory.csv"
)

TIER_A_PATH = (
    METADATA_DIR /
    "tcga_tier_a_published_mss.csv"
)

TIER_B_PATH = (
    METADATA_DIR /
    "tcga_tier_b_computational_low_msi.csv"
)

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.10 — RNA-SEQ SAMPLE SELECTION AUDIT")
print("=" * 65)

# ------------------------------------------------------------
# 2. Load research metadata
# ------------------------------------------------------------

required_files = [
    INVENTORY_PATH,
    TIER_A_PATH,
    TIER_B_PATH
]

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(
            f"Required file missing: {path}"
        )

inventory = pd.read_csv(
    INVENTORY_PATH,
    dtype=str,
    keep_default_na=False
)

tier_a = pd.read_csv(
    TIER_A_PATH,
    dtype=str,
    keep_default_na=False
)

tier_b = pd.read_csv(
    TIER_B_PATH,
    dtype=str,
    keep_default_na=False
)

print("\nMetadata loaded successfully.")

print(
    "Total RNA-seq files:",
    inventory["file_id"].nunique()
)

print("Tier A patients:", len(tier_a))

print("Tier B patients:", len(tier_b))

# ------------------------------------------------------------
# 3. Validate cohort identifiers
# ------------------------------------------------------------

if tier_a["patientId"].duplicated().any():
    raise RuntimeError(
        "Duplicate patients detected in Tier A."
    )

if tier_b["patientId"].duplicated().any():
    raise RuntimeError(
        "Duplicate patients detected in Tier B."
    )

tier_a_ids = set(tier_a["patientId"])

tier_b_ids = set(tier_b["patientId"])

if tier_a_ids & tier_b_ids:
    raise RuntimeError(
        "Tier A and Tier B contain overlapping patients."
    )

if inventory["file_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate file IDs in inventory. "
        "Review metadata before proceeding."
    )

print("\nCohort validation: PASSED")

# ------------------------------------------------------------
# 4. Construct patient-level cohort lookup
# ------------------------------------------------------------

tier_a_lookup = tier_a[
    ["patientId", "gdc_project"]
].copy()

tier_a_lookup["cohort_tier"] = "Tier A"

tier_b_lookup = tier_b[
    ["patientId", "gdc_project"]
].copy()

tier_b_lookup["cohort_tier"] = "Tier B"

cohort_lookup = pd.concat(
    [
        tier_a_lookup,
        tier_b_lookup
    ],
    ignore_index=True
)

cohort_lookup = cohort_lookup.rename(
    columns={
        "patientId": "patient_id",
        "gdc_project": "expected_project"
    }
)

# ------------------------------------------------------------
# 5. Prepare primary tumor RNA-seq inventory
# ------------------------------------------------------------

primary = inventory[
    inventory["sample_type"] == "Primary Tumor"
].copy()

primary = primary.merge(
    cohort_lookup,
    on="patient_id",
    how="inner",
    validate="many_to_one"
)

# Verify GDC project correspondence.

project_mismatch = (
    primary["project"]
    != primary["expected_project"]
)

if project_mismatch.any():
    raise RuntimeError(
        "GDC project mismatch detected."
    )

# Every eligible patient must have a primary tumor file.

matched_patients = set(
    primary["patient_id"]
)

expected_patients = (
    tier_a_ids | tier_b_ids
)

missing_patients = (
    expected_patients - matched_patients
)

if missing_patients:
    raise RuntimeError(
        f"{len(missing_patients)} cohort patients "
        "have no matching primary tumor RNA-seq file."
    )

print("\nPrimary tumor matching: PASSED")

# ------------------------------------------------------------
# 6. Convert file sizes
# ------------------------------------------------------------

primary["file_size_bytes"] = pd.to_numeric(
    primary["file_size_bytes"],
    errors="coerce"
)

if primary["file_size_bytes"].isna().any():
    raise RuntimeError(
        "Missing file sizes detected."
    )

primary["file_size_mb"] = (
    primary["file_size_bytes"]
    / (1024 ** 2)
)

# ------------------------------------------------------------
# 7. Cohort-specific RNA-seq inventory
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("COHORT RNA-SEQ FILE AVAILABILITY")
print("-" * 65)

cohort_summary = (
    primary.groupby(
        ["project", "cohort_tier"]
    )
    .agg(
        patients=("patient_id", "nunique"),
        samples=("sample_id", "nunique"),
        files=("file_id", "nunique"),
        total_size_mb=("file_size_mb", "sum")
    )
    .reset_index()
)

cohort_summary["total_size_mb"] = (
    cohort_summary["total_size_mb"].round(2)
)

print("\n")
print(
    cohort_summary.to_string(index=False)
)

# ------------------------------------------------------------
# 8. Identify patients with multiple tumor samples
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("MULTIPLE PRIMARY TUMOR SAMPLE AUDIT")
print("-" * 65)

patient_sample_counts = (
    primary.groupby(
        ["project", "cohort_tier", "patient_id"]
    )["sample_id"]
    .nunique()
    .reset_index(name="n_samples")
)

multi_sample_patients = (
    patient_sample_counts[
        patient_sample_counts["n_samples"] > 1
    ]
    .copy()
)

print(
    "\nPatients with multiple primary tumor samples:",
    len(multi_sample_patients)
)

if not multi_sample_patients.empty:

    print("\nPatients requiring sample selection review:")

    print(
        multi_sample_patients
        .sort_values(
            ["project", "cohort_tier", "patient_id"]
        )
        .to_string(index=False)
    )

else:

    print(
        "No patients with multiple primary tumor samples."
    )

# ------------------------------------------------------------
# 9. Identify samples with multiple RNA-seq files
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("MULTIPLE RNA-SEQ FILE AUDIT")
print("-" * 65)

sample_file_counts = (
    primary.groupby(
        [
            "project",
            "cohort_tier",
            "patient_id",
            "sample_id"
        ]
    )["file_id"]
    .nunique()
    .reset_index(name="n_files")
)

multi_file_samples = (
    sample_file_counts[
        sample_file_counts["n_files"] > 1
    ]
    .copy()
)

print(
    "\nSamples with multiple RNA-seq files:",
    len(multi_file_samples)
)

if not multi_file_samples.empty:

    print("\nSamples requiring file selection review:")

    print(
        multi_file_samples
        .sort_values(
            ["project", "cohort_tier", "patient_id"]
        )
        .to_string(index=False)
    )

# ------------------------------------------------------------
# 10. Generate detailed duplicate file audit
# ------------------------------------------------------------

duplicate_file_details = primary.merge(
    multi_file_samples[
        ["sample_id"]
    ],
    on="sample_id",
    how="inner"
)

duplicate_file_details = (
    duplicate_file_details
    .sort_values(
        ["project", "patient_id", "sample_id", "file_id"]
    )
)

if not duplicate_file_details.empty:

    print("\nDetailed duplicate file information:")

    display_columns = [
        "project",
        "cohort_tier",
        "patient_id",
        "sample_id",
        "file_id",
        "file_name",
        "file_size_mb"
    ]

    print(
        duplicate_file_details[
            display_columns
        ].to_string(index=False)
    )

# ------------------------------------------------------------
# 11. Inventory normal tissue samples
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("NORMAL TISSUE SAMPLE AVAILABILITY")
print("-" * 65)

normal = inventory[
    inventory["sample_type"] == "Solid Tissue Normal"
].copy()

normal["file_size_bytes"] = pd.to_numeric(
    normal["file_size_bytes"],
    errors="coerce"
)

if normal["file_size_bytes"].isna().any():
    raise RuntimeError(
        "Missing normal tissue file sizes."
    )

normal["file_size_mb"] = (
    normal["file_size_bytes"]
    / (1024 ** 2)
)

normal_summary = (
    normal.groupby("project")
    .agg(
        patients=("patient_id", "nunique"),
        samples=("sample_id", "nunique"),
        files=("file_id", "nunique"),
        total_size_mb=("file_size_mb", "sum")
    )
    .reset_index()
)

normal_summary["total_size_mb"] = (
    normal_summary["total_size_mb"].round(2)
)

print("\n")
print(
    normal_summary.to_string(index=False)
)

# ------------------------------------------------------------
# 12. Investigate paired normal availability
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("TUMOR / NORMAL PATIENT OVERLAP")
print("-" * 65)

normal_patients = set(
    normal["patient_id"]
)

for project in ["TCGA-COAD", "TCGA-READ"]:

    for tier_name, tier_ids in [
        ("Tier A", tier_a_ids),
        ("Tier B", tier_b_ids)
    ]:

        project_patients = set(
            primary.loc[
                (primary["project"] == project)
                & (primary["cohort_tier"] == tier_name),
                "patient_id"
            ]
        )

        paired_patients = (
            project_patients & normal_patients
        )

        print(
            f"{project} | {tier_name} | "
            f"Patients with available normal tissue: "
            f"{len(paired_patients)}"
        )

print("\nNOTE:")
print(
    "Patient overlap does not guarantee identical "
    "sample processing or a complete paired analysis."
)

# ------------------------------------------------------------
# 13. Storage requirement estimate
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("ESTIMATED RNA-SEQ STORAGE REQUIREMENTS")
print("-" * 65)

tumor_bytes = primary["file_size_bytes"].sum()

normal_bytes = normal["file_size_bytes"].sum()

combined_bytes = tumor_bytes + normal_bytes

tumor_gb = tumor_bytes / (1024 ** 3)

normal_gb = normal_bytes / (1024 ** 3)

combined_gb = combined_bytes / (1024 ** 3)

print(f"\nCandidate tumor files: {tumor_gb:.3f} GB")

print(f"Normal tissue files:   {normal_gb:.3f} GB")

print(f"Combined total:        {combined_gb:.3f} GB")

print("\nIMPORTANT:")
print(
    "This estimate includes all candidate files, "
    "including those requiring duplicate review."
)

print(
    "It excludes temporary processing overhead "
    "and additional downstream datasets."
)

# ------------------------------------------------------------
# 14. Save candidate file inventories
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING RESEARCH INVENTORIES")
print("-" * 65)

tumor_path = (
    METADATA_DIR /
    "tcga_candidate_tumor_rnaseq_files.csv"
)

normal_path = (
    METADATA_DIR /
    "tcga_candidate_normal_rnaseq_files.csv"
)

multi_sample_path = (
    METADATA_DIR /
    "tcga_multiple_tumor_samples_review.csv"
)

multi_file_path = (
    METADATA_DIR /
    "tcga_multiple_rnaseq_files_review.csv"
)

duplicate_details_path = (
    METADATA_DIR /
    "tcga_duplicate_file_details.csv"
)

summary_path = (
    RESULTS_DIR /
    "tcga_rnaseq_candidate_file_summary.csv"
)

primary.to_csv(
    tumor_path,
    index=False
)

normal.to_csv(
    normal_path,
    index=False
)

multi_sample_patients.to_csv(
    multi_sample_path,
    index=False
)

multi_file_samples.to_csv(
    multi_file_path,
    index=False
)

duplicate_file_details.to_csv(
    duplicate_details_path,
    index=False
)

cohort_summary.to_csv(
    summary_path,
    index=False
)

# ------------------------------------------------------------
# 15. Save audit provenance
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "1.10",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": "NCI Genomic Data Commons",

    "workflow": "STAR - Counts",

    "tier_a_patients": len(tier_a_ids),

    "tier_b_patients": len(tier_b_ids),

    "candidate_tumor_files": int(
        primary["file_id"].nunique()
    ),

    "candidate_normal_files": int(
        normal["file_id"].nunique()
    ),

    "patients_with_multiple_tumor_samples": int(
        len(multi_sample_patients)
    ),

    "samples_with_multiple_rnaseq_files": int(
        len(multi_file_samples)
    ),

    "candidate_storage_gb": round(
        float(combined_gb), 4
    ),

    "final_file_selection_performed": False,

    "molecular_data_downloaded": False,

    "limitations": [
        "Multiple tumor samples require review.",
        "Multiple RNA-seq files require review.",
        "Normal samples are not automatically matched controls.",
        "Tier B is computationally low-MSI, not confirmed MSS.",
        "Candidate file inventory is not a finalized download manifest."
    ]
}

provenance_path = (
    METADATA_DIR /
    "tcga_rnaseq_sample_audit_provenance.json"
)

with open(provenance_path, "w") as f:

    json.dump(
        provenance,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 16. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.10 SUMMARY")
print("=" * 65)

print("\nCandidate tumor patients:", len(expected_patients))

print(
    "Candidate tumor RNA-seq files:",
    primary["file_id"].nunique()
)

print(
    "Candidate normal RNA-seq files:",
    normal["file_id"].nunique()
)

print(
    "\nPatients requiring multiple-sample review:",
    len(multi_sample_patients)
)

print(
    "Samples requiring multiple-file review:",
    len(multi_file_samples)
)

print("\nFiles saved:")

print("1. tcga_candidate_tumor_rnaseq_files.csv")
print("2. tcga_candidate_normal_rnaseq_files.csv")
print("3. tcga_multiple_tumor_samples_review.csv")
print("4. tcga_multiple_rnaseq_files_review.csv")
print("5. tcga_duplicate_file_details.csv")
print("6. tcga_rnaseq_candidate_file_summary.csv")
print("7. tcga_rnaseq_sample_audit_provenance.json")

print("\nNo expression matrices downloaded.")

print("No duplicate files automatically selected.")

print("\nPATCH 1.10 COMPLETE")

print("=" * 65)

In [ ]:
# ============================================================
# SurfaceTarget-CRC
# Phase 1 — Patch 1.11
#
# RNA-seq Duplicate File and Biospecimen Investigation
#
# Objectives:
# 1. Identify patients with multiple tumor samples/files
# 2. Retrieve detailed file metadata from GDC
# 3. Investigate sample and aliquot identifiers
# 4. Examine workflow versions and file timestamps
# 5. Compare sample identifiers with MSI annotations
# 6. Save findings for reproducible sample selection
#
# IMPORTANT:
# No RNA-seq files are selected or downloaded.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import requests
import json

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/SurfaceTarget-CRC")

METADATA_DIR = PROJECT_ROOT / "data" / "metadata"

METADATA_DIR.mkdir(parents=True, exist_ok=True)

GDC_API = "https://api.gdc.cancer.gov"

CANDIDATE_PATH = (
    METADATA_DIR /
    "tcga_candidate_tumor_rnaseq_files.csv"
)

COHORT_PATH = (
    METADATA_DIR /
    "tcga_mss_evidence_tiers.csv"
)

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.11 — RNA-SEQ DUPLICATE INVESTIGATION")
print("=" * 65)

# ------------------------------------------------------------
# 2. Load candidate files and cohort information
# ------------------------------------------------------------

candidates = pd.read_csv(
    CANDIDATE_PATH,
    dtype=str,
    keep_default_na=False
)

cohort = pd.read_csv(
    COHORT_PATH,
    dtype=str
)

print("\nCandidate RNA-seq files:", len(candidates))

print(
    "Candidate patients:",
    candidates["patient_id"].nunique()
)

# ------------------------------------------------------------
# 3. Identify patients requiring review
# ------------------------------------------------------------

patient_summary = (
    candidates.groupby("patient_id")
    .agg(
        n_samples=("sample_id", "nunique"),
        n_files=("file_id", "nunique")
    )
    .reset_index()
)

review_patients = patient_summary[
    (patient_summary["n_samples"] > 1)
    | (patient_summary["n_files"] > 1)
].copy()

print("\n" + "-" * 65)
print("PATIENTS REQUIRING REVIEW")
print("-" * 65)

print(
    review_patients.to_string(index=False)
)

review_ids = review_patients["patient_id"].tolist()

if not review_ids:
    print("\nNo duplicate cases detected.")
else:
    print(
        "\nPatients requiring investigation:",
        len(review_ids)
    )

# ------------------------------------------------------------
# 4. Extract all candidate files for these patients
# ------------------------------------------------------------

review_files = candidates[
    candidates["patient_id"].isin(review_ids)
].copy()

file_ids = sorted(
    review_files["file_id"].unique().tolist()
)

print("\nFiles requiring metadata investigation:", len(file_ids))

print("\nOriginal inventory:")

print(
    review_files[
        [
            "patient_id",
            "sample_id",
            "file_id",
            "file_name"
        ]
    ].sort_values(
        ["patient_id", "sample_id", "file_id"]
    ).to_string(index=False)
)

# ------------------------------------------------------------
# 5. Retrieve detailed GDC file metadata
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("RETRIEVING GDC FILE PROVENANCE")
print("-" * 65)

session = requests.Session()

session.headers.update({
    "Accept": "application/json",
    "User-Agent": "SurfaceTarget-CRC-Research/1.0"
})

fields = [
    "file_id",
    "file_name",
    "file_size",
    "md5sum",
    "created_datetime",
    "updated_datetime",
    "analysis.workflow_type",
    "analysis.workflow_version",
    "analysis.input_files.file_id",
    "cases.submitter_id",
    "cases.samples.submitter_id",
    "cases.samples.sample_type",
    "cases.samples.portions.analytes.aliquots.submitter_id"
]

filters = {
    "op": "in",
    "content": {
        "field": "file_id",
        "value": file_ids
    }
}

params = {
    "filters": json.dumps(filters),
    "fields": ",".join(fields),
    "format": "JSON",
    "size": 100
}

if not file_ids:
    raise RuntimeError(
        "No ambiguous files found. "
        "No duplicate investigation is required."
    )

response = session.get(
    f"{GDC_API}/files",
    params=params,
    timeout=90
)

response.raise_for_status()

payload = response.json()

hits = payload["data"]["hits"]

print("\nFiles requested:", len(file_ids))
print("Files retrieved:", len(hits))

returned_ids = {
    item["file_id"]
    for item in hits
}

missing_ids = set(file_ids) - returned_ids

if missing_ids:
    raise RuntimeError(
        f"GDC did not return these file IDs: {missing_ids}"
    )

print("\nFile retrieval validation: PASSED")

# ------------------------------------------------------------
# 6. Save original GDC metadata response
# ------------------------------------------------------------

raw_metadata_path = (
    METADATA_DIR /
    "tcga_duplicate_files_gdc_raw_metadata.json"
)

raw_metadata = {
    "retrieved_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "source": "NCI Genomic Data Commons",
    "endpoint": f"{GDC_API}/files",
    "requested_file_ids": file_ids,
    "fields": fields,
    "records": hits
}

with open(raw_metadata_path, "w") as f:
    json.dump(raw_metadata, f, indent=4)

# ------------------------------------------------------------
# 7. Extract biospecimen and aliquot information
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("EXTRACTING BIOSPECIMEN INFORMATION")
print("-" * 65)

records = []

for item in hits:

    file_id = item.get("file_id")

    analysis = item.get("analysis") or {}

    input_files = analysis.get("input_files") or []

    input_file_ids = sorted({
        f.get("file_id")
        for f in input_files
        if f.get("file_id")
    })

    cases = item.get("cases") or []

    for case in cases:

        patient_id = case.get("submitter_id")

        samples = case.get("samples") or []

        sample_ids = set()
        sample_types = set()
        aliquot_ids = set()

        for sample in samples:

            sample_id = sample.get("submitter_id")

            if sample_id:
                sample_ids.add(sample_id)

            sample_type = sample.get("sample_type")

            if sample_type:
                sample_types.add(sample_type)

            portions = sample.get("portions") or []

            for portion in portions:

                analytes = portion.get("analytes") or []

                for analyte in analytes:

                    aliquots = analyte.get("aliquots") or []

                    for aliquot in aliquots:

                        aliquot_id = aliquot.get("submitter_id")

                        if aliquot_id:
                            aliquot_ids.add(aliquot_id)

        records.append({
            "patient_id": patient_id,
            "file_id": file_id,
            "file_name": item.get("file_name"),
            "file_size_bytes": item.get("file_size"),
            "md5sum": item.get("md5sum"),
            "created_datetime": item.get("created_datetime"),
            "updated_datetime": item.get("updated_datetime"),
            "workflow_type": analysis.get("workflow_type"),
            "workflow_version": analysis.get("workflow_version"),
            "input_file_ids": "; ".join(input_file_ids),
            "n_input_files": len(input_file_ids),
            "sample_ids": "; ".join(sorted(sample_ids)),
            "sample_types": "; ".join(sorted(sample_types)),
            "n_samples": len(sample_ids),
            "aliquot_ids": "; ".join(sorted(aliquot_ids)),
            "n_aliquots": len(aliquot_ids)
        })

details = pd.DataFrame(records)

if details.empty:
    raise RuntimeError(
        "No biospecimen metadata could be extracted."
    )

# ------------------------------------------------------------
# 8. Display detailed results
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("DETAILED FILE AND BIOSPECIMEN AUDIT")
print("-" * 65)

for patient_id in review_ids:

    print("\n" + "=" * 60)
    print("PATIENT:", patient_id)
    print("=" * 60)

    subset = details[
        details["patient_id"] == patient_id
    ].sort_values("file_id")

    for _, row in subset.iterrows():

        print("\nFile ID:", row["file_id"])

        print("File:", row["file_name"])

        print("Samples:", row["sample_ids"])

        print("Sample Types:", row["sample_types"])

        print("Aliquots:", row["aliquot_ids"])

        print("Workflow:", row["workflow_type"])

        print("Workflow Version:", row["workflow_version"])

        print("Input File IDs:", row["input_file_ids"])

        print("Created:", row["created_datetime"])

        print("Updated:", row["updated_datetime"])

        print("MD5:", row["md5sum"])

        print("-" * 50)

# ------------------------------------------------------------
# 9. Compare with existing RNA-seq inventory
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("ORIGINAL INVENTORY COMPARISON")
print("-" * 65)

for file_id in file_ids:

    original_samples = set(
        review_files.loc[
            review_files["file_id"] == file_id,
            "sample_id"
        ]
    )

    current_rows = details[
        details["file_id"] == file_id
    ]

    current_samples = set()

    for value in current_rows["sample_ids"]:

        for sample_id in str(value).split(";"):

            sample_id = sample_id.strip()

            if sample_id:
                current_samples.add(sample_id)

    print("\nFile:", file_id)

    print(
        "Original inventory samples:",
        sorted(original_samples)
    )

    print(
        "Current GDC samples:",
        sorted(current_samples)
    )

    if original_samples == current_samples:

        print("Metadata correspondence: MATCH")

    else:

        print("Metadata correspondence: REVIEW REQUIRED")

# ------------------------------------------------------------
# 10. Compare with MSI sample identifiers
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("MSI SAMPLE IDENTIFIER COMPARISON")
print("-" * 65)

msi_columns = [
    "patientId",
    "gdc_project",
    "msi_group",
    "mantis_sample_id",
    "msisensor_sample_id",
    "published_sample_id",
    "evidence_tier"
]

msi_review = cohort[
    cohort["patientId"].isin(review_ids)
][msi_columns]

print("\n")
print(
    msi_review.to_string(index=False)
)

# ------------------------------------------------------------
# 11. Save detailed audit
# ------------------------------------------------------------

details_path = (
    METADATA_DIR /
    "tcga_duplicate_rnaseq_provenance_audit.csv"
)

details.to_csv(
    details_path,
    index=False
)

msi_review_path = (
    METADATA_DIR /
    "tcga_duplicate_patients_msi_sample_review.csv"
)

msi_review.to_csv(
    msi_review_path,
    index=False
)

# ------------------------------------------------------------
# 12. Save methodology notes
# ------------------------------------------------------------

methodology = {
    "project": "SurfaceTarget-CRC",
    "patch": "1.11",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "patients_reviewed": review_ids,
    "files_reviewed": len(file_ids),
    "source": "NCI GDC API",
    "file_selection_completed": False,
    "notes": [
        "Multiple files were investigated using GDC metadata.",
        "File size alone is not a valid selection criterion.",
        "Timestamps alone do not establish biological superiority.",
        "Sample and aliquot relationships require interpretation.",
        "No duplicate RNA-seq files have been selected or removed."
    ]
}

methodology_path = (
    METADATA_DIR /
    "tcga_duplicate_file_investigation_methodology.json"
)

with open(methodology_path, "w") as f:
    json.dump(methodology, f, indent=4)

# ------------------------------------------------------------
# 13. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.11 SUMMARY")
print("=" * 65)

print("\nPatients investigated:", len(review_ids))

print("Files investigated:", len(file_ids))

print("\nFiles saved:")

print("1. tcga_duplicate_files_gdc_raw_metadata.json")
print("2. tcga_duplicate_rnaseq_provenance_audit.csv")
print("3. tcga_duplicate_patients_msi_sample_review.csv")
print("4. tcga_duplicate_file_investigation_methodology.json")

print("\nNo expression data downloaded.")
print("No duplicate files selected.")

print("\nPATCH 1.11 COMPLETE")
print("=" * 65)

In [ ]:
# ============================================================
# SurfaceTarget-CRC
# Phase 1 — Patch 1.12
#
# DUPLICATE RNA-SEQ QUALITY ASSESSMENT
#
# Objectives:
# 1. Download six duplicate-candidate RNA-seq files
# 2. Verify downloaded file integrity using MD5
# 3. Examine STAR gene-count statistics
# 4. Calculate preliminary RNA-seq quality indicators
# 5. Compare expression profiles within each patient
# 6. Save QC results and provenance
#
# IMPORTANT:
# No final file selection is performed.
# No files are excluded automatically.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import requests
import hashlib
import json
import time
import shutil

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/SurfaceTarget-CRC")

METADATA_DIR = PROJECT_ROOT / "data" / "metadata"

RAW_DIR = (
    PROJECT_ROOT /
    "data/raw/tcga_duplicate_qc"
)

RESULTS_DIR = (
    PROJECT_ROOT /
    "results/tables"
)

RAW_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

AUDIT_PATH = (
    METADATA_DIR /
    "tcga_duplicate_rnaseq_provenance_audit.csv"
)

GDC_DATA_API = "https://api.gdc.cancer.gov/data"

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.12 — DUPLICATE RNA-SEQ QUALITY ASSESSMENT")
print("=" * 65)

# ------------------------------------------------------------
# 2. Load file metadata
# ------------------------------------------------------------

if not AUDIT_PATH.exists():
    raise FileNotFoundError(
        "Duplicate provenance audit not found. "
        "Please complete Patch 1.11."
    )

audit = pd.read_csv(
    AUDIT_PATH,
    dtype=str,
    keep_default_na=False
)

required_columns = [
    "patient_id",
    "file_id",
    "file_size_bytes",
    "md5sum",
    "sample_ids",
    "aliquot_ids",
    "workflow_version"
]

missing = [
    col for col in required_columns
    if col not in audit.columns
]

if missing:
    raise RuntimeError(
        f"Missing metadata columns: {missing}"
    )

if audit["file_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate file identifiers in QC audit."
    )

print("\nMetadata loaded successfully.")

print("Patients:", audit["patient_id"].nunique())

print("Files:", len(audit))

expected_size_mb = (
    pd.to_numeric(audit["file_size_bytes"]).sum()
    / (1024 ** 2)
)

print(f"Expected download: {expected_size_mb:.2f} MB")

# ------------------------------------------------------------
# 3. Check available storage
# ------------------------------------------------------------

disk = shutil.disk_usage("/content")

available_gb = disk.free / (1024 ** 3)

print(f"\nAvailable Colab storage: {available_gb:.2f} GB")

if disk.free < 3 * expected_size_mb * (1024 ** 2):
    raise RuntimeError(
        "Insufficient free storage for safe download."
    )

# ------------------------------------------------------------
# 4. File integrity verification
# ------------------------------------------------------------

def calculate_md5(file_path):

    md5 = hashlib.md5()

    with open(file_path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            md5.update(chunk)

    return md5.hexdigest()


def verify_file(file_path, expected_size, expected_md5):

    if not file_path.exists():
        return False

    actual_size = file_path.stat().st_size

    if actual_size != expected_size:
        return False

    actual_md5 = calculate_md5(file_path)

    return actual_md5.lower() == expected_md5.lower()


# ------------------------------------------------------------
# 5. Download function with retries
# ------------------------------------------------------------

session = requests.Session()

session.headers.update({
    "User-Agent": "SurfaceTarget-CRC-Research/1.0"
})


def download_gdc_file(
    file_id,
    output_path,
    expected_size,
    expected_md5,
    max_attempts=3
):

    if verify_file(
        output_path,
        expected_size,
        expected_md5
    ):

        print("Already downloaded and verified.")

        return

    url = f"{GDC_DATA_API}/{file_id}"

    temporary_path = Path(
        str(output_path) + ".part"
    )

    for attempt in range(1, max_attempts + 1):

        try:

            print(
                f"Download attempt "
                f"{attempt}/{max_attempts}"
            )

            with session.get(
                url,
                stream=True,
                timeout=(30, 180)
            ) as response:

                response.raise_for_status()

                with open(
                    temporary_path,
                    "wb"
                ) as f:

                    for chunk in response.iter_content(
                        chunk_size=1024 * 1024
                    ):

                        if chunk:
                            f.write(chunk)

            # Validate before accepting the file.

            if not verify_file(
                temporary_path,
                expected_size,
                expected_md5
            ):

                raise RuntimeError(
                    "Downloaded file failed "
                    "size or MD5 verification."
                )

            temporary_path.replace(output_path)

            print("Download integrity: VERIFIED")

            return

        except Exception as e:

            print(f"Attempt failed: {e}")

            if temporary_path.exists():
                temporary_path.unlink()

            if attempt == max_attempts:
                raise

            time.sleep(2 ** attempt)


# ------------------------------------------------------------
# 6. Download the six RNA-seq files
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("DOWNLOADING RNA-SEQ FILES")
print("-" * 65)

download_records = []

for index, row in audit.iterrows():

    file_id = row["file_id"]

    patient_id = row["patient_id"]

    expected_size = int(row["file_size_bytes"])

    expected_md5 = row["md5sum"]

    output_path = (
        RAW_DIR /
        f"{file_id}.tsv"
    )

    print("\n" + "-" * 50)

    print(
        f"File {index + 1}/{len(audit)}"
    )

    print("Patient:", patient_id)

    print("File ID:", file_id)

    download_gdc_file(
        file_id,
        output_path,
        expected_size,
        expected_md5
    )

    download_records.append({
        "patient_id": patient_id,
        "file_id": file_id,
        "local_path": str(output_path),
        "expected_md5": expected_md5,
        "file_size_bytes": expected_size,
        "download_verified": True
    })

print("\nAll downloads completed and verified.")

# ------------------------------------------------------------
# 7. RNA-seq count file parsing
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PARSING STAR GENE-COUNT FILES")
print("-" * 65)

STAR_SUMMARY_ROWS = [
    "N_unmapped",
    "N_multimapping",
    "N_noFeature",
    "N_ambiguous"
]


def load_star_counts(file_path):

    # GDC files can contain a commented gene-model
    # header. comment="#" skips such lines.

    df = pd.read_csv(
        file_path,
        sep="\t",
        comment="#",
        low_memory=False
    )

    required = {
        "gene_id",
        "unstranded"
    }

    if not required.issubset(df.columns):

        raise RuntimeError(
            f"Unexpected STAR file format: {file_path}\n"
            f"Columns: {df.columns.tolist()}"
        )

    # Separate STAR count-summary rows.

    summary = df[
        df["gene_id"].isin(STAR_SUMMARY_ROWS)
    ].copy()

    genes = df[
        ~df["gene_id"].isin(STAR_SUMMARY_ROWS)
    ].copy()

    if genes.empty:
        raise RuntimeError(
            "No gene expression rows found."
        )

    if genes["gene_id"].isna().any():
        raise RuntimeError(
            "Missing gene identifiers detected."
        )

    if genes["gene_id"].duplicated().any():
        raise RuntimeError(
            "Duplicate gene identifiers detected."
        )

    genes["unstranded"] = pd.to_numeric(
        genes["unstranded"],
        errors="raise"
    )

    if genes["unstranded"].isna().any():
        raise RuntimeError(
            "Missing gene-count values detected."
        )

    if (genes["unstranded"] < 0).any():
        raise RuntimeError(
            "Negative gene counts detected."
        )

    # Convert summary statistics to numeric.

    summary["unstranded"] = pd.to_numeric(
        summary["unstranded"],
        errors="raise"
    )

    return genes, summary


# ------------------------------------------------------------
# 8. Calculate RNA-seq QC statistics
# ------------------------------------------------------------

qc_records = []

expression_profiles = {}

for _, row in audit.iterrows():

    file_id = row["file_id"]

    patient_id = row["patient_id"]

    file_path = (
        RAW_DIR /
        f"{file_id}.tsv"
    )

    genes, summary = load_star_counts(
        file_path
    )

    # Total counts assigned to genes.

    gene_counts = genes["unstranded"]

    assigned_counts = float(
        gene_counts.sum()
    )

    detected_genes = int(
        (gene_counts > 0).sum()
    )

    total_genes = len(genes)

    # STAR summary statistics.

    star_stats = dict(
        zip(
            summary["gene_id"],
            summary["unstranded"]
        )
    )

    missing_summary = (
        set(STAR_SUMMARY_ROWS)
        - set(star_stats)
    )

    if missing_summary:
        raise RuntimeError(
            f"Missing STAR summary rows: "
            f"{missing_summary}"
        )

    unmapped = float(
        star_stats["N_unmapped"]
    )

    multimapping = float(
        star_stats["N_multimapping"]
    )

    no_feature = float(
        star_stats["N_noFeature"]
    )

    ambiguous = float(
        star_stats["N_ambiguous"]
    )

    # Approximate total counted reads.

    total_counted = (
        assigned_counts
        + unmapped
        + multimapping
        + no_feature
        + ambiguous
    )

    assignment_fraction = (
        assigned_counts / total_counted
        if total_counted > 0
        else np.nan
    )

    # Gene detection percentage.

    detection_fraction = (
        detected_genes / total_genes
        if total_genes > 0
        else np.nan
    )

    # Preserve gene-level counts for comparisons.

    profile = genes.set_index(
        "gene_id"
    )["unstranded"]

    expression_profiles[file_id] = profile

    qc_records.append({

        "patient_id": patient_id,

        "file_id": file_id,

        "sample_ids": row["sample_ids"],

        "aliquot_ids": row["aliquot_ids"],

        "workflow_version": row["workflow_version"],

        "total_genes": total_genes,

        "detected_genes": detected_genes,

        "detection_fraction": detection_fraction,

        "assigned_gene_counts": assigned_counts,

        "unmapped_counts": unmapped,

        "multimapping_counts": multimapping,

        "no_feature_counts": no_feature,

        "ambiguous_counts": ambiguous,

        "total_counted_reads": total_counted,

        "assignment_fraction": assignment_fraction
    })

qc = pd.DataFrame(qc_records)

# ------------------------------------------------------------
# 9. Display quality metrics
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("RNA-SEQ QUALITY METRICS")
print("=" * 65)

qc_display = qc.copy()

qc_display["assigned_counts_millions"] = (
    qc_display["assigned_gene_counts"] / 1e6
)

qc_display["assignment_percent"] = (
    qc_display["assignment_fraction"] * 100
)

qc_display["detected_percent"] = (
    qc_display["detection_fraction"] * 100
)

display_columns = [
    "patient_id",
    "sample_ids",
    "aliquot_ids",
    "assigned_counts_millions",
    "detected_genes",
    "assignment_percent"
]

print("\n")

print(
    qc_display[
        display_columns
    ].round(2).to_string(index=False)
)

# ------------------------------------------------------------
# 10. Compare expression profiles
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("WITHIN-PATIENT EXPRESSION CORRELATION")
print("=" * 65)

correlation_records = []

for patient_id, group in audit.groupby("patient_id"):

    print("\n" + "-" * 60)

    print("Patient:", patient_id)

    file_ids = group["file_id"].tolist()

    # Combine counts by shared gene identifier.

    patient_profiles = pd.concat(
        [
            expression_profiles[fid].rename(fid)
            for fid in file_ids
        ],
        axis=1,
        join="inner"
    )

    if patient_profiles.empty:
        raise RuntimeError(
            f"No shared gene identifiers for {patient_id}"
        )

    # Apply log1p transformation for correlation.
    # This is descriptive QC, not normalization
    # for differential expression.

    log_profiles = np.log1p(
        patient_profiles
    )

    correlation_matrix = (
        log_profiles.corr(method="pearson")
    )

    # Short identifiers for readability.

    short_names = {
        fid: fid[:8]
        for fid in file_ids
    }

    display_matrix = correlation_matrix.rename(
        index=short_names,
        columns=short_names
    )

    print("\nPearson correlation (log1p raw counts):")

    print(
        display_matrix.round(4).to_string()
    )

    # Save each pair once.

    for i in range(len(file_ids)):

        for j in range(i + 1, len(file_ids)):

            fid_a = file_ids[i]

            fid_b = file_ids[j]

            correlation_records.append({

                "patient_id": patient_id,

                "file_a": fid_a,

                "file_b": fid_b,

                "pearson_log1p": (
                    correlation_matrix.loc[
                        fid_a,
                        fid_b
                    ]
                ),

                "shared_genes": len(
                    patient_profiles
                )
            })

correlations = pd.DataFrame(
    correlation_records
)

# ------------------------------------------------------------
# 11. Display QC interpretation reminders
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("QUALITY ASSESSMENT LIMITATIONS")
print("-" * 65)

print(
    "\n1. Higher read depth does not automatically "
    "indicate better biological quality."
)

print(
    "2. Expression correlation is descriptive; "
    "high correlation does not prove identical samples."
)

print(
    "3. Different RNA aliquots may contain "
    "biological or technical variation."
)

print(
    "4. Workflow-version differences require "
    "consideration before combining datasets."
)

print(
    "5. No automatic quality thresholds "
    "or file exclusions have been applied."
)

# ------------------------------------------------------------
# 12. Save QC outputs
# ------------------------------------------------------------

qc_path = (
    RESULTS_DIR /
    "tcga_duplicate_rnaseq_qc_metrics.csv"
)

correlation_path = (
    RESULTS_DIR /
    "tcga_duplicate_rnaseq_correlations.csv"
)

download_manifest_path = (
    METADATA_DIR /
    "tcga_duplicate_qc_download_records.csv"
)

qc.to_csv(
    qc_path,
    index=False
)

correlations.to_csv(
    correlation_path,
    index=False
)

pd.DataFrame(
    download_records
).to_csv(
    download_manifest_path,
    index=False
)

# ------------------------------------------------------------
# 13. Save research provenance
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "1.12",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": "NCI Genomic Data Commons",

    "workflow": "STAR - Counts",

    "files_assessed": len(qc),

    "patients_assessed": int(
        qc["patient_id"].nunique()
    ),

    "count_column": "unstranded",

    "integrity_check": "MD5 checksum and file size",

    "qc_metrics": [
        "assigned_gene_counts",
        "detected_genes",
        "unmapped_counts",
        "multimapping_counts",
        "no_feature_counts",
        "ambiguous_counts",
        "assignment_fraction"
    ],

    "correlation_method": (
        "Pearson correlation of log1p unstranded "
        "gene counts using shared gene identifiers"
    ),

    "file_selection_performed": False,

    "limitations": [
        "QC statistics are preliminary.",
        "No sequencing-level FastQC/Picard review performed.",
        "Read depth alone is not a selection criterion.",
        "Expression similarity does not establish biological equivalence.",
        "Different workflow versions may introduce technical differences.",
        "No files were automatically excluded."
    ]
}

provenance_path = (
    METADATA_DIR /
    "tcga_duplicate_rnaseq_qc_provenance.json"
)

with open(provenance_path, "w") as f:

    json.dump(
        provenance,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 14. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.12 SUMMARY")
print("=" * 65)

print("\nPatients assessed:", qc["patient_id"].nunique())

print("Files assessed:", len(qc))

print(
    "All downloaded files passed integrity verification."
)

print("\nSaved outputs:")

print("1. tcga_duplicate_rnaseq_qc_metrics.csv")

print("2. tcga_duplicate_rnaseq_correlations.csv")

print("3. tcga_duplicate_qc_download_records.csv")

print("4. tcga_duplicate_rnaseq_qc_provenance.json")

print("\nMolecular files stored in:")

print(RAW_DIR)

print("\nNo final file selection performed.")

print("\nPATCH 1.12 COMPLETE")

print("=" * 65)

In [ ]:
# ============================================================
# SurfaceTarget-CRC
# Phase 1 — Patch 1.13
#
# RNA-SEQ QUALITY CROSS-VALIDATION
#
# Objectives:
# 1. Examine all three STAR counting modes
# 2. Investigate gene assignment statistics
# 3. Evaluate protein-coding gene detection
# 4. Verify gene annotation consistency
# 5. Calculate library-normalized expression correlations
#
# IMPORTANT:
# This is a diagnostic assessment.
# No final RNA-seq file selection is performed.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from itertools import combinations

import pandas as pd
import numpy as np
import hashlib
import json

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/SurfaceTarget-CRC")

METADATA_DIR = PROJECT_ROOT / "data" / "metadata"

RAW_DIR = (
    PROJECT_ROOT /
    "data/raw/tcga_duplicate_qc"
)

RESULTS_DIR = PROJECT_ROOT / "results/tables"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

AUDIT_PATH = (
    METADATA_DIR /
    "tcga_duplicate_rnaseq_provenance_audit.csv"
)

COUNT_COLUMNS = [
    "unstranded",
    "stranded_first",
    "stranded_second"
]

STAR_SUMMARY_ROWS = [
    "N_unmapped",
    "N_multimapping",
    "N_noFeature",
    "N_ambiguous"
]

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.13 — RNA-SEQ QUALITY CROSS-VALIDATION")
print("=" * 65)

# ------------------------------------------------------------
# 2. Load metadata
# ------------------------------------------------------------

audit = pd.read_csv(
    AUDIT_PATH,
    dtype=str,
    keep_default_na=False
)

if audit["file_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate file IDs in provenance audit."
    )

print("\nPatients:", audit["patient_id"].nunique())
print("Files:", len(audit))

# ------------------------------------------------------------
# 3. Load and validate STAR count files
# ------------------------------------------------------------

def load_star_file(file_path):

    df = pd.read_csv(
        file_path,
        sep="\t",
        comment="#",
        low_memory=False,
        dtype={"gene_id": str}
    )

    required = {
        "gene_id",
        *COUNT_COLUMNS
    }

    missing = required - set(df.columns)

    if missing:
        raise RuntimeError(
            f"Missing STAR columns: {missing}"
        )

    summary = df[
        df["gene_id"].isin(STAR_SUMMARY_ROWS)
    ].copy()

    genes = df[
        ~df["gene_id"].isin(STAR_SUMMARY_ROWS)
    ].copy()

    if genes["gene_id"].duplicated().any():
        raise RuntimeError(
            "Duplicate gene identifiers."
        )

    if set(summary["gene_id"]) != set(STAR_SUMMARY_ROWS):
        raise RuntimeError(
            "Incomplete STAR count-summary rows."
        )

    for column in COUNT_COLUMNS:

        genes[column] = pd.to_numeric(
            genes[column],
            errors="raise"
        )

        summary[column] = pd.to_numeric(
            summary[column],
            errors="raise"
        )

        if genes[column].isna().any():
            raise RuntimeError(
                f"Missing counts: {column}"
            )

        if (genes[column] < 0).any():
            raise RuntimeError(
                f"Negative counts: {column}"
            )

    return genes, summary


# ------------------------------------------------------------
# 4. Examine counting statistics
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("STAR COUNTING MODE ASSESSMENT")
print("-" * 65)

qc_records = []

gene_profiles = {}

gene_id_signatures = {}

for _, row in audit.iterrows():

    file_id = row["file_id"]

    file_path = RAW_DIR / f"{file_id}.tsv"

    if not file_path.exists():
        raise FileNotFoundError(
            f"Missing downloaded file: {file_path}"
        )

    genes, summary = load_star_file(file_path)

    # Verify gene identifier structure.

    gene_ids = sorted(
        genes["gene_id"].astype(str).tolist()
    )

    signature = hashlib.sha256(
        "\n".join(gene_ids).encode("utf-8")
    ).hexdigest()

    gene_id_signatures[file_id] = signature

    # Store unstranded profiles.

    profile = genes.set_index(
        "gene_id"
    )["unstranded"]

    gene_profiles[file_id] = profile

    # Identify protein-coding genes.

    if "gene_type" in genes.columns:

        protein_coding = genes[
            genes["gene_type"] == "protein_coding"
        ].copy()

    else:

        protein_coding = pd.DataFrame()

    for column in COUNT_COLUMNS:

        assigned = float(
            genes[column].sum()
        )

        star_stats = (
            summary.set_index("gene_id")[column]
            .to_dict()
        )

        unmapped = float(
            star_stats["N_unmapped"]
        )

        multimapping = float(
            star_stats["N_multimapping"]
        )

        no_feature = float(
            star_stats["N_noFeature"]
        )

        ambiguous = float(
            star_stats["N_ambiguous"]
        )

        total = (
            assigned
            + unmapped
            + multimapping
            + no_feature
            + ambiguous
        )

        if total <= 0:
            raise RuntimeError(
                f"Invalid total counts: {file_id}"
            )

        assignment_pct = (
            100 * assigned / total
        )

        ambiguous_pct = (
            100 * ambiguous / total
        )

        no_feature_pct = (
            100 * no_feature / total
        )

        detected_genes = int(
            (genes[column] > 0).sum()
        )

        if not protein_coding.empty:

            detected_protein_coding = int(
                (protein_coding[column] > 0).sum()
            )

            total_protein_coding = len(
                protein_coding
            )

        else:

            detected_protein_coding = np.nan
            total_protein_coding = np.nan

        qc_records.append({

            "patient_id": row["patient_id"],

            "file_id": file_id,

            "file_short": file_id[:8],

            "sample_id": row["sample_ids"],

            "count_mode": column,

            "total_genes": len(genes),

            "detected_genes": detected_genes,

            "total_protein_coding": total_protein_coding,

            "detected_protein_coding":
                detected_protein_coding,

            "assigned_counts": assigned,

            "assignment_percent": assignment_pct,

            "ambiguous_percent": ambiguous_pct,

            "no_feature_percent": no_feature_pct,

            "unmapped_counts": unmapped,

            "multimapping_counts": multimapping
        })

qc = pd.DataFrame(qc_records)

# ------------------------------------------------------------
# 5. Display count-mode comparison
# ------------------------------------------------------------

print("\nUnstranded and stranded count statistics:")

display_columns = [
    "patient_id",
    "file_short",
    "sample_id",
    "count_mode",
    "detected_genes",
    "detected_protein_coding",
    "assignment_percent",
    "ambiguous_percent",
    "no_feature_percent"
]

print(
    qc[display_columns]
    .round(2)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 6. Verify gene identifier consistency
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENE ANNOTATION CONSISTENCY")
print("-" * 65)

unique_signatures = set(
    gene_id_signatures.values()
)

print(
    "\nUnique gene identifier signatures:",
    len(unique_signatures)
)

print("\nGene counts by file:")

for file_id, profile in gene_profiles.items():

    print(
        f"{file_id[:8]}: "
        f"{len(profile)} genes"
    )

if len(unique_signatures) == 1:

    print("\nGene identifier consistency: PASSED")

else:

    print(
        "\nWARNING: Gene identifier sets differ."
    )

    print(
        "Files may require additional annotation "
        "harmonization before comparison."
    )

# ------------------------------------------------------------
# 7. Protein-coding gene profiles
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("NORMALIZED EXPRESSION CORRELATIONS")
print("-" * 65)

# We use log2(CPM + 1) for descriptive QC.
# This is NOT the normalization used for
# downstream differential expression.

normalized_profiles = {}

protein_coding_id_sets = {}

for _, row in audit.iterrows():

    file_id = row["file_id"]

    file_path = RAW_DIR / f"{file_id}.tsv"

    genes, _ = load_star_file(file_path)

    if "gene_type" not in genes.columns:

        raise RuntimeError(
            "gene_type column unavailable. "
            "Protein-coding correlation cannot proceed."
        )

    coding = genes[
        genes["gene_type"] == "protein_coding"
    ].copy()

    if coding.empty:
        raise RuntimeError(
            "No protein-coding annotations found."
        )

    protein_coding_id_sets[file_id] = set(
        coding["gene_id"]
    )

    counts = (
        coding.set_index("gene_id")["unstranded"]
        .astype(float)
    )

    # Use total counts across all annotated genes
    # as the descriptive CPM denominator.

    total_assigned = float(
        genes["unstranded"].sum()
    )

    if total_assigned <= 0:
        raise RuntimeError(
            f"Zero assigned counts: {file_id}"
        )

    cpm = (
        counts / total_assigned
    ) * 1_000_000

    normalized_profiles[file_id] = np.log2(
        cpm + 1
    )

# ------------------------------------------------------------
# 8. Compare protein-coding expression
# ------------------------------------------------------------

correlation_records = []

for patient_id, group in audit.groupby("patient_id"):

    print("\nPatient:", patient_id)

    file_ids = group["file_id"].tolist()

    patient_profiles = pd.concat(
        [
            normalized_profiles[fid].rename(fid)
            for fid in file_ids
        ],
        axis=1,
        join="inner"
    )

    if patient_profiles.empty:
        raise RuntimeError(
            "No shared protein-coding genes."
        )

    # Pearson correlation on normalized profiles.

    correlation_matrix = (
        patient_profiles.corr(method="pearson")
    )

    short_names = {
        fid: fid[:8]
        for fid in file_ids
    }

    display_matrix = correlation_matrix.rename(
        index=short_names,
        columns=short_names
    )

    print("\nPearson correlation: log2(CPM + 1)")

    print(
        display_matrix.round(4).to_string()
    )

    # Save pairwise correlations.

    for file_a, file_b in combinations(file_ids, 2):

        correlation_records.append({

            "patient_id": patient_id,

            "file_a": file_a,

            "file_b": file_b,

            "pearson_log2_cpm": float(
                correlation_matrix.loc[
                    file_a,
                    file_b
                ]
            ),

            "shared_protein_coding_genes":
                len(patient_profiles)
        })

correlations = pd.DataFrame(
    correlation_records
)

# ------------------------------------------------------------
# 9. Display interpretation notes
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("QUALITY INTERPRETATION")
print("-" * 65)

print(
    "\n1. Assignment fraction is not "
    "the same as genome alignment rate."
)

print(
    "2. Gene detection depends on sequencing "
    "depth and library characteristics."
)

print(
    "3. Protein-coding gene detection can provide "
    "additional context for unusual samples."
)

print(
    "4. Normalized correlation is descriptive "
    "and is not an independent quality score."
)

print(
    "5. No sample has been automatically "
    "excluded based on these metrics."
)

# ------------------------------------------------------------
# 10. Save results
# ------------------------------------------------------------

qc_path = (
    RESULTS_DIR /
    "tcga_duplicate_rnaseq_extended_qc.csv"
)

correlation_path = (
    RESULTS_DIR /
    "tcga_duplicate_normalized_correlations.csv"
)

signature_path = (
    METADATA_DIR /
    "tcga_duplicate_gene_identifier_signatures.json"
)

qc.to_csv(
    qc_path,
    index=False
)

correlations.to_csv(
    correlation_path,
    index=False
)

with open(signature_path, "w") as f:

    json.dump(
        gene_id_signatures,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 11. Save provenance
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "1.13",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "files_assessed": int(
        audit["file_id"].nunique()
    ),

    "count_modes": COUNT_COLUMNS,

    "gene_identifier_consistent": (
        len(unique_signatures) == 1
    ),

    "correlation_method": (
        "Pearson correlation of log2(CPM + 1) "
        "protein-coding gene expression"
    ),

    "cpm_denominator": (
        "Total unstranded counts assigned "
        "to all annotated genes"
    ),

    "file_selection_performed": False,

    "limitations": [
        "Count-based QC is not sequencing-level QC.",
        "Assignment fraction is not alignment rate.",
        "CPM correlations are descriptive.",
        "Different aliquots can contain biological variation.",
        "Gene identifier consistency does not guarantee identical workflow parameters.",
        "No automatic file exclusion performed."
    ]
}

provenance_path = (
    METADATA_DIR /
    "tcga_duplicate_extended_qc_provenance.json"
)

with open(provenance_path, "w") as f:

    json.dump(
        provenance,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 12. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.13 SUMMARY")
print("=" * 65)

print("\nFiles assessed:", len(audit))

print(
    "Gene identifier consistency:",
    "PASSED"
    if len(unique_signatures) == 1
    else "REQUIRES REVIEW"
)

print("\nSaved outputs:")

print("1. tcga_duplicate_rnaseq_extended_qc.csv")

print("2. tcga_duplicate_normalized_correlations.csv")

print("3. tcga_duplicate_gene_identifier_signatures.json")

print("4. tcga_duplicate_extended_qc_provenance.json")

print("\nNo additional molecular files downloaded.")

print("No final file selection performed.")

print("\nPATCH 1.13 COMPLETE")
print("=" * 65)

In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PATCH 1.14
#
# COHORT-WIDE RNA-SEQ PROCESSING CONSISTENCY AUDIT
#
# Objectives:
# 1. Retrieve cohort-wide GDC processing metadata
# 2. Examine STAR workflow-version distributions
# 3. Verify file metadata integrity
# 4. Compare duplicate candidates with the wider cohort
# 5. Evaluate protein-coding gene detection at CPM >= 1
#
# No final file selection.
# No additional molecular data downloads.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import requests
import json
import time

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"
RESULTS = ROOT / "results/tables"

RAW = ROOT / "data/raw/tcga_duplicate_qc"

RESULTS.mkdir(parents=True, exist_ok=True)

TUMOR_PATH = META / "tcga_candidate_tumor_rnaseq_files.csv"
NORMAL_PATH = META / "tcga_candidate_normal_rnaseq_files.csv"

DUPLICATE_PATH = (
    META / "tcga_duplicate_rnaseq_provenance_audit.csv"
)

QC_PATH = (
    RESULTS / "tcga_duplicate_rnaseq_extended_qc.csv"
)

GDC_API = "https://api.gdc.cancer.gov/files"

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.14 — COHORT-WIDE WORKFLOW AUDIT")
print("=" * 65)

# ------------------------------------------------------------
# 2. Load inventories
# ------------------------------------------------------------

tumor = pd.read_csv(
    TUMOR_PATH,
    dtype=str,
    keep_default_na=False
)

normal = pd.read_csv(
    NORMAL_PATH,
    dtype=str,
    keep_default_na=False
)

normal["cohort_tier"] = "Normal control"

inventory = pd.concat(
    [tumor, normal],
    ignore_index=True
)

if inventory["file_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate file IDs detected across inventories."
    )

print("\nCandidate tumor files:", len(tumor))
print("Normal tissue files:", len(normal))

print("Total unique files:", len(inventory))

# ------------------------------------------------------------
# 3. Retrieve GDC metadata in batches
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("RETRIEVING COHORT-WIDE PROCESSING METADATA")
print("-" * 65)

session = requests.Session()

session.headers.update({
    "Accept": "application/json",
    "User-Agent": "SurfaceTarget-CRC-Research/1.0"
})

fields = [
    "file_id",
    "file_name",
    "file_size",
    "md5sum",
    "analysis.workflow_type",
    "analysis.workflow_version",
    "created_datetime",
    "updated_datetime"
]

file_ids = sorted(
    inventory["file_id"].unique()
)

BATCH_SIZE = 75

all_records = []

for start in range(0, len(file_ids), BATCH_SIZE):

    batch = file_ids[
        start:start + BATCH_SIZE
    ]

    filters = {
        "op": "in",
        "content": {
            "field": "file_id",
            "value": batch
        }
    }

    params = {
        "filters": json.dumps(filters),
        "fields": ",".join(fields),
        "format": "JSON",
        "size": len(batch)
    }

    for attempt in range(1, 4):

        try:

            response = session.get(
                GDC_API,
                params=params,
                timeout=90
            )

            response.raise_for_status()

            hits = response.json()["data"]["hits"]

            returned_ids = {
                item["file_id"]
                for item in hits
            }

            if returned_ids != set(batch):
                raise RuntimeError(
                    "GDC response missing requested files."
                )

            all_records.extend(hits)

            print(
                f"Retrieved {len(all_records)} "
                f"of {len(file_ids)} files"
            )

            break

        except Exception as e:

            if attempt == 3:
                raise

            print(
                f"Retry {attempt}: {e}"
            )

            time.sleep(2 ** attempt)

print("\nMetadata retrieval: COMPLETE")

# ------------------------------------------------------------
# 4. Extract processing metadata
# ------------------------------------------------------------

records = []

for item in all_records:

    analysis = item.get("analysis") or {}

    records.append({

        "file_id": item["file_id"],

        "gdc_file_name": item.get("file_name"),

        "gdc_file_size": item.get("file_size"),

        "gdc_md5sum": item.get("md5sum"),

        "workflow_type": analysis.get("workflow_type"),

        "workflow_version": analysis.get("workflow_version"),

        "created_datetime": item.get(
            "created_datetime"
        ),

        "updated_datetime": item.get(
            "updated_datetime"
        )
    })

metadata = pd.DataFrame(records)

if metadata["file_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate file IDs in GDC response."
    )

# ------------------------------------------------------------
# 5. Merge with cohort inventory
# ------------------------------------------------------------

merged = inventory.merge(
    metadata,
    on="file_id",
    how="left",
    validate="one_to_one"
)

if len(merged) != len(inventory):
    raise RuntimeError(
        "File count changed during metadata merge."
    )

if merged["workflow_type"].isna().any():
    raise RuntimeError(
        "Missing workflow metadata detected."
    )

# ------------------------------------------------------------
# 6. Verify file metadata
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("FILE METADATA VALIDATION")
print("-" * 65)

size_match = (
    pd.to_numeric(merged["file_size_bytes"])
    ==
    pd.to_numeric(merged["gdc_file_size"])
)

md5_match = (
    merged["md5sum"].str.lower()
    ==
    merged["gdc_md5sum"].str.lower()
)

workflow_match = (
    merged["workflow_type"] == "STAR - Counts"
)

print(
    "\nFile sizes matching:",
    int(size_match.sum()),
    "/",
    len(merged)
)

print(
    "MD5 checksums matching:",
    int(md5_match.sum()),
    "/",
    len(merged)
)

print(
    "STAR workflow confirmed:",
    int(workflow_match.sum()),
    "/",
    len(merged)
)

if not (
    size_match.all()
    and md5_match.all()
    and workflow_match.all()
):
    raise RuntimeError(
        "File metadata validation failed."
    )

print("\nFile metadata validation: PASSED")

# ------------------------------------------------------------
# 7. Examine workflow-version distribution
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("WORKFLOW VERSION DISTRIBUTION")
print("-" * 65)

merged["version_short"] = (
    merged["workflow_version"]
    .fillna("UNKNOWN")
    .astype(str)
    .str[:12]
)

version_summary = (
    merged.groupby(
        ["workflow_version"],
        dropna=False
    )
    .agg(
        n_files=("file_id", "nunique")
    )
    .reset_index()
    .sort_values(
        "n_files",
        ascending=False
    )
)

print(
    "\nUnique workflow versions:",
    merged["workflow_version"].nunique()
)

print("\nOverall distribution:")

print(
    version_summary.to_string(index=False)
)

# ------------------------------------------------------------
# 8. Project and sample-type distributions
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("WORKFLOW DISTRIBUTION BY PROJECT")
print("-" * 65)

project_versions = (
    merged.groupby(
        [
            "project",
            "sample_type",
            "version_short"
        ],
        dropna=False
    )
    .agg(
        files=("file_id", "nunique")
    )
    .reset_index()
)

print(
    "\n",
    project_versions.to_string(index=False)
)

# ------------------------------------------------------------
# 9. Investigate six duplicate candidates
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("DUPLICATE FILE WORKFLOW COMPARISON")
print("-" * 65)

duplicate_audit = pd.read_csv(
    DUPLICATE_PATH,
    dtype=str,
    keep_default_na=False
)

duplicate_ids = set(
    duplicate_audit["file_id"]
)

duplicate_versions = merged[
    merged["file_id"].isin(duplicate_ids)
].copy()

qc = pd.read_csv(QC_PATH)

qc_unstranded = qc[
    qc["count_mode"] == "unstranded"
].copy()

qc_columns = [
    "file_id",
    "detected_genes",
    "detected_protein_coding",
    "assignment_percent",
    "no_feature_percent"
]

duplicate_versions = duplicate_versions.merge(
    qc_unstranded[qc_columns],
    on="file_id",
    how="left",
    validate="one_to_one"
)

display_columns = [
    "patient_id",
    "sample_id",
    "file_id",
    "version_short",
    "detected_protein_coding",
    "assignment_percent",
    "no_feature_percent"
]

print(
    "\n",
    duplicate_versions[
        display_columns
    ]
    .round(2)
    .sort_values(
        ["patient_id", "sample_id", "file_id"]
    )
    .to_string(index=False)
)

# ------------------------------------------------------------
# 10. Protein-coding gene detection at CPM >= 1
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PROTEIN-CODING EXPRESSION DETECTION")
print("-" * 65)

expression_records = []

summary_rows = {
    "N_unmapped",
    "N_multimapping",
    "N_noFeature",
    "N_ambiguous"
}

for _, row in duplicate_versions.iterrows():

    file_id = row["file_id"]

    path = RAW / f"{file_id}.tsv"

    if not path.exists():
        raise FileNotFoundError(path)

    df = pd.read_csv(
        path,
        sep="\t",
        comment="#",
        usecols=[
            "gene_id",
            "gene_type",
            "unstranded"
        ]
    )

    df = df[
        ~df["gene_id"].isin(summary_rows)
    ].copy()

    df["unstranded"] = pd.to_numeric(
        df["unstranded"],
        errors="raise"
    )

    total_assigned = df["unstranded"].sum()

    if total_assigned <= 0:
        raise RuntimeError(
            f"Invalid library counts: {file_id}"
        )

    coding = df[
        df["gene_type"] == "protein_coding"
    ].copy()

    coding["cpm"] = (
        coding["unstranded"]
        / total_assigned
        * 1_000_000
    )

    detected_cpm1 = int(
        (coding["cpm"] >= 1).sum()
    )

    detected_cpm5 = int(
        (coding["cpm"] >= 5).sum()
    )

    expression_records.append({

        "patient_id": row["patient_id"],

        "sample_id": row["sample_id"],

        "file_id": file_id,

        "file_short": file_id[:8],

        "protein_coding_cpm_ge_1":
            detected_cpm1,

        "protein_coding_cpm_ge_5":
            detected_cpm5
    })

expression_qc = pd.DataFrame(
    expression_records
)

print(
    "\n",
    expression_qc[
        [
            "patient_id",
            "sample_id",
            "file_short",
            "protein_coding_cpm_ge_1",
            "protein_coding_cpm_ge_5"
        ]
    ]
    .sort_values(
        ["patient_id", "sample_id", "file_short"]
    )
    .to_string(index=False)
)

print("\nNOTE:")
print(
    "CPM thresholds are descriptive detection "
    "criteria, not automatic exclusion rules."
)

# ------------------------------------------------------------
# 11. Save results
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING WORKFLOW AUDIT")
print("-" * 65)

merged.to_csv(
    META / "tcga_cohort_workflow_metadata.csv",
    index=False
)

version_summary.to_csv(
    RESULTS / "tcga_workflow_version_summary.csv",
    index=False
)

project_versions.to_csv(
    RESULTS / "tcga_workflow_project_summary.csv",
    index=False
)

duplicate_versions.to_csv(
    RESULTS / "tcga_duplicate_workflow_comparison.csv",
    index=False
)

expression_qc.to_csv(
    RESULTS / "tcga_duplicate_cpm_detection_qc.csv",
    index=False
)

# ------------------------------------------------------------
# 12. Save provenance
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "1.14",

    "retrieved_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": "NCI GDC API",

    "files_audited": len(merged),

    "unique_workflow_versions": int(
        merged["workflow_version"].nunique()
    ),

    "file_integrity_metadata_consistent": True,

    "duplicate_files_assessed": len(expression_qc),

    "cpm_thresholds": [1, 5],

    "count_column": "unstranded",

    "final_file_selection_performed": False,

    "limitations": [
        "Workflow version identifiers do not establish quality rankings.",
        "CPM-based detection is descriptive.",
        "RNA-seq library characteristics may differ.",
        "Sequencing-level QC has not been independently assessed.",
        "No final RNA-seq file selection was performed."
    ]
}

with open(
    META / "tcga_workflow_audit_provenance.json",
    "w"
) as f:

    json.dump(
        provenance,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 13. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.14 SUMMARY")
print("=" * 65)

print("\nFiles audited:", len(merged))

print(
    "Workflow versions identified:",
    merged["workflow_version"].nunique()
)

print("\nAll file metadata checks passed.")

print("\nSaved outputs:")

print("1. tcga_cohort_workflow_metadata.csv")
print("2. tcga_workflow_version_summary.csv")
print("3. tcga_workflow_project_summary.csv")
print("4. tcga_duplicate_workflow_comparison.csv")
print("5. tcga_duplicate_cpm_detection_qc.csv")
print("6. tcga_workflow_audit_provenance.json")

print("\nNo molecular datasets downloaded.")
print("No final RNA-seq file selection performed.")

print("\nPATCH 1.14 COMPLETE")
print("=" * 65)

In [ ]:
# ============================================================
# SurfaceTarget-CRC
# PATCH 1.15
#
# PROVISIONAL RNA-SEQ FILE SELECTION
# AND DOWNLOAD MANIFEST GENERATION
#
# Objectives:
# 1. Select one primary tumor RNA-seq file per patient
# 2. Apply documented decisions for ambiguous patients
# 3. Preserve alternative files for sensitivity analysis
# 4. Validate sample identifiers and cohort membership
# 5. Generate reproducible download manifests
# 6. Examine selected workflow distributions
# 7. Create a research checkpoint
#
# IMPORTANT:
# Selection is provisional pending full-cohort QC.
# No additional molecular files are downloaded.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
import json

# ------------------------------------------------------------
# 1. Project configuration
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"

RESULTS = ROOT / "results/tables"

CHECKPOINTS = ROOT / "checkpoints"

RESULTS.mkdir(parents=True, exist_ok=True)
CHECKPOINTS.mkdir(parents=True, exist_ok=True)

WORKFLOW_PATH = (
    META / "tcga_cohort_workflow_metadata.csv"
)

COHORT_PATH = (
    META / "tcga_mss_evidence_tiers.csv"
)

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.15 — RNA-SEQ FILE SELECTION")
print("=" * 65)

# ------------------------------------------------------------
# 2. Load metadata
# ------------------------------------------------------------

for path in [WORKFLOW_PATH, COHORT_PATH]:

    if not path.exists():
        raise FileNotFoundError(
            f"Required metadata missing: {path}"
        )

inventory = pd.read_csv(
    WORKFLOW_PATH,
    dtype=str,
    keep_default_na=False
)

cohort = pd.read_csv(
    COHORT_PATH,
    dtype=str,
    keep_default_na=False
)

print("\nMetadata loaded successfully.")

print(
    "Total files in workflow inventory:",
    len(inventory)
)

print(
    "Total cohort records:",
    len(cohort)
)

# ------------------------------------------------------------
# 3. Separate tumor and normal tissue
# ------------------------------------------------------------

tumor = inventory[
    inventory["sample_type"] == "Primary Tumor"
].copy()

normal = inventory[
    inventory["sample_type"] == "Solid Tissue Normal"
].copy()

print("\nCandidate tumor files:", len(tumor))

print("Normal tissue files:", len(normal))

print(
    "Unique tumor patients:",
    tumor["patient_id"].nunique()
)

# ------------------------------------------------------------
# 4. Validate source inventory
# ------------------------------------------------------------

if inventory["file_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate file IDs in source inventory."
    )

if tumor["patient_id"].nunique() != 421:
    raise RuntimeError(
        "Unexpected tumor patient count."
    )

if len(tumor) != 425:
    raise RuntimeError(
        "Unexpected candidate tumor file count."
    )

if len(normal) != 51:
    raise RuntimeError(
        "Unexpected normal tissue file count."
    )

if not tumor["cohort_tier"].isin(
    ["Tier A", "Tier B"]
).all():
    raise RuntimeError(
        "Unexpected tumor cohort tier."
    )

print("\nSource inventory validation: PASSED")

# ------------------------------------------------------------
# 5. Define documented selections
# ------------------------------------------------------------

# These decisions are based on the detailed
# biospecimen and RNA-seq QC investigations
# completed in Patches 1.11 through 1.14.

MANUAL_SELECTIONS = {

    "TCGA-A6-2677": {

        "file_id":
        "c834cb75-259c-40bd-93cc-48f9756cc47a",

        "reason": (
            "Selected 01A aliquot. "
            "High gene-assignment fraction, "
            "consistent protein-coding detection, "
            "and strong correlation with the "
            "alternative 01A profile. "
            "Workflow version represented in "
            "the wider cohort."
        )
    },

    "TCGA-A6-3810": {

        "file_id":
        "cd10bd5e-b75b-4d47-a103-9c259dcf1a19",

        "reason": (
            "Selected 01A aliquot using the "
            "most common recorded workflow version. "
            "Higher assigned gene counts and "
            "protein-coding detection at CPM >= 1 "
            "than the alternative 01A profile. "
            "Strong within-vial expression correlation."
        )
    }
}

# ------------------------------------------------------------
# 6. Identify ambiguous patients
# ------------------------------------------------------------

patient_file_counts = (
    tumor.groupby("patient_id")["file_id"]
    .nunique()
)

ambiguous_patients = set(
    patient_file_counts[
        patient_file_counts > 1
    ].index
)

print("\n" + "-" * 65)
print("AMBIGUOUS PATIENT VALIDATION")
print("-" * 65)

print(
    "\nPatients with multiple files:",
    len(ambiguous_patients)
)

if ambiguous_patients != set(MANUAL_SELECTIONS):

    raise RuntimeError(
        "Ambiguous patient list differs "
        "from documented selection decisions."
    )

print("\nAmbiguous patient validation: PASSED")

# ------------------------------------------------------------
# 7. Construct selected file list
# ------------------------------------------------------------

selection_records = []

for patient_id, group in tumor.groupby("patient_id"):

    available_ids = set(
        group["file_id"]
    )

    if len(group) == 1:

        selected_id = group.iloc[0]["file_id"]

        selection_type = "Single available file"

        selection_reason = (
            "Only one eligible primary tumor "
            "STAR Counts file available."
        )

    else:

        decision = MANUAL_SELECTIONS[patient_id]

        selected_id = decision["file_id"]

        if selected_id not in available_ids:

            raise RuntimeError(
                f"Selected file unavailable: "
                f"{patient_id}"
            )

        selection_type = "Documented manual selection"

        selection_reason = decision["reason"]

    selection_records.append({

        "patient_id": patient_id,

        "selected_file_id": selected_id,

        "candidate_files": len(group),

        "selection_type": selection_type,

        "selection_reason": selection_reason
    })

selection_table = pd.DataFrame(
    selection_records
)

# ------------------------------------------------------------
# 8. Create selected tumor manifest
# ------------------------------------------------------------

selected = tumor.merge(
    selection_table,
    on="patient_id",
    how="inner",
    validate="many_to_one"
)

selected = selected[
    selected["file_id"]
    == selected["selected_file_id"]
].copy()

selected = selected.sort_values(
    ["project", "cohort_tier", "patient_id"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 9. Validate one file per patient
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SELECTED FILE VALIDATION")
print("-" * 65)

if len(selected) != 421:
    raise RuntimeError(
        "Selected tumor file count is not 421."
    )

if selected["patient_id"].nunique() != 421:
    raise RuntimeError(
        "Some patients have multiple or missing files."
    )

if selected["file_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate selected file IDs detected."
    )

selected_counts = (
    selected.groupby("patient_id")
    .size()
)

if not selected_counts.eq(1).all():
    raise RuntimeError(
        "One-file-per-patient validation failed."
    )

print("\nOne tumor file per patient: PASSED")

# ------------------------------------------------------------
# 10. Validate cohort evidence membership
# ------------------------------------------------------------

eligible_cohort = cohort[
    cohort["evidence_tier"].isin([
        "Tier A - Published MSS and concordant low MSI",
        "Tier B - Concordant low MSI only"
    ])
].copy()

if eligible_cohort["patientId"].duplicated().any():
    raise RuntimeError(
        "Duplicate patient identifiers in evidence tiers."
    )

if set(selected["patient_id"]) != set(
    eligible_cohort["patientId"]
):
    raise RuntimeError(
        "Selected patients differ from eligible cohorts."
    )

# Attach original MSI identifiers to the manifest.

msi_columns = [
    "patientId",
    "mantis_sample_id",
    "msisensor_sample_id",
    "evidence_tier"
]

selected = selected.merge(
    eligible_cohort[msi_columns],
    left_on="patient_id",
    right_on="patientId",
    how="left",
    validate="one_to_one"
)

# TCGA sample-type barcode correspondence.
#
# The first 15 characters identify the
# patient and sample-type code.
#
# This does not prove that the RNA-seq and
# MSI measurements used identical aliquots.

sample_matches = (
    selected["sample_id"].str[:15]
    ==
    selected["mantis_sample_id"].str[:15]
)

if not sample_matches.all():

    mismatches = selected.loc[
        ~sample_matches,
        ["patient_id", "sample_id", "mantis_sample_id"]
    ]

    print(mismatches.to_string(index=False))

    raise RuntimeError(
        "Selected RNA-seq/MSI sample linkage failed."
    )

print("MSI sample-type correspondence: PASSED")

# ------------------------------------------------------------
# 11. Validate cohort-specific counts
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("FINAL SELECTED TUMOR FILE COUNTS")
print("-" * 65)

cohort_counts = (
    selected.groupby(
        ["project", "cohort_tier"]
    )
    .agg(
        patients=("patient_id", "nunique"),
        selected_files=("file_id", "nunique")
    )
    .reset_index()
)

print("\n")
print(cohort_counts.to_string(index=False))

expected_counts = {
    ("TCGA-COAD", "Tier A"): 98,
    ("TCGA-COAD", "Tier B"): 194,
    ("TCGA-READ", "Tier A"): 46,
    ("TCGA-READ", "Tier B"): 83
}

for (project, tier), expected in expected_counts.items():

    actual = len(
        selected[
            (selected["project"] == project)
            & (selected["cohort_tier"] == tier)
        ]
    )

    if actual != expected:

        raise RuntimeError(
            f"Unexpected cohort size for "
            f"{project} {tier}: {actual}"
        )

print("\nCohort count validation: PASSED")

# ------------------------------------------------------------
# 12. Identify excluded alternative files
# ------------------------------------------------------------

selected_ids = set(
    selected["file_id"]
)

excluded = tumor[
    ~tumor["file_id"].isin(selected_ids)
].copy()

excluded["exclusion_reason"] = (
    "Alternative tumor RNA-seq file. "
    "Excluded from primary analysis to prevent "
    "multiple measurements per patient. "
    "Retained for sensitivity analysis."
)

print("\n" + "-" * 65)
print("ALTERNATIVE FILE AUDIT")
print("-" * 65)

print("\nAlternative files excluded:", len(excluded))

if len(excluded) != 4:
    raise RuntimeError(
        "Unexpected number of alternative files."
    )

print("\nExcluded alternatives:")

print(
    excluded[
        [
            "patient_id",
            "sample_id",
            "file_id",
            "version_short"
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# 13. Preserve normal tissue inventory
# ------------------------------------------------------------

normal = normal.sort_values(
    ["project", "patient_id"]
).reset_index(drop=True)

print("\n" + "-" * 65)
print("NORMAL TISSUE CONTROL INVENTORY")
print("-" * 65)

normal_summary = (
    normal.groupby("project")
    .agg(
        patients=("patient_id", "nunique"),
        files=("file_id", "nunique")
    )
    .reset_index()
)

print("\n")
print(normal_summary.to_string(index=False))

print("\nNormal tissues are not assigned MSS status.")

# ------------------------------------------------------------
# 14. Selected workflow version distribution
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SELECTED WORKFLOW VERSION DISTRIBUTION")
print("-" * 65)

workflow_distribution = pd.crosstab(
    [
        selected["project"],
        selected["cohort_tier"]
    ],
    selected["version_short"]
)

print("\nTumor samples:")

print(workflow_distribution.to_string())

print("\nNormal tissue workflow versions:")

normal_workflows = pd.crosstab(
    normal["project"],
    normal["version_short"]
)

print(normal_workflows.to_string())

# ------------------------------------------------------------
# 15. Storage requirements
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SELECTED DOWNLOAD STORAGE")
print("-" * 65)

selected["file_size_bytes"] = pd.to_numeric(
    selected["file_size_bytes"],
    errors="raise"
)

normal["file_size_bytes"] = pd.to_numeric(
    normal["file_size_bytes"],
    errors="raise"
)

tumor_gb = (
    selected["file_size_bytes"].sum()
    / (1024 ** 3)
)

normal_gb = (
    normal["file_size_bytes"].sum()
    / (1024 ** 3)
)

print(f"\nSelected tumor files: {tumor_gb:.3f} GiB")

print(f"Normal tissue files: {normal_gb:.3f} GiB")

print(
    f"Combined storage: {tumor_gb + normal_gb:.3f} GiB"
)

# ------------------------------------------------------------
# 16. Prepare local download paths
# ------------------------------------------------------------

# These paths will be used by the downloader
# in the next patch.

TUMOR_RAW_DIR = (
    ROOT / "data/raw/tcga_star_counts/tumor"
)

NORMAL_RAW_DIR = (
    ROOT / "data/raw/tcga_star_counts/normal"
)

selected["local_path"] = selected["file_id"].apply(
    lambda fid: str(
        TUMOR_RAW_DIR / f"{fid}.tsv"
    )
)

normal["local_path"] = normal["file_id"].apply(
    lambda fid: str(
        NORMAL_RAW_DIR / f"{fid}.tsv"
    )
)

selected["selection_status"] = (
    "Provisional - pending full cohort QC"
)

normal["selection_status"] = (
    "Candidate normal control - pending QC"
)

# ------------------------------------------------------------
# 17. Save final manifests
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING RNA-SEQ MANIFESTS")
print("-" * 65)

tumor_manifest_path = (
    META / "tcga_selected_tumor_rnaseq_manifest.csv"
)

normal_manifest_path = (
    META / "tcga_normal_rnaseq_manifest.csv"
)

excluded_path = (
    META / "tcga_excluded_alternative_rnaseq_files.csv"
)

selection_path = (
    META / "tcga_rnaseq_selection_decisions.csv"
)

workflow_path = (
    RESULTS / "tcga_selected_workflow_distribution.csv"
)

selected.to_csv(
    tumor_manifest_path,
    index=False
)

normal.to_csv(
    normal_manifest_path,
    index=False
)

excluded.to_csv(
    excluded_path,
    index=False
)

selection_table.to_csv(
    selection_path,
    index=False
)

workflow_distribution.to_csv(
    workflow_path
)

print("\nAll manifests saved successfully.")

# ------------------------------------------------------------
# 18. Save selection methodology
# ------------------------------------------------------------

methodology = {

    "project": "SurfaceTarget-CRC",

    "patch": "1.15",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "selection_policy_version": "1.15-v1",

    "selection_status": (
        "Provisional pending full cohort QC"
    ),

    "primary_rule": (
        "Retain one STAR Counts primary tumor "
        "expression file per patient."
    ),

    "ambiguous_patient_decisions": MANUAL_SELECTIONS,

    "selected_tumor_patients": int(
        selected["patient_id"].nunique()
    ),

    "selected_tumor_files": int(
        selected["file_id"].nunique()
    ),

    "candidate_normal_files": int(
        normal["file_id"].nunique()
    ),

    "excluded_alternative_files": int(
        len(excluded)
    ),

    "count_column": "unstranded",

    "download_performed": False,

    "limitations": [
        "File selection is provisional.",
        "Full-cohort RNA-seq QC remains pending.",
        "Normal samples are not automatically matched controls.",
        "Tier B is not categorically verified MSS.",
        "Alternative aliquots should be examined in sensitivity analyses.",
        "Workflow-version differences require downstream assessment.",
        "MSI and RNA-seq matching does not establish identical aliquots."
    ]
}

methodology_path = (
    META / "tcga_rnaseq_final_selection_methodology.json"
)

with open(methodology_path, "w") as f:

    json.dump(
        methodology,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 19. Create research checkpoint
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING RESEARCH CHECKPOINT")
print("-" * 65)

timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S")

checkpoint_path = (
    CHECKPOINTS /
    f"SurfaceTarget_CRC_Patch115_{timestamp}.zip"
)

backup_folders = [
    "data/metadata",
    "results/tables",
    "results/figures",
    "src",
    "reports",
    "logs"
]

with ZipFile(
    checkpoint_path,
    "w",
    ZIP_DEFLATED
) as archive:

    for folder in backup_folders:

        folder_path = ROOT / folder

        if folder_path.exists():

            for file_path in folder_path.rglob("*"):

                if file_path.is_file():

                    archive.write(
                        file_path,
                        arcname=str(
                            file_path.relative_to(ROOT)
                        )
                    )

checkpoint_size_mb = (
    checkpoint_path.stat().st_size
    / (1024 ** 2)
)

print("\nCheckpoint created successfully.")

print(
    f"Checkpoint size: {checkpoint_size_mb:.2f} MB"
)

print("\nCheckpoint location:")
print(checkpoint_path)

# ------------------------------------------------------------
# 20. Final project summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.15 SUMMARY")
print("=" * 65)

print("\nTumor patients:", len(selected))

print(
    "Selected tumor RNA-seq files:",
    selected["file_id"].nunique()
)

print(
    "Normal tissue files:",
    normal["file_id"].nunique()
)

print(
    "Excluded alternative files:",
    len(excluded)
)

print("\nFiles saved:")

print("1. tcga_selected_tumor_rnaseq_manifest.csv")

print("2. tcga_normal_rnaseq_manifest.csv")

print("3. tcga_excluded_alternative_rnaseq_files.csv")

print("4. tcga_rnaseq_selection_decisions.csv")

print("5. tcga_selected_workflow_distribution.csv")

print("6. tcga_rnaseq_final_selection_methodology.json")

print("\nResearch checkpoint: CREATED")

print("\nNo additional molecular files downloaded.")

print("\nRNA-seq manifests are ready for download.")

print("\nPATCH 1.15 COMPLETE")

print("=" * 65)

In [ ]:
# ============================================================
# SurfaceTarget-CRC
# PATCH 1.16
#
# FULL TCGA RNA-SEQ DOWNLOAD
# AND DATA INTEGRITY VERIFICATION
#
# Objectives:
# 1. Load selected tumor/normal manifests
# 2. Validate file metadata and access permissions
# 3. Download selected STAR Counts expression files
# 4. Reuse verified existing files
# 5. Verify file size and MD5 checksum
# 6. Maintain resumable download records
# 7. Generate final download summary
#
# IMPORTANT:
# No differential expression analysis is performed.
# No new biological classifications are assigned.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import requests
import hashlib
import shutil
import json
import time
import os
import uuid

# ------------------------------------------------------------
# 1. Project configuration
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"

RAW = ROOT / "data/raw/tcga_star_counts"

RESULTS = ROOT / "results/tables"

TUMOR_DIR = RAW / "tumor"

NORMAL_DIR = RAW / "normal"

for directory in [
    META,
    RAW,
    RESULTS,
    TUMOR_DIR,
    NORMAL_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

TUMOR_MANIFEST = (
    META / "tcga_selected_tumor_rnaseq_manifest.csv"
)

NORMAL_MANIFEST = (
    META / "tcga_normal_rnaseq_manifest.csv"
)

DOWNLOAD_LOG = (
    META / "tcga_rnaseq_download_log.csv"
)

GDC_DATA_API = "https://api.gdc.cancer.gov/data"

# Set to None to process the complete dataset.
# Set to an integer, e.g. 20, for a smaller test batch.

MAX_NEW_DOWNLOADS = None

MAX_ATTEMPTS = 4

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.16 — FULL RNA-SEQ DOWNLOAD")
print("=" * 65)


# ------------------------------------------------------------
# 2. Load manifests
# ------------------------------------------------------------

for path in [
    TUMOR_MANIFEST,
    NORMAL_MANIFEST
]:

    if not path.exists():

        raise FileNotFoundError(
            f"Required manifest missing: {path}"
        )

tumor = pd.read_csv(
    TUMOR_MANIFEST,
    dtype=str,
    keep_default_na=False
)

normal = pd.read_csv(
    NORMAL_MANIFEST,
    dtype=str,
    keep_default_na=False
)

tumor["dataset_group"] = "Tumor"

normal["dataset_group"] = "Normal"

manifest = pd.concat(
    [tumor, normal],
    ignore_index=True
)

print("\nManifest loaded successfully.")

print("Selected tumor files:", len(tumor))

print("Normal tissue files:", len(normal))

print("Total files:", len(manifest))


# ------------------------------------------------------------
# 3. Validate manifests
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("MANIFEST VALIDATION")
print("-" * 65)

required_columns = [
    "file_id",
    "file_size_bytes",
    "md5sum",
    "access",
    "project",
    "patient_id",
    "sample_id"
]

missing_columns = [
    column
    for column in required_columns
    if column not in manifest.columns
]

if missing_columns:

    raise RuntimeError(
        f"Missing columns: {missing_columns}"
    )

if len(tumor) != 421:

    raise RuntimeError(
        "Unexpected tumor file count."
    )

if len(normal) != 51:

    raise RuntimeError(
        "Unexpected normal file count."
    )

if manifest["file_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate file UUIDs detected."
    )

# Validate UUID formatting.

for file_id in manifest["file_id"]:

    try:

        if str(uuid.UUID(file_id)) != file_id.lower():

            raise ValueError("Noncanonical UUID")

    except ValueError:

        raise RuntimeError(
            f"Invalid GDC file UUID: {file_id}"
        )

# Validate access.

access_values = (
    manifest["access"]
    .str.strip()
    .str.lower()
)

if not access_values.eq("open").all():

    restricted = manifest.loc[
        ~access_values.eq("open"),
        ["file_id", "access"]
    ]

    raise RuntimeError(
        "Some files are not marked open-access:\n"
        + restricted.to_string(index=False)
    )

# Validate expected file sizes.

manifest["file_size_bytes"] = pd.to_numeric(
    manifest["file_size_bytes"],
    errors="raise"
).astype("int64")

if (
    manifest["file_size_bytes"] <= 0
).any():

    raise RuntimeError(
        "Invalid expected file sizes."
    )

# Validate MD5 formatting.

manifest["md5sum"] = (
    manifest["md5sum"]
    .str.strip()
    .str.lower()
)

valid_md5 = (
    manifest["md5sum"]
    .str.fullmatch(r"[0-9a-f]{32}")
)

if not valid_md5.all():

    raise RuntimeError(
        "Invalid MD5 checksums in manifest."
    )

print("\nFile count validation: PASSED")
print("Unique UUID validation: PASSED")
print("Open-access validation: PASSED")
print("File size validation: PASSED")
print("MD5 metadata validation: PASSED")


# ------------------------------------------------------------
# 4. Check available Colab storage
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("STORAGE ASSESSMENT")
print("-" * 65)

expected_bytes = int(
    manifest["file_size_bytes"].sum()
)

expected_gib = (
    expected_bytes / (1024 ** 3)
)

disk = shutil.disk_usage("/content")

free_gib = (
    disk.free / (1024 ** 3)
)

print(
    f"\nExpected total dataset: "
    f"{expected_gib:.3f} GiB"
)

print(
    f"Available Colab storage: "
    f"{free_gib:.2f} GiB"
)

# Conservative free-space requirement.

if disk.free < expected_bytes * 2:

    raise RuntimeError(
        "Insufficient free disk space "
        "for safe download."
    )

print("\nStorage assessment: PASSED")


# ------------------------------------------------------------
# 5. MD5 verification functions
# ------------------------------------------------------------

def calculate_md5(file_path):

    md5 = hashlib.md5()

    with open(file_path, "rb") as file:

        while True:

            chunk = file.read(
                1024 * 1024
            )

            if not chunk:
                break

            md5.update(chunk)

    return md5.hexdigest()


def verify_file(
    file_path,
    expected_size,
    expected_md5
):

    if not file_path.exists():
        return False

    if (
        file_path.stat().st_size
        != expected_size
    ):
        return False

    actual_md5 = calculate_md5(
        file_path
    )

    return (
        actual_md5.lower()
        == expected_md5.lower()
    )


# ------------------------------------------------------------
# 6. Initialize GDC download session
# ------------------------------------------------------------

session = requests.Session()

session.headers.update({
    "User-Agent": "SurfaceTarget-CRC-Research/1.0"
})


# ------------------------------------------------------------
# 7. Download individual GDC files
# ------------------------------------------------------------

def download_file(
    file_id,
    destination,
    expected_size,
    expected_md5
):

    # Reuse previously verified files.

    if verify_file(
        destination,
        expected_size,
        expected_md5
    ):

        return "Already verified"

    url = f"{GDC_DATA_API}/{file_id}"

    temporary = Path(
        str(destination) + ".part"
    )

    for attempt in range(
        1,
        MAX_ATTEMPTS + 1
    ):

        try:

            if temporary.exists():
                temporary.unlink()

            with session.get(
                url,
                stream=True,
                timeout=(30, 180)
            ) as response:

                response.raise_for_status()

                with open(
                    temporary,
                    "wb"
                ) as output:

                    for chunk in response.iter_content(
                        chunk_size=1024 * 1024
                    ):

                        if chunk:

                            output.write(chunk)

            # Accept only verified downloads.

            if not verify_file(
                temporary,
                expected_size,
                expected_md5
            ):

                raise RuntimeError(
                    "Size or MD5 verification failed."
                )

            os.replace(
                temporary,
                destination
            )

            return "Downloaded and verified"

        except Exception:

            if temporary.exists():
                temporary.unlink()

            if attempt == MAX_ATTEMPTS:
                raise

            time.sleep(
                min(2 ** attempt, 30)
            )


# ------------------------------------------------------------
# 8. Reuse files downloaded during Patch 1.12
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("REUSING PREVIOUSLY VERIFIED FILES")
print("-" * 65)

DUPLICATE_QC_DIR = (
    ROOT / "data/raw/tcga_duplicate_qc"
)

reused_count = 0

for _, row in manifest.iterrows():

    file_id = row["file_id"]

    expected_size = int(
        row["file_size_bytes"]
    )

    expected_md5 = row["md5sum"]

    destination_directory = (
        TUMOR_DIR
        if row["dataset_group"] == "Tumor"
        else NORMAL_DIR
    )

    destination = (
        destination_directory /
        f"{file_id}.tsv"
    )

    previous = (
        DUPLICATE_QC_DIR /
        f"{file_id}.tsv"
    )

    if verify_file(
        destination,
        expected_size,
        expected_md5
    ):
        continue

    if verify_file(
        previous,
        expected_size,
        expected_md5
    ):

        shutil.copy2(
            previous,
            destination
        )

        if not verify_file(
            destination,
            expected_size,
            expected_md5
        ):

            raise RuntimeError(
                "Copied file failed integrity verification."
            )

        reused_count += 1

print(
    "\nPreviously downloaded files reused:",
    reused_count
)


# ------------------------------------------------------------
# 9. Execute full RNA-seq download
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("DOWNLOADING SELECTED RNA-SEQ DATA")
print("-" * 65)

download_records = []

new_downloads = 0

successful = 0

failed = 0

processed = 0


def save_progress(records):

    progress_df = pd.DataFrame(records)

    temporary_log = Path(
        str(DOWNLOAD_LOG) + ".tmp"
    )

    progress_df.to_csv(
        temporary_log,
        index=False
    )

    os.replace(
        temporary_log,
        DOWNLOAD_LOG
    )


for _, row in manifest.iterrows():

    file_id = row["file_id"]

    dataset_group = row["dataset_group"]

    destination_directory = (
        TUMOR_DIR
        if dataset_group == "Tumor"
        else NORMAL_DIR
    )

    destination = (
        destination_directory /
        f"{file_id}.tsv"
    )

    expected_size = int(
        row["file_size_bytes"]
    )

    expected_md5 = row["md5sum"]

    record = {

        "file_id": file_id,

        "project": row["project"],

        "patient_id": row["patient_id"],

        "sample_id": row["sample_id"],

        "dataset_group": dataset_group,

        "local_path": str(destination),

        "expected_size_bytes": expected_size,

        "expected_md5": expected_md5,

        "status": None,

        "error": None
    }

    try:

        # If a new-download limit was specified,
        # retain already verified files but stop
        # before exceeding that limit.

        already_present = verify_file(
            destination,
            expected_size,
            expected_md5
        )

        if (
            MAX_NEW_DOWNLOADS is not None
            and new_downloads >= MAX_NEW_DOWNLOADS
            and not already_present
        ):

            record["status"] = "Pending"

        else:

            status = download_file(
                file_id,
                destination,
                expected_size,
                expected_md5
            )

            record["status"] = status

            successful += 1

            if status == "Downloaded and verified":
                new_downloads += 1

    except Exception as error:

        failed += 1

        record["status"] = "Failed"

        record["error"] = str(error)

        print(
            f"\nDOWNLOAD FAILED: {file_id}"
        )

        print(error)

    download_records.append(record)

    processed += 1

    # Save progress regularly.

    if processed % 25 == 0:

        save_progress(
            download_records
        )

        print(
            f"\nProgress: {processed}/{len(manifest)} "
            f"| Verified: {successful} "
            f"| Failed: {failed}"
        )

# Save final progress.

save_progress(
    download_records
)

print("\nDownload processing finished.")


# ------------------------------------------------------------
# 10. Verify downloaded dataset
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("FINAL DATA INTEGRITY AUDIT")
print("-" * 65)

verification_records = []

for _, row in manifest.iterrows():

    file_id = row["file_id"]

    directory = (
        TUMOR_DIR
        if row["dataset_group"] == "Tumor"
        else NORMAL_DIR
    )

    file_path = (
        directory /
        f"{file_id}.tsv"
    )

    verified = verify_file(
        file_path,
        int(row["file_size_bytes"]),
        row["md5sum"]
    )

    verification_records.append({

        "file_id": file_id,

        "project": row["project"],

        "dataset_group": row["dataset_group"],

        "verified": verified
    })

verification = pd.DataFrame(
    verification_records
)

print("\nOverall verification:")

print(
    verification["verified"]
    .value_counts()
    .to_string()
)

print("\nVerification by dataset:")

verification_summary = (
    verification.groupby(
        ["project", "dataset_group"]
    )
    .agg(
        files=("file_id", "count"),
        verified_files=("verified", "sum")
    )
    .reset_index()
)

print(
    verification_summary.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 11. Save final verification records
# ------------------------------------------------------------

verification_path = (
    META /
    "tcga_rnaseq_download_verification.csv"
)

verification.to_csv(
    verification_path,
    index=False
)

summary_path = (
    RESULTS /
    "tcga_rnaseq_download_summary.csv"
)

verification_summary.to_csv(
    summary_path,
    index=False
)


# ------------------------------------------------------------
# 12. Save research provenance
# ------------------------------------------------------------

all_verified = bool(
    verification["verified"].all()
)

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "1.16",

    "completed_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": "NCI Genomic Data Commons",

    "endpoint": GDC_DATA_API,

    "workflow": "STAR - Counts",

    "manifest_files": len(manifest),

    "tumor_files": len(tumor),

    "normal_files": len(normal),

    "verified_files": int(
        verification["verified"].sum()
    ),

    "failed_download_attempts": failed,

    "all_files_verified": all_verified,

    "integrity_checks": [
        "File size",
        "MD5 checksum"
    ],

    "count_column_for_downstream_analysis":
        "unstranded",

    "expression_matrix_constructed": False,

    "differential_expression_performed": False,

    "limitations": [
        "File integrity does not establish biological quality.",
        "Full-cohort expression QC remains pending.",
        "Workflow version and evidence tier are strongly associated.",
        "Normal samples are not all patient-matched.",
        "Tier B is computationally low-MSI, not categorically verified MSS."
    ]
}

provenance_path = (
    META /
    "tcga_rnaseq_download_provenance.json"
)

with open(provenance_path, "w") as output:

    json.dump(
        provenance,
        output,
        indent=4
    )


# ------------------------------------------------------------
# 13. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.16 SUMMARY")
print("=" * 65)

print(
    "\nTotal manifest files:",
    len(manifest)
)

print(
    "Successfully verified:",
    int(verification["verified"].sum())
)

print(
    "Files not verified:",
    int((~verification["verified"]).sum())
)

print(
    "Download attempts that failed:",
    failed
)

print(
    "\nAll files verified:",
    all_verified
)

print("\nOutput locations:")

print("Tumor:", TUMOR_DIR)

print("Normal:", NORMAL_DIR)

print("\nSaved metadata:")

print("1. tcga_rnaseq_download_log.csv")

print("2. tcga_rnaseq_download_verification.csv")

print("3. tcga_rnaseq_download_summary.csv")

print("4. tcga_rnaseq_download_provenance.json")

if all_verified:

    print("\nALL RNA-SEQ FILES VERIFIED SUCCESSFULLY")

    print("\nPATCH 1.16 COMPLETE")

else:

    print(
        "\nDownload incomplete. "
        "Rerun this patch to retry missing files."
    )

print("=" * 65)

In [ ]:
# ============================================================
# SurfaceTarget-CRC
# PATCH 1.17
#
# TCGA RNA-SEQ GENE EXPRESSION MATRIX CONSTRUCTION
#
# Objectives:
# 1. Load all verified STAR Counts files
# 2. Extract unstranded raw gene counts
# 3. Validate gene identifiers and annotations
# 4. Construct a unified gene-by-sample count matrix
# 5. Preserve sample and patient metadata
# 6. Calculate preliminary expression QC metrics
# 7. Save compressed expression data
#
# IMPORTANT:
# No gene filtering.
# No normalization.
# No differential expression analysis.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import hashlib
import json
import shutil

# ------------------------------------------------------------
# 1. Project configuration
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"

PROCESSED = ROOT / "data/processed"

RESULTS = ROOT / "results/tables"

PROCESSED.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

TUMOR_MANIFEST = (
    META / "tcga_selected_tumor_rnaseq_manifest.csv"
)

NORMAL_MANIFEST = (
    META / "tcga_normal_rnaseq_manifest.csv"
)

VERIFICATION_PATH = (
    META / "tcga_rnaseq_download_verification.csv"
)

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.17 — EXPRESSION MATRIX CONSTRUCTION")
print("=" * 65)


# ------------------------------------------------------------
# 2. Load sample manifests
# ------------------------------------------------------------

tumor = pd.read_csv(
    TUMOR_MANIFEST,
    dtype=str,
    keep_default_na=False
)

normal = pd.read_csv(
    NORMAL_MANIFEST,
    dtype=str,
    keep_default_na=False
)

tumor["dataset_group"] = "Tumor"
normal["dataset_group"] = "Normal"

manifest = pd.concat(
    [tumor, normal],
    ignore_index=True
)

if len(tumor) != 421:
    raise RuntimeError("Unexpected tumor sample count.")

if len(normal) != 51:
    raise RuntimeError("Unexpected normal sample count.")

if manifest["file_id"].duplicated().any():
    raise RuntimeError("Duplicate file IDs detected.")

print("\nSample manifests loaded.")

print("Tumor samples:", len(tumor))
print("Normal samples:", len(normal))
print("Total samples:", len(manifest))


# ------------------------------------------------------------
# 3. Verify previous download status
# ------------------------------------------------------------

verification = pd.read_csv(
    VERIFICATION_PATH,
    dtype=str
)

verified_ids = set(
    verification.loc[
        verification["verified"]
        .astype(str)
        .str.lower()
        .eq("true"),
        "file_id"
    ]
)

manifest_ids = set(manifest["file_id"])

if verified_ids != manifest_ids:
    raise RuntimeError(
        "Download verification records do not match "
        "the current manifest."
    )

print("\nDownload verification records: PASSED")


# ------------------------------------------------------------
# 4. Define STAR summary rows
# ------------------------------------------------------------

STAR_SUMMARY = {
    "N_unmapped",
    "N_multimapping",
    "N_noFeature",
    "N_ambiguous"
}

REQUIRED_COLUMNS = {
    "gene_id",
    "gene_name",
    "gene_type",
    "unstranded"
}


# ------------------------------------------------------------
# 5. STAR Counts file parser
# ------------------------------------------------------------

def load_gene_counts(file_path):

    df = pd.read_csv(
        file_path,
        sep="\t",
        comment="#",
        low_memory=False,
        dtype={"gene_id": str}
    )

    missing = REQUIRED_COLUMNS - set(df.columns)

    if missing:
        raise RuntimeError(
            f"Missing required columns in {file_path}: "
            f"{missing}"
        )

    summary = df[
        df["gene_id"].isin(STAR_SUMMARY)
    ].copy()

    genes = df[
        ~df["gene_id"].isin(STAR_SUMMARY)
    ].copy()

    if set(summary["gene_id"]) != STAR_SUMMARY:
        raise RuntimeError(
            f"Incomplete STAR summary: {file_path}"
        )

    if len(summary) != 4:
        raise RuntimeError(
            f"Unexpected STAR summary rows: {file_path}"
        )

    if genes.empty:
        raise RuntimeError(
            f"No gene counts found: {file_path}"
        )

    if genes["gene_id"].isna().any():
        raise RuntimeError(
            f"Missing gene IDs: {file_path}"
        )

    if genes["gene_id"].duplicated().any():
        raise RuntimeError(
            f"Duplicate gene IDs: {file_path}"
        )

    counts = pd.to_numeric(
        genes["unstranded"],
        errors="raise"
    )

    if counts.isna().any():
        raise RuntimeError(
            f"Missing counts: {file_path}"
        )

    if (counts < 0).any():
        raise RuntimeError(
            f"Negative counts: {file_path}"
        )

    if not np.all(
        counts.to_numpy() == np.floor(counts.to_numpy())
    ):
        raise RuntimeError(
            f"Non-integer raw counts: {file_path}"
        )

    genes["unstranded"] = counts.astype(np.int64)

    return genes, summary


# ------------------------------------------------------------
# 6. Initialize reference gene annotations
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("INITIALIZING GENE REFERENCE")
print("-" * 65)

first_file = Path(
    manifest.iloc[0]["local_path"]
)

if not first_file.exists():
    raise FileNotFoundError(first_file)

reference_genes, _ = load_gene_counts(first_file)

gene_metadata = reference_genes[
    ["gene_id", "gene_name", "gene_type"]
].copy()

gene_metadata = gene_metadata.fillna("")

reference_ids = (
    gene_metadata["gene_id"]
    .astype(str)
    .to_numpy()
)

reference_gene_types = (
    gene_metadata["gene_type"]
    .astype(str)
    .to_numpy()
)

reference_index = pd.Index(reference_ids)

reference_gene_set = set(reference_ids)

n_genes = len(reference_ids)
n_samples = len(manifest)

print("\nReference file:", first_file.name)

print("Total genes:", n_genes)

print(
    "Protein-coding genes:",
    int(
        (
            gene_metadata["gene_type"]
            == "protein_coding"
        ).sum()
    )
)

print("\nGene reference initialized.")


# ------------------------------------------------------------
# 7. Allocate expression matrix
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("ALLOCATING EXPRESSION MATRIX")
print("-" * 65)

# Rows = genes
# Columns = samples
#
# int64 preserves integer count values.

count_matrix = np.empty(
    (n_genes, n_samples),
    dtype=np.int64
)

matrix_memory_mb = (
    count_matrix.nbytes / (1024 ** 2)
)

print(
    f"\nMatrix dimensions: "
    f"{n_genes:,} genes x {n_samples:,} samples"
)

print(
    f"Allocated matrix memory: "
    f"{matrix_memory_mb:.2f} MiB"
)


# ------------------------------------------------------------
# 8. Process all RNA-seq files
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CONSTRUCTING GENE EXPRESSION MATRIX")
print("-" * 65)

qc_records = []

reordered_files = 0

for column_index, row in manifest.iterrows():

    file_id = row["file_id"]

    file_path = Path(row["local_path"])

    if not file_path.exists():
        raise FileNotFoundError(
            f"Missing RNA-seq file: {file_path}"
        )

    expected_size = int(row["file_size_bytes"])

    if file_path.stat().st_size != expected_size:
        raise RuntimeError(
            f"File size changed: {file_id}"
        )

    genes, summary = load_gene_counts(file_path)

    # --------------------------------------------
    # Validate gene identifiers
    # --------------------------------------------

    current_ids = (
        genes["gene_id"]
        .astype(str)
        .to_numpy()
    )

    if len(current_ids) != n_genes:
        raise RuntimeError(
            f"Gene count mismatch: {file_id}"
        )

    if not np.array_equal(
        current_ids,
        reference_ids
    ):

        if set(current_ids) != reference_gene_set:
            raise RuntimeError(
                f"Gene identifier mismatch: {file_id}"
            )

        # Same genes, different row order.
        # Reorder to the reference gene order.

        genes = (
            genes.set_index("gene_id")
            .loc[reference_index]
            .reset_index()
        )

        reordered_files += 1

    # --------------------------------------------
    # Validate gene-type annotations
    # --------------------------------------------

    current_types = (
        genes["gene_type"]
        .fillna("")
        .astype(str)
        .to_numpy()
    )

    if not np.array_equal(
        current_types,
        reference_gene_types
    ):
        raise RuntimeError(
            f"Gene-type annotation mismatch: {file_id}"
        )

    # --------------------------------------------
    # Store raw counts
    # --------------------------------------------

    counts = (
        genes["unstranded"]
        .to_numpy(dtype=np.int64)
    )

    count_matrix[:, column_index] = counts

    # --------------------------------------------
    # Calculate preliminary QC metrics
    # --------------------------------------------

    assigned_counts = int(counts.sum())

    detected_genes = int(
        np.count_nonzero(counts)
    )

    detected_protein_coding = int(
        np.count_nonzero(
            counts[
                reference_gene_types == "protein_coding"
            ]
        )
    )

    summary_values = dict(
        zip(
            summary["gene_id"],
            pd.to_numeric(
                summary["unstranded"],
                errors="raise"
            )
        )
    )

    total_counted = (
        assigned_counts
        + sum(summary_values.values())
    )

    assignment_fraction = (
        assigned_counts / total_counted
        if total_counted > 0
        else np.nan
    )

    qc_records.append({

        "file_id": file_id,

        "patient_id": row["patient_id"],

        "sample_id": row["sample_id"],

        "project": row["project"],

        "dataset_group": row["dataset_group"],

        "cohort_tier": row["cohort_tier"],

        "workflow_version": row["workflow_version"],

        "assigned_gene_counts": assigned_counts,

        "detected_genes": detected_genes,

        "detected_protein_coding":
            detected_protein_coding,

        "assignment_fraction":
            assignment_fraction,

        "N_unmapped":
            int(summary_values["N_unmapped"]),

        "N_multimapping":
            int(summary_values["N_multimapping"]),

        "N_noFeature":
            int(summary_values["N_noFeature"]),

        "N_ambiguous":
            int(summary_values["N_ambiguous"])
    })

    if (column_index + 1) % 25 == 0:

        print(
            f"Processed {column_index + 1} "
            f"of {n_samples} files"
        )

print(
    f"\nProcessed {n_samples} of {n_samples} files"
)

print("\nMatrix construction complete.")


# ------------------------------------------------------------
# 9. Validate expression matrix
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("EXPRESSION MATRIX VALIDATION")
print("-" * 65)

if count_matrix.shape != (
    n_genes,
    n_samples
):
    raise RuntimeError(
        "Expression matrix dimensions incorrect."
    )

if np.any(count_matrix < 0):
    raise RuntimeError(
        "Negative values detected in count matrix."
    )

if len(qc_records) != n_samples:
    raise RuntimeError(
        "QC record count mismatch."
    )

# Confirm that matrix column sums match
# independently recorded library sizes.

qc = pd.DataFrame(qc_records)

matrix_library_sizes = (
    count_matrix.sum(axis=0)
)

recorded_library_sizes = (
    qc["assigned_gene_counts"]
    .to_numpy(dtype=np.int64)
)

if not np.array_equal(
    matrix_library_sizes,
    recorded_library_sizes
):
    raise RuntimeError(
        "Library-size validation failed."
    )

print("\nMatrix dimensions: PASSED")

print("Nonnegative raw counts: PASSED")

print("Library-size consistency: PASSED")

print("Sample metadata correspondence: PASSED")

print(
    "Files requiring gene-order correction:",
    reordered_files
)


# ------------------------------------------------------------
# 10. Preliminary expression QC summary
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PRELIMINARY EXPRESSION QC")
print("-" * 65)

qc["library_size_millions"] = (
    qc["assigned_gene_counts"] / 1e6
)

qc["assignment_percent"] = (
    qc["assignment_fraction"] * 100
)

qc_summary = (
    qc.groupby(
        [
            "project",
            "dataset_group",
            "cohort_tier"
        ]
    )
    .agg(
        samples=("file_id", "count"),

        median_library_size_millions=(
            "library_size_millions",
            "median"
        ),

        median_detected_genes=(
            "detected_genes",
            "median"
        ),

        median_detected_protein_coding=(
            "detected_protein_coding",
            "median"
        ),

        median_assignment_percent=(
            "assignment_percent",
            "median"
        )
    )
    .reset_index()
)

print("\n")
print(
    qc_summary.round(2).to_string(index=False)
)


# ------------------------------------------------------------
# 11. Save gene annotation metadata
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING PROCESSED EXPRESSION DATA")
print("-" * 65)

GENE_METADATA_PATH = (
    META / "tcga_gene_annotations.csv"
)

SAMPLE_METADATA_PATH = (
    META / "tcga_expression_sample_metadata.csv"
)

QC_PATH = (
    RESULTS / "tcga_expression_initial_qc.csv"
)

QC_SUMMARY_PATH = (
    RESULTS / "tcga_expression_initial_qc_summary.csv"
)

gene_metadata.to_csv(
    GENE_METADATA_PATH,
    index=False
)

# Matrix column order is exactly the
# order of rows in this sample metadata file.

sample_metadata = manifest.copy()

sample_metadata["matrix_column_index"] = (
    np.arange(n_samples)
)

sample_metadata.to_csv(
    SAMPLE_METADATA_PATH,
    index=False
)

qc.to_csv(
    QC_PATH,
    index=False
)

qc_summary.to_csv(
    QC_SUMMARY_PATH,
    index=False
)


# ------------------------------------------------------------
# 12. Save compressed raw-count matrix
# ------------------------------------------------------------

MATRIX_PATH = (
    PROCESSED /
    "tcga_unstranded_raw_counts.npz"
)

np.savez_compressed(

    MATRIX_PATH,

    counts=count_matrix,

    gene_ids=reference_ids,

    file_ids=manifest["file_id"].to_numpy()
)

print("\nCompressed count matrix saved.")


# ------------------------------------------------------------
# 13. Verify saved matrix
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVED MATRIX VERIFICATION")
print("-" * 65)

with np.load(
    MATRIX_PATH,
    allow_pickle=False
) as saved:

    saved_shape = saved["counts"].shape

    saved_gene_ids = saved["gene_ids"]

    saved_file_ids = saved["file_ids"]

    first_column_matches = np.array_equal(
        saved["counts"][:, 0],
        count_matrix[:, 0]
    )

if saved_shape != count_matrix.shape:
    raise RuntimeError(
        "Saved matrix dimensions are incorrect."
    )

if not np.array_equal(
    saved_gene_ids,
    reference_ids
):
    raise RuntimeError(
        "Saved gene identifiers do not match."
    )

if not np.array_equal(
    saved_file_ids,
    manifest["file_id"].to_numpy()
):
    raise RuntimeError(
        "Saved sample identifiers do not match."
    )

if not first_column_matches:
    raise RuntimeError(
        "Saved count values failed verification."
    )

print("\nSaved matrix verification: PASSED")


# ------------------------------------------------------------
# 14. Calculate file checksum
# ------------------------------------------------------------

def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            digest.update(chunk)

    return digest.hexdigest()


matrix_sha256 = sha256_file(
    MATRIX_PATH
)

matrix_size_mb = (
    MATRIX_PATH.stat().st_size
    / (1024 ** 2)
)


# ------------------------------------------------------------
# 15. Save processing provenance
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "1.17",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": "NCI Genomic Data Commons",

    "workflow": "STAR - Counts",

    "count_column": "unstranded",

    "matrix_orientation": "genes_by_samples",

    "n_genes": int(n_genes),

    "n_samples": int(n_samples),

    "n_tumor_samples": int(len(tumor)),

    "n_normal_samples": int(len(normal)),

    "data_type": "int64",

    "gene_filtering_performed": False,

    "normalization_performed": False,

    "differential_expression_performed": False,

    "files_reordered_to_reference": int(
        reordered_files
    ),

    "matrix_file": str(MATRIX_PATH),

    "matrix_sha256": matrix_sha256,

    "matrix_size_mb": float(matrix_size_mb),

    "validation": [
        "Gene identifier consistency",
        "Gene-type annotation consistency",
        "Nonnegative integer counts",
        "Matrix dimensions",
        "Library-size consistency",
        "Saved matrix identifiers",
        "Saved matrix first-column values"
    ],

    "limitations": [
        "Full-cohort quality control remains pending.",
        "Workflow-version confounding remains unresolved.",
        "No biological conclusions have been drawn.",
        "Normal samples are not all paired with tumor samples.",
        "Tier B is computationally low-MSI, not confirmed MSS."
    ]
}

PROVENANCE_PATH = (
    META /
    "tcga_expression_matrix_provenance.json"
)

with open(
    PROVENANCE_PATH,
    "w"
) as f:

    json.dump(
        provenance,
        f,
        indent=4
    )


# ------------------------------------------------------------
# 16. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.17 SUMMARY")
print("=" * 65)

print("\nExpression matrix dimensions:")

print(
    f"{n_genes:,} genes x {n_samples:,} samples"
)

print("\nTumor samples:", len(tumor))

print("Normal samples:", len(normal))

print(
    "\nCompressed matrix size:",
    f"{matrix_size_mb:.2f} MiB"
)

print("\nSaved files:")

print("1. tcga_unstranded_raw_counts.npz")

print("2. tcga_gene_annotations.csv")

print("3. tcga_expression_sample_metadata.csv")

print("4. tcga_expression_initial_qc.csv")

print("5. tcga_expression_initial_qc_summary.csv")

print("6. tcga_expression_matrix_provenance.json")

print("\nAll matrix validation checks passed.")

print("\nNo normalization performed.")

print("No genes or samples excluded.")

print("\nPATCH 1.17 COMPLETE")

print("=" * 65)

In [ ]:
# ============================================================
# SurfaceTarget-CRC
# PATCH 1.17 — MATRIX SERIALIZATION REPAIR
#
# Fix:
# Convert gene IDs and file IDs to Unicode string arrays.
#
# IMPORTANT:
# Does NOT reprocess the 472 RNA-seq files.
# Does NOT download additional data.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import hashlib
import json
import os

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.17 — MATRIX SERIALIZATION REPAIR")
print("=" * 65)

# ------------------------------------------------------------
# 1. File paths
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"
PROCESSED = ROOT / "data/processed"
RESULTS = ROOT / "results/tables"

MATRIX_PATH = (
    PROCESSED /
    "tcga_unstranded_raw_counts.npz"
)

REPAIRED_PATH = (
    PROCESSED /
    "tcga_unstranded_raw_counts.repaired.npz"
)

GENES_PATH = META / "tcga_gene_annotations.csv"

SAMPLES_PATH = (
    META / "tcga_expression_sample_metadata.csv"
)

QC_PATH = (
    RESULTS / "tcga_expression_initial_qc.csv"
)

PROVENANCE_PATH = (
    META / "tcga_expression_matrix_provenance.json"
)

# ------------------------------------------------------------
# 2. Validate required files
# ------------------------------------------------------------

required_files = [
    MATRIX_PATH,
    GENES_PATH,
    SAMPLES_PATH,
    QC_PATH
]

for path in required_files:

    if not path.exists():
        raise FileNotFoundError(
            f"Required file missing: {path}"
        )

print("\nAll required files found.")

# ------------------------------------------------------------
# 3. Recover existing raw-count matrix
# ------------------------------------------------------------

print("\nRecovering previously saved count matrix...")

with np.load(
    MATRIX_PATH,
    allow_pickle=False
) as archive:

    # Only load numeric counts.
    # The original object-based identifiers
    # are deliberately not loaded.

    counts = archive["counts"]

print("\nCount matrix recovered successfully.")

print("Dimensions:", counts.shape)

print("Data type:", counts.dtype)

if counts.shape != (60660, 472):
    raise RuntimeError(
        "Unexpected count matrix dimensions."
    )

if not np.issubdtype(counts.dtype, np.integer):
    raise RuntimeError(
        "Raw-count matrix is not integer-valued."
    )

if np.any(counts < 0):
    raise RuntimeError(
        "Negative count values detected."
    )

# ------------------------------------------------------------
# 4. Recover gene and sample identifiers
# ------------------------------------------------------------

genes = pd.read_csv(
    GENES_PATH,
    dtype=str,
    keep_default_na=False
)

samples = pd.read_csv(
    SAMPLES_PATH,
    dtype=str,
    keep_default_na=False
)

qc = pd.read_csv(QC_PATH)

# Explicit Unicode conversion fixes
# the object-array serialization problem.

gene_ids = np.asarray(
    genes["gene_id"].tolist(),
    dtype=np.str_
)

file_ids = np.asarray(
    samples["file_id"].tolist(),
    dtype=np.str_
)

if len(gene_ids) != counts.shape[0]:
    raise RuntimeError(
        "Gene identifier count mismatch."
    )

if len(file_ids) != counts.shape[1]:
    raise RuntimeError(
        "Sample identifier count mismatch."
    )

if len(set(gene_ids)) != len(gene_ids):
    raise RuntimeError(
        "Duplicate gene identifiers detected."
    )

if len(set(file_ids)) != len(file_ids):
    raise RuntimeError(
        "Duplicate file identifiers detected."
    )

# Ensure sample metadata preserves matrix order.

column_indices = pd.to_numeric(
    samples["matrix_column_index"],
    errors="raise"
).to_numpy(dtype=np.int64)

if not np.array_equal(
    column_indices,
    np.arange(len(samples))
):
    raise RuntimeError(
        "Sample column order is inconsistent."
    )

# Compare with recorded QC library sizes.

if not np.array_equal(
    qc["file_id"].astype(str).to_numpy(),
    file_ids
):
    raise RuntimeError(
        "QC sample order does not match the matrix."
    )

expected_library_sizes = pd.to_numeric(
    qc["assigned_gene_counts"],
    errors="raise"
).to_numpy(dtype=np.int64)

actual_library_sizes = counts.sum(axis=0)

if not np.array_equal(
    expected_library_sizes,
    actual_library_sizes
):
    raise RuntimeError(
        "Library-size validation failed."
    )

print("\nGene identifier validation: PASSED")
print("Sample identifier validation: PASSED")
print("Sample column order: PASSED")
print("Library-size validation: PASSED")

# ------------------------------------------------------------
# 5. Save corrected archive
# ------------------------------------------------------------

print("\nSaving repaired expression matrix...")

np.savez_compressed(
    REPAIRED_PATH,
    counts=counts,
    gene_ids=gene_ids,
    file_ids=file_ids
)

# ------------------------------------------------------------
# 6. Verify repaired archive
# ------------------------------------------------------------

print("\nVerifying repaired matrix...")

with np.load(
    REPAIRED_PATH,
    allow_pickle=False
) as saved:

    saved_counts = saved["counts"]

    saved_gene_ids = saved["gene_ids"]

    saved_file_ids = saved["file_ids"]

    if not np.array_equal(
        saved_counts,
        counts
    ):
        raise RuntimeError(
            "Saved count values do not match."
        )

    if not np.array_equal(
        saved_gene_ids,
        gene_ids
    ):
        raise RuntimeError(
            "Saved gene identifiers do not match."
        )

    if not np.array_equal(
        saved_file_ids,
        file_ids
    ):
        raise RuntimeError(
            "Saved sample identifiers do not match."
        )

    if saved_gene_ids.dtype.kind != "U":
        raise RuntimeError(
            "Gene IDs were not saved as Unicode strings."
        )

    if saved_file_ids.dtype.kind != "U":
        raise RuntimeError(
            "File IDs were not saved as Unicode strings."
        )

print("\nRepaired matrix verification: PASSED")

# ------------------------------------------------------------
# 7. Replace original archive
# ------------------------------------------------------------

# Replace only after all verification checks pass.

os.replace(
    REPAIRED_PATH,
    MATRIX_PATH
)

print("\nOriginal archive replaced with verified version.")

# ------------------------------------------------------------
# 8. Calculate SHA256 checksum
# ------------------------------------------------------------

sha256 = hashlib.sha256()

with open(MATRIX_PATH, "rb") as f:

    for chunk in iter(
        lambda: f.read(1024 * 1024),
        b""
    ):

        sha256.update(chunk)

matrix_sha256 = sha256.hexdigest()

matrix_size_mib = (
    MATRIX_PATH.stat().st_size
    / (1024 ** 2)
)

# ------------------------------------------------------------
# 9. Save processing provenance
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "1.17",

    "repair": "Unicode identifier serialization",

    "completed_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": "NCI Genomic Data Commons",

    "workflow": "STAR - Counts",

    "count_column": "unstranded",

    "matrix_orientation": "genes_by_samples",

    "n_genes": int(len(gene_ids)),

    "n_samples": int(len(file_ids)),

    "n_tumor_samples": 421,

    "n_normal_samples": 51,

    "data_type": str(counts.dtype),

    "identifier_data_type": "Unicode strings",

    "gene_filtering_performed": False,

    "normalization_performed": False,

    "differential_expression_performed": False,

    "matrix_file": str(MATRIX_PATH),

    "matrix_sha256": matrix_sha256,

    "matrix_size_mib": float(matrix_size_mib),

    "validation": [
        "Gene identifier count",
        "Sample identifier count",
        "Unique gene identifiers",
        "Unique sample identifiers",
        "Sample column order",
        "Raw integer counts",
        "Nonnegative counts",
        "All sample library sizes",
        "Complete saved matrix equality",
        "Saved gene identifier equality",
        "Saved file identifier equality",
        "Unicode identifier serialization"
    ],

    "limitations": [
        "Full-cohort expression QC remains pending.",
        "Workflow-version confounding remains unresolved.",
        "Normal tissue controls are not all patient-matched.",
        "Tier B is computationally low-MSI, not confirmed MSS.",
        "No biological conclusions have been drawn."
    ]
}

with open(PROVENANCE_PATH, "w") as f:

    json.dump(
        provenance,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 10. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.17 — FINAL SUMMARY")
print("=" * 65)

print("\nExpression matrix:")

print(
    f"{len(gene_ids):,} genes x "
    f"{len(file_ids):,} samples"
)

print("\nTumor samples: 421")
print("Normal samples: 51")

print(
    f"\nCompressed matrix size: "
    f"{matrix_size_mib:.2f} MiB"
)

print("\nSaved matrix verification: PASSED")

print("Gene identifiers: VERIFIED")

print("Sample identifiers: VERIFIED")

print("Library sizes: VERIFIED")

print("Raw count values: VERIFIED")

print("\nProvenance: SAVED")

print("\nNo RNA-seq files reprocessed.")

print("No additional data downloaded.")

print("\nPATCH 1.17 COMPLETE")

print("=" * 65)

In [ ]:
# ============================================================
# SurfaceTarget-CRC
# PHASE 1 — PATCH 1.18
#
# FULL-COHORT RNA-SEQ QUALITY CONTROL
#
# Objectives:
# 1. Validate processed expression matrix
# 2. Examine library sizes and gene detection
# 3. Calculate protein-coding detection at CPM thresholds
# 4. Analyze gene-assignment statistics
# 5. Stratify QC by project, cohort and workflow
# 6. Flag unusual samples using robust statistics
# 7. Generate publication-oriented QC figures
# 8. Save results and a processed-data checkpoint
#
# IMPORTANT:
# - No samples will be excluded.
# - No differential expression is performed.
# - CPM is used for descriptive QC only.
# - Raw counts remain unchanged.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import hashlib
import json


# ------------------------------------------------------------
# 1. Project configuration
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"

PROCESSED = ROOT / "data/processed"

RESULTS = ROOT / "results/tables"

FIGURES = ROOT / "results/figures"

CHECKPOINTS = ROOT / "checkpoints"

for directory in [
    META,
    PROCESSED,
    RESULTS,
    FIGURES,
    CHECKPOINTS
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

MATRIX_PATH = (
    PROCESSED / "tcga_unstranded_raw_counts.npz"
)

GENES_PATH = (
    META / "tcga_gene_annotations.csv"
)

SAMPLES_PATH = (
    META / "tcga_expression_sample_metadata.csv"
)

INITIAL_QC_PATH = (
    RESULTS / "tcga_expression_initial_qc.csv"
)

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.18 — FULL-COHORT RNA-SEQ QC")
print("=" * 65)


# ------------------------------------------------------------
# 2. Load processed expression matrix
# ------------------------------------------------------------

required_files = [
    MATRIX_PATH,
    GENES_PATH,
    SAMPLES_PATH,
    INITIAL_QC_PATH
]

for path in required_files:

    if not path.exists():

        raise FileNotFoundError(
            f"Required file missing: {path}"
        )

with np.load(
    MATRIX_PATH,
    allow_pickle=False
) as archive:

    counts = archive["counts"]

    gene_ids = archive["gene_ids"]

    file_ids = archive["file_ids"]

genes = pd.read_csv(
    GENES_PATH,
    dtype=str,
    keep_default_na=False
)

samples = pd.read_csv(
    SAMPLES_PATH,
    dtype=str,
    keep_default_na=False
)

initial_qc = pd.read_csv(
    INITIAL_QC_PATH,
    dtype={"file_id": str}
)

print("\nExpression matrix loaded successfully.")

print("Matrix dimensions:", counts.shape)

print("Gene annotations:", len(genes))

print("Sample records:", len(samples))


# ------------------------------------------------------------
# 3. Validate matrix consistency
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("MATRIX INTEGRITY VALIDATION")
print("-" * 65)

if counts.shape != (60660, 472):

    raise RuntimeError(
        "Unexpected expression matrix dimensions."
    )

if not np.issubdtype(
    counts.dtype,
    np.integer
):

    raise RuntimeError(
        "Expression matrix contains non-integer values."
    )

if np.any(counts < 0):

    raise RuntimeError(
        "Negative raw counts detected."
    )

if not np.array_equal(
    gene_ids,
    genes["gene_id"].to_numpy()
):

    raise RuntimeError(
        "Gene identifier mismatch."
    )

if not np.array_equal(
    file_ids,
    samples["file_id"].to_numpy()
):

    raise RuntimeError(
        "Sample identifier mismatch."
    )

if not np.array_equal(
    file_ids,
    initial_qc["file_id"].to_numpy()
):

    raise RuntimeError(
        "Initial QC sample order mismatch."
    )

if samples["file_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate sample identifiers detected."
    )

required_sample_columns = [
    "file_id",
    "patient_id",
    "sample_id",
    "project",
    "dataset_group",
    "cohort_tier",
    "workflow_version"
]

missing_columns = [
    col for col in required_sample_columns
    if col not in samples.columns
]

if missing_columns:

    raise RuntimeError(
        f"Missing sample metadata: {missing_columns}"
    )

print("\nMatrix integrity: PASSED")

print("Gene annotation correspondence: PASSED")

print("Sample metadata correspondence: PASSED")


# ------------------------------------------------------------
# 4. Identify protein-coding genes
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENE ANNOTATION SUMMARY")
print("-" * 65)

protein_coding_mask = (
    genes["gene_type"]
    .eq("protein_coding")
    .to_numpy(dtype=bool)
)

n_protein_coding = int(
    protein_coding_mask.sum()
)

print("\nTotal genes:", len(genes))

print(
    "Protein-coding genes:",
    n_protein_coding
)

if n_protein_coding != 19962:

    raise RuntimeError(
        "Unexpected protein-coding gene count."
    )


# ------------------------------------------------------------
# 5. Calculate expression QC metrics
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CALCULATING FULL-COHORT EXPRESSION QC")
print("-" * 65)

# Raw library sizes:
# Total unstranded counts assigned to genes.

library_sizes = counts.sum(
    axis=0,
    dtype=np.int64
)

if np.any(library_sizes <= 0):

    raise RuntimeError(
        "One or more samples have zero assigned counts."
    )

# Validate against previously saved measurements.

recorded_library_sizes = pd.to_numeric(
    initial_qc["assigned_gene_counts"],
    errors="raise"
).to_numpy(dtype=np.int64)

if not np.array_equal(
    library_sizes,
    recorded_library_sizes
):

    raise RuntimeError(
        "Library sizes differ from Patch 1.17."
    )

# All genes with at least one assigned count.

detected_all = np.count_nonzero(
    counts,
    axis=0
)

# Protein-coding counts.

coding_counts = counts[
    protein_coding_mask,
    :
]

detected_coding = np.count_nonzero(
    coding_counts,
    axis=0
)

# Descriptive CPM calculations.
#
# Denominator = all assigned unstranded gene counts.
#
# This does not replace downstream
# count-based statistical normalization.

cpm_scaling = (
    1_000_000.0
    / library_sizes.astype(np.float64)
)

coding_cpm = (
    coding_counts
    * cpm_scaling[np.newaxis, :]
)

coding_cpm1 = np.count_nonzero(
    coding_cpm >= 1,
    axis=0
)

coding_cpm5 = np.count_nonzero(
    coding_cpm >= 5,
    axis=0
)

coding_assigned_counts = coding_counts.sum(
    axis=0,
    dtype=np.int64
)

coding_count_fraction = (
    coding_assigned_counts
    / library_sizes
)

print("\nGene detection metrics calculated.")

print("CPM >= 1 detection: COMPLETE")

print("CPM >= 5 detection: COMPLETE")


# ------------------------------------------------------------
# 6. Prepare sample-level QC table
# ------------------------------------------------------------

qc = samples[
    required_sample_columns
].copy()

qc["assigned_gene_counts"] = (
    library_sizes
)

qc["library_size_millions"] = (
    library_sizes / 1e6
)

qc["detected_genes"] = (
    detected_all
)

qc["detected_protein_coding"] = (
    detected_coding
)

qc["protein_coding_cpm_ge1"] = (
    coding_cpm1
)

qc["protein_coding_cpm_ge5"] = (
    coding_cpm5
)

qc["protein_coding_count_fraction"] = (
    coding_count_fraction
)

# STAR summary counts from Patch 1.17.

star_columns = [
    "N_unmapped",
    "N_multimapping",
    "N_noFeature",
    "N_ambiguous"
]

for column in star_columns:

    qc[column] = pd.to_numeric(
        initial_qc[column],
        errors="raise"
    ).to_numpy(dtype=np.int64)

star_total = (
    library_sizes
    + qc[star_columns].sum(
        axis=1
    ).to_numpy(dtype=np.int64)
)

if np.any(star_total <= 0):

    raise RuntimeError(
        "Invalid STAR counting totals."
    )

qc["assignment_percent"] = (
    100 * library_sizes / star_total
)

qc["no_feature_percent"] = (
    100 * qc["N_noFeature"]
    / star_total
)

qc["ambiguous_percent"] = (
    100 * qc["N_ambiguous"]
    / star_total
)

qc["multimapping_percent"] = (
    100 * qc["N_multimapping"]
    / star_total
)

qc["unmapped_percent"] = (
    100 * qc["N_unmapped"]
    / star_total
)

# Workflow identifiers are categorical.
# A hexadecimal workflow ID does not,
# by itself, establish chronological order.

qc["workflow_short"] = (
    qc["workflow_version"].str[:12]
)

qc["analysis_group"] = (
    qc["project"]
    + " | "
    + np.where(
        qc["dataset_group"].eq("Normal"),
        "Normal",
        qc["cohort_tier"]
    )
)

print("\nSample-level QC table constructed.")

print("Samples:", len(qc))


# ------------------------------------------------------------
# 7. Validate cohort composition
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("COHORT COMPOSITION VALIDATION")
print("-" * 65)

expected_groups = {

    "TCGA-COAD | Tier A": 98,

    "TCGA-COAD | Tier B": 194,

    "TCGA-COAD | Normal": 41,

    "TCGA-READ | Tier A": 46,

    "TCGA-READ | Tier B": 83,

    "TCGA-READ | Normal": 10
}

group_order = list(
    expected_groups.keys()
)

observed_groups = (
    qc["analysis_group"]
    .value_counts()
    .to_dict()
)

if observed_groups != expected_groups:

    raise RuntimeError(
        "Cohort composition differs from expected values."
    )

print("\nAll six analysis groups validated.")

print(
    qc["analysis_group"]
    .value_counts()
    .reindex(group_order)
    .to_string()
)


# ------------------------------------------------------------
# 8. Full-cohort QC summary
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("FULL-COHORT QUALITY SUMMARY")
print("-" * 65)

qc_summary = (
    qc.groupby("analysis_group")
    .agg(

        samples=(
            "file_id",
            "count"
        ),

        median_library_millions=(
            "library_size_millions",
            "median"
        ),

        median_detected_genes=(
            "detected_genes",
            "median"
        ),

        median_protein_coding_cpm1=(
            "protein_coding_cpm_ge1",
            "median"
        ),

        median_assignment_percent=(
            "assignment_percent",
            "median"
        ),

        median_no_feature_percent=(
            "no_feature_percent",
            "median"
        )

    )
    .reindex(group_order)
    .reset_index()
)

print("\n")
print(
    qc_summary
    .round(2)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 9. Library-size distribution
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LIBRARY-SIZE DISTRIBUTION")
print("-" * 65)

library_summary = (
    qc.groupby("analysis_group")[
        "library_size_millions"
    ]
    .agg(
        [
            "min",
            "median",
            "max"
        ]
    )
    .reindex(group_order)
)

print("\n")
print(
    library_summary
    .round(2)
    .to_string()
)


# ------------------------------------------------------------
# 10. Workflow-stratified quality assessment
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("QUALITY METRICS BY WORKFLOW VERSION")
print("-" * 65)

workflow_qc = (
    qc.groupby(
        [
            "analysis_group",
            "workflow_short"
        ]
    )
    .agg(

        samples=(
            "file_id",
            "count"
        ),

        median_library_millions=(
            "library_size_millions",
            "median"
        ),

        median_coding_cpm1=(
            "protein_coding_cpm_ge1",
            "median"
        ),

        median_assignment_percent=(
            "assignment_percent",
            "median"
        ),

        median_no_feature_percent=(
            "no_feature_percent",
            "median"
        )

    )
    .reset_index()
)

workflow_qc["analysis_group"] = (
    pd.Categorical(
        workflow_qc["analysis_group"],
        categories=group_order,
        ordered=True
    )
)

workflow_qc = (
    workflow_qc
    .sort_values(
        [
            "analysis_group",
            "workflow_short"
        ]
    )
)

print("\n")
print(
    workflow_qc
    .round(2)
    .to_string(index=False)
)

print("\nIMPORTANT:")
print(
    "Workflow and cohort associations are "
    "descriptive. These tables do not "
    "establish a causal technical effect."
)


# ------------------------------------------------------------
# 11. Robust sample outlier screening
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("ROBUST SAMPLE OUTLIER SCREENING")
print("-" * 65)

# Compare samples only within their own
# project and cohort/tissue group.
#
# Robust z-score = 0.67448975 * (x - median) / MAD
#
# |z| > 3.5 is used here only as a
# provisional review threshold.
#
# No samples will be removed.

qc["log10_library_size"] = np.log10(
    qc["assigned_gene_counts"]
)

metrics_to_screen = {

    "log10_library_size": "low",

    "protein_coding_cpm_ge1": "low",

    "assignment_percent": "low",

    "no_feature_percent": "high"
}

flag_reasons = {
    index: []
    for index in qc.index
}

flag_records = []

for group_name, group in qc.groupby(
    "analysis_group"
):

    group_indices = group.index.to_numpy()

    if len(group) < 10:
        continue

    for metric, direction in metrics_to_screen.items():

        values = group[
            metric
        ].to_numpy(dtype=float)

        median = np.median(values)

        mad = np.median(
            np.abs(values - median)
        )

        if not np.isfinite(mad) or mad == 0:

            # Cannot calculate a meaningful
            # robust score with zero MAD.

            continue

        robust_z = (
            0.67448975
            * (values - median)
            / mad
        )

        if direction == "low":

            flagged = robust_z < -3.5

        else:

            flagged = robust_z > 3.5

        for local_position in np.flatnonzero(flagged):

            row_index = group_indices[
                local_position
            ]

            reason = (
                f"{metric}: "
                f"robust_z={robust_z[local_position]:.2f}"
            )

            flag_reasons[
                row_index
            ].append(reason)

            flag_records.append({

                "file_id":
                    qc.loc[row_index, "file_id"],

                "patient_id":
                    qc.loc[row_index, "patient_id"],

                "analysis_group":
                    group_name,

                "metric":
                    metric,

                "observed_value":
                    float(values[local_position]),

                "group_median":
                    float(median),

                "robust_z":
                    float(
                        robust_z[local_position]
                    )
            })

qc["qc_review_flags"] = [
    "; ".join(
        flag_reasons[index]
    )
    for index in qc.index
]

qc["n_qc_review_flags"] = [
    len(
        flag_reasons[index]
    )
    for index in qc.index
]

flag_details = pd.DataFrame(
    flag_records,
    columns=[
        "file_id",
        "patient_id",
        "analysis_group",
        "metric",
        "observed_value",
        "group_median",
        "robust_z"
    ]
)

review_samples = qc[
    qc["n_qc_review_flags"] > 0
].copy()

review_samples = (
    review_samples
    .sort_values(
        [
            "n_qc_review_flags",
            "analysis_group"
        ],
        ascending=[
            False,
            True
        ]
    )
)

print(
    "\nSamples flagged for review:",
    len(review_samples)
)

if not review_samples.empty:

    print("\nFlagged samples (first 20):")

    print(
        review_samples[
            [
                "patient_id",
                "sample_id",
                "analysis_group",
                "workflow_short",
                "library_size_millions",
                "protein_coding_cpm_ge1",
                "assignment_percent",
                "no_feature_percent",
                "n_qc_review_flags"
            ]
        ]
        .head(20)
        .round(2)
        .to_string(index=False)
    )

else:

    print(
        "\nNo samples crossed the provisional "
        "outlier-review thresholds."
    )

print("\nNo samples have been excluded.")


# ------------------------------------------------------------
# 12. Generate QC figures
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENERATING RNA-SEQ QC FIGURES")
print("-" * 65)

figure_specs = [

    (
        "library_size_millions",
        "Assigned gene counts (millions)",
        "tcga_qc_library_sizes.png",
        "RNA-seq library size by cohort"
    ),

    (
        "protein_coding_cpm_ge1",
        "Protein-coding genes with CPM ≥ 1",
        "tcga_qc_protein_coding_detection.png",
        "Protein-coding gene detection by cohort"
    ),

    (
        "assignment_percent",
        "Gene-assignment fraction (%)",
        "tcga_qc_assignment_fraction.png",
        "STAR gene assignment by cohort"
    )

]

for metric, ylabel, filename, title in figure_specs:

    plot_data = [

        qc.loc[
            qc["analysis_group"] == group,
            metric
        ].to_numpy(dtype=float)

        for group in group_order
    ]

    fig, ax = plt.subplots(
        figsize=(11, 5.5)
    )

    ax.boxplot(
        plot_data,
        showfliers=True
    )

    ax.set_xticks(
        np.arange(
            1,
            len(group_order) + 1
        )
    )

    ax.set_xticklabels(
        group_order,
        rotation=35,
        ha="right"
    )

    ax.set_ylabel(ylabel)

    ax.set_title(title)

    ax.grid(
        axis="y",
        alpha=0.25
    )

    fig.tight_layout()

    figure_path = FIGURES / filename

    fig.savefig(
        figure_path,
        dpi=180,
        bbox_inches="tight"
    )

    plt.show()

    plt.close(fig)

    print(
        "Saved:",
        filename
    )


# ------------------------------------------------------------
# 13. Save QC tables
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING FULL-COHORT QC RESULTS")
print("-" * 65)

qc_path = (
    RESULTS / "tcga_full_cohort_qc.csv"
)

summary_path = (
    RESULTS / "tcga_full_cohort_qc_summary.csv"
)

workflow_path = (
    RESULTS / "tcga_full_cohort_qc_by_workflow.csv"
)

review_path = (
    RESULTS / "tcga_full_cohort_qc_review_samples.csv"
)

flag_path = (
    RESULTS / "tcga_full_cohort_qc_flag_details.csv"
)

qc.to_csv(
    qc_path,
    index=False
)

qc_summary.to_csv(
    summary_path,
    index=False
)

workflow_qc.to_csv(
    workflow_path,
    index=False
)

review_samples.to_csv(
    review_path,
    index=False
)

flag_details.to_csv(
    flag_path,
    index=False
)

print("\nAll QC tables saved.")


# ------------------------------------------------------------
# 14. Save research provenance
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "1.18",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "input_matrix": str(MATRIX_PATH),

    "matrix_dimensions": [
        int(counts.shape[0]),
        int(counts.shape[1])
    ],

    "samples_assessed": int(len(qc)),

    "protein_coding_genes": int(
        n_protein_coding
    ),

    "count_column": "unstranded",

    "cpm_denominator": (
        "Total unstranded counts assigned "
        "to all annotated genes"
    ),

    "qc_metrics": [
        "Assigned gene counts",
        "Detected genes",
        "Detected protein-coding genes",
        "Protein-coding detection at CPM >= 1",
        "Protein-coding detection at CPM >= 5",
        "STAR gene-assignment fraction",
        "STAR no-feature fraction",
        "STAR ambiguous fraction"
    ],

    "outlier_screening": {

        "method": (
            "Median absolute deviation "
            "robust z-score"
        ),

        "threshold": 3.5,

        "reference_groups": (
            "Project by tissue/cohort tier"
        ),

        "flags_are_exclusions": False

    },

    "flagged_samples": int(
        len(review_samples)
    ),

    "samples_excluded": 0,

    "normalization_performed": False,

    "descriptive_cpm_calculated": True,

    "differential_expression_performed": False,

    "limitations": [

        "QC flags require individual review.",

        "No sequencing-level FastQC or "
        "alignment QC was independently performed.",

        "Gene-assignment fraction is not "
        "genome alignment rate.",

        "CPM detection does not establish "
        "biological sample quality.",

        "Workflow version and cohort tier "
        "are strongly associated.",

        "Normal tissue comparisons require "
        "additional statistical design.",

        "Tier B is computationally low-MSI "
        "rather than categorically verified MSS."

    ]
}

provenance_path = (
    META / "tcga_full_cohort_qc_provenance.json"
)

with open(
    provenance_path,
    "w"
) as output:

    json.dump(
        provenance,
        output,
        indent=4
    )


# ------------------------------------------------------------
# 15. Create processed-data checkpoint
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING PROCESSED-DATA CHECKPOINT")
print("-" * 65)

timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S")

checkpoint_path = (
    CHECKPOINTS /
    f"SurfaceTarget_CRC_PostMatrix_Patch118_{timestamp}.zip"
)

# This checkpoint includes:
#
# - Processed count matrix
# - Sample and gene metadata
# - QC tables
# - QC figures
# - Existing project notes/code
#
# It intentionally excludes the 1.86 GiB
# original RNA-seq TSV files.

checkpoint_folders = [
    "data/metadata",
    "data/processed",
    "results/tables",
    "results/figures",
    "src",
    "reports",
    "logs"
]

with ZipFile(
    checkpoint_path,
    "w",
    ZIP_DEFLATED,
    compresslevel=3
) as archive:

    for folder in checkpoint_folders:

        folder_path = ROOT / folder

        if not folder_path.exists():
            continue

        for file_path in folder_path.rglob("*"):

            if file_path.is_file():

                archive.write(
                    file_path,
                    arcname=str(
                        file_path.relative_to(ROOT)
                    )
                )

checkpoint_size_mib = (
    checkpoint_path.stat().st_size
    / (1024 ** 2)
)

print("\nCheckpoint successfully created.")

print(
    f"Checkpoint size: "
    f"{checkpoint_size_mib:.2f} MiB"
)

print("\nCheckpoint path:")

print(checkpoint_path)


# ------------------------------------------------------------
# 16. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.18 SUMMARY")
print("=" * 65)

print("\nSamples analyzed:", len(qc))

print(
    "Protein-coding genes analyzed:",
    n_protein_coding
)

print(
    "\nSamples flagged for review:",
    len(review_samples)
)

print("Samples excluded: 0")

print("\nQC tables saved: 5")

print("QC figures saved: 3")

print("\nProcessed-data checkpoint: CREATED")

print(
    f"Checkpoint size: "
    f"{checkpoint_size_mib:.2f} MiB"
)

print("\nNo additional data downloaded.")

print("No changes made to raw expression counts.")

print("No differential expression performed.")

print("\nPATCH 1.18 COMPLETE")

print("=" * 65)

In [ ]:
# ============================================================
# SurfaceTarget-CRC
# PHASE 1 — PATCH 1.19
#
# RNA-SEQ PCA AND TECHNICAL CONFOUNDING ASSESSMENT
#
# Objectives:
# 1. Load the validated expression matrix
# 2. Select informative protein-coding genes
# 3. Calculate exploratory log2(CPM + 1) expression
# 4. Perform whole-cohort PCA
# 5. Examine workflow-associated expression structure
# 6. Perform workflow-restricted COAD and READ PCA
# 7. Highlight samples previously flagged for QC review
# 8. Save PCA coordinates, genes, figures and provenance
#
# IMPORTANT:
# - No samples are excluded.
# - No batch correction is applied.
# - No differential expression is performed.
# - PCA is exploratory, not a causal analysis.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

from sklearn.decomposition import PCA


# ------------------------------------------------------------
# 1. Project configuration
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"
PROCESSED = ROOT / "data/processed"
RESULTS = ROOT / "results/tables"
FIGURES = ROOT / "results/figures"

RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

MATRIX_PATH = (
    PROCESSED / "tcga_unstranded_raw_counts.npz"
)

GENES_PATH = META / "tcga_gene_annotations.csv"

SAMPLES_PATH = (
    META / "tcga_expression_sample_metadata.csv"
)

QC_PATH = RESULTS / "tcga_full_cohort_qc.csv"

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.19 — EXPRESSION PCA AND CONFOUNDING")
print("=" * 65)


# ------------------------------------------------------------
# 2. Load expression data
# ------------------------------------------------------------

for path in [
    MATRIX_PATH,
    GENES_PATH,
    SAMPLES_PATH,
    QC_PATH
]:
    if not path.exists():
        raise FileNotFoundError(path)

with np.load(
    MATRIX_PATH,
    allow_pickle=False
) as data:

    counts = data["counts"]
    gene_ids = data["gene_ids"]
    file_ids = data["file_ids"]

genes = pd.read_csv(
    GENES_PATH,
    dtype=str,
    keep_default_na=False
)

samples = pd.read_csv(
    SAMPLES_PATH,
    dtype=str,
    keep_default_na=False
)

qc = pd.read_csv(
    QC_PATH,
    dtype=str,
    keep_default_na=False
)

print("\nData loaded successfully.")

print("Expression matrix:", counts.shape)
print("Sample records:", len(samples))
print("QC records:", len(qc))


# ------------------------------------------------------------
# 3. Validate sample alignment
# ------------------------------------------------------------

if counts.shape != (60660, 472):
    raise RuntimeError(
        "Unexpected expression matrix dimensions."
    )

if not np.array_equal(
    file_ids,
    samples["file_id"].to_numpy()
):
    raise RuntimeError(
        "Matrix and sample metadata are misaligned."
    )

if not np.array_equal(
    file_ids,
    qc["file_id"].to_numpy()
):
    raise RuntimeError(
        "Matrix and QC records are misaligned."
    )

if not np.array_equal(
    gene_ids,
    genes["gene_id"].to_numpy()
):
    raise RuntimeError(
        "Gene identifiers are misaligned."
    )

if not np.issubdtype(
    counts.dtype,
    np.integer
):
    raise RuntimeError(
        "Raw count matrix must contain integers."
    )

qc["n_qc_review_flags"] = pd.to_numeric(
    qc["n_qc_review_flags"],
    errors="raise"
).astype(int)

qc["library_size_millions"] = pd.to_numeric(
    qc["library_size_millions"],
    errors="raise"
)

qc["assignment_percent"] = pd.to_numeric(
    qc["assignment_percent"],
    errors="raise"
)

print("\nSample and gene alignment: PASSED")


# ------------------------------------------------------------
# 4. Identify protein-coding genes
# ------------------------------------------------------------

coding_mask = (
    genes["gene_type"]
    .eq("protein_coding")
    .to_numpy(dtype=bool)
)

coding_counts = counts[
    coding_mask, :
]

coding_gene_ids = gene_ids[coding_mask]

print("\nProtein-coding genes:", len(coding_gene_ids))

if len(coding_gene_ids) != 19962:
    raise RuntimeError(
        "Unexpected protein-coding gene count."
    )

library_sizes = counts.sum(
    axis=0,
    dtype=np.int64
)

if np.any(library_sizes <= 0):
    raise RuntimeError(
        "Zero library size detected."
    )


# ------------------------------------------------------------
# 5. Identify shared workflow version
# ------------------------------------------------------------

# All normal samples should use the same
# recorded STAR workflow version.

normal_versions = samples.loc[
    samples["dataset_group"] == "Normal",
    "workflow_version"
].unique()

if len(normal_versions) != 1:
    raise RuntimeError(
        "Normal samples do not share one workflow."
    )

SHARED_WORKFLOW = normal_versions[0]

print("\nShared normal-tissue workflow:")
print(SHARED_WORKFLOW)

print("\nWorkflow-restricted sample availability:")

shared_samples = samples[
    samples["workflow_version"] == SHARED_WORKFLOW
]

print(
    pd.crosstab(
        shared_samples["project"],
        shared_samples["cohort_tier"]
    ).to_string()
)


# ------------------------------------------------------------
# 6. PCA analysis function
# ------------------------------------------------------------

def perform_pca(
    analysis_name,
    sample_indices,
    max_genes=2000
):

    sample_indices = np.asarray(
        sample_indices,
        dtype=int
    )

    subset = qc.iloc[
        sample_indices
    ].copy().reset_index(drop=True)

    n_samples = len(subset)

    if n_samples < 10:
        raise RuntimeError(
            f"Insufficient samples for {analysis_name}"
        )

    # --------------------------------------------
    # Extract protein-coding raw counts
    # --------------------------------------------

    subset_counts = coding_counts[
        :, sample_indices
    ]

    subset_libraries = library_sizes[
        sample_indices
    ]

    # --------------------------------------------
    # Calculate descriptive CPM
    # --------------------------------------------

    cpm = (
        subset_counts.astype(np.float64)
        / subset_libraries[np.newaxis, :]
        * 1_000_000
    )

    # --------------------------------------------
    # Filter minimally expressed genes
    #
    # CPM >= 1 in at least 10% of samples,
    # with a minimum of 10 samples.
    #
    # This filtering is ONLY for PCA.
    # --------------------------------------------

    min_detected_samples = max(
        10,
        int(np.ceil(n_samples * 0.10))
    )

    expressed_mask = (
        np.count_nonzero(
            cpm >= 1,
            axis=1
        )
        >= min_detected_samples
    )

    expressed_gene_indices = np.flatnonzero(
        expressed_mask
    )

    if len(expressed_gene_indices) < 100:
        raise RuntimeError(
            f"Too few expressed genes: {analysis_name}"
        )

    # --------------------------------------------
    # Log-transform expression
    # --------------------------------------------

    log_cpm = np.log2(
        cpm[expressed_mask, :] + 1.0
    )

    # --------------------------------------------
    # Select most variable genes
    # --------------------------------------------

    gene_variances = np.var(
        log_cpm,
        axis=1
    )

    ranked = np.argsort(
        -gene_variances,
        kind="stable"
    )

    n_selected = min(
        max_genes,
        int(np.count_nonzero(gene_variances > 0))
    )

    if n_selected < 100:
        raise RuntimeError(
            f"Too few variable genes: {analysis_name}"
        )

    top_indices = ranked[:n_selected]

    selected_gene_indices = (
        expressed_gene_indices[top_indices]
    )

    # Samples x genes.
    #
    # sklearn PCA automatically centers
    # each gene across samples.

    expression_for_pca = (
        log_cpm[top_indices, :].T
    )

    if not np.isfinite(
        expression_for_pca
    ).all():
        raise RuntimeError(
            "Nonfinite PCA expression values."
        )

    # --------------------------------------------
    # Fit PCA
    # --------------------------------------------

    model = PCA(
        n_components=3,
        svd_solver="randomized",
        random_state=42
    )

    coordinates = model.fit_transform(
        expression_for_pca
    )

    variance_explained = (
        model.explained_variance_ratio_ * 100
    )

    # --------------------------------------------
    # Save sample coordinates
    # --------------------------------------------

    for pc_index in range(3):

        subset[f"PC{pc_index + 1}"] = (
            coordinates[:, pc_index]
        )

    subset["pca_analysis"] = analysis_name

    # --------------------------------------------
    # Save genes used for PCA
    # --------------------------------------------

    selected_genes = pd.DataFrame({

        "gene_id":
            coding_gene_ids[selected_gene_indices],

        "log_cpm_variance":
            gene_variances[top_indices],

        "variance_rank":
            np.arange(1, n_selected + 1)
    })

    print("\n" + "-" * 60)
    print("PCA:", analysis_name)
    print("-" * 60)

    print("Samples:", n_samples)

    print(
        "Genes passing expression filter:",
        len(expressed_gene_indices)
    )

    print(
        "Variable genes selected:",
        n_selected
    )

    print(
        "Minimum samples for CPM filter:",
        min_detected_samples
    )

    print(
        "PC1 variance explained:",
        f"{variance_explained[0]:.2f}%"
    )

    print(
        "PC2 variance explained:",
        f"{variance_explained[1]:.2f}%"
    )

    print(
        "PC3 variance explained:",
        f"{variance_explained[2]:.2f}%"
    )

    print(
        "Previously flagged QC samples:",
        int(
            (subset["n_qc_review_flags"] > 0).sum()
        )
    )

    return (
        subset,
        selected_genes,
        variance_explained
    )


# ------------------------------------------------------------
# 7. PCA figure function
# ------------------------------------------------------------

def plot_pca(
    coordinates,
    variance,
    group_column,
    title,
    output_name
):

    fig, ax = plt.subplots(
        figsize=(10, 7)
    )

    groups = sorted(
        coordinates[group_column].unique()
    )

    for group in groups:

        group_data = coordinates[
            coordinates[group_column] == group
        ]

        ax.scatter(
            group_data["PC1"],
            group_data["PC2"],
            s=40,
            alpha=0.75,
            label=f"{group} (n={len(group_data)})"
        )

    # Highlight previously flagged QC samples
    # without removing them.

    flagged = coordinates[
        coordinates["n_qc_review_flags"] > 0
    ]

    if not flagged.empty:

        ax.scatter(
            flagged["PC1"],
            flagged["PC2"],
            marker="x",
            color="black",
            s=85,
            linewidths=1.3,
            label="QC review flag"
        )

    ax.set_xlabel(
        f"PC1 ({variance[0]:.1f}% variance)"
    )

    ax.set_ylabel(
        f"PC2 ({variance[1]:.1f}% variance)"
    )

    ax.set_title(title)

    ax.axhline(
        0,
        linewidth=0.5,
        alpha=0.35
    )

    ax.axvline(
        0,
        linewidth=0.5,
        alpha=0.35
    )

    ax.legend(
        fontsize=8,
        loc="best"
    )

    ax.grid(alpha=0.2)

    fig.tight_layout()

    output_path = FIGURES / output_name

    fig.savefig(
        output_path,
        dpi=200,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    print("Saved:", output_name)


# ------------------------------------------------------------
# 8. Whole-cohort PCA
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("WHOLE-COHORT EXPRESSION PCA")
print("=" * 65)

all_indices = np.arange(
    len(samples)
)

global_pca, global_genes, global_variance = (
    perform_pca(
        "global",
        all_indices
    )
)

plot_pca(
    global_pca,
    global_variance,
    "analysis_group",
    "TCGA RNA-seq PCA — Cohort Structure",
    "tcga_pca_global_by_cohort.png"
)

plot_pca(
    global_pca,
    global_variance,
    "workflow_short",
    "TCGA RNA-seq PCA — Workflow Versions",
    "tcga_pca_global_by_workflow.png"
)


# ------------------------------------------------------------
# 9. COAD PCA within common workflow
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("COAD WORKFLOW-RESTRICTED PCA")
print("=" * 65)

coad_mask = (
    samples["project"].eq("TCGA-COAD")
    & samples["workflow_version"].eq(
        SHARED_WORKFLOW
    )
)

coad_indices = np.flatnonzero(
    coad_mask.to_numpy()
)

coad_pca, coad_genes, coad_variance = (
    perform_pca(
        "coad_shared_workflow",
        coad_indices
    )
)

plot_pca(
    coad_pca,
    coad_variance,
    "analysis_group",
    "TCGA-COAD — Shared Workflow PCA",
    "tcga_coad_shared_workflow_pca.png"
)


# ------------------------------------------------------------
# 10. READ PCA within common workflow
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("READ WORKFLOW-RESTRICTED PCA")
print("=" * 65)

read_mask = (
    samples["project"].eq("TCGA-READ")
    & samples["workflow_version"].eq(
        SHARED_WORKFLOW
    )
)

read_indices = np.flatnonzero(
    read_mask.to_numpy()
)

read_pca, read_genes, read_variance = (
    perform_pca(
        "read_shared_workflow",
        read_indices
    )
)

plot_pca(
    read_pca,
    read_variance,
    "analysis_group",
    "TCGA-READ — Shared Workflow PCA",
    "tcga_read_shared_workflow_pca.png"
)


# ------------------------------------------------------------
# 11. Examine samples at PCA extremes
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("EXPRESSION STRUCTURE REVIEW")
print("=" * 65)

# Distance from the global PCA centroid
# is descriptive only.
#
# Samples far from the centroid are NOT
# automatically considered low-quality.

global_pca["pc12_distance"] = np.sqrt(
    global_pca["PC1"] ** 2
    + global_pca["PC2"] ** 2
)

extreme_samples = (
    global_pca
    .sort_values(
        "pc12_distance",
        ascending=False
    )
    .head(12)
)

print("\nSamples farthest from global PCA centroid:")

print(
    extreme_samples[
        [
            "patient_id",
            "sample_id",
            "analysis_group",
            "workflow_short",
            "PC1",
            "PC2",
            "library_size_millions",
            "assignment_percent",
            "n_qc_review_flags"
        ]
    ]
    .round(2)
    .to_string(index=False)
)

print("\nNOTE:")
print(
    "Extreme PCA coordinates do not "
    "automatically indicate poor sample quality."
)


# ------------------------------------------------------------
# 12. Save PCA outputs
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING PCA RESULTS")
print("-" * 65)

pca_tables = {

    "tcga_global_pca_coordinates.csv":
        global_pca,

    "tcga_coad_shared_workflow_pca.csv":
        coad_pca,

    "tcga_read_shared_workflow_pca.csv":
        read_pca,

    "tcga_global_pca_genes.csv":
        global_genes,

    "tcga_coad_pca_genes.csv":
        coad_genes,

    "tcga_read_pca_genes.csv":
        read_genes
}

for filename, dataframe in pca_tables.items():

    dataframe.to_csv(
        RESULTS / filename,
        index=False
    )

print("\nPCA tables saved:", len(pca_tables))


# ------------------------------------------------------------
# 13. Save PCA summary and provenance
# ------------------------------------------------------------

summary_records = []

for name, data, features, variance in [

    (
        "global",
        global_pca,
        global_genes,
        global_variance
    ),

    (
        "coad_shared_workflow",
        coad_pca,
        coad_genes,
        coad_variance
    ),

    (
        "read_shared_workflow",
        read_pca,
        read_genes,
        read_variance
    )

]:

    summary_records.append({

        "analysis": name,

        "samples": len(data),

        "selected_genes": len(features),

        "pc1_variance_percent": variance[0],

        "pc2_variance_percent": variance[1],

        "pc3_variance_percent": variance[2],

        "qc_flagged_samples": int(
            (data["n_qc_review_flags"] > 0).sum()
        )
    })

pca_summary = pd.DataFrame(
    summary_records
)

pca_summary.to_csv(
    RESULTS / "tcga_pca_analysis_summary.csv",
    index=False
)

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "1.19",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "input_matrix": str(MATRIX_PATH),

    "expression_measure": (
        "Unstranded raw gene counts"
    ),

    "pca_transformation": "log2(CPM + 1)",

    "cpm_denominator": (
        "Total unstranded counts assigned "
        "to all annotated genes"
    ),

    "gene_selection": (
        "Protein-coding genes with CPM >= 1 "
        "in at least 10% of samples, minimum "
        "10 samples, followed by the 2000 "
        "most variable log-CPM genes"
    ),

    "pca_components": 3,

    "random_state": 42,

    "shared_workflow_version":
        SHARED_WORKFLOW,

    "samples_excluded": 0,

    "batch_correction_performed": False,

    "differential_expression_performed": False,

    "limitations": [

        "Log-CPM PCA is exploratory.",

        "Composition-adjusted normalization "
        "and variance-stabilizing transformations "
        "have not yet been evaluated.",

        "PCA does not establish causes "
        "of expression variation.",

        "Workflow version and evidence tier "
        "are strongly confounded.",

        "The shared-workflow subset contains "
        "very few Tier A tumors.",

        "Shared workflow does not eliminate "
        "all potential batch effects.",

        "QC review flags are not exclusions.",

        "Normal tissues are not all paired "
        "with tumor samples."

    ]
}

with open(
    META / "tcga_pca_provenance.json",
    "w"
) as f:

    json.dump(
        provenance,
        f,
        indent=4
    )


# ------------------------------------------------------------
# 14. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.19 SUMMARY")
print("=" * 65)

print("\nPCA analysis summary:")

print(
    pca_summary
    .round(2)
    .to_string(index=False)
)

print("\nFigures generated: 4")

print("PCA coordinate/gene tables: 6")

print("Summary table: 1")

print("Provenance: SAVED")

print("\nSamples excluded: 0")

print("Batch correction performed: NO")

print("Differential expression performed: NO")

print("\nPATCH 1.19 COMPLETE")

print("=" * 65)

In [ ]:
# ============================================================
# SurfaceTarget-CRC
# PHASE 1 — PATCH 1.20 (CORRECTED)
#
# EXPLORATORY COAD DIFFERENTIAL EXPRESSION
#
# Discovery comparison:
# - Tier B COAD primary tumors
# - Patient-independent COAD normal tissues
# - Shared GDC STAR workflow version
#
# Additional evidence:
# - Tier A COAD tumor expression support
#
# FIX:
# Explicit Pandas string indexes for AnnData compatibility.
#
# IMPORTANT:
# 1. Tier B is computationally low-MSI, not verified MSS.
# 2. No sample is removed from the source dataset.
# 3. Patient overlap is avoided in the discovery contrast.
# 4. Results are exploratory pending sensitivity analyses.
# 5. Differential expression is not therapeutic validation.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError

import subprocess
import sys
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"
PROCESSED = ROOT / "data/processed"
RESULTS = ROOT / "results/tables"
FIGURES = ROOT / "results/figures"

RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

MATRIX_PATH = (
    PROCESSED / "tcga_unstranded_raw_counts.npz"
)

SAMPLES_PATH = (
    META / "tcga_expression_sample_metadata.csv"
)

GENES_PATH = (
    META / "tcga_gene_annotations.csv"
)

QC_PATH = (
    RESULTS / "tcga_full_cohort_qc.csv"
)

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.20 — EXPLORATORY COAD DIFFERENTIAL EXPRESSION")
print("=" * 65)


# ------------------------------------------------------------
# 2. Check PyDESeq2 installation
# ------------------------------------------------------------

print("\nChecking differential expression dependencies...")

REQUIRED_VERSION = "0.5.4"

try:
    installed_version = version("pydeseq2")

except PackageNotFoundError:
    installed_version = None

if installed_version != REQUIRED_VERSION:

    print(
        "Installing PyDESeq2",
        REQUIRED_VERSION
    )

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        f"pydeseq2=={REQUIRED_VERSION}"
    ])

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

print("\nDependency versions:")

print("PyDESeq2:", version("pydeseq2"))
print("Pandas:", pd.__version__)
print("AnnData:", version("anndata"))
print("NumPy:", np.__version__)


# ------------------------------------------------------------
# 3. Load validated data
# ------------------------------------------------------------

required_paths = [
    MATRIX_PATH,
    SAMPLES_PATH,
    GENES_PATH,
    QC_PATH
]

for path in required_paths:

    if not path.exists():

        raise FileNotFoundError(
            f"Required project file missing: {path}"
        )

with np.load(
    MATRIX_PATH,
    allow_pickle=False
) as archive:

    counts = archive["counts"]
    gene_ids = archive["gene_ids"]
    file_ids = archive["file_ids"]

samples = pd.read_csv(
    SAMPLES_PATH,
    dtype=str,
    keep_default_na=False
)

genes = pd.read_csv(
    GENES_PATH,
    dtype=str,
    keep_default_na=False
)

qc = pd.read_csv(
    QC_PATH,
    dtype=str,
    keep_default_na=False
)

print("\nExpression data loaded.")

print("Matrix:", counts.shape)
print("Samples:", len(samples))
print("Gene annotations:", len(genes))


# ------------------------------------------------------------
# 4. Validate sample and gene alignment
# ------------------------------------------------------------

if counts.shape != (60660, 472):

    raise RuntimeError(
        "Unexpected count matrix dimensions."
    )

if not np.array_equal(
    file_ids,
    samples["file_id"].to_numpy()
):

    raise RuntimeError(
        "Sample metadata alignment failed."
    )

if not np.array_equal(
    file_ids,
    qc["file_id"].to_numpy()
):

    raise RuntimeError(
        "QC metadata alignment failed."
    )

if not np.array_equal(
    gene_ids,
    genes["gene_id"].to_numpy()
):

    raise RuntimeError(
        "Gene annotation alignment failed."
    )

if not np.issubdtype(
    counts.dtype,
    np.integer
):

    raise RuntimeError(
        "Raw count matrix must be integer-valued."
    )

if np.any(counts < 0):

    raise RuntimeError(
        "Negative raw counts detected."
    )

print("\nData alignment validation: PASSED")


# ------------------------------------------------------------
# 5. Identify shared processing workflow
# ------------------------------------------------------------

normal_versions = samples.loc[
    samples["dataset_group"].eq("Normal"),
    "workflow_version"
].unique()

if len(normal_versions) != 1:

    raise RuntimeError(
        "Normal samples do not share one workflow."
    )

SHARED_WORKFLOW = normal_versions[0]

print("\nDiscovery workflow:")
print(SHARED_WORKFLOW)


# ------------------------------------------------------------
# 6. Define COAD discovery populations
# ------------------------------------------------------------

coad = samples["project"].eq("TCGA-COAD")

shared = samples["workflow_version"].eq(
    SHARED_WORKFLOW
)

tier_b_mask = (
    coad
    & shared
    & samples["dataset_group"].eq("Tumor")
    & samples["cohort_tier"].eq("Tier B")
)

normal_mask = (
    coad
    & shared
    & samples["dataset_group"].eq("Normal")
)

tier_a_mask = (
    coad
    & samples["dataset_group"].eq("Tumor")
    & samples["cohort_tier"].eq("Tier A")
)

tumor_patient_ids = set(
    samples.loc[
        tier_b_mask,
        "patient_id"
    ]
)

# Avoid including tumor and normal measurements
# from the same patient in the initial
# unpaired differential expression model.

overlapping_normals = (
    normal_mask
    & samples["patient_id"].isin(
        tumor_patient_ids
    )
)

independent_normal_mask = (
    normal_mask
    & ~samples["patient_id"].isin(
        tumor_patient_ids
    )
)

tumor_indices = np.flatnonzero(
    tier_b_mask.to_numpy()
)

normal_indices = np.flatnonzero(
    independent_normal_mask.to_numpy()
)

tier_a_indices = np.flatnonzero(
    tier_a_mask.to_numpy()
)

print("\n" + "-" * 65)
print("DISCOVERY COHORT DEFINITION")
print("-" * 65)

print("\nTier B COAD tumors:", len(tumor_indices))

print(
    "Available COAD normals:",
    int(normal_mask.sum())
)

print(
    "Normals overlapping Tier B patients:",
    int(overlapping_normals.sum())
)

print(
    "Patient-independent normals:",
    len(normal_indices)
)

print(
    "Tier A COAD expression-support samples:",
    len(tier_a_indices)
)

if len(tumor_indices) != 194:

    raise RuntimeError(
        "Unexpected Tier B tumor count."
    )

if len(tier_a_indices) != 98:

    raise RuntimeError(
        "Unexpected Tier A tumor count."
    )

if len(normal_indices) < 20:

    raise RuntimeError(
        "Insufficient independent normal samples."
    )

normal_patients = set(
    samples.iloc[normal_indices]["patient_id"]
)

if normal_patients & tumor_patient_ids:

    raise RuntimeError(
        "Patient overlap remains in discovery cohort."
    )

selected_indices = np.concatenate([
    normal_indices,
    tumor_indices
])

selected_samples = samples.iloc[
    selected_indices
].copy()

if selected_samples["patient_id"].duplicated().any():

    raise RuntimeError(
        "Repeated patient in discovery cohort."
    )

if selected_samples["workflow_version"].nunique() != 1:

    raise RuntimeError(
        "Discovery workflow is not consistent."
    )

print("\nPatient independence: PASSED")
print("Workflow restriction: PASSED")


# ------------------------------------------------------------
# 7. Save cohort selection metadata
# ------------------------------------------------------------

selection = selected_samples.copy()

selection["condition"] = (
    ["Normal"] * len(normal_indices)
    + ["Tumor"] * len(tumor_indices)
)

selection["analysis"] = (
    "COAD_TierB_vs_independent_normal"
)

selection["qc_review_flags"] = (
    qc.iloc[selected_indices][
        "n_qc_review_flags"
    ].to_numpy()
)

selection.to_csv(
    META / "tcga_coad_de_discovery_samples.csv",
    index=False
)

print("\nDiscovery sample manifest saved.")


# ------------------------------------------------------------
# 8. Extract protein-coding raw counts
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING DIFFERENTIAL EXPRESSION MATRIX")
print("-" * 65)

coding_mask = (
    genes["gene_type"]
    .eq("protein_coding")
    .to_numpy(dtype=bool)
)

coding_gene_ids = gene_ids[coding_mask]

# PyDESeq2 expects samples x genes.

analysis_counts = (
    counts[coding_mask, :][
        :, selected_indices
    ].T
)

if analysis_counts.shape[0] != len(
    selected_indices
):

    raise RuntimeError(
        "DE count matrix sample count mismatch."
    )


# ------------------------------------------------------------
# 9. Pre-filter very low-expression genes
# ------------------------------------------------------------

# CPM is used ONLY for expression filtering.
# The statistical model receives original
# integer raw counts.

total_library_sizes = counts[
    :, selected_indices
].sum(axis=0)

if np.any(total_library_sizes <= 0):

    raise RuntimeError(
        "Zero assigned-count library detected."
    )

cpm = (
    analysis_counts.astype(np.float64)
    / total_library_sizes[:, None]
    * 1_000_000
)

minimum_samples = max(
    10,
    int(np.ceil(
        0.10 * len(selected_indices)
    ))
)

expressed_mask = (
    np.count_nonzero(
        cpm >= 1,
        axis=0
    )
    >= minimum_samples
)

selected_gene_ids = (
    coding_gene_ids[expressed_mask]
)

filtered_counts = (
    analysis_counts[:, expressed_mask]
)

print(
    "\nOriginal protein-coding genes:",
    len(coding_gene_ids)
)

print(
    "Genes passing expression filter:",
    len(selected_gene_ids)
)

print(
    "Minimum samples with CPM >= 1:",
    minimum_samples
)

if len(selected_gene_ids) < 5000:

    raise RuntimeError(
        "Unexpectedly few genes passed filtering."
    )

if np.any(filtered_counts < 0):

    raise RuntimeError(
        "Negative counts in DE matrix."
    )


# ------------------------------------------------------------
# 10. Construct PyDESeq2 inputs
# ------------------------------------------------------------

selected_file_ids = (
    selected_samples["file_id"].tolist()
)

counts_df = pd.DataFrame(
    filtered_counts,
    index=selected_file_ids,
    columns=selected_gene_ids
)

metadata = pd.DataFrame(
    {
        "condition": (
            ["Normal"] * len(normal_indices)
            + ["Tumor"] * len(tumor_indices)
        )
    },
    index=selected_file_ids
)

metadata["condition"] = pd.Categorical(
    metadata["condition"],
    categories=["Normal", "Tumor"],
    ordered=True
)

if not counts_df.index.equals(metadata.index):

    raise RuntimeError(
        "DE count and metadata indices are misaligned."
    )

if counts_df.index.duplicated().any():

    raise RuntimeError(
        "Duplicate samples in DE input."
    )

print("\nPyDESeq2 input matrix:", counts_df.shape)

print("\nConditions:")

print(
    metadata["condition"]
    .value_counts()
    .to_string()
)


# ============================================================
# 10A. PANDAS / ANNDATA COMPATIBILITY FIX
#
# This explicitly converts the identifiers to
# Pandas StringDtype indexes.
#
# AnnData can then recognize the existing
# string dtype instead of attempting the
# incompatible StringDtype(na_value=...) call.
#
# The gene counts and biological metadata
# are not modified.
# ============================================================

print("\n" + "-" * 65)
print("APPLYING ANNDATA STRING-INDEX FIX")
print("-" * 65)

counts_df.index = pd.Index(
    counts_df.index,
    dtype="string"
)

counts_df.columns = pd.Index(
    counts_df.columns,
    dtype="string"
)

metadata.index = pd.Index(
    metadata.index,
    dtype="string"
)

# Verify that the sample identifiers
# remain aligned after conversion.

if not counts_df.index.equals(metadata.index):

    raise RuntimeError(
        "Sample index alignment failed after conversion."
    )

if not np.array_equal(
    counts_df.index.astype(str).to_numpy(),
    np.asarray(selected_file_ids, dtype=str)
):

    raise RuntimeError(
        "Sample identifiers changed during conversion."
    )

if counts_df.index.dtype != "string":

    raise RuntimeError(
        "Sample index is not Pandas StringDtype."
    )

if counts_df.columns.dtype != "string":

    raise RuntimeError(
        "Gene index is not Pandas StringDtype."
    )

if metadata.index.dtype != "string":

    raise RuntimeError(
        "Metadata index is not Pandas StringDtype."
    )

print("\nSample index conversion: PASSED")
print("Gene index conversion: PASSED")
print("Metadata index conversion: PASSED")
print("Sample alignment: PASSED")

print("\nAnnData compatibility fix applied.")


# ------------------------------------------------------------
# 11. Fit differential expression model
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FITTING DIFFERENTIAL EXPRESSION MODEL")
print("=" * 65)

print("\nModel: ~ condition")
print("Contrast: Tumor versus Normal")

dds = DeseqDataSet(
    counts=counts_df,
    metadata=metadata,
    design="~condition",
    refit_cooks=True,
    n_cpus=2
)

print("\nDESeq2 dataset initialized successfully.")

dds.deseq2()

print("\nDESeq2 model fitting: COMPLETE")


# ------------------------------------------------------------
# 12. Differential expression statistics
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CALCULATING DIFFERENTIAL EXPRESSION")
print("-" * 65)

stats = DeseqStats(
    dds,
    contrast=[
        "condition",
        "Tumor",
        "Normal"
    ],
    alpha=0.05,
    n_cpus=2
)

stats.summary()

results = (
    stats.results_df
    .copy()
    .rename_axis("gene_id")
    .reset_index()
)

required_result_columns = {
    "gene_id",
    "baseMean",
    "log2FoldChange",
    "pvalue",
    "padj"
}

if not required_result_columns.issubset(
    results.columns
):

    raise RuntimeError(
        "Unexpected PyDESeq2 result format."
    )

if results["gene_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate genes in DE results."
    )

print("\nDifferential expression statistics: COMPLETE")


# ------------------------------------------------------------
# 13. Attach gene annotations
# ------------------------------------------------------------

annotation_subset = genes[
    ["gene_id", "gene_name", "gene_type"]
].copy()

results = results.merge(
    annotation_subset,
    on="gene_id",
    how="left",
    validate="one_to_one"
)

if results["gene_name"].isna().any():

    raise RuntimeError(
        "Some genes have missing annotations."
    )

print("\nGene annotations attached successfully.")


# ------------------------------------------------------------
# 14. Add Tier A expression-support metrics
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CALCULATING TIER A EXPRESSION SUPPORT")
print("-" * 65)

# Evaluate expression prevalence
# separately in Tier A.
#
# This is NOT differential expression
# between Tier A and Tier B.

gene_lookup = {
    gene_id: index
    for index, gene_id in enumerate(gene_ids)
}

selected_gene_rows = np.array([
    gene_lookup[gene_id]
    for gene_id in selected_gene_ids
])


def descriptive_expression(indices):

    selected_raw = counts[
        np.ix_(
            selected_gene_rows,
            indices
        )
    ].astype(np.float64)

    libraries = counts[
        :, indices
    ].sum(axis=0)

    if np.any(libraries <= 0):

        raise RuntimeError(
            "Zero library in expression-support group."
        )

    group_cpm = (
        selected_raw
        / libraries[None, :]
        * 1_000_000
    )

    return pd.DataFrame({

        "gene_id": selected_gene_ids,

        "median_cpm": np.median(
            group_cpm,
            axis=1
        ),

        "fraction_cpm_ge1": np.mean(
            group_cpm >= 1,
            axis=1
        )
    })


tier_a_expression = descriptive_expression(
    tier_a_indices
)

tier_b_expression = descriptive_expression(
    tumor_indices
)

normal_expression = descriptive_expression(
    normal_indices
)


tier_a_expression = tier_a_expression.rename(
    columns={
        "median_cpm": "tier_a_median_cpm",
        "fraction_cpm_ge1": "tier_a_fraction_cpm_ge1"
    }
)

tier_b_expression = tier_b_expression.rename(
    columns={
        "median_cpm": "tier_b_median_cpm",
        "fraction_cpm_ge1": "tier_b_fraction_cpm_ge1"
    }
)

normal_expression = normal_expression.rename(
    columns={
        "median_cpm": "normal_median_cpm",
        "fraction_cpm_ge1": "normal_fraction_cpm_ge1"
    }
)


for expression_table in [
    tier_a_expression,
    tier_b_expression,
    normal_expression
]:

    results = results.merge(
        expression_table,
        on="gene_id",
        how="left",
        validate="one_to_one"
    )

print("\nTier A expression support: COMPLETE")


# ------------------------------------------------------------
# 15. Identify exploratory upregulated genes
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("EXPLORATORY DIFFERENTIAL EXPRESSION RESULTS")
print("=" * 65)

results["tumor_upregulated"] = (
    results["padj"].lt(0.05)
    & results["log2FoldChange"].gt(1)
)

results["tumor_downregulated"] = (
    results["padj"].lt(0.05)
    & results["log2FoldChange"].lt(-1)
)

results["tier_a_expression_supported"] = (
    results["tier_a_fraction_cpm_ge1"].ge(0.50)
    & results["tier_a_median_cpm"].ge(1)
)

# Provisional transcriptomic screening genes.
# These are not validated therapeutic targets.

candidates = results[
    results["tumor_upregulated"]
    & results["tier_a_expression_supported"]
].copy()

candidates = candidates.sort_values(
    [
        "padj",
        "log2FoldChange"
    ],
    ascending=[
        True,
        False
    ]
)

print("\nGenes tested:", len(results))

print(
    "Tumor-upregulated genes "
    "(padj < 0.05, log2FC > 1):",
    int(results["tumor_upregulated"].sum())
)

print(
    "Tumor-downregulated genes "
    "(padj < 0.05, log2FC < -1):",
    int(results["tumor_downregulated"].sum())
)

print(
    "Upregulated genes with Tier A "
    "expression support:",
    len(candidates)
)

print("\nTop 20 exploratory screening genes:")

display_columns = [
    "gene_name",
    "log2FoldChange",
    "padj",
    "tier_a_median_cpm",
    "tier_a_fraction_cpm_ge1",
    "normal_median_cpm"
]

print(
    candidates[display_columns]
    .head(20)
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4g}"
    )
)


# ------------------------------------------------------------
# 16. Generate exploratory volcano plot
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENERATING DIFFERENTIAL EXPRESSION FIGURE")
print("-" * 65)

plot_results = results[
    results["padj"].notna()
    & results["log2FoldChange"].notna()
].copy()

plot_results["neg_log10_padj"] = (
    -np.log10(
        plot_results["padj"].clip(
            lower=np.finfo(float).tiny
        )
    )
)

fig, ax = plt.subplots(
    figsize=(10, 7)
)

ax.scatter(
    plot_results["log2FoldChange"],
    plot_results["neg_log10_padj"],
    s=8,
    alpha=0.45
)

ax.axvline(
    -1,
    linestyle="--",
    linewidth=0.8
)

ax.axvline(
    1,
    linestyle="--",
    linewidth=0.8
)

ax.axhline(
    -np.log10(0.05),
    linestyle="--",
    linewidth=0.8
)

ax.set_xlabel(
    "Log2 fold change: Tier B tumor / normal"
)

ax.set_ylabel(
    "-log10 adjusted p-value"
)

ax.set_title(
    "TCGA-COAD — Exploratory Tumor vs Normal"
)

ax.grid(alpha=0.2)

fig.tight_layout()

figure_path = (
    FIGURES /
    "tcga_coad_tierb_exploratory_volcano.png"
)

fig.savefig(
    figure_path,
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print("Volcano plot saved.")


# ------------------------------------------------------------
# 17. Save expression and statistical results
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING DIFFERENTIAL EXPRESSION RESULTS")
print("-" * 65)

all_results_path = (
    RESULTS /
    "tcga_coad_tierb_vs_normal_deseq2_results.csv"
)

candidate_path = (
    RESULTS /
    "tcga_coad_exploratory_upregulated_genes.csv"
)

results.to_csv(
    all_results_path,
    index=False
)

candidates.to_csv(
    candidate_path,
    index=False
)

size_factors = pd.DataFrame({

    "file_id": dds.obs.index.astype(str),

    "size_factor": dds.obs[
        "size_factors"
    ].to_numpy()
})

size_factors.to_csv(
    RESULTS / "tcga_coad_deseq2_size_factors.csv",
    index=False
)

print("\nDifferential expression files saved.")


# ------------------------------------------------------------
# 18. Save analysis provenance
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "1.20",

    "patch_version": "1.20-corrected",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "analysis": (
        "Exploratory COAD Tier B tumor "
        "versus patient-independent normal"
    ),

    "software": "PyDESeq2",

    "software_version": version("pydeseq2"),

    "pandas_version": pd.__version__,

    "anndata_version": version("anndata"),

    "compatibility_fix": (
        "Explicit Pandas StringDtype indexes "
        "for AnnData sample and gene identifiers"
    ),

    "design": "~condition",

    "contrast": [
        "condition",
        "Tumor",
        "Normal"
    ],

    "workflow_version": SHARED_WORKFLOW,

    "tumor_samples": int(
        len(tumor_indices)
    ),

    "normal_samples": int(
        len(normal_indices)
    ),

    "overlapping_normals_withheld": int(
        overlapping_normals.sum()
    ),

    "tier_a_expression_support_samples": int(
        len(tier_a_indices)
    ),

    "count_column": "unstranded",

    "genes_tested": int(
        len(results)
    ),

    "expression_prefilter": {

        "cpm_threshold": 1,

        "minimum_samples": int(
            minimum_samples
        )
    },

    "tumor_upregulated_genes": int(
        results["tumor_upregulated"].sum()
    ),

    "tier_a_supported_screening_genes": int(
        len(candidates)
    ),

    "multiple_testing": (
        "Benjamini-Hochberg adjusted p-values "
        "through PyDESeq2"
    ),

    "qc_flagged_samples_excluded": False,

    "clinical_covariates_adjusted": False,

    "paired_model": False,

    "sensitivity_analysis_completed": False,

    "limitations": [

        "Tier B is not independently confirmed MSS.",

        "Normal tissues may be tumor-adjacent.",

        "Tissue composition may influence "
        "differential expression.",

        "Clinical covariates are not adjusted.",

        "Other technical confounding may remain.",

        "QC-flagged sample sensitivity "
        "analysis remains pending.",

        "A patient-paired analysis remains pending.",

        "Tier A expression support is descriptive.",

        "RNA expression does not establish "
        "protein abundance.",

        "RNA expression does not establish "
        "cell-surface localization.",

        "Screening genes are not validated "
        "therapeutic targets."
    ]
}

with open(
    META /
    "tcga_coad_exploratory_de_provenance.json",
    "w"
) as f:

    json.dump(
        provenance,
        f,
        indent=4
    )

print("\nAnalysis provenance saved.")


# ------------------------------------------------------------
# 19. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.20 SUMMARY")
print("=" * 65)

print(
    "\nDiscovery tumor samples:",
    len(tumor_indices)
)

print(
    "Independent normal samples:",
    len(normal_indices)
)

print(
    "Tier A support samples:",
    len(tier_a_indices)
)

print(
    "\nProtein-coding genes tested:",
    len(results)
)

print(
    "Exploratory upregulated genes:",
    int(results["tumor_upregulated"].sum())
)

print(
    "Upregulated genes with Tier A "
    "expression support:",
    len(candidates)
)

print("\nSaved outputs:")

print("1. tcga_coad_de_discovery_samples.csv")

print("2. tcga_coad_tierb_vs_normal_deseq2_results.csv")

print("3. tcga_coad_exploratory_upregulated_genes.csv")

print("4. tcga_coad_deseq2_size_factors.csv")

print("5. tcga_coad_tierb_exploratory_volcano.png")

print("6. tcga_coad_exploratory_de_provenance.json")

print("\nAnnData compatibility fix: APPLIED")

print("No raw counts modified.")

print("No permanent sample exclusions.")

print("\nPATCH 1.20 COMPLETE")

print("=" * 65)

In [ ]:
%pip install -q --upgrade "pandas==2.3.3" #upgraded as the cell above this was not executing

In [ ]:
# ============================================================
# SurfaceTarget-CRC
# PHASE 1 — PATCH 1.21
#
# DIFFERENTIAL EXPRESSION ROBUSTNESS ASSESSMENT
#
# Sensitivity analysis:
# Repeat COAD Tier B vs normal differential expression
# after withholding previously QC-flagged samples.
#
# Objectives:
# 1. Load validated baseline DE results
# 2. Identify QC-flagged discovery samples
# 3. Create a QC-screened sensitivity cohort
# 4. Refit PyDESeq2 using original raw counts
# 5. Compare expression effect sizes
# 6. Assess significance and candidate stability
# 7. Preserve results and provenance
#
# IMPORTANT:
# No permanent sample exclusions.
# Original DE results remain unchanged.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

from scipy.stats import spearmanr
from packaging.version import Version

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"
PROCESSED = ROOT / "data/processed"
RESULTS = ROOT / "results/tables"
FIGURES = ROOT / "results/figures"
CHECKPOINTS = ROOT / "checkpoints"

FIGURES.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
CHECKPOINTS.mkdir(parents=True, exist_ok=True)

MATRIX_PATH = (
    PROCESSED / "tcga_unstranded_raw_counts.npz"
)

BASELINE_PATH = (
    RESULTS /
    "tcga_coad_tierb_vs_normal_deseq2_results.csv"
)

DISCOVERY_PATH = (
    META / "tcga_coad_de_discovery_samples.csv"
)

QC_PATH = (
    RESULTS / "tcga_full_cohort_qc.csv"
)

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.21 — DIFFERENTIAL EXPRESSION ROBUSTNESS")
print("=" * 65)


# ------------------------------------------------------------
# 2. Validate software environment
# ------------------------------------------------------------

print("\nSoftware environment:")

print("PyDESeq2:", version("pydeseq2"))
print("Pandas:", pd.__version__)
print("AnnData:", version("anndata"))

if Version(pd.__version__) < Version("2.3.0"):

    raise RuntimeError(
        "Pandas >= 2.3.0 is required for "
        "the current AnnData environment."
    )

if version("pydeseq2") != "0.5.4":

    raise RuntimeError(
        "PyDESeq2 version differs from "
        "the baseline analysis."
    )


# ------------------------------------------------------------
# 3. Load existing research data
# ------------------------------------------------------------

for path in [
    MATRIX_PATH,
    BASELINE_PATH,
    DISCOVERY_PATH,
    QC_PATH
]:

    if not path.exists():

        raise FileNotFoundError(path)

with np.load(
    MATRIX_PATH,
    allow_pickle=False
) as archive:

    counts = archive["counts"]
    gene_ids = archive["gene_ids"]
    file_ids = archive["file_ids"]

baseline = pd.read_csv(
    BASELINE_PATH,
    dtype={"gene_id": str}
)

discovery = pd.read_csv(
    DISCOVERY_PATH,
    dtype=str,
    keep_default_na=False
)

qc = pd.read_csv(
    QC_PATH,
    dtype=str,
    keep_default_na=False
)

print("\nData loaded successfully.")

print("Expression matrix:", counts.shape)

print("Baseline genes:", len(baseline))

print("Discovery samples:", len(discovery))

print("QC samples:", len(qc))


# ------------------------------------------------------------
# 4. Validate baseline analysis
# ------------------------------------------------------------

if counts.shape != (60660, 472):

    raise RuntimeError(
        "Unexpected expression matrix dimensions."
    )

if len(baseline) != 14832:

    raise RuntimeError(
        "Unexpected baseline gene count."
    )

if len(discovery) != 223:

    raise RuntimeError(
        "Unexpected discovery sample count."
    )

if baseline["gene_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate gene IDs in baseline."
    )

if discovery["file_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate discovery sample IDs."
    )

if qc["file_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate QC file identifiers."
    )

if len(set(discovery["file_id"]) - set(file_ids)):

    raise RuntimeError(
        "Discovery samples missing from count matrix."
    )

if discovery["patient_id"].duplicated().any():

    raise RuntimeError(
        "Discovery cohort contains repeated patients."
    )

condition_counts = (
    discovery["condition"].value_counts()
)

if (
    condition_counts.get("Tumor", 0) != 194
    or condition_counts.get("Normal", 0) != 29
):

    raise RuntimeError(
        "Baseline cohort composition changed."
    )

print("\nBaseline validation: PASSED")


# ------------------------------------------------------------
# 5. Attach QC review flags
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("DISCOVERY COHORT QC AUDIT")
print("-" * 65)

qc_flags = qc[
    ["file_id", "n_qc_review_flags"]
].copy()

qc_flags["n_qc_review_flags"] = pd.to_numeric(
    qc_flags["n_qc_review_flags"],
    errors="raise"
).astype(int)

discovery = discovery.drop(
    columns=["n_qc_review_flags"],
    errors="ignore"
)

discovery = discovery.merge(
    qc_flags,
    on="file_id",
    how="left",
    validate="one_to_one"
)

if discovery["n_qc_review_flags"].isna().any():

    raise RuntimeError(
        "Missing QC flags for discovery samples."
    )

discovery["qc_flagged"] = (
    discovery["n_qc_review_flags"] > 0
)

discovery["included_in_sensitivity"] = (
    ~discovery["qc_flagged"]
)

print("\nOriginal discovery composition:")

print(
    discovery.groupby("condition")
    .agg(
        total=("file_id", "count"),
        flagged=("qc_flagged", "sum")
    )
    .to_string()
)

print(
    "\nTotal flagged discovery samples:",
    int(discovery["qc_flagged"].sum())
)

print("\nFlagged sample identifiers:")

flagged_samples = discovery[
    discovery["qc_flagged"]
].copy()

if not flagged_samples.empty:

    print(
        flagged_samples[
            [
                "patient_id",
                "sample_id",
                "condition",
                "n_qc_review_flags"
            ]
        ]
        .to_string(index=False)
    )

else:

    print("None.")


# ------------------------------------------------------------
# 6. Construct sensitivity cohort
# ------------------------------------------------------------

sensitivity_samples = discovery[
    discovery["included_in_sensitivity"]
].copy()

sensitivity_counts = (
    sensitivity_samples["condition"]
    .value_counts()
)

n_tumors = int(
    sensitivity_counts.get("Tumor", 0)
)

n_normals = int(
    sensitivity_counts.get("Normal", 0)
)

print("\n" + "-" * 65)
print("SENSITIVITY COHORT DEFINITION")
print("-" * 65)

print("\nRetained tumors:", n_tumors)

print("Retained normals:", n_normals)

print(
    "Total retained samples:",
    len(sensitivity_samples)
)

if n_tumors < 150 or n_normals < 20:

    raise RuntimeError(
        "Too few samples remain for the "
        "planned sensitivity comparison."
    )

if sensitivity_samples["patient_id"].duplicated().any():

    raise RuntimeError(
        "Repeated patient in sensitivity cohort."
    )

if sensitivity_samples["workflow_version"].nunique() != 1:

    raise RuntimeError(
        "Multiple workflow versions detected."
    )

print("\nSensitivity cohort validation: PASSED")


# ------------------------------------------------------------
# 7. Align matrix rows and columns
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING SENSITIVITY COUNT MATRIX")
print("-" * 65)

gene_index = pd.Index(gene_ids)

sample_index = pd.Index(file_ids)

gene_positions = gene_index.get_indexer(
    baseline["gene_id"].tolist()
)

sample_positions = sample_index.get_indexer(
    sensitivity_samples["file_id"].tolist()
)

if np.any(gene_positions < 0):

    raise RuntimeError(
        "Baseline genes missing from count matrix."
    )

if np.any(sample_positions < 0):

    raise RuntimeError(
        "Sensitivity samples missing from count matrix."
    )

# Extract original unmodified raw counts.
#
# Rows: genes
# Columns: retained samples

subset_counts = counts[
    np.ix_(
        gene_positions,
        sample_positions
    )
]

# Drop only genes with zero counts across
# all retained samples.
#
# This is necessary if a gene becomes entirely
# unexpressed after excluding QC-flagged samples.
#
# All remaining genes use the same baseline
# testing universe.

nonzero_gene_mask = (
    subset_counts.sum(axis=1) > 0
)

retained_gene_ids = (
    baseline["gene_id"]
    .to_numpy()[nonzero_gene_mask]
)

subset_counts = subset_counts[
    nonzero_gene_mask, :
]

print("\nBaseline gene universe:", len(baseline))

print(
    "Genes retained for sensitivity analysis:",
    len(retained_gene_ids)
)

print(
    "All-zero genes removed:",
    int((~nonzero_gene_mask).sum())
)

# PyDESeq2 expects samples x genes.

sensitivity_matrix = subset_counts.T

print(
    "\nSensitivity matrix:",
    sensitivity_matrix.shape
)

if np.any(sensitivity_matrix < 0):

    raise RuntimeError(
        "Negative raw counts detected."
    )


# ------------------------------------------------------------
# 8. Create PyDESeq2 input objects
# ------------------------------------------------------------

sample_ids = (
    sensitivity_samples["file_id"].tolist()
)

counts_df = pd.DataFrame(
    sensitivity_matrix,
    index=sample_ids,
    columns=retained_gene_ids
)

metadata = pd.DataFrame(
    {
        "condition":
            sensitivity_samples["condition"].tolist()
    },
    index=sample_ids
)

metadata["condition"] = pd.Categorical(
    metadata["condition"],
    categories=["Normal", "Tumor"],
    ordered=True
)

# Compatible string indexes for AnnData.

counts_df.index = pd.Index(
    counts_df.index,
    dtype="string"
)

counts_df.columns = pd.Index(
    counts_df.columns,
    dtype="string"
)

metadata.index = pd.Index(
    metadata.index,
    dtype="string"
)

if not counts_df.index.equals(metadata.index):

    raise RuntimeError(
        "Count and metadata indices are misaligned."
    )

print("\nPyDESeq2 input alignment: PASSED")


# ------------------------------------------------------------
# 9. Fit sensitivity differential expression model
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FITTING QC-SCREENED DIFFERENTIAL EXPRESSION")
print("=" * 65)

dds = DeseqDataSet(
    counts=counts_df,
    metadata=metadata,
    design="~condition",
    refit_cooks=True,
    n_cpus=2
)

dds.deseq2()

print("\nSensitivity model fitting: COMPLETE")


# ------------------------------------------------------------
# 10. Calculate sensitivity DE statistics
# ------------------------------------------------------------

stats = DeseqStats(
    dds,
    contrast=[
        "condition",
        "Tumor",
        "Normal"
    ],
    alpha=0.05,
    n_cpus=2
)

stats.summary()

sensitivity = (
    stats.results_df
    .copy()
    .rename_axis("gene_id")
    .reset_index()
)

if sensitivity["gene_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate genes in sensitivity results."
    )

print("\nSensitivity statistics: COMPLETE")


# ------------------------------------------------------------
# 11. Compare baseline and sensitivity results
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("BASELINE VS SENSITIVITY COMPARISON")
print("=" * 65)

baseline_comparison = baseline[
    [
        "gene_id",
        "gene_name",
        "log2FoldChange",
        "padj",
        "tier_a_median_cpm",
        "tier_a_fraction_cpm_ge1"
    ]
].copy()

baseline_comparison = baseline_comparison.rename(
    columns={
        "log2FoldChange":
            "baseline_log2fc",

        "padj":
            "baseline_padj"
    }
)

sensitivity_comparison = sensitivity[
    [
        "gene_id",
        "log2FoldChange",
        "padj"
    ]
].copy()

sensitivity_comparison = sensitivity_comparison.rename(
    columns={
        "log2FoldChange":
            "sensitivity_log2fc",

        "padj":
            "sensitivity_padj"
    }
)

comparison = baseline_comparison.merge(
    sensitivity_comparison,
    on="gene_id",
    how="left",
    validate="one_to_one"
)

comparison["baseline_upregulated"] = (
    comparison["baseline_padj"].lt(0.05)
    & comparison["baseline_log2fc"].gt(1)
)

comparison["sensitivity_upregulated"] = (
    comparison["sensitivity_padj"].lt(0.05)
    & comparison["sensitivity_log2fc"].gt(1)
)

comparison["tier_a_supported"] = (
    comparison["tier_a_fraction_cpm_ge1"].ge(0.50)
    & comparison["tier_a_median_cpm"].ge(1)
)

comparison["baseline_screening_candidate"] = (
    comparison["baseline_upregulated"]
    & comparison["tier_a_supported"]
)

comparison["robust_screening_candidate"] = (
    comparison["baseline_screening_candidate"]
    & comparison["sensitivity_upregulated"]
)

comparison["log2fc_difference"] = (
    comparison["sensitivity_log2fc"]
    - comparison["baseline_log2fc"]
)

# ------------------------------------------------------------
# 12. Calculate agreement statistics
# ------------------------------------------------------------

valid = comparison[
    comparison["baseline_log2fc"].notna()
    & comparison["sensitivity_log2fc"].notna()
].copy()

if len(valid) < 10000:

    raise RuntimeError(
        "Unexpectedly few comparable genes."
    )

rho, rho_pvalue = spearmanr(
    valid["baseline_log2fc"],
    valid["sensitivity_log2fc"]
)

baseline_up_count = int(
    comparison["baseline_upregulated"].sum()
)

sensitivity_up_count = int(
    comparison["sensitivity_upregulated"].sum()
)

baseline_candidate_count = int(
    comparison["baseline_screening_candidate"].sum()
)

robust_candidate_count = int(
    comparison["robust_screening_candidate"].sum()
)

stable_up_count = int(
    (
        comparison["baseline_upregulated"]
        & comparison["sensitivity_upregulated"]
    ).sum()
)

print("\nGenes compared:", len(valid))

print(
    "Spearman correlation of log2FC:",
    round(float(rho), 4)
)

print(
    "\nBaseline tumor-upregulated:",
    baseline_up_count
)

print(
    "Sensitivity tumor-upregulated:",
    sensitivity_up_count
)

print(
    "Baseline upregulated genes retaining "
    "significance and log2FC > 1:",
    stable_up_count
)

print(
    "\nOriginal Tier A-supported screening genes:",
    baseline_candidate_count
)

print(
    "Screening genes supported by both analyses:",
    robust_candidate_count
)


# ------------------------------------------------------------
# 13. Examine the original top 20 genes
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("TOP-GENE ROBUSTNESS REVIEW")
print("-" * 65)

top20 = comparison[
    comparison["baseline_screening_candidate"]
].sort_values(
    "baseline_padj"
).head(20).copy()

print("\nTop 20 original screening genes:")

print(
    top20[
        [
            "gene_name",
            "baseline_log2fc",
            "sensitivity_log2fc",
            "baseline_padj",
            "sensitivity_padj",
            "sensitivity_upregulated"
        ]
    ]
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4g}"
    )
)

print(
    "\nTop 20 retaining sensitivity significance:",
    int(top20["sensitivity_upregulated"].sum())
)


# ------------------------------------------------------------
# 14. Plot baseline vs sensitivity effect sizes
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENERATING ROBUSTNESS FIGURE")
print("-" * 65)

fig, ax = plt.subplots(
    figsize=(8.5, 8.5)
)

ax.scatter(
    valid["baseline_log2fc"],
    valid["sensitivity_log2fc"],
    s=9,
    alpha=0.22
)

limits = [
    min(
        valid["baseline_log2fc"].min(),
        valid["sensitivity_log2fc"].min()
    ),
    max(
        valid["baseline_log2fc"].max(),
        valid["sensitivity_log2fc"].max()
    )
]

ax.plot(
    limits,
    limits,
    linestyle="--",
    linewidth=1
)

ax.set_xlabel(
    "Baseline log2 fold change"
)

ax.set_ylabel(
    "QC-screened sensitivity log2 fold change"
)

ax.set_title(
    "TCGA-COAD — Differential Expression Robustness"
)

ax.grid(alpha=0.2)

fig.tight_layout()

figure_path = (
    FIGURES /
    "tcga_coad_qc_sensitivity_lfc_comparison.png"
)

fig.savefig(
    figure_path,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

plt.close(fig)

print("\nRobustness figure saved.")


# ------------------------------------------------------------
# 15. Save analysis tables
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING SENSITIVITY RESULTS")
print("-" * 65)

sensitivity.to_csv(
    RESULTS /
    "tcga_coad_qc_sensitivity_deseq2_results.csv",
    index=False
)

comparison.to_csv(
    RESULTS /
    "tcga_coad_qc_sensitivity_comparison.csv",
    index=False
)

discovery.to_csv(
    META /
    "tcga_coad_qc_sensitivity_sample_manifest.csv",
    index=False
)

top20.to_csv(
    RESULTS /
    "tcga_coad_qc_sensitivity_top20_review.csv",
    index=False
)

size_factors = pd.DataFrame({
    "file_id": dds.obs.index.astype(str),
    "size_factor": dds.obs[
        "size_factors"
    ].to_numpy()
})

size_factors.to_csv(
    RESULTS /
    "tcga_coad_qc_sensitivity_size_factors.csv",
    index=False
)


# ------------------------------------------------------------
# 16. Save research provenance
# ------------------------------------------------------------

summary = {

    "project": "SurfaceTarget-CRC",

    "patch": "1.21",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "analysis": (
        "QC-screened sensitivity analysis "
        "of COAD Tier B vs normal"
    ),

    "software": "PyDESeq2",

    "software_version": version("pydeseq2"),

    "baseline_tumors": 194,

    "baseline_normals": 29,

    "sensitivity_tumors": n_tumors,

    "sensitivity_normals": n_normals,

    "qc_flagged_samples_withheld": int(
        discovery["qc_flagged"].sum()
    ),

    "baseline_genes": int(
        len(baseline)
    ),

    "sensitivity_genes": int(
        len(sensitivity)
    ),

    "baseline_upregulated_genes": baseline_up_count,

    "sensitivity_upregulated_genes": sensitivity_up_count,

    "stable_upregulated_genes": stable_up_count,

    "baseline_screening_candidates":
        baseline_candidate_count,

    "robust_screening_candidates":
        robust_candidate_count,

    "log2fc_spearman": float(rho),

    "model": "~condition",

    "contrast": [
        "condition",
        "Tumor",
        "Normal"
    ],

    "cook_outlier_refitting": True,

    "baseline_results_overwritten": False,

    "source_matrix_modified": False,

    "limitations": [
        "QC flags are provisional.",
        "Sample removal is a sensitivity analysis.",
        "No clinical covariates are adjusted.",
        "Tumor purity and tissue composition may confound results.",
        "Workflow matching does not eliminate every technical bias.",
        "Tier B is not independently verified MSS.",
        "Tier A support is descriptive expression prevalence.",
        "Statistical robustness does not establish targetability.",
        "Cook's-distance refitting remains enabled.",
        "Patient-matched sensitivity analysis remains pending."
    ]
}

with open(
    META /
    "tcga_coad_qc_sensitivity_provenance.json",
    "w"
) as output:

    json.dump(
        summary,
        output,
        indent=4
    )

print("\nSensitivity results saved.")


# ------------------------------------------------------------
# 17. Create research checkpoint
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING RESEARCH CHECKPOINT")
print("-" * 65)

timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S")

checkpoint = (
    CHECKPOINTS /
    f"SurfaceTarget_CRC_PostDE_Patch121_{timestamp}.zip"
)

backup_folders = [
    "data/metadata",
    "data/processed",
    "results/tables",
    "results/figures",
    "src",
    "reports",
    "logs"
]

with ZipFile(
    checkpoint,
    "w",
    ZIP_DEFLATED,
    compresslevel=3
) as archive:

    for folder in backup_folders:

        folder_path = ROOT / folder

        if not folder_path.exists():
            continue

        for file_path in folder_path.rglob("*"):

            if file_path.is_file():

                archive.write(
                    file_path,
                    arcname=str(
                        file_path.relative_to(ROOT)
                    )
                )

checkpoint_mib = (
    checkpoint.stat().st_size
    / (1024 ** 2)
)

print("\nCheckpoint created.")

print(
    f"Checkpoint size: {checkpoint_mib:.2f} MiB"
)

print("Checkpoint location:", checkpoint)


# ------------------------------------------------------------
# 18. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.21 SUMMARY")
print("=" * 65)

print(
    "\nQC-flagged discovery samples withheld:",
    int(discovery["qc_flagged"].sum())
)

print("Retained tumors:", n_tumors)

print("Retained normals:", n_normals)

print("\nBaseline upregulated genes:", baseline_up_count)

print(
    "Sensitivity upregulated genes:",
    sensitivity_up_count
)

print(
    "Stable upregulated genes:",
    stable_up_count
)

print(
    "\nRobust Tier A-supported screening genes:",
    robust_candidate_count
)

print(
    "Log2FC Spearman correlation:",
    round(float(rho), 4)
)

print("\nSensitivity analysis: COMPLETE")

print("Research checkpoint: CREATED")

print("\nNo permanent sample exclusions.")

print("Original DE results preserved.")

print("\nPATCH 1.21 COMPLETE")

print("=" * 65)

In [ ]:
# ============================================================
# SurfaceTarget-CRC
# PHASE 1 — PATCH 1.22
#
# PATIENT-MATCHED COAD TUMOR–NORMAL ANALYSIS
#
# Objectives:
# 1. Identify Tier B patients with paired normal tissue
# 2. Validate patient and workflow correspondence
# 3. Fit a patient-adjusted differential expression model
# 4. Calculate descriptive within-patient expression changes
# 5. Evaluate robustness of existing screening candidates
# 6. Save reproducible results and provenance
#
# IMPORTANT:
# - This is a complementary analysis, not independent validation.
# - The paired cohort is small.
# - No samples or raw counts are permanently modified.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"
PROCESSED = ROOT / "data/processed"
RESULTS = ROOT / "results/tables"
FIGURES = ROOT / "results/figures"

RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

MATRIX_PATH = (
    PROCESSED / "tcga_unstranded_raw_counts.npz"
)

SAMPLES_PATH = (
    META / "tcga_expression_sample_metadata.csv"
)

GENES_PATH = (
    META / "tcga_gene_annotations.csv"
)

BASELINE_PATH = (
    RESULTS / "tcga_coad_qc_sensitivity_comparison.csv"
)

QC_PATH = (
    RESULTS / "tcga_full_cohort_qc.csv"
)

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.22 — PATIENT-MATCHED COAD ANALYSIS")
print("=" * 65)


# ------------------------------------------------------------
# 2. Validate dependencies and load data
# ------------------------------------------------------------

print("\nSoftware versions:")
print("PyDESeq2:", version("pydeseq2"))
print("Pandas:", pd.__version__)

if version("pydeseq2") != "0.5.4":
    raise RuntimeError("Unexpected PyDESeq2 version.")

for path in [
    MATRIX_PATH,
    SAMPLES_PATH,
    GENES_PATH,
    BASELINE_PATH,
    QC_PATH
]:
    if not path.exists():
        raise FileNotFoundError(path)

with np.load(MATRIX_PATH, allow_pickle=False) as archive:
    counts = archive["counts"]
    gene_ids = archive["gene_ids"]
    file_ids = archive["file_ids"]

samples = pd.read_csv(
    SAMPLES_PATH,
    dtype=str,
    keep_default_na=False
)

genes = pd.read_csv(
    GENES_PATH,
    dtype=str,
    keep_default_na=False
)

baseline = pd.read_csv(BASELINE_PATH)

qc = pd.read_csv(
    QC_PATH,
    dtype=str,
    keep_default_na=False
)

if counts.shape != (60660, 472):
    raise RuntimeError("Unexpected matrix shape.")

if not np.array_equal(file_ids, samples["file_id"].to_numpy()):
    raise RuntimeError("Sample identifiers are misaligned.")

if not np.array_equal(gene_ids, genes["gene_id"].to_numpy()):
    raise RuntimeError("Gene identifiers are misaligned.")

if baseline["gene_id"].duplicated().any():
    raise RuntimeError("Duplicate baseline genes.")

print("\nData loaded and validated.")


# ------------------------------------------------------------
# 3. Identify the shared workflow
# ------------------------------------------------------------

normal_workflows = samples.loc[
    samples["dataset_group"].eq("Normal"),
    "workflow_version"
].unique()

if len(normal_workflows) != 1:
    raise RuntimeError("Unexpected normal workflow structure.")

SHARED_WORKFLOW = normal_workflows[0]

coad = samples[
    samples["project"].eq("TCGA-COAD")
    & samples["workflow_version"].eq(SHARED_WORKFLOW)
].copy()

tumors = coad[
    coad["dataset_group"].eq("Tumor")
    & coad["cohort_tier"].eq("Tier B")
].copy()

normals = coad[
    coad["dataset_group"].eq("Normal")
].copy()

paired_patients = sorted(
    set(tumors["patient_id"])
    & set(normals["patient_id"])
)

print("\n" + "-" * 65)
print("MATCHED PATIENT IDENTIFICATION")
print("-" * 65)

print("\nTier B tumors:", len(tumors))
print("Available normals:", len(normals))
print("Patients with both:", len(paired_patients))

if len(paired_patients) != 12:
    raise RuntimeError(
        "Expected 12 matched Tier B COAD patients. "
        "Review the cohort before proceeding."
    )


# ------------------------------------------------------------
# 4. Construct paired sample manifest
# ------------------------------------------------------------

pair_records = []

for patient_id in paired_patients:

    tumor_rows = tumors[
        tumors["patient_id"].eq(patient_id)
    ]

    normal_rows = normals[
        normals["patient_id"].eq(patient_id)
    ]

    if len(tumor_rows) != 1 or len(normal_rows) != 1:
        raise RuntimeError(
            f"Ambiguous paired samples: {patient_id}"
        )

    tumor = tumor_rows.iloc[0]
    normal = normal_rows.iloc[0]

    pair_records.append({
        "patient_id": patient_id,
        "tumor_file_id": tumor["file_id"],
        "normal_file_id": normal["file_id"],
        "tumor_sample_id": tumor["sample_id"],
        "normal_sample_id": normal["sample_id"],
        "workflow_version": SHARED_WORKFLOW
    })

pairs = pd.DataFrame(pair_records)

if pairs["patient_id"].duplicated().any():
    raise RuntimeError("Duplicate matched patients.")

pairs_path = (
    META / "tcga_coad_tierb_matched_pairs.csv"
)

pairs.to_csv(pairs_path, index=False)

print("\nMatched pairs:")
print(
    pairs[
        ["patient_id", "tumor_sample_id", "normal_sample_id"]
    ].to_string(index=False)
)

print("\nMatched-patient validation: PASSED")


# ------------------------------------------------------------
# 5. Prepare sample ordering
# ------------------------------------------------------------

# Each patient contributes:
# Normal sample followed by tumor sample.

ordered_file_ids = []

conditions = []

patient_labels = []

for _, row in pairs.iterrows():

    ordered_file_ids.extend([
        row["normal_file_id"],
        row["tumor_file_id"]
    ])

    conditions.extend(["Normal", "Tumor"])

    patient_labels.extend([
        row["patient_id"],
        row["patient_id"]
    ])

file_lookup = {
    file_id: index
    for index, file_id in enumerate(file_ids)
}

sample_positions = np.array([
    file_lookup[fid]
    for fid in ordered_file_ids
])

if len(sample_positions) != 24:
    raise RuntimeError("Unexpected paired sample count.")

if len(set(ordered_file_ids)) != 24:
    raise RuntimeError("Repeated RNA-seq file in paired analysis.")

print("\nPaired samples prepared:", len(sample_positions))


# ------------------------------------------------------------
# 6. Check existing QC flags
# ------------------------------------------------------------

qc_lookup = qc.set_index("file_id")

paired_qc = qc_lookup.loc[
    ordered_file_ids
].copy()

paired_qc["n_qc_review_flags"] = pd.to_numeric(
    paired_qc["n_qc_review_flags"],
    errors="raise"
)

n_flagged = int(
    (paired_qc["n_qc_review_flags"] > 0).sum()
)

print("\nPreviously QC-flagged paired samples:", n_flagged)

print(
    "No samples will be excluded from this paired analysis."
)


# ------------------------------------------------------------
# 7. Prepare protein-coding gene counts
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING PAIRED EXPRESSION MATRIX")
print("-" * 65)

coding_mask = (
    genes["gene_type"]
    .eq("protein_coding")
    .to_numpy(dtype=bool)
)

coding_gene_ids = gene_ids[coding_mask]

paired_counts = counts[
    coding_mask, :
][:, sample_positions]

# Library sizes from all annotated genes.

library_sizes = counts[
    :, sample_positions
].sum(axis=0)

if np.any(library_sizes <= 0):
    raise RuntimeError("Zero library size detected.")

paired_cpm = (
    paired_counts.astype(np.float64)
    / library_sizes[np.newaxis, :]
    * 1_000_000
)

# Exploratory expression filter:
# CPM >= 1 in at least 4 of 24 samples.

expressed_mask = (
    np.count_nonzero(
        paired_cpm >= 1,
        axis=1
    ) >= 4
)

selected_gene_ids = coding_gene_ids[
    expressed_mask
]

selected_counts = paired_counts[
    expressed_mask, :
]

selected_cpm = paired_cpm[
    expressed_mask, :
]

if len(selected_gene_ids) < 5000:
    raise RuntimeError(
        "Unexpectedly few expressed genes."
    )

print("\nProtein-coding genes:", len(coding_gene_ids))
print("Genes passing paired expression filter:", len(selected_gene_ids))
print("Paired samples:", selected_counts.shape[1])


# ------------------------------------------------------------
# 8. Descriptive within-patient expression changes
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("WITHIN-PATIENT EXPRESSION ASSESSMENT")
print("-" * 65)

# Sample order:
# Normal, Tumor, Normal, Tumor, ...
#
# Calculate log2((tumor CPM + 1) / (normal CPM + 1)).
#
# This is descriptive and not the DESeq2 model estimate.

normal_cpm = selected_cpm[:, 0::2]
tumor_cpm = selected_cpm[:, 1::2]

paired_log2_ratio = np.log2(
    (tumor_cpm + 1.0)
    / (normal_cpm + 1.0)
)

descriptive = pd.DataFrame({

    "gene_id": selected_gene_ids,

    "paired_median_log2_cpm_ratio":
        np.median(paired_log2_ratio, axis=1),

    "paired_fraction_tumor_higher":
        np.mean(paired_log2_ratio > 0, axis=1),

    "paired_fraction_log2ratio_gt1":
        np.mean(paired_log2_ratio > 1, axis=1),

    "paired_median_tumor_cpm":
        np.median(tumor_cpm, axis=1),

    "paired_median_normal_cpm":
        np.median(normal_cpm, axis=1)
})

print("\nDescriptive paired expression metrics calculated.")


# ------------------------------------------------------------
# 9. Prepare PyDESeq2 inputs
# ------------------------------------------------------------

counts_df = pd.DataFrame(
    selected_counts.T,
    index=ordered_file_ids,
    columns=selected_gene_ids
)

metadata = pd.DataFrame({

    "patient_id": patient_labels,

    "condition": conditions

}, index=ordered_file_ids)

metadata["patient_id"] = pd.Categorical(
    metadata["patient_id"],
    categories=paired_patients
)

metadata["condition"] = pd.Categorical(
    metadata["condition"],
    categories=["Normal", "Tumor"],
    ordered=True
)

# Preserve AnnData compatibility.

counts_df.index = pd.Index(
    counts_df.index,
    dtype="string"
)

counts_df.columns = pd.Index(
    counts_df.columns,
    dtype="string"
)

metadata.index = pd.Index(
    metadata.index,
    dtype="string"
)

if not counts_df.index.equals(metadata.index):
    raise RuntimeError("PyDESeq2 input alignment failed.")

# Confirm every patient contributes exactly
# one normal and one tumor sample.

pair_check = pd.crosstab(
    metadata["patient_id"],
    metadata["condition"]
)

if not (
    pair_check["Normal"].eq(1).all()
    and pair_check["Tumor"].eq(1).all()
):
    raise RuntimeError("Invalid paired design.")

print("\nPaired model input validation: PASSED")
print("Model input dimensions:", counts_df.shape)


# ------------------------------------------------------------
# 10. Fit patient-adjusted differential expression
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FITTING PATIENT-ADJUSTED DESEQ2 MODEL")
print("=" * 65)

print("\nDesign: ~ patient_id + condition")
print("Contrast: Tumor versus Normal")

# Disable Cook's outlier replacement because
# each patient-condition combination has only
# one sample in this small paired dataset.

dds = DeseqDataSet(
    counts=counts_df,
    metadata=metadata,
    design="~ patient_id + condition",
    refit_cooks=False,
    n_cpus=2
)

dds.deseq2()

print("\nPaired model fitting: COMPLETE")


# ------------------------------------------------------------
# 11. Calculate paired DE statistics
# ------------------------------------------------------------

stats = DeseqStats(
    dds,
    contrast=["condition", "Tumor", "Normal"],
    alpha=0.05,
    n_cpus=2
)

stats.summary()

paired_results = (
    stats.results_df
    .copy()
    .rename_axis("gene_id")
    .reset_index()
)

if paired_results["gene_id"].duplicated().any():
    raise RuntimeError("Duplicate paired-result genes.")

paired_results = paired_results.merge(
    descriptive,
    on="gene_id",
    how="left",
    validate="one_to_one"
)

paired_results = paired_results.merge(
    genes[["gene_id", "gene_name"]],
    on="gene_id",
    how="left",
    validate="one_to_one"
)

paired_results["paired_upregulated"] = (
    paired_results["padj"].lt(0.05)
    & paired_results["log2FoldChange"].gt(1)
)

print("\nPaired DE statistics: COMPLETE")


# ------------------------------------------------------------
# 12. Compare with existing screening candidates
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("CROSS-ANALYSIS GENE SUPPORT")
print("=" * 65)

required_baseline_columns = {
    "gene_id",
    "gene_name",
    "baseline_log2fc",
    "robust_screening_candidate"
}

if not required_baseline_columns.issubset(
    baseline.columns
):
    raise RuntimeError(
        "Unexpected baseline sensitivity table format."
    )

baseline["robust_screening_candidate"] = (
    baseline["robust_screening_candidate"]
    .astype(str)
    .str.lower()
    .eq("true")
)

comparison = baseline.merge(
    paired_results[
        [
            "gene_id",
            "log2FoldChange",
            "padj",
            "paired_median_log2_cpm_ratio",
            "paired_fraction_tumor_higher",
            "paired_fraction_log2ratio_gt1",
            "paired_upregulated"
        ]
    ],
    on="gene_id",
    how="left",
    validate="one_to_one"
)

comparison = comparison.rename(
    columns={
        "log2FoldChange": "paired_model_log2fc",
        "padj": "paired_model_padj"
    }
)

# Descriptive support requires:
# - Tumor CPM greater than normal in >= 75% of pairs
# - Median paired log2 CPM ratio > 1

comparison["paired_direction_supported"] = (
    comparison["paired_fraction_tumor_higher"].ge(0.75)
    & comparison["paired_median_log2_cpm_ratio"].gt(1)
)

comparison["combined_transcriptomic_support"] = (
    comparison["robust_screening_candidate"]
    & comparison["paired_direction_supported"]
)

robust_candidates = comparison[
    comparison["robust_screening_candidate"]
].copy()

supported_candidates = robust_candidates[
    robust_candidates["paired_direction_supported"]
].copy()

print("\nPreviously robust screening genes:", len(robust_candidates))

print(
    "Genes with additional paired directional support:",
    len(supported_candidates)
)

print(
    "Genes significant in the paired model:",
    int(paired_results["paired_upregulated"].sum())
)

print("\nPaired directional support is descriptive,")
print("not an independent statistical validation.")


# ------------------------------------------------------------
# 13. Examine original leading genes
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("TOP 20 GENE SUPPORT")
print("-" * 65)

top20 = robust_candidates.sort_values(
    "baseline_padj"
).head(20).copy()

display_columns = [
    "gene_name",
    "baseline_log2fc",
    "paired_model_log2fc",
    "paired_model_padj",
    "paired_median_log2_cpm_ratio",
    "paired_fraction_tumor_higher",
    "paired_direction_supported"
]

print(
    top20[display_columns]
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4g}"
    )
)


# ------------------------------------------------------------
# 14. Generate comparison figure
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENERATING MATCHED-PAIR FIGURE")
print("-" * 65)

plot_data = robust_candidates.dropna(
    subset=[
        "baseline_log2fc",
        "paired_model_log2fc"
    ]
)

fig, ax = plt.subplots(figsize=(8, 7))

ax.scatter(
    plot_data["baseline_log2fc"],
    plot_data["paired_model_log2fc"],
    s=12,
    alpha=0.4
)

ax.axhline(0, linestyle="--", linewidth=0.8)
ax.axvline(0, linestyle="--", linewidth=0.8)

ax.set_xlabel("Original discovery log2 fold change")
ax.set_ylabel("Patient-adjusted paired log2 fold change")

ax.set_title(
    "TCGA-COAD — Matched-Pair Expression Support"
)

ax.grid(alpha=0.2)

fig.tight_layout()

figure_path = (
    FIGURES / "tcga_coad_matched_pair_expression_support.png"
)

fig.savefig(
    figure_path,
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print("\nMatched-pair figure saved.")


# ------------------------------------------------------------
# 15. Save results
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING MATCHED-PAIR RESULTS")
print("-" * 65)

paired_results.to_csv(
    RESULTS / "tcga_coad_paired_deseq2_results.csv",
    index=False
)

comparison.to_csv(
    RESULTS / "tcga_coad_paired_support_comparison.csv",
    index=False
)

supported_candidates.to_csv(
    RESULTS / "tcga_coad_paired_supported_candidates.csv",
    index=False
)

top20.to_csv(
    RESULTS / "tcga_coad_paired_top20_review.csv",
    index=False
)

metadata.reset_index().rename(
    columns={"index": "file_id"}
).to_csv(
    META / "tcga_coad_paired_model_metadata.csv",
    index=False
)


# ------------------------------------------------------------
# 16. Save provenance
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "1.22",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "analysis": "Patient-matched Tier B COAD tumor-normal",

    "software": "PyDESeq2",

    "software_version": version("pydeseq2"),

    "pandas_version": pd.__version__,

    "patients": len(paired_patients),

    "tumor_samples": len(paired_patients),

    "normal_samples": len(paired_patients),

    "workflow_version": SHARED_WORKFLOW,

    "design": "~ patient_id + condition",

    "contrast": ["condition", "Tumor", "Normal"],

    "count_column": "unstranded",

    "expression_filter": (
        "CPM >= 1 in at least 4 of 24 samples"
    ),

    "genes_tested": len(paired_results),

    "refit_cooks": False,

    "qc_flagged_samples_included": n_flagged,

    "previously_robust_candidates": len(
        robust_candidates
    ),

    "paired_direction_supported_candidates": len(
        supported_candidates
    ),

    "limitations": [
        "Only 12 matched patients are available.",
        "This is not an independent validation cohort.",
        "Normal samples may be tumor-adjacent.",
        "Patient matching does not eliminate tissue-composition effects.",
        "Small-sample statistical estimates may be unstable.",
        "No clinical covariates beyond patient identity are included.",
        "CPM ratios are descriptive.",
        "No therapeutic conclusions are established."
    ]
}

with open(
    META / "tcga_coad_paired_analysis_provenance.json",
    "w"
) as f:
    json.dump(provenance, f, indent=4)


# ------------------------------------------------------------
# 17. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.22 SUMMARY")
print("=" * 65)

print("\nMatched patients:", len(paired_patients))

print("Paired RNA-seq samples:", len(sample_positions))

print("Genes tested:", len(paired_results))

print(
    "Previously robust screening candidates:",
    len(robust_candidates)
)

print(
    "Candidates with paired directional support:",
    len(supported_candidates)
)

print("\nFiles saved:")
print("1. tcga_coad_tierb_matched_pairs.csv")
print("2. tcga_coad_paired_deseq2_results.csv")
print("3. tcga_coad_paired_support_comparison.csv")
print("4. tcga_coad_paired_supported_candidates.csv")
print("5. tcga_coad_paired_top20_review.csv")
print("6. tcga_coad_paired_model_metadata.csv")
print("7. tcga_coad_paired_analysis_provenance.json")

print("\nFigure saved:")
print("tcga_coad_matched_pair_expression_support.png")

print("\nNo raw counts modified.")
print("Original analyses preserved.")
print("No permanent sample exclusions.")

print("\nPATCH 1.22 COMPLETE")
print("=" * 65)

In [ ]:
# ============================================================
# SurfaceTarget-CRC
# PHASE 2 — PATCH 2.1
#
# CPTAC COLON PROTEOMICS ACQUISITION AND AUDIT
#
# Objectives:
# 1. Download processed CPTAC colon proteomics datasets
# 2. Verify downloaded file structure
# 3. Record file integrity checksums
# 4. Load CPTAC clinical annotations
# 5. Identify published MSS and MSI-H classifications
# 6. Assess protein measurement availability
# 7. Evaluate transcriptomic candidate coverage
# 8. Save reproducible source and audit metadata
#
# IMPORTANT:
# - No proteomic differential expression yet
# - No protein abundance imputation
# - No merging of TCGA and CPTAC patient IDs
# - Missing protein measurements are not zeros
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import requests

import hashlib
import json
import time
import os


# ------------------------------------------------------------
# 1. Project configuration
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"

RAW = ROOT / "data/raw/cptac_colon"

RESULTS = ROOT / "results/tables"

for directory in [META, RAW, RESULTS]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

CANDIDATE_PATH = (
    RESULTS /
    "tcga_coad_paired_support_comparison.csv"
)

print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 2 — PATCH 2.1")
print("CPTAC COLON PROTEOMICS DATA ACQUISITION")
print("=" * 65)


# ------------------------------------------------------------
# 2. Define source datasets
# ------------------------------------------------------------

# Source:
# CPTAC Prospective Colon Cancer Study
# NCI Proteomic Data Commons: PDC000116
#
# Processed matrices distributed through LinkedOmics.
#
# Tumor and normal unshared TMT log-ratio matrices
# are retained separately.
#
# The paired tumor-normal matrix contains
# reported within-patient protein log2 ratios.
#
# We will verify cohort membership and
# measurement availability before analysis.

BASE_URL = (
    "https://linkedomics.org/cptac-colon/"
)

SOURCE_FILES = {

    "tumor_proteomics": (
        "Human__CPTAC_COAD__PNNL__Proteome__TMT__"
        "03_01_2017__BCM__Gene__"
        "PNNL_Tumor_TMT_UnsharedLogRatio.cct"
    ),

    "normal_proteomics": (
        "Human__CPTAC_COAD__PNNL__Proteome__TMT__"
        "03_01_2017__BCM__Gene__"
        "PNNL_Normal_TMT_UnsharedLogRatio.cct"
    ),

    "paired_proteomics": (
        "Human__CPTAC_COAD__PNNL__Proteome__TMT__"
        "03_01_2017__BCM__Gene__"
        "Tumor_Normal_log2FC.cct"
    ),

    "clinical": (
        "Human__CPTAC_COAD__MS__Clinical__Clinical__"
        "03_01_2017__CPTAC__Clinical__BCM.tsi"
    )
}

SOURCE_URLS = {
    name: BASE_URL + filename
    for name, filename in SOURCE_FILES.items()
}

print("\nStudy: CPTAC Prospective Colon Cancer")
print("PDC study identifier: PDC000116")
print("Processed-data source: LinkedOmics")
print("Datasets requested:", len(SOURCE_FILES))


# ------------------------------------------------------------
# 3. Create download session
# ------------------------------------------------------------

session = requests.Session()

session.headers.update({
    "User-Agent": "SurfaceTarget-CRC-Research/1.0",
    "Accept": "*/*"
})


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            digest.update(chunk)

    return digest.hexdigest()


def valid_matrix_header(path):

    if not path.exists():
        return False

    if path.stat().st_size < 1000:
        return False

    with open(
        path,
        "rb"
    ) as f:

        first_line = f.readline(
            1024 * 1024
        )

    return first_line.startswith(
        b"attrib_name\t"
    )


def download_dataset(
    url,
    destination,
    max_attempts=3
):

    if valid_matrix_header(destination):

        print("Previously downloaded file found.")

        return "Existing file"

    temporary = Path(
        str(destination) + ".part"
    )

    for attempt in range(
        1,
        max_attempts + 1
    ):

        try:

            if temporary.exists():
                temporary.unlink()

            with session.get(
                url,
                stream=True,
                timeout=(30, 120)
            ) as response:

                response.raise_for_status()

                downloaded_bytes = 0

                with open(
                    temporary,
                    "wb"
                ) as output:

                    for chunk in response.iter_content(
                        chunk_size=1024 * 1024
                    ):

                        if chunk:

                            output.write(chunk)

                            downloaded_bytes += len(chunk)

                            if downloaded_bytes > 40 * 1024**2:
                                raise RuntimeError(
                                    "Unexpectedly large dataset."
                                )

            if not valid_matrix_header(temporary):

                raise RuntimeError(
                    "Downloaded file is not a valid "
                    "LinkedOmics matrix."
                )

            os.replace(
                temporary,
                destination
            )

            return "Downloaded"

        except Exception as error:

            print(
                f"Attempt {attempt} failed: {error}"
            )

            if temporary.exists():
                temporary.unlink()

            if attempt == max_attempts:
                raise

            time.sleep(
                2 ** attempt
            )


# ------------------------------------------------------------
# 4. Download source files
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("DOWNLOADING CPTAC DATASETS")
print("-" * 65)

source_records = []

for name, filename in SOURCE_FILES.items():

    print("\nDataset:", name)

    destination = RAW / filename

    status = download_dataset(
        SOURCE_URLS[name],
        destination
    )

    source_records.append({

        "dataset": name,

        "filename": filename,

        "source_url": SOURCE_URLS[name],

        "local_path": str(destination),

        "download_status": status,

        "file_size_bytes": destination.stat().st_size,

        "sha256": sha256_file(destination)
    })

    print("Status:", status)

    print(
        "Size:",
        round(
            destination.stat().st_size / 1024**2,
            2
        ),
        "MiB"
    )

print("\nAll source files acquired.")


# ------------------------------------------------------------
# 5. Load quantitative proteomics matrices
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING PROTEOMICS MATRICES")
print("-" * 65)


def load_proteomics(name):

    path = RAW / SOURCE_FILES[name]

    raw = pd.read_csv(
        path,
        sep="\t",
        index_col=0,
        na_values=["NA", "N/A", ""],
        low_memory=False
    )

    raw.index = (
        raw.index.astype(str).str.strip()
    )

    raw.columns = (
        raw.columns.astype(str).str.strip()
    )

    if raw.index.isna().any():
        raise RuntimeError(
            f"Missing protein identifiers: {name}"
        )

    if raw.columns.duplicated().any():
        raise RuntimeError(
            f"Duplicate sample identifiers: {name}"
        )

    # Convert to numeric while detecting
    # unexpected nonnumeric entries.

    numeric = raw.apply(
        pd.to_numeric,
        errors="coerce"
    )

    invalid = (
        raw.notna() & numeric.isna()
    )

    if invalid.to_numpy().any():

        raise RuntimeError(
            f"Unexpected nonnumeric values: {name}"
        )

    if numeric.shape[0] < 3000:
        raise RuntimeError(
            f"Unexpectedly few proteins: {name}"
        )

    if numeric.shape[1] < 50:
        raise RuntimeError(
            f"Unexpectedly few samples: {name}"
        )

    return numeric


proteomics = {

    "tumor_proteomics":
        load_proteomics("tumor_proteomics"),

    "normal_proteomics":
        load_proteomics("normal_proteomics"),

    "paired_proteomics":
        load_proteomics("paired_proteomics")
}

for name, matrix in proteomics.items():

    print("\nDataset:", name)

    print(
        "Dimensions:",
        matrix.shape
    )

    print(
        "Unique gene symbols:",
        matrix.index.nunique()
    )

    print(
        "Duplicated gene-symbol rows:",
        int(matrix.index.duplicated().sum())
    )

    print(
        "Missing measurement percentage:",
        round(
            100 * matrix.isna().to_numpy().mean(),
            2
        )
    )


# ------------------------------------------------------------
# 6. Load CPTAC clinical metadata
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING CPTAC CLINICAL METADATA")
print("-" * 65)

clinical_path = (
    RAW / SOURCE_FILES["clinical"]
)

clinical_raw = pd.read_csv(
    clinical_path,
    sep="\t",
    index_col=0,
    na_values=["NA", "N/A", ""],
    low_memory=False
)

# LinkedOmics clinical file orientation:
#
# Rows = clinical attributes
# Columns = patients
#
# Transpose to patients x attributes.

clinical = clinical_raw.T.copy()

clinical.index = (
    clinical.index.astype(str).str.strip()
)

clinical.index.name = "cptac_patient_id"

if clinical.index.duplicated().any():

    raise RuntimeError(
        "Duplicate CPTAC clinical patient IDs."
    )

if "Mutation_Phenotype" not in clinical.columns:

    raise RuntimeError(
        "Published MSI classification "
        "not found in CPTAC clinical metadata."
    )

clinical["Mutation_Phenotype"] = (
    clinical["Mutation_Phenotype"]
    .astype("string")
    .str.strip()
    .str.upper()
)

allowed_statuses = {
    "MSS",
    "MSI-H"
}

observed_statuses = set(
    clinical["Mutation_Phenotype"].dropna()
)

unexpected = (
    observed_statuses - allowed_statuses
)

if unexpected:

    print(
        "WARNING: Additional phenotype labels:",
        unexpected
    )

print("\nClinical patients:", len(clinical))

print("\nPublished mutation phenotype:")

print(
    clinical["Mutation_Phenotype"]
    .fillna("NOT AVAILABLE")
    .value_counts()
    .to_string()
)


# ------------------------------------------------------------
# 7. Validate sample correspondence
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PROTEOMICS / CLINICAL SAMPLE CORRESPONDENCE")
print("-" * 65)

tumor_ids = set(
    proteomics["tumor_proteomics"].columns
)

normal_ids = set(
    proteomics["normal_proteomics"].columns
)

paired_ids = set(
    proteomics["paired_proteomics"].columns
)

clinical_ids = set(
    clinical.index
)

print("\nTumor proteomics patients:", len(tumor_ids))

print("Normal proteomics patients:", len(normal_ids))

print("Paired ratio patients:", len(paired_ids))

print("Clinical patients:", len(clinical_ids))

print(
    "\nTumor/normal identifier overlap:",
    len(tumor_ids & normal_ids)
)

print(
    "Paired ratio/clinical overlap:",
    len(paired_ids & clinical_ids)
)

print(
    "All three protein matrices overlap:",
    len(tumor_ids & normal_ids & paired_ids)
)

# Sample correspondence is audited,
# not assumed to be complete.


# ------------------------------------------------------------
# 8. Identify CPTAC MSS samples
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CPTAC MSS COHORT IDENTIFICATION")
print("-" * 65)

paired = proteomics["paired_proteomics"]

paired_metadata = clinical.reindex(
    paired.columns
).copy()

paired_metadata["clinical_match"] = (
    paired_metadata.index.isin(
        clinical.index
    )
)

paired_metadata["msi_classification"] = (
    paired_metadata["Mutation_Phenotype"]
    .fillna("NOT AVAILABLE")
)

paired_distribution = (
    paired_metadata["msi_classification"]
    .value_counts()
)

print("\nPaired proteomics by published MSI status:")

print(
    paired_distribution.to_string()
)

mss_ids = (
    paired_metadata.index[
        paired_metadata[
            "msi_classification"
        ].eq("MSS")
    ].tolist()
)

msih_ids = (
    paired_metadata.index[
        paired_metadata[
            "msi_classification"
        ].eq("MSI-H")
    ].tolist()
)

print("\nPublished MSS paired patients:", len(mss_ids))

print("Published MSI-H paired patients:", len(msih_ids))

if len(mss_ids) < 30:

    raise RuntimeError(
        "Unexpectedly few published MSS "
        "samples in paired proteomics data."
    )

print("\nCPTAC MSS cohort identification: PASSED")


# ------------------------------------------------------------
# 9. Validate transcriptomic screening genes
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("TRANSCRIPTOMIC CANDIDATE COVERAGE")
print("-" * 65)

if not CANDIDATE_PATH.exists():

    raise FileNotFoundError(
        "Patch 1.22 candidate comparison "
        "table not found."
    )

candidates = pd.read_csv(
    CANDIDATE_PATH,
    dtype={"gene_id": str},
    low_memory=False
)

required_candidate_columns = {
    "gene_id",
    "gene_name",
    "combined_transcriptomic_support"
}

if not required_candidate_columns.issubset(
    candidates.columns
):

    raise RuntimeError(
        "Required screening candidate "
        "columns are missing."
    )

supported_mask = (
    candidates["combined_transcriptomic_support"]
    .astype(str)
    .str.lower()
    .eq("true")
)

supported = candidates[
    supported_mask
].copy()

print(
    "\nTranscriptomic screening genes:",
    len(supported)
)

if len(supported) != 948:

    raise RuntimeError(
        "Transcriptomic screening gene count "
        "differs from Patch 1.22."
    )

supported["gene_name"] = (
    supported["gene_name"]
    .astype(str)
    .str.strip()
)

# Coverage counts report whether a gene
# symbol appears in the protein assay.
#
# They do not establish protein enrichment.

paired_gene_counts = (
    paired.index.value_counts()
)

tumor_gene_set = set(
    proteomics["tumor_proteomics"].index
)

normal_gene_set = set(
    proteomics["normal_proteomics"].index
)

paired_gene_set = set(
    paired.index
)

supported["in_tumor_proteomics"] = (
    supported["gene_name"].isin(
        tumor_gene_set
    )
)

supported["in_normal_proteomics"] = (
    supported["gene_name"].isin(
        normal_gene_set
    )
)

supported["in_paired_proteomics"] = (
    supported["gene_name"].isin(
        paired_gene_set
    )
)

supported["paired_symbol_row_count"] = (
    supported["gene_name"]
    .map(paired_gene_counts)
    .fillna(0)
    .astype(int)
)

# We deliberately do not collapse duplicate
# protein identifiers in this acquisition patch.

print("\nProtein assay coverage:")

for column in [
    "in_tumor_proteomics",
    "in_normal_proteomics",
    "in_paired_proteomics"
]:

    print(
        column,
        ":",
        int(supported[column].sum()),
        "/",
        len(supported)
    )

print(
    "\nCandidates with ambiguous duplicate "
    "paired protein-symbol rows:",
    int(
        (
            supported["paired_symbol_row_count"] > 1
        ).sum()
    )
)


# ------------------------------------------------------------
# 10. Inspect leading transcriptomic genes
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LEADING GENE PROTEOMICS COVERAGE")
print("-" * 65)

top_genes = [
    "CDH3",
    "CLDN1",
    "KRT80",
    "ETV4",
    "FOXQ1",
    "ESM1",
    "CEMIP",
    "INHBA",
    "SIM2",
    "WNT2"
]

top_coverage = supported[
    supported["gene_name"].isin(
        top_genes
    )
].copy()

top_coverage["gene_name"] = pd.Categorical(
    top_coverage["gene_name"],
    categories=top_genes,
    ordered=True
)

top_coverage = (
    top_coverage
    .sort_values("gene_name")
)

print("\n")

print(
    top_coverage[
        [
            "gene_name",
            "in_tumor_proteomics",
            "in_normal_proteomics",
            "in_paired_proteomics",
            "paired_symbol_row_count"
        ]
    ].to_string(index=False)
)

print("\nIMPORTANT:")
print(
    "Protein measurement availability "
    "is not evidence of tumor enrichment."
)


# ------------------------------------------------------------
# 11. Save CPTAC dataset audit
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING CPTAC AUDIT RESULTS")
print("-" * 65)

source_manifest = pd.DataFrame(
    source_records
)

source_manifest.to_csv(
    META / "cptac_colon_source_manifest.csv",
    index=False
)

clinical.reset_index().to_csv(
    META / "cptac_colon_clinical_metadata.csv",
    index=False
)

paired_metadata.reset_index().to_csv(
    META / "cptac_colon_paired_sample_metadata.csv",
    index=False
)

supported.to_csv(
    RESULTS / "cptac_transcriptomic_candidate_coverage.csv",
    index=False
)

top_coverage.to_csv(
    RESULTS / "cptac_top_gene_protein_coverage.csv",
    index=False
)

inventory_records = []

for name, matrix in proteomics.items():

    inventory_records.append({

        "dataset": name,

        "n_gene_rows": len(matrix),

        "n_unique_gene_symbols":
            matrix.index.nunique(),

        "n_patient_columns":
            len(matrix.columns),

        "missing_measurements": int(
            matrix.isna().to_numpy().sum()
        ),

        "missing_percent": float(
            100 * matrix.isna().to_numpy().mean()
        ),

        "duplicate_gene_symbol_rows": int(
            matrix.index.duplicated().sum()
        )
    })

inventory_df = pd.DataFrame(
    inventory_records
)

inventory_df.to_csv(
    RESULTS / "cptac_colon_proteomics_inventory.csv",
    index=False
)

print("\nCPTAC metadata and audit tables saved.")


# ------------------------------------------------------------
# 12. Save reproducibility metadata
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "2.1",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "study": "CPTAC Prospective Colon Cancer",

    "pdc_study_id": "PDC000116",

    "processed_data_source": "LinkedOmics",

    "dataset_sources": SOURCE_URLS,

    "source_file_sha256": {
        record["dataset"]: record["sha256"]
        for record in source_records
    },

    "protein_measurement_type": (
        "TMT-based relative protein abundance "
        "and reported matched tumor-normal log2 ratios"
    ),

    "paired_proteomics_patients": int(
        len(paired_ids)
    ),

    "published_mss_paired_patients": int(
        len(mss_ids)
    ),

    "published_msih_paired_patients": int(
        len(msih_ids)
    ),

    "transcriptomic_screening_genes": int(
        len(supported)
    ),

    "protein_differential_expression_performed": False,

    "protein_imputation_performed": False,

    "tcga_cptac_patient_level_merging": False,

    "limitations": [
        "CPTAC prospective colon and TCGA are separate cohorts.",
        "Published MSS annotations must be retained.",
        "TMT protein measurements are relative abundances.",
        "Missing protein measurements are not biological zeros.",
        "Gene-symbol mapping can be ambiguous.",
        "Protein quantification does not prove surface accessibility.",
        "No tumor-enrichment statistics calculated yet.",
        "No therapeutic target has been validated."
    ]
}

with open(
    META / "cptac_colon_acquisition_provenance.json",
    "w"
) as output:

    json.dump(
        provenance,
        output,
        indent=4
    )


# ------------------------------------------------------------
# 13. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 2.1 SUMMARY")
print("=" * 65)

print("\nCPTAC source datasets:", len(source_records))

print(
    "Paired proteomics patients:",
    len(paired_ids)
)

print(
    "Published MSS paired patients:",
    len(mss_ids)
)

print(
    "Published MSI-H paired patients:",
    len(msih_ids)
)

print(
    "\nTranscriptomic screening genes:",
    len(supported)
)

print(
    "Screening genes represented "
    "in paired protein assay:",
    int(
        supported["in_paired_proteomics"].sum()
    )
)

print("\nSource files stored in:")
print(RAW)

print("\nAll acquisition metadata saved.")

print("\nNo protein differential expression performed.")

print("No source measurements modified.")

print("No TCGA/CPTAC patient IDs merged.")

print("\nPATCH 2.1 COMPLETE")
print("=" * 65)

In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 2 — PATCH 2.2 (CORRECTED)
#
# CPTAC MSS PROTEIN ENRICHMENT AND IDENTIFIER AUDIT
#
# Objectives:
# 1. Validate previously downloaded CPTAC source files
# 2. Audit tumor/normal/paired sample identifiers
# 3. Correct clinical patient identifier handling
# 4. Select published MSS paired-proteomics patients
# 5. Analyze reported tumor-normal protein log2 ratios
# 6. Apply multiple-testing correction
# 7. Integrate transcriptomic screening evidence
# 8. Preserve missing-measurement information
# 9. Generate an exploratory proteomics figure
# 10. Save results, provenance, and research checkpoint
#
# CORRECTIONS:
# - Handle clinical patient ID saved as "index",
#   "Unnamed: 0", or "cptac_patient_id".
# - Explicitly validate clinical/sample correspondence.
# - Use strictly Boolean NumPy masks in np.select.
# - Preserve missing proteomics evidence as missing.
#
# IMPORTANT:
# - No protein imputation
# - No TCGA/CPTAC patient-level merging
# - No separate tumor-normal matrix subtraction
# - No therapeutic target validation
# ============================================================


# ------------------------------------------------------------
# 1. IMPORTS AND PROJECT CONFIGURATION
# ------------------------------------------------------------

from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import wilcoxon, binomtest
from statsmodels.stats.multitest import multipletests

import hashlib
import json
import re


ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"
RAW = ROOT / "data/raw/cptac_colon"
RESULTS = ROOT / "results/tables"
FIGURES = ROOT / "results/figures"
CHECKPOINTS = ROOT / "checkpoints"

for folder in [
    META,
    RAW,
    RESULTS,
    FIGURES,
    CHECKPOINTS
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


SOURCE_MANIFEST = (
    META / "cptac_colon_source_manifest.csv"
)

PAIRED_METADATA = (
    META / "cptac_colon_paired_sample_metadata.csv"
)

TRANSCRIPTOMIC_PATH = (
    RESULTS / "tcga_coad_paired_support_comparison.csv"
)

MIN_OBSERVED_MSS = 30


print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 2 — PATCH 2.2")
print("CPTAC MSS PROTEIN ENRICHMENT")
print("=" * 65)


# ------------------------------------------------------------
# 2. SOURCE DATA INTEGRITY VALIDATION
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SOURCE DATA INTEGRITY VALIDATION")
print("-" * 65)


if not SOURCE_MANIFEST.exists():
    raise FileNotFoundError(
        "Patch 2.1 source manifest is missing."
    )


source_manifest = pd.read_csv(
    SOURCE_MANIFEST,
    dtype=str,
    keep_default_na=False
)


expected_datasets = {
    "tumor_proteomics",
    "normal_proteomics",
    "paired_proteomics",
    "clinical"
}


if set(source_manifest["dataset"]) != expected_datasets:
    raise RuntimeError(
        "Unexpected CPTAC source manifest."
    )


if source_manifest["dataset"].duplicated().any():
    raise RuntimeError(
        "Duplicate datasets in source manifest."
    )


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


source_paths = {}

for _, row in source_manifest.iterrows():

    path = RAW / row["filename"]

    if not path.exists():
        raise FileNotFoundError(path)

    actual_sha = sha256_file(path)

    if actual_sha.lower() != row["sha256"].lower():

        raise RuntimeError(
            "Source checksum mismatch: "
            f"{row['dataset']}"
        )

    source_paths[row["dataset"]] = path


print("\nAll four source files verified.")

print(
    "Checksums match the source manifest "
    "recorded in Patch 2.1."
)


# ------------------------------------------------------------
# 3. LOAD PROTEOMICS MATRICES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING CPTAC PROTEOMICS MATRICES")
print("-" * 65)


def load_matrix(dataset):

    df = pd.read_csv(
        source_paths[dataset],
        sep="\t",
        index_col=0,
        na_values=["NA", "N/A", ""],
        low_memory=False
    )

    df.index = (
        df.index.astype(str).str.strip()
    )

    df.columns = (
        df.columns.astype(str).str.strip()
    )

    if df.index.duplicated().any():

        raise RuntimeError(
            f"Duplicate protein identifiers: {dataset}"
        )

    if df.columns.duplicated().any():

        raise RuntimeError(
            f"Duplicate sample identifiers: {dataset}"
        )

    if len(df) == 0 or len(df.columns) == 0:

        raise RuntimeError(
            f"Empty proteomics matrix: {dataset}"
        )

    numeric = df.apply(
        pd.to_numeric,
        errors="coerce"
    )

    invalid = (
        df.notna() & numeric.isna()
    )

    if invalid.to_numpy().any():

        raise RuntimeError(
            f"Unexpected nonnumeric measurements: {dataset}"
        )

    values = numeric.to_numpy(dtype=float)

    if np.isinf(values).any():

        raise RuntimeError(
            f"Infinite measurements detected: {dataset}"
        )

    return numeric


tumor = load_matrix("tumor_proteomics")

normal = load_matrix("normal_proteomics")

paired = load_matrix("paired_proteomics")


print("\nProteomics matrices loaded.")

print("Tumor:", tumor.shape)

print("Normal:", normal.shape)

print("Paired ratios:", paired.shape)


if tumor.shape != (8067, 97):

    raise RuntimeError(
        "Unexpected tumor proteomics dimensions."
    )

if normal.shape != (8067, 100):

    raise RuntimeError(
        "Unexpected normal proteomics dimensions."
    )

if paired.shape != (6422, 96):

    raise RuntimeError(
        "Unexpected paired proteomics dimensions."
    )


print("\nProteomics matrix validation: PASSED")


# ------------------------------------------------------------
# 4. CPTAC SAMPLE IDENTIFIER AUDIT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CPTAC SAMPLE IDENTIFIER AUDIT")
print("-" * 65)


print("\nFirst 8 tumor identifiers:")
print(tumor.columns[:8].tolist())

print("\nFirst 8 normal identifiers:")
print(normal.columns[:8].tolist())

print("\nFirst 8 paired-ratio identifiers:")
print(paired.columns[:8].tolist())


# CPTAC colon sample IDs may contain:
#
# 01CO005  = tumor/patient identifier
# 01CO005N = corresponding normal specimen
#
# Extract the shared patient identifier.
#
# This establishes identifier correspondence,
# not sample-processing equivalence.


PATIENT_PATTERN = re.compile(
    r"\d{2}CO\d{3}"
)


def extract_patient_id(sample_id):

    match = PATIENT_PATTERN.search(
        str(sample_id)
    )

    if match is None:
        return None

    return match.group(0)


identifier_records = []

for dataset_name, matrix in [
    ("Tumor", tumor),
    ("Normal", normal),
    ("Paired", paired)
]:

    for sample_id in matrix.columns:

        identifier_records.append({

            "dataset": dataset_name,

            "original_sample_id": sample_id,

            "extracted_patient_id":
                extract_patient_id(sample_id)

        })


identifier_audit = pd.DataFrame(
    identifier_records
)


identifier_summary = (
    identifier_audit.groupby("dataset")
    .agg(

        samples=(
            "original_sample_id",
            "count"
        ),

        mapped_patients=(
            "extracted_patient_id",
            "nunique"
        ),

        unmatched=(
            "extracted_patient_id",
            lambda x: int(x.isna().sum())
        )

    )
)


print("\nPatient identifier extraction:")

print(identifier_summary.to_string())


complete_mapping = (
    identifier_audit["extracted_patient_id"]
    .notna()
    .all()
)


unique_mapping = all(

    not group["extracted_patient_id"]
    .duplicated()
    .any()

    for _, group in identifier_audit.groupby(
        "dataset"
    )

)


if complete_mapping and unique_mapping:

    patient_sets = {

        name: set(group["extracted_patient_id"])

        for name, group in identifier_audit.groupby(
            "dataset"
        )

    }

    tumor_normal_overlap = len(
        patient_sets["Tumor"]
        & patient_sets["Normal"]
    )

    paired_tumor_overlap = len(
        patient_sets["Paired"]
        & patient_sets["Tumor"]
    )

    paired_normal_overlap = len(
        patient_sets["Paired"]
        & patient_sets["Normal"]
    )

    all_three_overlap = len(
        patient_sets["Tumor"]
        & patient_sets["Normal"]
        & patient_sets["Paired"]
    )

    print("\nPatient-level overlaps:")

    print(
        "Tumor and normal:",
        tumor_normal_overlap
    )

    print(
        "Paired and tumor:",
        paired_tumor_overlap
    )

    print(
        "Paired and normal:",
        paired_normal_overlap
    )

    print(
        "All three:",
        all_three_overlap
    )

else:

    print(
        "\nWARNING: One-to-one patient "
        "identifier mapping could not "
        "be established."
    )


print("\nNo separate tumor-normal subtraction performed.")


# ============================================================
# 5. CPTAC MSS CLINICAL COHORT IDENTIFICATION
#
# CORRECTED SECTION
#
# The CSV saved by Patch 2.1 may contain:
#
# - cptac_patient_id
# - index
# - Unnamed: 0
#
# This section detects and standardizes
# the identifier column before continuing.
# ============================================================

print("\n" + "-" * 65)
print("MSS PROTEOMICS COHORT")
print("-" * 65)


if not PAIRED_METADATA.exists():

    raise FileNotFoundError(
        "CPTAC paired clinical metadata missing."
    )


clinical = pd.read_csv(
    PAIRED_METADATA,
    dtype=str,
    keep_default_na=False
)


print("\nSaved clinical metadata columns:")

print(clinical.columns.tolist())


# ------------------------------------------------------------
# 5A. Identify patient ID column
# ------------------------------------------------------------

possible_id_columns = [
    "cptac_patient_id",
    "index",
    "Unnamed: 0"
]


available_id_columns = [
    column
    for column in possible_id_columns
    if column in clinical.columns
]


if len(available_id_columns) != 1:

    raise RuntimeError(
        "Could not uniquely identify the "
        "CPTAC patient identifier column. "
        f"Candidates found: {available_id_columns}"
    )


original_id_column = available_id_columns[0]


clinical = clinical.rename(
    columns={
        original_id_column: "cptac_patient_id"
    }
)


print(
    "\nOriginal patient ID column:",
    original_id_column
)

print(
    "Standardized patient ID column:",
    "cptac_patient_id"
)


# ------------------------------------------------------------
# 5B. Validate clinical fields
# ------------------------------------------------------------

required_clinical = {
    "cptac_patient_id",
    "msi_classification"
}


missing_columns = (
    required_clinical - set(clinical.columns)
)


if missing_columns:

    raise RuntimeError(
        "Missing required clinical columns: "
        f"{missing_columns}"
    )


clinical["cptac_patient_id"] = (
    clinical["cptac_patient_id"]
    .astype(str)
    .str.strip()
)


clinical["msi_classification"] = (
    clinical["msi_classification"]
    .astype(str)
    .str.strip()
    .str.upper()
)


if clinical["cptac_patient_id"].eq("").any():

    raise RuntimeError(
        "Empty CPTAC patient identifiers detected."
    )


if clinical["cptac_patient_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate CPTAC clinical patient identifiers."
    )


clinical = clinical.set_index(
    "cptac_patient_id"
)


# ------------------------------------------------------------
# 5C. Validate paired-proteomics correspondence
# ------------------------------------------------------------

missing_patients = (
    set(paired.columns)
    - set(clinical.index)
)


if missing_patients:

    raise RuntimeError(
        "Paired-ratio patients missing clinical "
        "metadata: "
        f"{sorted(missing_patients)}"
    )


print("\nClinical patient correspondence: PASSED")


# ------------------------------------------------------------
# 5D. Identify published MSS patients
# ------------------------------------------------------------

msi_status = (
    clinical.loc[
        paired.columns,
        "msi_classification"
    ]
)


print("\nMSI classifications:")

print(
    msi_status.value_counts().to_string()
)


mss_patients = msi_status.index[
    msi_status.eq("MSS")
].tolist()


if len(mss_patients) != 73:

    raise RuntimeError(
        "Expected 73 published MSS patients. "
        f"Found {len(mss_patients)}."
    )


mss = paired.loc[
    :,
    mss_patients
].copy()


print("\nMSS patients:", len(mss_patients))

print("Measured protein rows:", len(mss))

print("\nMSS cohort identification: PASSED")


# ------------------------------------------------------------
# 6. CALCULATE MSS PROTEIN ENRICHMENT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CALCULATING MSS PROTEIN ENRICHMENT")
print("-" * 65)


# The paired matrix contains reported
# tumor-normal protein log2 ratios.
#
# Positive ratio = higher reported abundance
# in tumor relative to normal.
#
# Missing measurements are excluded
# individually for each protein.
#
# No missing values are replaced with zero.


protein_records = []


for gene_symbol, row in mss.iterrows():

    values = row.to_numpy(
        dtype=float
    )

    observed = values[
        np.isfinite(values)
    ]

    n = len(observed)

    # --------------------------------------------
    # Descriptive protein statistics
    # --------------------------------------------

    if n > 0:

        median_ratio = float(
            np.median(observed)
        )

        mean_ratio = float(
            np.mean(observed)
        )

        fraction_positive = float(
            np.mean(observed > 0)
        )

        fraction_ge1 = float(
            np.mean(observed >= 1)
        )

        positive = int(
            np.count_nonzero(observed > 0)
        )

        negative = int(
            np.count_nonzero(observed < 0)
        )

        zero = int(
            np.count_nonzero(observed == 0)
        )

    else:

        median_ratio = np.nan

        mean_ratio = np.nan

        fraction_positive = np.nan

        fraction_ge1 = np.nan

        positive = 0
        negative = 0
        zero = 0

    # --------------------------------------------
    # Statistical tests
    # --------------------------------------------

    wilcoxon_p = np.nan

    sign_p = np.nan


    if n >= MIN_OBSERVED_MSS:

        if np.all(observed == 0):

            wilcoxon_p = 1.0

        else:

            wilcoxon_p = float(
                wilcoxon(
                    observed,
                    alternative="two-sided",
                    zero_method="wilcox",
                    method="auto"
                ).pvalue
            )

        nonzero = positive + negative

        if nonzero > 0:

            sign_p = float(
                binomtest(
                    positive,
                    n=nonzero,
                    p=0.5,
                    alternative="two-sided"
                ).pvalue
            )

        else:

            sign_p = 1.0


    protein_records.append({

        "protein_gene_symbol": gene_symbol,

        "mss_patients_total": len(mss_patients),

        "mss_measurements": n,

        "missing_mss_measurements":
            len(mss_patients) - n,

        "mss_measurement_fraction":
            n / len(mss_patients),

        "median_tumor_normal_log2ratio":
            median_ratio,

        "mean_tumor_normal_log2ratio":
            mean_ratio,

        "fraction_tumor_higher":
            fraction_positive,

        "fraction_log2ratio_ge1":
            fraction_ge1,

        "positive_patients":
            positive,

        "negative_patients":
            negative,

        "zero_ratio_patients":
            zero,

        "wilcoxon_pvalue":
            wilcoxon_p,

        "sign_test_pvalue":
            sign_p

    })


protein_stats = pd.DataFrame(
    protein_records
)


print("\nProtein statistics calculated.")

print("Total protein rows:", len(protein_stats))

print(
    "Minimum required MSS measurements:",
    MIN_OBSERVED_MSS
)


# ------------------------------------------------------------
# 7. MULTIPLE-TESTING CORRECTION
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("MULTIPLE-TESTING CORRECTION")
print("-" * 65)


protein_stats["wilcoxon_fdr"] = np.nan

protein_stats["sign_test_fdr"] = np.nan


for p_column, fdr_column in [

    ("wilcoxon_pvalue", "wilcoxon_fdr"),

    ("sign_test_pvalue", "sign_test_fdr")

]:

    valid = (
        protein_stats[p_column]
        .notna()
        .to_numpy(dtype=bool)
    )

    if valid.sum() == 0:

        raise RuntimeError(
            f"No valid protein tests: {p_column}"
        )

    adjusted = multipletests(

        protein_stats.loc[
            valid,
            p_column
        ].to_numpy(dtype=float),

        method="fdr_bh"

    )[1]


    protein_stats.loc[
        valid,
        fdr_column
    ] = adjusted


# ------------------------------------------------------------
# 7A. Apply exploratory enrichment criteria
# ------------------------------------------------------------

protein_stats["protein_enriched"] = (

    protein_stats["wilcoxon_fdr"].lt(0.05)

    & protein_stats[
        "median_tumor_normal_log2ratio"
    ].ge(1.0)

    & protein_stats[
        "fraction_tumor_higher"
    ].ge(0.70)

    & protein_stats[
        "mss_measurements"
    ].ge(MIN_OBSERVED_MSS)

)


proteins_tested = int(
    protein_stats["wilcoxon_pvalue"]
    .notna()
    .sum()
)


proteins_enriched = int(
    protein_stats["protein_enriched"].sum()
)


print(
    "\nProteins with sufficient MSS measurements:",
    proteins_tested
)

print(
    "Proteins meeting enrichment criteria:",
    proteins_enriched
)

print(
    "Proteins lacking sufficient measurements:",
    int(
        protein_stats["wilcoxon_pvalue"]
        .isna()
        .sum()
    )
)


# ------------------------------------------------------------
# 8. LOAD TRANSCRIPTOMIC SCREENING EVIDENCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("TRANSCRIPTOMIC / PROTEOMIC INTEGRATION")
print("-" * 65)


if not TRANSCRIPTOMIC_PATH.exists():

    raise FileNotFoundError(
        "Patch 1.22 transcriptomic comparison "
        "table is missing."
    )


transcriptomic = pd.read_csv(
    TRANSCRIPTOMIC_PATH,
    dtype={"gene_id": str},
    low_memory=False
)


required_columns = {
    "gene_id",
    "gene_name",
    "combined_transcriptomic_support"
}


if not required_columns.issubset(
    transcriptomic.columns
):

    raise RuntimeError(
        "Missing transcriptomic evidence columns."
    )


supported_mask = (

    transcriptomic[
        "combined_transcriptomic_support"
    ]

    .astype(str)
    .str.lower()
    .eq("true")

)


supported = transcriptomic[
    supported_mask
].copy()


if len(supported) != 948:

    raise RuntimeError(
        "Expected 948 transcriptomic screening genes. "
        f"Found {len(supported)}."
    )


supported["gene_name"] = (
    supported["gene_name"]
    .astype(str)
    .str.strip()
)


print("\nTranscriptomic screening genes:", len(supported))


# ------------------------------------------------------------
# 8A. Integrate by gene symbol
# ------------------------------------------------------------

integrated = supported.merge(

    protein_stats,

    left_on="gene_name",

    right_on="protein_gene_symbol",

    how="left",

    validate="many_to_one"

)


if len(integrated) != 948:

    raise RuntimeError(
        "Transcriptomic integration changed row count."
    )


print("\nGene-symbol integration: PASSED")


# ============================================================
# 9. CLASSIFY PROTEIN EVIDENCE
#
# CORRECTED:
# Convert all masks to explicit Boolean arrays
# to avoid nullable Boolean compatibility errors.
# ============================================================

print("\n" + "-" * 65)
print("PROTEIN EVIDENCE CLASSIFICATION")
print("-" * 65)


integrated["paired_assay_covered"] = (
    integrated["protein_gene_symbol"].notna()
)


enriched_mask = (

    integrated["protein_enriched"]

    .fillna(False)

    .astype(bool)

    .to_numpy(dtype=bool)

)


covered_mask = (

    integrated["paired_assay_covered"]

    .fillna(False)

    .astype(bool)

    .to_numpy(dtype=bool)

)


insufficient_mask = (

    covered_mask

    & integrated[
        "wilcoxon_pvalue"
    ].isna().to_numpy(dtype=bool)

)


integrated["protein_support_status"] = np.select(

    [

        enriched_mask,

        insufficient_mask,

        covered_mask

    ],

    [

        "MSS protein enrichment supported",

        "Measured; insufficient MSS observations",

        "Measured; enrichment criteria not met"

    ],

    default="Not represented in paired assay"

)


print("\nProtein evidence categories:")

print(
    integrated[
        "protein_support_status"
    ].value_counts().to_string()
)


# ------------------------------------------------------------
# 10. EXAMINE LEADING GENES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LEADING GENE PROTEIN EVIDENCE")
print("-" * 65)


leading_genes = [
    "CDH3",
    "CLDN1",
    "KRT80",
    "ETV4",
    "FOXQ1",
    "ESM1",
    "CEMIP",
    "INHBA",
    "SIM2",
    "WNT2"
]


leading = integrated[
    integrated["gene_name"].isin(
        leading_genes
    )
].copy()


gene_order = {
    gene: index
    for index, gene in enumerate(
        leading_genes
    )
}


leading["gene_order"] = (
    leading["gene_name"].map(
        gene_order
    )
)


leading = leading.sort_values(
    "gene_order"
)


display_columns = [
    "gene_name",
    "paired_assay_covered",
    "mss_measurements",
    "median_tumor_normal_log2ratio",
    "fraction_tumor_higher",
    "wilcoxon_fdr",
    "protein_support_status"
]


print("\n")

print(
    leading[display_columns]
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4g}"
    )
)


print("\nIMPORTANT:")

print(
    "Missing paired-proteomics evidence "
    "does not establish protein absence."
)


# ------------------------------------------------------------
# 11. IDENTIFY PROTEOMICS-SUPPORTED GENES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PROTEOMICS-SUPPORTED SCREENING GENES")
print("-" * 65)


protein_supported = integrated[

    integrated["protein_support_status"]
    .eq("MSS protein enrichment supported")

].copy()


protein_supported = (
    protein_supported
    .sort_values(
        [
            "wilcoxon_fdr",
            "median_tumor_normal_log2ratio"
        ],
        ascending=[True, False]
    )
)


print(
    "\nTranscriptomic screening genes with "
    "MSS protein-enrichment support:",
    len(protein_supported)
)


if not protein_supported.empty:

    print("\nTop 20:")

    print(

        protein_supported[
            [
                "gene_name",
                "baseline_log2fc",
                "mss_measurements",
                "median_tumor_normal_log2ratio",
                "fraction_tumor_higher",
                "wilcoxon_fdr"
            ]
        ]

        .head(20)

        .to_string(
            index=False,
            float_format=lambda x: f"{x:.4g}"
        )

    )

else:

    print(
        "No proteins passed the prespecified "
        "combined screening criteria."
    )


# ------------------------------------------------------------
# 12. GENERATE PROTEOMICS FIGURE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENERATING PROTEOMICS FIGURE")
print("-" * 65)


plot_data = protein_stats.dropna(

    subset=[
        "median_tumor_normal_log2ratio",
        "wilcoxon_fdr"
    ]

).copy()


plot_data["neg_log10_fdr"] = (

    -np.log10(

        plot_data["wilcoxon_fdr"]
        .clip(
            lower=np.finfo(float).tiny
        )

    )

)


fig, ax = plt.subplots(
    figsize=(10, 7)
)


ax.scatter(

    plot_data[
        "median_tumor_normal_log2ratio"
    ],

    plot_data[
        "neg_log10_fdr"
    ],

    s=12,

    alpha=0.4

)


ax.axvline(
    1,
    linestyle="--",
    linewidth=0.9
)

ax.axvline(
    0,
    linestyle=":",
    linewidth=0.8
)

ax.axhline(
    -np.log10(0.05),
    linestyle="--",
    linewidth=0.9
)


ax.set_xlabel(
    "Median reported tumor-normal protein log2 ratio"
)

ax.set_ylabel(
    "-log10 adjusted Wilcoxon p-value"
)

ax.set_title(
    "CPTAC Colon — MSS Paired Protein Enrichment"
)

ax.grid(alpha=0.2)

fig.tight_layout()


figure_path = (
    FIGURES /
    "cptac_mss_protein_enrichment_volcano.png"
)


fig.savefig(
    figure_path,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

plt.close(fig)


print("\nProteomics figure saved.")


# ------------------------------------------------------------
# 13. SAVE ANALYSIS OUTPUTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING PROTEOMICS RESULTS")
print("-" * 65)


identifier_audit.to_csv(

    META / "cptac_colon_identifier_audit.csv",

    index=False

)


protein_stats.to_csv(

    RESULTS / "cptac_mss_paired_protein_statistics.csv",

    index=False

)


integrated.to_csv(

    RESULTS / "cptac_mss_transcriptomic_integration.csv",

    index=False

)


protein_supported.to_csv(

    RESULTS / "cptac_mss_protein_supported_candidates.csv",

    index=False

)


leading.to_csv(

    RESULTS / "cptac_mss_leading_gene_review.csv",

    index=False

)


print("\nAll proteomics analysis tables saved.")


# ------------------------------------------------------------
# 14. SAVE RESEARCH PROVENANCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING ANALYSIS PROVENANCE")
print("-" * 65)


provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "2.2",

    "patch_version": "2.2-corrected",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "study": "CPTAC Prospective Colon Cancer",

    "data_source": "LinkedOmics",

    "msi_group": "Published MSS",

    "mss_patients": int(
        len(mss_patients)
    ),

    "reported_measurement": (
        "Paired tumor-normal TMT protein log2 ratio"
    ),

    "minimum_observed_mss_patients":
        MIN_OBSERVED_MSS,

    "statistical_test": (
        "Two-sided Wilcoxon signed-rank test "
        "against zero"
    ),

    "additional_test": (
        "Two-sided binomial sign test"
    ),

    "multiple_testing": (
        "Benjamini-Hochberg FDR across "
        "all tested paired proteins"
    ),

    "protein_enrichment_criteria": {

        "wilcoxon_fdr": 0.05,

        "minimum_median_log2ratio": 1.0,

        "minimum_fraction_positive": 0.70

    },

    "proteins_tested": int(
        proteins_tested
    ),

    "proteins_enriched": int(
        proteins_enriched
    ),

    "transcriptomic_screening_genes": 948,

    "integrated_protein_supported_genes": int(
        len(protein_supported)
    ),

    "sample_identifier_mapping": (
        "CPTAC patient ID extracted from "
        "tumor, normal and paired sample labels"
    ),

    "clinical_identifier_fix": (
        "Standardized saved patient identifier "
        "column to cptac_patient_id"
    ),

    "protein_imputation_performed": False,

    "tcga_cptac_patient_level_merging": False,

    "separate_tumor_normal_subtraction": False,

    "limitations": [

        "Paired ratios are reported processed "
        "TMT measurements, not absolute "
        "protein concentrations.",

        "Minimum observation count is an "
        "exploratory screening criterion.",

        "Wilcoxon signed-rank testing assumes "
        "an appropriately symmetric distribution "
        "of paired differences for location inference.",

        "Missing proteomic measurements "
        "may not be random.",

        "Tumor-adjacent normal tissue is not "
        "equivalent to a healthy-organ "
        "safety atlas.",

        "Protein-level enrichment does not "
        "establish extracellular accessibility.",

        "Proteins absent from the paired-ratio "
        "matrix may still be measured in the "
        "separate tumor and normal matrices.",

        "Sample identifier correspondence "
        "does not establish identical "
        "quantification procedures.",

        "Gene-symbol integration is provisional "
        "and requires identifier validation.",

        "No therapeutic targets have been validated."

    ]

}


provenance_path = (
    META /
    "cptac_mss_protein_analysis_provenance.json"
)


with open(
    provenance_path,
    "w"
) as f:

    json.dump(
        provenance,
        f,
        indent=4
    )


print("\nAnalysis provenance saved.")


# ------------------------------------------------------------
# 15. CREATE PHASE 2 RESEARCH CHECKPOINT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING PHASE 2 RESEARCH CHECKPOINT")
print("-" * 65)


timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S")


checkpoint = (

    CHECKPOINTS /

    f"SurfaceTarget_CRC_Proteomics_Patch22_{timestamp}.zip"

)


backup_folders = [
    "data/metadata",
    "data/processed",
    "data/raw/cptac_colon",
    "results/tables",
    "results/figures",
    "src",
    "reports",
    "logs"
]


with ZipFile(
    checkpoint,
    "w",
    ZIP_DEFLATED,
    compresslevel=3
) as archive:

    for folder in backup_folders:

        folder_path = ROOT / folder

        if not folder_path.exists():
            continue

        for file_path in folder_path.rglob("*"):

            if file_path.is_file():

                archive.write(

                    file_path,

                    arcname=str(
                        file_path.relative_to(ROOT)
                    )

                )


checkpoint_mib = (
    checkpoint.stat().st_size
    / (1024 ** 2)
)


print("\nCheckpoint created.")

print(
    f"Checkpoint size: {checkpoint_mib:.2f} MiB"
)

print("Checkpoint path:")

print(checkpoint)


# ------------------------------------------------------------
# 16. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 2.2 SUMMARY")
print("=" * 65)


print("\nPublished MSS patients:", len(mss_patients))

print(
    "Proteins in paired assay:",
    len(paired)
)

print(
    "Proteins eligible for statistical testing:",
    proteins_tested
)

print(
    "MSS-enriched proteins:",
    proteins_enriched
)


print("\nTranscriptomic screening genes:", 948)

print(
    "Genes with protein-enrichment support:",
    len(protein_supported)
)


print("\nSaved outputs:")

print("1. cptac_colon_identifier_audit.csv")

print("2. cptac_mss_paired_protein_statistics.csv")

print("3. cptac_mss_transcriptomic_integration.csv")

print("4. cptac_mss_protein_supported_candidates.csv")

print("5. cptac_mss_leading_gene_review.csv")

print("6. cptac_mss_protein_analysis_provenance.json")


print("\nFigure:")

print("cptac_mss_protein_enrichment_volcano.png")


print("\nResearch checkpoint: CREATED")

print(
    f"Checkpoint size: {checkpoint_mib:.2f} MiB"
)


print("\nClinical identifier fix: APPLIED")

print("No source measurements modified.")

print("No protein imputation performed.")

print("No TCGA/CPTAC patient-level merging.")


print("\nPATCH 2.2 COMPLETE")

print("=" * 65)

In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 2 — PATCH 2.3
#
# CPTAC CROSS-MATRIX QUANTIFICATION AUDIT
#
# Objectives:
# 1. Validate CPTAC source matrices
# 2. Harmonize tumor and normal patient identifiers
# 3. Compare derived differences with reported paired ratios
# 4. Evaluate global and patient-level concordance
# 5. Examine CDH3, CLDN1 and other leading genes
# 6. Save audit tables, figure and provenance
#
# IMPORTANT:
# - No new protein differential expression tests
# - No imputation
# - No automatic acceptance of derived ratios
# - No source data modification
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import spearmanr

import hashlib
import json
import re


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"
RAW = ROOT / "data/raw/cptac_colon"
RESULTS = ROOT / "results/tables"
FIGURES = ROOT / "results/figures"

RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

SOURCE_PATH = (
    META / "cptac_colon_source_manifest.csv"
)

CLINICAL_PATH = (
    META / "cptac_colon_paired_sample_metadata.csv"
)

print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 2 — PATCH 2.3")
print("CPTAC CROSS-MATRIX QUANTIFICATION AUDIT")
print("=" * 65)


# ------------------------------------------------------------
# 2. Validate source files
# ------------------------------------------------------------

source = pd.read_csv(
    SOURCE_PATH,
    dtype=str,
    keep_default_na=False
)

if source["dataset"].duplicated().any():
    raise RuntimeError(
        "Duplicate CPTAC source datasets."
    )


def calculate_sha256(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


source_paths = {}

for _, row in source.iterrows():

    path = RAW / row["filename"]

    if not path.exists():
        raise FileNotFoundError(path)

    if calculate_sha256(path) != row["sha256"]:
        raise RuntimeError(
            f"Checksum mismatch: {row['dataset']}"
        )

    source_paths[row["dataset"]] = path

print("\nSource checksum validation: PASSED")


# ------------------------------------------------------------
# 3. Load proteomics matrices
# ------------------------------------------------------------

def load_matrix(dataset):

    df = pd.read_csv(
        source_paths[dataset],
        sep="\t",
        index_col=0,
        na_values=["NA", "N/A", ""],
        low_memory=False
    )

    df.index = (
        df.index.astype(str).str.strip()
    )

    df.columns = (
        df.columns.astype(str).str.strip()
    )

    if df.index.duplicated().any():
        raise RuntimeError(
            f"Duplicate gene symbols: {dataset}"
        )

    if df.columns.duplicated().any():
        raise RuntimeError(
            f"Duplicate sample identifiers: {dataset}"
        )

    numeric = df.apply(
        pd.to_numeric,
        errors="raise"
    )

    if np.isinf(
        numeric.to_numpy(dtype=float)
    ).any():
        raise RuntimeError(
            f"Infinite values: {dataset}"
        )

    return numeric


tumor = load_matrix("tumor_proteomics")

normal = load_matrix("normal_proteomics")

paired = load_matrix("paired_proteomics")


print("\nMatrices loaded.")

print("Tumor:", tumor.shape)
print("Normal:", normal.shape)
print("Reported paired:", paired.shape)


# ------------------------------------------------------------
# 4. Harmonize patient identifiers
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PATIENT IDENTIFIER VALIDATION")
print("-" * 65)


PATTERN = re.compile(
    r"^(\d{2}CO\d{3})(N)?$"
)


def build_patient_map(columns, dataset):

    mapping = {}

    for sample_id in columns:

        match = PATTERN.fullmatch(
            str(sample_id)
        )

        if match is None:
            raise RuntimeError(
                f"Unrecognized {dataset} identifier: "
                f"{sample_id}"
            )

        patient_id = match.group(1)
        normal_suffix = match.group(2)

        if dataset == "Normal":

            if normal_suffix != "N":
                raise RuntimeError(
                    f"Unexpected normal label: {sample_id}"
                )

        else:

            if normal_suffix is not None:
                raise RuntimeError(
                    f"Unexpected {dataset} label: {sample_id}"
                )

        if patient_id in mapping:
            raise RuntimeError(
                f"Duplicate patient: {patient_id}"
            )

        mapping[patient_id] = sample_id

    return mapping


tumor_map = build_patient_map(
    tumor.columns,
    "Tumor"
)

normal_map = build_patient_map(
    normal.columns,
    "Normal"
)

paired_map = build_patient_map(
    paired.columns,
    "Paired"
)


shared_patients = sorted(
    set(tumor_map)
    & set(normal_map)
    & set(paired_map)
)


print("\nPatients in all three matrices:",
      len(shared_patients))


if len(shared_patients) != 96:
    raise RuntimeError(
        "Expected 96 fully matched CPTAC patients."
    )


print("Patient correspondence: PASSED")


# ------------------------------------------------------------
# 5. Identify comparable protein rows
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PROTEIN IDENTIFIER CORRESPONDENCE")
print("-" * 65)


common_genes = (
    paired.index
    .intersection(tumor.index)
    .intersection(normal.index)
)


print("\nTumor proteins:", len(tumor))
print("Normal proteins:", len(normal))
print("Reported paired proteins:", len(paired))

print(
    "Proteins represented in all three:",
    len(common_genes)
)


if len(common_genes) < 3000:
    raise RuntimeError(
        "Unexpectedly low protein overlap."
    )


# ------------------------------------------------------------
# 6. Align measurements
# ------------------------------------------------------------

tumor_columns = [
    tumor_map[p]
    for p in shared_patients
]

normal_columns = [
    normal_map[p]
    for p in shared_patients
]

paired_columns = [
    paired_map[p]
    for p in shared_patients
]


tumor_values = (
    tumor.loc[
        common_genes,
        tumor_columns
    ]
    .to_numpy(dtype=float)
)

normal_values = (
    normal.loc[
        common_genes,
        normal_columns
    ]
    .to_numpy(dtype=float)
)

reported_values = (
    paired.loc[
        common_genes,
        paired_columns
    ]
    .to_numpy(dtype=float)
)


# Diagnostic subtraction only.
#
# We are NOT yet assuming the different
# matrices have equivalent normalization.

derived_values = (
    tumor_values - normal_values
)


valid = (
    np.isfinite(derived_values)
    & np.isfinite(reported_values)
)


n_comparable = int(valid.sum())


print("\n" + "-" * 65)
print("CROSS-MATRIX MEASUREMENT AVAILABILITY")
print("-" * 65)


print(
    "\nPotential gene-patient comparisons:",
    int(derived_values.size)
)

print(
    "Complete comparisons:",
    n_comparable
)

print(
    "Complete comparison percentage:",
    round(
        100 * n_comparable / derived_values.size,
        2
    )
)


if n_comparable < 5000:
    raise RuntimeError(
        "Insufficient complete measurements."
    )


# ------------------------------------------------------------
# 7. Global concordance statistics
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GLOBAL QUANTIFICATION CONCORDANCE")
print("-" * 65)


derived_observed = derived_values[valid]

reported_observed = reported_values[valid]


difference = (
    derived_observed - reported_observed
)


pearson = float(
    np.corrcoef(
        derived_observed,
        reported_observed
    )[0, 1]
)


spearman = float(
    spearmanr(
        derived_observed,
        reported_observed
    ).statistic
)


median_absolute_error = float(
    np.median(
        np.abs(difference)
    )
)


median_difference = float(
    np.median(difference)
)


p95_absolute_error = float(
    np.percentile(
        np.abs(difference),
        95
    )
)


print("\nComplete observations:", n_comparable)

print(
    "Pearson correlation:",
    round(pearson, 5)
)

print(
    "Spearman correlation:",
    round(spearman, 5)
)

print(
    "Median derived-minus-reported:",
    round(median_difference, 5)
)

print(
    "Median absolute difference:",
    round(median_absolute_error, 5)
)

print(
    "95th percentile absolute difference:",
    round(p95_absolute_error, 5)
)


print("\nIMPORTANT:")
print(
    "Correlation alone does not establish "
    "measurement-scale equivalence."
)


# ------------------------------------------------------------
# 8. Patient-level comparison
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PATIENT-LEVEL CONCORDANCE")
print("-" * 65)


patient_records = []


for column_index, patient_id in enumerate(
    shared_patients
):

    x = derived_values[:, column_index]

    y = reported_values[:, column_index]

    mask = (
        np.isfinite(x)
        & np.isfinite(y)
    )

    n = int(mask.sum())

    if n >= 50:

        px = x[mask]
        py = y[mask]

        correlation = float(
            np.corrcoef(px, py)[0, 1]
        )

        median_error = float(
            np.median(
                np.abs(px - py)
            )
        )

        median_offset = float(
            np.median(px - py)
        )

    else:

        correlation = np.nan
        median_error = np.nan
        median_offset = np.nan

    patient_records.append({

        "patient_id": patient_id,

        "n_complete_proteins": n,

        "pearson_correlation":
            correlation,

        "median_absolute_difference":
            median_error,

        "median_derived_minus_reported":
            median_offset

    })


patient_audit = pd.DataFrame(
    patient_records
)


print("\nPatient-level summary:")

print(
    patient_audit[
        [
            "n_complete_proteins",
            "pearson_correlation",
            "median_absolute_difference",
            "median_derived_minus_reported"
        ]
    ]
    .describe()
    .round(4)
    .to_string()
)


# ------------------------------------------------------------
# 9. Load MSS clinical annotations
# ------------------------------------------------------------

clinical = pd.read_csv(
    CLINICAL_PATH,
    dtype=str,
    keep_default_na=False
)


id_columns = [
    c for c in [
        "cptac_patient_id",
        "index",
        "Unnamed: 0"
    ]
    if c in clinical.columns
]


if len(id_columns) != 1:
    raise RuntimeError(
        "Clinical patient ID column is ambiguous."
    )


clinical = clinical.rename(
    columns={
        id_columns[0]: "cptac_patient_id"
    }
)


if clinical["cptac_patient_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate clinical patient IDs."
    )


clinical = clinical.set_index(
    "cptac_patient_id"
)


msi_status = (
    clinical.loc[
        shared_patients,
        "msi_classification"
    ]
    .astype(str)
    .str.strip()
    .str.upper()
)


mss_patients = [
    p for p in shared_patients
    if msi_status.loc[p] == "MSS"
]


print("\nPublished MSS matched patients:",
      len(mss_patients))


if len(mss_patients) != 73:
    raise RuntimeError(
        "Expected 73 published MSS patients."
    )


# ------------------------------------------------------------
# 10. Examine leading genes
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LEADING GENE MEASUREMENT AUDIT")
print("-" * 65)


TARGET_GENES = [
    "CDH3",
    "CLDN1",
    "KRT80",
    "CEMIP",
    "INHBA",
    "WNT2",
    "FAP",
    "DPEP1",
    "SLC7A5",
    "SLC5A6",
    "CTHRC1",
    "S100P"
]


mss_tumor_columns = [
    tumor_map[p]
    for p in mss_patients
]

mss_normal_columns = [
    normal_map[p]
    for p in mss_patients
]

mss_paired_columns = [
    paired_map[p]
    for p in mss_patients
]


gene_records = []


for gene in TARGET_GENES:

    in_tumor = gene in tumor.index
    in_normal = gene in normal.index
    in_reported = gene in paired.index

    n_complete_separate = 0

    derived_median = np.nan
    derived_fraction_positive = np.nan

    reported_n = 0
    reported_median = np.nan

    if in_tumor and in_normal:

        t = tumor.loc[
            gene,
            mss_tumor_columns
        ].to_numpy(dtype=float)

        n = normal.loc[
            gene,
            mss_normal_columns
        ].to_numpy(dtype=float)

        complete = (
            np.isfinite(t)
            & np.isfinite(n)
        )

        n_complete_separate = int(
            complete.sum()
        )

        if n_complete_separate > 0:

            delta = (
                t[complete] - n[complete]
            )

            derived_median = float(
                np.median(delta)
            )

            derived_fraction_positive = float(
                np.mean(delta > 0)
            )

    if in_reported:

        r = paired.loc[
            gene,
            mss_paired_columns
        ].to_numpy(dtype=float)

        r = r[np.isfinite(r)]

        reported_n = len(r)

        if reported_n > 0:

            reported_median = float(
                np.median(r)
            )

    gene_records.append({

        "gene_name": gene,

        "in_tumor_matrix": in_tumor,

        "in_normal_matrix": in_normal,

        "in_reported_paired_matrix": in_reported,

        "mss_complete_separate_pairs":
            n_complete_separate,

        "diagnostic_derived_median_log2_difference":
            derived_median,

        "diagnostic_derived_fraction_positive":
            derived_fraction_positive,

        "reported_paired_mss_measurements":
            reported_n,

        "reported_paired_median_log2ratio":
            reported_median

    })


gene_audit = pd.DataFrame(
    gene_records
)


print("\n")

print(
    gene_audit.to_string(
        index=False,
        float_format=lambda x: f"{x:.4g}"
    )
)


print("\nCAUTION:")
print(
    "Derived differences are diagnostic only. "
    "They are not yet accepted as validated "
    "tumor-normal protein enrichment measurements."
)


# ------------------------------------------------------------
# 11. Generate concordance figure
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENERATING CROSS-MATRIX FIGURE")
print("-" * 65)


rng = np.random.default_rng(42)

n_plot = min(
    15000,
    len(derived_observed)
)

plot_indices = rng.choice(
    len(derived_observed),
    size=n_plot,
    replace=False
)


x_plot = reported_observed[
    plot_indices
]

y_plot = derived_observed[
    plot_indices
]


fig, ax = plt.subplots(
    figsize=(8, 7)
)


ax.scatter(
    x_plot,
    y_plot,
    s=6,
    alpha=0.25
)


limits = [
    float(min(
        np.min(x_plot),
        np.min(y_plot)
    )),
    float(max(
        np.max(x_plot),
        np.max(y_plot)
    ))
]


ax.plot(
    limits,
    limits,
    linestyle="--",
    linewidth=1,
    label="Exact agreement"
)


ax.set_xlabel(
    "Reported paired tumor-normal log2 ratio"
)

ax.set_ylabel(
    "Tumor minus normal (separate log-ratio matrices)"
)

ax.set_title(
    "CPTAC Colon — Cross-Matrix Quantification Audit"
)

ax.grid(alpha=0.2)

ax.legend()

fig.tight_layout()


figure_path = (
    FIGURES /
    "cptac_crossmatrix_ratio_concordance.png"
)


fig.savefig(
    figure_path,
    dpi=200,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


print("\nFigure saved.")


# ------------------------------------------------------------
# 12. Save audit tables
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING CROSS-MATRIX AUDIT")
print("-" * 65)


patient_audit.to_csv(

    RESULTS /
    "cptac_crossmatrix_patient_concordance.csv",

    index=False

)


gene_audit.to_csv(

    RESULTS /
    "cptac_crossmatrix_leading_gene_review.csv",

    index=False

)


# ------------------------------------------------------------
# 13. Save provenance
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "2.3",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": "CPTAC Colon via LinkedOmics",

    "analysis": (
        "Diagnostic concordance between "
        "reported paired protein log2 ratios "
        "and differences derived from separate "
        "tumor and normal log-ratio matrices"
    ),

    "matched_patients": int(
        len(shared_patients)
    ),

    "published_mss_patients": int(
        len(mss_patients)
    ),

    "common_proteins": int(
        len(common_genes)
    ),

    "complete_comparisons": n_comparable,

    "global_pearson": pearson,

    "global_spearman": spearman,

    "median_derived_minus_reported":
        median_difference,

    "median_absolute_difference":
        median_absolute_error,

    "p95_absolute_difference":
        p95_absolute_error,

    "diagnostic_subtraction_performed": True,

    "derived_ratios_accepted_for_inference": False,

    "additional_protein_differential_expression":
        False,

    "imputation_performed": False,

    "limitations": [

        "The separate matrices may have "
        "different quantification or "
        "normalization characteristics.",

        "Agreement between datasets does not "
        "independently establish biological validity.",

        "Missing proteomic measurements "
        "may be nonrandom.",

        "Derived differences are not yet "
        "approved for formal enrichment inference.",

        "Protein abundance does not establish "
        "cell-surface accessibility."

    ]
}


with open(
    META / "cptac_crossmatrix_audit_provenance.json",
    "w"
) as f:

    json.dump(
        provenance,
        f,
        indent=4
    )


# ------------------------------------------------------------
# 14. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 2.3 SUMMARY")
print("=" * 65)


print("\nMatched patients:", len(shared_patients))

print("Published MSS patients:", len(mss_patients))

print("Comparable proteins:", len(common_genes))

print("Complete measurement comparisons:",
      n_comparable)


print("\nGlobal Pearson:", round(pearson, 5))

print("Global Spearman:", round(spearman, 5))

print(
    "Median absolute difference:",
    round(median_absolute_error, 5)
)


print("\nSaved outputs:")

print("1. cptac_crossmatrix_patient_concordance.csv")

print("2. cptac_crossmatrix_leading_gene_review.csv")

print("3. cptac_crossmatrix_audit_provenance.json")

print("4. cptac_crossmatrix_ratio_concordance.png")


print("\nNo new protein enrichment tests performed.")

print("No source measurements modified.")

print("No additional datasets downloaded.")


print("\nPATCH 2.3 COMPLETE")

print("=" * 65)

In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 2 — PATCH 2.4 (CORRECTED)
#
# EXTENDED CPTAC MSS PROTEIN COVERAGE AND ENRICHMENT
#
# Objectives:
# 1. Validate CPTAC source files
# 2. Reconstruct paired protein differences
# 3. Verify numerical and missingness agreement
# 4. Examine all 8,067 protein rows
# 5. Apply MSS protein-enrichment criteria
# 6. Correct multiple testing across expanded universe
# 7. Integrate 948 transcriptomic screening genes
# 8. Investigate sparsely measured leading candidates
# 9. Preserve original analyses
# 10. Create a research checkpoint
#
# CORRECTIONS:
# - Allow reconstruction differences up to 0.00011
# - Apply the same tolerance to median comparisons
# - Record the numerical tolerance in provenance
# - Report measurements exceeding the old threshold
#
# IMPORTANT:
# - No protein imputation
# - No independent validation claimed
# - No TCGA/CPTAC patient-level merging
# - No source measurements modified
# - Insufficient coverage is not negative evidence
# ============================================================


# ------------------------------------------------------------
# 1. PROJECT CONFIGURATION
# ------------------------------------------------------------

from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

import hashlib
import json


ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"
RAW = ROOT / "data/raw/cptac_colon"
RESULTS = ROOT / "results/tables"
FIGURES = ROOT / "results/figures"
CHECKPOINTS = ROOT / "checkpoints"

for folder in [
    META,
    RAW,
    RESULTS,
    FIGURES,
    CHECKPOINTS
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


SOURCE_MANIFEST = (
    META / "cptac_colon_source_manifest.csv"
)

CLINICAL_PATH = (
    META / "cptac_colon_paired_sample_metadata.csv"
)

ORIGINAL_PROTEIN_PATH = (
    RESULTS / "cptac_mss_paired_protein_statistics.csv"
)

TRANSCRIPTOMIC_PATH = (
    RESULTS / "tcga_coad_paired_support_comparison.csv"
)


MIN_OBSERVATIONS = 30

# CORRECTED:
# Tolerance for numerical comparisons between
# separately reconstructed and reported ratios.
#
# The observed maximum discrepancy in the
# previous execution was approximately 0.0001.
#
# The tolerance permits that small discrepancy,
# while larger disagreements still stop execution.

RECONSTRUCTION_TOLERANCE = 1.1e-4


print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 2 — PATCH 2.4")
print("EXTENDED CPTAC MSS PROTEIN ANALYSIS")
print("=" * 65)


# ------------------------------------------------------------
# 2. VALIDATE CPTAC SOURCE FILES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SOURCE DATA VALIDATION")
print("-" * 65)


required_files = [
    SOURCE_MANIFEST,
    CLINICAL_PATH,
    ORIGINAL_PROTEIN_PATH,
    TRANSCRIPTOMIC_PATH
]

for path in required_files:

    if not path.exists():
        raise FileNotFoundError(path)


source = pd.read_csv(
    SOURCE_MANIFEST,
    dtype=str,
    keep_default_na=False
)


if source["dataset"].duplicated().any():

    raise RuntimeError(
        "Duplicate source dataset identifiers."
    )


expected_datasets = {
    "tumor_proteomics",
    "normal_proteomics",
    "paired_proteomics",
    "clinical"
}

if set(source["dataset"]) != expected_datasets:

    raise RuntimeError(
        "Unexpected source dataset inventory."
    )


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            digest.update(chunk)

    return digest.hexdigest()


source_paths = {}

for _, row in source.iterrows():

    path = RAW / row["filename"]

    if not path.exists():

        raise FileNotFoundError(path)

    if sha256_file(path) != row["sha256"]:

        raise RuntimeError(
            f"Checksum mismatch: {row['dataset']}"
        )

    source_paths[row["dataset"]] = path


print("\nSource checksum validation: PASSED")


# ------------------------------------------------------------
# 3. LOAD PROTEOMICS MATRICES
# ------------------------------------------------------------

def load_matrix(dataset):

    df = pd.read_csv(
        source_paths[dataset],
        sep="\t",
        index_col=0,
        na_values=["NA", "N/A", ""],
        low_memory=False
    )

    df.index = (
        df.index.astype(str).str.strip()
    )

    df.columns = (
        df.columns.astype(str).str.strip()
    )

    if df.index.duplicated().any():

        raise RuntimeError(
            f"Duplicate protein identifiers: {dataset}"
        )

    if df.columns.duplicated().any():

        raise RuntimeError(
            f"Duplicate sample identifiers: {dataset}"
        )

    df = df.apply(
        pd.to_numeric,
        errors="raise"
    )

    if np.isinf(
        df.to_numpy(dtype=float)
    ).any():

        raise RuntimeError(
            f"Infinite values: {dataset}"
        )

    return df


tumor = load_matrix("tumor_proteomics")

normal = load_matrix("normal_proteomics")

reported = load_matrix("paired_proteomics")


print("\nMatrices loaded.")

print("Tumor:", tumor.shape)
print("Normal:", normal.shape)
print("Reported paired:", reported.shape)


if tumor.shape != (8067, 97):

    raise RuntimeError(
        "Unexpected tumor matrix."
    )

if normal.shape != (8067, 100):

    raise RuntimeError(
        "Unexpected normal matrix."
    )

if reported.shape != (6422, 96):

    raise RuntimeError(
        "Unexpected paired matrix."
    )


if set(tumor.index) != set(normal.index):

    raise RuntimeError(
        "Tumor and normal protein identifiers differ."
    )


if not set(reported.index).issubset(
    set(tumor.index)
):

    raise RuntimeError(
        "Reported proteins missing from larger assay."
    )


print("\nProtein identifier validation: PASSED")


# ------------------------------------------------------------
# 4. LOAD PUBLISHED MSS ANNOTATIONS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("MSS PATIENT IDENTIFICATION")
print("-" * 65)


clinical = pd.read_csv(
    CLINICAL_PATH,
    dtype=str,
    keep_default_na=False
)


possible_id_columns = [
    "cptac_patient_id",
    "index",
    "Unnamed: 0"
]


id_columns = [
    column
    for column in possible_id_columns
    if column in clinical.columns
]


if len(id_columns) != 1:

    raise RuntimeError(
        "Ambiguous CPTAC clinical identifier column."
    )


clinical = clinical.rename(
    columns={
        id_columns[0]: "cptac_patient_id"
    }
)


if "msi_classification" not in clinical.columns:

    raise RuntimeError(
        "Missing MSS clinical classification."
    )


clinical["cptac_patient_id"] = (
    clinical["cptac_patient_id"]
    .str.strip()
)


if clinical["cptac_patient_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate clinical patients."
    )


clinical = clinical.set_index(
    "cptac_patient_id"
)


paired_patients = reported.columns.tolist()


missing_clinical = (
    set(paired_patients)
    - set(clinical.index)
)


if missing_clinical:

    raise RuntimeError(
        "Missing clinical annotations for paired patients."
    )


status = (
    clinical.loc[
        paired_patients,
        "msi_classification"
    ]
    .astype(str)
    .str.strip()
    .str.upper()
)


mss_patients = status.index[
    status.eq("MSS")
].tolist()


print("\nPublished MSS patients:", len(mss_patients))


if len(mss_patients) != 73:

    raise RuntimeError(
        "Unexpected MSS patient count."
    )


# Confirm tumor and normal sample names.

missing_tumors = [
    p for p in mss_patients
    if p not in tumor.columns
]

missing_normals = [
    p for p in mss_patients
    if p + "N" not in normal.columns
]


if missing_tumors or missing_normals:

    raise RuntimeError(
        "Incomplete MSS sample identifier mapping."
    )


print("Tumor/normal patient correspondence: PASSED")


# ------------------------------------------------------------
# 5. RECONSTRUCT COMPLETE PROTEIN DIFFERENCES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("RECONSTRUCTING PROTEIN DIFFERENCES")
print("-" * 65)


# Align all 8,067 protein rows and 73 MSS patients.
#
# Tumor columns:
# 01CO005
#
# Normal columns:
# 01CO005N
#
# This subtraction follows the relationship
# investigated in Patch 2.3.
#
# Quantitative equivalence will be checked
# against the published paired measurements.


all_genes = tumor.index.tolist()


tumor_mss = tumor.loc[
    all_genes,
    mss_patients
].copy()


normal_mss = normal.loc[
    all_genes,
    [p + "N" for p in mss_patients]
].copy()


normal_mss.columns = mss_patients


if not tumor_mss.index.equals(
    normal_mss.index
):

    raise RuntimeError(
        "Protein row alignment failed."
    )


if not tumor_mss.columns.equals(
    normal_mss.columns
):

    raise RuntimeError(
        "Patient column alignment failed."
    )


reconstructed = (
    tumor_mss - normal_mss
)


if reconstructed.shape != (8067, 73):

    raise RuntimeError(
        "Unexpected reconstructed matrix dimensions."
    )


print("\nReconstructed matrix:", reconstructed.shape)


print(
    "Complete protein-patient observations:",
    int(
        reconstructed.notna()
        .to_numpy()
        .sum()
    )
)


# ============================================================
# 6. REPORTED-RATIO NUMERICAL VALIDATION
#
# CORRECTED:
#
# Allow discrepancies up to 0.00011.
#
# The previous run showed:
# - 419,118 comparable observations
# - Zero missingness disagreements
# - Median absolute difference = 0
# - Maximum absolute difference = 0.0001
#
# These results are consistent with very
# small representation/rounding differences.
#
# We retain strict missingness validation
# and explicitly report numerical discrepancies.
# ============================================================

print("\n" + "-" * 65)
print("REPORTED-RATIO NUMERICAL VALIDATION")
print("-" * 65)


reported_subset = reported.loc[
    reported.index,
    mss_patients
]


derived_subset = reconstructed.loc[
    reported.index,
    mss_patients
]


expected = reported_subset.to_numpy(
    dtype=float
)

observed = derived_subset.to_numpy(
    dtype=float
)


expected_finite = np.isfinite(expected)

observed_finite = np.isfinite(observed)


missingness_disagreements = int(
    np.count_nonzero(
        expected_finite != observed_finite
    )
)


valid = (
    expected_finite & observed_finite
)


n_valid = int(valid.sum())


if n_valid == 0:

    raise RuntimeError(
        "No comparable reported measurements."
    )


absolute_errors = np.abs(
    expected[valid] - observed[valid]
)


max_absolute_error = float(
    np.max(absolute_errors)
)


median_absolute_error = float(
    np.median(absolute_errors)
)


p95_absolute_error = float(
    np.percentile(
        absolute_errors,
        95
    )
)


# Report how many measurements exceed
# the previous strict tolerance.

n_above_previous_tolerance = int(
    np.count_nonzero(
        absolute_errors > 1e-6
    )
)


# Also report how many exceed the new
# permitted tolerance.

n_above_reconstruction_tolerance = int(
    np.count_nonzero(
        absolute_errors > RECONSTRUCTION_TOLERANCE
    )
)


print("\nCompared measurements:", n_valid)


print(
    "Missingness disagreements:",
    missingness_disagreements
)


print(
    "Median absolute difference:",
    f"{median_absolute_error:.12g}"
)


print(
    "95th percentile absolute difference:",
    f"{p95_absolute_error:.12g}"
)


print(
    "Maximum absolute difference:",
    f"{max_absolute_error:.12g}"
)


print(
    "Measurements differing by more than 1e-6:",
    n_above_previous_tolerance
)


print(
    "Permitted numerical tolerance:",
    RECONSTRUCTION_TOLERANCE
)


print(
    "Measurements exceeding permitted tolerance:",
    n_above_reconstruction_tolerance
)


# ------------------------------------------------------------
# 6A. Validate missingness
# ------------------------------------------------------------

if missingness_disagreements != 0:

    raise RuntimeError(
        "Reported and reconstructed missingness differ."
    )


# ------------------------------------------------------------
# 6B. Validate numerical differences
# ------------------------------------------------------------

if max_absolute_error > RECONSTRUCTION_TOLERANCE:

    raise RuntimeError(
        "Reconstructed measurements exceed "
        "the permitted numerical tolerance."
    )


print("\nFull overlap correspondence: PASSED")


print(
    "Small numerical differences have been "
    "documented, not discarded."
)


# ------------------------------------------------------------
# 7. EXTENDED MSS PROTEIN STATISTICS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("EXTENDED MSS PROTEIN STATISTICS")
print("-" * 65)


original_gene_set = set(
    reported.index
)


records = []


for gene_symbol, row in reconstructed.iterrows():

    values = row.to_numpy(
        dtype=float
    )

    observed_values = values[
        np.isfinite(values)
    ]

    n = len(observed_values)


    median_ratio = np.nan
    fraction_positive = np.nan

    positive_count = 0
    negative_count = 0


    if n > 0:

        median_ratio = float(
            np.median(observed_values)
        )

        fraction_positive = float(
            np.mean(observed_values > 0)
        )

        positive_count = int(
            np.count_nonzero(
                observed_values > 0
            )
        )

        negative_count = int(
            np.count_nonzero(
                observed_values < 0
            )
        )


    pvalue = np.nan


    if n >= MIN_OBSERVATIONS:

        if np.all(observed_values == 0):

            pvalue = 1.0

        else:

            pvalue = float(
                wilcoxon(
                    observed_values,
                    alternative="two-sided",
                    zero_method="wilcox",
                    method="auto"
                ).pvalue
            )


    records.append({

        "protein_gene_symbol": gene_symbol,

        "in_original_paired_matrix":
            gene_symbol in original_gene_set,

        "mss_complete_pairs": n,

        "mss_missing_pairs":
            len(mss_patients) - n,

        "measurement_fraction":
            n / len(mss_patients),

        "median_tumor_normal_log2ratio":
            median_ratio,

        "fraction_tumor_higher":
            fraction_positive,

        "positive_patients":
            positive_count,

        "negative_patients":
            negative_count,

        "wilcoxon_pvalue":
            pvalue

    })


extended = pd.DataFrame(
    records
)


print("\nTotal protein rows:", len(extended))


print(
    "Proteins with >=30 complete MSS pairs:",
    int(
        extended["wilcoxon_pvalue"]
        .notna()
        .sum()
    )
)


newly_testable = extended[
    ~extended["in_original_paired_matrix"]
    & extended["wilcoxon_pvalue"].notna()
].copy()


print(
    "Additional proteins eligible for testing:",
    len(newly_testable)
)


# ------------------------------------------------------------
# 8. EXPANDED MULTIPLE-TESTING CORRECTION
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("MULTIPLE-TESTING CORRECTION")
print("-" * 65)


extended["wilcoxon_fdr"] = np.nan


testable = (
    extended["wilcoxon_pvalue"].notna()
)


if not testable.any():

    raise RuntimeError(
        "No eligible protein tests."
    )


adjusted = multipletests(

    extended.loc[
        testable,
        "wilcoxon_pvalue"
    ].to_numpy(dtype=float),

    method="fdr_bh"

)[1]


extended.loc[
    testable,
    "wilcoxon_fdr"
] = adjusted


# ------------------------------------------------------------
# 8A. Apply exploratory enrichment criteria
# ------------------------------------------------------------

extended["protein_enriched"] = (

    extended["mss_complete_pairs"].ge(
        MIN_OBSERVATIONS
    )

    & extended["wilcoxon_fdr"].lt(0.05)

    & extended[
        "median_tumor_normal_log2ratio"
    ].ge(1.0)

    & extended[
        "fraction_tumor_higher"
    ].ge(0.70)

)


print(
    "\nProteins meeting expanded enrichment criteria:",
    int(
        extended["protein_enriched"].sum()
    )
)


print(
    "Newly testable proteins meeting criteria:",
    int(
        (
            ~extended["in_original_paired_matrix"]
            & extended["protein_enriched"]
        ).sum()
    )
)


# ------------------------------------------------------------
# 9. COMPARE WITH ORIGINAL CPTAC RESULTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("ORIGINAL ANALYSIS CONSISTENCY")
print("-" * 65)


original = pd.read_csv(
    ORIGINAL_PROTEIN_PATH
)


if original[
    "protein_gene_symbol"
].duplicated().any():

    raise RuntimeError(
        "Duplicate original protein symbols."
    )


comparison = original[
    [
        "protein_gene_symbol",
        "mss_measurements",
        "median_tumor_normal_log2ratio",
        "protein_enriched"
    ]
].merge(

    extended[
        [
            "protein_gene_symbol",
            "mss_complete_pairs",
            "median_tumor_normal_log2ratio",
            "protein_enriched"
        ]
    ],

    on="protein_gene_symbol",

    how="left",

    suffixes=(
        "_original",
        "_extended"
    ),

    validate="one_to_one"

)


if len(comparison) != 6422:

    raise RuntimeError(
        "Original protein universe changed."
    )


# ------------------------------------------------------------
# 9A. Validate measurement coverage
# ------------------------------------------------------------

if not np.array_equal(

    comparison[
        "mss_measurements"
    ].to_numpy(),

    comparison[
        "mss_complete_pairs"
    ].to_numpy()

):

    raise RuntimeError(
        "Original and extended coverage differ."
    )


# ------------------------------------------------------------
# 9B. Validate median protein differences
#
# CORRECTED:
# Use the same reconstruction tolerance
# established in Section 6.
# ------------------------------------------------------------

original_medians = comparison[
    "median_tumor_normal_log2ratio_original"
].to_numpy(
    dtype=float
)


extended_medians = comparison[
    "median_tumor_normal_log2ratio_extended"
].to_numpy(
    dtype=float
)


if not np.allclose(

    original_medians,

    extended_medians,

    equal_nan=True,

    atol=RECONSTRUCTION_TOLERANCE,

    rtol=0.0

):

    raise RuntimeError(
        "Original and reconstructed protein "
        "medians differ beyond permitted tolerance."
    )


print("\nOriginal protein coverage: PASSED")

print("Original median ratios: PASSED")


original_enriched = (
    comparison[
        "protein_enriched_original"
    ].astype(bool)
)


extended_enriched = (
    comparison[
        "protein_enriched_extended"
    ].astype(bool)
)


print(
    "\nOriginal enriched proteins:",
    int(original_enriched.sum())
)


print(
    "Original enriched proteins retaining "
    "expanded-analysis criteria:",
    int(
        (
            original_enriched
            & extended_enriched
        ).sum()
    )
)


print(
    "\nNOTE: Expanded multiple-testing correction "
    "may change adjusted p-values."
)


# ------------------------------------------------------------
# 10. INTEGRATE TRANSCRIPTOMIC SCREENING GENES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("TRANSCRIPTOMIC / PROTEOMIC INTEGRATION")
print("-" * 65)


transcriptomic = pd.read_csv(
    TRANSCRIPTOMIC_PATH,
    dtype={"gene_id": str}
)


required_columns = {
    "gene_id",
    "gene_name",
    "combined_transcriptomic_support"
}


if not required_columns.issubset(
    transcriptomic.columns
):

    raise RuntimeError(
        "Unexpected transcriptomic table structure."
    )


supported_mask = (

    transcriptomic[
        "combined_transcriptomic_support"
    ]

    .astype(str)

    .str.lower()

    .eq("true")

)


supported = transcriptomic[
    supported_mask
].copy()


if len(supported) != 948:

    raise RuntimeError(
        "Expected 948 transcriptomic screening genes."
    )


supported["gene_name"] = (
    supported["gene_name"]
    .astype(str)
    .str.strip()
)


integrated = supported.merge(

    extended,

    left_on="gene_name",

    right_on="protein_gene_symbol",

    how="left",

    validate="many_to_one"

)


if len(integrated) != 948:

    raise RuntimeError(
        "Transcriptomic row count changed."
    )


covered = (
    integrated[
        "protein_gene_symbol"
    ].notna()
)


testable_integrated = (
    integrated[
        "wilcoxon_pvalue"
    ].notna()
)


enriched_integrated = (

    integrated[
        "protein_enriched"
    ]

    .fillna(False)

    .astype(bool)

)


print("\nTranscriptomic screening genes:", len(integrated))


print(
    "Represented in expanded assay:",
    int(covered.sum())
)


print(
    "With >=30 MSS paired observations:",
    int(testable_integrated.sum())
)


print(
    "Meeting expanded protein enrichment criteria:",
    int(enriched_integrated.sum())
)


additional_supported = integrated[

    enriched_integrated

    & ~integrated[
        "in_original_paired_matrix"
    ]
    .fillna(False)
    .astype(bool)

].copy()


print(
    "New screening genes supported "
    "outside original paired matrix:",
    len(additional_supported)
)


# ------------------------------------------------------------
# 11. REVIEW LEADING GENES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LEADING PROTEIN COVERAGE REVIEW")
print("-" * 65)


leading_genes = [
    "CDH3",
    "CLDN1",
    "CEMIP",
    "INHBA",
    "WNT2",
    "KRT80",
    "FAP",
    "DPEP1",
    "SLC7A5",
    "SLC5A6",
    "CTHRC1",
    "S100P"
]


leading = integrated[
    integrated["gene_name"].isin(
        leading_genes
    )
].copy()


order_lookup = {

    gene: index

    for index, gene in enumerate(
        leading_genes
    )

}


leading["display_order"] = (
    leading["gene_name"]
    .map(order_lookup)
)


leading = leading.sort_values(
    "display_order"
)


print("\n")


print(

    leading[
        [
            "gene_name",
            "in_original_paired_matrix",
            "mss_complete_pairs",
            "median_tumor_normal_log2ratio",
            "fraction_tumor_higher",
            "wilcoxon_fdr",
            "protein_enriched"
        ]
    ]

    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4g}"
    )

)


print("\nIMPORTANT:")

print(
    "Proteins with fewer than 30 paired "
    "measurements remain insufficiently "
    "covered for formal enrichment testing."
)


# ------------------------------------------------------------
# 12. GENERATE PROTEIN COVERAGE FIGURE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENERATING PROTEIN COVERAGE FIGURE")
print("-" * 65)


original_coverage = extended.loc[
    extended["in_original_paired_matrix"],
    "mss_complete_pairs"
].to_numpy()


additional_coverage = extended.loc[
    ~extended["in_original_paired_matrix"],
    "mss_complete_pairs"
].to_numpy()


fig, ax = plt.subplots(
    figsize=(10, 6)
)


bins = np.arange(
    -0.5,
    74.5,
    3
)


ax.hist(
    original_coverage,
    bins=bins,
    alpha=0.6,
    label="Original paired-matrix proteins"
)


ax.hist(
    additional_coverage,
    bins=bins,
    alpha=0.6,
    label="Additional protein rows"
)


ax.axvline(
    MIN_OBSERVATIONS,
    linestyle="--",
    linewidth=1,
    label="Minimum testing coverage"
)


ax.set_xlabel(
    "Complete MSS tumor-normal patient pairs"
)


ax.set_ylabel(
    "Number of proteins"
)


ax.set_title(
    "CPTAC Colon — Extended Protein Coverage"
)


ax.legend()


ax.grid(
    axis="y",
    alpha=0.2
)


fig.tight_layout()


figure_path = (
    FIGURES /
    "cptac_mss_extended_protein_coverage.png"
)


fig.savefig(
    figure_path,
    dpi=200,
    bbox_inches="tight"
)


plt.show()
plt.close(fig)


print("\nCoverage figure saved.")


# ------------------------------------------------------------
# 13. SAVE ANALYSIS OUTPUTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING EXTENDED PROTEOMICS RESULTS")
print("-" * 65)


extended.to_csv(

    RESULTS /
    "cptac_mss_extended_protein_statistics.csv",

    index=False

)


integrated.to_csv(

    RESULTS /
    "cptac_mss_extended_transcriptomic_integration.csv",

    index=False

)


newly_testable.to_csv(

    RESULTS /
    "cptac_mss_newly_testable_proteins.csv",

    index=False

)


additional_supported.to_csv(

    RESULTS /
    "cptac_mss_additional_supported_genes.csv",

    index=False

)


leading.to_csv(

    RESULTS /
    "cptac_mss_extended_leading_gene_review.csv",

    index=False

)


comparison.to_csv(

    RESULTS /
    "cptac_mss_original_extended_consistency.csv",

    index=False

)


print("\nAll extended analysis tables saved.")


# ------------------------------------------------------------
# 14. SAVE RESEARCH PROVENANCE
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "2.4",

    "patch_version": "2.4-corrected",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "study": "CPTAC Prospective Colon Cancer",

    "source": "LinkedOmics",

    "analysis": (
        "Extended paired protein coverage "
        "and MSS enrichment using separate "
        "tumor and normal TMT log-ratio matrices"
    ),

    "published_mss_patients": int(
        len(mss_patients)
    ),

    "protein_rows_assessed": int(
        len(extended)
    ),

    "reported_overlap_measurements": int(
        n_valid
    ),

    "missingness_disagreements": int(
        missingness_disagreements
    ),

    # Numerical reconstruction audit

    "reconstruction_tolerance": float(
        RECONSTRUCTION_TOLERANCE
    ),

    "maximum_overlap_absolute_error": float(
        max_absolute_error
    ),

    "median_overlap_absolute_error": float(
        median_absolute_error
    ),

    "p95_overlap_absolute_error": float(
        p95_absolute_error
    ),

    "measurements_above_previous_1e-6_tolerance":
        int(n_above_previous_tolerance),

    "measurements_above_accepted_tolerance":
        int(n_above_reconstruction_tolerance),

    "reconstruction_validation_passed": True,

    # Statistical analysis

    "minimum_complete_pairs_for_testing":
        MIN_OBSERVATIONS,

    "proteins_tested": int(
        testable.sum()
    ),

    "newly_testable_proteins": int(
        len(newly_testable)
    ),

    "expanded_protein_enriched": int(
        extended["protein_enriched"].sum()
    ),

    # Cross-omics integration

    "transcriptomic_screening_genes": int(
        len(integrated)
    ),

    "expanded_protein_supported_screening_genes":
        int(enriched_integrated.sum()),

    "newly_supported_screening_genes":
        int(len(additional_supported)),

    # Statistical methods

    "statistical_test": (
        "Two-sided Wilcoxon signed-rank test"
    ),

    "multiple_testing": (
        "Benjamini-Hochberg across all "
        "eligible expanded protein tests"
    ),

    "protein_enrichment_criteria": {

        "minimum_mss_pairs": 30,

        "maximum_fdr": 0.05,

        "minimum_median_log2_difference": 1.0,

        "minimum_positive_fraction": 0.70
    },

    # Data handling

    "original_analysis_overwritten": False,

    "protein_imputation_performed": False,

    "tcga_cptac_patient_level_merging": False,

    # Scientific limitations

    "limitations": [

        "Extended values are reconstructed "
        "from the same CPTAC experiment.",

        "This is not independent replication.",

        "Small numerical differences between "
        "reported and reconstructed ratios "
        "are documented and bounded by tolerance.",

        "Sparse protein coverage limits inference.",

        "Missingness may be nonrandom.",

        "Only proteins with sufficient "
        "coverage receive statistical tests.",

        "Expanded testing changes the "
        "multiple-testing universe.",

        "Gene-symbol integration remains provisional.",

        "Tumor tissue includes malignant "
        "and nonmalignant cell populations.",

        "Protein enrichment does not prove "
        "cell-surface accessibility.",

        "No therapeutic targets have "
        "been validated."

    ]
}


with open(

    META /
    "cptac_mss_extended_analysis_provenance.json",

    "w"

) as f:

    json.dump(
        provenance,
        f,
        indent=4
    )


print("\nAnalysis provenance saved.")


# ------------------------------------------------------------
# 15. CREATE RESEARCH CHECKPOINT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING RESEARCH CHECKPOINT")
print("-" * 65)


timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S")


checkpoint = (

    CHECKPOINTS /

    f"SurfaceTarget_CRC_CPTAC_Patch24_{timestamp}.zip"

)


backup_folders = [
    "data/metadata",
    "data/processed",
    "data/raw/cptac_colon",
    "results/tables",
    "results/figures",
    "src",
    "reports",
    "logs"
]


with ZipFile(

    checkpoint,

    "w",

    ZIP_DEFLATED,

    compresslevel=3

) as archive:

    for folder in backup_folders:

        folder_path = ROOT / folder

        if not folder_path.exists():
            continue

        for file_path in folder_path.rglob("*"):

            if file_path.is_file():

                archive.write(

                    file_path,

                    arcname=str(
                        file_path.relative_to(ROOT)
                    )

                )


checkpoint_mib = (

    checkpoint.stat().st_size
    / (1024 ** 2)

)


print("\nCheckpoint created successfully.")


print(
    f"Checkpoint size: "
    f"{checkpoint_mib:.2f} MiB"
)


print("Checkpoint location:")
print(checkpoint)


# ------------------------------------------------------------
# 16. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 2.4 SUMMARY")
print("=" * 65)


print("\nPublished MSS patients:", len(mss_patients))


print(
    "Total protein rows assessed:",
    len(extended)
)


print(
    "Proteins eligible for statistical testing:",
    int(testable.sum())
)


print(
    "Additional proteins eligible for testing:",
    len(newly_testable)
)


print(
    "Proteins meeting expanded enrichment criteria:",
    int(
        extended["protein_enriched"].sum()
    )
)


print(
    "\nTranscriptomic screening genes:",
    len(integrated)
)


print(
    "Screening genes with expanded protein support:",
    int(enriched_integrated.sum())
)


print(
    "New screening genes supported:",
    len(additional_supported)
)


print("\nCross-matrix validation: PASSED")


print(
    "Numerical tolerance:",
    RECONSTRUCTION_TOLERANCE
)


print(
    "Maximum reconstruction difference:",
    f"{max_absolute_error:.12g}"
)


print("\nSaved outputs:")


print("1. cptac_mss_extended_protein_statistics.csv")

print("2. cptac_mss_extended_transcriptomic_integration.csv")

print("3. cptac_mss_newly_testable_proteins.csv")

print("4. cptac_mss_additional_supported_genes.csv")

print("5. cptac_mss_extended_leading_gene_review.csv")

print("6. cptac_mss_original_extended_consistency.csv")

print("7. cptac_mss_extended_analysis_provenance.json")


print("\nFigure:")

print("cptac_mss_extended_protein_coverage.png")


print("\nResearch checkpoint: CREATED")


print(
    f"Checkpoint size: "
    f"{checkpoint_mib:.2f} MiB"
)


print("\nOriginal CPTAC results preserved.")

print("No source measurements modified.")

print("No protein imputation performed.")


print("\nPATCH 2.4 COMPLETE")

print("=" * 65)

In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 3 — PATCH 3.1
#
# HUMAN PROTEIN ATLAS ACQUISITION AND ANNOTATION AUDIT
#
# Objectives:
# 1. Load validated transcriptomic/proteomic evidence
# 2. Download official HPA subcellular annotations
# 3. Download official HPA normal-tissue IHC data
# 4. Preserve source files and SHA256 checksums
# 5. Match annotations using stable Ensembl gene IDs
# 6. Identify reported plasma-membrane localization
# 7. Summarize normal-tissue IHC coverage
# 8. Review leading genes
# 9. Save annotations and provenance
#
# IMPORTANT:
# - No final target ranking
# - No automatic candidate exclusions
# - No assumption of extracellular accessibility
# - No assumption that missing HPA data means absence
# - No changes to TCGA or CPTAC analyses
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np

import requests
import hashlib
import json
import time
import os

from zipfile import ZipFile, is_zipfile


# ------------------------------------------------------------
# 1. PROJECT CONFIGURATION
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"

RAW = ROOT / "data/raw/hpa"

RESULTS = ROOT / "results/tables"

for folder in [META, RAW, RESULTS]:
    folder.mkdir(parents=True, exist_ok=True)

INPUT_PATH = (
    RESULTS /
    "cptac_mss_extended_transcriptomic_integration.csv"
)

print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 3 — PATCH 3.1")
print("HUMAN PROTEIN ATLAS ANNOTATION AUDIT")
print("=" * 65)


# ------------------------------------------------------------
# 2. LOAD EXISTING CROSS-OMICS CANDIDATES
# ------------------------------------------------------------

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        "Patch 2.4 integrated candidate table missing."
    )

candidates = pd.read_csv(
    INPUT_PATH,
    dtype={"gene_id": str},
    low_memory=False
)

required_columns = {
    "gene_id",
    "gene_name",
    "protein_enriched",
    "combined_transcriptomic_support"
}

if not required_columns.issubset(candidates.columns):
    raise RuntimeError(
        "Required candidate columns are missing."
    )

if len(candidates) != 948:
    raise RuntimeError(
        "Unexpected transcriptomic candidate count."
    )

if candidates["gene_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate Ensembl gene IDs in candidate table."
    )

# TCGA Ensembl IDs include version suffixes.
#
# Example:
# ENSG00000062038.15 -> ENSG00000062038
#
# We retain original IDs while creating a
# separate stable identifier for HPA matching.

candidates["ensembl_stable_id"] = (
    candidates["gene_id"]
    .astype(str)
    .str.replace(
        r"\.\d+$",
        "",
        regex=True
    )
)

valid_ensembl = (
    candidates["ensembl_stable_id"]
    .str.fullmatch(r"ENSG\d{11}")
)

if not valid_ensembl.all():
    raise RuntimeError(
        "Unexpected Ensembl gene identifier format."
    )

if candidates["ensembl_stable_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate stable Ensembl identifiers."
    )

candidates["protein_enriched"] = (
    candidates["protein_enriched"]
    .fillna(False)
    .astype(str)
    .str.lower()
    .eq("true")
)

print("\nCandidate table loaded.")

print("Transcriptomic candidates:", len(candidates))

print(
    "Cross-omics-supported genes:",
    int(candidates["protein_enriched"].sum())
)

if int(candidates["protein_enriched"].sum()) != 20:
    raise RuntimeError(
        "Expected 20 cross-omics-supported genes."
    )

print("\nCandidate identifier validation: PASSED")


# ------------------------------------------------------------
# 3. DEFINE OFFICIAL HPA DOWNLOADS
# ------------------------------------------------------------

HPA_SOURCES = {

    "subcellular": {
        "url": (
            "https://www.proteinatlas.org/download/"
            "tsv/subcellular_location.tsv.zip"
        ),
        "filename": "subcellular_location.tsv.zip"
    },

    "normal_ihc": {
        "url": (
            "https://www.proteinatlas.org/download/"
            "tsv/normal_ihc_data.tsv.zip"
        ),
        "filename": "normal_ihc_data.tsv.zip"
    }
}

print("\nOfficial HPA datasets requested:", len(HPA_SOURCES))


# ------------------------------------------------------------
# 4. DOWNLOAD AND INTEGRITY FUNCTIONS
# ------------------------------------------------------------

session = requests.Session()

session.headers.update({
    "User-Agent": "SurfaceTarget-CRC-Research/1.0",
    "Accept": "application/zip,*/*"
})


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


def inspect_zip(path):

    if not is_zipfile(path):
        raise RuntimeError(
            f"File is not a valid ZIP archive: {path}"
        )

    with ZipFile(path) as archive:

        members = [
            info
            for info in archive.infolist()
            if not info.is_dir()
            and info.filename.lower().endswith(".tsv")
        ]

        if len(members) != 1:
            raise RuntimeError(
                "Expected one TSV file per HPA archive. "
                f"Found: {[m.filename for m in members]}"
            )

        member = members[0]

        if member.file_size > 500 * 1024**2:
            raise RuntimeError(
                "Unexpectedly large uncompressed HPA file."
            )

        with archive.open(member) as handle:
            header = (
                handle.readline()
                .decode("utf-8-sig")
                .rstrip("\r\n")
                .split("\t")
            )

    return member.filename, header


def download_hpa(url, destination):

    if destination.exists():

        try:
            inspect_zip(destination)

            print("Existing valid archive reused.")

            return "Existing"

        except Exception:
            print("Existing archive invalid; downloading again.")

    temporary = Path(str(destination) + ".part")

    for attempt in range(1, 4):

        try:

            if temporary.exists():
                temporary.unlink()

            with session.get(
                url,
                stream=True,
                timeout=(30, 120)
            ) as response:

                response.raise_for_status()

                total_bytes = 0

                with open(temporary, "wb") as output:

                    for chunk in response.iter_content(
                        chunk_size=1024 * 1024
                    ):

                        if not chunk:
                            continue

                        output.write(chunk)

                        total_bytes += len(chunk)

                        if total_bytes > 150 * 1024**2:
                            raise RuntimeError(
                                "Download exceeded the safety limit."
                            )

            inspect_zip(temporary)

            os.replace(temporary, destination)

            return "Downloaded"

        except Exception as error:

            print(
                f"Download attempt {attempt}/3 failed: {error}"
            )

            if temporary.exists():
                temporary.unlink()

            if attempt == 3:
                raise

            time.sleep(2 ** attempt)


# ------------------------------------------------------------
# 5. ACQUIRE HPA DATASETS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("DOWNLOADING HUMAN PROTEIN ATLAS DATA")
print("-" * 65)

source_records = []

source_paths = {}

for name, source in HPA_SOURCES.items():

    destination = RAW / source["filename"]

    print("\nDataset:", name)

    status = download_hpa(
        source["url"],
        destination
    )

    archive_member, header = inspect_zip(
        destination
    )

    source_paths[name] = destination

    source_records.append({

        "dataset": name,
        "source_url": source["url"],
        "local_path": str(destination),
        "archive_member": archive_member,
        "download_status": status,
        "size_bytes": destination.stat().st_size,
        "sha256": sha256_file(destination),
        "columns": header

    })

    print("Status:", status)

    print(
        "Compressed size (MiB):",
        round(destination.stat().st_size / 1024**2, 2)
    )

    print("Archive member:", archive_member)

    print("Columns:", header)

print("\nHPA source acquisition: COMPLETE")


# ------------------------------------------------------------
# 6. LOAD SUBCELLULAR LOCATION DATA
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING HPA SUBCELLULAR ANNOTATIONS")
print("-" * 65)

sub_path = source_paths["subcellular"]

with ZipFile(sub_path) as archive:

    member = next(
        info.filename
        for info in archive.infolist()
        if info.filename.lower().endswith(".tsv")
    )

    with archive.open(member) as handle:

        subcellular = pd.read_csv(
            handle,
            sep="\t",
            dtype=str,
            keep_default_na=False
        )

subcellular.columns = (
    subcellular.columns.str.strip()
)

required_sub_columns = {
    "Gene",
    "Gene name",
    "Reliability",
    "Enhanced",
    "Supported",
    "Approved",
    "Uncertain"
}

if not required_sub_columns.issubset(
    subcellular.columns
):
    raise RuntimeError(
        "Unexpected HPA subcellular data schema. "
        f"Missing: {required_sub_columns - set(subcellular.columns)}"
    )

subcellular["Gene"] = (
    subcellular["Gene"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.\d+$", "", regex=True)
)

if subcellular["Gene"].duplicated().any():
    raise RuntimeError(
        "Duplicate HPA subcellular Ensembl IDs."
    )

print("\nTotal HPA subcellular records:", len(subcellular))


# ------------------------------------------------------------
# 7. SELECT CANDIDATE SUBCELLULAR ANNOTATIONS
# ------------------------------------------------------------

candidate_gene_ids = set(
    candidates["ensembl_stable_id"]
)

sub_candidate = subcellular[
    subcellular["Gene"].isin(candidate_gene_ids)
].copy()

sub_candidate = sub_candidate.rename(
    columns={
        "Gene": "ensembl_stable_id",
        "Gene name": "hpa_gene_name",
        "Reliability": "hpa_subcellular_reliability",
        "Enhanced": "hpa_enhanced",
        "Supported": "hpa_supported",
        "Approved": "hpa_approved",
        "Uncertain": "hpa_uncertain"
    }
)

sub_columns = [
    "ensembl_stable_id",
    "hpa_gene_name",
    "hpa_subcellular_reliability",
    "hpa_enhanced",
    "hpa_supported",
    "hpa_approved",
    "hpa_uncertain"
]

sub_candidate = sub_candidate[
    sub_columns
].copy()

print(
    "Candidates with HPA subcellular records:",
    len(sub_candidate)
)


# ------------------------------------------------------------
# 8. IDENTIFY REPORTED PLASMA-MEMBRANE LOCATION
# ------------------------------------------------------------

# This identifies explicitly reported HPA
# plasma-membrane localization.
#
# It does NOT establish:
# - Extracellular epitope accessibility
# - Antibody binding
# - Tumor-cell-specific localization
# - Therapeutic suitability

confident_columns = [
    "hpa_enhanced",
    "hpa_supported",
    "hpa_approved"
]


def combine_locations(row, columns):

    values = [
        str(row[col]).strip()
        for col in columns
        if str(row[col]).strip()
    ]

    return "; ".join(values)


sub_candidate["hpa_reported_locations"] = (
    sub_candidate.apply(
        lambda row: combine_locations(
            row,
            confident_columns
        ),
        axis=1
    )
)

sub_candidate["hpa_plasma_membrane_reported"] = (
    sub_candidate["hpa_reported_locations"]
    .str.contains(
        r"\bplasma membrane\b",
        case=False,
        regex=True,
        na=False
    )
)

sub_candidate["hpa_cell_junctions_reported"] = (
    sub_candidate["hpa_reported_locations"]
    .str.contains(
        r"\bcell junctions?\b",
        case=False,
        regex=True,
        na=False
    )
)

sub_candidate["hpa_uncertain_plasma_membrane"] = (
    sub_candidate["hpa_uncertain"]
    .str.contains(
        r"\bplasma membrane\b",
        case=False,
        regex=True,
        na=False
    )
)

print("\nHPA subcellular annotation summary:")

print(
    "Plasma membrane reported:",
    int(
        sub_candidate[
            "hpa_plasma_membrane_reported"
        ].sum()
    )
)

print(
    "Cell junctions reported:",
    int(
        sub_candidate[
            "hpa_cell_junctions_reported"
        ].sum()
    )
)

print(
    "Uncertain plasma-membrane annotation:",
    int(
        sub_candidate[
            "hpa_uncertain_plasma_membrane"
        ].sum()
    )
)


# ------------------------------------------------------------
# 9. LOAD NORMAL-TISSUE IHC RECORDS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING HPA NORMAL-TISSUE IHC")
print("-" * 65)

# The full HPA normal-tissue table can be large.
#
# Read in chunks and retain only records
# corresponding to our 948 candidate genes.
#
# All original downloaded files are preserved.

ihc_path = source_paths["normal_ihc"]

ihc_chunks = []

total_ihc_records = 0

with ZipFile(ihc_path) as archive:

    member = next(
        info.filename
        for info in archive.infolist()
        if info.filename.lower().endswith(".tsv")
    )

    with archive.open(member) as handle:

        reader = pd.read_csv(
            handle,
            sep="\t",
            dtype=str,
            keep_default_na=False,
            chunksize=75000,
            low_memory=False
        )

        for chunk in reader:

            chunk.columns = (
                chunk.columns.str.strip()
            )

            required_ihc_columns = {
                "Gene",
                "Tissue",
                "Cell type",
                "Level",
                "Reliability"
            }

            if not required_ihc_columns.issubset(
                chunk.columns
            ):
                raise RuntimeError(
                    "Unexpected normal-tissue IHC schema. "
                    f"Missing: {required_ihc_columns - set(chunk.columns)}"
                )

            total_ihc_records += len(chunk)

            chunk["Gene"] = (
                chunk["Gene"]
                .astype(str)
                .str.strip()
                .str.replace(
                    r"\.\d+$",
                    "",
                    regex=True
                )
            )

            matched = chunk[
                chunk["Gene"].isin(candidate_gene_ids)
            ].copy()

            if not matched.empty:
                ihc_chunks.append(matched)

print("\nTotal HPA normal IHC records scanned:",
      total_ihc_records)

if not ihc_chunks:
    raise RuntimeError(
        "No normal-tissue IHC records matched candidates."
    )

ihc_candidate = pd.concat(
    ihc_chunks,
    ignore_index=True
)

print(
    "Candidate normal-tissue IHC records:",
    len(ihc_candidate)
)

print(
    "Candidates with normal-tissue IHC records:",
    ihc_candidate["Gene"].nunique()
)


# ------------------------------------------------------------
# 10. NORMAL-TISSUE EXPRESSION COVERAGE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("NORMAL-TISSUE EXPRESSION SUMMARY")
print("-" * 65)

ihc_candidate["level_normalized"] = (
    ihc_candidate["Level"]
    .astype(str)
    .str.strip()
    .str.lower()
)

print("\nObserved IHC expression labels:")

print(
    ihc_candidate[
        "level_normalized"
    ].value_counts().to_string()
)

# These are descriptive HPA IHC categories.
#
# We do not convert categorical staining
# levels into absolute protein concentrations.
#
# "High" and "Medium" are reported
# expression categories, not safety outcomes.

ihc_candidate["is_high"] = (
    ihc_candidate["level_normalized"]
    .eq("high")
)

ihc_candidate["is_medium_or_high"] = (
    ihc_candidate["level_normalized"]
    .isin(["medium", "high"])
)


summary_records = []

for gene_id, group in ihc_candidate.groupby(
    "Gene",
    sort=False
):

    high = group[
        group["is_high"]
    ]

    medium_high = group[
        group["is_medium_or_high"]
    ]

    summary_records.append({

        "ensembl_stable_id": gene_id,

        "hpa_ihc_records": len(group),

        "hpa_normal_tissues_profiled":
            group["Tissue"].nunique(),

        "hpa_normal_tissues_high":
            high["Tissue"].nunique(),

        "hpa_normal_tissues_medium_high":
            medium_high["Tissue"].nunique(),

        "hpa_high_tissue_names":
            "; ".join(
                sorted(
                    high["Tissue"]
                    .dropna()
                    .unique()
                )
            ),

        "hpa_medium_high_tissue_names":
            "; ".join(
                sorted(
                    medium_high["Tissue"]
                    .dropna()
                    .unique()
                )
            )
    })


ihc_summary = pd.DataFrame(
    summary_records
)

if ihc_summary["ensembl_stable_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate gene-level IHC summaries."
    )

print("\nIHC summaries calculated:",
      len(ihc_summary))


# ------------------------------------------------------------
# 11. INTEGRATE HPA ANNOTATIONS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("INTEGRATING HPA EVIDENCE")
print("-" * 65)

integrated = candidates.merge(
    sub_candidate,
    on="ensembl_stable_id",
    how="left",
    validate="one_to_one"
)

integrated = integrated.merge(
    ihc_summary,
    on="ensembl_stable_id",
    how="left",
    validate="one_to_one"
)

if len(integrated) != 948:
    raise RuntimeError(
        "HPA integration changed candidate count."
    )

integrated["hpa_subcellular_record_available"] = (
    integrated["hpa_gene_name"].notna()
)

integrated["hpa_normal_ihc_record_available"] = (
    integrated["hpa_ihc_records"].notna()
)

print("\nIntegration: PASSED")

print("Candidate genes preserved:", len(integrated))

print(
    "With HPA subcellular records:",
    int(
        integrated[
            "hpa_subcellular_record_available"
        ].sum()
    )
)

print(
    "With normal-tissue IHC records:",
    int(
        integrated[
            "hpa_normal_ihc_record_available"
        ].sum()
    )
)

print(
    "With reported plasma-membrane localization:",
    int(
        integrated[
            "hpa_plasma_membrane_reported"
        ].fillna(False).sum()
    )
)

print(
    "Cross-omics-supported genes:",
    int(integrated["protein_enriched"].sum())
)


# ------------------------------------------------------------
# 12. REVIEW LEADING GENES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LEADING GENE HPA ANNOTATIONS")
print("-" * 65)

leading_genes = [
    "CDH3",
    "CLDN1",
    "FAP",
    "DPEP1",
    "SLC7A5",
    "SLC5A6",
    "CTHRC1",
    "S100P",
    "KRT80",
    "CEMIP",
    "INHBA",
    "WNT2"
]

leading = integrated[
    integrated["gene_name"].isin(leading_genes)
].copy()

leading["display_order"] = (
    leading["gene_name"].map({
        gene: index
        for index, gene in enumerate(leading_genes)
    })
)

leading = leading.sort_values(
    "display_order"
)

display_columns = [
    "gene_name",
    "protein_enriched",
    "hpa_subcellular_record_available",
    "hpa_reported_locations",
    "hpa_plasma_membrane_reported",
    "hpa_normal_tissues_profiled",
    "hpa_normal_tissues_medium_high"
]

print("\n")

print(
    leading[display_columns]
    .to_string(index=False)
)

print("\nIMPORTANT:")

print(
    "HPA subcellular localization is not "
    "equivalent to extracellular accessibility."
)

print(
    "Normal-tissue IHC expression is not "
    "equivalent to demonstrated clinical toxicity."
)

print(
    "Missing HPA annotation is not "
    "evidence of biological absence."
)


# ------------------------------------------------------------
# 13. SAVE HPA AUDIT OUTPUTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING HPA ANNOTATION TABLES")
print("-" * 65)

pd.DataFrame(source_records).to_csv(
    META / "hpa_source_manifest.csv",
    index=False
)

sub_candidate.to_csv(
    RESULTS / "hpa_candidate_subcellular_annotations.csv",
    index=False
)

ihc_candidate.to_csv(
    RESULTS / "hpa_candidate_normal_ihc_records.csv",
    index=False
)

ihc_summary.to_csv(
    RESULTS / "hpa_candidate_normal_ihc_summary.csv",
    index=False
)

integrated.to_csv(
    RESULTS / "surfacetarget_hpa_annotation_audit.csv",
    index=False
)

leading.to_csv(
    RESULTS / "surfacetarget_hpa_leading_gene_review.csv",
    index=False
)

print("\nAll HPA annotation tables saved.")


# ------------------------------------------------------------
# 14. SAVE PROVENANCE
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "3.1",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": "Human Protein Atlas",

    "source_urls": {
        name: source["url"]
        for name, source in HPA_SOURCES.items()
    },

    "source_sha256": {
        record["dataset"]: record["sha256"]
        for record in source_records
    },

    "identifier_matching": (
        "Version-stripped Ensembl gene IDs"
    ),

    "transcriptomic_screening_genes":
        int(len(integrated)),

    "cross_omics_supported_genes":
        int(integrated["protein_enriched"].sum()),

    "subcellular_annotated_candidates":
        int(
            integrated[
                "hpa_subcellular_record_available"
            ].sum()
        ),

    "normal_ihc_annotated_candidates":
        int(
            integrated[
                "hpa_normal_ihc_record_available"
            ].sum()
        ),

    "plasma_membrane_annotation_candidates":
        int(
            integrated[
                "hpa_plasma_membrane_reported"
            ].fillna(False).sum()
        ),

    "final_target_ranking_performed": False,

    "extracellular_topology_verified": False,

    "normal_tissue_safety_established": False,

    "candidate_exclusions": 0,

    "limitations": [
        "HPA IF annotations derive from examined cell models.",
        "Absence of an IF annotation does not establish absence of a protein.",
        "Plasma-membrane localization does not prove extracellular epitope accessibility.",
        "IHC staining is categorical and depends on antibody performance.",
        "Normal-tissue expression does not by itself establish clinical toxicity.",
        "RNA and proteomic measurement limitations remain applicable.",
        "HPA evidence requires integration with UniProt topology and other resources.",
        "No therapeutic targets have been validated."
    ]
}

with open(
    META / "hpa_annotation_audit_provenance.json",
    "w"
) as output:

    json.dump(
        provenance,
        output,
        indent=4
    )

print("\nResearch provenance saved.")


# ------------------------------------------------------------
# 15. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 3.1 SUMMARY")
print("=" * 65)

print("\nTranscriptomic screening genes:", len(integrated))

print(
    "Cross-omics-supported genes:",
    int(integrated["protein_enriched"].sum())
)

print(
    "\nHPA subcellular annotations:",
    int(
        integrated[
            "hpa_subcellular_record_available"
        ].sum()
    )
)

print(
    "HPA normal-tissue IHC annotations:",
    int(
        integrated[
            "hpa_normal_ihc_record_available"
        ].sum()
    )
)

print(
    "Reported plasma-membrane annotations:",
    int(
        integrated[
            "hpa_plasma_membrane_reported"
        ].fillna(False).sum()
    )
)

print("\nSource datasets downloaded or reused: 2")

print("Annotation and audit tables saved: 6")

print("Source manifest and provenance saved.")

print("\nNo candidate genes excluded.")

print("No therapeutic target scores assigned.")

print("No TCGA or CPTAC results overwritten.")

print("\nPATCH 3.1 COMPLETE")

print("=" * 65)

In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 3 — PATCH 3.2
#
# UNIPROT PROTEIN TOPOLOGY AND EXTRACELLULAR-DOMAIN AUDIT
#
# Objectives:
# 1. Load HPA-integrated candidate table
# 2. Map Ensembl gene IDs to UniProtKB entries
# 3. Prefer uniquely mapped reviewed human entries
# 4. Preserve ambiguous and missing mappings
# 5. Extract subcellular localization annotations
# 6. Identify transmembrane and GPI-anchor evidence
# 7. Extract annotated extracellular topological regions
# 8. Integrate results with RNA, CPTAC and HPA evidence
# 9. Review leading candidates
# 10. Save raw source data, tables and provenance
#
# IMPORTANT:
# - No gene-symbol-only mapping
# - No arbitrary choice among reviewed mappings
# - No automatic candidate exclusions
# - No final therapeutic ranking
# - No assumption that membrane association proves
#   extracellular antibody accessibility
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

import pandas as pd
import numpy as np
import requests

import hashlib
import json
import gzip
import re
import time


# ------------------------------------------------------------
# 1. PROJECT CONFIGURATION
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"
RAW = ROOT / "data/raw/uniprot"
RESULTS = ROOT / "results/tables"

for directory in [META, RAW, RESULTS]:
    directory.mkdir(parents=True, exist_ok=True)

INPUT_PATH = (
    RESULTS /
    "surfacetarget_hpa_annotation_audit.csv"
)

RAW_MAPPING_PATH = (
    RAW /
    "uniprot_ensembl_mapping_results.json.gz"
)

print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 3 — PATCH 3.2")
print("UNIPROT PROTEIN TOPOLOGY AUDIT")
print("=" * 65)


# ------------------------------------------------------------
# 2. LOAD CANDIDATE EVIDENCE
# ------------------------------------------------------------

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        "Patch 3.1 HPA annotation table missing."
    )

candidates = pd.read_csv(
    INPUT_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)

required_columns = {
    "gene_id",
    "gene_name",
    "ensembl_stable_id",
    "protein_enriched"
}

if not required_columns.issubset(candidates.columns):
    raise RuntimeError(
        "Required candidate columns missing."
    )

if len(candidates) != 948:
    raise RuntimeError(
        "Expected 948 candidate genes."
    )

if candidates["ensembl_stable_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate stable Ensembl gene IDs."
    )

candidate_ids = (
    candidates["ensembl_stable_id"]
    .astype(str)
    .tolist()
)

candidate_id_set = set(candidate_ids)

input_fingerprint = hashlib.sha256(
    "\n".join(sorted(candidate_ids)).encode("utf-8")
).hexdigest()

print("\nCandidate evidence loaded.")

print("Candidate genes:", len(candidates))

print("Stable Ensembl identifiers: VERIFIED")


# ------------------------------------------------------------
# 3. UNIPROT API CONFIGURATION
# ------------------------------------------------------------

API_BASE = "https://rest.uniprot.org"

session = requests.Session()

retry_policy = Retry(
    total=5,
    backoff_factor=1,
    status_forcelist=[
        429, 500, 502, 503, 504
    ],
    allowed_methods=["GET", "POST"]
)

session.mount(
    "https://",
    HTTPAdapter(max_retries=retry_policy)
)

session.headers.update({
    "User-Agent": "SurfaceTarget-CRC-Research/1.0",
    "Accept": "application/json"
})


def get_json(url, **kwargs):

    response = session.get(
        url,
        timeout=(30, 120),
        **kwargs
    )

    response.raise_for_status()

    return response.json()


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


# ------------------------------------------------------------
# 4. ACQUIRE UNIPROT IDENTIFIER MAPPINGS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("UNIPROT IDENTIFIER MAPPING")
print("-" * 65)

mapping_data = None
mapping_job_id = None

# Reuse previously completed API results
# only when the requested Ensembl IDs match.

if RAW_MAPPING_PATH.exists():

    with gzip.open(
        RAW_MAPPING_PATH,
        "rt",
        encoding="utf-8"
    ) as handle:

        cached = json.load(handle)

    if (
        cached.get("input_fingerprint")
        == input_fingerprint
        and "results" in cached
    ):

        mapping_data = cached

        mapping_job_id = cached.get("job_id")

        print("\nExisting matching UniProt results reused.")


if mapping_data is None:

    print("\nSubmitting Ensembl-to-UniProtKB mapping job.")

    response = session.post(
        f"{API_BASE}/idmapping/run",
        data={
            "from": "Ensembl",
            "to": "UniProtKB",
            "ids": ",".join(candidate_ids)
        },
        timeout=(30, 120)
    )

    response.raise_for_status()

    submission = response.json()

    mapping_job_id = submission.get("jobId")

    if not mapping_job_id:
        raise RuntimeError(
            "UniProt mapping job ID not returned."
        )

    print("Mapping job submitted.")

    print("\nChecking mapping status...")

    completed = False

    for attempt in range(120):

        status_response = get_json(
            f"{API_BASE}/idmapping/status/"
            f"{mapping_job_id}"
        )

        job_status = status_response.get("jobStatus")

        if job_status in {"NEW", "RUNNING"}:

            if attempt % 10 == 0:
                print("Mapping job processing...")

            time.sleep(3)
            continue

        if job_status in {"ERROR", "FAILED"}:
            raise RuntimeError(
                f"UniProt mapping failed: {status_response}"
            )

        # A completed mapping response may contain
        # results or failedIds without jobStatus.

        if (
            job_status == "FINISHED"
            or "results" in status_response
            or "failedIds" in status_response
        ):

            completed = True
            break

        raise RuntimeError(
            f"Unexpected mapping status: {status_response}"
        )

    if not completed:
        raise RuntimeError(
            "UniProt mapping job did not complete "
            "during this execution."
        )

    # Retrieve the official result URL.

    details = get_json(
        f"{API_BASE}/idmapping/details/"
        f"{mapping_job_id}"
    )

    results_url = details.get("redirectURL")

    if not results_url:
        raise RuntimeError(
            "UniProt results URL missing."
        )

    separator = "&" if "?" in results_url else "?"

    next_url = (
        results_url
        + separator
        + "format=json&size=500"
    )

    all_results = []
    failed_ids = set()
    page_count = 0

    # Retrieve all pages, not only the first 500
    # mappings returned by the API.

    while next_url:

        page_count += 1

        if page_count > 100:
            raise RuntimeError(
                "Unexpectedly many UniProt result pages."
            )

        response = session.get(
            next_url,
            timeout=(30, 120)
        )

        response.raise_for_status()

        payload = response.json()

        all_results.extend(
            payload.get("results", [])
        )

        for failed in payload.get("failedIds", []):

            if isinstance(failed, dict):
                failed = failed.get("from", "")

            failed_ids.add(str(failed))

        link_header = response.headers.get(
            "Link", ""
        )

        next_match = re.search(
            r'<([^>]+)>;\s*rel="next"',
            link_header
        )

        next_url = (
            next_match.group(1)
            if next_match
            else None
        )

        print(
            f"Retrieved page {page_count}: "
            f"{len(all_results)} mapping records"
        )

    mapping_data = {
        "source": "UniProt REST API",
        "from_database": "Ensembl",
        "to_database": "UniProtKB",
        "job_id": mapping_job_id,
        "retrieved_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "input_fingerprint": input_fingerprint,
        "input_count": len(candidate_ids),
        "results": all_results,
        "failedIds": sorted(failed_ids)
    }

    # Preserve complete API mapping results,
    # including alternative UniProt entries.

    with gzip.open(
        RAW_MAPPING_PATH,
        "wt",
        encoding="utf-8"
    ) as handle:

        json.dump(
            mapping_data,
            handle,
            separators=(",", ":")
        )

    print("\nRaw UniProt mapping results saved.")


print("\nMapping records:",
      len(mapping_data["results"]))

print("Unmapped identifiers reported:",
      len(mapping_data.get("failedIds", [])))


# ------------------------------------------------------------
# 5. VALIDATE MAPPING RESPONSES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("UNIPROT MAPPING VALIDATION")
print("-" * 65)

mapping_by_gene = defaultdict(dict)

unexpected_source_ids = set()

for record in mapping_data["results"]:

    ensembl_id = str(
        record.get("from", "")
    ).strip()

    if ensembl_id not in candidate_id_set:
        unexpected_source_ids.add(ensembl_id)
        continue

    entry = record.get("to")

    if not isinstance(entry, dict):
        raise RuntimeError(
            "UniProt mapping response lacks "
            "a structured protein entry."
        )

    accession = entry.get(
        "primaryAccession"
    )

    if not accession:
        raise RuntimeError(
            "Mapped UniProt entry missing accession."
        )

    organism = entry.get(
        "organism", {}
    )

    taxon_id = organism.get(
        "taxonId"
    )

    if str(taxon_id) != "9606":
        # Do not accept nonhuman entries.
        continue

    mapping_by_gene[ensembl_id][accession] = entry


if unexpected_source_ids:
    raise RuntimeError(
        "UniProt returned unexpected input IDs: "
        f"{sorted(unexpected_source_ids)[:10]}"
    )

print("\nHuman UniProt mappings parsed.")

print(
    "Candidate genes with any human mapping:",
    len(mapping_by_gene)
)


# ------------------------------------------------------------
# 6. SELECT UNAMBIGUOUS REVIEWED ENTRIES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("REVIEWED ENTRY SELECTION")
print("-" * 65)

selection_records = []

selected_entries = {}


def is_reviewed(entry):

    entry_type = str(
        entry.get("entryType", "")
    ).lower()

    return (
        "reviewed" in entry_type
        and "unreviewed" not in entry_type
    )


for gene_id in candidate_ids:

    entries = list(
        mapping_by_gene.get(
            gene_id, {}
        ).values()
    )

    reviewed = [
        entry
        for entry in entries
        if is_reviewed(entry)
    ]

    if len(reviewed) == 1:

        status = "Unique reviewed mapping"

        selected_entries[gene_id] = reviewed[0]

    elif len(reviewed) > 1:

        status = "Ambiguous reviewed mappings"

    elif entries:

        status = "Unreviewed mappings only"

    else:

        status = "No human UniProt mapping"

    selection_records.append({

        "ensembl_stable_id": gene_id,

        "total_human_uniprot_entries":
            len(entries),

        "reviewed_entries":
            len(reviewed),

        "mapping_status":
            status,

        "reviewed_accessions":
            "; ".join(
                sorted(
                    entry["primaryAccession"]
                    for entry in reviewed
                )
            )
    })


selection_table = pd.DataFrame(
    selection_records
)

print("\nMapping status distribution:")

print(
    selection_table[
        "mapping_status"
    ].value_counts().to_string()
)

print(
    "\nUniquely mapped reviewed proteins:",
    len(selected_entries)
)

print(
    "Ambiguous mappings are retained "
    "for review, not automatically resolved."
)


# ------------------------------------------------------------
# 7. EXTRACT UNIPROT TOPOLOGY ANNOTATIONS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("EXTRACTING PROTEIN TOPOLOGY")
print("-" * 65)


def extract_gene_symbol(entry):

    for gene in entry.get("genes", []):

        name = gene.get(
            "geneName", {}
        )

        if isinstance(name, dict):

            value = name.get("value")

            if value:
                return str(value)

    return ""


def extract_locations(entry):

    locations = []

    for comment in entry.get("comments", []):

        if comment.get("commentType") != (
            "SUBCELLULAR LOCATION"
        ):
            continue

        for item in comment.get(
            "subcellularLocations", []
        ):

            location = item.get(
                "location", {}
            )

            if isinstance(location, dict):

                value = location.get(
                    "value", ""
                )

                if value:
                    locations.append(str(value))

    return sorted(set(locations))


def feature_length(feature):

    location = feature.get(
        "location", {}
    )

    start = location.get(
        "start", {}
    )

    end = location.get(
        "end", {}
    )

    if not isinstance(start, dict):
        return None

    if not isinstance(end, dict):
        return None

    first = start.get("value")
    last = end.get("value")

    if first is None or last is None:
        return None

    try:

        first = int(first)
        last = int(last)

        if last < first:
            return None

        return last - first + 1

    except (TypeError, ValueError):

        return None


annotation_records = []


for ensembl_id, entry in selected_entries.items():

    accession = entry["primaryAccession"]

    features = entry.get(
        "features", []
    )

    locations = extract_locations(
        entry
    )

    location_text = "; ".join(
        locations
    )

    # Identify reported localization.

    cell_membrane = any(
        re.search(
            r"\b(cell|plasma) membrane\b",
            location,
            flags=re.IGNORECASE
        )
        for location in locations
    )

    secreted = any(
        "secreted" in location.lower()
        or "extracellular space" in location.lower()
        for location in locations
    )

    # Examine sequence features.

    transmembrane_features = []

    extracellular_features = []

    signal_features = []

    lipidation_features = []

    for feature in features:

        feature_type = feature.get(
            "type", ""
        )

        description = str(
            feature.get(
                "description", ""
            )
        )

        if feature_type == "Transmembrane":

            transmembrane_features.append(
                feature
            )

        elif feature_type == "Topological domain":

            if re.search(
                r"\bextracellular\b",
                description,
                flags=re.IGNORECASE
            ):

                extracellular_features.append(
                    feature
                )

        elif feature_type == "Signal":

            signal_features.append(
                feature
            )

        elif feature_type == "Lipidation":

            lipidation_features.append(
                feature
            )

    # GPI-anchor annotations may appear
    # in localization comments or features.

    gpi_feature = any(
        "gpi" in str(
            feature.get(
                "description", ""
            )
        ).lower()
        for feature in lipidation_features
    )

    gpi_location = any(
        "gpi" in location.lower()
        for location in locations
    )

    gpi_anchor = (
        gpi_feature or gpi_location
    )

    # Count only explicitly annotated
    # extracellular topological regions.

    extracellular_lengths = [
        feature_length(feature)
        for feature in extracellular_features
    ]

    extracellular_lengths = [
        length
        for length in extracellular_lengths
        if length is not None
    ]

    total_extracellular_aa = (
        sum(extracellular_lengths)
        if extracellular_lengths
        else np.nan
    )

    transmembrane_count = len(
        transmembrane_features
    )

    extracellular_count = len(
        extracellular_features
    )

    # Descriptive evidence category.
    #
    # This is NOT an antibody-accessibility
    # prediction or target ranking.

    if (
        cell_membrane
        and (
            transmembrane_count > 0
            or gpi_anchor
        )
        and extracellular_count > 0
    ):

        topology_category = (
            "Cell membrane with annotated "
            "extracellular region"
        )

    elif (
        cell_membrane
        and (
            transmembrane_count > 0
            or gpi_anchor
        )
    ):

        topology_category = (
            "Cell membrane; extracellular "
            "topology not explicitly annotated"
        )

    elif cell_membrane:

        topology_category = (
            "Cell-membrane localization reported; "
            "attachment mechanism unresolved"
        )

    elif secreted:

        topology_category = (
            "Secreted or extracellular; "
            "not established as membrane-bound"
        )

    elif transmembrane_count > 0:

        topology_category = (
            "Transmembrane protein; "
            "cell-surface localization unresolved"
        )

    else:

        topology_category = (
            "No qualifying cell-surface "
            "topology evidence in extracted annotations"
        )

    sequence = entry.get(
        "sequence", {}
    )

    annotation_records.append({

        "ensembl_stable_id": ensembl_id,

        "uniprot_accession": accession,

        "uniprot_entry_name":
            entry.get("uniProtkbId", ""),

        "uniprot_gene_symbol":
            extract_gene_symbol(entry),

        "uniprot_entry_type":
            entry.get("entryType", ""),

        "uniprot_sequence_length":
            sequence.get("length"),

        "uniprot_subcellular_locations":
            location_text,

        "uniprot_cell_membrane":
            cell_membrane,

        "uniprot_secreted":
            secreted,

        "uniprot_transmembrane_count":
            transmembrane_count,

        "uniprot_signal_peptide":
            len(signal_features) > 0,

        "uniprot_gpi_anchor":
            gpi_anchor,

        "uniprot_extracellular_domain_count":
            extracellular_count,

        "uniprot_annotated_extracellular_aa":
            total_extracellular_aa,

        "uniprot_topology_category":
            topology_category,

        "uniprot_entry_url":
            (
                "https://www.uniprot.org/"
                "uniprotkb/"
                f"{accession}/entry"
            )
    })


annotations = pd.DataFrame(
    annotation_records
)


if annotations["ensembl_stable_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate selected protein annotations."
    )


print("\nProtein topology extraction: COMPLETE")

print("Reviewed protein annotations:", len(annotations))


# ------------------------------------------------------------
# 8. INTEGRATE UNIPROT WITH EXISTING EVIDENCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("INTEGRATING PROTEIN TOPOLOGY EVIDENCE")
print("-" * 65)


integrated = candidates.merge(
    selection_table,
    on="ensembl_stable_id",
    how="left",
    validate="one_to_one"
)

integrated = integrated.merge(
    annotations,
    on="ensembl_stable_id",
    how="left",
    validate="one_to_one"
)


if len(integrated) != 948:
    raise RuntimeError(
        "Candidate count changed during integration."
    )


integrated["uniprot_reviewed_available"] = (
    integrated["uniprot_accession"].notna()
)


# Gene-symbol concordance is recorded for review.
# We do not discard mappings based solely
# on gene-symbol changes or aliases.

integrated["uniprot_gene_symbol_match"] = (
    integrated["gene_name"]
    .astype(str)
    .str.upper()
    .eq(
        integrated["uniprot_gene_symbol"]
        .fillna("")
        .astype(str)
        .str.upper()
    )
)

integrated.loc[
    ~integrated["uniprot_reviewed_available"],
    "uniprot_gene_symbol_match"
] = False


print("\nCandidate integration: PASSED")

print("Total genes:", len(integrated))

print(
    "With uniquely mapped reviewed UniProt entries:",
    int(
        integrated[
            "uniprot_reviewed_available"
        ].sum()
    )
)

print(
    "With UniProt cell-membrane localization:",
    int(
        integrated[
            "uniprot_cell_membrane"
        ].fillna(False).sum()
    )
)

print(
    "With annotated extracellular regions:",
    int(
        integrated[
            "uniprot_extracellular_domain_count"
        ].fillna(0).gt(0).sum()
    )
)

print(
    "With GPI-anchor annotations:",
    int(
        integrated[
            "uniprot_gpi_anchor"
        ].fillna(False).sum()
    )
)

print("\nTopology categories:")

print(
    integrated[
        "uniprot_topology_category"
    ]
    .fillna("UniProt mapping unresolved")
    .value_counts()
    .to_string()
)


# ------------------------------------------------------------
# 9. REVIEW LEADING CANDIDATES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LEADING CANDIDATE TOPOLOGY REVIEW")
print("-" * 65)


leading_genes = [
    "CDH3",
    "CLDN1",
    "FAP",
    "DPEP1",
    "SLC7A5",
    "SLC5A6",
    "CTHRC1",
    "S100P",
    "KRT80",
    "CEMIP",
    "INHBA",
    "WNT2"
]


leading = integrated[
    integrated["gene_name"].isin(
        leading_genes
    )
].copy()


order_lookup = {
    gene: index
    for index, gene in enumerate(
        leading_genes
    )
}

leading["display_order"] = (
    leading["gene_name"].map(
        order_lookup
    )
)

leading = leading.sort_values(
    "display_order"
)


display_columns = [
    "gene_name",
    "mapping_status",
    "uniprot_accession",
    "uniprot_subcellular_locations",
    "uniprot_transmembrane_count",
    "uniprot_gpi_anchor",
    "uniprot_extracellular_domain_count",
    "uniprot_annotated_extracellular_aa"
]


print("\n")

print(
    leading[display_columns]
    .to_string(
        index=False
    )
)


print("\nIMPORTANT:")

print(
    "Annotated extracellular regions support "
    "topological plausibility, not demonstrated "
    "therapeutic antibody accessibility."
)

print(
    "Missing topology annotations must not "
    "be interpreted as evidence of absence."
)


# ------------------------------------------------------------
# 10. REVIEW CROSS-OMICS-SUPPORTED PROTEINS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CROSS-OMICS PROTEIN TOPOLOGY")
print("-" * 65)


cross_omics_mask = (
    integrated["protein_enriched"]
    .fillna(False)
    .astype(str)
    .str.lower()
    .eq("true")
)


cross_omics = integrated[
    cross_omics_mask
].copy()


if len(cross_omics) != 20:
    raise RuntimeError(
        "Expected 20 cross-omics-supported genes."
    )


print("\nCross-omics-supported proteins:", len(cross_omics))

print("\nTopology categories among those proteins:")

print(
    cross_omics[
        "uniprot_topology_category"
    ]
    .fillna("UniProt mapping unresolved")
    .value_counts()
    .to_string()
)


# ------------------------------------------------------------
# 11. SAVE ANNOTATION AND MAPPING TABLES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING UNIPROT ANNOTATION TABLES")
print("-" * 65)


selection_table.to_csv(
    RESULTS /
    "uniprot_ensembl_mapping_audit.csv",
    index=False
)

annotations.to_csv(
    RESULTS /
    "uniprot_candidate_topology_annotations.csv",
    index=False
)

integrated.to_csv(
    RESULTS /
    "surfacetarget_uniprot_topology_integration.csv",
    index=False
)

leading.to_csv(
    RESULTS /
    "surfacetarget_uniprot_leading_gene_review.csv",
    index=False
)

cross_omics.to_csv(
    RESULTS /
    "surfacetarget_crossomics_topology_review.csv",
    index=False
)


print("\nAnnotation tables saved successfully.")


# ------------------------------------------------------------
# 12. SAVE PROVENANCE
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "3.2",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "data_source": "UniProtKB REST API",

    "api_base": API_BASE,

    "mapping_from": "Ensembl",

    "mapping_to": "UniProtKB",

    "mapping_job_id": mapping_job_id,

    "mapping_input_fingerprint":
        input_fingerprint,

    "mapping_input_genes":
        len(candidate_ids),

    "mapping_result_records":
        len(mapping_data["results"]),

    "raw_mapping_path":
        str(RAW_MAPPING_PATH),

    "raw_mapping_sha256":
        sha256_file(RAW_MAPPING_PATH),

    "reviewed_mapping_policy": (
        "Accept exactly one reviewed human "
        "UniProtKB accession per Ensembl gene; "
        "retain ambiguous mappings for review"
    ),

    "uniquely_mapped_reviewed_genes":
        len(selected_entries),

    "cross_omics_supported_genes":
        len(cross_omics),

    "annotation_types": [
        "SUBCELLULAR LOCATION",
        "Transmembrane",
        "Topological domain",
        "Signal",
        "Lipidation"
    ],

    "extracellular_region_rule": (
        "Explicit extracellular description "
        "within UniProt Topological domain features"
    ),

    "candidate_exclusions": 0,

    "antibody_accessibility_verified": False,

    "target_ranking_performed": False,

    "limitations": [

        "UniProt sequence topology does not "
        "prove antibody binding.",

        "Different protein isoforms may have "
        "different topology.",

        "Only uniquely mapped reviewed human "
        "entries receive topology annotations.",

        "Ambiguous mappings remain unresolved.",

        "Missing UniProt annotations are "
        "not negative biological evidence.",

        "GPI attachment and transmembrane "
        "anchoring are distinct mechanisms.",

        "Secreted proteins are not necessarily "
        "cell-surface anchored.",

        "Cell-membrane annotations may reflect "
        "non-tumor biological contexts.",

        "CPTAC protein enrichment does not "
        "establish malignant-cell expression.",

        "Normal-tissue safety is not established.",

        "No therapeutic targets have "
        "been validated."
    ]
}


with open(
    META /
    "uniprot_topology_audit_provenance.json",
    "w"
) as handle:

    json.dump(
        provenance,
        handle,
        indent=4
    )


print("\nResearch provenance saved.")


# ------------------------------------------------------------
# 13. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 3.2 SUMMARY")
print("=" * 65)


print("\nTranscriptomic screening genes:", len(integrated))

print("Cross-omics-supported genes:", len(cross_omics))

print(
    "Uniquely mapped reviewed proteins:",
    len(selected_entries)
)

print(
    "Cell-membrane-localized proteins:",
    int(
        integrated[
            "uniprot_cell_membrane"
        ].fillna(False).sum()
    )
)

print(
    "Proteins with extracellular-domain annotations:",
    int(
        integrated[
            "uniprot_extracellular_domain_count"
        ].fillna(0).gt(0).sum()
    )
)


print("\nFiles saved:")

print("1. uniprot_ensembl_mapping_audit.csv")

print("2. uniprot_candidate_topology_annotations.csv")

print("3. surfacetarget_uniprot_topology_integration.csv")

print("4. surfacetarget_uniprot_leading_gene_review.csv")

print("5. surfacetarget_crossomics_topology_review.csv")

print("6. uniprot_topology_audit_provenance.json")

print("7. uniprot_ensembl_mapping_results.json.gz")


print("\nNo genes automatically excluded.")

print("No therapeutic target scores assigned.")

print("No TCGA, CPTAC or HPA results overwritten.")


print("\nPATCH 3.2 COMPLETE")

print("=" * 65)

In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 3 — PATCH 3.3
#
# DETAILED NORMAL-TISSUE EXPRESSION AND CELL-TYPE AUDIT
#
# Objectives:
# 1. Load validated HPA and UniProt evidence
# 2. Verify HPA source integrity
# 3. Audit IHC expression and reliability categories
# 4. Construct tissue-level expression profiles
# 5. Preserve individual cell-type observations
# 6. Quantify normal-tissue staining breadth
# 7. Compare all observations with reliability-supported data
# 8. Examine leading cell-surface candidates
# 9. Review the 20 cross-omics-supported genes
# 10. Generate a descriptive tissue-expression heatmap
# 11. Save results, provenance and checkpoint
#
# IMPORTANT:
# - No final safety scores
# - No automatic target exclusions
# - Missing IHC observations are NOT zero expression
# - Normal-tissue staining is NOT clinical toxicity
# - IHC staining is NOT antibody-accessibility evidence
# - No previous analyses are overwritten
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import hashlib
import json


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"
RAW = ROOT / "data/raw/hpa"

RESULTS = ROOT / "results/tables"
FIGURES = ROOT / "results/figures"
CHECKPOINTS = ROOT / "checkpoints"

for directory in [
    META,
    RAW,
    RESULTS,
    FIGURES,
    CHECKPOINTS
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


INTEGRATED_PATH = (
    RESULTS /
    "surfacetarget_uniprot_topology_integration.csv"
)

IHC_PATH = (
    RESULTS /
    "hpa_candidate_normal_ihc_records.csv"
)

PREVIOUS_SUMMARY_PATH = (
    RESULTS /
    "hpa_candidate_normal_ihc_summary.csv"
)

SOURCE_PATH = (
    META /
    "hpa_source_manifest.csv"
)


LEADING_GENES = [
    "CDH3",
    "CLDN1",
    "FAP",
    "DPEP1",
    "SLC7A5",
    "SLC5A6",
    "CTHRC1",
    "S100P",
    "KRT80",
    "CEMIP",
    "INHBA",
    "WNT2"
]


# HPA staining levels are ordinal
# descriptive categories, not continuous
# measurements of protein abundance.

LEVEL_SCORES = {
    "not detected": 0,
    "low": 1,
    "medium": 2,
    "high": 3
}

SCORE_LABELS = {
    0: "Not detected",
    1: "Low",
    2: "Medium",
    3: "High"
}


# HPA protein-expression reliability
# categories stronger than "Uncertain".
#
# This is a secondary evidence-stratified
# view, not a guarantee that individual
# staining observations are correct.

RELIABLE_LEVELS = {
    "enhanced",
    "supported",
    "approved"
}


print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 3 — PATCH 3.3")
print("DETAILED NORMAL-TISSUE EXPRESSION AUDIT")
print("=" * 65)


# ------------------------------------------------------------
# 2. VALIDATE REQUIRED FILES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING RESEARCH INPUTS")
print("-" * 65)


required_paths = [
    INTEGRATED_PATH,
    IHC_PATH,
    PREVIOUS_SUMMARY_PATH,
    SOURCE_PATH
]

for path in required_paths:

    if not path.exists():
        raise FileNotFoundError(path)


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


source = pd.read_csv(
    SOURCE_PATH,
    dtype=str,
    keep_default_na=False
)


if source["dataset"].duplicated().any():

    raise RuntimeError(
        "Duplicate HPA source records."
    )


normal_source = source[
    source["dataset"].eq("normal_ihc")
]


if len(normal_source) != 1:

    raise RuntimeError(
        "Expected exactly one HPA normal-IHC source."
    )


source_record = normal_source.iloc[0]

source_file = (
    RAW /
    Path(source_record["local_path"]).name
)


if not source_file.exists():

    raise FileNotFoundError(source_file)


actual_checksum = sha256_file(source_file)


if actual_checksum.lower() != (
    source_record["sha256"].lower()
):

    raise RuntimeError(
        "HPA normal-IHC source checksum mismatch."
    )


print("\nHPA normal-IHC archive: VERIFIED")


# ------------------------------------------------------------
# 3. LOAD INTEGRATED CANDIDATE TABLE
# ------------------------------------------------------------

candidates = pd.read_csv(
    INTEGRATED_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)


required_columns = {
    "gene_id",
    "gene_name",
    "ensembl_stable_id",
    "protein_enriched",
    "uniprot_accession",
    "uniprot_topology_category"
}


if not required_columns.issubset(
    candidates.columns
):

    raise RuntimeError(
        "Unexpected UniProt integration schema."
    )


if len(candidates) != 948:

    raise RuntimeError(
        "Expected 948 candidate genes."
    )


if candidates[
    "ensembl_stable_id"
].duplicated().any():

    raise RuntimeError(
        "Duplicate candidate Ensembl IDs."
    )


candidates["protein_enriched"] = (
    candidates["protein_enriched"]
    .astype(str)
    .str.lower()
    .eq("true")
)


if int(candidates["protein_enriched"].sum()) != 20:

    raise RuntimeError(
        "Expected 20 cross-omics-supported genes."
    )


candidate_gene_ids = set(
    candidates["ensembl_stable_id"]
)


print("\nCandidate integration loaded.")

print("Transcriptomic candidates:", len(candidates))

print(
    "Cross-omics-supported genes:",
    int(candidates["protein_enriched"].sum())
)


# ------------------------------------------------------------
# 4. LOAD NORMAL-TISSUE IHC RECORDS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING HPA NORMAL-TISSUE OBSERVATIONS")
print("-" * 65)


ihc = pd.read_csv(
    IHC_PATH,
    dtype=str,
    keep_default_na=False,
    low_memory=False
)


required_ihc_columns = {
    "Gene",
    "Tissue",
    "Cell type",
    "Level",
    "Reliability"
}


if not required_ihc_columns.issubset(
    ihc.columns
):

    raise RuntimeError(
        "Unexpected HPA IHC record schema."
    )


ihc["Gene"] = (
    ihc["Gene"]
    .astype(str)
    .str.strip()
    .str.replace(
        r"\.\d+$",
        "",
        regex=True
    )
)


for column in [
    "Tissue",
    "Cell type",
    "Level",
    "Reliability"
]:

    ihc[column] = (
        ihc[column]
        .astype(str)
        .str.strip()
    )


if not set(ihc["Gene"]).issubset(
    candidate_gene_ids
):

    raise RuntimeError(
        "IHC records contain genes outside "
        "the candidate inventory."
    )


if ihc.empty:

    raise RuntimeError(
        "HPA candidate IHC table is empty."
    )


print("\nIHC records:", len(ihc))

print(
    "Candidates with IHC observations:",
    ihc["Gene"].nunique()
)

print(
    "Normal tissues represented:",
    ihc["Tissue"].nunique()
)


# ------------------------------------------------------------
# 5. EXPRESSION LEVEL AND RELIABILITY AUDIT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("IHC LEVEL AND RELIABILITY AUDIT")
print("-" * 65)


ihc["level_clean"] = (
    ihc["Level"]
    .str.lower()
)


ihc["reliability_clean"] = (
    ihc["Reliability"]
    .str.lower()
)


ihc["level_score"] = (
    ihc["level_clean"]
    .map(LEVEL_SCORES)
)


ihc["interpretable_level"] = (
    ihc["level_score"].notna()
)


ihc["reliability_supported"] = (
    ihc["reliability_clean"].isin(
        RELIABLE_LEVELS
    )
)


ihc["is_high"] = (
    ihc["level_score"].eq(3)
)


ihc["is_medium_high"] = (
    ihc["level_score"].ge(2)
)


ihc["is_detected"] = (
    ihc["level_score"].ge(1)
)


ihc["is_not_detected"] = (
    ihc["level_score"].eq(0)
)


print("\nObserved expression levels:")

print(
    ihc["level_clean"]
    .value_counts()
    .to_string()
)


print("\nObserved reliability categories:")

print(
    ihc["reliability_clean"]
    .replace("", "missing")
    .value_counts()
    .to_string()
)


nonstandard = ihc[
    ~ihc["interpretable_level"]
].copy()


print(
    "\nRecords without a standard "
    "expression-level interpretation:",
    len(nonstandard)
)


if not nonstandard.empty:

    print(
        nonstandard[
            "level_clean"
        ].value_counts().to_string()
    )


print("\nIMPORTANT:")

print(
    "Nonstandard values such as ascending "
    "or descending are preserved separately."
)

print(
    "They are not treated as high, medium, "
    "low, or undetected expression."
)


# ------------------------------------------------------------
# 6. CONSTRUCT TISSUE-LEVEL PROFILES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CONSTRUCTING TISSUE-LEVEL PROFILES")
print("-" * 65)


# Each gene can have multiple IHC records
# within one tissue because different cell
# types are annotated separately.
#
# The maximum observed level is used only
# as a descriptive tissue-level indicator.
#
# A tissue receives a "High" designation
# when at least one recorded cell type
# has high staining.


tissue_profiles = (
    ihc.groupby(
        ["Gene", "Tissue"],
        sort=True
    )
    .agg(

        n_cell_type_records=(
            "Cell type",
            "size"
        ),

        n_interpretable_records=(
            "level_score",
            "count"
        ),

        max_level_score=(
            "level_score",
            "max"
        ),

        n_high_records=(
            "is_high",
            "sum"
        ),

        n_medium_high_records=(
            "is_medium_high",
            "sum"
        ),

        n_detected_records=(
            "is_detected",
            "sum"
        ),

        n_not_detected_records=(
            "is_not_detected",
            "sum"
        ),

        n_reliability_supported_records=(
            "reliability_supported",
            "sum"
        )

    )
    .reset_index()
)


tissue_profiles["max_level_label"] = (
    tissue_profiles["max_level_score"]
    .map(SCORE_LABELS)
    .fillna("No interpretable level")
)


# ------------------------------------------------------------
# 6A. Reliability-supported subset
# ------------------------------------------------------------

reliable_records = ihc[
    ihc["reliability_supported"]
].copy()


reliable_tissue = (
    reliable_records.groupby(
        ["Gene", "Tissue"]
    )
    .agg(

        max_reliable_score=(
            "level_score",
            "max"
        ),

        reliable_interpretable_records=(
            "level_score",
            "count"
        )

    )
    .reset_index()
)


tissue_profiles = tissue_profiles.merge(

    reliable_tissue,

    on=["Gene", "Tissue"],

    how="left",

    validate="one_to_one"

)


tissue_profiles[
    "reliable_max_level_label"
] = (

    tissue_profiles[
        "max_reliable_score"
    ]

    .map(SCORE_LABELS)

    .fillna(
        "No reliability-supported interpretable record"
    )

)


print(
    "\nGene-tissue profiles:",
    len(tissue_profiles)
)


print(
    "Profiles with reliability-supported "
    "interpretable observations:",
    int(
        tissue_profiles[
            "max_reliable_score"
        ].notna().sum()
    )
)


# ------------------------------------------------------------
# 7. SUMMARIZE NORMAL-TISSUE BREADTH
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CALCULATING NORMAL-TISSUE STAINING BREADTH")
print("-" * 65)


summary_records = []


for gene_id, group in tissue_profiles.groupby(
    "Gene",
    sort=False
):

    interpretable = group[
        group["max_level_score"].notna()
    ]

    high = group[
        group["max_level_score"].eq(3)
    ]

    medium_high = group[
        group["max_level_score"].ge(2)
    ]

    detected = group[
        group["max_level_score"].ge(1)
    ]

    not_detected = group[
        group["max_level_score"].eq(0)
    ]

    reliable_interpretable = group[
        group["max_reliable_score"].notna()
    ]

    reliable_medium_high = group[
        group["max_reliable_score"].ge(2)
    ]

    summary_records.append({

        "ensembl_stable_id": gene_id,

        "ihc_record_available": True,

        "hpa_tissues_with_records":
            len(group),

        "hpa_tissues_interpretable":
            len(interpretable),

        "hpa_tissues_high":
            len(high),

        "hpa_tissues_medium_high":
            len(medium_high),

        "hpa_tissues_detected":
            len(detected),

        "hpa_tissues_not_detected":
            len(not_detected),

        "hpa_tissues_reliability_supported":
            len(reliable_interpretable),

        "hpa_reliable_medium_high_tissues":
            len(reliable_medium_high),

        "hpa_high_tissue_names":
            "; ".join(
                sorted(
                    high["Tissue"].tolist()
                )
            ),

        "hpa_medium_high_tissue_names":
            "; ".join(
                sorted(
                    medium_high["Tissue"].tolist()
                )
            ),

        "hpa_reliable_medium_high_tissue_names":
            "; ".join(
                sorted(
                    reliable_medium_high["Tissue"].tolist()
                )
            )

    })


gene_summary = pd.DataFrame(
    summary_records
)


if gene_summary[
    "ensembl_stable_id"
].duplicated().any():

    raise RuntimeError(
        "Duplicate gene-level IHC summaries."
    )


print(
    "\nGenes with normal-tissue summaries:",
    len(gene_summary)
)


# ------------------------------------------------------------
# 8. CROSS-VALIDATE PATCH 3.1 RESULTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CROSS-VALIDATING PREVIOUS HPA SUMMARY")
print("-" * 65)


previous = pd.read_csv(
    PREVIOUS_SUMMARY_PATH,
    dtype={
        "ensembl_stable_id": str
    }
)


previous_required = {
    "ensembl_stable_id",
    "hpa_normal_tissues_profiled",
    "hpa_normal_tissues_medium_high"
}


if not previous_required.issubset(
    previous.columns
):

    raise RuntimeError(
        "Patch 3.1 IHC summary schema differs."
    )


validation = gene_summary.merge(

    previous[
        list(previous_required)
    ],

    on="ensembl_stable_id",

    how="outer",

    indicator=True,

    validate="one_to_one"

)


if not validation["_merge"].eq("both").all():

    raise RuntimeError(
        "IHC gene coverage changed since Patch 3.1."
    )


profiled_equal = np.array_equal(

    validation[
        "hpa_tissues_with_records"
    ].to_numpy(dtype=int),

    validation[
        "hpa_normal_tissues_profiled"
    ].to_numpy(dtype=int)

)


medium_high_equal = np.array_equal(

    validation[
        "hpa_tissues_medium_high"
    ].to_numpy(dtype=int),

    validation[
        "hpa_normal_tissues_medium_high"
    ].to_numpy(dtype=int)

)


if not profiled_equal:

    raise RuntimeError(
        "Tissue coverage differs from Patch 3.1."
    )


if not medium_high_equal:

    raise RuntimeError(
        "Medium/high tissue counts differ "
        "from Patch 3.1."
    )


print("\nCandidate IHC coverage: PASSED")

print("Tissue counts: PASSED")

print("Medium/high staining counts: PASSED")


# ------------------------------------------------------------
# 9. INTEGRATE WITH RNA, CPTAC, HPA AND UNIPROT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("INTEGRATING NORMAL-TISSUE EVIDENCE")
print("-" * 65)


integrated = candidates.merge(

    gene_summary,

    on="ensembl_stable_id",

    how="left",

    validate="one_to_one"

)


if len(integrated) != 948:

    raise RuntimeError(
        "Candidate count changed during integration."
    )


integrated["ihc_record_available"] = (
    integrated["ihc_record_available"]
    .fillna(False)
    .astype(bool)
)


count_columns = [
    "hpa_tissues_with_records",
    "hpa_tissues_interpretable",
    "hpa_tissues_high",
    "hpa_tissues_medium_high",
    "hpa_tissues_detected",
    "hpa_tissues_not_detected",
    "hpa_tissues_reliability_supported",
    "hpa_reliable_medium_high_tissues"
]


for column in count_columns:

    integrated[column] = (
        integrated[column]
        .fillna(0)
        .astype(int)
    )


# ------------------------------------------------------------
# 9A. Descriptive observation categories
# ------------------------------------------------------------

# These are NOT target-risk grades.
#
# They describe what the available
# normal-tissue IHC annotations report.


def describe_normal_observations(row):

    if not row["ihc_record_available"]:

        return "No HPA normal-IHC record"

    if row["hpa_tissues_interpretable"] == 0:

        return "IHC available; levels not interpretable"

    if row["hpa_tissues_high"] > 0:

        return "High staining reported in normal tissue"

    if row["hpa_tissues_medium_high"] > 0:

        return "Medium staining reported; no high observed"

    if row["hpa_tissues_detected"] > 0:

        return "Low staining only in observed tissues"

    return "Not detected in observed annotated cell types"


integrated["normal_ihc_observation_category"] = (
    integrated.apply(
        describe_normal_observations,
        axis=1
    )
)


print("\nIntegration: PASSED")

print("Candidate genes preserved:", len(integrated))

print(
    "With normal-IHC records:",
    int(integrated["ihc_record_available"].sum())
)


print("\nNormal-tissue observation categories:")

print(
    integrated[
        "normal_ihc_observation_category"
    ]
    .value_counts()
    .to_string()
)


print("\nIMPORTANT:")

print(
    "No IHC record does not mean "
    "no healthy-tissue expression."
)

print(
    "Observed staining breadth is not "
    "a quantitative therapeutic safety score."
)


# ------------------------------------------------------------
# 10. CROSS-OMICS NORMAL-TISSUE REVIEW
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CROSS-OMICS NORMAL-TISSUE REVIEW")
print("-" * 65)


cross_omics = integrated[
    integrated["protein_enriched"]
].copy()


if len(cross_omics) != 20:

    raise RuntimeError(
        "Expected 20 cross-omics-supported genes."
    )


cross_omics = cross_omics.sort_values(
    [
        "hpa_tissues_medium_high",
        "gene_name"
    ],
    ascending=[
        False,
        True
    ]
)


cross_columns = [
    "gene_name",
    "uniprot_topology_category",
    "hpa_tissues_with_records",
    "hpa_tissues_high",
    "hpa_tissues_medium_high",
    "hpa_reliable_medium_high_tissues"
]


print("\nCross-omics-supported proteins:")

print(
    cross_omics[
        cross_columns
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 11. LEADING CANDIDATE REVIEW
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LEADING CANDIDATE NORMAL-TISSUE REVIEW")
print("-" * 65)


leading = integrated[
    integrated["gene_name"].isin(
        LEADING_GENES
    )
].copy()


leading["display_order"] = (
    leading["gene_name"].map({
        gene: index
        for index, gene in enumerate(
            LEADING_GENES
        )
    })
)


leading = leading.sort_values(
    "display_order"
)


leading_columns = [
    "gene_name",
    "protein_enriched",
    "uniprot_cell_membrane",
    "uniprot_gpi_anchor",
    "hpa_tissues_with_records",
    "hpa_tissues_high",
    "hpa_tissues_medium_high",
    "hpa_reliable_medium_high_tissues"
]


print("\n")

print(
    leading[
        leading_columns
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 12. EXAMINE SPECIFIC NORMAL CELL TYPES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LEADING-GENE CELL-TYPE STAINING")
print("-" * 65)


leading_ids = set(
    leading["ensembl_stable_id"]
)


leading_ihc = ihc[
    ihc["Gene"].isin(
        leading_ids
    )
    & ihc["is_medium_high"]
].copy()


leading_ihc = leading_ihc.merge(

    leading[
        [
            "ensembl_stable_id",
            "gene_name"
        ]
    ],

    left_on="Gene",

    right_on="ensembl_stable_id",

    how="left",

    validate="many_to_one"

)


leading_ihc = leading_ihc.sort_values(
    [
        "gene_name",
        "level_score",
        "Tissue",
        "Cell type"
    ],
    ascending=[
        True,
        False,
        True,
        True
    ]
)


print(
    "\nLeading-gene medium/high "
    "normal cell-type observations:",
    len(leading_ihc)
)


for gene in [
    "CDH3",
    "CLDN1",
    "FAP",
    "DPEP1",
    "SLC7A5",
    "SLC5A6"
]:

    subset = leading_ihc[
        leading_ihc["gene_name"].eq(
            gene
        )
    ]

    print("\n" + "-" * 50)

    print("Gene:", gene)

    print(
        "Medium/high cell-type records:",
        len(subset)
    )

    if subset.empty:

        print(
            "No medium/high observations "
            "in the available IHC records."
        )

        continue

    print(
        subset[
            [
                "Tissue",
                "Cell type",
                "Level",
                "Reliability"
            ]
        ]
        .head(8)
        .to_string(index=False)
    )


# ------------------------------------------------------------
# 13. GENERATE NORMAL-TISSUE EXPRESSION HEATMAP
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENERATING NORMAL-TISSUE HEATMAP")
print("-" * 65)


all_tissues = sorted(
    tissue_profiles["Tissue"].unique()
)


leading_lookup = dict(
    zip(
        leading["gene_name"],
        leading["ensembl_stable_id"]
    )
)


heatmap = np.full(

    (
        len(LEADING_GENES),
        len(all_tissues)
    ),

    np.nan,

    dtype=float

)


tissue_lookup = {
    tissue: index
    for index, tissue in enumerate(
        all_tissues
    )
}


for gene_index, gene_name in enumerate(
    LEADING_GENES
):

    gene_id = leading_lookup.get(
        gene_name
    )

    if gene_id is None:
        continue

    rows = tissue_profiles[
        tissue_profiles["Gene"].eq(
            gene_id
        )
    ]

    for _, row in rows.iterrows():

        tissue_index = tissue_lookup[
            row["Tissue"]
        ]

        heatmap[
            gene_index,
            tissue_index
        ] = row[
            "max_level_score"
        ]


fig, ax = plt.subplots(
    figsize=(16, 7)
)


image = ax.imshow(
    heatmap,
    aspect="auto",
    interpolation="nearest",
    vmin=0,
    vmax=3
)


ax.set_xticks(
    np.arange(len(all_tissues))
)


ax.set_xticklabels(
    all_tissues,
    rotation=90,
    fontsize=8
)


ax.set_yticks(
    np.arange(len(LEADING_GENES))
)


ax.set_yticklabels(
    LEADING_GENES
)


ax.set_title(
    "HPA Normal-Tissue IHC — "
    "Leading SurfaceTarget-CRC Candidates"
)


ax.set_xlabel(
    "Normal tissue"
)


ax.set_ylabel(
    "Candidate gene"
)


colorbar = fig.colorbar(
    image,
    ax=ax,
    fraction=0.025,
    pad=0.02
)


colorbar.set_ticks(
    [0, 1, 2, 3]
)


colorbar.set_ticklabels(
    [
        "Not detected",
        "Low",
        "Medium",
        "High"
    ]
)


colorbar.set_label(
    "Maximum observed cell-type staining level"
)


fig.tight_layout()


figure_path = (
    FIGURES /
    "surfacetarget_normal_tissue_ihc_heatmap.png"
)


fig.savefig(
    figure_path,
    dpi=200,
    bbox_inches="tight"
)


plt.show()
plt.close(fig)


print("\nNormal-tissue heatmap saved.")

print(
    "Missing heatmap values indicate "
    "unavailable or uninterpretable IHC data."
)


# ------------------------------------------------------------
# 14. SAVE ANALYSIS TABLES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING NORMAL-TISSUE AUDIT RESULTS")
print("-" * 65)


ihc.to_csv(

    RESULTS /
    "hpa_candidate_celltype_staining_audit.csv",

    index=False

)


tissue_profiles.to_csv(

    RESULTS /
    "hpa_candidate_tissue_level_profiles.csv",

    index=False

)


gene_summary.to_csv(

    RESULTS /
    "hpa_candidate_detailed_tissue_summary.csv",

    index=False

)


integrated.to_csv(

    RESULTS /
    "surfacetarget_normal_tissue_integration.csv",

    index=False

)


cross_omics.to_csv(

    RESULTS /
    "surfacetarget_crossomics_normal_tissue_review.csv",

    index=False

)


leading_ihc.to_csv(

    RESULTS /
    "surfacetarget_leading_normal_celltypes.csv",

    index=False

)


leading.to_csv(

    RESULTS /
    "surfacetarget_leading_normal_tissue_review.csv",

    index=False

)


nonstandard.to_csv(

    RESULTS /
    "hpa_nonstandard_ihc_level_review.csv",

    index=False

)


print("\nAll normal-tissue audit tables saved.")


# ------------------------------------------------------------
# 15. SAVE RESEARCH PROVENANCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING RESEARCH PROVENANCE")
print("-" * 65)


provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "3.3",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": "Human Protein Atlas",

    "source_dataset": "normal_ihc_data.tsv.zip",

    "source_sha256": actual_checksum,

    "candidate_count": int(
        len(integrated)
    ),

    "cross_omics_supported_count": int(
        len(cross_omics)
    ),

    "ihc_records": int(
        len(ihc)
    ),

    "ihc_annotated_genes": int(
        len(gene_summary)
    ),

    "gene_tissue_profiles": int(
        len(tissue_profiles)
    ),

    "normal_tissues_represented": int(
        len(all_tissues)
    ),

    "nonstandard_expression_records": int(
        len(nonstandard)
    ),

    "expression_level_mapping":
        LEVEL_SCORES,

    "reliability_supported_categories":
        sorted(RELIABLE_LEVELS),

    "tissue_level_aggregation": (
        "Maximum observed categorical IHC "
        "level across annotated cell types"
    ),

    "reliability_analysis": (
        "All observations retained, with "
        "a separate Enhanced/Supported/Approved "
        "evidence-stratified summary"
    ),

    "previous_hpa_summary_validation": True,

    "normal_tissue_safety_score_created": False,

    "therapeutic_target_ranking_performed": False,

    "candidate_exclusions": 0,

    "limitations": [

        "IHC expression levels are ordinal categories.",

        "Maximum tissue staining represents "
        "the highest observed annotated cell type.",

        "Tissue-level summaries can obscure "
        "important cell-type differences.",

        "HPA reliability is not a direct "
        "measure of therapeutic accessibility.",

        "Uncertain reliability records are "
        "retained rather than discarded.",

        "Uninterpretable expression labels "
        "are not converted to numeric levels.",

        "Missing IHC measurements are "
        "not treated as absent expression.",

        "Normal-tissue staining does not "
        "establish clinical toxicity.",

        "Tumor and normal measurements "
        "originate from different resources.",

        "No cross-platform quantitative "
        "tumor-normal ratios were calculated.",

        "Antibody binding and internalization "
        "have not been established.",

        "No therapeutic targets have been validated."

    ]
}


with open(

    META /
    "hpa_detailed_normal_tissue_provenance.json",

    "w"

) as handle:

    json.dump(
        provenance,
        handle,
        indent=4
    )


print("\nResearch provenance saved.")


# ------------------------------------------------------------
# 16. CREATE PHASE 3 RESEARCH CHECKPOINT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING PHASE 3 RESEARCH CHECKPOINT")
print("-" * 65)


timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S"
)


checkpoint = (

    CHECKPOINTS /

    f"SurfaceTarget_CRC_Phase3_Patch33_{timestamp}.zip"

)


backup_folders = [
    "data/metadata",
    "data/processed",
    "data/raw/cptac_colon",
    "data/raw/hpa",
    "data/raw/uniprot",
    "results/tables",
    "results/figures",
    "src",
    "reports",
    "logs"
]


with ZipFile(
    checkpoint,
    "w",
    ZIP_DEFLATED,
    compresslevel=3
) as archive:

    for folder in backup_folders:

        folder_path = ROOT / folder

        if not folder_path.exists():
            continue

        for file_path in folder_path.rglob("*"):

            if file_path.is_file():

                archive.write(

                    file_path,

                    arcname=str(
                        file_path.relative_to(ROOT)
                    )

                )


checkpoint_mib = (
    checkpoint.stat().st_size
    / (1024 ** 2)
)


print("\nCheckpoint created.")

print(
    "Checkpoint size:",
    f"{checkpoint_mib:.2f} MiB"
)

print("Checkpoint location:")
print(checkpoint)


# ------------------------------------------------------------
# 17. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 3.3 SUMMARY")
print("=" * 65)


print(
    "\nTranscriptomic candidates:",
    len(integrated)
)

print(
    "Cross-omics-supported genes:",
    len(cross_omics)
)

print(
    "Genes with normal-tissue IHC:",
    int(
        integrated[
            "ihc_record_available"
        ].sum()
    )
)

print(
    "Gene-tissue profiles constructed:",
    len(tissue_profiles)
)

print(
    "Individual cell-type records audited:",
    len(ihc)
)

print(
    "Nonstandard expression records preserved:",
    len(nonstandard)
)


print("\nPrevious HPA summary validation: PASSED")

print("Normal-tissue audit tables: SAVED")

print("Normal-tissue heatmap: SAVED")

print("Research provenance: SAVED")

print("Phase 3 checkpoint: CREATED")


print("\nNo final target ranking.")

print("No safety scores assigned.")

print("No candidate genes excluded.")

print("No previous analyses overwritten.")


print("\nPATCH 3.3 COMPLETE")

print("=" * 65)

In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 4 — PATCH 4.1
#
# DEPMAP RELEASE AND INPUT DATA AUDIT
#
# Objectives:
# 1. Validate Phase 3 candidate evidence
# 2. Establish DepMap 26Q3 as the intended release
# 3. Check official DepMap release metadata
# 4. Prepare directories for CRISPR and model data
# 5. Detect and inspect any existing source files
# 6. Preserve a reproducible candidate reference
# 7. Save the Phase 4 research plan and provenance
#
# IMPORTANT:
# - No CRISPR dependency analysis yet
# - No patient-level TCGA/CPTAC/DepMap merging
# - No automatic use of older DepMap releases
# - No bypassing DepMap browser verification
# - No candidate exclusions
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import requests

import hashlib
import json
import io
import re


# ------------------------------------------------------------
# 1. Project configuration
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"
RESULTS = ROOT / "results/tables"

DEPMAP_RAW = (
    ROOT / "data/raw/depmap/26Q3"
)

for directory in [
    META,
    RESULTS,
    DEPMAP_RAW
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

CANDIDATE_PATH = (
    RESULTS /
    "surfacetarget_normal_tissue_integration.csv"
)

RELEASE = "26Q3"

REQUIRED_FILES = [
    "Model.csv",
    "CRISPRGeneEffect.csv"
]

CATALOGUE_URL = (
    "https://depmap.org/portal/"
    "api/no-captcha/download/files"
)

PORTAL_URL = (
    "https://depmap.org/portal/data_page/"
)

print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 4 — PATCH 4.1")
print("DEPMAP RELEASE AND INPUT AUDIT")
print("=" * 65)


# ------------------------------------------------------------
# 2. Load current research candidates
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING EXISTING CANDIDATE EVIDENCE")
print("-" * 65)

if not CANDIDATE_PATH.exists():
    raise FileNotFoundError(
        "Phase 3 integrated candidate table missing."
    )

candidates = pd.read_csv(
    CANDIDATE_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)

required_columns = {
    "gene_id",
    "gene_name",
    "ensembl_stable_id",
    "protein_enriched",
    "uniprot_accession",
    "uniprot_topology_category"
}

missing_columns = (
    required_columns - set(candidates.columns)
)

if missing_columns:
    raise RuntimeError(
        f"Missing candidate columns: {missing_columns}"
    )

if len(candidates) != 948:
    raise RuntimeError(
        "Expected 948 candidate genes."
    )

if candidates["gene_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate transcriptomic gene IDs."
    )

if candidates[
    "ensembl_stable_id"
].duplicated().any():
    raise RuntimeError(
        "Duplicate stable Ensembl IDs."
    )

protein_supported = (
    candidates["protein_enriched"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)

if int(protein_supported.sum()) != 20:
    raise RuntimeError(
        "Expected 20 cross-omics-supported genes."
    )

print("\nCandidate table validated.")

print("Transcriptomic candidates:", len(candidates))

print(
    "Cross-omics-supported candidates:",
    int(protein_supported.sum())
)


# ------------------------------------------------------------
# 3. Freeze Phase 4 gene reference
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING DEPMAP GENE REFERENCE")
print("-" * 65)

reference = candidates[
    [
        "gene_id",
        "ensembl_stable_id",
        "gene_name",
        "protein_enriched",
        "uniprot_accession",
        "uniprot_topology_category"
    ]
].copy()

reference["protein_enriched"] = (
    protein_supported.to_numpy(dtype=bool)
)

reference["gene_name"] = (
    reference["gene_name"]
    .astype("string")
    .str.strip()
)

missing_symbols = int(
    reference["gene_name"].isna().sum()
)

duplicate_symbols = int(
    reference["gene_name"]
    .dropna()
    .duplicated()
    .sum()
)

print("\nGenes in reference:", len(reference))

print("Missing gene symbols:", missing_symbols)

print(
    "Repeated gene symbols:",
    duplicate_symbols
)

print(
    "\nGene symbols are retained for inspection."
)

print(
    "DepMap gene identifiers will be validated "
    "before any CRISPR integration."
)

REFERENCE_PATH = (
    META / "depmap_phase4_candidate_reference.csv"
)

reference.to_csv(
    REFERENCE_PATH,
    index=False
)


# ------------------------------------------------------------
# 4. Retrieve official release catalogue
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CHECKING OFFICIAL DEPMAP RELEASE METADATA")
print("-" * 65)

catalogue = None
catalogue_status = "Unavailable"

try:

    response = requests.get(
        CATALOGUE_URL,
        headers={
            "User-Agent":
                "SurfaceTarget-CRC-Academic-Research/1.0",
            "Accept": "text/csv"
        },
        timeout=(15, 45)
    )

    response.raise_for_status()

    beginning = response.text[:1000].lower()

    if (
        "<html" in beginning
        or "<!doctype html" in beginning
    ):
        raise ValueError(
            "DepMap returned browser verification "
            "instead of release metadata."
        )

    parsed = pd.read_csv(
        io.StringIO(response.text),
        dtype=str
    )

    required_catalogue_columns = {
        "release",
        "filename"
    }

    if not required_catalogue_columns.issubset(
        parsed.columns
    ):
        raise ValueError(
            "Unexpected DepMap catalogue schema."
        )

    catalogue = parsed

    catalogue_status = "Retrieved"

    print("\nOfficial catalogue retrieved.")

    print("Catalogue records:", len(catalogue))

except Exception as error:

    print("\nOfficial catalogue unavailable.")

    print("Reason:", str(error))

    print(
        "This does not indicate that "
        "the DepMap release is unavailable."
    )


# ------------------------------------------------------------
# 5. Inspect intended release
# ------------------------------------------------------------

release_files = pd.DataFrame()

if catalogue is not None:

    release_mask = (
        catalogue["release"]
        .astype(str)
        .str.contains(
            r"26Q3",
            case=False,
            regex=True,
            na=False
        )
    )

    release_rows = catalogue[
        release_mask
    ].copy()

    release_files = release_rows[
        release_rows["filename"].isin(
            REQUIRED_FILES
        )
    ].copy()

    print("\n26Q3 catalogue entries:")

    print(
        release_files.to_string(
            index=False
        )
    )

    release_files.to_csv(
        META / "depmap_26q3_source_catalogue.csv",
        index=False
    )

    for filename in REQUIRED_FILES:

        matches = release_files[
            release_files["filename"].eq(filename)
        ]

        print(
            f"{filename}: "
            f"{len(matches)} catalogue entry/entries"
        )


# ------------------------------------------------------------
# 6. Audit local DepMap files
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("DEPMAP INPUT FILE AVAILABILITY")
print("-" * 65)

input_records = []

for filename in REQUIRED_FILES:

    path = DEPMAP_RAW / filename

    exists = path.exists()

    size_mib = (
        path.stat().st_size / (1024**2)
        if exists else np.nan
    )

    header_columns = []

    header_valid = False

    validation_note = "Not yet provided"

    if exists:

        try:

            preview = pd.read_csv(
                path,
                nrows=2,
                low_memory=False
            )

            header_columns = (
                preview.columns.astype(str).tolist()
            )

            header_valid = (
                "ModelID" in header_columns
            )

            if not header_valid:

                validation_note = (
                    "ModelID column not found; "
                    "requires review"
                )

            elif filename == "CRISPRGeneEffect.csv":

                if len(header_columns) < 1000:

                    header_valid = False

                    validation_note = (
                        "Unexpectedly few gene columns"
                    )

                else:

                    validation_note = (
                        "Preliminary header validation passed"
                    )

            else:

                validation_note = (
                    "Preliminary header validation passed"
                )

        except Exception as error:

            validation_note = str(error)

    input_records.append({

        "release": RELEASE,

        "filename": filename,

        "local_path": str(path),

        "available": exists,

        "size_mib": size_mib,

        "header_valid": header_valid,

        "header_preview":
            "; ".join(header_columns[:8]),

        "validation_note": validation_note

    })


input_audit = pd.DataFrame(
    input_records
)

print("\n")

print(
    input_audit[
        [
            "filename",
            "available",
            "size_mib",
            "header_valid",
            "validation_note"
        ]
    ].to_string(index=False)
)

input_audit.to_csv(
    META / "depmap_26q3_input_audit.csv",
    index=False
)


# ------------------------------------------------------------
# 7. Record analysis strategy
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("DEPMAP ANALYSIS STRATEGY")
print("-" * 65)

analysis_strategy = {

    "primary_focus":
        "Colorectal cancer cell models",

    "model_annotation_source":
        "DepMap Public 26Q3 Model.csv",

    "dependency_source":
        "DepMap Public 26Q3 CRISPRGeneEffect.csv",

    "model_selection": (
        "Identify colorectal models using "
        "the release's actual OncoTree metadata; "
        "review colon and rectal subgroups."
    ),

    "msi_handling": (
        "Do not assign MSS status without "
        "verified model-specific MSI annotations."
    ),

    "dependency_interpretation": (
        "More negative Chronos gene-effect "
        "scores indicate greater knockout "
        "fitness effects."
    ),

    "primary_analyses": [
        "Colorectal model dependency distribution",
        "Fraction of models with strong negative effects",
        "Comparison with noncolorectal lineages",
        "Cancer-lineage selectivity assessment",
        "Candidate-level coverage audit"
    ],

    "important_limitation": (
        "CRISPR fitness dependency is "
        "not a necessary condition for "
        "antibody-mediated therapeutic targeting."
    )
}


# ------------------------------------------------------------
# 8. Save provenance
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "4.1",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "intended_depmap_release":
        "DepMap Public 26Q3",

    "official_download_page": PORTAL_URL,

    "official_catalogue_url": CATALOGUE_URL,

    "catalogue_status": catalogue_status,

    "candidate_count": int(
        len(reference)
    ),

    "cross_omics_supported_count": 20,

    "input_files_required": REQUIRED_FILES,

    "input_files_available": int(
        input_audit["available"].sum()
    ),

    "preliminary_validated_files": int(
        input_audit["header_valid"].sum()
    ),

    "analysis_strategy": analysis_strategy,

    "crispr_analysis_performed": False,

    "target_ranking_performed": False,

    "candidate_exclusions": 0
}


with open(
    META / "depmap_phase4_setup_provenance.json",
    "w"
) as handle:

    json.dump(
        provenance,
        handle,
        indent=4
    )


# ------------------------------------------------------------
# 9. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 4.1 SUMMARY")
print("=" * 65)

print("\nIntended release: DepMap Public 26Q3")

print("Candidate genes:", len(reference))

print("Required DepMap files:", len(REQUIRED_FILES))

print(
    "Files currently available:",
    int(input_audit["available"].sum())
)

print(
    "Files passing preliminary checks:",
    int(input_audit["header_valid"].sum())
)

print("\nExpected input directory:")
print(DEPMAP_RAW)

print("\nResearch reference: SAVED")

print("Input audit: SAVED")

print("Analysis strategy: SAVED")

print("Provenance: SAVED")

print("\nNo dependency statistics calculated.")
print("No candidate genes excluded.")
print("No previous research files modified.")

if not input_audit["header_valid"].all():

    print("\nNEXT ACTION:")
    print(
        "Obtain official 26Q3 Model.csv and "
        "CRISPRGeneEffect.csv files."
    )

else:

    print("\nBoth required inputs are available.")
    print("Ready for full source verification.")

print("\nPATCH 4.1 COMPLETE")
print("=" * 65)

In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 4 — PATCH 4.1A
#
# DEPMAP 26Q3 FILE IMPORT AND PRELIMINARY VERIFICATION
#
# Required input files:
#   Model.csv
#   CRISPRGeneEffect.csv
#
# Both files must originate from DepMap Public 26Q3.
#
# No dependency analysis performed.
# No existing research results overwritten.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import hashlib
import shutil
import json


ROOT = Path("/content/SurfaceTarget-CRC")

UPLOAD_DIR = Path("/content")

DEPMAP_DIR = (
    ROOT / "data/raw/depmap/26Q3"
)

META = ROOT / "data/metadata"

DEPMAP_DIR.mkdir(
    parents=True,
    exist_ok=True
)

META.mkdir(
    parents=True,
    exist_ok=True
)


REQUIRED_FILES = [
    "Model.csv",
    "CRISPRGeneEffect.csv"
]


print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 4 — PATCH 4.1A")
print("DEPMAP 26Q3 FILE IMPORT")
print("=" * 65)


# ------------------------------------------------------------
# 1. Locate uploaded files
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CHECKING UPLOADED FILES")
print("-" * 65)


for filename in REQUIRED_FILES:

    source = UPLOAD_DIR / filename
    destination = DEPMAP_DIR / filename

    if destination.exists():

        print(
            f"\n{filename}: Already in research directory."
        )

        if source.exists():
            print(
                "Additional upload detected; existing "
                "research file will not be overwritten."
            )

    elif source.exists():

        print(f"\nMoving {filename}...")

        shutil.move(
            str(source),
            str(destination)
        )

        print("Move successful.")

    else:

        raise FileNotFoundError(
            f"{filename} not found. "
            "Upload the official 26Q3 file "
            "using Colab's Files panel."
        )


# ------------------------------------------------------------
# 2. SHA256 checksum function
# ------------------------------------------------------------

def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):

            digest.update(chunk)

    return digest.hexdigest()


# ------------------------------------------------------------
# 3. Inspect file headers
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING DEPMAP FILE STRUCTURES")
print("-" * 65)


records = []


for filename in REQUIRED_FILES:

    path = DEPMAP_DIR / filename

    print("\nFile:", filename)

    if path.stat().st_size == 0:

        raise RuntimeError(
            f"Empty file detected: {filename}"
        )

    # Read only header and first three rows.
    # The full CRISPR matrix is not loaded.

    preview = pd.read_csv(
        path,
        nrows=3,
        low_memory=False
    )

    columns = (
        preview.columns.astype(str).tolist()
    )

    print(
        "File size (MiB):",
        round(
            path.stat().st_size / (1024**2),
            2
        )
    )

    print("Total columns:", len(columns))

    print("First 8 columns:")
    print(columns[:8])


    if "ModelID" not in columns:

        raise RuntimeError(
            f"ModelID column missing: {filename}"
        )


    if filename == "CRISPRGeneEffect.csv":

        if len(columns) < 1000:

            raise RuntimeError(
                "Unexpectedly few CRISPR gene columns."
            )

        gene_columns = [
            column
            for column in columns
            if column != "ModelID"
        ]

        print(
            "CRISPR gene columns:",
            len(gene_columns)
        )

        print(
            "Example gene identifiers:",
            gene_columns[:5]
        )

    else:

        print(
            "Model metadata columns:",
            len(columns)
        )


    checksum = sha256_file(path)

    print("SHA256:", checksum)

    records.append({

        "release": "DepMap Public 26Q3",

        "filename": filename,

        "local_path": str(path),

        "file_size_bytes":
            path.stat().st_size,

        "total_columns":
            len(columns),

        "header_valid": True,

        "sha256": checksum,

        "source": (
            "User-provided download from "
            "official DepMap portal"
        )

    })


# ------------------------------------------------------------
# 4. Save acquisition records
# ------------------------------------------------------------

audit = pd.DataFrame(records)

audit.to_csv(
    META / "depmap_26q3_import_verification.csv",
    index=False
)


provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "4.1A",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "declared_release": "DepMap Public 26Q3",

    "download_source": (
        "Official DepMap portal; "
        "manual browser download"
    ),

    "files": records,

    "validation": (
        "Local SHA256 checksums and "
        "preliminary CSV schema checks"
    ),

    "official_checksum_comparison": False,

    "important_note": (
        "Local hashes establish file identity "
        "for reproducibility. Release authenticity "
        "still requires comparison with official "
        "release metadata when available."
    ),

    "dependency_analysis_performed": False,

    "candidate_exclusions": 0

}


with open(
    META / "depmap_26q3_import_provenance.json",
    "w"
) as handle:

    json.dump(
        provenance,
        handle,
        indent=4
    )


# ------------------------------------------------------------
# 5. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 4.1A SUMMARY")
print("=" * 65)

print("\nRequired files:", len(REQUIRED_FILES))

print(
    "Files passing preliminary checks:",
    len(audit)
)

print("\nDepMap directory:")
print(DEPMAP_DIR)

print("\nAcquisition records: SAVED")

print("Local SHA256 checksums: SAVED")

print("Preliminary CSV validation: PASSED")

print("\nNo dependency calculations performed.")

print("No candidate genes excluded.")

print("No previous research results overwritten.")

print("\nPATCH 4.1A COMPLETE")
print("=" * 65)

In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 4 — PATCH 4.1A REPAIR
#
# DEPMAP 26Q3 INPUT REPAIR AND VERIFICATION
#
# Fixes:
# 1. Accepts unnamed CRISPR model-ID column
# 2. Detects suspiciously incomplete downloads
# 3. Preserves the previous incomplete file
# 4. Validates CRISPR model identifiers
# 5. Saves reproducibility checksums
#
# No CRISPR dependency analysis performed.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import hashlib
import shutil
import json


ROOT = Path("/content/SurfaceTarget-CRC")

UPLOAD = Path("/content")

DEPMAP = ROOT / "data/raw/depmap/26Q3"

META = ROOT / "data/metadata"

DEPMAP.mkdir(parents=True, exist_ok=True)
META.mkdir(parents=True, exist_ok=True)


print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 4.1A — DEPMAP IMPORT REPAIR")
print("=" * 65)


# ------------------------------------------------------------
# 1. Validate the complete uploaded CRISPR file
# ------------------------------------------------------------

print("\nCHECKING CRISPR FILE SIZE")

uploaded = UPLOAD / "CRISPRGeneEffect.csv"

stored = DEPMAP / "CRISPRGeneEffect.csv"

# The official file was displayed as 497.1 MB.
# This conservative minimum catches the
# previously observed incomplete 54 MiB upload.

MIN_EXPECTED_BYTES = 350_000_000


if uploaded.exists():

    upload_size = uploaded.stat().st_size

    print(
        "New upload size:",
        round(upload_size / 1024**2, 2),
        "MiB"
    )

    if upload_size < MIN_EXPECTED_BYTES:

        raise RuntimeError(
            "The new CRISPR upload appears incomplete. "
            "Wait for the upload to finish and verify "
            "the original file is approximately 497 MB."
        )

    if stored.exists():

        stored_size = stored.stat().st_size

        if stored_size < MIN_EXPECTED_BYTES:

            timestamp = datetime.now(
                timezone.utc
            ).strftime("%Y%m%d_%H%M%S")

            backup = DEPMAP / (
                f"CRISPRGeneEffect_incomplete_{timestamp}.csv"
            )

            stored.rename(backup)

            print(
                "Previous incomplete file preserved:",
                backup.name
            )

        else:

            raise RuntimeError(
                "An apparently complete CRISPR file "
                "already exists in the research directory. "
                "Manual comparison is required before "
                "replacing it."
            )

    shutil.move(
        str(uploaded),
        str(stored)
    )

    print("Complete CRISPR upload moved successfully.")

else:

    print("No new CRISPR upload found.")

    if not stored.exists():

        raise FileNotFoundError(
            "CRISPRGeneEffect.csv is missing."
        )


# ------------------------------------------------------------
# 2. Verify final input files
# ------------------------------------------------------------

model_path = DEPMAP / "Model.csv"

crispr_path = DEPMAP / "CRISPRGeneEffect.csv"

for path in [model_path, crispr_path]:

    if not path.exists():
        raise FileNotFoundError(path)


crispr_bytes = crispr_path.stat().st_size

print("\nFINAL CRISPR FILE SIZE")

print(
    "Size:",
    round(crispr_bytes / 1024**2, 2),
    "MiB"
)

if crispr_bytes < MIN_EXPECTED_BYTES:

    raise RuntimeError(
        "Stored CRISPR file is substantially smaller "
        "than the expected official download."
    )


# ------------------------------------------------------------
# 3. Validate Model.csv
# ------------------------------------------------------------

print("\nVALIDATING MODEL METADATA")

models = pd.read_csv(
    model_path,
    dtype={"ModelID": str},
    low_memory=False
)

if "ModelID" not in models.columns:

    raise RuntimeError(
        "Model.csv does not contain ModelID."
    )

if models["ModelID"].duplicated().any():

    raise RuntimeError(
        "Duplicate ModelIDs in Model.csv."
    )

model_ids = set(
    models["ModelID"].dropna()
)

print("Models in metadata:", len(models))
print("Metadata columns:", len(models.columns))

print("Model metadata validation: PASSED")


# ------------------------------------------------------------
# 4. Validate CRISPR header
# ------------------------------------------------------------

print("\nVALIDATING CRISPR GENE-EFFECT DATA")

header = pd.read_csv(
    crispr_path,
    nrows=0
).columns.tolist()

print("Total CRISPR columns:", len(header))

print("Original first column:", header[0])

print("First gene columns:", header[1:6])


if len(header) < 15000:

    raise RuntimeError(
        "Unexpectedly few CRISPR gene columns."
    )


if header[0] not in [
    "ModelID",
    "Unnamed: 0"
]:

    raise RuntimeError(
        "Unrecognized CRISPR model identifier column."
    )


# The first column is treated as ModelID
# regardless of whether its header is blank.
#
# The original source file is not rewritten.

print("\nModel identifier handling: CORRECTED")


# ------------------------------------------------------------
# 5. Validate all CRISPR model identifiers
# ------------------------------------------------------------

print("\nREADING CRISPR MODEL IDENTIFIERS")

# Read only the first column into memory.
# Pandas parses the complete CSV while retaining
# only the model identifier column.

crispr_index = pd.read_csv(
    crispr_path,
    usecols=[0],
    dtype=str
)

crispr_index.columns = ["ModelID"]

crispr_index["ModelID"] = (
    crispr_index["ModelID"]
    .astype(str)
    .str.strip()
)

crispr_ids = crispr_index["ModelID"]

if crispr_ids.duplicated().any():

    raise RuntimeError(
        "Duplicate CRISPR model identifiers."
    )

valid_format = (
    crispr_ids.str.fullmatch(
        r"ACH-\d+"
    )
)

if not valid_format.all():

    raise RuntimeError(
        "Unexpected CRISPR model identifier format."
    )


matched = crispr_ids.isin(
    model_ids
)

print("CRISPR model rows:", len(crispr_ids))

print(
    "Models matching Model.csv:",
    int(matched.sum())
)

print(
    "Models without metadata match:",
    int((~matched).sum())
)


if not matched.all():

    raise RuntimeError(
        "Some CRISPR model IDs are missing "
        "from the supplied Model.csv. "
        "Confirm both files are from 26Q3."
    )


print("\nCRISPR model identifier validation: PASSED")


# ------------------------------------------------------------
# 6. Save checksums
# ------------------------------------------------------------

print("\nCALCULATING SOURCE CHECKSUMS")


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


records = []

for path in [model_path, crispr_path]:

    checksum = sha256_file(path)

    records.append({

        "release": "DepMap Public 26Q3",

        "filename": path.name,

        "local_path": str(path),

        "file_size_bytes":
            path.stat().st_size,

        "sha256": checksum,

        "header_valid": True,

        "source": "Official DepMap browser download"
    })

    print(path.name)
    print("SHA256:", checksum)


audit = pd.DataFrame(records)

audit.to_csv(
    META / "depmap_26q3_import_verification.csv",
    index=False
)


provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "4.1A-repair",

    "release": "DepMap Public 26Q3",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "crispr_model_column_original": header[0],

    "crispr_model_column_interpretation": "ModelID",

    "crispr_model_rows": int(len(crispr_ids)),

    "crispr_gene_columns": int(len(header) - 1),

    "matched_model_ids": int(matched.sum()),

    "source_files": records,

    "official_hash_comparison_performed": False,

    "dependency_analysis_performed": False,

    "original_crispr_file_modified": False

}

with open(
    META / "depmap_26q3_import_provenance.json",
    "w"
) as handle:

    json.dump(
        provenance,
        handle,
        indent=4
    )


# ------------------------------------------------------------
# 7. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 4.1A — REPAIR SUMMARY")
print("=" * 65)

print("\nDepMap release: 26Q3")

print("Files verified: 2")

print(
    "CRISPR model rows:",
    len(crispr_ids)
)

print(
    "CRISPR gene columns:",
    len(header) - 1
)

print(
    "Models matched to metadata:",
    int(matched.sum())
)

print("\nModel identifier fix: APPLIED")

print("File size validation: PASSED")

print("Model correspondence: PASSED")

print("SHA256 checksums: SAVED")

print("Source provenance: SAVED")

print("\nNo dependency analysis performed.")

print("\nPATCH 4.1A COMPLETE")

print("=" * 65)

In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 4 — PATCH 4.1B
#
# DEPMAP CRISPR FILE FRAGMENT RECOVERY
#
# Purpose:
# 1. Locate both preserved upload fragments
# 2. Validate their combined expected file size
# 3. Inspect their joining boundary
# 4. Construct a recovery candidate
# 5. Validate the complete CSV structure
# 6. Verify model identifiers against Model.csv
# 7. Preserve the original fragments
# 8. Save recovery provenance and checksums
#
# IMPORTANT:
# - Original fragments are preserved
# - No dependency calculations performed
# - No source measurements intentionally modified
# - Official source checksum comparison remains pending
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import hashlib
import shutil
import csv
import re
import os
import json
import numpy as np


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

DEPMAP = ROOT / "data/raw/depmap/26Q3"
META = ROOT / "data/metadata"

META.mkdir(parents=True, exist_ok=True)

FIRST_FRAGMENT = (
    DEPMAP /
    "CRISPRGeneEffect_incomplete_20261121_081207.csv"
)

SECOND_FRAGMENT = (
    DEPMAP /
    "CRISPRGeneEffect.csv"
)

RECOVERY_CANDIDATE = (
    DEPMAP /
    "CRISPRGeneEffect_recovery_candidate.csv"
)

MODEL_PATH = DEPMAP / "Model.csv"


print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 4 — PATCH 4.1B")
print("DEPMAP CRISPR FRAGMENT RECOVERY")
print("=" * 65)


# ------------------------------------------------------------
# 2. VALIDATE INPUT FILES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING PRESERVED FILE FRAGMENTS")
print("-" * 65)


for path in [
    FIRST_FRAGMENT,
    SECOND_FRAGMENT,
    MODEL_PATH
]:

    if not path.exists():

        raise FileNotFoundError(
            f"Required file missing: {path}"
        )


first_size = FIRST_FRAGMENT.stat().st_size

second_size = SECOND_FRAGMENT.stat().st_size

combined_size = first_size + second_size


print("\nFirst fragment:")

print(
    f"Size: {first_size / 1024**2:.2f} MiB"
)


print("\nSecond fragment:")

print(
    f"Size: {second_size / 1024**2:.2f} MiB"
)


print("\nCombined:")

print(
    f"Size: {combined_size / 1024**2:.2f} MiB"
)

print(
    f"Size: {combined_size / 1_000_000:.2f} MB"
)


# The website displayed approximately 497.1 MB.
# This is a preliminary plausibility check,
# not an official file-integrity verification.

if not (490_000_000 <= combined_size <= 505_000_000):

    raise RuntimeError(
        "Combined fragment size is inconsistent "
        "with the expected DepMap download."
    )


print("\nCombined file-size plausibility: PASSED")


# ------------------------------------------------------------
# 3. INSPECT THE FRAGMENT BOUNDARY
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("INSPECTING FRAGMENT BOUNDARY")
print("-" * 65)


with open(FIRST_FRAGMENT, "rb") as f:

    f.seek(max(0, first_size - 100))

    first_tail = f.read()


with open(SECOND_FRAGMENT, "rb") as f:

    second_head = f.read(100)


print("\nEnd of first fragment:")

print(repr(first_tail.decode("utf-8", errors="replace")))


print("\nBeginning of second fragment:")

print(repr(second_head.decode("utf-8", errors="replace")))


print("\nCombined boundary preview:")

boundary = first_tail + second_head

print(repr(boundary.decode("utf-8", errors="replace")))


# ------------------------------------------------------------
# 4. VALIDATE ORIGINAL CSV HEADER
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING ORIGINAL CRISPR HEADER")
print("-" * 65)


with open(
    FIRST_FRAGMENT,
    "r",
    encoding="utf-8-sig",
    newline=""
) as f:

    reader = csv.reader(f)

    original_header = next(reader)


expected_columns = len(original_header)


print("\nOriginal header columns:", expected_columns)

print("First five headers:")

print(original_header[:5])


if expected_columns < 18000:

    raise RuntimeError(
        "The first fragment does not contain "
        "the expected DepMap gene header."
    )


if original_header[0] not in ["", "ModelID"]:

    raise RuntimeError(
        "Unexpected first column in original header."
    )


# Check the standard gene naming format:
# GeneSymbol (EntrezGeneID)

gene_header_pattern = re.compile(
    r".+\s\(\d+\)$"
)


matching_gene_headers = sum(
    bool(gene_header_pattern.fullmatch(name))
    for name in original_header[1:]
)


header_match_fraction = (
    matching_gene_headers / (expected_columns - 1)
)


print(
    "Gene columns matching expected format:",
    matching_gene_headers
)

print(
    "Matching header fraction:",
    round(header_match_fraction, 5)
)


if header_match_fraction < 0.98:

    raise RuntimeError(
        "Unexpected CRISPR gene-column naming structure."
    )


print("\nOriginal gene header: VALIDATED")


# ------------------------------------------------------------
# 5. LOAD MODEL IDENTIFIERS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING MODEL METADATA")
print("-" * 65)


models = pd.read_csv(
    MODEL_PATH,
    usecols=["ModelID"],
    dtype=str
)


if models["ModelID"].duplicated().any():

    raise RuntimeError(
        "Duplicate ModelIDs in Model.csv."
    )


model_ids = set(
    models["ModelID"]
    .dropna()
    .astype(str)
    .str.strip()
)


print("\nMetadata models:", len(model_ids))


# ------------------------------------------------------------
# 6. RECONSTRUCT CSV WITHOUT MODIFYING FRAGMENTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("RECONSTRUCTING CRISPR CSV")
print("-" * 65)


if RECOVERY_CANDIDATE.exists():

    raise RuntimeError(
        "A previous recovery candidate already exists. "
        "Inspect or remove that candidate before rerunning."
    )


# Copy the original byte sequences in order.
#
# Fragment 1 contains the original header.
# Fragment 2 appears to continue its data.
#
# The next section validates the resulting CSV
# before it can replace the working file.

with open(RECOVERY_CANDIDATE, "wb") as output:

    for fragment in [
        FIRST_FRAGMENT,
        SECOND_FRAGMENT
    ]:

        with open(fragment, "rb") as source:

            shutil.copyfileobj(
                source,
                output,
                length=1024 * 1024
            )


reconstructed_size = (
    RECOVERY_CANDIDATE.stat().st_size
)


if reconstructed_size != combined_size:

    raise RuntimeError(
        "Reconstructed file size does not "
        "match the sum of both fragments."
    )


print("\nRecovery candidate created.")

print(
    "Size:",
    round(reconstructed_size / 1024**2, 2),
    "MiB"
)


# ------------------------------------------------------------
# 7. VALIDATE EVERY RECONSTRUCTED CSV ROW
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING RECONSTRUCTED CSV STRUCTURE")
print("-" * 65)


observed_ids = set()

row_count = 0

numeric_checks = 0

model_matching_count = 0


# Representative fields across each row.
# The full file is streamed rather than
# materialized as a huge DataFrame.

sample_column_positions = [
    1,
    100,
    1000,
    5000,
    10000,
    expected_columns - 1
]


try:

    with open(
        RECOVERY_CANDIDATE,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as f:

        reader = csv.reader(
            f,
            strict=True
        )

        reconstructed_header = next(reader)

        if reconstructed_header != original_header:

            raise RuntimeError(
                "Reconstructed CSV header differs "
                "from the preserved original header."
            )

        for row in reader:

            row_count += 1

            # ------------------------------------
            # Check complete CSV row width
            # ------------------------------------

            if len(row) != expected_columns:

                raise RuntimeError(
                    "Malformed CSV row detected. "
                    f"Row: {row_count}, "
                    f"Expected fields: {expected_columns}, "
                    f"Observed fields: {len(row)}"
                )

            # ------------------------------------
            # Validate ModelID
            # ------------------------------------

            model_id = str(row[0]).strip()

            if not re.fullmatch(
                r"ACH-\d+",
                model_id
            ):

                raise RuntimeError(
                    "Unexpected CRISPR ModelID. "
                    f"Row: {row_count}, "
                    f"Observed: {model_id}"
                )

            if model_id in observed_ids:

                raise RuntimeError(
                    f"Duplicate CRISPR ModelID: {model_id}"
                )

            observed_ids.add(model_id)

            if model_id not in model_ids:

                raise RuntimeError(
                    "CRISPR ModelID not present in Model.csv: "
                    f"{model_id}"
                )

            model_matching_count += 1

            # ------------------------------------
            # Check representative gene values
            # ------------------------------------

            for position in sample_column_positions:

                value = row[position].strip()

                if value == "" or value.lower() in {
                    "na", "nan"
                }:

                    continue

                try:

                    numeric_value = float(value)

                except ValueError:

                    raise RuntimeError(
                        "Invalid numerical gene-effect value. "
                        f"Row: {row_count}, "
                        f"Column: {position}, "
                        f"Value: {value}"
                    )

                if not np.isfinite(numeric_value):

                    raise RuntimeError(
                        "Non-finite gene-effect value detected."
                    )

                numeric_checks += 1

            if row_count % 250 == 0:

                print(
                    f"Validated {row_count} CRISPR model rows"
                )

except Exception:

    print(
        "\nRECOVERY VALIDATION FAILED."
    )

    print(
        "Both original fragments remain preserved."
    )

    print(
        "No source measurements have been replaced."
    )

    raise


# ------------------------------------------------------------
# 8. COMPLETE STRUCTURE VALIDATION
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("RECONSTRUCTION VALIDATION SUMMARY")
print("-" * 65)


print("\nGene columns:", expected_columns - 1)

print("CRISPR model rows:", row_count)

print("Unique ModelIDs:", len(observed_ids))

print("Models matching metadata:", model_matching_count)

print("Numerical spot-checks:", numeric_checks)


if row_count < 500:

    raise RuntimeError(
        "Unexpectedly few CRISPR model rows."
    )


if row_count > len(model_ids):

    raise RuntimeError(
        "CRISPR model count exceeds metadata models."
    )


if len(observed_ids) != row_count:

    raise RuntimeError(
        "CRISPR model identifier uniqueness failed."
    )


if model_matching_count != row_count:

    raise RuntimeError(
        "Not all CRISPR models matched metadata."
    )


print("\nCSV structure validation: PASSED")

print("Model correspondence: PASSED")

print("Numerical spot-checks: PASSED")


# ------------------------------------------------------------
# 9. CALCULATE SOURCE HASHES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CALCULATING RECONSTRUCTION CHECKSUMS")
print("-" * 65)


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            digest.update(chunk)

    return digest.hexdigest()


first_hash = sha256_file(FIRST_FRAGMENT)

second_hash = sha256_file(SECOND_FRAGMENT)

reconstructed_hash = sha256_file(
    RECOVERY_CANDIDATE
)


print("\nFirst fragment SHA256:", first_hash)

print("Second fragment SHA256:", second_hash)

print("Reconstructed SHA256:", reconstructed_hash)


# ------------------------------------------------------------
# 10. PRESERVE FRAGMENTS AND INSTALL VERIFIED CSV
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("INSTALLING STRUCTURALLY VALIDATED CSV")
print("-" * 65)


timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S")


tail_backup = (
    DEPMAP /
    f"CRISPRGeneEffect_tail_fragment_{timestamp}.csv"
)


# Preserve the second fragment separately.
# The first fragment is already preserved.

os.replace(
    SECOND_FRAGMENT,
    tail_backup
)


try:

    os.replace(
        RECOVERY_CANDIDATE,
        SECOND_FRAGMENT
    )

except Exception:

    # Restore original working file if needed.

    if (
        tail_backup.exists()
        and not SECOND_FRAGMENT.exists()
    ):

        os.replace(
            tail_backup,
            SECOND_FRAGMENT
        )

    raise


print("\nRecovered CSV installed successfully.")

print("Original first fragment: PRESERVED")

print("Original second fragment: PRESERVED")

print("Recovered full CSV: INSTALLED")


# ------------------------------------------------------------
# 11. SAVE IMPORT AUDIT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING RECOVERY PROVENANCE")
print("-" * 65)


records = [

    {
        "release": "DepMap Public 26Q3",
        "filename": "Model.csv",
        "local_path": str(MODEL_PATH),
        "file_size_bytes": MODEL_PATH.stat().st_size,
        "sha256": sha256_file(MODEL_PATH),
        "header_valid": True,
        "source": "Official DepMap browser download"
    },

    {
        "release": "DepMap Public 26Q3",
        "filename": "CRISPRGeneEffect.csv",
        "local_path": str(SECOND_FRAGMENT),
        "file_size_bytes":
            SECOND_FRAGMENT.stat().st_size,
        "sha256": reconstructed_hash,
        "header_valid": True,
        "source": (
            "Reconstructed from two preserved "
            "Colab upload fragments"
        )
    }
]


pd.DataFrame(records).to_csv(
    META / "depmap_26q3_import_verification.csv",
    index=False
)


provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "4.1B",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "declared_release": "DepMap Public 26Q3",

    "reconstruction_method": (
        "Bytewise concatenation of preserved "
        "first and second upload fragments"
    ),

    "first_fragment_path": str(FIRST_FRAGMENT),

    "second_fragment_backup_path": str(tail_backup),

    "first_fragment_bytes": first_size,

    "second_fragment_bytes": second_size,

    "reconstructed_bytes": reconstructed_size,

    "first_fragment_sha256": first_hash,

    "second_fragment_sha256": second_hash,

    "reconstructed_sha256": reconstructed_hash,

    "crispr_model_rows": row_count,

    "crispr_gene_columns": expected_columns - 1,

    "model_metadata_rows": len(model_ids),

    "models_matching_metadata": model_matching_count,

    "csv_structure_validation": True,

    "numerical_spot_checks": numeric_checks,

    "official_release_checksum_verified": False,

    "important_limitation": (
        "Structural and size validation support "
        "the reconstruction, but do not independently "
        "prove byte-for-byte identity with the "
        "official DepMap download."
    ),

    "dependency_analysis_performed": False,

    "candidate_exclusions": 0
}


with open(
    META / "depmap_26q3_import_provenance.json",
    "w"
) as f:

    json.dump(
        provenance,
        f,
        indent=4
    )


print("\nImport audit: SAVED")

print("Reconstruction provenance: SAVED")


# ------------------------------------------------------------
# 12. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 4.1B SUMMARY")
print("=" * 65)


print("\nDepMap release: 26Q3")

print(
    "Recovered CSV size:",
    round(reconstructed_size / 1024**2, 2),
    "MiB"
)

print("CRISPR model rows:", row_count)

print("CRISPR gene columns:", expected_columns - 1)

print(
    "Model metadata correspondence:",
    "PASSED"
)

print("\nStructural CSV validation: PASSED")

print("Numerical spot-checks: PASSED")

print("Both upload fragments: PRESERVED")

print("Source checksums: SAVED")

print("\nOfficial source checksum verified: NO")

print("CRISPR dependency analysis performed: NO")

print("\nPATCH 4.1B COMPLETE")

print("=" * 65)

In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 4 — PATCH 4.2
#
# COLORECTAL MODEL IDENTIFICATION AND
# CANDIDATE CRISPR GENE-EFFECT AUDIT
#
# Objectives:
# 1. Validate reconstructed DepMap 26Q3 inputs
# 2. Identify screened colorectal cancer models
# 3. Review colorectal disease and histology annotations
# 4. Audit candidate gene identifier correspondence
# 5. Extract only relevant CRISPR gene-effect columns
# 6. Validate model/sample and numerical integrity
# 7. Calculate descriptive colorectal dependency metrics
# 8. Compare with other screened cancer models
# 9. Review leading research candidates
# 10. Save results and reproducibility metadata
#
# IMPORTANT:
# - No MSS status inferred from tissue origin
# - No automatic gene exclusions
# - No final therapeutic target ranking
# - No causal lineage-specificity claims
# - No modification of original DepMap data
# - Reconstructed input not officially hash-verified
# ============================================================


# ------------------------------------------------------------
# 1. IMPORTS AND CONFIGURATION
# ------------------------------------------------------------

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np

import hashlib
import json
import re


ROOT = Path("/content/SurfaceTarget-CRC")

DEPMAP = ROOT / "data/raw/depmap/26Q3"

META = ROOT / "data/metadata"

RESULTS = ROOT / "results/tables"

for directory in [META, RESULTS]:
    directory.mkdir(parents=True, exist_ok=True)


MODEL_PATH = DEPMAP / "Model.csv"

CRISPR_PATH = DEPMAP / "CRISPRGeneEffect.csv"

IMPORT_PATH = (
    META / "depmap_26q3_import_verification.csv"
)

REFERENCE_PATH = (
    META / "depmap_phase4_candidate_reference.csv"
)


print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 4 — PATCH 4.2")
print("COLORECTAL CRISPR DEPENDENCY AUDIT")
print("=" * 65)


# ------------------------------------------------------------
# 2. VALIDATE REQUIRED RESEARCH INPUTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING DEPMAP INPUT FILES")
print("-" * 65)


required_paths = [
    MODEL_PATH,
    CRISPR_PATH,
    IMPORT_PATH,
    REFERENCE_PATH
]


for path in required_paths:

    if not path.exists():

        raise FileNotFoundError(
            f"Required research file missing: {path}"
        )


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


import_audit = pd.read_csv(
    IMPORT_PATH,
    dtype=str,
    keep_default_na=False
)


if import_audit["filename"].duplicated().any():

    raise RuntimeError(
        "Duplicate file entries in import audit."
    )


import_audit = import_audit.set_index(
    "filename"
)


for path in [MODEL_PATH, CRISPR_PATH]:

    if path.name not in import_audit.index:

        raise RuntimeError(
            f"File missing from import audit: {path.name}"
        )

    expected_hash = import_audit.loc[
        path.name,
        "sha256"
    ]

    actual_hash = sha256_file(path)

    if actual_hash != expected_hash:

        raise RuntimeError(
            f"File changed since verification: {path.name}"
        )

    expected_size = int(
        import_audit.loc[
            path.name,
            "file_size_bytes"
        ]
    )

    if path.stat().st_size != expected_size:

        raise RuntimeError(
            f"File size changed: {path.name}"
        )


print("\nDepMap input integrity: PASSED")

print(
    "Source files match the locally recorded "
    "Patch 4.1B checksums."
)

print(
    "Official release checksum comparison: "
    "NOT AVAILABLE"
)


# ------------------------------------------------------------
# 3. LOAD MODEL METADATA
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING DEPMAP MODEL METADATA")
print("-" * 65)


models = pd.read_csv(
    MODEL_PATH,
    dtype=str,
    keep_default_na=False,
    low_memory=False
)


required_model_columns = {
    "ModelID",
    "CellLineName",
    "OncotreeLineage",
    "OncotreePrimaryDisease",
    "OncotreeSubtype"
}


if not required_model_columns.issubset(
    models.columns
):

    raise RuntimeError(
        "Required DepMap model annotations missing."
    )


if models["ModelID"].duplicated().any():

    raise RuntimeError(
        "Duplicate ModelIDs in metadata."
    )


models["ModelID"] = (
    models["ModelID"].str.strip()
)


if models["ModelID"].eq("").any():

    raise RuntimeError(
        "Empty model identifiers detected."
    )


print("\nModels in metadata:", len(models))

print("Metadata columns:", len(models.columns))

print("Unique ModelIDs:", models["ModelID"].nunique())


# ------------------------------------------------------------
# 4. LOAD SCREENED MODEL IDENTIFIERS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("IDENTIFYING MODELS WITH CRISPR MEASUREMENTS")
print("-" * 65)


# The reconstructed CRISPR CSV retains
# its original blank first-column header.
#
# Read only the first column initially.
# No genome-wide matrix is loaded here.


crispr_ids = pd.read_csv(
    CRISPR_PATH,
    usecols=[0],
    dtype=str,
    low_memory=False
)


crispr_ids.columns = ["ModelID"]


crispr_ids["ModelID"] = (
    crispr_ids["ModelID"]
    .astype(str)
    .str.strip()
)


if crispr_ids["ModelID"].duplicated().any():

    raise RuntimeError(
        "Duplicate CRISPR model identifiers."
    )


if crispr_ids["ModelID"].eq("").any():

    raise RuntimeError(
        "Missing CRISPR model identifiers."
    )


screened_ids = set(
    crispr_ids["ModelID"]
)


metadata_ids = set(
    models["ModelID"]
)


if not screened_ids.issubset(metadata_ids):

    raise RuntimeError(
        "Some CRISPR models lack model metadata."
    )


screened_models = models[
    models["ModelID"].isin(
        screened_ids
    )
].copy()


print("\nModels with CRISPR measurements:",
      len(screened_ids))

print(
    "Screened models matching metadata:",
    len(screened_models)
)


if len(screened_ids) != 1361:

    raise RuntimeError(
        "Unexpected screened model count. "
        "Expected 1,361 from Patch 4.1B."
    )


print("\nModel correspondence: PASSED")


# ------------------------------------------------------------
# 5. EXPLORE COLORECTAL MODEL ANNOTATIONS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("COLORECTAL MODEL ANNOTATION DISCOVERY")
print("-" * 65)


# Do not assume all Bowel-lineage models
# are colorectal adenocarcinoma models.
#
# First inspect disease classifications
# using the actual release metadata.


bowel_mask = (
    screened_models["OncotreeLineage"]
    .str.contains(
        r"bowel|colorectal",
        case=False,
        regex=True,
        na=False
    )
)


bowel_models = screened_models[
    bowel_mask
].copy()


print("\nScreened models in bowel-related lineages:",
      len(bowel_models))


print("\nPrimary disease labels in bowel-related models:")


print(
    bowel_models[
        "OncotreePrimaryDisease"
    ]
    .value_counts()
    .to_string()
)


# ------------------------------------------------------------
# 6. IDENTIFY COLORECTAL CANCER MODELS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SELECTING COLORECTAL CANCER MODELS")
print("-" * 65)


# The primary disease annotation must
# explicitly identify colon, rectal,
# or colorectal malignancy.
#
# We do not include every bowel-lineage
# model automatically.


colorectal_mask = (
    screened_models["OncotreePrimaryDisease"]
    .str.contains(
        r"colorectal|colon|rectal",
        case=False,
        regex=True,
        na=False
    )
)


colorectal_models = screened_models[
    colorectal_mask
].copy()


print("\nScreened colorectal models:",
      len(colorectal_models))


print("\nColorectal primary disease distribution:")

print(
    colorectal_models[
        "OncotreePrimaryDisease"
    ]
    .value_counts()
    .to_string()
)


print("\nColorectal subtype distribution:")

print(
    colorectal_models[
        "OncotreeSubtype"
    ]
    .replace("", "NOT AVAILABLE")
    .value_counts()
    .to_string()
)


if colorectal_models.empty:

    raise RuntimeError(
        "No screened colorectal models identified. "
        "Inspect the printed OncoTree annotations "
        "before modifying selection criteria."
    )


# ------------------------------------------------------------
# 6A. Review unusual colorectal histologies
# ------------------------------------------------------------

# Most of our patient-tissue discovery analysis
# concerns colorectal adenocarcinoma.
#
# Flag clearly distinct histologies for review
# rather than automatically treating them
# as representative adenocarcinoma models.


ATYPICAL_HISTOLOGY_PATTERN = (
    r"neuroendocrine|small.cell|"
    r"sarcoma|lymphoma|gastrointestinal.stromal"
)


colorectal_models["histology_review_flag"] = (
    colorectal_models["OncotreeSubtype"]
    .str.contains(
        ATYPICAL_HISTOLOGY_PATTERN,
        case=False,
        regex=True,
        na=False
    )
)


histology_review = colorectal_models[
    colorectal_models["histology_review_flag"]
].copy()


print("\nModels requiring distinct-histology review:",
      len(histology_review))


if not histology_review.empty:

    print(
        histology_review[
            [
                "ModelID",
                "CellLineName",
                "OncotreePrimaryDisease",
                "OncotreeSubtype"
            ]
        ].to_string(index=False)
    )


# Use the main colorectal group while
# withholding clearly distinct histologies
# from the primary descriptive summaries.
#
# All reviewed models are preserved in
# the full colorectal model inventory.


primary_crc_models = colorectal_models[
    ~colorectal_models["histology_review_flag"]
].copy()


print("\nPrimary colorectal analysis models:",
      len(primary_crc_models))


if len(primary_crc_models) < 3:

    raise RuntimeError(
        "Insufficient colorectal models "
        "for descriptive analysis."
    )


# ------------------------------------------------------------
# 6B. Check possible patient-level duplication
# ------------------------------------------------------------

if "PatientID" in primary_crc_models.columns:

    patient_values = (
        primary_crc_models["PatientID"]
        .replace("", np.nan)
        .dropna()
    )

    repeated_patients = int(
        patient_values.duplicated().sum()
    )

else:

    repeated_patients = np.nan


print("\nRepeated nonmissing PatientIDs:",
      repeated_patients)


print(
    "Multiple models from one patient, when present, "
    "must not be described as independent patients."
)


# ------------------------------------------------------------
# 6C. Inspect possible MSI annotations
# ------------------------------------------------------------

msi_related_columns = [
    column
    for column in models.columns
    if re.search(
        r"msi|microsatellite|mantis",
        column,
        flags=re.IGNORECASE
    )
]


print("\nPotential MSI annotation columns:")

print(
    msi_related_columns
    if msi_related_columns
    else "None identified in Model.csv"
)


print(
    "\nNo models have been assigned MSS status."
)


# ------------------------------------------------------------
# 7. LOAD THE PHASE 4 CANDIDATE REFERENCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING CANDIDATE GENE REFERENCE")
print("-" * 65)


reference = pd.read_csv(
    REFERENCE_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str,
        "gene_name": str
    },
    low_memory=False
)


if len(reference) != 948:

    raise RuntimeError(
        "Expected 948 candidate genes."
    )


if reference["gene_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate transcriptomic gene IDs."
    )


if reference["gene_name"].duplicated().any():

    raise RuntimeError(
        "Duplicate candidate gene symbols."
    )


reference["gene_name"] = (
    reference["gene_name"]
    .astype(str)
    .str.strip()
)


reference["protein_enriched"] = (
    reference["protein_enriched"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)


if int(reference["protein_enriched"].sum()) != 20:

    raise RuntimeError(
        "Expected 20 cross-omics-supported genes."
    )


print("\nCandidate genes:", len(reference))

print(
    "Cross-omics-supported genes:",
    int(reference["protein_enriched"].sum())
)


# ------------------------------------------------------------
# 8. INSPECT CRISPR GENE IDENTIFIERS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CRISPR GENE IDENTIFIER AUDIT")
print("-" * 65)


# DepMap headers have the general format:
#
# A1BG (1)
# CDH3 (1001)
#
# Gene symbol and Entrez ID are preserved.
#
# Matching candidate gene symbols is an
# initial correspondence rule, not a
# fully verified Ensembl-to-Entrez crosswalk.


crispr_header = pd.read_csv(
    CRISPR_PATH,
    nrows=0
).columns.tolist()


print("\nTotal CRISPR columns:",
      len(crispr_header))


if len(crispr_header) != 18435:

    raise RuntimeError(
        "Unexpected CRISPR column count."
    )


model_column = crispr_header[0]


if model_column not in [
    "Unnamed: 0",
    "ModelID"
]:

    raise RuntimeError(
        "Unexpected CRISPR model column."
    )


GENE_PATTERN = re.compile(
    r"^(.*?) \((\d+)\)$"
)


gene_header_records = []

for column in crispr_header[1:]:

    match = GENE_PATTERN.fullmatch(
        column
    )

    if match:

        gene_symbol = match.group(1)

        entrez_id = match.group(2)

    else:

        gene_symbol = None

        entrez_id = None

    gene_header_records.append({

        "depmap_column": column,

        "depmap_gene_symbol": gene_symbol,

        "depmap_entrez_id": entrez_id,

        "header_parsed": match is not None
    })


gene_headers = pd.DataFrame(
    gene_header_records
)


parsed_count = int(
    gene_headers["header_parsed"].sum()
)


print("\nCRISPR gene columns:",
      len(gene_headers))

print(
    "Headers matching symbol/Entrez format:",
    parsed_count
)


if parsed_count < 18000:

    raise RuntimeError(
        "Unexpected DepMap gene header format."
    )


# ------------------------------------------------------------
# 9. MAP CANDIDATES TO CRISPR GENE COLUMNS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("MATCHING RESEARCH CANDIDATES TO DEPMAP")
print("-" * 65)


gene_matches = {}

for symbol, group in gene_headers.groupby(
    "depmap_gene_symbol",
    dropna=True
):

    gene_matches[symbol] = group.to_dict(
        orient="records"
    )


mapping_records = []


for _, row in reference.iterrows():

    symbol = row["gene_name"]

    matches = gene_matches.get(
        symbol,
        []
    )

    if len(matches) == 1:

        mapping_status = (
            "Unique exact gene-symbol match"
        )

        selected_column = matches[0][
            "depmap_column"
        ]

        entrez_id = matches[0][
            "depmap_entrez_id"
        ]

    elif len(matches) > 1:

        mapping_status = (
            "Ambiguous gene-symbol match"
        )

        selected_column = None
        entrez_id = None

    else:

        mapping_status = (
            "No exact gene-symbol match"
        )

        selected_column = None
        entrez_id = None


    mapping_records.append({

        "gene_id": row["gene_id"],

        "ensembl_stable_id":
            row["ensembl_stable_id"],

        "gene_name": symbol,

        "protein_enriched":
            bool(row["protein_enriched"]),

        "depmap_matching_columns":
            len(matches),

        "depmap_mapping_status":
            mapping_status,

        "depmap_column":
            selected_column,

        "depmap_entrez_id":
            entrez_id
    })


gene_mapping = pd.DataFrame(
    mapping_records
)


print("\nMapping status distribution:")

print(
    gene_mapping[
        "depmap_mapping_status"
    ]
    .value_counts()
    .to_string()
)


uniquely_mapped = gene_mapping[
    gene_mapping["depmap_mapping_status"]
    .eq("Unique exact gene-symbol match")
].copy()


print("\nUniquely matched candidates:",
      len(uniquely_mapped))


unresolved = gene_mapping[
    ~gene_mapping["depmap_mapping_status"]
    .eq("Unique exact gene-symbol match")
].copy()


print("\nUnresolved candidate mappings:",
      len(unresolved))


if not unresolved.empty:

    print("\nFirst 25 unresolved genes:")

    print(
        unresolved[
            [
                "gene_name",
                "depmap_mapping_status"
            ]
        ]
        .head(25)
        .to_string(index=False)
    )


if len(uniquely_mapped) < 500:

    raise RuntimeError(
        "Unexpectedly low candidate gene coverage. "
        "Review identifier correspondence."
    )


# ------------------------------------------------------------
# 10. EXTRACT CANDIDATE CRISPR GENE EFFECTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("EXTRACTING CANDIDATE CRISPR MEASUREMENTS")
print("-" * 65)


selected_gene_columns = (
    uniquely_mapped["depmap_column"]
    .tolist()
)


columns_to_read = (
    [model_column]
    + selected_gene_columns
)


# Read only the candidate gene columns.
#
# This avoids loading all 18,434 gene
# columns into memory.


effects = pd.read_csv(
    CRISPR_PATH,
    usecols=columns_to_read,
    dtype={model_column: str},
    low_memory=False
)


effects = effects.rename(
    columns={
        model_column: "ModelID"
    }
)


effects["ModelID"] = (
    effects["ModelID"]
    .astype(str)
    .str.strip()
)


if effects["ModelID"].duplicated().any():

    raise RuntimeError(
        "Duplicate CRISPR model rows."
    )


effects = effects.set_index(
    "ModelID"
)


if set(effects.index) != screened_ids:

    raise RuntimeError(
        "CRISPR model inventory changed "
        "during candidate extraction."
    )


if set(effects.columns) != set(
    selected_gene_columns
):

    raise RuntimeError(
        "Extracted CRISPR gene columns differ "
        "from the validated mapping."
    )


# Convert only extracted gene-effect columns
# to numeric measurements.


effects = effects.apply(
    pd.to_numeric,
    errors="raise"
)


if np.isinf(
    effects.to_numpy(dtype=float)
).any():

    raise RuntimeError(
        "Infinite CRISPR gene-effect values detected."
    )


print("\nCandidate CRISPR matrix extracted.")

print("Matrix dimensions:", effects.shape)

print(
    "Models represented:",
    effects.index.nunique()
)

print(
    "Gene columns represented:",
    len(effects.columns)
)


print("\nCandidate matrix validation: PASSED")


# ------------------------------------------------------------
# 11. CREATE COLORECTAL AND COMPARATOR MATRICES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CONSTRUCTING FUNCTIONAL ANALYSIS GROUPS")
print("-" * 65)


crc_ids = (
    primary_crc_models["ModelID"]
    .tolist()
)


crc_effects = effects.loc[
    crc_ids
].copy()


other_effects = effects.drop(
    index=crc_ids
).copy()


print("\nColorectal models:",
      len(crc_effects))

print("Other screened models:",
      len(other_effects))


if len(crc_effects) < 3:

    raise RuntimeError(
        "Insufficient colorectal CRISPR observations."
    )


if len(other_effects) < 50:

    raise RuntimeError(
        "Insufficient comparator models."
    )


print("\nAnalysis groups: PASSED")


# ------------------------------------------------------------
# 12. CALCULATE DESCRIPTIVE GENE-EFFECT STATISTICS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CALCULATING COLORECTAL CRISPR GENE EFFECTS")
print("-" * 65)


def summarize_gene_effect(
    matrix,
    gene_column
):

    values = (
        matrix[gene_column]
        .dropna()
        .to_numpy(dtype=float)
    )

    n = len(values)

    if n == 0:

        return {
            "n": 0,
            "median": np.nan,
            "q25": np.nan,
            "q75": np.nan,
            "fraction_below_minus_05": np.nan,
            "fraction_below_minus_1": np.nan
        }

    return {

        "n": n,

        "median": float(
            np.median(values)
        ),

        "q25": float(
            np.percentile(values, 25)
        ),

        "q75": float(
            np.percentile(values, 75)
        ),

        "fraction_below_minus_05": float(
            np.mean(values < -0.5)
        ),

        "fraction_below_minus_1": float(
            np.mean(values < -1.0)
        )

    }


summary_records = []


for _, row in gene_mapping.iterrows():

    gene_name = row["gene_name"]

    gene_column = row["depmap_column"]

    record = row.to_dict()

    if pd.isna(gene_column):

        record.update({

            "crc_models_measured": 0,

            "crc_median_gene_effect": np.nan,

            "crc_q25_gene_effect": np.nan,

            "crc_q75_gene_effect": np.nan,

            "crc_fraction_below_minus_05": np.nan,

            "crc_fraction_below_minus_1": np.nan,

            "other_models_measured": 0,

            "other_median_gene_effect": np.nan,

            "other_fraction_below_minus_05": np.nan,

            "crc_minus_other_median_effect": np.nan
        })

    else:

        crc_stats = summarize_gene_effect(
            crc_effects,
            gene_column
        )

        other_stats = summarize_gene_effect(
            other_effects,
            gene_column
        )

        if (
            crc_stats["n"] > 0
            and other_stats["n"] > 0
        ):

            median_difference = (
                crc_stats["median"]
                - other_stats["median"]
            )

        else:

            median_difference = np.nan

        record.update({

            "crc_models_measured":
                crc_stats["n"],

            "crc_median_gene_effect":
                crc_stats["median"],

            "crc_q25_gene_effect":
                crc_stats["q25"],

            "crc_q75_gene_effect":
                crc_stats["q75"],

            "crc_fraction_below_minus_05":
                crc_stats[
                    "fraction_below_minus_05"
                ],

            "crc_fraction_below_minus_1":
                crc_stats[
                    "fraction_below_minus_1"
                ],

            "other_models_measured":
                other_stats["n"],

            "other_median_gene_effect":
                other_stats["median"],

            "other_fraction_below_minus_05":
                other_stats[
                    "fraction_below_minus_05"
                ],

            "crc_minus_other_median_effect":
                median_difference
        })


    summary_records.append(record)


dependency_summary = pd.DataFrame(
    summary_records
)


if len(dependency_summary) != 948:

    raise RuntimeError(
        "Dependency summary changed candidate count."
    )


print("\nCandidate dependency summaries:",
      len(dependency_summary))


print(
    "Candidates with colorectal CRISPR data:",
    int(
        dependency_summary[
            "crc_models_measured"
        ].gt(0).sum()
    )
)


print(
    "Candidates with >=5 colorectal measurements:",
    int(
        dependency_summary[
            "crc_models_measured"
        ].ge(5).sum()
    )
)


# ------------------------------------------------------------
# 13. EXPLORE STRONGER NEGATIVE GENE EFFECTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("EXPLORATORY COLORECTAL FITNESS EFFECTS")
print("-" * 65)


# Descriptive screening only.
#
# No statistical claims about selective
# dependency are made here.


rankable = dependency_summary[
    dependency_summary[
        "crc_models_measured"
    ].ge(5)
].copy()


rankable = rankable.sort_values(
    "crc_median_gene_effect",
    ascending=True
)


print("\nTop 20 candidates by most negative "
      "median colorectal gene effect:")


print(
    rankable[
        [
            "gene_name",
            "crc_models_measured",
            "crc_median_gene_effect",
            "crc_fraction_below_minus_05",
            "other_median_gene_effect",
            "crc_minus_other_median_effect"
        ]
    ]
    .head(20)
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ------------------------------------------------------------
# 14. REVIEW LEADING CANCER-TARGET CANDIDATES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LEADING CANDIDATE FUNCTIONAL REVIEW")
print("-" * 65)


LEADING_GENES = [
    "CDH3",
    "CLDN1",
    "FAP",
    "DPEP1",
    "SLC7A5",
    "SLC5A6",
    "CTHRC1",
    "S100P",
    "KRT80",
    "CEMIP",
    "INHBA",
    "WNT2"
]


leading = dependency_summary[
    dependency_summary["gene_name"]
    .isin(LEADING_GENES)
].copy()


leading["display_order"] = (
    leading["gene_name"]
    .map({
        gene: index
        for index, gene in enumerate(
            LEADING_GENES
        )
    })
)


leading = leading.sort_values(
    "display_order"
)


print("\n")

print(
    leading[
        [
            "gene_name",
            "crc_models_measured",
            "crc_median_gene_effect",
            "crc_fraction_below_minus_05",
            "other_median_gene_effect",
            "crc_minus_other_median_effect"
        ]
    ]
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


print("\nIMPORTANT:")

print(
    "CRISPR dependency is not necessary "
    "for antibody-drug conjugate targeting."
)

print(
    "Weak dependency in cancer cell lines "
    "does not exclude stromal or "
    "antigen-mediated targeting strategies."
)


# ------------------------------------------------------------
# 15. SAVE COLORECTAL MODEL INVENTORIES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING DEPMAP MODEL INVENTORIES")
print("-" * 65)


screened_models.to_csv(
    RESULTS /
    "depmap_26q3_screened_model_metadata.csv",
    index=False
)


colorectal_models.to_csv(
    RESULTS /
    "depmap_26q3_colorectal_model_inventory.csv",
    index=False
)


primary_crc_models.to_csv(
    RESULTS /
    "depmap_26q3_primary_crc_analysis_models.csv",
    index=False
)


histology_review.to_csv(
    RESULTS /
    "depmap_26q3_histology_review_models.csv",
    index=False
)


# ------------------------------------------------------------
# 16. SAVE CANDIDATE GENE MAPPINGS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING CRISPR GENE MAPPINGS")
print("-" * 65)


gene_mapping.to_csv(
    RESULTS /
    "depmap_26q3_candidate_gene_mapping.csv",
    index=False
)


unresolved.to_csv(
    RESULTS /
    "depmap_26q3_unresolved_gene_mapping.csv",
    index=False
)


# ------------------------------------------------------------
# 17. SAVE COLORECTAL CANDIDATE MATRIX
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING CANDIDATE CRISPR MATRIX")
print("-" * 65)


column_to_symbol = dict(
    zip(
        uniquely_mapped["depmap_column"],
        uniquely_mapped["gene_name"]
    )
)


crc_matrix_export = crc_effects.rename(
    columns=column_to_symbol
).copy()


crc_matrix_export.index.name = "ModelID"


crc_matrix_export.to_csv(
    RESULTS /
    "depmap_26q3_crc_candidate_gene_effects.csv"
)


dependency_summary.to_csv(
    RESULTS /
    "depmap_26q3_candidate_gene_effect_summary.csv",
    index=False
)


leading.to_csv(
    RESULTS /
    "depmap_26q3_leading_candidate_dependency.csv",
    index=False
)


print("\nAll dependency audit tables saved.")


# ------------------------------------------------------------
# 18. SAVE RESEARCH PROVENANCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING PHASE 4 PROVENANCE")
print("-" * 65)


provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "4.2",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "depmap_release": "26Q3",

    "model_source": str(MODEL_PATH),

    "crispr_source": str(CRISPR_PATH),

    "model_sha256": sha256_file(
        MODEL_PATH
    ),

    "crispr_sha256": sha256_file(
        CRISPR_PATH
    ),

    "source_integrity": (
        "Locally checksum-verified reconstructed "
        "CSV; official source checksum unavailable"
    ),

    "screened_models": int(
        len(screened_ids)
    ),

    "colorectal_models_identified": int(
        len(colorectal_models)
    ),

    "primary_crc_models": int(
        len(primary_crc_models)
    ),

    "histology_review_models": int(
        len(histology_review)
    ),

    "candidate_genes": int(
        len(gene_mapping)
    ),

    "unique_exact_symbol_matches": int(
        len(uniquely_mapped)
    ),

    "unresolved_gene_mappings": int(
        len(unresolved)
    ),

    "candidate_gene_matching": (
        "Unique exact gene-symbol correspondence "
        "to DepMap Symbol (EntrezID) headers; "
        "Entrez ID retained for future crosswalk validation"
    ),

    "model_selection": (
        "Screened models with primary disease "
        "explicitly annotated as colorectal, "
        "colon or rectal; distinct histologies "
        "separately flagged"
    ),

    "msi_status_assigned": False,

    "gene_effect_summary": [
        "Median",
        "25th percentile",
        "75th percentile",
        "Fraction below -0.5",
        "Fraction below -1",
        "Other-model median",
        "Difference between group medians"
    ],

    "inferential_dependency_testing":
        False,

    "target_ranking_performed": False,

    "candidate_exclusions": 0,

    "limitations": [

        "The CRISPR file was reconstructed "
        "from upload fragments.",

        "Official byte-for-byte checksum "
        "verification is unavailable.",

        "Colorectal model status does not "
        "establish MSS status.",

        "Different colorectal histologies "
        "require biological review.",

        "DepMap models may not be independent "
        "at the patient level.",

        "Initial gene mapping uses exact "
        "symbols rather than validated "
        "Ensembl-to-Entrez correspondence.",

        "More negative gene-effect values "
        "suggest stronger knockout fitness effects.",

        "Median differences are descriptive "
        "and do not establish lineage selectivity.",

        "The noncolorectal comparator includes "
        "heterogeneous cancer lineages.",

        "CRISPR dependency is not a necessary "
        "criterion for ADC antigen suitability.",

        "Stromal antigens may not show dependency "
        "in cancer-cell-intrinsic screens.",

        "No therapeutic targets have been validated."
    ]
}


with open(
    META /
    "depmap_26q3_candidate_dependency_provenance.json",
    "w"
) as handle:

    json.dump(
        provenance,
        handle,
        indent=4
    )


print("\nResearch provenance saved.")


# ------------------------------------------------------------
# 19. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 4.2 SUMMARY")
print("=" * 65)


print("\nDepMap release: Public 26Q3")


print(
    "Screened cancer models:",
    len(screened_ids)
)


print(
    "Colorectal models identified:",
    len(colorectal_models)
)


print(
    "Primary colorectal analysis models:",
    len(primary_crc_models)
)


print(
    "Distinct-histology models flagged:",
    len(histology_review)
)


print(
    "\nTranscriptomic screening genes:",
    len(gene_mapping)
)


print(
    "Uniquely matched DepMap genes:",
    len(uniquely_mapped)
)


print(
    "Unresolved gene mappings:",
    len(unresolved)
)


print(
    "Genes with available colorectal CRISPR data:",
    int(
        dependency_summary[
            "crc_models_measured"
        ].gt(0).sum()
    )
)


print("\nSaved outputs:")

print("1. depmap_26q3_screened_model_metadata.csv")

print("2. depmap_26q3_colorectal_model_inventory.csv")

print("3. depmap_26q3_primary_crc_analysis_models.csv")

print("4. depmap_26q3_histology_review_models.csv")

print("5. depmap_26q3_candidate_gene_mapping.csv")

print("6. depmap_26q3_unresolved_gene_mapping.csv")

print("7. depmap_26q3_crc_candidate_gene_effects.csv")

print("8. depmap_26q3_candidate_gene_effect_summary.csv")

print("9. depmap_26q3_leading_candidate_dependency.csv")

print("10. depmap_26q3_candidate_dependency_provenance.json")


print("\nModel matching: PASSED")

print("Gene identifier audit: COMPLETE")

print("Descriptive CRISPR analysis: COMPLETE")


print("\nNo MSS status inferred.")

print("No final target scores assigned.")

print("No source files modified.")

print("No candidate genes excluded.")


print("\nPATCH 4.2 COMPLETE")

print("=" * 65)

In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 4 — PATCH 4.3
#
# COLORECTAL COHORT REFINEMENT AND
# CRISPR DEPENDENCY ROBUSTNESS
#
# Objectives:
# 1. Validate existing DepMap inputs
# 2. Correct colorectal adenocarcinoma cohort definition
# 3. Exclude benign adenoma from primary CRC analysis
# 4. Audit repeated originating patient identifiers
# 5. Recalculate candidate CRISPR dependency summaries
# 6. Calculate one-model-per-patient sensitivity estimates
# 7. Account for gene-level missingness
# 8. Review other cancer lineage measurements
# 9. Generate descriptive dependency figure
# 10. Save corrected results and provenance
#
# IMPORTANT:
# - No MSS status inferred
# - No permanent exclusion of research candidates
# - No therapeutic target scores assigned
# - No modification of original DepMap datasets
# - No changes to Patch 4.2 outputs
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import hashlib
import json
import math


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"

DEPMAP = ROOT / "data/raw/depmap/26Q3"

RESULTS = ROOT / "results/tables"

FIGURES = ROOT / "results/figures"

for directory in [META, RESULTS, FIGURES]:
    directory.mkdir(parents=True, exist_ok=True)


MODEL_PATH = DEPMAP / "Model.csv"

CRISPR_PATH = DEPMAP / "CRISPRGeneEffect.csv"

IMPORT_PATH = (
    META / "depmap_26q3_import_verification.csv"
)

MAPPING_PATH = (
    RESULTS / "depmap_26q3_candidate_gene_mapping.csv"
)


LEADING_GENES = [
    "CDH3",
    "CLDN1",
    "FAP",
    "DPEP1",
    "SLC7A5",
    "SLC5A6",
    "CTHRC1",
    "S100P",
    "KRT80",
    "CEMIP",
    "INHBA",
    "WNT2"
]


print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 4 — PATCH 4.3")
print("CRC DEPENDENCY ROBUSTNESS AUDIT")
print("=" * 65)


# ------------------------------------------------------------
# 2. VALIDATE INPUTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING DEPMAP INPUTS")
print("-" * 65)


for path in [
    MODEL_PATH,
    CRISPR_PATH,
    IMPORT_PATH,
    MAPPING_PATH
]:

    if not path.exists():
        raise FileNotFoundError(path)


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


audit = pd.read_csv(
    IMPORT_PATH,
    dtype=str,
    keep_default_na=False
)

audit = audit.set_index("filename")


for path in [MODEL_PATH, CRISPR_PATH]:

    if path.name not in audit.index:
        raise RuntimeError(
            f"Missing checksum record: {path.name}"
        )

    expected = audit.loc[
        path.name,
        "sha256"
    ]

    actual = sha256_file(path)

    if expected != actual:
        raise RuntimeError(
            f"Source checksum mismatch: {path.name}"
        )


print("\nLocally recorded source checksums: PASSED")

print(
    "Official release checksum comparison "
    "remains unavailable."
)


# ------------------------------------------------------------
# 3. LOAD MODEL METADATA
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING SCREENED CANCER MODELS")
print("-" * 65)


models = pd.read_csv(
    MODEL_PATH,
    dtype=str,
    keep_default_na=False
)


if models["ModelID"].duplicated().any():
    raise RuntimeError(
        "Duplicate DepMap model IDs."
    )


models["ModelID"] = (
    models["ModelID"]
    .astype(str)
    .str.strip()
)


model_columns = [
    "ModelID",
    "PatientID",
    "CellLineName",
    "OncotreeLineage",
    "OncotreePrimaryDisease",
    "OncotreeSubtype"
]


if not set(model_columns).issubset(models.columns):
    raise RuntimeError(
        "Required model metadata columns missing."
    )


# ------------------------------------------------------------
# 4. LOAD CANDIDATE GENE MAPPING
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING PREVIOUS GENE MAPPINGS")
print("-" * 65)


mapping = pd.read_csv(
    MAPPING_PATH,
    dtype=str,
    keep_default_na=False
)


if len(mapping) != 948:
    raise RuntimeError(
        "Expected 948 candidate genes."
    )


valid_mapping = mapping[
    mapping["depmap_mapping_status"].eq(
        "Unique exact gene-symbol match"
    )
].copy()


if len(valid_mapping) != 930:
    raise RuntimeError(
        "Expected 930 previously matched genes."
    )


if valid_mapping["depmap_column"].duplicated().any():
    raise RuntimeError(
        "Duplicate selected DepMap columns."
    )


print("\nCandidates:", len(mapping))

print("Unique exact symbol matches:", len(valid_mapping))

print("Unresolved mappings:", len(mapping) - len(valid_mapping))


# ------------------------------------------------------------
# 5. EXTRACT CANDIDATE CRISPR MATRIX
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING CANDIDATE CRISPR MEASUREMENTS")
print("-" * 65)


header = pd.read_csv(
    CRISPR_PATH,
    nrows=0
).columns.tolist()


if len(header) != 18435:
    raise RuntimeError(
        "Unexpected CRISPR gene-column count."
    )


model_column = header[0]


if model_column not in [
    "Unnamed: 0",
    "ModelID"
]:
    raise RuntimeError(
        "Unexpected CRISPR model-ID column."
    )


selected_columns = (
    valid_mapping["depmap_column"]
    .tolist()
)


if not set(selected_columns).issubset(set(header)):
    raise RuntimeError(
        "Candidate columns missing from CRISPR file."
    )


effects = pd.read_csv(
    CRISPR_PATH,
    usecols=[model_column] + selected_columns,
    dtype={model_column: str},
    low_memory=False
)


effects = effects.rename(
    columns={model_column: "ModelID"}
)


effects["ModelID"] = (
    effects["ModelID"]
    .astype(str)
    .str.strip()
)


if effects["ModelID"].duplicated().any():
    raise RuntimeError(
        "Duplicate screened ModelIDs."
    )


effects = effects.set_index("ModelID")


effects = effects.apply(
    pd.to_numeric,
    errors="raise"
)


if np.isinf(
    effects.to_numpy(dtype=float)
).any():
    raise RuntimeError(
        "Infinite CRISPR gene-effect values."
    )


if effects.shape != (1361, 930):
    raise RuntimeError(
        f"Unexpected candidate matrix: {effects.shape}"
    )


print("\nCandidate CRISPR matrix:", effects.shape)

print("Candidate matrix validation: PASSED")


# ------------------------------------------------------------
# 6. ALIGN SCREENED MODEL METADATA
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("ALIGNING SCREENED MODEL METADATA")
print("-" * 65)


model_indexed = models.set_index("ModelID")


if not set(effects.index).issubset(
    set(model_indexed.index)
):
    raise RuntimeError(
        "Screened models missing metadata."
    )


screened = model_indexed.loc[
    effects.index
].copy()


screened.index.name = "ModelID"


print("\nScreened models:", len(screened))

print("Metadata correspondence: PASSED")


# ------------------------------------------------------------
# 7. CORRECT CRC ADENOCARCINOMA DEFINITION
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("REFINING COLORECTAL CANCER COHORT")
print("-" * 65)


# The previous patch included one model
# classified as Tubular Adenoma of the Colon.
#
# The primary cohort must explicitly carry
# the colorectal adenocarcinoma disease label.


crc_mask = (
    screened["OncotreePrimaryDisease"]
    .str.strip()
    .eq("Colorectal Adenocarcinoma")
)


crc_models = screened[
    crc_mask
].copy()


adenoma_mask = (
    screened["OncotreePrimaryDisease"]
    .eq("Tubular Adenoma of the Colon")
)


adenoma_models = screened[
    adenoma_mask
].copy()


print("\nColorectal adenocarcinoma models:",
      len(crc_models))

print("Tubular colon adenoma models:",
      len(adenoma_models))


if len(crc_models) != 81:
    raise RuntimeError(
        "Expected 81 colorectal adenocarcinoma models."
    )


if len(adenoma_models) != 1:
    raise RuntimeError(
        "Expected one tubular adenoma model."
    )


print("\nCRC subtype distribution:")

print(
    crc_models[
        "OncotreeSubtype"
    ].value_counts().to_string()
)


print("\nExcluded precursor model:")

print(
    adenoma_models[
        [
            "CellLineName",
            "OncotreePrimaryDisease",
            "OncotreeSubtype"
        ]
    ].to_string()
)


print("\nPrimary CRC cohort refinement: PASSED")


# ------------------------------------------------------------
# 8. DEFINE OTHER-LINEAGE COMPARATOR
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("DEFINING OTHER-LINEAGE COMPARATOR")
print("-" * 65)


# Exclude Bowel-lineage models from the
# primary comparator.
#
# This prevents benign colorectal precursor
# models and other bowel diseases from entering
# the comparator group.
#
# Also exclude models explicitly annotated
# as benign/non-cancerous elsewhere.
#
# This is a descriptive comparator, not
# a clinically matched control cohort.


bowel_mask = (
    screened["OncotreeLineage"]
    .str.contains(
        r"bowel",
        case=False,
        regex=True,
        na=False
    )
)


nonmalignant_mask = (
    screened["OncotreePrimaryDisease"]
    .str.contains(
        r"non[- ]?cancerous|"
        r"benign|adenoma|"
        r"normal tissue|"
        r"hyperplasia|"
        r"dysplasia|"
        r"sessile serrated",
        case=False,
        regex=True,
        na=False
    )
)


other_mask = (
    ~bowel_mask
    & ~nonmalignant_mask
    & ~crc_mask
)


other_models = screened[
    other_mask
].copy()


print("\nCRC adenocarcinoma models:",
      len(crc_models))

print(
    "Other-lineage comparator models:",
    len(other_models)
)

print(
    "Screened bowel-lineage models:",
    int(bowel_mask.sum())
)

print(
    "Explicit nonmalignant/precursor "
    "annotations:",
    int(nonmalignant_mask.sum())
)


if len(other_models) < 500:
    raise RuntimeError(
        "Unexpectedly small other-lineage comparator."
    )


print("\nComparator definition: PASSED")

print(
    "Other-lineage models are heterogeneous "
    "and are not independent biological controls."
)


# ------------------------------------------------------------
# 9. REPEATED PATIENT AUDIT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("ORIGINATING PATIENT DUPLICATION AUDIT")
print("-" * 65)


crc_patient = crc_models.reset_index().copy()


crc_patient["PatientID"] = (
    crc_patient["PatientID"]
    .astype(str)
    .str.strip()
)


# Missing PatientIDs are treated as separate
# unknown-origin models, rather than incorrectly
# grouping all missing values as one patient.


crc_patient["patient_group"] = np.where(
    crc_patient["PatientID"].ne(""),
    crc_patient["PatientID"],
    "UNKNOWN_" + crc_patient["ModelID"]
)


duplicate_patients = crc_patient[
    crc_patient["patient_group"].duplicated(
        keep=False
    )
].copy()


print("\nModels sharing a recorded patient ID:",
      len(duplicate_patients))


if not duplicate_patients.empty:

    print("\nRepeated-patient models:")

    print(
        duplicate_patients[
            [
                "PatientID",
                "ModelID",
                "CellLineName",
                "OncotreeSubtype"
            ]
        ]
        .sort_values(
            ["PatientID", "ModelID"]
        )
        .to_string(index=False)
    )


# ------------------------------------------------------------
# 10. ONE-MODEL-PER-PATIENT SENSITIVITY COHORT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CONSTRUCTING PATIENT-DEDUPLICATED COHORT")
print("-" * 65)


crc_patient["measured_candidate_genes"] = (
    effects.loc[
        crc_patient["ModelID"]
    ]
    .notna()
    .sum(axis=1)
    .to_numpy()
)


# Deterministic sensitivity selection:
#
# 1. Keep model with most measured candidate genes.
# 2. Break ties by ModelID.
#
# This is a robustness assessment, not
# a biological claim that one model is better.


crc_patient_sorted = crc_patient.sort_values(
    [
        "measured_candidate_genes",
        "ModelID"
    ],
    ascending=[
        False,
        True
    ]
)


deduplicated = (
    crc_patient_sorted
    .drop_duplicates(
        subset="patient_group",
        keep="first"
    )
    .copy()
)


deduplicated_ids = (
    deduplicated["ModelID"]
    .tolist()
)


print("\nOriginal CRC models:", len(crc_models))

print(
    "One-model-per-patient sensitivity models:",
    len(deduplicated_ids)
)


if len(deduplicated_ids) > len(crc_models):
    raise RuntimeError(
        "Sensitivity cohort unexpectedly expanded."
    )


print("Patient deduplication: PASSED")


# ------------------------------------------------------------
# 11. BUILD ANALYSIS MATRICES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING CRISPR ANALYSIS GROUPS")
print("-" * 65)


crc_effects = effects.loc[
    crc_models.index
].copy()


other_effects = effects.loc[
    other_models.index
].copy()


sensitivity_effects = effects.loc[
    deduplicated_ids
].copy()


print("\nCRC matrix:", crc_effects.shape)

print("Other-lineage matrix:", other_effects.shape)

print(
    "Patient-deduplicated matrix:",
    sensitivity_effects.shape
)


# ------------------------------------------------------------
# 12. GENE-EFFECT SUMMARY FUNCTION
# ------------------------------------------------------------

def summarize_effect(values):

    values = np.asarray(
        values,
        dtype=float
    )

    values = values[
        np.isfinite(values)
    ]

    n = len(values)

    if n == 0:

        return {
            "n": 0,
            "median": np.nan,
            "q25": np.nan,
            "q75": np.nan,
            "fraction_lt_05": np.nan,
            "fraction_lt_1": np.nan
        }

    return {

        "n": int(n),

        "median": float(
            np.median(values)
        ),

        "q25": float(
            np.percentile(values, 25)
        ),

        "q75": float(
            np.percentile(values, 75)
        ),

        "fraction_lt_05": float(
            np.mean(values < -0.5)
        ),

        "fraction_lt_1": float(
            np.mean(values < -1.0)
        )

    }


# ------------------------------------------------------------
# 13. RECALCULATE CANDIDATE DEPENDENCIES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CALCULATING CORRECTED CRISPR DEPENDENCIES")
print("-" * 65)


records = []


for _, row in mapping.iterrows():

    gene_name = row["gene_name"]

    depmap_col = row["depmap_column"]

    result = row.to_dict()

    if (
        not depmap_col
        or depmap_col not in effects.columns
    ):

        crc_stats = summarize_effect([])
        other_stats = summarize_effect([])
        sensitivity_stats = summarize_effect([])

    else:

        crc_stats = summarize_effect(
            crc_effects[depmap_col]
        )

        other_stats = summarize_effect(
            other_effects[depmap_col]
        )

        sensitivity_stats = summarize_effect(
            sensitivity_effects[depmap_col]
        )


    crc_med = crc_stats["median"]

    other_med = other_stats["median"]

    sensitivity_med = sensitivity_stats["median"]


    result.update({

        "crc_models_total":
            len(crc_effects),

        "crc_models_measured":
            crc_stats["n"],

        "crc_measurement_fraction":
            crc_stats["n"] / len(crc_effects),

        "crc_median_gene_effect":
            crc_med,

        "crc_q25_gene_effect":
            crc_stats["q25"],

        "crc_q75_gene_effect":
            crc_stats["q75"],

        "crc_fraction_below_minus_05":
            crc_stats["fraction_lt_05"],

        "crc_fraction_below_minus_1":
            crc_stats["fraction_lt_1"],

        "other_models_measured":
            other_stats["n"],

        "other_median_gene_effect":
            other_med,

        "other_fraction_below_minus_05":
            other_stats["fraction_lt_05"],

        "crc_minus_other_median_effect":
            (
                crc_med - other_med
                if np.isfinite(crc_med)
                and np.isfinite(other_med)
                else np.nan
            ),

        "deduplicated_models_measured":
            sensitivity_stats["n"],

        "deduplicated_median_gene_effect":
            sensitivity_med,

        "dedup_minus_primary_median":
            (
                sensitivity_med - crc_med
                if np.isfinite(sensitivity_med)
                and np.isfinite(crc_med)
                else np.nan
            )

    })

    records.append(result)


summary = pd.DataFrame(records)


if len(summary) != 948:
    raise RuntimeError(
        "Candidate count changed."
    )


print("\nCandidates summarized:", len(summary))


print(
    "Candidates with CRC measurements:",
    int(
        summary["crc_models_measured"]
        .gt(0)
        .sum()
    )
)


# ------------------------------------------------------------
# 14. COVERAGE-AWARE DEPENDENCY REVIEW
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENE COVERAGE AND DEPENDENCY REVIEW")
print("-" * 65)


# An 80% coverage threshold is used for
# descriptive robustness screening.
#
# It is not a biological dependency threshold.


MIN_CRC_COVERAGE = math.ceil(
    0.80 * len(crc_effects)
)


summary["high_crc_coverage"] = (
    summary["crc_models_measured"]
    .ge(MIN_CRC_COVERAGE)
)


print("\nCRC models:", len(crc_effects))

print(
    "Minimum for high coverage:",
    MIN_CRC_COVERAGE
)


print(
    "Candidates meeting high coverage:",
    int(
        summary["high_crc_coverage"].sum()
    )
)


low_coverage = summary[
    summary["crc_models_measured"].gt(0)
    & ~summary["high_crc_coverage"]
].copy()


print(
    "Mapped candidates with lower coverage:",
    len(low_coverage)
)


high_coverage_ranked = summary[
    summary["high_crc_coverage"]
].sort_values(
    "crc_median_gene_effect"
)


print("\nTop 20 negative effects among "
      "high-coverage candidates:")


print(
    high_coverage_ranked[
        [
            "gene_name",
            "crc_models_measured",
            "crc_median_gene_effect",
            "crc_fraction_below_minus_05",
            "other_median_gene_effect",
            "crc_minus_other_median_effect"
        ]
    ]
    .head(20)
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ------------------------------------------------------------
# 15. LEADING CANDIDATE ROBUSTNESS REVIEW
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LEADING CANDIDATE DEPENDENCY ROBUSTNESS")
print("-" * 65)


leading = summary[
    summary["gene_name"].isin(
        LEADING_GENES
    )
].copy()


leading["display_order"] = (
    leading["gene_name"].map({
        gene: idx
        for idx, gene in enumerate(
            LEADING_GENES
        )
    })
)


leading = leading.sort_values(
    "display_order"
)


print("\n")


print(
    leading[
        [
            "gene_name",
            "crc_models_measured",
            "crc_median_gene_effect",
            "crc_fraction_below_minus_05",
            "other_median_gene_effect",
            "deduplicated_median_gene_effect",
            "dedup_minus_primary_median"
        ]
    ]
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ------------------------------------------------------------
# 16. CROSS-LINEAGE DESCRIPTIVE AUDIT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CROSS-LINEAGE DEPENDENCY COVERAGE")
print("-" * 65)


lineage_counts = (
    other_models["OncotreeLineage"]
    .replace("", "NOT AVAILABLE")
    .value_counts()
)


print("\nOther-lineage model distribution:")

print(
    lineage_counts.to_string()
)


# Save descriptive summaries for the
# biologically relevant leading genes.
#
# No inferential lineage-specificity
# claims are made.


lineage_records = []


for gene in LEADING_GENES:

    matching = valid_mapping[
        valid_mapping["gene_name"].eq(gene)
    ]

    if len(matching) != 1:
        continue

    column = matching.iloc[0]["depmap_column"]

    for lineage, group in other_models.groupby(
        "OncotreeLineage"
    ):

        values = effects.loc[
            group.index,
            column
        ]

        stats = summarize_effect(values)

        lineage_records.append({

            "gene_name": gene,

            "lineage": lineage,

            "models_total": len(group),

            "models_measured": stats["n"],

            "median_gene_effect": stats["median"],

            "fraction_below_minus_05":
                stats["fraction_lt_05"]

        })


lineage_summary = pd.DataFrame(
    lineage_records
)


print(
    "\nLeading-gene lineage summaries:",
    len(lineage_summary)
)


print(
    "These summaries are descriptive; "
    "differences may reflect lineage composition."
)


# ------------------------------------------------------------
# 17. GENERATE LEADING-GENE DEPENDENCY FIGURE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENERATING DEPENDENCY COMPARISON FIGURE")
print("-" * 65)


plot_data = leading.dropna(
    subset=[
        "crc_median_gene_effect",
        "other_median_gene_effect"
    ]
).copy()


positions = np.arange(
    len(plot_data)
)


fig, ax = plt.subplots(
    figsize=(11, 7)
)


ax.scatter(
    plot_data["crc_median_gene_effect"],
    positions,
    s=70,
    label="CRC adenocarcinoma"
)


ax.scatter(
    plot_data["other_median_gene_effect"],
    positions,
    s=70,
    marker="x",
    label="Other screened lineages"
)


for position, (_, row) in enumerate(
    plot_data.iterrows()
):

    ax.plot(
        [
            row["crc_median_gene_effect"],
            row["other_median_gene_effect"]
        ],
        [position, position],
        alpha=0.35
    )


ax.axvline(
    0,
    linestyle="--",
    linewidth=1
)


ax.axvline(
    -0.5,
    linestyle=":",
    linewidth=1
)


ax.set_yticks(positions)


ax.set_yticklabels(
    plot_data["gene_name"]
)


ax.invert_yaxis()


ax.set_xlabel(
    "Median CRISPR gene-effect score"
)


ax.set_title(
    "DepMap 26Q3 — Leading SurfaceTarget-CRC Candidates"
)


ax.legend()


ax.grid(
    axis="x",
    alpha=0.2
)


fig.tight_layout()


figure_path = (
    FIGURES /
    "depmap_26q3_crc_dependency_comparison.png"
)


fig.savefig(
    figure_path,
    dpi=200,
    bbox_inches="tight"
)


plt.show()
plt.close(fig)


print("\nDependency comparison figure saved.")


# ------------------------------------------------------------
# 18. SAVE CORRECTED ANALYSIS OUTPUTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING CORRECTED DEPMAP RESULTS")
print("-" * 65)


crc_models.reset_index().to_csv(
    RESULTS /
    "depmap_26q3_refined_crc_models.csv",
    index=False
)


adenoma_models.reset_index().to_csv(
    RESULTS /
    "depmap_26q3_excluded_precursor_models.csv",
    index=False
)


duplicate_patients.to_csv(
    RESULTS /
    "depmap_26q3_repeated_patient_review.csv",
    index=False
)


deduplicated.to_csv(
    RESULTS /
    "depmap_26q3_patient_deduplicated_models.csv",
    index=False
)


summary.to_csv(
    RESULTS /
    "depmap_26q3_refined_dependency_summary.csv",
    index=False
)


low_coverage.to_csv(
    RESULTS /
    "depmap_26q3_low_coverage_gene_review.csv",
    index=False
)


leading.to_csv(
    RESULTS /
    "depmap_26q3_refined_leading_dependency.csv",
    index=False
)


lineage_summary.to_csv(
    RESULTS /
    "depmap_26q3_leading_gene_lineage_summary.csv",
    index=False
)


# Save corrected colorectal candidate matrix.
#
# Only the refined 81-model cohort is exported.

column_to_gene = dict(
    zip(
        valid_mapping["depmap_column"],
        valid_mapping["gene_name"]
    )
)


crc_export = crc_effects.rename(
    columns=column_to_gene
)


crc_export.index.name = "ModelID"


crc_export.to_csv(
    RESULTS /
    "depmap_26q3_refined_crc_candidate_matrix.csv"
)


print("\nAll refined analysis tables saved.")


# ------------------------------------------------------------
# 19. SAVE PROVENANCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING RESEARCH PROVENANCE")
print("-" * 65)


provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "4.3",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "depmap_release": "Public 26Q3",

    "model_source_sha256":
        sha256_file(MODEL_PATH),

    "crispr_source_sha256":
        sha256_file(CRISPR_PATH),

    "official_source_checksum_verified": False,

    "original_crc_related_models": 82,

    "refined_crc_adenocarcinoma_models":
        int(len(crc_models)),

    "excluded_tubular_adenoma_models":
        int(len(adenoma_models)),

    "crc_primary_disease_rule": (
        "Exact match: Colorectal Adenocarcinoma"
    ),

    "other_lineage_comparator_models":
        int(len(other_models)),

    "comparator_definition": (
        "Screened non-Bowel models, excluding "
        "explicit benign and precursor annotations"
    ),

    "patient_deduplicated_crc_models":
        int(len(deduplicated_ids)),

    "patient_deduplication_rule": (
        "One model per nonmissing PatientID, "
        "selected by candidate gene coverage "
        "then lexicographic ModelID"
    ),

    "candidate_genes": 948,

    "exact_symbol_mapped_genes":
        int(len(valid_mapping)),

    "unresolved_gene_mappings":
        int(len(mapping) - len(valid_mapping)),

    "high_coverage_definition": (
        "Measurements in at least 80% "
        "of refined CRC adenocarcinoma models"
    ),

    "high_coverage_minimum_models":
        int(MIN_CRC_COVERAGE),

    "gene_effect_summary_statistics": [
        "Median",
        "25th percentile",
        "75th percentile",
        "Fraction below -0.5",
        "Fraction below -1.0",
        "Other-lineage median",
        "Patient-deduplicated CRC median"
    ],

    "msi_status_inferred": False,

    "formal_lineage_selectivity_testing": False,

    "therapeutic_target_ranking": False,

    "original_patch_42_outputs_overwritten": False,

    "limitations": [

        "Source checksum comparison with "
        "official DepMap release unavailable.",

        "Colorectal models are not confirmed MSS.",

        "Some models share originating patients.",

        "Initial gene matching uses exact "
        "gene-symbol correspondence.",

        "Gene effects are cancer-model "
        "fitness measurements.",

        "Other-lineage comparator is heterogeneous.",

        "Colorectal versus other-lineage median "
        "differences are descriptive.",

        "Low CRISPR dependency does not "
        "exclude antibody-targeting relevance.",

        "Stromal targets are not adequately "
        "modeled by cancer-cell-intrinsic screens.",

        "No therapeutically validated "
        "targets have been established."

    ]
}


with open(
    META /
    "depmap_26q3_refined_dependency_provenance.json",
    "w"
) as handle:

    json.dump(
        provenance,
        handle,
        indent=4
    )


print("\nProvenance saved.")


# ------------------------------------------------------------
# 20. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 4.3 SUMMARY")
print("=" * 65)


print("\nDepMap release: Public 26Q3")


print(
    "Original colorectal-related models: 82"
)


print(
    "Refined CRC adenocarcinoma models:",
    len(crc_models)
)


print(
    "Tubular adenoma models withheld:",
    len(adenoma_models)
)


print(
    "Patient-deduplicated sensitivity models:",
    len(deduplicated_ids)
)


print(
    "\nCandidate genes:",
    len(mapping)
)


print(
    "Mapped genes:",
    len(valid_mapping)
)


print(
    "Genes with >=80% CRC coverage:",
    int(
        summary["high_crc_coverage"].sum()
    )
)


print(
    "\nOther-lineage comparator models:",
    len(other_models)
)


print("\nOutputs:")

print("1. Refined CRC model inventory")

print("2. Tubular adenoma exclusion audit")

print("3. Repeated-patient review")

print("4. Patient-deduplicated model inventory")

print("5. Corrected candidate dependency summary")

print("6. Low-coverage gene review")

print("7. Leading-candidate robustness review")

print("8. Other-lineage dependency summary")

print("9. Corrected CRC candidate matrix")

print("10. Dependency comparison figure")

print("11. Research provenance")


print("\nPrimary CRC model correction: COMPLETE")

print("Patient-duplication sensitivity: COMPLETE")

print("Coverage-aware dependency analysis: COMPLETE")


print("\nNo MSS status inferred.")

print("No source measurements modified.")

print("No original results overwritten.")

print("No final target ranking performed.")


print("\nPATCH 4.3 COMPLETE")

print("=" * 65)

In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 4 — PATCH 4.4
#
# HGNC GENE IDENTIFIER CROSS-VALIDATION
#
# Objectives:
# 1. Validate existing DepMap candidate mappings
# 2. Download official HGNC complete gene annotations
# 3. Match Ensembl IDs to HGNC records
# 4. Extract NCBI Gene (Entrez) identifiers
# 5. Cross-check DepMap gene-column identifiers
# 6. Validate existing exact-symbol matches
# 7. Investigate previously unmatched genes
# 8. Flag ambiguous or conflicting annotations
# 9. Preserve existing dependency statistics
# 10. Save mapping audits and provenance
# 11. Create a Phase 4 research checkpoint
#
# IMPORTANT:
# - No automatic replacement of disputed mappings
# - No guessing from gene aliases
# - No automatic candidate exclusions
# - No CRISPR statistics recalculated
# - No original research results overwritten
# ============================================================


# ------------------------------------------------------------
# 1. IMPORTS
# ------------------------------------------------------------

from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

import pandas as pd
import numpy as np
import requests
import hashlib
import json
import re
import os


# ------------------------------------------------------------
# 2. PROJECT CONFIGURATION
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"

DEPMAP = ROOT / "data/raw/depmap/26Q3"

HGNC_RAW = ROOT / "data/raw/hgnc"

RESULTS = ROOT / "results/tables"

CHECKPOINTS = ROOT / "checkpoints"

for folder in [
    META,
    HGNC_RAW,
    RESULTS,
    CHECKPOINTS
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


MAPPING_PATH = (
    RESULTS /
    "depmap_26q3_candidate_gene_mapping.csv"
)

SUMMARY_PATH = (
    RESULTS /
    "depmap_26q3_refined_dependency_summary.csv"
)

CRISPR_PATH = (
    DEPMAP /
    "CRISPRGeneEffect.csv"
)

HGNC_PATH = (
    HGNC_RAW /
    "hgnc_complete_set.txt"
)


HGNC_URL = (
    "https://storage.googleapis.com/"
    "public-download-files/hgnc/tsv/tsv/"
    "hgnc_complete_set.txt"
)


LEADING_GENES = [
    "CDH3",
    "CLDN1",
    "FAP",
    "DPEP1",
    "SLC7A5",
    "SLC5A6",
    "CTHRC1",
    "S100P",
    "KRT80",
    "CEMIP",
    "INHBA",
    "WNT2"
]


print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 4 — PATCH 4.4")
print("HGNC GENE IDENTIFIER CROSS-VALIDATION")
print("=" * 65)


# ------------------------------------------------------------
# 3. VERIFY EXISTING RESEARCH FILES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING RESEARCH INPUTS")
print("-" * 65)


required_paths = [
    MAPPING_PATH,
    SUMMARY_PATH,
    CRISPR_PATH
]


for path in required_paths:

    if not path.exists():
        raise FileNotFoundError(path)


mapping = pd.read_csv(
    MAPPING_PATH,
    dtype=str,
    keep_default_na=False
)


summary = pd.read_csv(
    SUMMARY_PATH,
    dtype=str,
    keep_default_na=False
)


if len(mapping) != 948:
    raise RuntimeError(
        "Unexpected DepMap candidate count."
    )


if len(summary) != 948:
    raise RuntimeError(
        "Unexpected refined dependency summary."
    )


if mapping["gene_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate original gene identifiers."
    )


required_mapping_columns = {
    "gene_id",
    "ensembl_stable_id",
    "gene_name",
    "depmap_mapping_status",
    "depmap_column",
    "depmap_entrez_id"
}


if not required_mapping_columns.issubset(
    mapping.columns
):

    raise RuntimeError(
        "Unexpected candidate mapping schema."
    )


mapping["ensembl_stable_id"] = (
    mapping["ensembl_stable_id"]
    .astype(str)
    .str.strip()
)


if mapping[
    "ensembl_stable_id"
].duplicated().any():

    raise RuntimeError(
        "Duplicate stable Ensembl identifiers."
    )


originally_matched = (
    mapping["depmap_mapping_status"]
    .eq("Unique exact gene-symbol match")
)


if int(originally_matched.sum()) != 930:

    raise RuntimeError(
        "Expected 930 original symbol matches."
    )


print("\nCandidate genes:", len(mapping))

print(
    "Previously symbol-matched genes:",
    int(originally_matched.sum())
)

print(
    "Previously unmatched genes:",
    int((~originally_matched).sum())
)

print("\nResearch input validation: PASSED")


# ------------------------------------------------------------
# 4. DOWNLOAD OFFICIAL HGNC DATA
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("ACQUIRING OFFICIAL HGNC GENE ANNOTATIONS")
print("-" * 65)


session = requests.Session()

retry_policy = Retry(
    total=4,
    backoff_factor=1,
    status_forcelist=[
        429, 500, 502, 503, 504
    ],
    allowed_methods=["GET"]
)

session.mount(
    "https://",
    HTTPAdapter(
        max_retries=retry_policy
    )
)


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


if HGNC_PATH.exists():

    print("\nExisting HGNC file found.")

    print("Reusing existing file.")

    download_status = "Reused"

else:

    print("\nDownloading HGNC complete dataset...")

    temporary = Path(
        str(HGNC_PATH) + ".part"
    )

    with session.get(
        HGNC_URL,
        stream=True,
        timeout=(30, 120)
    ) as response:

        response.raise_for_status()

        total_bytes = 0

        with open(
            temporary,
            "wb"
        ) as output:

            for chunk in response.iter_content(
                chunk_size=1024 * 1024
            ):

                if not chunk:
                    continue

                output.write(chunk)

                total_bytes += len(chunk)

                if total_bytes > 50_000_000:

                    raise RuntimeError(
                        "HGNC download exceeded "
                        "the configured safety limit."
                    )

    os.replace(
        temporary,
        HGNC_PATH
    )

    download_status = "Downloaded"


hgnc_checksum = sha256_file(
    HGNC_PATH
)


print("\nHGNC acquisition:", download_status)

print(
    "Source size (MiB):",
    round(
        HGNC_PATH.stat().st_size / 1024**2,
        2
    )
)

print("SHA256:", hgnc_checksum)


# ------------------------------------------------------------
# 5. LOAD HGNC GENE ANNOTATIONS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING HGNC GENE ANNOTATIONS")
print("-" * 65)


hgnc = pd.read_csv(
    HGNC_PATH,
    sep="\t",
    dtype=str,
    keep_default_na=False,
    low_memory=False
)


required_hgnc_columns = {
    "hgnc_id",
    "symbol",
    "status",
    "ensembl_gene_id",
    "entrez_id"
}


missing_columns = (
    required_hgnc_columns
    - set(hgnc.columns)
)


if missing_columns:

    raise RuntimeError(
        "HGNC schema missing columns: "
        f"{missing_columns}"
    )


print("\nHGNC records:", len(hgnc))

print("HGNC columns:", len(hgnc.columns))


if hgnc["hgnc_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate HGNC identifiers."
    )


# Preserve original values while
# normalizing identifier fields.


for column in [
    "hgnc_id",
    "symbol",
    "status",
    "ensembl_gene_id",
    "entrez_id"
]:

    hgnc[column] = (
        hgnc[column]
        .astype(str)
        .str.strip()
    )


print("\nHGNC status distribution:")

print(
    hgnc["status"]
    .value_counts()
    .head(10)
    .to_string()
)


# ------------------------------------------------------------
# 6. PREPARE APPROVED ENSEMBL-TO-ENTREZ CROSSWALK
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING HGNC IDENTIFIER CROSSWALK")
print("-" * 65)


approved = hgnc[
    hgnc["status"].eq("Approved")
].copy()


approved = approved[
    approved["ensembl_gene_id"].ne("")
].copy()


approved["ensembl_gene_id"] = (
    approved["ensembl_gene_id"]
    .str.replace(
        r"\.\d+$",
        "",
        regex=True
    )
)


hgnc_by_ensembl = {

    gene_id: group.to_dict(
        orient="records"
    )

    for gene_id, group in approved.groupby(
        "ensembl_gene_id"
    )

}


print(
    "\nApproved HGNC records with Ensembl IDs:",
    len(approved)
)

print(
    "Unique Ensembl identifiers:",
    len(hgnc_by_ensembl)
)


# ------------------------------------------------------------
# 7. PARSE DEPMAP GENE IDENTIFIERS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PARSING DEPMAP GENE IDENTIFIERS")
print("-" * 65)


crispr_header = pd.read_csv(
    CRISPR_PATH,
    nrows=0
).columns.tolist()


if len(crispr_header) != 18435:

    raise RuntimeError(
        "Unexpected CRISPR gene header count."
    )


GENE_PATTERN = re.compile(
    r"^(.*?) \((\d+)\)$"
)


depmap_records = []

for column in crispr_header[1:]:

    match = GENE_PATTERN.fullmatch(
        column
    )

    if not match:

        raise RuntimeError(
            "Unexpected DepMap gene header: "
            f"{column}"
        )

    depmap_records.append({

        "depmap_column": column,

        "depmap_symbol":
            match.group(1),

        "depmap_entrez_id":
            match.group(2)
    })


depmap_genes = pd.DataFrame(
    depmap_records
)


depmap_by_entrez = {

    gene_id: group.to_dict(
        orient="records"
    )

    for gene_id, group in depmap_genes.groupby(
        "depmap_entrez_id"
    )

}


print("\nDepMap gene columns:", len(depmap_genes))

print(
    "Distinct NCBI Gene IDs:",
    len(depmap_by_entrez)
)


duplicate_entrez = (
    depmap_genes["depmap_entrez_id"]
    .duplicated()
    .sum()
)


print(
    "Repeated NCBI Gene IDs:",
    int(duplicate_entrez)
)


# ------------------------------------------------------------
# 8. CROSS-VALIDATE ALL 948 CANDIDATES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CROSS-VALIDATING ENSEMBL AND ENTREZ IDS")
print("-" * 65)


records = []


for _, row in mapping.iterrows():

    ensembl_id = row[
        "ensembl_stable_id"
    ]

    gene_name = row[
        "gene_name"
    ]

    prior_column = row[
        "depmap_column"
    ]

    prior_entrez = row[
        "depmap_entrez_id"
    ]

    prior_matched = (
        row["depmap_mapping_status"]
        == "Unique exact gene-symbol match"
    )


    hgnc_matches = hgnc_by_ensembl.get(
        ensembl_id,
        []
    )


    hgnc_symbol = ""

    hgnc_id = ""

    hgnc_entrez = ""

    proposed_column = ""

    proposed_depmap_symbol = ""

    status = ""


    # ----------------------------------------
    # No unambiguous HGNC Ensembl mapping
    # ----------------------------------------

    if len(hgnc_matches) == 0:

        status = (
            "No approved HGNC Ensembl mapping"
        )


    elif len(hgnc_matches) > 1:

        status = (
            "Ambiguous HGNC Ensembl mapping"
        )


    else:

        record = hgnc_matches[0]

        hgnc_symbol = record["symbol"]

        hgnc_id = record["hgnc_id"]

        hgnc_entrez = record[
            "entrez_id"
        ]


        # ------------------------------------
        # HGNC record lacks NCBI Gene ID
        # ------------------------------------

        if not hgnc_entrez:

            status = (
                "HGNC record lacks NCBI Gene ID"
            )


        else:

            depmap_matches = (
                depmap_by_entrez.get(
                    hgnc_entrez,
                    []
                )
            )


            # --------------------------------
            # No DepMap gene with this ID
            # --------------------------------

            if len(depmap_matches) == 0:

                status = (
                    "HGNC NCBI Gene ID absent "
                    "from DepMap"
                )


            # --------------------------------
            # Multiple DepMap gene columns
            # --------------------------------

            elif len(depmap_matches) > 1:

                status = (
                    "Ambiguous DepMap NCBI Gene ID"
                )


            else:

                depmap_match = depmap_matches[0]

                proposed_column = (
                    depmap_match["depmap_column"]
                )

                proposed_depmap_symbol = (
                    depmap_match["depmap_symbol"]
                )


                # ----------------------------
                # Existing symbol match agrees
                # with NCBI identifier
                # ----------------------------

                if (
                    prior_matched
                    and prior_entrez == hgnc_entrez
                    and prior_column == proposed_column
                ):

                    status = (
                        "Verified existing mapping"
                    )


                # ----------------------------
                # A previously unmatched gene
                # can now be located using ID
                # ----------------------------

                elif not prior_matched:

                    status = (
                        "Newly resolved by NCBI Gene ID"
                    )


                # ----------------------------
                # Symbol match and NCBI IDs
                # disagree: manual review
                # ----------------------------

                else:

                    status = (
                        "CONFLICT - identifier disagreement"
                    )


    records.append({

        "gene_id":
            row["gene_id"],

        "ensembl_stable_id":
            ensembl_id,

        "gene_name":
            gene_name,

        "previous_mapping_status":
            row["depmap_mapping_status"],

        "previous_depmap_column":
            prior_column,

        "previous_depmap_entrez_id":
            prior_entrez,

        "hgnc_id":
            hgnc_id,

        "hgnc_approved_symbol":
            hgnc_symbol,

        "hgnc_entrez_id":
            hgnc_entrez,

        "proposed_depmap_column":
            proposed_column,

        "proposed_depmap_symbol":
            proposed_depmap_symbol,

        "identity_validation_status":
            status,

        "symbol_changed":
            bool(
                hgnc_symbol
                and hgnc_symbol != gene_name
            )
    })


crosswalk = pd.DataFrame(
    records
)


if len(crosswalk) != 948:

    raise RuntimeError(
        "Gene count changed during crosswalk."
    )


if crosswalk["gene_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate candidate records "
        "after identifier validation."
    )


print("\nCross-validation status:")

print(
    crosswalk[
        "identity_validation_status"
    ]
    .value_counts()
    .to_string()
)


# ------------------------------------------------------------
# 9. SEPARATE VERIFIED AND UNRESOLVED MAPPINGS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("REVIEWING MAPPING RESOLUTION")
print("-" * 65)


verified = crosswalk[
    crosswalk["identity_validation_status"]
    .eq("Verified existing mapping")
].copy()


newly_resolved = crosswalk[
    crosswalk["identity_validation_status"]
    .eq("Newly resolved by NCBI Gene ID")
].copy()


conflicts = crosswalk[
    crosswalk["identity_validation_status"]
    .str.contains(
        "CONFLICT",
        na=False
    )
].copy()


unresolved = crosswalk[
    ~crosswalk["identity_validation_status"]
    .isin([
        "Verified existing mapping",
        "Newly resolved by NCBI Gene ID"
    ])
].copy()


print("\nVerified existing mappings:", len(verified))

print(
    "Newly resolved mappings:",
    len(newly_resolved)
)

print(
    "Identifier disagreements:",
    len(conflicts)
)

print(
    "Remaining unresolved or "
    "unverified mappings:",
    len(unresolved)
)


# ------------------------------------------------------------
# 10. INVESTIGATE THE ORIGINAL 18 UNMATCHED GENES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("ORIGINAL UNMATCHED GENE REVIEW")
print("-" * 65)


original_unmatched = crosswalk[
    ~crosswalk["previous_mapping_status"]
    .eq("Unique exact gene-symbol match")
].copy()


if len(original_unmatched) != 18:

    raise RuntimeError(
        "Unexpected number of previously "
        "unmatched genes."
    )


print("\nOriginal unmatched genes:", len(original_unmatched))


print("\n")

print(
    original_unmatched[
        [
            "gene_name",
            "hgnc_approved_symbol",
            "hgnc_entrez_id",
            "proposed_depmap_symbol",
            "identity_validation_status"
        ]
    ].to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 11. REVIEW IDENTIFIER DISAGREEMENTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("IDENTIFIER CONFLICT REVIEW")
print("-" * 65)


if conflicts.empty:

    print(
        "\nNo disagreements detected "
        "between validated existing mappings "
        "and the available HGNC crosswalk."
    )

else:

    print("\n")

    print(
        conflicts[
            [
                "gene_name",
                "previous_depmap_entrez_id",
                "hgnc_entrez_id",
                "previous_depmap_column",
                "proposed_depmap_column"
            ]
        ].to_string(
            index=False
        )
    )


print("\nIMPORTANT:")

print(
    "Conflicting mappings are not "
    "automatically replaced."
)

print(
    "Candidates without an HGNC match "
    "remain unverified, not absent."
)


# ------------------------------------------------------------
# 12. REVIEW LEADING CANDIDATES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LEADING CANDIDATE IDENTITY AUDIT")
print("-" * 65)


leading = crosswalk[
    crosswalk["gene_name"]
    .isin(LEADING_GENES)
].copy()


leading["display_order"] = (
    leading["gene_name"]
    .map({
        gene: index
        for index, gene in enumerate(
            LEADING_GENES
        )
    })
)


leading = leading.sort_values(
    "display_order"
)


print("\n")

print(
    leading[
        [
            "gene_name",
            "hgnc_approved_symbol",
            "hgnc_entrez_id",
            "proposed_depmap_symbol",
            "identity_validation_status"
        ]
    ].to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 13. SAVE IDENTIFIER VALIDATION TABLES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING IDENTIFIER VALIDATION RESULTS")
print("-" * 65)


crosswalk.to_csv(

    RESULTS /
    "depmap_26q3_hgnc_crosswalk_audit.csv",

    index=False
)


verified.to_csv(

    RESULTS /
    "depmap_26q3_verified_gene_mapping.csv",

    index=False
)


newly_resolved.to_csv(

    RESULTS /
    "depmap_26q3_newly_resolved_genes.csv",

    index=False
)


conflicts.to_csv(

    RESULTS /
    "depmap_26q3_identifier_conflicts.csv",

    index=False
)


unresolved.to_csv(

    RESULTS /
    "depmap_26q3_unverified_gene_mapping.csv",

    index=False
)


original_unmatched.to_csv(

    RESULTS /
    "depmap_26q3_original_unmatched_review.csv",

    index=False
)


leading.to_csv(

    RESULTS /
    "depmap_26q3_leading_gene_identity_audit.csv",

    index=False
)


print("\nAll identifier audit tables saved.")


# ------------------------------------------------------------
# 14. SAVE RESEARCH PROVENANCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING CROSSWALK PROVENANCE")
print("-" * 65)


provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "4.4",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "reference_source": (
        "HUGO Gene Nomenclature Committee"
    ),

    "reference_url": HGNC_URL,

    "reference_sha256": hgnc_checksum,

    "reference_local_path": str(HGNC_PATH),

    "reference_download_status": download_status,

    "reference_version_policy": (
        "HGNC complete dataset as retrieved "
        "at the recorded acquisition time"
    ),

    "depmap_release": "Public 26Q3",

    "candidate_count": int(
        len(crosswalk)
    ),

    "original_symbol_matched_genes": 930,

    "verified_existing_mappings": int(
        len(verified)
    ),

    "newly_resolved_mappings": int(
        len(newly_resolved)
    ),

    "identifier_conflicts": int(
        len(conflicts)
    ),

    "remaining_unverified_mappings": int(
        len(unresolved)
    ),

    "crosswalk_method": (
        "Ensembl stable gene ID to HGNC "
        "approved record to NCBI Gene ID "
        "to DepMap gene column"
    ),

    "ambiguous_mappings_automatically_selected":
        False,

    "original_gene_mapping_overwritten":
        False,

    "dependency_statistics_recalculated":
        False,

    "therapeutic_target_ranking_performed":
        False,

    "limitations": [

        "HGNC mappings reflect the database "
        "snapshot available at acquisition.",

        "Genes without HGNC NCBI IDs remain "
        "unverified rather than classified absent.",

        "Gene nomenclature can change between "
        "TCGA, HGNC, and DepMap releases.",

        "A verified identifier does not guarantee "
        "complete CRISPR measurement coverage.",

        "Disputed mappings require manual review.",

        "DepMap CRISPR gene effects are not "
        "direct measurements of antibody efficacy.",

        "No MSS status has been inferred.",

        "No therapeutic targets have been validated."

    ]
}


with open(

    META /
    "depmap_26q3_hgnc_crosswalk_provenance.json",

    "w"

) as handle:

    json.dump(
        provenance,
        handle,
        indent=4
    )


print("\nCrosswalk provenance saved.")


# ------------------------------------------------------------
# 15. CREATE PHASE 4 CHECKPOINT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING PHASE 4 RESEARCH CHECKPOINT")
print("-" * 65)


timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S"
)


checkpoint = (

    CHECKPOINTS /

    f"SurfaceTarget_CRC_Phase4_Patch44_{timestamp}.zip"

)


# Preserve processed research outputs and
# metadata without duplicating the large
# raw DepMap and TCGA molecular datasets.
#
# The original DepMap CSV should also
# remain backed up on the user's computer.


backup_folders = [
    "data/metadata",
    "data/processed",
    "data/raw/cptac_colon",
    "data/raw/hpa",
    "data/raw/uniprot",
    "data/raw/hgnc",
    "results/tables",
    "results/figures",
    "src",
    "reports",
    "logs"
]


with ZipFile(
    checkpoint,
    "w",
    ZIP_DEFLATED,
    compresslevel=3
) as archive:

    for folder in backup_folders:

        folder_path = ROOT / folder

        if not folder_path.exists():
            continue

        for file_path in folder_path.rglob("*"):

            if file_path.is_file():

                archive.write(

                    file_path,

                    arcname=str(
                        file_path.relative_to(ROOT)
                    )

                )


checkpoint_mib = (
    checkpoint.stat().st_size
    / (1024 ** 2)
)


print("\nCheckpoint created successfully.")

print(
    "Checkpoint size:",
    f"{checkpoint_mib:.2f} MiB"
)

print("Checkpoint location:")
print(checkpoint)


# ------------------------------------------------------------
# 16. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 4.4 SUMMARY")
print("=" * 65)


print("\nCandidate genes:", len(crosswalk))


print(
    "Original exact-symbol matches:",
    int(originally_matched.sum())
)


print(
    "Verified existing mappings:",
    len(verified)
)


print(
    "Newly resolved mappings:",
    len(newly_resolved)
)


print(
    "Identifier disagreements:",
    len(conflicts)
)


print(
    "Unresolved or unverified mappings:",
    len(unresolved)
)


print("\nSource: HGNC complete gene dataset")

print("HGNC source checksum: SAVED")

print("Crosswalk audit: SAVED")

print("Leading-gene identity review: SAVED")

print("Original 18-gene review: SAVED")

print("Research provenance: SAVED")

print("\nPhase 4 checkpoint: CREATED")

print(
    "Checkpoint size:",
    f"{checkpoint_mib:.2f} MiB"
)


print("\nNo dependency statistics recalculated.")

print("No existing gene mappings overwritten.")

print("No candidate genes excluded.")

print("No therapeutic target scores assigned.")


print("\nPATCH 4.4 COMPLETE")

print("=" * 65)

In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 4 — PATCH 4.5
#
# FINAL HGNC-VALIDATED DEPMAP DEPENDENCY INTEGRATION
#
# Objectives:
# 1. Validate previously saved analysis inputs
# 2. Confirm DepMap source checksums
# 3. Import 13 newly resolved HGNC gene mappings
# 4. Extract the corresponding CRISPR measurements
# 5. Calculate corrected CRC dependency statistics
# 6. Calculate patient-deduplicated sensitivity statistics
# 7. Compare with the established other-lineage cohort
# 8. Preserve all 930 original dependency results
# 9. Retain 5 genes without DepMap measurements
# 10. Save final Phase 4 tables and provenance
#
# IMPORTANT:
# - No original results overwritten
# - No gene symbols guessed
# - No MSS status inferred
# - No automatic candidate exclusions
# - No therapeutic target ranking
# ============================================================


# ------------------------------------------------------------
# 1. IMPORTS AND CONFIGURATION
# ------------------------------------------------------------

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np

import hashlib
import json
import math


ROOT = Path("/content/SurfaceTarget-CRC")

DEPMAP = ROOT / "data/raw/depmap/26Q3"

META = ROOT / "data/metadata"

RESULTS = ROOT / "results/tables"

RESULTS.mkdir(parents=True, exist_ok=True)
META.mkdir(parents=True, exist_ok=True)


CRISPR_PATH = DEPMAP / "CRISPRGeneEffect.csv"

MODEL_PATH = DEPMAP / "Model.csv"

IMPORT_PATH = (
    META /
    "depmap_26q3_import_verification.csv"
)

CROSSWALK_PATH = (
    RESULTS /
    "depmap_26q3_hgnc_crosswalk_audit.csv"
)

PREVIOUS_SUMMARY_PATH = (
    RESULTS /
    "depmap_26q3_refined_dependency_summary.csv"
)

CRC_MODELS_PATH = (
    RESULTS /
    "depmap_26q3_refined_crc_models.csv"
)

DEDUP_MODELS_PATH = (
    RESULTS /
    "depmap_26q3_patient_deduplicated_models.csv"
)


print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 4 — PATCH 4.5")
print("FINAL DEPMAP DEPENDENCY INTEGRATION")
print("=" * 65)


# ------------------------------------------------------------
# 2. VALIDATE REQUIRED FILES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING PHASE 4 RESEARCH INPUTS")
print("-" * 65)


required_paths = [
    CRISPR_PATH,
    MODEL_PATH,
    IMPORT_PATH,
    CROSSWALK_PATH,
    PREVIOUS_SUMMARY_PATH,
    CRC_MODELS_PATH,
    DEDUP_MODELS_PATH
]


for path in required_paths:

    if not path.exists():

        raise FileNotFoundError(
            f"Required file missing: {path}"
        )


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):

            digest.update(chunk)

    return digest.hexdigest()


import_audit = pd.read_csv(
    IMPORT_PATH,
    dtype=str,
    keep_default_na=False
)


if import_audit["filename"].duplicated().any():

    raise RuntimeError(
        "Duplicate source files in import audit."
    )


import_audit = import_audit.set_index(
    "filename"
)


for path in [MODEL_PATH, CRISPR_PATH]:

    if path.name not in import_audit.index:

        raise RuntimeError(
            f"Missing source checksum: {path.name}"
        )

    expected_hash = import_audit.loc[
        path.name,
        "sha256"
    ]

    actual_hash = sha256_file(path)

    if actual_hash != expected_hash:

        raise RuntimeError(
            f"Source checksum mismatch: {path.name}"
        )


print("\nLocally recorded source checksums: PASSED")

print(
    "Official DepMap source checksum "
    "comparison remains unavailable."
)


# ------------------------------------------------------------
# 3. LOAD HGNC-VALIDATED GENE MAPPINGS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING HGNC-VALIDATED CANDIDATE MAPPINGS")
print("-" * 65)


crosswalk = pd.read_csv(
    CROSSWALK_PATH,
    dtype=str,
    keep_default_na=False
)


required_columns = {
    "gene_id",
    "ensembl_stable_id",
    "gene_name",
    "previous_depmap_column",
    "hgnc_entrez_id",
    "proposed_depmap_column",
    "proposed_depmap_symbol",
    "identity_validation_status"
}


if not required_columns.issubset(
    crosswalk.columns
):

    raise RuntimeError(
        "Unexpected HGNC crosswalk schema."
    )


if len(crosswalk) != 948:

    raise RuntimeError(
        "Expected 948 research genes."
    )


if crosswalk["gene_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate research gene IDs."
    )


status_counts = (
    crosswalk["identity_validation_status"]
    .value_counts()
)


print("\nIdentifier validation distribution:")

print(status_counts.to_string())


verified = crosswalk[
    crosswalk["identity_validation_status"]
    .eq("Verified existing mapping")
].copy()


newly_resolved = crosswalk[
    crosswalk["identity_validation_status"]
    .eq("Newly resolved by NCBI Gene ID")
].copy()


unavailable = crosswalk[
    crosswalk["identity_validation_status"]
    .eq("HGNC NCBI Gene ID absent from DepMap")
].copy()


if len(verified) != 930:

    raise RuntimeError(
        "Expected 930 verified original mappings."
    )


if len(newly_resolved) != 13:

    raise RuntimeError(
        "Expected 13 newly resolved mappings."
    )


if len(unavailable) != 5:

    raise RuntimeError(
        "Expected five genes absent by HGNC "
        "NCBI Gene ID from DepMap."
    )


if 930 + 13 + 5 != len(crosswalk):

    raise RuntimeError(
        "Unexpected identifier-validation categories."
    )


print("\nVerified existing genes:", len(verified))

print("Newly resolved genes:", len(newly_resolved))

print("Genes without DepMap columns:", len(unavailable))


# ------------------------------------------------------------
# 4. VERIFY COMPLETE GENE MAPPING UNIQUENESS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING FINAL GENE COLUMN CORRESPONDENCE")
print("-" * 65)


mapped = crosswalk[
    crosswalk["identity_validation_status"]
    .isin([
        "Verified existing mapping",
        "Newly resolved by NCBI Gene ID"
    ])
].copy()


if mapped[
    "proposed_depmap_column"
].eq("").any():

    raise RuntimeError(
        "Mapped genes contain empty DepMap columns."
    )


if mapped[
    "proposed_depmap_column"
].duplicated().any():

    raise RuntimeError(
        "Multiple research genes map to "
        "the same DepMap gene column."
    )


if mapped["hgnc_entrez_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate NCBI Gene IDs among mapped genes."
    )


if len(mapped) != 943:

    raise RuntimeError(
        "Expected 943 uniquely mapped genes."
    )


# Verify previously established mappings
# were not changed by HGNC integration.

if not (
    verified["previous_depmap_column"]
    == verified["proposed_depmap_column"]
).all():

    raise RuntimeError(
        "Previously verified gene columns changed."
    )


print("\nFinal verified DepMap genes:", len(mapped))

print("Existing mappings unchanged: PASSED")

print("Gene-column uniqueness: PASSED")


# ------------------------------------------------------------
# 5. VERIFY THE 13 NEW DEPMAP COLUMNS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING NEWLY RESOLVED CRISPR COLUMNS")
print("-" * 65)


crispr_header = pd.read_csv(
    CRISPR_PATH,
    nrows=0
).columns.tolist()


if len(crispr_header) != 18435:

    raise RuntimeError(
        "Unexpected CRISPR gene-column count."
    )


model_column = crispr_header[0]


if model_column not in [
    "Unnamed: 0",
    "ModelID"
]:

    raise RuntimeError(
        "Unexpected CRISPR model identifier column."
    )


new_columns = (
    newly_resolved["proposed_depmap_column"]
    .tolist()
)


if len(set(new_columns)) != 13:

    raise RuntimeError(
        "New gene mappings are not unique."
    )


missing_columns = (
    set(new_columns) - set(crispr_header)
)


if missing_columns:

    raise RuntimeError(
        "New gene columns missing from DepMap: "
        f"{sorted(missing_columns)}"
    )


print("\nNewly resolved gene columns:", len(new_columns))

print("CRISPR header correspondence: PASSED")


# ------------------------------------------------------------
# 6. LOAD ONLY THE NEW CRISPR MEASUREMENTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("EXTRACTING 13 NEW GENE-EFFECT COLUMNS")
print("-" * 65)


effects = pd.read_csv(
    CRISPR_PATH,
    usecols=[model_column] + new_columns,
    dtype={model_column: str},
    low_memory=False
)


effects = effects.rename(
    columns={
        model_column: "ModelID"
    }
)


effects["ModelID"] = (
    effects["ModelID"]
    .astype(str)
    .str.strip()
)


if effects["ModelID"].duplicated().any():

    raise RuntimeError(
        "Duplicate CRISPR model identifiers."
    )


effects = effects.set_index(
    "ModelID"
)


effects = effects.apply(
    pd.to_numeric,
    errors="raise"
)


if effects.shape != (1361, 13):

    raise RuntimeError(
        f"Unexpected extracted matrix: {effects.shape}"
    )


if np.isinf(
    effects.to_numpy(dtype=float)
).any():

    raise RuntimeError(
        "Infinite gene-effect values detected."
    )


print("\nExtracted matrix:", effects.shape)

print("Numerical values: VALIDATED")


# ------------------------------------------------------------
# 7. LOAD COLORECTAL AND DEDUPLICATED MODELS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("RESTORING VALIDATED COLORECTAL COHORTS")
print("-" * 65)


crc_models = pd.read_csv(
    CRC_MODELS_PATH,
    dtype=str,
    keep_default_na=False
)


dedup_models = pd.read_csv(
    DEDUP_MODELS_PATH,
    dtype=str,
    keep_default_na=False
)


if (
    "ModelID" not in crc_models.columns
    or "ModelID" not in dedup_models.columns
):

    raise RuntimeError(
        "Model identifier column missing."
    )


crc_ids = crc_models["ModelID"].tolist()

dedup_ids = dedup_models["ModelID"].tolist()


if len(crc_ids) != 81:

    raise RuntimeError(
        "Expected 81 CRC adenocarcinoma models."
    )


if len(dedup_ids) != 79:

    raise RuntimeError(
        "Expected 79 patient-deduplicated models."
    )


if len(set(crc_ids)) != len(crc_ids):

    raise RuntimeError(
        "Duplicate CRC model IDs."
    )


if len(set(dedup_ids)) != len(dedup_ids):

    raise RuntimeError(
        "Duplicate sensitivity model IDs."
    )


if not set(dedup_ids).issubset(
    set(crc_ids)
):

    raise RuntimeError(
        "Deduplicated cohort is not a subset "
        "of the primary CRC cohort."
    )


if not set(crc_ids).issubset(
    set(effects.index)
):

    raise RuntimeError(
        "CRC models missing from CRISPR measurements."
    )


print("\nPrimary CRC models:", len(crc_ids))

print("Patient-deduplicated models:", len(dedup_ids))


# ------------------------------------------------------------
# 8. RECONSTRUCT THE SAME COMPARATOR AS PATCH 4.3
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("RESTORING OTHER-LINEAGE COMPARATOR")
print("-" * 65)


models = pd.read_csv(
    MODEL_PATH,
    dtype=str,
    keep_default_na=False
)


if models["ModelID"].duplicated().any():

    raise RuntimeError(
        "Duplicate ModelIDs in metadata."
    )


models = models.set_index("ModelID")


if not set(effects.index).issubset(
    set(models.index)
):

    raise RuntimeError(
        "CRISPR models missing from metadata."
    )


screened = models.loc[
    effects.index
].copy()


crc_mask = (
    screened["OncotreePrimaryDisease"]
    .str.strip()
    .eq("Colorectal Adenocarcinoma")
)


bowel_mask = (
    screened["OncotreeLineage"]
    .str.contains(
        r"bowel",
        case=False,
        regex=True,
        na=False
    )
)


nonmalignant_mask = (
    screened["OncotreePrimaryDisease"]
    .str.contains(
        r"non[- ]?cancerous|"
        r"benign|adenoma|"
        r"normal tissue|"
        r"hyperplasia|"
        r"dysplasia|"
        r"sessile serrated",
        case=False,
        regex=True,
        na=False
    )
)


other_mask = (
    ~bowel_mask
    & ~nonmalignant_mask
    & ~crc_mask
)


other_ids = (
    screened.index[other_mask]
    .tolist()
)


if len(other_ids) != 1260:

    raise RuntimeError(
        "Other-lineage comparator differs "
        "from Patch 4.3."
    )


if set(crc_ids) != set(
    screened.index[crc_mask]
):

    raise RuntimeError(
        "Primary CRC model definition "
        "differs from Patch 4.3."
    )


print("\nOther-lineage models:", len(other_ids))

print("Comparator validation: PASSED")


# ------------------------------------------------------------
# 9. BUILD ANALYSIS MATRICES
# ------------------------------------------------------------

crc_effects = effects.loc[
    crc_ids
].copy()


dedup_effects = effects.loc[
    dedup_ids
].copy()


other_effects = effects.loc[
    other_ids
].copy()


print("\nNew-gene matrix dimensions:")

print("CRC:", crc_effects.shape)

print("Deduplicated CRC:", dedup_effects.shape)

print("Other lineages:", other_effects.shape)


# ------------------------------------------------------------
# 10. GENE-EFFECT SUMMARY FUNCTION
# ------------------------------------------------------------

def summarize_effect(values):

    values = np.asarray(
        values,
        dtype=float
    )

    values = values[
        np.isfinite(values)
    ]

    n = len(values)

    if n == 0:

        return {
            "n": 0,
            "median": np.nan,
            "q25": np.nan,
            "q75": np.nan,
            "fraction_lt_05": np.nan,
            "fraction_lt_1": np.nan
        }

    return {

        "n": int(n),

        "median": float(
            np.median(values)
        ),

        "q25": float(
            np.percentile(values, 25)
        ),

        "q75": float(
            np.percentile(values, 75)
        ),

        "fraction_lt_05": float(
            np.mean(values < -0.5)
        ),

        "fraction_lt_1": float(
            np.mean(values < -1.0)
        )
    }


# ------------------------------------------------------------
# 11. CALCULATE NEW GENE DEPENDENCIES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CALCULATING NEWLY RESOLVED GENE DEPENDENCIES")
print("-" * 65)


new_records = []


for _, row in newly_resolved.iterrows():

    column = row[
        "proposed_depmap_column"
    ]

    crc_stats = summarize_effect(
        crc_effects[column]
    )

    other_stats = summarize_effect(
        other_effects[column]
    )

    dedup_stats = summarize_effect(
        dedup_effects[column]
    )


    crc_med = crc_stats["median"]

    other_med = other_stats["median"]

    dedup_med = dedup_stats["median"]


    if (
        np.isfinite(crc_med)
        and np.isfinite(other_med)
    ):

        median_difference = (
            crc_med - other_med
        )

    else:

        median_difference = np.nan


    if (
        np.isfinite(crc_med)
        and np.isfinite(dedup_med)
    ):

        dedup_difference = (
            dedup_med - crc_med
        )

    else:

        dedup_difference = np.nan


    new_records.append({

        "gene_id":
            row["gene_id"],

        "ensembl_stable_id":
            row["ensembl_stable_id"],

        "gene_name":
            row["gene_name"],

        "hgnc_approved_symbol":
            row["hgnc_approved_symbol"],

        "depmap_current_symbol":
            row["proposed_depmap_symbol"],

        "depmap_column":
            column,

        "depmap_entrez_id":
            row["hgnc_entrez_id"],

        "crc_models_total":
            len(crc_ids),

        "crc_models_measured":
            crc_stats["n"],

        "crc_measurement_fraction":
            crc_stats["n"] / len(crc_ids),

        "crc_median_gene_effect":
            crc_med,

        "crc_q25_gene_effect":
            crc_stats["q25"],

        "crc_q75_gene_effect":
            crc_stats["q75"],

        "crc_fraction_below_minus_05":
            crc_stats["fraction_lt_05"],

        "crc_fraction_below_minus_1":
            crc_stats["fraction_lt_1"],

        "other_models_measured":
            other_stats["n"],

        "other_median_gene_effect":
            other_med,

        "other_fraction_below_minus_05":
            other_stats["fraction_lt_05"],

        "crc_minus_other_median_effect":
            median_difference,

        "deduplicated_models_measured":
            dedup_stats["n"],

        "deduplicated_median_gene_effect":
            dedup_med,

        "dedup_minus_primary_median":
            dedup_difference,

        "high_crc_coverage":
            crc_stats["n"] >= 65

    })


new_results = pd.DataFrame(
    new_records
)


if len(new_results) != 13:

    raise RuntimeError(
        "Expected 13 newly calculated gene records."
    )


print("\nNew gene dependency calculations: COMPLETE")


# ------------------------------------------------------------
# 12. DISPLAY NEWLY RESOLVED GENES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("NEW GENE FUNCTIONAL RESULTS")
print("-" * 65)


display_columns = [
    "gene_name",
    "depmap_current_symbol",
    "crc_models_measured",
    "crc_median_gene_effect",
    "crc_fraction_below_minus_05",
    "other_median_gene_effect",
    "deduplicated_median_gene_effect"
]


print("\n")

print(
    new_results[
        display_columns
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ------------------------------------------------------------
# 13. LOAD PREVIOUS REFINED DEPENDENCY SUMMARY
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PRESERVING PREVIOUS CRISPR RESULTS")
print("-" * 65)


previous = pd.read_csv(
    PREVIOUS_SUMMARY_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)


if len(previous) != 948:

    raise RuntimeError(
        "Unexpected previous summary size."
    )


if previous["gene_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate gene IDs in previous summary."
    )


previous = previous.set_index(
    "gene_id",
    drop=False
)


# Preserve the original summary.
# All modifications below occur in a copy.

final = previous.copy(deep=True)


# ------------------------------------------------------------
# 14. ATTACH VALIDATED IDENTITY METADATA
# ------------------------------------------------------------

crosswalk_indexed = crosswalk.set_index(
    "gene_id"
)


if set(final.index) != set(
    crosswalk_indexed.index
):

    raise RuntimeError(
        "Gene IDs differ between crosswalk "
        "and dependency summary."
    )


if not (
    final["ensembl_stable_id"]
    == crosswalk_indexed.loc[
        final.index,
        "ensembl_stable_id"
    ]
).all():

    raise RuntimeError(
        "Ensembl identifiers differ "
        "between research tables."
    )


final["previous_depmap_mapping_status"] = (
    final["depmap_mapping_status"].copy()
)


final["hgnc_identity_status"] = (
    crosswalk_indexed.loc[
        final.index,
        "identity_validation_status"
    ]
)


final["hgnc_approved_symbol"] = (
    crosswalk_indexed.loc[
        final.index,
        "hgnc_approved_symbol"
    ]
)


final["hgnc_entrez_id"] = (
    crosswalk_indexed.loc[
        final.index,
        "hgnc_entrez_id"
    ]
)


final["depmap_current_symbol"] = (
    crosswalk_indexed.loc[
        final.index,
        "proposed_depmap_symbol"
    ]
)


final["depmap_final_mapping_available"] = (
    final["hgnc_identity_status"].isin([
        "Verified existing mapping",
        "Newly resolved by NCBI Gene ID"
    ])
)


# ------------------------------------------------------------
# 15. INTEGRATE THE 13 NEW DEPENDENCY RESULTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("INTEGRATING NEWLY RESOLVED DEPENDENCIES")
print("-" * 65)


statistic_columns = [
    "crc_models_total",
    "crc_models_measured",
    "crc_measurement_fraction",
    "crc_median_gene_effect",
    "crc_q25_gene_effect",
    "crc_q75_gene_effect",
    "crc_fraction_below_minus_05",
    "crc_fraction_below_minus_1",
    "other_models_measured",
    "other_median_gene_effect",
    "other_fraction_below_minus_05",
    "crc_minus_other_median_effect",
    "deduplicated_models_measured",
    "deduplicated_median_gene_effect",
    "dedup_minus_primary_median",
    "high_crc_coverage"
]


missing_statistic_columns = (
    set(statistic_columns) - set(final.columns)
)


if missing_statistic_columns:

    raise RuntimeError(
        "Previous summary missing statistics: "
        f"{missing_statistic_columns}"
    )


for _, row in new_results.iterrows():

    gene_id = row["gene_id"]

    if gene_id not in final.index:

        raise RuntimeError(
            f"New gene missing from summary: {gene_id}"
        )

    # Insert the verified DepMap identifier.

    final.at[
        gene_id,
        "depmap_column"
    ] = row["depmap_column"]


    final.at[
        gene_id,
        "depmap_entrez_id"
    ] = row["depmap_entrez_id"]


    final.at[
        gene_id,
        "depmap_mapping_status"
    ] = "HGNC NCBI Gene ID validated"


    # Insert the newly calculated statistics.

    for column in statistic_columns:

        final.at[
            gene_id,
            column
        ] = row[column]


print("\nNewly resolved genes integrated:", len(new_results))


# ------------------------------------------------------------
# 16. VALIDATE ORIGINAL 930 STATISTICS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VERIFYING PREVIOUS RESULTS WERE PRESERVED")
print("-" * 65)


verified_ids = (
    verified["gene_id"]
    .tolist()
)


for column in statistic_columns:

    if column == "high_crc_coverage":

        if not (
            final.loc[
                verified_ids,
                column
            ].astype(bool).to_numpy()

            ==

            previous.loc[
                verified_ids,
                column
            ].astype(bool).to_numpy()
        ).all():

            raise RuntimeError(
                f"Existing coverage flag changed: {column}"
            )

    else:

        old_values = pd.to_numeric(
            previous.loc[
                verified_ids,
                column
            ],
            errors="raise"
        ).to_numpy(dtype=float)


        new_values = pd.to_numeric(
            final.loc[
                verified_ids,
                column
            ],
            errors="raise"
        ).to_numpy(dtype=float)


        if not np.allclose(
            old_values,
            new_values,
            equal_nan=True,
            rtol=0.0,
            atol=0.0
        ):

            raise RuntimeError(
                "Previously calculated statistics changed: "
                f"{column}"
            )


print("\nOriginal 930 gene statistics: UNCHANGED")


# ------------------------------------------------------------
# 17. FINAL GENE COVERAGE VALIDATION
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("FINAL DEPMAP COVERAGE VALIDATION")
print("-" * 65)


if len(final) != 948:

    raise RuntimeError(
        "Final candidate count changed."
    )


mapping_available = int(
    final["depmap_final_mapping_available"].sum()
)


if mapping_available != 943:

    raise RuntimeError(
        "Expected 943 validated mappings."
    )


unmapped_final = final[
    ~final["depmap_final_mapping_available"]
].copy()


if len(unmapped_final) != 5:

    raise RuntimeError(
        "Expected five genes without "
        "DepMap gene-effect columns."
    )


# Genes without a DepMap column must not
# receive invented functional measurements.

if unmapped_final[
    "crc_models_measured"
].fillna(0).gt(0).any():

    raise RuntimeError(
        "Unmapped genes unexpectedly have "
        "CRISPR measurements."
    )


print("\nResearch candidate genes:", len(final))

print("Verified DepMap gene mappings:", mapping_available)

print("Without DepMap gene columns:", len(unmapped_final))


print(
    "Genes with CRC measurements:",
    int(
        final["crc_models_measured"]
        .fillna(0)
        .gt(0)
        .sum()
    )
)


print(
    "Genes meeting >=80% CRC coverage:",
    int(
        final["high_crc_coverage"]
        .fillna(False)
        .astype(bool)
        .sum()
    )
)


print("\nCandidate preservation: PASSED")

print("Gene identity validation: PASSED")


# ------------------------------------------------------------
# 18. REVIEW REMAINING UNMEASURED GENES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENES WITHOUT DEPMAP MEASUREMENTS")
print("-" * 65)


print("\n")

print(
    unmapped_final[
        [
            "gene_name",
            "hgnc_approved_symbol",
            "hgnc_entrez_id",
            "hgnc_identity_status"
        ]
    ].to_string(index=False)
)


print("\nIMPORTANT:")

print(
    "Absent DepMap gene-effect columns "
    "are not biological evidence of "
    "nonessentiality or lack of expression."
)


# ------------------------------------------------------------
# 19. PREPARE NEW-GENE CRC MATRIX
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING NEWLY RESOLVED CRC MATRIX")
print("-" * 65)


column_to_research_symbol = dict(
    zip(
        newly_resolved["proposed_depmap_column"],
        newly_resolved["gene_name"]
    )
)


crc_export = crc_effects.rename(
    columns=column_to_research_symbol
).copy()


crc_export.index.name = "ModelID"


if crc_export.shape != (81, 13):

    raise RuntimeError(
        "Unexpected new-gene CRC matrix."
    )


print("\nNewly resolved CRC matrix:", crc_export.shape)


# ------------------------------------------------------------
# 20. SAVE FINAL DEPMAP RESULTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING FINAL PHASE 4 RESULTS")
print("-" * 65)


final_output = final.reset_index(
    drop=True
)


final_output.to_csv(

    RESULTS /
    "depmap_26q3_hgnc_integrated_dependency_summary.csv",

    index=False
)


new_results.to_csv(

    RESULTS /
    "depmap_26q3_hgnc_resolved_gene_dependency.csv",

    index=False
)


unmapped_final.reset_index(
    drop=True
).to_csv(

    RESULTS /
    "depmap_26q3_remaining_unmeasured_candidates.csv",

    index=False
)


crc_export.to_csv(

    RESULTS /
    "depmap_26q3_hgnc_resolved_crc_matrix.csv"
)


# Separate, compact final mapping reference.

mapping_columns = [
    "gene_id",
    "ensembl_stable_id",
    "gene_name",
    "hgnc_approved_symbol",
    "hgnc_entrez_id",
    "depmap_current_symbol",
    "depmap_column",
    "depmap_final_mapping_available",
    "hgnc_identity_status"
]


final_output[
    mapping_columns
].to_csv(

    RESULTS /
    "depmap_26q3_final_candidate_gene_mapping.csv",

    index=False
)


print("\nAll final Phase 4 tables saved.")


# ------------------------------------------------------------
# 21. SAVE RESEARCH PROVENANCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING FINAL DEPMAP PROVENANCE")
print("-" * 65)


provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "4.5",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "depmap_release": "Public 26Q3",

    "source_model_sha256":
        sha256_file(MODEL_PATH),

    "source_crispr_sha256":
        sha256_file(CRISPR_PATH),

    "official_source_checksum_verified": False,

    "candidate_genes": 948,

    "previously_verified_genes": 930,

    "newly_resolved_genes": 13,

    "final_verified_gene_mappings": 943,

    "genes_without_depmap_columns": 5,

    "refined_crc_adenocarcinoma_models": 81,

    "patient_deduplicated_crc_models": 79,

    "other_lineage_comparator_models": 1260,

    "new_dependency_statistics_calculated_for": 13,

    "existing_dependency_statistics_preserved": True,

    "gene_mapping_method": (
        "TCGA Ensembl gene ID -> HGNC approved "
        "record -> NCBI Gene ID -> DepMap column"
    ),

    "reference_gene_naming": (
        "Original TCGA gene symbols retained; "
        "current HGNC/DepMap symbols recorded "
        "in separate columns"
    ),

    "unmapped_gene_policy": (
        "Preserve candidates with unavailable "
        "CRISPR values; do not assign zero effects"
    ),

    "primary_crc_cohort": (
        "81 models annotated as "
        "Colorectal Adenocarcinoma"
    ),

    "sensitivity_crc_cohort": (
        "79 models after deterministic "
        "patient-level deduplication"
    ),

    "dependency_interpretation": (
        "Descriptive CRISPR fitness effects; "
        "not direct antibody efficacy measures"
    ),

    "msi_status_inferred": False,

    "therapeutic_target_ranking_performed": False,

    "original_phase4_results_overwritten": False,

    "limitations": [

        "The working CRISPR CSV was reconstructed "
        "from two upload fragments.",

        "Official source checksum comparison "
        "has not been performed.",

        "The colorectal model cohort is "
        "not confirmed to be entirely MSS.",

        "Models may not capture primary tumor "
        "heterogeneity or stromal biology.",

        "Other-lineage comparisons are descriptive.",

        "Gene-effect measurements do not "
        "directly measure antibody binding.",

        "Weak dependency does not rule out "
        "ADC antigen suitability.",

        "Five candidate genes do not have "
        "gene-effect columns in this release.",

        "No therapeutic targets have "
        "been validated."

    ]
}


provenance_path = (
    META /
    "depmap_26q3_final_integration_provenance.json"
)


with open(
    provenance_path,
    "w"
) as handle:

    json.dump(
        provenance,
        handle,
        indent=4
    )


print("\nFinal Phase 4 provenance saved.")


# ------------------------------------------------------------
# 22. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 4.5 SUMMARY")
print("=" * 65)


print("\nDepMap release: Public 26Q3")


print(
    "Research candidates preserved:",
    len(final_output)
)


print(
    "Verified DepMap gene mappings:",
    mapping_available
)


print(
    "Newly recovered gene measurements:",
    len(new_results)
)


print(
    "Genes without DepMap columns:",
    len(unmapped_final)
)


print(
    "\nCRC adenocarcinoma models:",
    len(crc_ids)
)


print(
    "Patient-deduplicated models:",
    len(dedup_ids)
)


print(
    "Other-lineage models:",
    len(other_ids)
)


print("\nSaved files:")

print(
    "1. depmap_26q3_hgnc_integrated_dependency_summary.csv"
)

print(
    "2. depmap_26q3_hgnc_resolved_gene_dependency.csv"
)

print(
    "3. depmap_26q3_remaining_unmeasured_candidates.csv"
)

print(
    "4. depmap_26q3_hgnc_resolved_crc_matrix.csv"
)

print(
    "5. depmap_26q3_final_candidate_gene_mapping.csv"
)

print(
    "6. depmap_26q3_final_integration_provenance.json"
)


print("\nExisting 930 gene statistics: VERIFIED UNCHANGED")

print("Additional 13 genes: INTEGRATED")

print("All 948 candidates: PRESERVED")

print("No missing CRISPR values imputed.")

print("No source files modified.")

print("No therapeutic target scores assigned.")


print("\nPATCH 4.5 COMPLETE")

print("=" * 65)

In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 5 — PATCH 5.1
#
# CROSS-OMICS EVIDENCE INTEGRATION AND VALIDATION
#
# OBJECTIVES:
# 1. Load completed Phase 2-4 evidence
# 2. Validate all candidate identifiers
# 3. Preserve the original 948-gene research universe
# 4. Validate CPTAC support across previous analyses
# 5. Integrate HGNC-verified DepMap results
# 6. Audit evidence coverage and missingness
# 7. Preserve normal-tissue uncertainty
# 8. Review leading candidates, including TDGF1
# 9. Save canonical cross-omics evidence table
# 10. Save data dictionary and provenance
# 11. Create a new research checkpoint
#
# IMPORTANT:
# - No target ranking
# - No arbitrary scoring weights
# - No automatic candidate exclusions
# - Missing evidence is not negative evidence
# - No cross-study patient-level merging
# - No previous research results overwritten
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
import numpy as np

import hashlib
import json
import re


# ------------------------------------------------------------
# 1. PROJECT CONFIGURATION
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"
RESULTS = ROOT / "results/tables"
CHECKPOINTS = ROOT / "checkpoints"

for folder in [META, RESULTS, CHECKPOINTS]:
    folder.mkdir(parents=True, exist_ok=True)


PHASE3_PATH = (
    RESULTS /
    "surfacetarget_normal_tissue_integration.csv"
)

CPTAC_PATH = (
    RESULTS /
    "cptac_mss_extended_transcriptomic_integration.csv"
)

DEPMAP_PATH = (
    RESULTS /
    "depmap_26q3_hgnc_integrated_dependency_summary.csv"
)

CROSSWALK_PATH = (
    RESULTS /
    "depmap_26q3_hgnc_crosswalk_audit.csv"
)


print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 5 — PATCH 5.1")
print("CROSS-OMICS EVIDENCE INTEGRATION")
print("=" * 65)


# ------------------------------------------------------------
# 2. VALIDATE INPUT FILES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING RESEARCH INPUT FILES")
print("-" * 65)


required_files = {
    "Phase 3 integration": PHASE3_PATH,
    "CPTAC integration": CPTAC_PATH,
    "DepMap integration": DEPMAP_PATH,
    "HGNC crosswalk": CROSSWALK_PATH
}


for name, path in required_files.items():

    if not path.exists():

        raise FileNotFoundError(
            f"{name} missing: {path}"
        )

    print(
        f"{name}: FOUND"
    )


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


source_hashes = {

    name: sha256_file(path)

    for name, path in required_files.items()

}


print("\nSource files: VERIFIED")


# ------------------------------------------------------------
# 3. LOAD RESEARCH TABLES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING COMPLETED RESEARCH EVIDENCE")
print("-" * 65)


phase3 = pd.read_csv(
    PHASE3_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)


cptac = pd.read_csv(
    CPTAC_PATH,
    dtype={"gene_id": str},
    low_memory=False
)


depmap = pd.read_csv(
    DEPMAP_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)


crosswalk = pd.read_csv(
    CROSSWALK_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)


tables = {
    "Phase3": phase3,
    "CPTAC": cptac,
    "DepMap": depmap,
    "HGNC": crosswalk
}


for name, table in tables.items():

    print(
        f"{name}: {table.shape[0]} genes, "
        f"{table.shape[1]} columns"
    )


# ------------------------------------------------------------
# 4. VALIDATE GENE IDENTITIES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CROSS-SOURCE GENE IDENTIFIER VALIDATION")
print("-" * 65)


for name, table in tables.items():

    if len(table) != 948:

        raise RuntimeError(
            f"{name}: expected 948 genes."
        )

    if "gene_id" not in table.columns:

        raise RuntimeError(
            f"{name}: gene_id column missing."
        )

    if table["gene_id"].isna().any():

        raise RuntimeError(
            f"{name}: missing gene identifiers."
        )

    if table["gene_id"].duplicated().any():

        raise RuntimeError(
            f"{name}: duplicate gene identifiers."
        )


reference_ids = set(
    phase3["gene_id"]
)


for name, table in tables.items():

    if set(table["gene_id"]) != reference_ids:

        raise RuntimeError(
            f"Candidate gene inventory differs "
            f"between Phase 3 and {name}."
        )


# Verify stable Ensembl IDs wherever available.

for name, table in tables.items():

    if "ensembl_stable_id" not in table.columns:

        continue

    expected = (
        table["gene_id"]
        .astype(str)
        .str.replace(
            r"\.\d+$",
            "",
            regex=True
        )
    )

    observed = (
        table["ensembl_stable_id"]
        .astype(str)
        .str.strip()
    )

    if not expected.eq(observed).all():

        raise RuntimeError(
            f"{name}: stable Ensembl ID mismatch."
        )


print("\nAll four candidate universes agree.")

print("Unique research genes:", len(reference_ids))

print("Ensembl identifier consistency: PASSED")


# ------------------------------------------------------------
# 5. VALIDATE ORIGINAL GENE SYMBOL CORRESPONDENCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENE SYMBOL CORRESPONDENCE AUDIT")
print("-" * 65)


for name, table in tables.items():

    if "gene_name" not in table.columns:

        continue

    comparison = phase3[
        ["gene_id", "gene_name"]
    ].merge(

        table[
            ["gene_id", "gene_name"]
        ],

        on="gene_id",

        how="inner",

        suffixes=(
            "_phase3",
            "_comparison"
        ),

        validate="one_to_one"
    )

    if name == "Phase3":

        continue

    differences = comparison[
        comparison["gene_name_phase3"]
        .astype(str)
        .ne(
            comparison["gene_name_comparison"]
            .astype(str)
        )
    ]

    print(
        f"{name} symbol differences:",
        len(differences)
    )

    if not differences.empty:

        raise RuntimeError(
            f"{name}: unexpected changes to "
            "original research gene symbols."
        )


print("\nOriginal TCGA gene symbols: PRESERVED")

print(
    "Updated HGNC and DepMap symbols "
    "remain separate annotations."
)


# ------------------------------------------------------------
# 6. CROSS-VALIDATE CPTAC PROTEIN SUPPORT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING PROTEOMICS EVIDENCE")
print("-" * 65)


if (
    "protein_enriched" not in phase3.columns
    or "protein_enriched" not in cptac.columns
):

    raise RuntimeError(
        "Protein enrichment evidence missing."
    )


def to_boolean(series):

    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("true")
    )


phase3_protein = (
    phase3[
        ["gene_id", "protein_enriched"]
    ]
    .copy()
)


cptac_protein = (
    cptac[
        ["gene_id", "protein_enriched"]
    ]
    .copy()
)


phase3_protein["protein_enriched"] = (
    to_boolean(
        phase3_protein["protein_enriched"]
    )
)


cptac_protein["protein_enriched"] = (
    to_boolean(
        cptac_protein["protein_enriched"]
    )
)


protein_check = phase3_protein.merge(

    cptac_protein,

    on="gene_id",

    validate="one_to_one",

    suffixes=(
        "_phase3",
        "_cptac"
    )
)


if not (
    protein_check[
        "protein_enriched_phase3"
    ]
    ==
    protein_check[
        "protein_enriched_cptac"
    ]
).all():

    raise RuntimeError(
        "CPTAC enrichment status differs "
        "between Phase 2 and Phase 3."
    )


supported_genes = int(
    phase3_protein[
        "protein_enriched"
    ].sum()
)


if supported_genes != 20:

    raise RuntimeError(
        "Expected 20 cross-omics-supported genes."
    )


print("\nCPTAC status correspondence: PASSED")

print(
    "Candidates meeting exploratory "
    "cross-omics enrichment criteria:",
    supported_genes
)


print(
    "\nImportant: other genes may have "
    "insufficient proteomic coverage."
)


# ------------------------------------------------------------
# 7. VALIDATE FINAL DEPMAP INTEGRATION
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING FINAL DEPMAP COVERAGE")
print("-" * 65)


required_depmap_columns = {
    "gene_id",
    "depmap_final_mapping_available",
    "crc_models_measured",
    "crc_median_gene_effect",
    "hgnc_identity_status"
}


if not required_depmap_columns.issubset(
    depmap.columns
):

    raise RuntimeError(
        "Final DepMap integration schema differs."
    )


depmap["depmap_final_mapping_available"] = (
    to_boolean(
        depmap[
            "depmap_final_mapping_available"
        ]
    )
)


mapped_count = int(
    depmap[
        "depmap_final_mapping_available"
    ].sum()
)


if mapped_count != 943:

    raise RuntimeError(
        "Expected 943 verified DepMap mappings."
    )


depmap["crc_models_measured"] = (
    pd.to_numeric(
        depmap["crc_models_measured"],
        errors="raise"
    )
)


measured_count = int(
    depmap[
        "crc_models_measured"
    ].gt(0).sum()
)


if measured_count != 943:

    raise RuntimeError(
        "Expected 943 genes with CRC measurements."
    )


high_coverage_count = int(
    depmap[
        "crc_models_measured"
    ].ge(65).sum()
)


if high_coverage_count != 923:

    raise RuntimeError(
        "Expected 923 high-coverage genes."
    )


print("\nVerified DepMap gene mappings:", mapped_count)

print("Genes with CRC measurements:", measured_count)

print(
    "Genes with >=80% CRC coverage:",
    high_coverage_count
)

print("\nDepMap evidence: VALIDATED")


# ------------------------------------------------------------
# 8. PREPARE DEPMAP EVIDENCE FIELDS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING FUNCTIONAL EVIDENCE FIELDS")
print("-" * 65)


depmap_fields = [
    "depmap_column",
    "depmap_entrez_id",
    "depmap_current_symbol",
    "hgnc_identity_status",
    "depmap_final_mapping_available",
    "crc_models_measured",
    "crc_measurement_fraction",
    "crc_median_gene_effect",
    "crc_q25_gene_effect",
    "crc_q75_gene_effect",
    "crc_fraction_below_minus_05",
    "crc_fraction_below_minus_1",
    "other_models_measured",
    "other_median_gene_effect",
    "crc_minus_other_median_effect",
    "deduplicated_models_measured",
    "deduplicated_median_gene_effect",
    "dedup_minus_primary_median",
    "high_crc_coverage"
]


available_depmap_fields = [
    column
    for column in depmap_fields
    if column in depmap.columns
]


missing_depmap_fields = (
    set(depmap_fields)
    - set(available_depmap_fields)
)


if missing_depmap_fields:

    raise RuntimeError(
        "Expected DepMap evidence missing: "
        f"{missing_depmap_fields}"
    )


depmap_subset = depmap[
    ["gene_id"] + available_depmap_fields
].copy()


# Prefix functional measurements to keep
# their source unambiguous.
#
# Fields already beginning with depmap_
# retain their names.


rename_map = {

    column: (
        column
        if column.startswith("depmap_")
        else f"depmap_{column}"
    )

    for column in available_depmap_fields

}


depmap_subset = depmap_subset.rename(
    columns=rename_map
)


if depmap_subset.columns.duplicated().any():

    raise RuntimeError(
        "Duplicate names in prepared DepMap evidence."
    )


overlap = (
    set(depmap_subset.columns)
    & set(phase3.columns)
) - {"gene_id"}


if overlap:

    raise RuntimeError(
        "Overlapping evidence fields require "
        f"review before integration: {overlap}"
    )


print("\nDepMap evidence fields prepared:",
      len(available_depmap_fields))


# ------------------------------------------------------------
# 9. CONSTRUCT CANONICAL CROSS-OMICS TABLE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CONSTRUCTING CANONICAL EVIDENCE TABLE")
print("-" * 65)


integrated = phase3.merge(

    depmap_subset,

    on="gene_id",

    how="left",

    validate="one_to_one",

    indicator=True

)


if not integrated["_merge"].eq("both").all():

    raise RuntimeError(
        "Missing gene-level DepMap reconciliation."
    )


integrated = integrated.drop(
    columns="_merge"
)


if len(integrated) != 948:

    raise RuntimeError(
        "Candidate count changed during integration."
    )


if integrated["gene_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate genes after integration."
    )


integrated["protein_enriched"] = (
    to_boolean(
        integrated["protein_enriched"]
    )
)


print("\nCross-omics integration: PASSED")

print("Candidate genes preserved:", len(integrated))

print("Integrated columns:", len(integrated.columns))


# ------------------------------------------------------------
# 10. PRESERVE NORMAL-TISSUE UNCERTAINTY
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("NORMAL-TISSUE EVIDENCE VALIDATION")
print("-" * 65)


if "ihc_record_available" not in integrated.columns:

    raise RuntimeError(
        "Normal-IHC availability indicator missing."
    )


ihc_available = to_boolean(
    integrated["ihc_record_available"]
)


integrated[
    "phase5_normal_ihc_available"
] = ihc_available


ihc_coverage = int(
    ihc_available.sum()
)


if ihc_coverage != 690:

    raise RuntimeError(
        "Expected 690 normal-IHC annotated genes."
    )


# Preserve a separate missing-aware value.
#
# A gene without HPA IHC records should
# not be treated as having zero normal
# tissues with detectable expression.


for source_column in [
    "hpa_tissues_medium_high",
    "hpa_tissues_high",
    "hpa_reliable_medium_high_tissues"
]:

    if source_column not in integrated.columns:

        raise RuntimeError(
            f"Normal-tissue evidence missing: "
            f"{source_column}"
        )

    target_column = (
        "phase5_"
        + source_column
        + "_missing_aware"
    )

    values = pd.to_numeric(
        integrated[source_column],
        errors="raise"
    )

    integrated[target_column] = (
        values.where(
            ihc_available,
            np.nan
        )
    )


print("\nGenes with HPA normal-IHC:", ihc_coverage)

print(
    "Genes without HPA normal-IHC:",
    len(integrated) - ihc_coverage
)


print(
    "Missing normal-tissue evidence "
    "preserved as missing."
)


# ------------------------------------------------------------
# 11. EVIDENCE COVERAGE SUMMARY
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CROSS-OMICS EVIDENCE COVERAGE")
print("-" * 65)


def count_true(column):

    if column not in integrated.columns:
        return None

    return int(
        to_boolean(
            integrated[column]
        ).sum()
    )


coverage_records = [

    {
        "source": "TCGA",
        "evidence": "Research screening candidates",
        "genes": 948,
        "interpretation": (
            "Genes retained after the "
            "exploratory transcriptomic workflow"
        )
    },

    {
        "source": "CPTAC",
        "evidence": (
            "Qualifying tumor-associated "
            "protein enrichment"
        ),
        "genes": supported_genes,
        "interpretation": (
            "Exploratory enrichment criteria met; "
            "not therapeutic validation"
        )
    },

    {
        "source": "HPA",
        "evidence": "Normal-tissue IHC records",
        "genes": ihc_coverage,
        "interpretation": (
            "Normal-tissue observations available; "
            "not clinical safety assessment"
        )
    },

    {
        "source": "UniProt",
        "evidence": "Reviewed protein mappings",
        "genes": count_true(
            "uniprot_reviewed_available"
        ),
        "interpretation": (
            "Reviewed protein annotation available"
        )
    },

    {
        "source": "HPA",
        "evidence": "Subcellular annotations",
        "genes": count_true(
            "hpa_subcellular_record_available"
        ),
        "interpretation": (
            "HPA immunofluorescence annotation "
            "available"
        )
    },

    {
        "source": "DepMap",
        "evidence": "Verified gene mappings",
        "genes": mapped_count,
        "interpretation": (
            "HGNC-validated gene identity"
        )
    },

    {
        "source": "DepMap",
        "evidence": "High-coverage CRC gene effects",
        "genes": high_coverage_count,
        "interpretation": (
            "Measurements in at least "
            "65 of 81 CRC models"
        )
    }

]


coverage = pd.DataFrame(
    coverage_records
)


print("\n")

print(
    coverage[
        ["source", "evidence", "genes"]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 12. AUDIT AVAILABLE EVIDENCE COLUMNS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("AUDITING CANDIDATE EVIDENCE SCHEMA")
print("-" * 65)


rna_keywords = (
    r"log2|fold|padj|tier_a|"
    r"paired|expression|tumor"
)


protein_keywords = (
    r"protein|proteomic|cptac|"
    r"wilcoxon|mss_complete"
)


rna_columns = [
    column
    for column in phase3.columns
    if re.search(
        rna_keywords,
        column,
        flags=re.IGNORECASE
    )
]


protein_columns = [
    column
    for column in phase3.columns
    if re.search(
        protein_keywords,
        column,
        flags=re.IGNORECASE
    )
]


print("\nPotential RNA evidence fields:")

for column in rna_columns[:35]:
    print(" ", column)


print("\nPotential protein evidence fields:")

for column in protein_columns[:35]:
    print(" ", column)


print(
    "\nFull field inventory will be saved."
)


# ------------------------------------------------------------
# 13. REVIEW LEADING CANDIDATES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LEADING CROSS-OMICS CANDIDATE REVIEW")
print("-" * 65)


LEADING_GENES = [
    "CDH3",
    "CLDN1",
    "FAP",
    "DPEP1",
    "SLC7A5",
    "SLC5A6",
    "TDGF1",
    "CTHRC1",
    "S100P",
    "KRT80",
    "CEMIP",
    "INHBA",
    "WNT2"
]


leading = integrated[
    integrated["gene_name"]
    .isin(LEADING_GENES)
].copy()


leading["display_order"] = (
    leading["gene_name"]
    .map({
        gene: index
        for index, gene in enumerate(
            LEADING_GENES
        )
    })
)


leading = leading.sort_values(
    "display_order"
)


leading_columns = [
    "gene_name",
    "protein_enriched",
    "uniprot_topology_category",
    "uniprot_gpi_anchor",
    "phase5_normal_ihc_available",
    "phase5_hpa_tissues_medium_high_missing_aware",
    "depmap_crc_models_measured",
    "depmap_crc_median_gene_effect",
    "depmap_crc_fraction_below_minus_05"
]


missing_leading_columns = (
    set(leading_columns)
    - set(integrated.columns)
)


if missing_leading_columns:

    raise RuntimeError(
        "Leading evidence columns missing: "
        f"{missing_leading_columns}"
    )


print("\n")

print(
    leading[
        leading_columns
    ]
    .to_string(
        index=False
    )
)


print("\nIMPORTANT:")

print(
    "Protein enrichment, membrane topology, "
    "normal-tissue expression and CRISPR "
    "dependency are distinct evidence types."
)


print(
    "No biological target rankings "
    "have been assigned."
)


# ------------------------------------------------------------
# 14. FINAL INTEGRITY CHECKS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("FINAL CROSS-OMICS INTEGRITY VALIDATION")
print("-" * 65)


if int(
    integrated["protein_enriched"].sum()
) != 20:

    raise RuntimeError(
        "Protein support count changed."
    )


if int(
    to_boolean(
        integrated[
            "depmap_final_mapping_available"
        ]
    ).sum()
) != 943:

    raise RuntimeError(
        "DepMap mapping count changed."
    )


if int(
    integrated[
        "phase5_normal_ihc_available"
    ].sum()
) != 690:

    raise RuntimeError(
        "Normal-IHC coverage changed."
    )


unmapped = integrated[
    ~to_boolean(
        integrated[
            "depmap_final_mapping_available"
        ]
    )
].copy()


if len(unmapped) != 5:

    raise RuntimeError(
        "Expected five candidates without "
        "DepMap mappings."
    )


if pd.to_numeric(
    unmapped[
        "depmap_crc_models_measured"
    ],
    errors="coerce"
).fillna(0).gt(0).any():

    raise RuntimeError(
        "Unmapped genes unexpectedly have "
        "CRISPR measurements."
    )


print("\nCandidate preservation: PASSED")

print("Gene identifier correspondence: PASSED")

print("CPTAC evidence consistency: PASSED")

print("DepMap measurement coverage: PASSED")

print("Normal-IHC missingness handling: PASSED")


# ------------------------------------------------------------
# 15. CREATE EVIDENCE FIELD DICTIONARY
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING EVIDENCE FIELD DICTIONARY")
print("-" * 65)


field_records = []


for column in integrated.columns:

    if column.startswith("depmap_"):

        source = "DepMap"

    elif column.startswith("phase5_"):

        source = "Phase 5 derived audit field"

    else:

        source = (
            "Phase 3 integrated evidence "
            "(TCGA/CPTAC/HPA/UniProt)"
        )


    field_records.append({

        "column_name": column,

        "source": source,

        "dtype": str(
            integrated[column].dtype
        ),

        "nonmissing_records": int(
            integrated[column]
            .notna()
            .sum()
        ),

        "missing_records": int(
            integrated[column]
            .isna()
            .sum()
        )

    })


field_dictionary = pd.DataFrame(
    field_records
)


print(
    "\nFields documented:",
    len(field_dictionary)
)


# ------------------------------------------------------------
# 16. SAVE CANONICAL EVIDENCE TABLE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING CANONICAL CROSS-OMICS RESULTS")
print("-" * 65)


OUTPUT_PATH = (
    RESULTS /
    "surfacetarget_phase5_integrated_evidence_v1.csv"
)


integrated.to_csv(
    OUTPUT_PATH,
    index=False
)


coverage.to_csv(
    RESULTS /
    "surfacetarget_phase5_data_coverage.csv",
    index=False
)


field_dictionary.to_csv(
    RESULTS /
    "surfacetarget_phase5_field_dictionary.csv",
    index=False
)


leading.to_csv(
    RESULTS /
    "surfacetarget_phase5_leading_gene_audit.csv",
    index=False
)


unmapped.to_csv(
    RESULTS /
    "surfacetarget_phase5_unmeasured_depmap_genes.csv",
    index=False
)


# Verify that the complete canonical table
# can be reopened without losing gene rows.


reloaded = pd.read_csv(
    OUTPUT_PATH,
    dtype={"gene_id": str},
    low_memory=False
)


if len(reloaded) != 948:

    raise RuntimeError(
        "Saved integrated table row count differs."
    )


if (
    reloaded["gene_id"].tolist()
    != integrated["gene_id"].tolist()
):

    raise RuntimeError(
        "Saved gene ordering or identifiers differ."
    )


print("\nCanonical evidence table saved.")

print("Saved table verification: PASSED")


# ------------------------------------------------------------
# 17. SAVE RESEARCH PROVENANCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING PHASE 5 PROVENANCE")
print("-" * 65)


provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "5.1",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source_files": {

        name: str(path)

        for name, path in required_files.items()

    },

    "source_sha256": source_hashes,

    "candidate_genes": 948,

    "cross_omics_enriched_candidates": 20,

    "normal_ihc_annotated_candidates": 690,

    "depmap_verified_mappings": 943,

    "depmap_high_coverage_genes": 923,

    "canonical_output": str(OUTPUT_PATH),

    "canonical_output_sha256":
        sha256_file(OUTPUT_PATH),

    "integration_key": "gene_id",

    "integration_policy": (
        "One-to-one integration using the "
        "original versioned TCGA Ensembl ID"
    ),

    "source_separation": (
        "TCGA, CPTAC and DepMap observations "
        "remain independent datasets"
    ),

    "normal_ihc_missingness": (
        "Missing HPA IHC records are retained "
        "as unavailable evidence"
    ),

    "protein_missingness": (
        "Failure to meet CPTAC enrichment "
        "criteria does not imply absent protein"
    ),

    "msi_limitations": (
        "TCGA discovery Tier B is computationally "
        "low-MSI, not independently confirmed MSS; "
        "DepMap CRC models are not MSS-confirmed"
    ),

    "target_ranking_performed": False,

    "candidate_exclusions": 0,

    "limitations": [

        "Transcriptomic differential expression "
        "and CPTAC enrichment originate from "
        "different patient cohorts.",

        "Bulk tumor measurements cannot "
        "establish malignant-cell-specific expression.",

        "HPA staining is descriptive "
        "and not a clinical safety measurement.",

        "UniProt topology does not "
        "demonstrate antibody accessibility.",

        "DepMap effects do not directly "
        "measure therapeutic antibody efficacy.",

        "Missing evidence has not been "
        "converted to negative evidence.",

        "No therapeutic targets "
        "have been experimentally validated."

    ]

}


with open(
    META /
    "surfacetarget_phase5_integration_provenance.json",
    "w"
) as handle:

    json.dump(
        provenance,
        handle,
        indent=4
    )


print("\nIntegration provenance saved.")


# ------------------------------------------------------------
# 18. CREATE RESEARCH CHECKPOINT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING PHASE 5 RESEARCH CHECKPOINT")
print("-" * 65)


timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S"
)


checkpoint = (
    CHECKPOINTS /
    f"SurfaceTarget_CRC_Phase5_Patch51_{timestamp}.zip"
)


# Includes reproducible processed results
# and compact research sources.
#
# Excludes the large raw TCGA and DepMap
# datasets and preserved upload fragments.


backup_folders = [
    "data/metadata",
    "data/processed",
    "data/raw/cptac_colon",
    "data/raw/hpa",
    "data/raw/uniprot",
    "data/raw/hgnc",
    "results/tables",
    "results/figures",
    "src",
    "reports",
    "logs"
]


with ZipFile(
    checkpoint,
    "w",
    ZIP_DEFLATED,
    compresslevel=3
) as archive:

    for folder in backup_folders:

        source_folder = ROOT / folder

        if not source_folder.exists():
            continue

        for file_path in source_folder.rglob("*"):

            if file_path.is_file():

                archive.write(
                    file_path,
                    arcname=str(
                        file_path.relative_to(ROOT)
                    )
                )


checkpoint_size = (
    checkpoint.stat().st_size
    / 1024**2
)


print("\nCheckpoint created.")

print(
    "Size:",
    f"{checkpoint_size:.2f} MiB"
)

print("Location:")
print(checkpoint)


# ------------------------------------------------------------
# 19. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 5.1 SUMMARY")
print("=" * 65)


print("\nIntegrated candidate genes:", len(integrated))

print("Integrated evidence columns:", len(integrated.columns))

print("CPTAC-supported candidates:", supported_genes)

print("HPA normal-IHC annotated genes:", ihc_coverage)

print("Verified DepMap mappings:", mapped_count)

print(
    "Genes with high DepMap coverage:",
    high_coverage_count
)


print("\nSaved outputs:")

print("1. surfacetarget_phase5_integrated_evidence_v1.csv")

print("2. surfacetarget_phase5_data_coverage.csv")

print("3. surfacetarget_phase5_field_dictionary.csv")

print("4. surfacetarget_phase5_leading_gene_audit.csv")

print("5. surfacetarget_phase5_unmeasured_depmap_genes.csv")

print("6. surfacetarget_phase5_integration_provenance.json")


print("\nResearch checkpoint: CREATED")

print(
    "Checkpoint size:",
    f"{checkpoint_size:.2f} MiB"
)


print("\nNo candidate genes excluded.")

print("No therapeutic target scores assigned.")

print("No earlier analyses overwritten.")

print("\nPATCH 5.1 COMPLETE")

print("=" * 65)

In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 5 — PATCH 5.2
#
# BIOLOGICAL TARGET CLASSIFICATION
# AND EVIDENCE ELIGIBILITY AUDIT
#
# OBJECTIVES:
# 1. Validate the canonical Phase 5 evidence table
# 2. Classify protein topology evidence
# 3. Distinguish GPI-anchored and transmembrane proteins
# 4. Preserve uncertainty about antibody accessibility
# 5. Classify CPTAC protein-evidence completeness
# 6. Examine HPA normal colon, kidney and skin staining
# 7. Preserve normal-tissue IHC reliability information
# 8. Identify candidates requiring biological review
# 9. Review the 20 cross-omics-supported candidates
# 10. Examine leading membrane-related candidates
# 11. Save classification tables and provenance
# 12. Create a research checkpoint
#
# IMPORTANT:
# - No therapeutic efficacy claims
# - No clinical safety scores
# - No candidate exclusions
# - No arbitrary target-ranking weights
# - Missing evidence is not negative evidence
# - Bulk expression does not identify malignant cell origin
# - Membrane localization does not prove antibody binding
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
import numpy as np

import hashlib
import json


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"
RESULTS = ROOT / "results/tables"
CHECKPOINTS = ROOT / "checkpoints"

for folder in [META, RESULTS, CHECKPOINTS]:
    folder.mkdir(parents=True, exist_ok=True)


CANONICAL_PATH = (
    RESULTS /
    "surfacetarget_phase5_integrated_evidence_v1.csv"
)

TISSUE_PATH = (
    RESULTS /
    "hpa_candidate_tissue_level_profiles.csv"
)

PREVIOUS_PROVENANCE_PATH = (
    META /
    "surfacetarget_phase5_integration_provenance.json"
)


LEADING_GENES = [
    "CDH3",
    "CLDN1",
    "FAP",
    "DPEP1",
    "SLC7A5",
    "SLC5A6",
    "TDGF1",
    "CTHRC1",
    "S100P",
    "KRT80",
    "CEMIP",
    "INHBA",
    "WNT2"
]


print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 5 — PATCH 5.2")
print("BIOLOGICAL TARGET CLASSIFICATION")
print("=" * 65)


# ------------------------------------------------------------
# 2. VERIFY SOURCE FILES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING RESEARCH INPUTS")
print("-" * 65)


for path in [
    CANONICAL_PATH,
    TISSUE_PATH,
    PREVIOUS_PROVENANCE_PATH
]:

    if not path.exists():
        raise FileNotFoundError(path)


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


with open(
    PREVIOUS_PROVENANCE_PATH,
    "r"
) as handle:

    previous_provenance = json.load(handle)


expected_hash = previous_provenance.get(
    "canonical_output_sha256"
)

actual_hash = sha256_file(
    CANONICAL_PATH
)


if not expected_hash:

    raise RuntimeError(
        "Canonical checksum missing from Patch 5.1 provenance."
    )


if actual_hash != expected_hash:

    raise RuntimeError(
        "Phase 5 canonical evidence checksum mismatch."
    )


print("\nCanonical evidence checksum: VERIFIED")


# ------------------------------------------------------------
# 3. LOAD CANONICAL EVIDENCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING CANONICAL CROSS-OMICS EVIDENCE")
print("-" * 65)


source = pd.read_csv(
    CANONICAL_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)


required_columns = {
    "gene_id",
    "ensembl_stable_id",
    "gene_name",
    "protein_enriched",
    "mss_complete_pairs",
    "uniprot_topology_category",
    "uniprot_gpi_anchor",
    "uniprot_reviewed_available",
    "phase5_normal_ihc_available",
    "hpa_tissues_high",
    "hpa_tissues_medium_high",
    "hpa_reliable_medium_high_tissues",
    "depmap_final_mapping_available",
    "depmap_crc_models_measured",
    "depmap_crc_median_gene_effect"
}


missing_columns = (
    required_columns - set(source.columns)
)


if missing_columns:

    raise RuntimeError(
        f"Required evidence columns missing: "
        f"{sorted(missing_columns)}"
    )


if len(source) != 948:

    raise RuntimeError(
        "Expected 948 candidate genes."
    )


if source["gene_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate research gene identifiers."
    )


if source["ensembl_stable_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate stable Ensembl identifiers."
    )


print("\nCanonical candidates:", len(source))

print("Evidence columns:", len(source.columns))


# ------------------------------------------------------------
# 4. PREPARE STRICT BOOLEAN INTERPRETATION
# ------------------------------------------------------------

def parse_boolean(series, column_name):

    values = (
        series.astype("string")
        .str.strip()
        .str.lower()
    )

    accepted = {
        "true",
        "false",
        "1",
        "0"
    }

    missing_tokens = {
        "",
        "nan",
        "none",
        "<na>"
    }

    invalid = values[
        values.notna()
        & ~values.isin(
            list(accepted | missing_tokens)
        )
    ]

    if not invalid.empty:

        raise RuntimeError(
            f"Unexpected boolean values in "
            f"{column_name}: "
            f"{invalid.unique().tolist()[:10]}"
        )

    return (
        values.isin(["true", "1"])
        .fillna(False)
        .astype(bool)
    )


data = source.copy(deep=True)


data["_protein_enriched"] = parse_boolean(
    data["protein_enriched"],
    "protein_enriched"
)


data["_uniprot_gpi"] = parse_boolean(
    data["uniprot_gpi_anchor"],
    "uniprot_gpi_anchor"
)


data["_uniprot_reviewed"] = parse_boolean(
    data["uniprot_reviewed_available"],
    "uniprot_reviewed_available"
)


data["_normal_ihc_available"] = parse_boolean(
    data["phase5_normal_ihc_available"],
    "phase5_normal_ihc_available"
)


data["_depmap_available"] = parse_boolean(
    data["depmap_final_mapping_available"],
    "depmap_final_mapping_available"
)


if int(data["_protein_enriched"].sum()) != 20:

    raise RuntimeError(
        "Expected 20 CPTAC-supported candidates."
    )


if int(data["_uniprot_reviewed"].sum()) != 944:

    raise RuntimeError(
        "Expected 944 reviewed UniProt mappings."
    )


if int(data["_normal_ihc_available"].sum()) != 690:

    raise RuntimeError(
        "Expected 690 HPA normal-IHC annotated genes."
    )


if int(data["_depmap_available"].sum()) != 943:

    raise RuntimeError(
        "Expected 943 verified DepMap mappings."
    )


print("\nBoolean evidence validation: PASSED")


# ------------------------------------------------------------
# 5. CLASSIFY UNIPROT TOPOLOGY
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CLASSIFYING PROTEIN TOPOLOGY")
print("-" * 65)


def classify_topology(row):

    category = str(
        row["uniprot_topology_category"]
    ).strip().lower()

    gpi = bool(
        row["_uniprot_gpi"]
    )

    if (
        "cell membrane with annotated "
        "extracellular region" in category
    ):

        return (
            "Membrane with annotated "
            "extracellular region"
        )

    if gpi and (
        "cell membrane" in category
        or "cell-membrane" in category
    ):

        return (
            "GPI-anchored membrane-associated"
        )

    if gpi:

        return (
            "GPI anchor annotated; "
            "membrane residency requires review"
        )

    if (
        "cell membrane" in category
        or "cell-membrane" in category
    ):

        return (
            "Membrane-associated; "
            "extracellular topology unresolved"
        )

    if (
        "transmembrane protein" in category
        and "surface localization unresolved"
        in category
    ):

        return (
            "Transmembrane; "
            "surface localization unresolved"
        )

    if (
        "secreted or extracellular" in category
    ):

        return (
            "Secreted/extracellular; "
            "membrane attachment not established"
        )

    if (
        "no qualifying cell-surface topology"
        in category
    ):

        return (
            "No qualifying surface-topology "
            "evidence in extracted annotations"
        )

    return (
        "Other or insufficient topology annotation"
    )


data["phase5_topology_class"] = (
    data.apply(
        classify_topology,
        axis=1
    )
)


print("\nTopology class distribution:")

print(
    data["phase5_topology_class"]
    .value_counts()
    .to_string()
)


# ------------------------------------------------------------
# 6. ASSIGN DESCRIPTIVE SURFACE-REVIEW GROUPS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING SURFACE-EVIDENCE REVIEW GROUPS")
print("-" * 65)


def surface_review_group(category):

    if category == (
        "Membrane with annotated "
        "extracellular region"
    ):

        return (
            "Membrane evidence: "
            "extracellular region annotated"
        )

    if category == (
        "GPI-anchored membrane-associated"
    ):

        return (
            "Membrane evidence: GPI anchor"
        )

    if category in [
        "Membrane-associated; "
        "extracellular topology unresolved",

        "GPI anchor annotated; "
        "membrane residency requires review"
    ]:

        return (
            "Membrane evidence: "
            "accessibility requires review"
        )

    if category == (
        "Transmembrane; "
        "surface localization unresolved"
    ):

        return (
            "Transmembrane evidence: "
            "surface residency unresolved"
        )

    if category == (
        "Secreted/extracellular; "
        "membrane attachment not established"
    ):

        return (
            "Extracellular non-membrane review"
        )

    return (
        "No established surface-targeting "
        "topology from current annotations"
    )


data["phase5_surface_review_group"] = (
    data["phase5_topology_class"]
    .map(surface_review_group)
)


print("\nSurface-review group distribution:")

print(
    data["phase5_surface_review_group"]
    .value_counts()
    .to_string()
)


print("\nIMPORTANT:")

print(
    "These are evidence groups, "
    "not therapeutic target rankings."
)

print(
    "Membrane annotation does not establish "
    "antibody binding, epitope accessibility "
    "or internalization."
)


# ------------------------------------------------------------
# 7. PRESERVE HPA PLASMA-MEMBRANE EVIDENCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("AUDITING INDEPENDENT HPA LOCALIZATION")
print("-" * 65)


if "hpa_subcellular_record_available" in data.columns:

    hpa_localization_available = parse_boolean(
        data["hpa_subcellular_record_available"],
        "hpa_subcellular_record_available"
    )

else:

    hpa_localization_available = pd.Series(
        False,
        index=data.index
    )


if "hpa_plasma_membrane_reported" in data.columns:

    hpa_plasma_membrane = parse_boolean(
        data["hpa_plasma_membrane_reported"],
        "hpa_plasma_membrane_reported"
    )

else:

    hpa_plasma_membrane = pd.Series(
        False,
        index=data.index
    )


data["phase5_hpa_plasma_membrane_status"] = np.select(

    [
        hpa_plasma_membrane.to_numpy(dtype=bool),

        (
            hpa_localization_available
            & ~hpa_plasma_membrane
        ).to_numpy(dtype=bool)
    ],

    [
        "Plasma membrane reported",

        "Localization record available; "
        "plasma membrane not reported"
    ],

    default="HPA localization record unavailable"
)


print("\nHPA localization status:")

print(
    data["phase5_hpa_plasma_membrane_status"]
    .value_counts()
    .to_string()
)


# ------------------------------------------------------------
# 8. CLASSIFY CPTAC EVIDENCE COMPLETENESS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CLASSIFYING CPTAC PROTEIN EVIDENCE")
print("-" * 65)


data["_mss_complete_pairs"] = pd.to_numeric(
    data["mss_complete_pairs"],
    errors="coerce"
)


def protein_evidence_state(row):

    if row["_protein_enriched"]:

        return (
            "Qualifying MSS paired-protein enrichment"
        )

    n = row["_mss_complete_pairs"]

    if pd.isna(n):

        return (
            "Paired-protein coverage unavailable"
        )

    if n >= 30:

        return (
            "Adequately covered in paired assay; "
            "enrichment criteria not met"
        )

    if n > 0:

        return (
            "Sparse paired-protein measurements"
        )

    return (
        "No complete MSS paired-protein measurement"
    )


data["phase5_protein_evidence_state"] = (
    data.apply(
        protein_evidence_state,
        axis=1
    )
)


print("\nCPTAC evidence categories:")

print(
    data["phase5_protein_evidence_state"]
    .value_counts()
    .to_string()
)


print("\nIMPORTANT:")

print(
    "Absence of qualifying protein enrichment "
    "does not demonstrate protein absence."
)


# ------------------------------------------------------------
# 9. PREPARE NORMAL-TISSUE EVIDENCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING NORMAL-TISSUE EVIDENCE")
print("-" * 65)


normal_columns = [
    "hpa_tissues_high",
    "hpa_tissues_medium_high",
    "hpa_reliable_medium_high_tissues"
]


for column in normal_columns:

    data["_" + column] = pd.to_numeric(
        data[column],
        errors="coerce"
    )


def normal_ihc_state(row):

    if not row["_normal_ihc_available"]:

        return (
            "Normal-IHC evidence unavailable"
        )

    high = row["_hpa_tissues_high"]

    medium_high = row[
        "_hpa_tissues_medium_high"
    ]

    if pd.notna(high) and high > 0:

        return (
            "High staining reported "
            "in at least one normal tissue"
        )

    if (
        pd.notna(medium_high)
        and medium_high > 0
    ):

        return (
            "Medium staining reported; "
            "no high tissue-level staining observed"
        )

    return (
        "No medium/high staining in "
        "available summarized IHC observations"
    )


data["phase5_normal_ihc_state"] = (
    data.apply(
        normal_ihc_state,
        axis=1
    )
)


print("\nNormal-IHC evidence categories:")

print(
    data["phase5_normal_ihc_state"]
    .value_counts()
    .to_string()
)


# ------------------------------------------------------------
# 10. LOAD HPA TISSUE-LEVEL PROFILES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING NORMAL-TISSUE CELL-TYPE SUMMARIES")
print("-" * 65)


tissues = pd.read_csv(
    TISSUE_PATH,
    dtype={"Gene": str},
    low_memory=False
)


required_tissue_columns = {
    "Gene",
    "Tissue",
    "max_level_score",
    "max_reliable_score"
}


if not required_tissue_columns.issubset(
    tissues.columns
):

    raise RuntimeError(
        "Unexpected HPA tissue-level schema."
    )


if tissues[
    ["Gene", "Tissue"]
].duplicated().any():

    raise RuntimeError(
        "Duplicate gene-tissue profiles."
    )


if not set(tissues["Gene"]).issubset(
    set(data["ensembl_stable_id"])
):

    raise RuntimeError(
        "HPA tissue profiles contain "
        "unexpected candidate genes."
    )


for column in [
    "max_level_score",
    "max_reliable_score"
]:

    tissues[column] = pd.to_numeric(
        tissues[column],
        errors="coerce"
    )


print("\nGene-tissue profiles:", len(tissues))

print("Tissues represented:", tissues["Tissue"].nunique())


# ------------------------------------------------------------
# 11. NORMAL COLON, KIDNEY AND SKIN REVIEW
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("NORMAL COLON, KIDNEY AND SKIN REVIEW")
print("-" * 65)


FOCUS_TISSUES = [
    "Colon",
    "Kidney",
    "Skin"
]


SCORE_LABELS = {
    0: "Not detected",
    1: "Low",
    2: "Medium",
    3: "High"
}


for tissue_name in FOCUS_TISSUES:

    key = tissue_name.lower()

    subset = tissues[
        tissues["Tissue"]
        .astype(str)
        .str.lower()
        .eq(key)
    ].copy()


    if subset["Gene"].duplicated().any():

        raise RuntimeError(
            f"Duplicate {tissue_name} gene profiles."
        )


    primary_lookup = subset.set_index(
        "Gene"
    )["max_level_score"]


    reliable_lookup = subset.set_index(
        "Gene"
    )["max_reliable_score"]


    score_column = (
        f"phase5_normal_{key}_max_ihc_score"
    )

    reliable_column = (
        f"phase5_normal_{key}_reliable_ihc_score"
    )

    label_column = (
        f"phase5_normal_{key}_ihc_label"
    )


    data[score_column] = (
        data["ensembl_stable_id"]
        .map(primary_lookup)
    )


    data[reliable_column] = (
        data["ensembl_stable_id"]
        .map(reliable_lookup)
    )


    data[label_column] = (
        data[score_column]
        .map(SCORE_LABELS)
        .fillna(
            "No interpretable tissue observation"
        )
    )


    print(
        f"\n{tissue_name}: "
        f"{len(subset)} gene-level profiles"
    )


print("\nImportant:")

print(
    "Tissue-level IHC represents the maximum "
    "reported level among annotated cell types."
)

print(
    "It does not establish clinical toxicity "
    "or accessibility of the protein."
)


# ------------------------------------------------------------
# 12. SEPARATE BIOLOGICAL TARGETING HYPOTHESES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("ANNOTATING PUTATIVE TARGETING MECHANISMS")
print("-" * 65)


def targeting_hypothesis(row):

    gene = row["gene_name"]

    category = row[
        "phase5_topology_class"
    ]


    # Explicit biological review hypothesis.
    # This does NOT establish localization
    # to tumor-associated fibroblasts
    # from our bulk datasets.

    if gene == "FAP":

        return (
            "Stromal-associated targeting hypothesis; "
            "tumor cell-type origin requires validation"
        )


    if category in [
        "Membrane with annotated extracellular region",
        "GPI-anchored membrane-associated",
        "Membrane-associated; "
        "extracellular topology unresolved",
        "GPI anchor annotated; "
        "membrane residency requires review"
    ]:

        return (
            "Cell-surface antibody-targeting hypothesis; "
            "cellular origin and access unverified"
        )


    if category == (
        "Transmembrane; "
        "surface localization unresolved"
    ):

        return (
            "Potential membrane target; "
            "plasma-membrane residency unresolved"
        )


    if category == (
        "Secreted/extracellular; "
        "membrane attachment not established"
    ):

        return (
            "Extracellular or microenvironment "
            "targeting hypothesis"
        )


    return (
        "No supported antibody-accessible "
        "mechanism established by current topology"
    )


data["phase5_targeting_hypothesis"] = (
    data.apply(
        targeting_hypothesis,
        axis=1
    )
)


print("\nWorking targeting-hypothesis distribution:")

print(
    data["phase5_targeting_hypothesis"]
    .value_counts()
    .to_string()
)


# ------------------------------------------------------------
# 13. IDENTIFY EVIDENCE-REVIEW FLAGS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENERATING EVIDENCE-REVIEW FLAGS")
print("-" * 65)


def generate_review_flags(row):

    flags = []


    if not row["_uniprot_reviewed"]:

        flags.append(
            "Reviewed UniProt mapping unavailable"
        )


    if not row["_normal_ihc_available"]:

        flags.append(
            "Normal-tissue IHC unavailable"
        )


    if not row["_depmap_available"]:

        flags.append(
            "CRISPR gene-effect measurement unavailable"
        )


    if (
        row["phase5_protein_evidence_state"]
        == "Sparse paired-protein measurements"
    ):

        flags.append(
            "Sparse CPTAC paired-protein coverage"
        )


    if (
        row["phase5_protein_evidence_state"]
        == "No complete MSS paired-protein measurement"
    ):

        flags.append(
            "No complete CPTAC paired-protein observations"
        )


    colon_score = row[
        "phase5_normal_colon_max_ihc_score"
    ]

    kidney_score = row[
        "phase5_normal_kidney_max_ihc_score"
    ]


    if (
        pd.notna(colon_score)
        and colon_score >= 2
    ):

        flags.append(
            "Normal colon medium/high IHC reported"
        )


    if (
        pd.notna(kidney_score)
        and kidney_score >= 2
    ):

        flags.append(
            "Normal kidney medium/high IHC reported"
        )


    if (
        row["gene_name"] == "FAP"
    ):

        flags.append(
            "Stromal-vs-tumor-cell localization review"
        )


    return "; ".join(flags)


data["phase5_evidence_review_flags"] = (
    data.apply(
        generate_review_flags,
        axis=1
    )
)


data["phase5_has_review_flags"] = (
    data["phase5_evidence_review_flags"]
    .ne("")
)


print(
    "\nCandidates with at least one "
    "evidence-review flag:",
    int(
        data["phase5_has_review_flags"].sum()
    )
)


print(
    "Flags indicate questions requiring "
    "review, not candidate disqualification."
)


# ------------------------------------------------------------
# 14. VERIFY SOURCE EVIDENCE WAS PRESERVED
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING SOURCE EVIDENCE PRESERVATION")
print("-" * 65)


original_columns = source.columns.tolist()


if not source.equals(
    data[original_columns]
):

    raise RuntimeError(
        "Original canonical evidence changed "
        "during classification."
    )


if len(data) != 948:

    raise RuntimeError(
        "Candidate count changed."
    )


if int(
    data["_protein_enriched"].sum()
) != 20:

    raise RuntimeError(
        "Cross-omics support count changed."
    )


if int(
    data["_depmap_available"].sum()
) != 943:

    raise RuntimeError(
        "DepMap coverage changed."
    )


print("\nOriginal evidence columns: UNCHANGED")

print("Candidate count: PASSED")

print("Evidence identity: PASSED")


# ------------------------------------------------------------
# 15. REVIEW 20 CROSS-OMICS-SUPPORTED GENES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CROSS-OMICS-SUPPORTED CANDIDATE REVIEW")
print("-" * 65)


cross_omics = data[
    data["_protein_enriched"]
].copy()


if len(cross_omics) != 20:

    raise RuntimeError(
        "Expected 20 protein-supported genes."
    )


cross_columns = [
    "gene_name",
    "phase5_topology_class",
    "phase5_protein_evidence_state",
    "hpa_tissues_medium_high",
    "depmap_crc_median_gene_effect"
]


print("\n")

print(
    cross_omics[
        cross_columns
    ].to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 16. REVIEW LEADING CANDIDATES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LEADING THERAPEUTIC HYPOTHESIS REVIEW")
print("-" * 65)


leading = data[
    data["gene_name"].isin(
        LEADING_GENES
    )
].copy()


leading["display_order"] = (
    leading["gene_name"].map({
        gene: index
        for index, gene in enumerate(
            LEADING_GENES
        )
    })
)


leading = leading.sort_values(
    "display_order"
)


leading_columns = [
    "gene_name",
    "phase5_topology_class",
    "protein_enriched",
    "mss_complete_pairs",
    "phase5_normal_colon_ihc_label",
    "phase5_normal_kidney_ihc_label",
    "depmap_crc_median_gene_effect"
]


print("\n")

print(
    leading[
        leading_columns
    ].to_string(
        index=False
    )
)


print("\nTargeting hypotheses:")

print(
    leading[
        [
            "gene_name",
            "phase5_targeting_hypothesis"
        ]
    ].to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 17. PREPARE CLASSIFICATION SUMMARY
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING CLASSIFICATION SUMMARY")
print("-" * 65)


class_summary = (
    data.groupby(
        "phase5_topology_class",
        dropna=False
    )
    .agg(

        candidates=(
            "gene_id",
            "size"
        ),

        cptac_supported=(
            "_protein_enriched",
            "sum"
        ),

        normal_ihc_available=(
            "_normal_ihc_available",
            "sum"
        ),

        depmap_available=(
            "_depmap_available",
            "sum"
        )

    )
    .reset_index()
)


print("\n")

print(
    class_summary.to_string(
        index=False
    )
)


if int(
    class_summary["candidates"].sum()
) != 948:

    raise RuntimeError(
        "Classification summary lost candidates."
    )


# ------------------------------------------------------------
# 18. PREPARE CLEAN OUTPUT TABLE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING CLASSIFICATION OUTPUTS")
print("-" * 65)


# Temporary internal calculation fields
# are not exported.

temporary_columns = [
    column
    for column in data.columns
    if column.startswith("_")
    and column not in original_columns
]


final = data.drop(
    columns=temporary_columns
).copy()


if len(final) != 948:

    raise RuntimeError(
        "Output candidate count changed."
    )


review_flags = final[
    final["phase5_has_review_flags"]
].copy()


focus_tissue_columns = [
    "gene_id",
    "gene_name",
    "phase5_topology_class",
    "phase5_normal_colon_ihc_label",
    "phase5_normal_colon_reliable_ihc_score",
    "phase5_normal_kidney_ihc_label",
    "phase5_normal_kidney_reliable_ihc_score",
    "phase5_normal_skin_ihc_label",
    "phase5_normal_skin_reliable_ihc_score"
]


normal_focus = final[
    focus_tissue_columns
].copy()


surface_groups = final[
    [
        "gene_id",
        "gene_name",
        "phase5_topology_class",
        "phase5_surface_review_group",
        "phase5_targeting_hypothesis",
        "phase5_protein_evidence_state",
        "phase5_normal_ihc_state"
    ]
].copy()


# ------------------------------------------------------------
# 19. SAVE CLASSIFICATION TABLES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING BIOLOGICAL CLASSIFICATION RESULTS")
print("-" * 65)


output_path = (
    RESULTS /
    "surfacetarget_phase5_target_classification_v1.csv"
)


final.to_csv(
    output_path,
    index=False
)


class_summary.to_csv(
    RESULTS /
    "surfacetarget_phase5_topology_class_summary.csv",
    index=False
)


cross_omics.drop(
    columns=temporary_columns,
    errors="ignore"
).to_csv(
    RESULTS /
    "surfacetarget_phase5_crossomics_target_review.csv",
    index=False
)


leading.drop(
    columns=temporary_columns,
    errors="ignore"
).to_csv(
    RESULTS /
    "surfacetarget_phase5_leading_target_review.csv",
    index=False
)


surface_groups.to_csv(
    RESULTS /
    "surfacetarget_phase5_surface_review_groups.csv",
    index=False
)


normal_focus.to_csv(
    RESULTS /
    "surfacetarget_phase5_normal_tissue_focus_review.csv",
    index=False
)


review_flags.to_csv(
    RESULTS /
    "surfacetarget_phase5_evidence_review_flags.csv",
    index=False
)


reloaded = pd.read_csv(
    output_path,
    dtype={"gene_id": str},
    low_memory=False
)


if len(reloaded) != 948:

    raise RuntimeError(
        "Saved classification table has "
        "an unexpected gene count."
    )


if (
    reloaded["gene_id"].tolist()
    != final["gene_id"].tolist()
):

    raise RuntimeError(
        "Saved candidate identifiers changed."
    )


print("\nAll classification tables saved.")

print("Saved table validation: PASSED")


# ------------------------------------------------------------
# 20. SAVE RESEARCH PROVENANCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING CLASSIFICATION PROVENANCE")
print("-" * 65)


provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "5.2",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source_canonical_table": str(
        CANONICAL_PATH
    ),

    "source_canonical_sha256": actual_hash,

    "hpa_tissue_source": str(
        TISSUE_PATH
    ),

    "hpa_tissue_sha256": sha256_file(
        TISSUE_PATH
    ),

    "candidate_count": int(
        len(final)
    ),

    "protein_supported_candidates": int(
        data["_protein_enriched"].sum()
    ),

    "normal_ihc_annotated_candidates": int(
        data["_normal_ihc_available"].sum()
    ),

    "verified_depmap_mappings": int(
        data["_depmap_available"].sum()
    ),

    "classification_counts": {

        str(row["phase5_topology_class"]):
            int(row["candidates"])

        for _, row in class_summary.iterrows()

    },

    "classification_method": (
        "Rule-based interpretation of existing "
        "UniProt protein-topology categories"
    ),

    "gpi_policy": (
        "GPI anchoring is evaluated separately "
        "from transmembrane segments"
    ),

    "protein_evidence_policy": (
        "Separate qualifying enrichment, "
        "adequate nonqualifying measurements, "
        "sparse coverage and missing observations"
    ),

    "normal_ihc_policy": (
        "Preserve tissue-level staining and "
        "reliability annotations without "
        "assigning clinical safety scores"
    ),

    "targeting_hypothesis_policy": (
        "Working mechanism categories only; "
        "tumor-cell specificity not inferred "
        "from bulk expression"
    ),

    "stromal_hypothesis": (
        "FAP separately flagged for "
        "tumor-stroma localization review"
    ),

    "dependency_policy": (
        "DepMap CRISPR fitness effects retained "
        "as an orthogonal evidence dimension"
    ),

    "original_evidence_preserved": True,

    "candidate_exclusions": 0,

    "therapeutic_target_ranking_performed": False,

    "limitations": [

        "Membrane localization does not "
        "demonstrate antibody accessibility.",

        "Extracellular topology annotations "
        "do not establish binding epitopes.",

        "Bulk tumor enrichment cannot distinguish "
        "malignant from stromal expression.",

        "GPI-anchor annotations do not "
        "prove therapeutic suitability.",

        "Normal-tissue IHC is not "
        "a clinical toxicity measurement.",

        "Maximum tissue staining summarizes "
        "multiple annotated cell types.",

        "Missing protein measurements are "
        "not treated as absent protein.",

        "CRISPR dependency is not required "
        "for antibody-drug conjugate targeting.",

        "The DepMap cohort is not "
        "independently confirmed MSS.",

        "No experimental target validation "
        "has been performed."

    ]
}


with open(
    META /
    "surfacetarget_phase5_classification_provenance.json",
    "w"
) as handle:

    json.dump(
        provenance,
        handle,
        indent=4
    )


print("\nResearch provenance saved.")


# ------------------------------------------------------------
# 21. CREATE RESEARCH CHECKPOINT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING PHASE 5 RESEARCH CHECKPOINT")
print("-" * 65)


timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S"
)


checkpoint = (
    CHECKPOINTS /
    f"SurfaceTarget_CRC_Phase5_Patch52_{timestamp}.zip"
)


backup_folders = [
    "data/metadata",
    "data/processed",
    "data/raw/cptac_colon",
    "data/raw/hpa",
    "data/raw/uniprot",
    "data/raw/hgnc",
    "results/tables",
    "results/figures",
    "src",
    "reports",
    "logs"
]


with ZipFile(
    checkpoint,
    "w",
    ZIP_DEFLATED,
    compresslevel=3
) as archive:

    for folder in backup_folders:

        folder_path = ROOT / folder

        if not folder_path.exists():
            continue

        for file_path in folder_path.rglob("*"):

            if file_path.is_file():

                archive.write(
                    file_path,
                    arcname=str(
                        file_path.relative_to(ROOT)
                    )
                )


checkpoint_size = (
    checkpoint.stat().st_size / 1024**2
)


print("\nCheckpoint created successfully.")

print(
    "Checkpoint size:",
    f"{checkpoint_size:.2f} MiB"
)

print("Checkpoint location:")
print(checkpoint)


# ------------------------------------------------------------
# 22. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 5.2 SUMMARY")
print("=" * 65)


print("\nCandidate genes:", len(final))

print(
    "Biological topology classes:",
    class_summary.shape[0]
)

print(
    "Cross-omics-supported candidates:",
    len(cross_omics)
)

print(
    "Candidates with normal-IHC evidence:",
    int(
        data["_normal_ihc_available"].sum()
    )
)

print(
    "Verified DepMap mappings:",
    int(
        data["_depmap_available"].sum()
    )
)

print(
    "Candidates with evidence-review flags:",
    len(review_flags)
)


print("\nSaved outputs:")

print("1. surfacetarget_phase5_target_classification_v1.csv")

print("2. surfacetarget_phase5_topology_class_summary.csv")

print("3. surfacetarget_phase5_crossomics_target_review.csv")

print("4. surfacetarget_phase5_leading_target_review.csv")

print("5. surfacetarget_phase5_surface_review_groups.csv")

print("6. surfacetarget_phase5_normal_tissue_focus_review.csv")

print("7. surfacetarget_phase5_evidence_review_flags.csv")

print("8. surfacetarget_phase5_classification_provenance.json")


print("\nClassification validation: PASSED")

print("Original evidence: PRESERVED")

print("Research checkpoint: CREATED")

print(
    "Checkpoint size:",
    f"{checkpoint_size:.2f} MiB"
)


print("\nNo candidate genes excluded.")

print("No clinical safety scores assigned.")

print("No final target rankings performed.")

print("\nPATCH 5.2 COMPLETE")

print("=" * 65)

In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 5 — PATCH 5.3
#
# EVIDENCE-BASED CANDIDATE TRIAGE
#
# Objectives:
# 1. Validate Phase 5.2 classification results
# 2. Audit RNA evidence consistency
# 3. Identify candidates with annotated surface topology
# 4. Distinguish proteomic support from missing coverage
# 5. Preserve normal-tissue expression concerns
# 6. Preserve HPA staining reliability information
# 7. Separate tumor-cell and stromal hypotheses
# 8. Create research follow-up categories
# 9. Identify candidate evidence gaps
# 10. Save results and provenance
#
# IMPORTANT:
# - No therapeutic efficacy scores
# - No clinical safety scores
# - No candidate genes excluded
# - No arbitrary weighted ranking
# - DepMap dependency does not determine ADC eligibility
# - Missing evidence is not negative evidence
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import hashlib
import json


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

RESULTS = ROOT / "results/tables"
META = ROOT / "data/metadata"

RESULTS.mkdir(parents=True, exist_ok=True)
META.mkdir(parents=True, exist_ok=True)

SOURCE_PATH = (
    RESULTS /
    "surfacetarget_phase5_target_classification_v1.csv"
)

PROVENANCE_PATH = (
    META /
    "surfacetarget_phase5_classification_provenance.json"
)

OUTPUT_PATH = (
    RESULTS /
    "surfacetarget_phase5_evidence_triage_v1.csv"
)

LEADING_GENES = [
    "CDH3",
    "CLDN1",
    "FAP",
    "DPEP1",
    "SLC7A5",
    "SLC5A6",
    "TDGF1",
    "CTHRC1",
    "S100P",
    "KRT80",
    "CEMIP",
    "INHBA",
    "WNT2"
]

print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 5 — PATCH 5.3")
print("EVIDENCE-BASED CANDIDATE TRIAGE")
print("=" * 65)


# ------------------------------------------------------------
# 2. VALIDATE RESEARCH INPUTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING RESEARCH INPUTS")
print("-" * 65)

for path in [SOURCE_PATH, PROVENANCE_PATH]:

    if not path.exists():
        raise FileNotFoundError(path)


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


with open(PROVENANCE_PATH, "r") as handle:
    previous_provenance = json.load(handle)

data = pd.read_csv(
    SOURCE_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)

if len(data) != 948:
    raise RuntimeError("Expected 948 candidates.")

if data["gene_id"].duplicated().any():
    raise RuntimeError("Duplicate gene identifiers.")

required = {
    "gene_id",
    "gene_name",
    "baseline_log2fc",
    "baseline_padj",
    "sensitivity_log2fc",
    "sensitivity_padj",
    "tier_a_median_cpm",
    "tier_a_fraction_cpm_ge1",
    "paired_direction_supported",
    "protein_enriched",
    "mss_complete_pairs",
    "phase5_topology_class",
    "phase5_surface_review_group",
    "phase5_protein_evidence_state",
    "phase5_normal_ihc_available",
    "hpa_reliable_medium_high_tissues",
    "phase5_normal_colon_max_ihc_score",
    "phase5_normal_colon_reliable_ihc_score",
    "phase5_normal_kidney_max_ihc_score",
    "phase5_normal_kidney_reliable_ihc_score",
    "phase5_normal_skin_max_ihc_score",
    "phase5_normal_skin_reliable_ihc_score",
    "depmap_crc_models_measured",
    "depmap_crc_median_gene_effect"
}

missing = required - set(data.columns)

if missing:
    raise RuntimeError(
        f"Missing required fields: {sorted(missing)}"
    )

if int(
    previous_provenance.get("candidate_count", -1)
) != 948:

    raise RuntimeError(
        "Patch 5.2 provenance count mismatch."
    )

source_hash = sha256_file(SOURCE_PATH)

print("\nSource candidates:", len(data))
print("Source fields:", len(data.columns))
print("Source SHA256:", source_hash)
print("Input validation: PASSED")


# ------------------------------------------------------------
# 3. BOOLEAN AND NUMERIC HELPERS
# ------------------------------------------------------------

def as_bool(series):

    values = (
        series.astype("string")
        .str.strip()
        .str.lower()
    )

    valid = {
        "true", "false", "1", "0",
        "", "nan", "none", "<na>"
    }

    unexpected = (
        values.notna()
        & ~values.isin(list(valid))
    )

    if unexpected.any():
        raise RuntimeError(
            "Unexpected boolean values: "
            f"{values[unexpected].unique().tolist()}"
        )

    return (
        values.isin(["true", "1"])
        .fillna(False)
        .astype(bool)
    )


def as_numeric(column):

    return pd.to_numeric(
        data[column],
        errors="coerce"
    )


protein = as_bool(data["protein_enriched"])
paired = as_bool(data["paired_direction_supported"])
ihc = as_bool(data["phase5_normal_ihc_available"])

if protein.sum() != 20:
    raise RuntimeError("Expected 20 protein-supported genes.")

if ihc.sum() != 690:
    raise RuntimeError("Expected 690 IHC-annotated genes.")


# ------------------------------------------------------------
# 4. AUDIT RNA EVIDENCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("AUDITING TRANSCRIPTOMIC SUPPORT")
print("-" * 65)

baseline_lfc = as_numeric("baseline_log2fc")
baseline_padj = as_numeric("baseline_padj")

sensitivity_lfc = as_numeric("sensitivity_log2fc")
sensitivity_padj = as_numeric("sensitivity_padj")

tier_a_cpm = as_numeric("tier_a_median_cpm")
tier_a_fraction = as_numeric(
    "tier_a_fraction_cpm_ge1"
)

# These thresholds are exploratory evidence
# descriptors, not therapeutic eligibility rules.

baseline_support = (
    baseline_lfc > 1
) & (
    baseline_padj < 0.05
)

sensitivity_support = (
    sensitivity_lfc > 1
) & (
    sensitivity_padj < 0.05
)

tier_a_expression_support = (
    tier_a_cpm >= 1
) & (
    tier_a_fraction >= 0.5
)

# Matched-patient directional support is
# supplementary and should not be described
# as independent external validation.

data["phase5_rna_baseline_supported"] = (
    baseline_support.fillna(False)
)

data["phase5_rna_sensitivity_supported"] = (
    sensitivity_support.fillna(False)
)

data["phase5_rna_tier_a_expression_supported"] = (
    tier_a_expression_support.fillna(False)
)

data["phase5_rna_paired_direction_supported"] = paired

data["phase5_rna_evidence_pattern"] = np.select(

    [
        (
            baseline_support
            & sensitivity_support
            & tier_a_expression_support
            & paired
        ).to_numpy(dtype=bool),

        (
            baseline_support
            & sensitivity_support
            & tier_a_expression_support
        ).to_numpy(dtype=bool),

        (
            baseline_support
            & sensitivity_support
        ).to_numpy(dtype=bool)
    ],

    [
        "Reproducible DE + Tier A expression + paired direction",
        "Reproducible DE + Tier A expression",
        "Reproducible DE; additional support requires review"
    ],

    default="Partial or inconsistent RNA evidence"
)

print("\nRNA evidence patterns:")

print(
    data["phase5_rna_evidence_pattern"]
    .value_counts()
    .to_string()
)


# ------------------------------------------------------------
# 5. DEFINE SURFACE-TOPOLOGY EVIDENCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CLASSIFYING SURFACE-TOPOLOGY EVIDENCE")
print("-" * 65)

STRONGER_TOPOLOGY = {
    "Membrane with annotated extracellular region",
    "GPI-anchored membrane-associated"
}

UNRESOLVED_MEMBRANE = {
    "Membrane-associated; extracellular topology unresolved",
    "Transmembrane; surface localization unresolved"
}

EXTRACELLULAR_NONMEMBRANE = {
    "Secreted/extracellular; membrane attachment not established"
}

topology = data["phase5_topology_class"]

stronger_surface = topology.isin(
    STRONGER_TOPOLOGY
)

unresolved_surface = topology.isin(
    UNRESOLVED_MEMBRANE
)

extracellular = topology.isin(
    EXTRACELLULAR_NONMEMBRANE
)

data["phase5_stronger_surface_annotation"] = (
    stronger_surface
)

data["phase5_unresolved_membrane_annotation"] = (
    unresolved_surface
)

data["phase5_extracellular_nonmembrane_annotation"] = (
    extracellular
)

print(
    "\nStronger surface-related annotations:",
    int(stronger_surface.sum())
)

print(
    "Membrane-related annotations requiring verification:",
    int(unresolved_surface.sum())
)

print(
    "Secreted/extracellular non-membrane annotations:",
    int(extracellular.sum())
)

print(
    "Other or insufficient topology evidence:",
    int(
        (
            ~stronger_surface
            & ~unresolved_surface
            & ~extracellular
        ).sum()
    )
)


# ------------------------------------------------------------
# 6. CLASSIFY EVIDENCE PACKAGES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CLASSIFYING CROSS-OMICS EVIDENCE PACKAGES")
print("-" * 65)

# These categories represent available evidence.
# They do not measure therapeutic efficacy.

conditions = [

    (
        stronger_surface & protein
    ).to_numpy(dtype=bool),

    (
        stronger_surface & ~protein
    ).to_numpy(dtype=bool),

    (
        unresolved_surface & protein
    ).to_numpy(dtype=bool),

    (
        unresolved_surface & ~protein
    ).to_numpy(dtype=bool),

    (
        extracellular & protein
    ).to_numpy(dtype=bool),

    (
        extracellular & ~protein
    ).to_numpy(dtype=bool)
]

choices = [
    "RNA + qualifying protein + stronger surface annotation",

    "RNA + stronger surface annotation; protein support incomplete or nonqualifying",

    "RNA + qualifying protein; membrane accessibility unresolved",

    "RNA + unresolved membrane evidence; protein support incomplete or nonqualifying",

    "RNA + qualifying protein; extracellular non-membrane hypothesis",

    "RNA + extracellular non-membrane annotation"
]

data["phase5_evidence_package"] = np.select(
    conditions,
    choices,
    default=(
        "RNA evidence; no qualifying surface topology "
        "in current annotations"
    )
)

print("\nEvidence-package distribution:")

print(
    data["phase5_evidence_package"]
    .value_counts()
    .to_string()
)


# ------------------------------------------------------------
# 7. DEFINE TARGETING REVIEW LANES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("ASSIGNING BIOLOGICAL FOLLOW-UP LANES")
print("-" * 65)

# FAP is handled separately because its
# stromal biology requires a different
# validation strategy.
#
# Other proteins are not assumed to be
# tumor-cell-specific.

fap = data["gene_name"].eq("FAP")

data["phase5_followup_lane"] = np.select(

    [
        fap.to_numpy(dtype=bool),

        (
            stronger_surface & ~fap
        ).to_numpy(dtype=bool),

        unresolved_surface.to_numpy(dtype=bool),

        extracellular.to_numpy(dtype=bool)
    ],

    [
        "Stromal-associated targeting review",

        "Putative cell-surface antigen review",

        "Membrane localization/accessibility verification",

        "Extracellular microenvironment review"
    ],

    default=(
        "Topology clarification or alternative modality review"
    )
)

print("\nFollow-up lane distribution:")

print(
    data["phase5_followup_lane"]
    .value_counts()
    .to_string()
)


# ------------------------------------------------------------
# 8. AUDIT NORMAL-TISSUE EXPRESSION
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("AUDITING NORMAL-TISSUE SELECTIVITY QUESTIONS")
print("-" * 65)

FOCUS_TISSUES = [
    "colon",
    "kidney",
    "skin"
]

reported_medium_high = pd.Series(
    False,
    index=data.index
)

reliably_reported_medium_high = pd.Series(
    False,
    index=data.index
)

for tissue in FOCUS_TISSUES:

    observed = as_numeric(
        f"phase5_normal_{tissue}_max_ihc_score"
    )

    reliable = as_numeric(
        f"phase5_normal_{tissue}_reliable_ihc_score"
    )

    reported_medium_high |= (
        observed >= 2
    ).fillna(False)

    reliably_reported_medium_high |= (
        reliable >= 2
    ).fillna(False)


data["phase5_focus_normal_medium_high"] = (
    reported_medium_high
)

data["phase5_focus_normal_reliable_medium_high"] = (
    reliably_reported_medium_high
)

reliable_tissues = as_numeric(
    "hpa_reliable_medium_high_tissues"
)

# Evidence-review categories, not toxicity levels.

data["phase5_normal_tissue_review_state"] = np.select(

    [
        (~ihc).to_numpy(dtype=bool),

        reliably_reported_medium_high.to_numpy(dtype=bool),

        reported_medium_high.to_numpy(dtype=bool),

        (
            ihc
            & (reliable_tissues > 0).fillna(False)
        ).to_numpy(dtype=bool)
    ],

    [
        "Normal-tissue IHC unavailable",

        "Medium/high staining in focus organ with stronger IHC reliability",

        "Medium/high staining in focus organ; reliability requires review",

        "Reliable medium/high staining reported in other normal tissues"
    ],

    default=(
        "No medium/high focus-organ signal in available summary; "
        "not evidence of safety"
    )
)

print("\nNormal-tissue review states:")

print(
    data["phase5_normal_tissue_review_state"]
    .value_counts()
    .to_string()
)


# ------------------------------------------------------------
# 9. PRESERVE DEPMAP FUNCTIONAL EVIDENCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PRESERVING FUNCTIONAL DEPENDENCY CONTEXT")
print("-" * 65)

dependency = as_numeric(
    "depmap_crc_median_gene_effect"
)

depmap_coverage = as_numeric(
    "depmap_crc_models_measured"
)

data["phase5_depmap_functional_review"] = np.select(

    [
        (
            depmap_coverage == 0
        ).fillna(False).to_numpy(dtype=bool),

        (
            depmap_coverage >= 65
        ).fillna(False).to_numpy(dtype=bool)
    ],

    [
        "No colorectal CRISPR measurements",

        "High-coverage colorectal CRISPR observations"
    ],

    default="Partial colorectal CRISPR coverage"
)

print("\nFunctional evidence coverage:")

print(
    data["phase5_depmap_functional_review"]
    .value_counts()
    .to_string()
)

print("\nImportant:")

print(
    "CRISPR dependency is recorded but is "
    "not used to determine antibody-target eligibility."
)


# ------------------------------------------------------------
# 10. GENERATE FOLLOW-UP QUESTIONS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENERATING BIOLOGICAL REVIEW QUESTIONS")
print("-" * 65)


def followup_questions(row):

    questions = []

    lane = row["phase5_followup_lane"]

    protein_state = str(
        row["phase5_protein_evidence_state"]
    )

    normal_state = str(
        row["phase5_normal_tissue_review_state"]
    )

    if lane == "Stromal-associated targeting review":

        questions.append(
            "Confirm fibroblast versus malignant-cell expression"
        )

    elif lane == "Putative cell-surface antigen review":

        questions.append(
            "Validate malignant-cell surface expression and epitope access"
        )

    elif lane == (
        "Membrane localization/accessibility verification"
    ):

        questions.append(
            "Confirm plasma-membrane residency and extracellular accessibility"
        )

    elif lane == "Extracellular microenvironment review":

        questions.append(
            "Investigate extracellular localization and targeting mechanism"
        )

    else:

        questions.append(
            "Resolve protein topology before antibody-target ranking"
        )


    if protein_state != (
        "Qualifying MSS paired-protein enrichment"
    ):

        questions.append(
            "Review protein measurement coverage and tumor enrichment"
        )


    if normal_state == "Normal-tissue IHC unavailable":

        questions.append(
            "Acquire independent normal-tissue expression evidence"
        )

    elif (
        "Medium/high staining" in normal_state
        or "Reliable medium/high" in normal_state
    ):

        questions.append(
            "Review normal-tissue cell types and staining reliability"
        )


    if row["gene_name"] == "TDGF1":

        questions.append(
            "Evaluate CRIPTO surface presentation and normal-tissue selectivity"
        )


    return "; ".join(questions)


data["phase5_followup_questions"] = (
    data.apply(
        followup_questions,
        axis=1
    )
)


# ------------------------------------------------------------
# 11. REVIEW SURFACE-RELATED CANDIDATES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SURFACE-RELATED CANDIDATE REVIEW")
print("-" * 65)

surface_review = data[
    data["phase5_followup_lane"].isin([
        "Putative cell-surface antigen review",
        "Stromal-associated targeting review"
    ])
].copy()

print(
    "\nCandidates with stronger surface-related annotations:",
    len(surface_review)
)

surface_protein = surface_review[
    as_bool(surface_review["protein_enriched"])
].copy()

print(
    "With qualifying CPTAC protein enrichment:",
    len(surface_protein)
)

print("\nCross-omics-supported surface-related candidates:")

print(
    surface_protein[
        [
            "gene_name",
            "phase5_followup_lane",
            "phase5_topology_class",
            "phase5_protein_evidence_state",
            "phase5_normal_tissue_review_state",
            "depmap_crc_median_gene_effect"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 12. REVIEW ALL 20 PROTEIN-SUPPORTED GENES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("COMPLETE CROSS-OMICS CANDIDATE REVIEW")
print("-" * 65)

protein_review = data[
    protein
].copy()

if len(protein_review) != 20:
    raise RuntimeError(
        "Cross-omics candidate count changed."
    )

print("\n")

print(
    protein_review[
        [
            "gene_name",
            "phase5_followup_lane",
            "phase5_evidence_package"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 13. REVIEW LEADING GENES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LEADING CANDIDATE EVIDENCE REVIEW")
print("-" * 65)

leading = data[
    data["gene_name"].isin(
        LEADING_GENES
    )
].copy()

leading["display_order"] = (
    leading["gene_name"]
    .map({
        gene: i
        for i, gene in enumerate(
            LEADING_GENES
        )
    })
)

leading = leading.sort_values(
    "display_order"
)

print("\n")

print(
    leading[
        [
            "gene_name",
            "phase5_rna_evidence_pattern",
            "phase5_followup_lane",
            "phase5_protein_evidence_state",
            "phase5_normal_tissue_review_state"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 14. VERIFY SOURCE DATA WERE NOT MODIFIED
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VERIFYING DATA PRESERVATION")
print("-" * 65)

if len(data) != 948:
    raise RuntimeError(
        "Candidate count changed."
    )

if data["gene_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate genes after triage."
    )

if int(
    as_bool(data["protein_enriched"]).sum()
) != 20:

    raise RuntimeError(
        "CPTAC support count changed."
    )

if int(
    as_bool(
        data["phase5_normal_ihc_available"]
    ).sum()
) != 690:

    raise RuntimeError(
        "Normal-IHC count changed."
    )

print("\nCandidate preservation: PASSED")
print("CPTAC support count: PASSED")
print("Normal-IHC evidence count: PASSED")


# ------------------------------------------------------------
# 15. PREPARE SUMMARY TABLE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING EVIDENCE TRIAGE SUMMARY")
print("-" * 65)

data["_protein_supported_temp"] = protein.to_numpy()

lane_summary = (
    data.groupby(
        "phase5_followup_lane",
        dropna=False
    )
    .agg(
        candidates=("gene_id", "size"),

        cptac_supported=(
            "_protein_supported_temp",
            "sum"
        ),

        stronger_surface=(
            "phase5_stronger_surface_annotation",
            "sum"
        ),

        normal_ihc_available=(
            "phase5_normal_ihc_available",
            lambda x: int(as_bool(x).sum())
        )
    )
    .reset_index()
)

data = data.drop(
    columns=["_protein_supported_temp"]
)

print("\n")

print(
    lane_summary.to_string(index=False)
)

if int(lane_summary["candidates"].sum()) != 948:
    raise RuntimeError(
        "Triage summary lost candidates."
    )


# ------------------------------------------------------------
# 16. SAVE TRIAGE RESULTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING EVIDENCE TRIAGE RESULTS")
print("-" * 65)

data.to_csv(
    OUTPUT_PATH,
    index=False
)

lane_summary.to_csv(
    RESULTS /
    "surfacetarget_phase5_followup_lane_summary.csv",
    index=False
)

surface_review.to_csv(
    RESULTS /
    "surfacetarget_phase5_surface_candidate_review.csv",
    index=False
)

surface_protein.to_csv(
    RESULTS /
    "surfacetarget_phase5_surface_crossomics_review.csv",
    index=False
)

protein_review.to_csv(
    RESULTS /
    "surfacetarget_phase5_protein_supported_review.csv",
    index=False
)

leading.to_csv(
    RESULTS /
    "surfacetarget_phase5_leading_evidence_triage.csv",
    index=False
)


reloaded = pd.read_csv(
    OUTPUT_PATH,
    dtype={"gene_id": str},
    low_memory=False
)

if len(reloaded) != 948:
    raise RuntimeError(
        "Saved triage table has incorrect dimensions."
    )

if reloaded["gene_id"].tolist() != data["gene_id"].tolist():
    raise RuntimeError(
        "Saved gene order changed."
    )

print("\nAll evidence triage tables saved.")
print("Saved table validation: PASSED")


# ------------------------------------------------------------
# 17. SAVE PROVENANCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING TRIAGE PROVENANCE")
print("-" * 65)

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "5.3",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": str(SOURCE_PATH),

    "source_sha256": source_hash,

    "output": str(OUTPUT_PATH),

    "output_sha256": sha256_file(OUTPUT_PATH),

    "candidate_count": 948,

    "protein_supported_count": 20,

    "surface_annotation_policy": (
        "Annotated extracellular membrane regions "
        "and GPI-associated membrane proteins "
        "form the stronger surface review group"
    ),

    "rna_policy": (
        "Exploratory baseline and sensitivity DE, "
        "Tier A expression and patient-paired "
        "directional support retained separately"
    ),

    "protein_policy": (
        "Preserve qualifying enrichment and "
        "nonqualifying or unavailable measurements"
    ),

    "normal_tissue_policy": (
        "Normal colon, kidney and skin staining "
        "flagged for biological and reliability review"
    ),

    "dependency_policy": (
        "CRISPR effects retained as descriptive "
        "functional evidence, not an ADC requirement"
    ),

    "stromal_policy": (
        "FAP assigned a separate stromal-associated "
        "targeting review lane"
    ),

    "candidate_exclusions": 0,

    "weighted_scoring_performed": False,

    "final_target_ranking_performed": False,

    "limitations": [

        "Surface-related annotations do not "
        "demonstrate antibody accessibility.",

        "Bulk tumor RNA and protein enrichment "
        "do not establish malignant-cell origin.",

        "CPTAC protein missingness is not "
        "interpreted as protein absence.",

        "HPA normal-tissue staining is not "
        "a clinical toxicity measurement.",

        "Normal-tissue review categories "
        "are not clinical safety categories.",

        "TCGA discovery Tier B is computationally "
        "low MSI, not independently confirmed MSS.",

        "DepMap colorectal models are not "
        "confirmed MSS by this analysis.",

        "No experimental target validation "
        "has been performed."

    ]
}

with open(
    META /
    "surfacetarget_phase5_evidence_triage_provenance.json",
    "w"
) as handle:

    json.dump(
        provenance,
        handle,
        indent=4
    )

print("\nResearch provenance saved.")


# ------------------------------------------------------------
# 18. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 5.3 SUMMARY")
print("=" * 65)

print("\nTotal candidates:", len(data))

print(
    "Stronger surface-related annotations:",
    int(stronger_surface.sum())
)

print(
    "Cross-omics-supported candidates:",
    int(protein.sum())
)

print(
    "Cross-omics-supported, stronger "
    "surface-related candidates:",
    len(surface_protein)
)

print(
    "Biological follow-up lanes:",
    len(lane_summary)
)

print("\nSaved files:")

print(
    "1. surfacetarget_phase5_evidence_triage_v1.csv"
)

print(
    "2. surfacetarget_phase5_followup_lane_summary.csv"
)

print(
    "3. surfacetarget_phase5_surface_candidate_review.csv"
)

print(
    "4. surfacetarget_phase5_surface_crossomics_review.csv"
)

print(
    "5. surfacetarget_phase5_protein_supported_review.csv"
)

print(
    "6. surfacetarget_phase5_leading_evidence_triage.csv"
)

print(
    "7. surfacetarget_phase5_evidence_triage_provenance.json"
)

print("\nSource evidence: PRESERVED")
print("Candidate count: VALIDATED")
print("No candidates excluded.")
print("No therapeutic efficacy scores assigned.")
print("No final target ranking performed.")

print("\nPATCH 5.3 COMPLETE")
print("=" * 65)

In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 5 — PATCH 5.4
#
# INTEGRATED CANDIDATE PRIORITIZATION
# AND RANKING SENSITIVITY ANALYSIS
#
# OBJECTIVES:
# 1. Validate Patch 5.3 evidence triage
# 2. Preserve all 948 research candidates
# 3. Construct quantitative RNA evidence measures
# 4. Preserve qualifying CPTAC evidence
# 5. Preserve normal-tissue expression concerns
# 6. Separate biological targeting categories
# 7. Calculate three transparent priority scenarios
# 8. Compare ranking sensitivity
# 9. Identify stable research follow-up candidates
# 10. Review leading genes separately by targeting lane
# 11. Save prioritization tables and provenance
# 12. Create a research checkpoint
#
# IMPORTANT:
# - Research-priority rankings, NOT therapeutic efficacy scores
# - No clinical safety predictions
# - No automatic gene exclusions
# - No single ranking across incompatible target mechanisms
# - Missing HPA evidence is NOT favorable safety evidence
# - Missing CPTAC evidence is NOT absent protein
# - CRISPR dependency is NOT required for ADC targeting
# - No experimentally validated targets are claimed
# ============================================================


# ------------------------------------------------------------
# 1. IMPORTS AND CONFIGURATION
# ------------------------------------------------------------

from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
import numpy as np

import hashlib
import json


ROOT = Path("/content/SurfaceTarget-CRC")

RESULTS = ROOT / "results/tables"
META = ROOT / "data/metadata"
CHECKPOINTS = ROOT / "checkpoints"

for folder in [
    RESULTS,
    META,
    CHECKPOINTS
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


SOURCE_PATH = (
    RESULTS /
    "surfacetarget_phase5_evidence_triage_v1.csv"
)

PROVENANCE_PATH = (
    META /
    "surfacetarget_phase5_evidence_triage_provenance.json"
)


LEADING_GENES = [
    "CDH3",
    "CLDN1",
    "FAP",
    "DPEP1",
    "SLC7A5",
    "SLC5A6",
    "TDGF1",
    "CTHRC1",
    "S100P",
    "KRT80",
    "CEMIP",
    "INHBA",
    "WNT2"
]


SURFACE_LANE = (
    "Putative cell-surface antigen review"
)

STROMAL_LANE = (
    "Stromal-associated targeting review"
)

MEMBRANE_REVIEW_LANE = (
    "Membrane localization/accessibility verification"
)

EXTRACELLULAR_LANE = (
    "Extracellular microenvironment review"
)

ALTERNATIVE_LANE = (
    "Topology clarification or alternative modality review"
)


EXPECTED_LANES = {
    SURFACE_LANE: 111,
    STROMAL_LANE: 1,
    MEMBRANE_REVIEW_LANE: 152,
    EXTRACELLULAR_LANE: 128,
    ALTERNATIVE_LANE: 556
}


print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 5 — PATCH 5.4")
print("INTEGRATED CANDIDATE PRIORITIZATION")
print("=" * 65)


# ------------------------------------------------------------
# 2. VALIDATE SOURCE FILES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING RESEARCH INPUTS")
print("-" * 65)


for path in [
    SOURCE_PATH,
    PROVENANCE_PATH
]:

    if not path.exists():

        raise FileNotFoundError(path)


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


with open(
    PROVENANCE_PATH,
    "r"
) as handle:

    previous_provenance = json.load(handle)


actual_hash = sha256_file(SOURCE_PATH)

expected_hash = previous_provenance.get(
    "output_sha256"
)


if not expected_hash:

    raise RuntimeError(
        "Patch 5.3 provenance has no output checksum."
    )


if actual_hash != expected_hash:

    raise RuntimeError(
        "Patch 5.3 triage table checksum mismatch."
    )


print("\nSource checksum: VERIFIED")


# ------------------------------------------------------------
# 3. LOAD AND VALIDATE RESEARCH CANDIDATES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING EVIDENCE TRIAGE DATA")
print("-" * 65)


source = pd.read_csv(
    SOURCE_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)


required = {
    "gene_id",
    "ensembl_stable_id",
    "gene_name",
    "phase5_followup_lane",
    "phase5_topology_class",
    "phase5_protein_evidence_state",
    "phase5_normal_tissue_review_state",
    "phase5_rna_evidence_pattern",
    "phase5_stronger_surface_annotation",
    "protein_enriched",
    "baseline_log2fc",
    "sensitivity_log2fc",
    "tier_a_median_cpm",
    "tier_a_fraction_cpm_ge1",
    "mss_complete_pairs",
    "phase5_normal_ihc_available",
    "depmap_crc_models_measured",
    "depmap_crc_median_gene_effect"
}


missing = required - set(source.columns)

if missing:

    raise RuntimeError(
        f"Required evidence columns missing: "
        f"{sorted(missing)}"
    )


if len(source) != 948:

    raise RuntimeError(
        "Expected 948 research candidates."
    )


if source["gene_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate research gene IDs."
    )


if source["gene_name"].duplicated().any():

    raise RuntimeError(
        "Duplicate research gene symbols."
    )


print("\nCandidate genes:", len(source))

print("Evidence fields:", len(source.columns))

print("Candidate identity: PASSED")


# ------------------------------------------------------------
# 4. VALIDATE BIOLOGICAL FOLLOW-UP LANES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING TARGETING CATEGORIES")
print("-" * 65)


observed_lanes = (
    source["phase5_followup_lane"]
    .value_counts()
    .to_dict()
)


if observed_lanes != EXPECTED_LANES:

    raise RuntimeError(
        "Biological follow-up categories differ "
        "from the validated Patch 5.3 results."
    )


print("\nFollow-up lane distribution:")

print(
    source["phase5_followup_lane"]
    .value_counts()
    .to_string()
)


print("\nBiological lane validation: PASSED")


# ------------------------------------------------------------
# 5. PREPARE ANALYSIS HELPERS
# ------------------------------------------------------------

def as_bool(series):

    values = (
        series.astype("string")
        .str.strip()
        .str.lower()
    )

    allowed = {
        "true",
        "false",
        "1",
        "0",
        "",
        "nan",
        "none",
        "<na>"
    }

    unexpected = (
        values.notna()
        & ~values.isin(list(allowed))
    )

    if unexpected.any():

        raise RuntimeError(
            "Unexpected boolean annotation: "
            f"{values[unexpected].unique().tolist()}"
        )

    return (
        values.isin(["true", "1"])
        .fillna(False)
        .astype(bool)
    )


def numeric(column):

    return pd.to_numeric(
        source[column],
        errors="coerce"
    )


data = source.copy(deep=True)


# ------------------------------------------------------------
# 6. QUANTITATIVE RNA EVIDENCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING TRANSCRIPTOMIC PRIORITIZATION EVIDENCE")
print("-" * 65)


baseline = numeric(
    "baseline_log2fc"
)

sensitivity = numeric(
    "sensitivity_log2fc"
)

tier_a_cpm = numeric(
    "tier_a_median_cpm"
)

tier_a_fraction = numeric(
    "tier_a_fraction_cpm_ge1"
)


if baseline.isna().any():

    raise RuntimeError(
        "Missing baseline RNA effect sizes."
    )


if sensitivity.isna().any():

    raise RuntimeError(
        "Missing sensitivity RNA effect sizes."
    )


if tier_a_fraction.isna().any():

    raise RuntimeError(
        "Missing Tier A RNA detection fractions."
    )


if not tier_a_fraction.between(
    0,
    1
).all():

    raise RuntimeError(
        "Tier A expression fractions outside [0,1]."
    )


# Conservative descriptive RNA magnitude:
#
# Use the smaller of the baseline and
# QC-screened log2 fold changes.
#
# This prevents a candidate from benefiting
# exclusively from a larger estimate in
# just one of the two related analyses.
#
# This is NOT an independent validation score.


data["phase5_rna_conservative_log2fc"] = (
    np.minimum(
        baseline.to_numpy(dtype=float),
        sensitivity.to_numpy(dtype=float)
    )
)


data["phase5_rna_tier_a_detection_fraction"] = (
    tier_a_fraction
)


data["phase5_rna_tier_a_median_cpm"] = (
    tier_a_cpm
)


rna_patterns = (
    data["phase5_rna_evidence_pattern"]
    .value_counts()
)


print("\nRNA evidence pattern distribution:")

print(
    rna_patterns.to_string()
)


if len(rna_patterns) == 1:

    print(
        "\nAll candidates share the same RNA "
        "evidence category."
    )

    print(
        "Continuous effect sizes, rather than "
        "the shared eligibility category, "
        "will distinguish candidates."
    )


print(
    "\nConservative log2FC range:",
    round(
        data[
            "phase5_rna_conservative_log2fc"
        ].min(),
        3
    ),
    "to",
    round(
        data[
            "phase5_rna_conservative_log2fc"
        ].max(),
        3
    )
)


# ------------------------------------------------------------
# 7. PREPARE QUALIFYING PROTEIN EVIDENCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING CPTAC PROTEIN EVIDENCE")
print("-" * 65)


protein_supported = as_bool(
    data["protein_enriched"]
)


if int(protein_supported.sum()) != 20:

    raise RuntimeError(
        "Expected 20 qualifying protein-enrichment genes."
    )


data["phase5_qualifying_protein_evidence"] = (
    protein_supported
)


# A qualifying enrichment result is rewarded
# as positive evidence.
#
# Missing protein measurements are NOT
# assigned a numerical penalty.
#
# The difference between inadequate coverage
# and adequately measured nonqualifying
# results remains visible in the source field.


data["phase5_protein_evidence_label"] = np.where(

    protein_supported,

    "Qualifying CPTAC enrichment observed",

    "Qualifying CPTAC enrichment not established"
)


print("\nQualifying protein-enrichment evidence:",
      int(protein_supported.sum()))


print(
    "Other candidates:",
    int((~protein_supported).sum())
)


print(
    "\nUnestablished enrichment is not "
    "interpreted as absent protein."
)


# ------------------------------------------------------------
# 8. PREPARE NORMAL-TISSUE REVIEW CATEGORIES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING NORMAL-TISSUE EVIDENCE")
print("-" * 65)


normal_state_order = {

    (
        "No medium/high focus-organ signal "
        "in available summary; not evidence of safety"
    ): 0,

    (
        "Reliable medium/high staining "
        "reported in other normal tissues"
    ): 1,

    (
        "Medium/high staining in focus organ; "
        "reliability requires review"
    ): 2,

    (
        "Medium/high staining in focus organ "
        "with stronger IHC reliability"
    ): 3,

    "Normal-tissue IHC unavailable": 4

}


unexpected_states = (
    set(data["phase5_normal_tissue_review_state"])
    - set(normal_state_order)
)


if unexpected_states:

    raise RuntimeError(
        "Unrecognized normal-tissue review states: "
        f"{unexpected_states}"
    )


data["phase5_normal_review_order"] = (
    data["phase5_normal_tissue_review_state"]
    .map(normal_state_order)
    .astype(int)
)


normal_available = as_bool(
    data["phase5_normal_ihc_available"]
)


if int(normal_available.sum()) != 690:

    raise RuntimeError(
        "Expected 690 candidates with normal-IHC evidence."
    )


# The ordinal review order is only a
# scenario-specific research follow-up rule.
#
# It is NOT a toxicity score.
#
# Unknown normal-tissue expression is placed
# last in the normal-review-first scenario
# to prevent missing evidence from being
# interpreted as favorable selectivity.


print("\nNormal-tissue category distribution:")

print(
    data["phase5_normal_tissue_review_state"]
    .value_counts()
    .to_string()
)


print(
    "\nUnknown IHC evidence is kept separate "
    "from observed low staining."
)


# ------------------------------------------------------------
# 9. VALIDATE SURFACE-RELATED EVIDENCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING SURFACE-RELATED CANDIDATES")
print("-" * 65)


stronger_surface = as_bool(
    data["phase5_stronger_surface_annotation"]
)


if int(stronger_surface.sum()) != 112:

    raise RuntimeError(
        "Expected 112 stronger surface annotations."
    )


surface_mask = (
    data["phase5_followup_lane"]
    .eq(SURFACE_LANE)
)


stromal_mask = (
    data["phase5_followup_lane"]
    .eq(STROMAL_LANE)
)


if int(surface_mask.sum()) != 111:

    raise RuntimeError(
        "Expected 111 putative cell-surface candidates."
    )


if int(stromal_mask.sum()) != 1:

    raise RuntimeError(
        "Expected one stromal-review candidate."
    )


if not stronger_surface[
    surface_mask | stromal_mask
].all():

    raise RuntimeError(
        "Surface-review lane contains candidates "
        "without the expected topology annotation."
    )


print("\nPutative cell-surface candidates:", 111)

print("Stromal-associated candidates:", 1)

print(
    "Qualifying CPTAC-supported surface candidates:",
    int(
        (
            surface_mask
            & protein_supported
        ).sum()
    )
)


print(
    "Qualifying CPTAC-supported stromal candidates:",
    int(
        (
            stromal_mask
            & protein_supported
        ).sum()
    )
)


# ------------------------------------------------------------
# 10. DEFINE TRANSPARENT PRIORITIZATION SCENARIOS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("DEFINING PRIORITIZATION SCENARIOS")
print("-" * 65)


# We do not combine the evidence into
# a single weighted numerical score.
#
# Instead, each scenario uses a transparent
# ordering of the same observed variables.
#
# Ranking is performed separately inside
# each biological follow-up lane.


SCENARIOS = {

    "protein_first": {

        "columns": [
            "phase5_qualifying_protein_evidence",
            "phase5_normal_review_order",
            "phase5_rna_conservative_log2fc",
            "phase5_rna_tier_a_detection_fraction",
            "gene_id"
        ],

        "ascending": [
            False,
            True,
            False,
            False,
            True
        ],

        "description": (
            "Qualifying protein enrichment first; "
            "then normal-tissue review status; "
            "then conservative RNA effect size"
        )
    },

    "normal_review_first": {

        "columns": [
            "phase5_normal_review_order",
            "phase5_qualifying_protein_evidence",
            "phase5_rna_conservative_log2fc",
            "phase5_rna_tier_a_detection_fraction",
            "gene_id"
        ],

        "ascending": [
            True,
            False,
            False,
            False,
            True
        ],

        "description": (
            "Normal-tissue expression review first; "
            "then qualifying protein evidence; "
            "then conservative RNA effect size"
        )
    },

    "rna_discovery_first": {

        "columns": [
            "phase5_rna_conservative_log2fc",
            "phase5_rna_tier_a_detection_fraction",
            "phase5_qualifying_protein_evidence",
            "phase5_normal_review_order",
            "gene_id"
        ],

        "ascending": [
            False,
            False,
            False,
            True,
            True
        ],

        "description": (
            "Conservative RNA enrichment first; "
            "then Tier A detection breadth; "
            "then protein and normal-tissue context"
        )
    }

}


for name, settings in SCENARIOS.items():

    print("\nScenario:", name)

    print(settings["description"])


print("\nIMPORTANT:")

print(
    "These are research review priorities, "
    "not clinical target scores."
)

print(
    "The normal-review scenario evaluates "
    "annotation concerns, not actual toxicity."
)


# ------------------------------------------------------------
# 11. CALCULATE WITHIN-LANE SCENARIO RANKS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CALCULATING BIOLOGICAL LANE-SPECIFIC RANKS")
print("-" * 65)


for scenario_name, settings in SCENARIOS.items():

    ranking_lookup = {}

    for lane, group in data.groupby(
        "phase5_followup_lane",
        sort=True
    ):

        ordered = group.sort_values(

            by=settings["columns"],

            ascending=settings["ascending"],

            na_position="last",

            kind="mergesort"

        )


        for rank, gene_id in enumerate(
            ordered["gene_id"].tolist(),
            start=1
        ):

            if gene_id in ranking_lookup:

                raise RuntimeError(
                    "Gene assigned multiple scenario ranks."
                )

            ranking_lookup[gene_id] = rank


    if len(ranking_lookup) != 948:

        raise RuntimeError(
            "Not all genes received a scenario rank."
        )


    output_column = (
        f"phase5_rank_{scenario_name}"
    )


    data[output_column] = (
        data["gene_id"]
        .map(ranking_lookup)
        .astype(int)
    )


print("\nThree ranking scenarios calculated.")


# ------------------------------------------------------------
# 12. VALIDATE WITHIN-LANE RANK UNIQUENESS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING SCENARIO RANKINGS")
print("-" * 65)


rank_columns = [
    f"phase5_rank_{name}"
    for name in SCENARIOS
]


for lane, group in data.groupby(
    "phase5_followup_lane"
):

    expected_ranks = set(
        range(1, len(group) + 1)
    )

    for column in rank_columns:

        observed_ranks = set(
            group[column].astype(int)
        )

        if observed_ranks != expected_ranks:

            raise RuntimeError(
                "Invalid within-lane ranks: "
                f"{lane}, {column}"
            )


print("\nWithin-lane ranking uniqueness: PASSED")

print(
    "No cross-lane therapeutic hierarchy "
    "has been imposed."
)


# ------------------------------------------------------------
# 13. SUMMARIZE RANKING SENSITIVITY
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CALCULATING RANKING SENSITIVITY")
print("-" * 65)


data["phase5_median_scenario_rank"] = (
    data[rank_columns]
    .median(axis=1)
)


data["phase5_best_scenario_rank"] = (
    data[rank_columns]
    .min(axis=1)
)


data["phase5_worst_scenario_rank"] = (
    data[rank_columns]
    .max(axis=1)
)


data["phase5_rank_range"] = (

    data["phase5_worst_scenario_rank"]
    -
    data["phase5_best_scenario_rank"]

)


# The top-20 threshold is a convenient
# shortlist size for research review.
#
# It is not a biological threshold.


TOP_N = 20


data["phase5_scenarios_in_top20"] = (
    data[rank_columns]
    .le(TOP_N)
    .sum(axis=1)
)


data["phase5_consistent_top20_review"] = (
    data["phase5_scenarios_in_top20"]
    .ge(2)
)


print(
    "\nCandidates appearing in the top 20 "
    "of at least two scenarios, by lane:"
)


print(

    data.groupby(
        "phase5_followup_lane"
    )["phase5_consistent_top20_review"]

    .sum()

    .to_string()

)


print("\nIMPORTANT:")

print(
    "Median scenario rank summarizes robustness "
    "to three explicit ordering assumptions."
)

print(
    "It is not a therapeutic efficacy "
    "or clinical development score."
)


# ------------------------------------------------------------
# 14. PREPARE THE PRIMARY SURFACE SHORTLIST
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PRIMARY CELL-SURFACE CANDIDATE SHORTLIST")
print("-" * 65)


surface = data[
    data["phase5_followup_lane"]
    .eq(SURFACE_LANE)
].copy()


surface = surface.sort_values(

    [
        "phase5_median_scenario_rank",
        "phase5_rank_range",
        "gene_id"
    ],

    ascending=[
        True,
        True,
        True
    ]

)


if len(surface) != 111:

    raise RuntimeError(
        "Surface candidate inventory changed."
    )


surface["phase5_surface_consensus_position"] = (
    np.arange(
        1,
        len(surface) + 1
    )
)


surface_top20 = surface.head(
    TOP_N
).copy()


print("\nTop 20 exploratory surface-review candidates:")


display_columns = [
    "gene_name",
    "phase5_surface_consensus_position",
    "phase5_rank_protein_first",
    "phase5_rank_normal_review_first",
    "phase5_rank_rna_discovery_first",
    "phase5_scenarios_in_top20",
    "phase5_qualifying_protein_evidence",
    "phase5_normal_tissue_review_state",
    "phase5_rna_conservative_log2fc"
]


print("\n")

print(

    surface_top20[
        display_columns
    ]

    .to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )

)


# ------------------------------------------------------------
# 15. REVIEW SHORTLIST STABILITY
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PRIMARY SURFACE RANKING STABILITY")
print("-" * 65)


scenario_top_sets = {}


for scenario_name in SCENARIOS:

    rank_column = (
        f"phase5_rank_{scenario_name}"
    )

    scenario_top_sets[scenario_name] = set(

        surface.loc[
            surface[rank_column].le(TOP_N),
            "gene_id"
        ]

    )


stability_records = []


scenario_names = list(
    SCENARIOS.keys()
)


for i in range(len(scenario_names)):

    for j in range(i + 1, len(scenario_names)):

        first = scenario_names[i]

        second = scenario_names[j]

        first_set = scenario_top_sets[first]

        second_set = scenario_top_sets[second]

        overlap = len(
            first_set & second_set
        )

        union = len(
            first_set | second_set
        )

        jaccard = (
            overlap / union
            if union > 0
            else np.nan
        )

        stability_records.append({

            "scenario_a": first,

            "scenario_b": second,

            "top_n": TOP_N,

            "shared_candidates": overlap,

            "jaccard_similarity": jaccard

        })


stability = pd.DataFrame(
    stability_records
)


print("\nPairwise top-20 overlap:")

print(
    stability.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)


stable_surface = surface[
    surface["phase5_consistent_top20_review"]
].copy()


print(
    "\nSurface candidates in top 20 "
    "under at least two scenarios:",
    len(stable_surface)
)


print("\nStable exploratory follow-up genes:")


if not stable_surface.empty:

    print(

        stable_surface[
            [
                "gene_name",
                "phase5_scenarios_in_top20",
                "phase5_median_scenario_rank",
                "phase5_qualifying_protein_evidence",
                "phase5_normal_tissue_review_state"
            ]
        ]

        .to_string(
            index=False
        )

    )

else:

    print(
        "No genes meet the specified "
        "ranking-consistency criterion."
    )


# ------------------------------------------------------------
# 16. REVIEW STROMAL CANDIDATES SEPARATELY
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("STROMAL TARGETING REVIEW")
print("-" * 65)


stromal = data[
    data["phase5_followup_lane"]
    .eq(STROMAL_LANE)
].copy()


if len(stromal) != 1:

    raise RuntimeError(
        "Expected one stromal-associated candidate."
    )


print("\n")

print(
    stromal[
        [
            "gene_name",
            "phase5_protein_evidence_state",
            "phase5_normal_tissue_review_state",
            "phase5_rna_conservative_log2fc",
            "depmap_crc_median_gene_effect"
        ]
    ].to_string(index=False)
)


print(
    "\nStromal and tumor-cell antigen "
    "ranks must not be compared directly."
)


# ------------------------------------------------------------
# 17. REVIEW OTHER BIOLOGICAL LANES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("OTHER BIOLOGICAL FOLLOW-UP CATEGORIES")
print("-" * 65)


other_lane_records = []


for lane in [
    MEMBRANE_REVIEW_LANE,
    EXTRACELLULAR_LANE,
    ALTERNATIVE_LANE
]:

    subset = data[
        data["phase5_followup_lane"]
        .eq(lane)
    ].copy()

    subset = subset.sort_values(
        [
            "phase5_median_scenario_rank",
            "phase5_rank_range",
            "gene_id"
        ]
    )

    other_lane_records.append(
        subset
    )

    print("\n" + "-" * 50)

    print("Lane:", lane)

    print("Candidates:", len(subset))

    print("\nFirst 10 exploratory review candidates:")

    print(

        subset[
            [
                "gene_name",
                "phase5_median_scenario_rank",
                "phase5_qualifying_protein_evidence",
                "phase5_normal_tissue_review_state"
            ]
        ]

        .head(10)

        .to_string(index=False)

    )


other_lanes = pd.concat(
    other_lane_records,
    ignore_index=True
)


# ------------------------------------------------------------
# 18. REVIEW LEADING BIOLOGICAL CANDIDATES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LEADING CANDIDATE SENSITIVITY REVIEW")
print("-" * 65)


leading = data[
    data["gene_name"].isin(
        LEADING_GENES
    )
].copy()


leading["display_order"] = (
    leading["gene_name"]
    .map({
        gene: index
        for index, gene in enumerate(
            LEADING_GENES
        )
    })
)


leading = leading.sort_values(
    "display_order"
)


leading_columns = [
    "gene_name",
    "phase5_followup_lane",
    "phase5_qualifying_protein_evidence",
    "phase5_normal_tissue_review_state",
    "phase5_rna_conservative_log2fc",
    "phase5_rank_protein_first",
    "phase5_rank_normal_review_first",
    "phase5_rank_rna_discovery_first",
    "phase5_median_scenario_rank"
]


print("\n")

print(

    leading[
        leading_columns
    ]

    .to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )

)


print("\nINTERPRETATION:")

print(
    "Ranks are comparable only within "
    "the same biological follow-up lane."
)

print(
    "A candidate with missing normal-IHC "
    "evidence is not assumed to be selective."
)

print(
    "A candidate with weak CRISPR dependency "
    "is not automatically excluded."
)


# ------------------------------------------------------------
# 19. VALIDATE SOURCE EVIDENCE PRESERVATION
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING SOURCE EVIDENCE PRESERVATION")
print("-" * 65)


original_columns = source.columns.tolist()


if not source.equals(
    data[original_columns]
):

    raise RuntimeError(
        "Source evidence changed during prioritization."
    )


if len(data) != 948:

    raise RuntimeError(
        "Candidate count changed."
    )


if data["gene_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate candidate identifiers."
    )


if int(
    data["phase5_qualifying_protein_evidence"]
    .sum()
) != 20:

    raise RuntimeError(
        "Qualifying protein-support count changed."
    )


if int(
    stronger_surface.sum()
) != 112:

    raise RuntimeError(
        "Surface-topology evidence count changed."
    )


print("\nOriginal evidence: UNCHANGED")

print("Candidate preservation: PASSED")

print("Protein support: PASSED")

print("Surface topology: PASSED")


# ------------------------------------------------------------
# 20. PREPARE FINAL REVIEW TABLES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING PRIORITIZATION OUTPUTS")
print("-" * 65)


# Full research table.
#
# These ranks are exploratory and are
# explicitly limited to their own lanes.


ranked = data.copy()


# Add surface consensus position
# without assigning artificial positions
# to candidates in other biological lanes.


surface_positions = dict(
    zip(
        surface["gene_id"],
        surface["phase5_surface_consensus_position"]
    )
)


ranked["phase5_surface_consensus_position"] = (
    ranked["gene_id"]
    .map(surface_positions)
)


if int(
    ranked["phase5_surface_consensus_position"]
    .notna()
    .sum()
) != 111:

    raise RuntimeError(
        "Surface ranking coverage mismatch."
    )


# Scenario methodology table.

scenario_methodology = pd.DataFrame([

    {
        "scenario": name,

        "ordering_columns":
            "; ".join(settings["columns"]),

        "ascending_flags":
            "; ".join(
                str(value)
                for value in settings["ascending"]
            ),

        "interpretation":
            settings["description"]

    }

    for name, settings in SCENARIOS.items()

])


# Research review categories are never
# labeled as safe/unsafe or clinically
# recommended/not recommended.


# ------------------------------------------------------------
# 21. SAVE PRIORITIZATION RESULTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING PRIORITIZATION RESULTS")
print("-" * 65)


OUTPUT_PATH = (
    RESULTS /
    "surfacetarget_phase5_prioritization_v1.csv"
)


ranked.to_csv(
    OUTPUT_PATH,
    index=False
)


surface.to_csv(

    RESULTS /
    "surfacetarget_phase5_surface_ranked_candidates.csv",

    index=False
)


surface_top20.to_csv(

    RESULTS /
    "surfacetarget_phase5_surface_top20_exploratory.csv",

    index=False
)


stable_surface.to_csv(

    RESULTS /
    "surfacetarget_phase5_surface_stable_followup.csv",

    index=False
)


stromal.to_csv(

    RESULTS /
    "surfacetarget_phase5_stromal_target_review.csv",

    index=False
)


other_lanes.to_csv(

    RESULTS /
    "surfacetarget_phase5_alternative_lane_priorities.csv",

    index=False
)


leading.to_csv(

    RESULTS /
    "surfacetarget_phase5_leading_rank_sensitivity.csv",

    index=False
)


stability.to_csv(

    RESULTS /
    "surfacetarget_phase5_ranking_stability.csv",

    index=False
)


scenario_methodology.to_csv(

    RESULTS /
    "surfacetarget_phase5_ranking_scenarios.csv",

    index=False
)


# Reopen and validate the main output.

reloaded = pd.read_csv(
    OUTPUT_PATH,
    dtype={"gene_id": str},
    low_memory=False
)


if len(reloaded) != 948:

    raise RuntimeError(
        "Saved prioritization table lost candidates."
    )


if (
    reloaded["gene_id"].tolist()
    != ranked["gene_id"].tolist()
):

    raise RuntimeError(
        "Saved candidate identifiers changed."
    )


print("\nPrioritization tables: SAVED")

print("Saved table validation: PASSED")


# ------------------------------------------------------------
# 22. SAVE RESEARCH PROVENANCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING PRIORITIZATION PROVENANCE")
print("-" * 65)


provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "5.4",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": str(SOURCE_PATH),

    "source_sha256": actual_hash,

    "output": str(OUTPUT_PATH),

    "output_sha256":
        sha256_file(OUTPUT_PATH),

    "candidate_count": 948,

    "surface_lane_candidates": 111,

    "stromal_lane_candidates": 1,

    "stronger_surface_annotations": 112,

    "qualifying_cptac_enrichment_genes": 20,

    "ranking_method": (
        "Three lexicographic research-priority "
        "scenarios within biological follow-up lanes"
    ),

    "scenarios": {

        name: {
            "ordering_columns": settings["columns"],
            "ascending": settings["ascending"],
            "description": settings["description"]
        }

        for name, settings in SCENARIOS.items()

    },

    "rna_metric": (
        "Minimum of baseline and QC-screened "
        "tumor-vs-normal log2 fold changes"
    ),

    "rna_independence_note": (
        "Baseline and QC-screened analyses "
        "are related sensitivity analyses, "
        "not independent experiments"
    ),

    "protein_policy": (
        "Qualifying enrichment treated as "
        "supporting evidence; absence of qualifying "
        "enrichment is not interpreted as absent protein"
    ),

    "normal_tissue_policy": (
        "Missing IHC data assigned a distinct "
        "unassessed category, never a favorable "
        "selectivity classification"
    ),

    "normal_review_order_interpretation": (
        "Scenario-specific annotation-review priority; "
        "not a quantitative clinical toxicity estimate"
    ),

    "dependency_policy": (
        "DepMap CRISPR gene effects retained "
        "as context but not used to determine "
        "antibody-antigen prioritization"
    ),

    "cross_lane_rank_comparison_allowed": False,

    "top_n_for_sensitivity": TOP_N,

    "stable_followup_rule": (
        "Within the same lane, appears in the top "
        "20 under at least two of three scenarios"
    ),

    "candidate_exclusions": 0,

    "therapeutic_efficacy_scores": False,

    "clinical_safety_predictions": False,

    "experimentally_validated_targets": False,

    "limitations": [

        "The research-priority ordering depends "
        "on selected evidence hierarchy assumptions.",

        "A high ranking does not imply a "
        "clinically favorable therapeutic window.",

        "Normal-tissue IHC measurements "
        "are not direct toxicity assessments.",

        "Missing normal-tissue IHC evidence "
        "does not imply low expression.",

        "Bulk tumor RNA and proteomics do "
        "not establish malignant-cell origin.",

        "CPTAC enrichment is exploratory "
        "and based on measured protein coverage.",

        "GPI anchoring and extracellular topology "
        "do not establish antibody accessibility.",

        "CRISPR dependency is not required "
        "for antibody-drug conjugate targeting.",

        "The DepMap colorectal cohort "
        "has not been confirmed MSS.",

        "TCGA Tier B is computationally "
        "low-MSI rather than independently "
        "confirmed MSS.",

        "No target has been experimentally validated."

    ]

}


with open(

    META /
    "surfacetarget_phase5_prioritization_provenance.json",

    "w"

) as handle:

    json.dump(
        provenance,
        handle,
        indent=4
    )


print("\nResearch provenance: SAVED")


# ------------------------------------------------------------
# 23. CREATE RESEARCH CHECKPOINT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING RESEARCH CHECKPOINT")
print("-" * 65)


timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S"
)


checkpoint = (

    CHECKPOINTS /

    f"SurfaceTarget_CRC_Phase5_Patch54_{timestamp}.zip"

)


# Preserve analysis results, provenance
# and compact research sources.
#
# Exclude raw TCGA and reconstructed
# DepMap molecular datasets.


backup_folders = [
    "data/metadata",
    "data/processed",
    "data/raw/cptac_colon",
    "data/raw/hpa",
    "data/raw/uniprot",
    "data/raw/hgnc",
    "results/tables",
    "results/figures",
    "src",
    "reports",
    "logs"
]


with ZipFile(
    checkpoint,
    "w",
    ZIP_DEFLATED,
    compresslevel=3
) as archive:

    for folder in backup_folders:

        folder_path = ROOT / folder

        if not folder_path.exists():
            continue

        for file_path in folder_path.rglob("*"):

            if file_path.is_file():

                archive.write(

                    file_path,

                    arcname=str(
                        file_path.relative_to(ROOT)
                    )

                )


checkpoint_size_mib = (
    checkpoint.stat().st_size
    / (1024 ** 2)
)


print("\nCheckpoint created.")

print(
    "Size:",
    f"{checkpoint_size_mib:.2f} MiB"
)

print("Location:")
print(checkpoint)


# ------------------------------------------------------------
# 24. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 5.4 SUMMARY")
print("=" * 65)


print("\nResearch candidates:", len(ranked))


print(
    "Primary surface-review candidates:",
    len(surface)
)


print(
    "Stromal-review candidates:",
    len(stromal)
)


print(
    "Qualifying CPTAC-supported genes:",
    int(protein_supported.sum())
)


print(
    "Ranking scenarios:",
    len(SCENARIOS)
)


print(
    "Stable surface follow-up candidates:",
    len(stable_surface)
)


print("\nSaved outputs:")

print("1. surfacetarget_phase5_prioritization_v1.csv")

print("2. surfacetarget_phase5_surface_ranked_candidates.csv")

print("3. surfacetarget_phase5_surface_top20_exploratory.csv")

print("4. surfacetarget_phase5_surface_stable_followup.csv")

print("5. surfacetarget_phase5_stromal_target_review.csv")

print("6. surfacetarget_phase5_alternative_lane_priorities.csv")

print("7. surfacetarget_phase5_leading_rank_sensitivity.csv")

print("8. surfacetarget_phase5_ranking_stability.csv")

print("9. surfacetarget_phase5_ranking_scenarios.csv")

print("10. surfacetarget_phase5_prioritization_provenance.json")


print("\nSource evidence: VERIFIED UNCHANGED")

print("Biological lane separation: PRESERVED")

print("Ranking sensitivity: COMPLETE")

print("Research checkpoint: CREATED")


print(
    "Checkpoint size:",
    f"{checkpoint_size_mib:.2f} MiB"
)


print("\nNo candidates excluded.")

print("No clinical safety scores assigned.")

print("No therapeutic efficacy claims.")

print("No experimentally validated targets claimed.")


print("\nPATCH 5.4 COMPLETE")

print("=" * 65)

In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 5 — PATCH 5.5
#
# RANKING ROBUSTNESS AND CANDIDATE EVIDENCE DOSSIERS
#
# Objectives:
# 1. Validate Patch 5.4 ranking results
# 2. Audit correlations between ranking scenarios
# 3. Compare top-10, top-20 and top-30 membership
# 4. Distinguish 2-scenario from 3-scenario agreement
# 5. Audit normal-tissue IHC across all available tissues
# 6. Prepare candidate-specific evidence dossiers
# 7. Identify experimental validation priorities
# 8. Preserve original 948-gene research universe
# 9. Save diagnostic results and provenance
# 10. Create a research checkpoint
#
# IMPORTANT:
# - No therapeutic efficacy scores
# - No clinical safety predictions
# - No additional ranking weights
# - No source data overwritten
# - No candidates excluded
# - No malignant-cell localization inferred from bulk data
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
import numpy as np
import hashlib
import json


# ------------------------------------------------------------
# 1. PROJECT CONFIGURATION
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

RESULTS = ROOT / "results/tables"
META = ROOT / "data/metadata"
CHECKPOINTS = ROOT / "checkpoints"

for folder in [RESULTS, META, CHECKPOINTS]:
    folder.mkdir(parents=True, exist_ok=True)


RANKING_PATH = (
    RESULTS /
    "surfacetarget_phase5_prioritization_v1.csv"
)

PROVENANCE_PATH = (
    META /
    "surfacetarget_phase5_prioritization_provenance.json"
)

TISSUE_PATH = (
    RESULTS /
    "hpa_candidate_tissue_level_profiles.csv"
)


SURFACE_LANE = "Putative cell-surface antigen review"

STROMAL_LANE = "Stromal-associated targeting review"

MEMBRANE_LANE = (
    "Membrane localization/accessibility verification"
)

EXTRACELLULAR_LANE = (
    "Extracellular microenvironment review"
)


SCENARIO_COLUMNS = {
    "protein_first": "phase5_rank_protein_first",
    "normal_review_first": "phase5_rank_normal_review_first",
    "rna_discovery_first": "phase5_rank_rna_discovery_first"
}


REVIEW_GENES = [
    "CA9",
    "ADAM12",
    "NPC1L1",
    "LGR5",
    "CDH3",
    "CLDN1",
    "FAP",
    "DPEP1",
    "SLC7A5",
    "SLC5A6",
    "TDGF1",
    "CEMIP",
    "CTHRC1",
    "MXRA5",
    "SFRP4"
]


print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 5 — PATCH 5.5")
print("RANKING ROBUSTNESS AND EVIDENCE DOSSIERS")
print("=" * 65)


# ------------------------------------------------------------
# 2. SOURCE INTEGRITY VALIDATION
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING RESEARCH INPUTS")
print("-" * 65)


for path in [
    RANKING_PATH,
    PROVENANCE_PATH,
    TISSUE_PATH
]:
    if not path.exists():
        raise FileNotFoundError(path)


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


with open(PROVENANCE_PATH, "r") as handle:
    provenance = json.load(handle)


expected_hash = provenance.get("output_sha256")
actual_hash = sha256_file(RANKING_PATH)

if not expected_hash:
    raise RuntimeError(
        "Missing Patch 5.4 output checksum."
    )

if expected_hash != actual_hash:
    raise RuntimeError(
        "Patch 5.4 ranking checksum mismatch."
    )


print("\nRanking source checksum: VERIFIED")


# ------------------------------------------------------------
# 3. LOAD CANONICAL RANKING DATA
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING EXISTING CANDIDATE PRIORITIES")
print("-" * 65)


source = pd.read_csv(
    RANKING_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)


if len(source) != 948:
    raise RuntimeError(
        "Expected 948 research candidates."
    )

if source["gene_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate research gene identifiers."
    )

if source["ensembl_stable_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate Ensembl stable identifiers."
    )


required = {
    "gene_id",
    "ensembl_stable_id",
    "gene_name",
    "phase5_followup_lane",
    "phase5_topology_class",
    "phase5_protein_evidence_state",
    "phase5_normal_tissue_review_state",
    "phase5_normal_ihc_available",
    "phase5_rna_conservative_log2fc",
    "phase5_median_scenario_rank",
    "phase5_surface_consensus_position",
    "baseline_log2fc",
    "sensitivity_log2fc",
    "mss_complete_pairs",
    "protein_enriched",
    "depmap_crc_median_gene_effect",
    *SCENARIO_COLUMNS.values()
}


missing = required - set(source.columns)

if missing:
    raise RuntimeError(
        f"Missing required fields: {sorted(missing)}"
    )


surface = source[
    source["phase5_followup_lane"].eq(SURFACE_LANE)
].copy()


if len(surface) != 111:
    raise RuntimeError(
        "Expected 111 surface-review candidates."
    )


for column in SCENARIO_COLUMNS.values():

    ranks = pd.to_numeric(
        surface[column],
        errors="raise"
    ).astype(int)

    if set(ranks) != set(range(1, 112)):
        raise RuntimeError(
            f"Invalid scenario rank inventory: {column}"
        )


print("\nAll research candidates:", len(source))
print("Surface-review candidates:", len(surface))
print("Ranking scenarios:", len(SCENARIO_COLUMNS))
print("Ranking source validation: PASSED")


# ------------------------------------------------------------
# 4. RANK CORRELATION DIAGNOSTICS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CALCULATING SCENARIO RANK CORRELATIONS")
print("-" * 65)


rank_data = surface[
    list(SCENARIO_COLUMNS.values())
].copy()


rank_correlations = rank_data.corr(
    method="spearman"
)


rank_correlations.index = list(
    SCENARIO_COLUMNS.keys()
)

rank_correlations.columns = list(
    SCENARIO_COLUMNS.keys()
)


print("\nSpearman rank correlations:")

print(
    rank_correlations.to_string(
        float_format=lambda x: f"{x:.4f}"
    )
)


print(
    "\nThese correlations describe agreement "
    "between selected prioritization rules."
)

print(
    "They do not establish biological "
    "or therapeutic validation."
)


# ------------------------------------------------------------
# 5. TOP-N SCENARIO OVERLAP
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("AUDITING TOP-N RANKING OVERLAP")
print("-" * 65)


TOP_N_VALUES = [10, 20, 30]

overlap_records = []
consensus_records = []

scenario_names = list(
    SCENARIO_COLUMNS.keys()
)


for top_n in TOP_N_VALUES:

    top_sets = {}

    for name, column in SCENARIO_COLUMNS.items():

        top_sets[name] = set(
            surface.loc[
                surface[column] <= top_n,
                "gene_id"
            ]
        )

    for i in range(len(scenario_names)):

        for j in range(i + 1, len(scenario_names)):

            a = scenario_names[i]
            b = scenario_names[j]

            intersection = (
                top_sets[a] & top_sets[b]
            )

            union = (
                top_sets[a] | top_sets[b]
            )

            overlap_records.append({

                "top_n": top_n,
                "scenario_a": a,
                "scenario_b": b,
                "shared_candidates": len(intersection),
                "jaccard_similarity": (
                    len(intersection) / len(union)
                    if union
                    else np.nan
                )
            })


    # Distinguish agreement under all three
    # scenarios from agreement under only two.

    all_three = set.intersection(
        *top_sets.values()
    )

    at_least_two = {
        gene_id
        for gene_id in surface["gene_id"]
        if sum(
            gene_id in selected
            for selected in top_sets.values()
        ) >= 2
    }

    consensus_records.append({

        "top_n": top_n,

        "all_three_scenarios":
            len(all_three),

        "at_least_two_scenarios":
            len(at_least_two),

        "two_but_not_three":
            len(at_least_two - all_three)
    })


overlap = pd.DataFrame(overlap_records)

consensus = pd.DataFrame(consensus_records)


print("\nPairwise overlap:")

print(
    overlap.to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)


print("\nAgreement across scenarios:")

print(
    consensus.to_string(index=False)
)


print("\nInterpretation:")

print(
    "Agreement under two strongly correlated "
    "scenarios is not equivalent to agreement "
    "under three independent evidence assumptions."
)


# ------------------------------------------------------------
# 6. IDENTIFY ALL-THREE TOP-20 CANDIDATES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("IDENTIFYING THREE-SCENARIO AGREEMENT")
print("-" * 65)


three_way_mask = np.logical_and.reduce([

    surface[column].le(20).to_numpy()

    for column in SCENARIO_COLUMNS.values()

])


three_way = surface.loc[
    three_way_mask
].copy()


three_way = three_way.sort_values(
    "phase5_median_scenario_rank"
)


print("\nCandidates appearing in all three top-20 lists:")

if not three_way.empty:

    print(
        three_way[
            [
                "gene_name",
                *SCENARIO_COLUMNS.values(),
                "protein_enriched",
                "phase5_normal_tissue_review_state"
            ]
        ].to_string(index=False)
    )

else:

    print("None.")


print(
    "\nThree-scenario agreement remains "
    "a ranking-consistency observation, "
    "not experimental confirmation."
)


# ------------------------------------------------------------
# 7. LOAD ALL HPA NORMAL-TISSUE PROFILES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("AUDITING ALL AVAILABLE NORMAL TISSUES")
print("-" * 65)


tissues = pd.read_csv(
    TISSUE_PATH,
    dtype={
        "Gene": str,
        "Tissue": str
    },
    low_memory=False
)


required_tissue_columns = {
    "Gene",
    "Tissue",
    "max_level_score",
    "max_reliable_score"
}


if not required_tissue_columns.issubset(
    tissues.columns
):

    raise RuntimeError(
        "Unexpected HPA tissue-profile schema."
    )


if tissues[
    ["Gene", "Tissue"]
].duplicated().any():

    raise RuntimeError(
        "Duplicate HPA gene-tissue profiles."
    )


if not set(tissues["Gene"]).issubset(
    set(source["ensembl_stable_id"])
):

    raise RuntimeError(
        "Unexpected Ensembl gene identifiers "
        "in HPA tissue profiles."
    )


for column in [
    "max_level_score",
    "max_reliable_score"
]:

    tissues[column] = pd.to_numeric(
        tissues[column],
        errors="coerce"
    )


print("\nGene-tissue profiles:", len(tissues))
print("Normal tissues:", tissues["Tissue"].nunique())
print("Genes represented:", tissues["Gene"].nunique())


# ------------------------------------------------------------
# 8. SUMMARIZE NORMAL-TISSUE EXPRESSION
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING ALL-TISSUE IHC SUMMARIES")
print("-" * 65)


tissue_records = []


for ensembl_id, group in tissues.groupby(
    "Gene",
    sort=False
):

    measured = group[
        "max_level_score"
    ].notna()

    medium_high = group[
        "max_level_score"
    ].ge(2)

    high = group[
        "max_level_score"
    ].ge(3)

    reliable_medium_high = group[
        "max_reliable_score"
    ].ge(2)


    tissue_records.append({

        "ensembl_stable_id": ensembl_id,

        "phase55_normal_tissues_recorded":
            int(len(group)),

        "phase55_normal_tissues_interpretable":
            int(measured.sum()),

        "phase55_normal_tissues_medium_high":
            int(medium_high.sum()),

        "phase55_normal_tissues_high":
            int(high.sum()),

        "phase55_normal_tissues_reliable_medium_high":
            int(reliable_medium_high.sum()),

        "phase55_normal_medium_high_tissue_names":
            "; ".join(
                sorted(
                    group.loc[
                        medium_high,
                        "Tissue"
                    ].astype(str).tolist()
                )
            ),

        "phase55_normal_high_tissue_names":
            "; ".join(
                sorted(
                    group.loc[
                        high,
                        "Tissue"
                    ].astype(str).tolist()
                )
            ),

        "phase55_normal_reliable_medium_high_tissue_names":
            "; ".join(
                sorted(
                    group.loc[
                        reliable_medium_high,
                        "Tissue"
                    ].astype(str).tolist()
                )
            )
    })


tissue_summary = pd.DataFrame(
    tissue_records
)


if tissue_summary[
    "ensembl_stable_id"
].duplicated().any():

    raise RuntimeError(
        "Duplicate HPA gene summaries."
    )


print(
    "\nGenes with normal-tissue summaries:",
    len(tissue_summary)
)


# ------------------------------------------------------------
# 9. MERGE NORMAL-TISSUE AUDIT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("INTEGRATING ALL-TISSUE CONTEXT")
print("-" * 65)


annotated = source.merge(

    tissue_summary,

    on="ensembl_stable_id",

    how="left",

    validate="one_to_one"

)


if len(annotated) != 948:

    raise RuntimeError(
        "Gene count changed after HPA integration."
    )


if not annotated[
    source.columns
].equals(source):

    raise RuntimeError(
        "Existing research evidence changed."
    )


def parse_bool(series):

    values = (
        series.astype("string")
        .str.strip()
        .str.lower()
    )

    accepted = {
        "true", "false", "1", "0",
        "", "nan", "none", "<na>"
    }

    unexpected = (
        values.notna()
        & ~values.isin(list(accepted))
    )

    if unexpected.any():

        raise RuntimeError(
            "Unexpected boolean values: "
            f"{values[unexpected].unique().tolist()}"
        )

    return (
        values.isin(["true", "1"])
        .fillna(False)
        .astype(bool)
    )


ihc_available = parse_bool(
    annotated["phase5_normal_ihc_available"]
)


tissue_available = (
    annotated[
        "phase55_normal_tissues_recorded"
    ].notna()
)


mismatch = (
    ihc_available != tissue_available
)


print(
    "\nGenes with canonical normal-IHC evidence:",
    int(ihc_available.sum())
)

print(
    "Genes with tissue-profile records:",
    int(tissue_available.sum())
)

print(
    "Availability disagreements:",
    int(mismatch.sum())
)


if mismatch.any():

    print(
        "\nWARNING: Some IHC availability fields "
        "disagree with the tissue-profile inventory."
    )

    print(
        "These records require review before "
        "making tissue-selectivity claims."
    )


print(
    "\nMissing tissue profiles remain missing, "
    "not zero expression."
)


# ------------------------------------------------------------
# 10. PREPARE EXPERIMENTAL VALIDATION QUESTIONS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENERATING CANDIDATE VALIDATION PRIORITIES")
print("-" * 65)


def validation_questions(row):

    questions = []

    lane = str(
        row["phase5_followup_lane"]
    )

    protein_state = str(
        row["phase5_protein_evidence_state"]
    )

    normal_state = str(
        row["phase5_normal_tissue_review_state"]
    )


    if lane == STROMAL_LANE:

        questions.append(
            "Verify expression in tumor-associated "
            "fibroblasts versus malignant cells"
        )

    elif lane == SURFACE_LANE:

        questions.append(
            "Confirm malignant-cell surface expression "
            "using single-cell/spatial and protein assays"
        )

        questions.append(
            "Evaluate extracellular epitope accessibility"
        )

    elif lane == MEMBRANE_LANE:

        questions.append(
            "Verify plasma-membrane residency "
            "and extracellular topology"
        )

    elif lane == EXTRACELLULAR_LANE:

        questions.append(
            "Establish extracellular localization "
            "and appropriate targeting mechanism"
        )

    else:

        questions.append(
            "Clarify protein localization "
            "before antibody-target consideration"
        )


    if protein_state == (
        "Sparse paired-protein measurements"
    ):

        questions.append(
            "Acquire adequately covered tumor/normal "
            "protein measurements"
        )

    elif protein_state in [
        "Paired-protein coverage unavailable",
        "No complete MSS paired-protein measurement"
    ]:

        questions.append(
            "Obtain independent tumor/normal "
            "protein-level evidence"
        )

    elif protein_state.startswith(
        "Adequately covered"
    ):

        questions.append(
            "Review nonqualifying protein-enrichment "
            "results and assay detectability"
        )


    if normal_state == (
        "Normal-tissue IHC unavailable"
    ):

        questions.append(
            "Acquire independent normal-tissue "
            "expression and selectivity data"
        )

    else:

        n_high = row[
            "phase55_normal_tissues_high"
        ]

        n_medium_high = row[
            "phase55_normal_tissues_medium_high"
        ]


        if (
            pd.notna(n_high)
            and n_high > 0
        ):

            questions.append(
                "Review cell-type-specific high staining "
                "in healthy tissues"
            )

        elif (
            pd.notna(n_medium_high)
            and n_medium_high > 0
        ):

            questions.append(
                "Review medium staining "
                "in healthy tissues"
            )


    if row["gene_name"] == "TDGF1":

        questions.append(
            "Verify CRIPTO membrane presentation, "
            "shedding and normal-tissue selectivity"
        )


    if row["gene_name"] == "FAP":

        questions.append(
            "Investigate tumor-stroma expression "
            "and stromal targeting strategy"
        )


    return "; ".join(questions)


annotated["phase55_validation_questions"] = (
    annotated.apply(
        validation_questions,
        axis=1
    )
)


annotated["phase55_cell_origin_status"] = np.where(

    annotated["gene_name"].eq("FAP"),

    "Stromal-associated working hypothesis; "
    "not verified in this dataset",

    "Cellular origin unresolved from bulk RNA/proteomics"
)


print(
    "\nCandidate validation questions prepared:",
    len(annotated)
)


# ------------------------------------------------------------
# 11. BUILD EVIDENCE DOSSIER PANEL
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("BUILDING CANDIDATE EVIDENCE DOSSIERS")
print("-" * 65)


# Include the union of all three surface
# top-20 lists, plus leading biological genes
# and representative alternative-lane candidates.
#
# This is a review panel, not a final shortlist.


panel_ids = set()


for column in SCENARIO_COLUMNS.values():

    panel_ids.update(
        surface.loc[
            surface[column].le(20),
            "gene_id"
        ].tolist()
    )


panel_ids.update(

    source.loc[
        source["gene_name"].isin(
            REVIEW_GENES
        ),
        "gene_id"
    ].tolist()

)


for lane in [
    MEMBRANE_LANE,
    EXTRACELLULAR_LANE
]:

    group = source[
        source["phase5_followup_lane"].eq(lane)
    ].copy()

    group = group.sort_values(
        [
            "phase5_median_scenario_rank",
            "gene_id"
        ]
    )

    panel_ids.update(
        group.head(5)["gene_id"].tolist()
    )


panel = annotated[
    annotated["gene_id"].isin(panel_ids)
].copy()


panel = panel.sort_values(
    [
        "phase5_followup_lane",
        "phase5_median_scenario_rank",
        "gene_id"
    ]
)


print("\nEvidence dossiers created:", len(panel))

print(
    "These candidates are selected "
    "for detailed review, not declared "
    "therapeutically superior."
)


# ------------------------------------------------------------
# 12. REVIEW SELECTED BIOLOGICAL CANDIDATES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LEADING BIOLOGICAL CANDIDATE DOSSIERS")
print("-" * 65)


leading = panel[
    panel["gene_name"].isin(REVIEW_GENES)
].copy()


leading["display_order"] = (
    leading["gene_name"]
    .map({
        gene: i
        for i, gene in enumerate(
            REVIEW_GENES
        )
    })
)


leading = leading.sort_values(
    "display_order"
)


display_columns = [
    "gene_name",
    "phase5_followup_lane",
    "phase5_protein_evidence_state",
    "phase55_normal_tissues_medium_high",
    "phase55_normal_tissues_high",
    "phase55_normal_tissues_reliable_medium_high",
    "phase5_rna_conservative_log2fc"
]


print("\n")

print(
    leading[
        display_columns
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)


print(
    "\nNormal-tissue counts are descriptive "
    "and do not establish clinical safety."
)


# ------------------------------------------------------------
# 13. PREPARE NORMAL-TISSUE DETAIL TABLE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING TISSUE-LEVEL DOSSIER EVIDENCE")
print("-" * 65)


panel_ensembl_ids = set(
    panel["ensembl_stable_id"]
)


panel_tissues = tissues[
    tissues["Gene"].isin(
        panel_ensembl_ids
    )
].copy()


panel_tissues = panel_tissues.merge(

    panel[
        [
            "ensembl_stable_id",
            "gene_name",
            "phase5_followup_lane"
        ]
    ],

    left_on="Gene",

    right_on="ensembl_stable_id",

    how="left",

    validate="many_to_one"

)


print(
    "\nTissue-level dossier records:",
    len(panel_tissues)
)


# ------------------------------------------------------------
# 14. VALIDATE ORIGINAL RESEARCH EVIDENCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING SOURCE DATA PRESERVATION")
print("-" * 65)


if len(annotated) != 948:

    raise RuntimeError(
        "Original candidate universe changed."
    )


if annotated["gene_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate genes in evidence audit."
    )


if not annotated[
    source.columns
].equals(source):

    raise RuntimeError(
        "Original ranking evidence was modified."
    )


print("\nOriginal 948 candidates: PRESERVED")

print("Original ranking columns: UNCHANGED")

print("No scoring weights introduced.")


# ------------------------------------------------------------
# 15. SAVE RANKING ROBUSTNESS RESULTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING ROBUSTNESS AND DOSSIER TABLES")
print("-" * 65)


rank_correlations.to_csv(
    RESULTS /
    "surfacetarget_phase5_scenario_rank_correlations.csv"
)


overlap.to_csv(
    RESULTS /
    "surfacetarget_phase5_topn_overlap_audit.csv",
    index=False
)


consensus.to_csv(
    RESULTS /
    "surfacetarget_phase5_topn_consensus_audit.csv",
    index=False
)


three_way.to_csv(
    RESULTS /
    "surfacetarget_phase5_three_scenario_top20.csv",
    index=False
)


annotated.to_csv(
    RESULTS /
    "surfacetarget_phase5_all_tissue_evidence_audit.csv",
    index=False
)


panel.to_csv(
    RESULTS /
    "surfacetarget_phase5_candidate_evidence_dossiers.csv",
    index=False
)


leading.to_csv(
    RESULTS /
    "surfacetarget_phase5_leading_candidate_dossiers.csv",
    index=False
)


panel_tissues.to_csv(
    RESULTS /
    "surfacetarget_phase5_dossier_tissue_details.csv",
    index=False
)


print("\nAll ranking diagnostics: SAVED")

print("All candidate evidence dossiers: SAVED")


# ------------------------------------------------------------
# 16. SAVE PROVENANCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING RESEARCH PROVENANCE")
print("-" * 65)


provenance_output = {

    "project": "SurfaceTarget-CRC",

    "patch": "5.5",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "ranking_source": str(RANKING_PATH),

    "ranking_source_sha256": actual_hash,

    "hpa_tissue_source": str(TISSUE_PATH),

    "hpa_tissue_sha256":
        sha256_file(TISSUE_PATH),

    "candidate_count": 948,

    "surface_review_candidates": 111,

    "ranking_scenarios": list(
        SCENARIO_COLUMNS.keys()
    ),

    "top_n_sensitivity": TOP_N_VALUES,

    "three_scenario_top20_count":
        int(len(three_way)),

    "evidence_dossier_count":
        int(len(panel)),

    "ranking_robustness_policy": (
        "Distinguish agreement under two "
        "scenarios from agreement under "
        "all three scenarios"
    ),

    "normal_tissue_policy": (
        "Inspect all available HPA tissue "
        "profiles, preserving missingness "
        "and reliability information"
    ),

    "malignant_cell_origin_inferred": False,

    "clinical_safety_inferred": False,

    "therapeutic_efficacy_scores": False,

    "previous_rankings_modified": False,

    "candidate_exclusions": 0,

    "limitations": [

        "Ranking scenarios are not independent "
        "experimental validation studies.",

        "Agreement under correlated scenarios "
        "may overstate apparent rank stability.",

        "HPA tissue-level maximum staining "
        "does not resolve all cell types.",

        "Normal IHC evidence does not "
        "establish clinical toxicity or safety.",

        "Bulk tumor enrichment does not "
        "identify malignant-cell expression.",

        "Protein enrichment does not "
        "demonstrate antibody accessibility.",

        "CRISPR dependency does not directly "
        "measure antibody efficacy.",

        "No candidate has been "
        "experimentally validated."

    ]
}


with open(
    META /
    "surfacetarget_phase5_robustness_provenance.json",
    "w"
) as handle:

    json.dump(
        provenance_output,
        handle,
        indent=4
    )


print("\nResearch provenance saved.")


# ------------------------------------------------------------
# 17. CREATE PHASE 5 CHECKPOINT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING PHASE 5 CHECKPOINT")
print("-" * 65)


timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S"
)


checkpoint = (
    CHECKPOINTS /
    f"SurfaceTarget_CRC_Phase5_Patch55_{timestamp}.zip"
)


backup_folders = [
    "data/metadata",
    "data/processed",
    "data/raw/cptac_colon",
    "data/raw/hpa",
    "data/raw/uniprot",
    "data/raw/hgnc",
    "results/tables",
    "results/figures",
    "src",
    "reports",
    "logs"
]


with ZipFile(
    checkpoint,
    "w",
    ZIP_DEFLATED,
    compresslevel=3
) as archive:

    for folder in backup_folders:

        folder_path = ROOT / folder

        if not folder_path.exists():
            continue

        for file_path in folder_path.rglob("*"):

            if file_path.is_file():

                archive.write(
                    file_path,
                    arcname=str(
                        file_path.relative_to(ROOT)
                    )
                )


checkpoint_size = (
    checkpoint.stat().st_size / 1024**2
)


print("\nCheckpoint created successfully.")

print(
    "Checkpoint size:",
    f"{checkpoint_size:.2f} MiB"
)

print("Checkpoint location:")
print(checkpoint)


# ------------------------------------------------------------
# 18. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 5.5 SUMMARY")
print("=" * 65)


print("\nResearch candidates:", len(annotated))

print("Surface-review candidates:", len(surface))

print("Ranking scenarios:", len(SCENARIO_COLUMNS))

print(
    "Three-scenario top-20 agreement:",
    len(three_way)
)

print(
    "Candidate evidence dossiers:",
    len(panel)
)

print(
    "HPA normal tissues reviewed:",
    tissues["Tissue"].nunique()
)


print("\nOutputs:")

print("1. Scenario rank-correlation audit")

print("2. Top-N scenario overlap analysis")

print("3. Two-versus-three scenario consensus")

print("4. Three-scenario top-20 candidate table")

print("5. Full normal-tissue evidence audit")

print("6. Candidate evidence dossiers")

print("7. Leading candidate dossiers")

print("8. Detailed HPA tissue profiles")

print("9. Research provenance")


print("\nOriginal research evidence: PRESERVED")

print("No ranking weights added.")

print("No clinical safety claims.")

print("No candidates excluded.")

print("No experimental validation claimed.")


print("\nCheckpoint: CREATED")

print(
    "Checkpoint size:",
    f"{checkpoint_size:.2f} MiB"
)


print("\nPATCH 5.5 COMPLETE")

print("=" * 65)

In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 5 — PATCH 5.6
#
# RESEARCH PANEL AND EXPERIMENTAL VALIDATION ROADMAP
#
# OBJECTIVES:
# 1. Verify completed Phase 5.4 and 5.5 analyses
# 2. Preserve all 948 candidate genes
# 3. Confirm cross-scenario ranking agreement
# 4. Create a transparent 15-candidate review panel
# 5. Retain distinct biological targeting mechanisms
# 6. Audit protein, topology and functional evidence
# 7. Review all-tissue normal IHC observations
# 8. Construct candidate-specific validation priorities
# 9. Produce a human-readable research summary
# 10. Save provenance and a research checkpoint
#
# IMPORTANT:
# - Expert review panel, not an exhaustive target list
# - No therapeutic efficacy scores
# - No clinical safety predictions
# - No experimentally validated target claims
# - No previous data overwritten
# - No candidate genes excluded
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
import numpy as np
import hashlib
import json


# ------------------------------------------------------------
# 1. PROJECT CONFIGURATION
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

RESULTS = ROOT / "results/tables"
META = ROOT / "data/metadata"
REPORTS = ROOT / "reports"
CHECKPOINTS = ROOT / "checkpoints"

for directory in [
    RESULTS,
    META,
    REPORTS,
    CHECKPOINTS
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


RANKING_PATH = (
    RESULTS /
    "surfacetarget_phase5_prioritization_v1.csv"
)

ALL_TISSUE_PATH = (
    RESULTS /
    "surfacetarget_phase5_all_tissue_evidence_audit.csv"
)

THREE_WAY_PATH = (
    RESULTS /
    "surfacetarget_phase5_three_scenario_top20.csv"
)

CORRELATION_PATH = (
    RESULTS /
    "surfacetarget_phase5_scenario_rank_correlations.csv"
)

P54_PROVENANCE = (
    META /
    "surfacetarget_phase5_prioritization_provenance.json"
)

P55_PROVENANCE = (
    META /
    "surfacetarget_phase5_robustness_provenance.json"
)


print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 5 — PATCH 5.6")
print("RESEARCH PANEL AND VALIDATION ROADMAP")
print("=" * 65)


# ------------------------------------------------------------
# 2. VERIFY RESEARCH INPUTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING RESEARCH INPUTS")
print("-" * 65)


required_files = {
    "Phase 5.4 ranking": RANKING_PATH,
    "Phase 5.5 tissue audit": ALL_TISSUE_PATH,
    "Three-scenario candidates": THREE_WAY_PATH,
    "Rank correlations": CORRELATION_PATH,
    "Phase 5.4 provenance": P54_PROVENANCE,
    "Phase 5.5 provenance": P55_PROVENANCE
}


for name, path in required_files.items():

    if not path.exists():

        raise FileNotFoundError(
            f"{name} missing: {path}"
        )


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


with open(P54_PROVENANCE, "r") as handle:

    p54 = json.load(handle)


with open(P55_PROVENANCE, "r") as handle:

    p55 = json.load(handle)


ranking_hash = sha256_file(RANKING_PATH)


if ranking_hash != p54.get("output_sha256"):

    raise RuntimeError(
        "Phase 5.4 ranking checksum mismatch."
    )


if ranking_hash != p55.get(
    "ranking_source_sha256"
):

    raise RuntimeError(
        "Phase 5.5 source checksum mismatch."
    )


print("\nPhase 5.4 ranking checksum: PASSED")

print("Phase 5.5 provenance correspondence: PASSED")


# ------------------------------------------------------------
# 3. LOAD COMPLETED RESEARCH DATA
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING COMPLETE RESEARCH EVIDENCE")
print("-" * 65)


ranking = pd.read_csv(
    RANKING_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)


evidence = pd.read_csv(
    ALL_TISSUE_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)


three_way = pd.read_csv(
    THREE_WAY_PATH,
    dtype={"gene_id": str},
    low_memory=False
)


correlations = pd.read_csv(
    CORRELATION_PATH,
    index_col=0
)


if len(ranking) != 948:

    raise RuntimeError(
        "Unexpected ranking candidate count."
    )


if len(evidence) != 948:

    raise RuntimeError(
        "Unexpected evidence-audit candidate count."
    )


for name, table in {
    "Ranking": ranking,
    "Evidence": evidence
}.items():

    if table["gene_id"].duplicated().any():

        raise RuntimeError(
            f"{name} has duplicate gene IDs."
        )


if set(ranking["gene_id"]) != set(
    evidence["gene_id"]
):

    raise RuntimeError(
        "Candidate inventories disagree."
    )


# Patch 5.5 retained the Phase 5.4 columns.
# Verify exact correspondence rather than
# assuming that the tables are aligned.


ranking_index = ranking.set_index("gene_id")

evidence_index = evidence.set_index("gene_id")


original_fields = [
    column
    for column in ranking.columns
    if column != "gene_id"
]


if not ranking_index[
    original_fields
].equals(

    evidence_index.loc[
        ranking_index.index,
        original_fields
    ]

):

    raise RuntimeError(
        "Phase 5.4 and Phase 5.5 evidence differ."
    )


print("\nResearch candidates:", len(evidence))

print("Original ranking columns: VERIFIED")

print("Evidence reconciliation: PASSED")


# ------------------------------------------------------------
# 4. VERIFY THREE-SCENARIO AGREEMENT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING RANKING ROBUSTNESS")
print("-" * 65)


SCENARIO_COLUMNS = [
    "phase5_rank_protein_first",
    "phase5_rank_normal_review_first",
    "phase5_rank_rna_discovery_first"
]


surface = evidence[
    evidence["phase5_followup_lane"].eq(
        "Putative cell-surface antigen review"
    )
].copy()


if len(surface) != 111:

    raise RuntimeError(
        "Expected 111 surface-review candidates."
    )


for column in SCENARIO_COLUMNS:

    ranks = pd.to_numeric(
        surface[column],
        errors="raise"
    ).astype(int)

    if set(ranks) != set(
        range(1, 112)
    ):

        raise RuntimeError(
            f"Invalid rank inventory: {column}"
        )


calculated_three_way = surface[

    surface[SCENARIO_COLUMNS]
    .le(20)
    .all(axis=1)

].copy()


if len(calculated_three_way) != 3:

    raise RuntimeError(
        "Expected three candidates in "
        "all scenario top-20 lists."
    )


if set(
    calculated_three_way["gene_id"]
) != set(
    three_way["gene_id"]
):

    raise RuntimeError(
        "Three-scenario membership differs "
        "from Patch 5.5."
    )


expected_three = {
    "CA9",
    "ADAM12",
    "NPC1L1"
}


if set(
    calculated_three_way["gene_name"]
) != expected_three:

    raise RuntimeError(
        "Unexpected three-scenario candidate identities."
    )


print("\nThree-scenario candidates:")

print(
    calculated_three_way[
        [
            "gene_name",
            *SCENARIO_COLUMNS
        ]
    ].to_string(index=False)
)


print("\nScenario rank correlations:")

print(
    correlations.to_string(
        float_format=lambda x: f"{x:.4f}"
    )
)


print(
    "\nRanking scenarios are not independent "
    "biological validation experiments."
)


# ------------------------------------------------------------
# 5. DEFINE TRANSPARENT RESEARCH PANEL
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("DEFINING BIOLOGICAL RESEARCH PANEL")
print("-" * 65)


# This is a deliberately selected
# mechanistic review panel.
#
# Only Group A is chosen through the
# all-three-scenario ranking criterion.
#
# Groups B-E retain complementary
# pre-existing research hypotheses.
#
# Selection is not a claim of therapeutic
# superiority or an exhaustive search.


REVIEW_GROUPS = {

    "A | Cross-scenario surface hypotheses": [
        "CA9",
        "ADAM12",
        "NPC1L1"
    ],

    "B | Additional surface-antigen hypotheses": [
        "CDH3",
        "CLDN1",
        "DPEP1",
        "SLC7A5",
        "TDGF1",
        "LGR5"
    ],

    "C | Stromal-targeting hypothesis": [
        "FAP"
    ],

    "D | Membrane-accessibility investigations": [
        "SLC5A6",
        "CEMIP"
    ],

    "E | Extracellular microenvironment hypotheses": [
        "CTHRC1",
        "MXRA5",
        "SFRP4"
    ]

}


panel_records = []


for group, genes in REVIEW_GROUPS.items():

    for position, gene in enumerate(
        genes,
        start=1
    ):

        panel_records.append({

            "gene_name": gene,

            "phase56_review_group": group,

            "phase56_group_display_order":
                position
        })


panel_definition = pd.DataFrame(
    panel_records
)


if len(panel_definition) != 15:

    raise RuntimeError(
        "Expected 15 research-panel candidates."
    )


if panel_definition[
    "gene_name"
].duplicated().any():

    raise RuntimeError(
        "Duplicate gene in research panel."
    )


if evidence["gene_name"].duplicated().any():

    raise RuntimeError(
        "Duplicate original research gene symbols."
    )


panel = panel_definition.merge(

    evidence,

    on="gene_name",

    how="left",

    validate="one_to_one",

    indicator=True

)


if not panel["_merge"].eq("both").all():

    missing_genes = panel.loc[
        panel["_merge"].ne("both"),
        "gene_name"
    ].tolist()

    raise RuntimeError(
        "Panel genes missing from evidence: "
        f"{missing_genes}"
    )


panel = panel.drop(
    columns="_merge"
)


print("\nResearch-panel genes:", len(panel))

print("\nGroup distribution:")

print(
    panel[
        "phase56_review_group"
    ].value_counts(
        sort=False
    ).to_string()
)


# ------------------------------------------------------------
# 6. VALIDATE BIOLOGICAL LANE ASSIGNMENTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING BIOLOGICAL TARGETING GROUPS")
print("-" * 65)


EXPECTED_LANES = {

    "A": "Putative cell-surface antigen review",

    "B": "Putative cell-surface antigen review",

    "C": "Stromal-associated targeting review",

    "D": (
        "Membrane localization/"
        "accessibility verification"
    ),

    "E": "Extracellular microenvironment review"
}


for _, row in panel.iterrows():

    group_letter = row[
        "phase56_review_group"
    ][0]

    expected_lane = EXPECTED_LANES[
        group_letter
    ]

    actual_lane = row[
        "phase5_followup_lane"
    ]

    if actual_lane != expected_lane:

        raise RuntimeError(
            "Biological lane disagreement for "
            f"{row['gene_name']}: {actual_lane}"
        )


print("\nAll review-group assignments: PASSED")

print(
    "Stromal, membrane and extracellular "
    "mechanisms remain separate."
)


# ------------------------------------------------------------
# 7. VALIDATE PANEL EVIDENCE FIELDS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("AUDITING PANEL EVIDENCE COMPLETENESS")
print("-" * 65)


required_panel_fields = {

    "gene_id",
    "ensembl_stable_id",
    "gene_name",

    "baseline_log2fc",
    "sensitivity_log2fc",
    "phase5_rna_conservative_log2fc",

    "protein_enriched",
    "phase5_protein_evidence_state",
    "mss_complete_pairs",

    "phase5_topology_class",
    "phase5_followup_lane",

    "phase5_normal_ihc_available",

    "phase55_normal_tissues_medium_high",
    "phase55_normal_tissues_high",
    "phase55_normal_tissues_reliable_medium_high",

    "phase55_normal_medium_high_tissue_names",
    "phase55_normal_high_tissue_names",

    "depmap_crc_models_measured",
    "depmap_crc_median_gene_effect",

    *SCENARIO_COLUMNS

}


missing_fields = (
    required_panel_fields
    - set(panel.columns)
)


if missing_fields:

    raise RuntimeError(
        "Panel evidence fields missing: "
        f"{sorted(missing_fields)}"
    )


print("\nRequired evidence fields: FOUND")

print("Panel gene identities: VALIDATED")


# ------------------------------------------------------------
# 8. PRESERVE NORMAL-TISSUE MISSINGNESS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING NORMAL-TISSUE EVIDENCE")
print("-" * 65)


def parse_bool(series):

    values = (
        series.astype("string")
        .str.strip()
        .str.lower()
    )

    allowed = {
        "true", "false", "1", "0",
        "", "nan", "none", "<na>"
    }

    unexpected = (
        values.notna()
        & ~values.isin(list(allowed))
    )

    if unexpected.any():

        raise RuntimeError(
            "Unexpected boolean values: "
            f"{values[unexpected].unique().tolist()}"
        )

    return (
        values.isin(["true", "1"])
        .fillna(False)
        .astype(bool)
    )


ihc_available = parse_bool(
    panel["phase5_normal_ihc_available"]
)


for column in [
    "phase55_normal_tissues_medium_high",
    "phase55_normal_tissues_high",
    "phase55_normal_tissues_reliable_medium_high"
]:

    panel[column] = pd.to_numeric(
        panel[column],
        errors="coerce"
    )

    if panel.loc[
        ~ihc_available,
        column
    ].notna().any():

        raise RuntimeError(
            "Unobserved normal-IHC measurements "
            f"were assigned values: {column}"
        )


panel["phase56_normal_ihc_evidence_status"] = (
    np.where(
        ihc_available,

        "HPA normal-IHC observations available",

        "HPA normal-IHC observations unavailable"
    )
)


print(
    "\nPanel genes with HPA IHC:",
    int(ihc_available.sum())
)

print(
    "Panel genes without HPA IHC:",
    int((~ihc_available).sum())
)


print(
    "Missing normal expression has not "
    "been treated as low expression."
)


# ------------------------------------------------------------
# 9. PREPARE EXPERIMENTAL VALIDATION PLANS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING EXPERIMENTAL VALIDATION ROADMAP")
print("-" * 65)


def validation_plan(row):

    gene = row["gene_name"]

    group = row[
        "phase56_review_group"
    ][0]


    if group in ["A", "B"]:

        primary_question = (
            "Is the antigen expressed and "
            "accessible on malignant MSS CRC cells?"
        )

        primary_assay = (
            "Confirm expression in verified MSS CRC "
            "cell models using non-permeabilized "
            "flow cytometry or surface proteomics"
        )

        orthogonal_assay = (
            "Use single-cell/spatial expression "
            "analysis and matched normal-colon "
            "organoids to assess cell-type selectivity"
        )


    elif group == "C":

        primary_question = (
            "Is the protein enriched in "
            "tumor-associated fibroblasts?"
        )

        primary_assay = (
            "Perform multiplex immunostaining "
            "with fibroblast and epithelial markers"
        )

        orthogonal_assay = (
            "Evaluate spatial protein expression "
            "in tumor stroma and corresponding "
            "healthy tissue"
        )


    elif group == "D":

        primary_question = (
            "Does the protein present an "
            "accessible extracellular epitope?"
        )

        primary_assay = (
            "Measure live-cell extracellular "
            "antibody binding and plasma-membrane "
            "localization"
        )

        orthogonal_assay = (
            "Verify extracellular topology "
            "using orthogonal biochemical or "
            "surface-proteomics methods"
        )


    else:

        primary_question = (
            "Where is the protein deposited "
            "or retained in the tumor environment?"
        )

        primary_assay = (
            "Characterize extracellular matrix "
            "or secreted-protein localization "
            "in tumor sections"
        )

        orthogonal_assay = (
            "Compare tumor versus normal "
            "tissue localization using "
            "spatial protein measurements"
        )


    additional_questions = []


    protein_state = str(
        row["phase5_protein_evidence_state"]
    )


    if protein_state != (
        "Qualifying MSS paired-protein enrichment"
    ):

        additional_questions.append(
            "Obtain stronger tumor/normal "
            "protein-level evidence"
        )


    normal_available = str(
        row["phase56_normal_ihc_evidence_status"]
    )


    if "unavailable" in normal_available.lower():

        additional_questions.append(
            "Acquire independent normal-tissue "
            "expression measurements"
        )


    if gene in ["DPEP1", "SLC7A5"]:

        additional_questions.append(
            "Investigate binding to normal "
            "colon and kidney epithelia"
        )


    if gene in ["CDH3", "CLDN1"]:

        additional_questions.append(
            "Investigate normal epithelial "
            "expression and improve proteomic coverage"
        )


    if gene == "TDGF1":

        additional_questions.append(
            "Validate GPI-associated surface "
            "presentation and potential shedding"
        )


    if gene == "FAP":

        additional_questions.append(
            "Distinguish stromal antigen "
            "presentation from malignant-cell expression"
        )


    return {

        "primary_question": primary_question,

        "primary_assay": primary_assay,

        "orthogonal_assay": orthogonal_assay,

        "additional_review": (
            "; ".join(additional_questions)
            if additional_questions
            else "Standard specificity and validation review"
        )
    }


plan_records = []


for _, row in panel.iterrows():

    plan = validation_plan(row)

    plan_records.append({

        "gene_id": row["gene_id"],

        "gene_name": row["gene_name"],

        "review_group":
            row["phase56_review_group"],

        **plan

    })


validation = pd.DataFrame(
    plan_records
)


if len(validation) != 15:

    raise RuntimeError(
        "Experimental roadmap lost candidates."
    )


print("\nCandidate-specific plans:", len(validation))

print("Experimental roadmap: PREPARED")


# ------------------------------------------------------------
# 10. CREATE PANEL EVIDENCE SUMMARY
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("REVIEWING PANEL EVIDENCE")
print("-" * 65)


panel_display_columns = [

    "gene_name",

    "phase56_review_group",

    "phase5_rna_conservative_log2fc",

    "phase5_protein_evidence_state",

    "phase5_topology_class",

    "phase55_normal_tissues_medium_high",

    "depmap_crc_median_gene_effect"

]


print("\n")

print(
    panel[
        panel_display_columns
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)


print("\nImportant:")

print(
    "These genes belong to different "
    "biological targeting mechanisms."
)

print(
    "Their row order does not represent "
    "therapeutic superiority."
)


# ------------------------------------------------------------
# 11. PREPARE 948-GENE AUDIT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PRESERVING COMPLETE CANDIDATE UNIVERSE")
print("-" * 65)


panel_lookup = panel.set_index(
    "gene_id"
)[
    "phase56_review_group"
].to_dict()


full_audit = evidence.copy(deep=True)


full_audit["phase56_in_review_panel"] = (
    full_audit["gene_id"].isin(
        set(panel["gene_id"])
    )
)


full_audit["phase56_review_group"] = (
    full_audit["gene_id"]
    .map(panel_lookup)
)


if len(full_audit) != 948:

    raise RuntimeError(
        "Complete candidate inventory changed."
    )


if int(
    full_audit["phase56_in_review_panel"].sum()
) != 15:

    raise RuntimeError(
        "Unexpected panel membership count."
    )


# Verify original evidence is unchanged.

if not evidence.equals(
    full_audit[evidence.columns]
):

    raise RuntimeError(
        "Original evidence was modified."
    )


print("\nComplete candidates preserved:", len(full_audit))

print("Selected for detailed panel:", 15)

print("Retained outside review panel:", 933)


# ------------------------------------------------------------
# 12. NORMAL-TISSUE DETAIL REVIEW
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING NORMAL-TISSUE REVIEW TABLE")
print("-" * 65)


normal_review_columns = [

    "gene_id",
    "gene_name",
    "phase56_review_group",

    "phase56_normal_ihc_evidence_status",

    "phase55_normal_tissues_medium_high",
    "phase55_normal_tissues_high",
    "phase55_normal_tissues_reliable_medium_high",

    "phase55_normal_medium_high_tissue_names",
    "phase55_normal_high_tissue_names"

]


normal_review = panel[
    normal_review_columns
].copy()


print("\nNormal-tissue review prepared.")

print(
    "Counts are descriptive observations, "
    "not clinical toxicity estimates."
)


# ------------------------------------------------------------
# 13. SAVE RESEARCH PANEL OUTPUTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING RESEARCH PANEL RESULTS")
print("-" * 65)


PANEL_PATH = (
    RESULTS /
    "surfacetarget_phase5_research_panel_v1.csv"
)

FULL_PATH = (
    RESULTS /
    "surfacetarget_phase5_full_universe_panel_audit.csv"
)

VALIDATION_PATH = (
    RESULTS /
    "surfacetarget_phase5_experimental_validation_roadmap.csv"
)

NORMAL_PATH = (
    RESULTS /
    "surfacetarget_phase5_panel_normal_tissue_review.csv"
)


panel.to_csv(
    PANEL_PATH,
    index=False
)


full_audit.to_csv(
    FULL_PATH,
    index=False
)


validation.to_csv(
    VALIDATION_PATH,
    index=False
)


normal_review.to_csv(
    NORMAL_PATH,
    index=False
)


# Validate saved files.

saved_panel = pd.read_csv(
    PANEL_PATH,
    dtype={"gene_id": str}
)


saved_full = pd.read_csv(
    FULL_PATH,
    dtype={"gene_id": str}
)


if len(saved_panel) != 15:

    raise RuntimeError(
        "Saved research panel has incorrect size."
    )


if len(saved_full) != 948:

    raise RuntimeError(
        "Saved full audit has incorrect size."
    )


if set(saved_panel["gene_id"]) != set(
    panel["gene_id"]
):

    raise RuntimeError(
        "Saved panel identifiers changed."
    )


print("\nResearch tables: SAVED")

print("Output validation: PASSED")


# ------------------------------------------------------------
# 14. GENERATE HUMAN-READABLE RESEARCH SUMMARY
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING RESEARCH INTERPRETATION SUMMARY")
print("-" * 65)


REPORT_PATH = (
    REPORTS /
    "SurfaceTarget_CRC_Phase5_Research_Panel.md"
)


report_lines = [

    "# SurfaceTarget-CRC",

    "## Phase 5 — Research Panel and Validation Roadmap",

    "",

    (
        "This document summarizes an exploratory "
        "15-gene review panel derived from a larger "
        "948-gene cross-omics candidate universe."
    ),

    "",

    (
        "The panel is intended to guide research "
        "follow-up. It is not a validated therapeutic "
        "ranking or clinical recommendation."
    ),

    "",

    "## Research context",

    "",

    (
        "- TCGA: colorectal tumor-associated RNA "
        "enrichment, including computationally "
        "low-MSI discovery samples."
    ),

    (
        "- CPTAC: paired MSS colon tumor/normal "
        "proteomic evidence, subject to coverage limits."
    ),

    (
        "- HPA/UniProt: protein topology and "
        "descriptive healthy-tissue annotations."
    ),

    (
        "- DepMap: colorectal adenocarcinoma "
        "CRISPR fitness measurements; "
        "cell lines are not MSS-confirmed here."
    ),

    "",

    "## Prioritization limitations",

    "",

    (
        "- Protein-first and normal-review-first "
        "ranking scenarios are strongly correlated."
    ),

    (
        "- Only CA9, ADAM12 and NPC1L1 "
        "were in the top 20 of all three "
        "surface-review scenarios."
    ),

    (
        "- Cross-scenario agreement is not "
        "independent biological validation."
    ),

    (
        "- No candidate has been experimentally "
        "validated by this project."
    ),

    "",

    "## Candidate hypotheses",

    ""
]


for group, subset in panel.groupby(
    "phase56_review_group",
    sort=False
):

    report_lines.append(
        f"### {group}"
    )

    report_lines.append("")

    for _, row in subset.iterrows():

        gene = row["gene_name"]

        rna = row[
            "phase5_rna_conservative_log2fc"
        ]

        protein_state = row[
            "phase5_protein_evidence_state"
        ]

        topology = row[
            "phase5_topology_class"
        ]

        normal_n = row[
            "phase55_normal_tissues_medium_high"
        ]

        depmap_effect = row[
            "depmap_crc_median_gene_effect"
        ]


        normal_text = (
            "unavailable"
            if pd.isna(normal_n)
            else str(int(normal_n))
        )


        depmap_text = (
            "unavailable"
            if pd.isna(depmap_effect)
            else f"{depmap_effect:.3f}"
        )


        matching_plan = validation.loc[
            validation["gene_name"].eq(gene)
        ].iloc[0]


        report_lines.extend([

            f"**{gene}**",

            "",

            (
                f"- Conservative RNA log2FC: "
                f"{rna:.3f}"
            ),

            f"- CPTAC evidence: {protein_state}",

            f"- Topology: {topology}",

            (
                "- Normal tissues with medium/high "
                f"IHC: {normal_text}"
            ),

            (
                "- Median colorectal CRISPR "
                f"gene effect: {depmap_text}"
            ),

            (
                "- Primary validation question: "
                f"{matching_plan['primary_question']}"
            ),

            (
                "- Suggested first assay: "
                f"{matching_plan['primary_assay']}"
            ),

            ""
        ])


report_lines.extend([

    "## Scientific limitations",

    "",

    (
        "Normal-tissue IHC is not a direct "
        "measurement of therapeutic toxicity."
    ),

    "",

    (
        "Bulk tumor RNA and proteomics "
        "cannot establish malignant-cell-specific "
        "expression without additional evidence."
    ),

    "",

    (
        "Membrane annotation alone does not "
        "prove extracellular epitope accessibility "
        "or antibody internalization."
    ),

    "",

    (
        "CRISPR gene effects measure fitness "
        "following knockout and do not directly "
        "predict antibody-drug conjugate efficacy."
    ),

    "",

    (
        "Missing protein, IHC or functional "
        "measurements have not been interpreted "
        "as negative biological evidence."
    ),

    "",

    (
        "All 948 original candidate genes "
        "remain in the project dataset."
    ),

    ""
])


REPORT_PATH.write_text(
    "\n".join(report_lines),
    encoding="utf-8"
)


print("\nResearch summary created.")

print("Report path:")
print(REPORT_PATH)


# ------------------------------------------------------------
# 15. SAVE PROVENANCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING FINAL PHASE 5 PANEL PROVENANCE")
print("-" * 65)


PANEL_PROVENANCE_PATH = (
    META /
    "surfacetarget_phase5_research_panel_provenance.json"
)


panel_provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "5.6",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source_files": {

        name: str(path)
        for name, path in required_files.items()

    },

    "source_sha256": {

        name: sha256_file(path)
        for name, path in required_files.items()

    },

    "candidate_universe": 948,

    "research_panel_size": 15,

    "panel_selection": REVIEW_GROUPS,

    "cross_scenario_consistent_genes": sorted(
        expected_three
    ),

    "panel_method": (
        "Three-scenario ranking overlap "
        "combined with explicitly selected "
        "complementary biological hypotheses"
    ),

    "selection_limitations": (
        "The additional hypothesis groups "
        "are a transparent expert-review selection, "
        "not a data-derived optimal target ranking."
    ),

    "therapeutic_scores_assigned": False,

    "clinical_safety_inferred": False,

    "experimental_validation_completed": False,

    "all_candidate_genes_preserved": True,

    "panel_output": str(PANEL_PATH),

    "panel_sha256": sha256_file(PANEL_PATH),

    "full_audit_output": str(FULL_PATH),

    "full_audit_sha256": sha256_file(FULL_PATH),

    "report_output": str(REPORT_PATH),

    "report_sha256": sha256_file(REPORT_PATH),

    "limitations": [

        "Selection criteria reflect research "
        "follow-up decisions and are not "
        "clinical development criteria.",

        "Ranking scenarios are strongly correlated "
        "in two of the three comparisons.",

        "RNA evidence includes computationally "
        "low-MSI cases, not exclusively "
        "independently confirmed MSS cases.",

        "DepMap colorectal models are "
        "not confirmed MSS in this analysis.",

        "CPTAC measurements have gene-specific "
        "paired-sample coverage limitations.",

        "Tumor bulk measurements cannot "
        "assign expression to malignant cells.",

        "Normal-tissue IHC does not "
        "establish clinical safety.",

        "Surface topology does not "
        "demonstrate antibody accessibility.",

        "No new wet-lab experiments "
        "were performed by this project."

    ]
}


with open(
    PANEL_PROVENANCE_PATH,
    "w"
) as handle:

    json.dump(
        panel_provenance,
        handle,
        indent=4
    )


print("\nPanel provenance: SAVED")


# ------------------------------------------------------------
# 16. CREATE CHECKPOINT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING PHASE 5 CHECKPOINT")
print("-" * 65)


timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S"
)


checkpoint = (
    CHECKPOINTS /
    f"SurfaceTarget_CRC_Phase5_Patch56_{timestamp}.zip"
)


backup_folders = [
    "data/metadata",
    "data/processed",
    "data/raw/cptac_colon",
    "data/raw/hpa",
    "data/raw/uniprot",
    "data/raw/hgnc",
    "results/tables",
    "results/figures",
    "src",
    "reports",
    "logs"
]


with ZipFile(
    checkpoint,
    "w",
    ZIP_DEFLATED,
    compresslevel=3
) as archive:

    for folder in backup_folders:

        folder_path = ROOT / folder

        if not folder_path.exists():
            continue

        for file_path in folder_path.rglob("*"):

            if file_path.is_file():

                archive.write(
                    file_path,
                    arcname=str(
                        file_path.relative_to(ROOT)
                    )
                )


checkpoint_size = (
    checkpoint.stat().st_size / 1024**2
)


print("\nCheckpoint created.")

print(
    "Checkpoint size:",
    f"{checkpoint_size:.2f} MiB"
)

print("Checkpoint location:")
print(checkpoint)


# ------------------------------------------------------------
# 17. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 5.6 SUMMARY")
print("=" * 65)


print("\nResearch candidate universe:", 948)

print("Detailed research-panel candidates:", len(panel))

print(
    "Cross-scenario-consistent surface genes:",
    len(calculated_three_way)
)

print(
    "Biological research groups:",
    len(REVIEW_GROUPS)
)


print("\nSaved outputs:")

print("1. surfacetarget_phase5_research_panel_v1.csv")

print("2. surfacetarget_phase5_full_universe_panel_audit.csv")

print("3. surfacetarget_phase5_experimental_validation_roadmap.csv")

print("4. surfacetarget_phase5_panel_normal_tissue_review.csv")

print("5. SurfaceTarget_CRC_Phase5_Research_Panel.md")

print("6. surfacetarget_phase5_research_panel_provenance.json")


print("\nOriginal 948 candidates: PRESERVED")

print("Source rankings: UNCHANGED")

print("Normal-tissue missingness: PRESERVED")

print("No therapeutic efficacy scores assigned.")

print("No clinical safety claims made.")

print("No experimental target validation claimed.")


print("\nPhase 5 research checkpoint: CREATED")

print(
    "Checkpoint size:",
    f"{checkpoint_size:.2f} MiB"
)


print("\nPATCH 5.6 COMPLETE")

print("=" * 65)

In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 5 — PATCH 5.6A
#
# CORRECTED RESEARCH PANEL AND VALIDATION ROADMAP
#
# Fix:
# Robust cross-CSV scientific data integrity verification
#
# Objectives:
# 1. Validate Phase 5.4 and 5.5 source files
# 2. Compare evidence without false CSV precision failures
# 3. Detect and report genuine data discrepancies
# 4. Verify three-scenario ranking agreement
# 5. Construct the 15-candidate research panel
# 6. Preserve all 948 original candidates
# 7. Create experimental validation roadmap
# 8. Save normal-tissue evidence and research report
# 9. Save provenance and create checkpoint
#
# IMPORTANT:
# - No original data overwritten
# - No therapeutic efficacy scores
# - No clinical safety predictions
# - No candidate exclusions
# - No experimental validation claimed
# ============================================================


# ------------------------------------------------------------
# 1. IMPORTS AND CONFIGURATION
# ------------------------------------------------------------

from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
import numpy as np
import hashlib
import json


ROOT = Path("/content/SurfaceTarget-CRC")

RESULTS = ROOT / "results/tables"
META = ROOT / "data/metadata"
REPORTS = ROOT / "reports"
CHECKPOINTS = ROOT / "checkpoints"

for directory in [
    RESULTS,
    META,
    REPORTS,
    CHECKPOINTS
]:
    directory.mkdir(parents=True, exist_ok=True)


RANKING_PATH = (
    RESULTS /
    "surfacetarget_phase5_prioritization_v1.csv"
)

ALL_TISSUE_PATH = (
    RESULTS /
    "surfacetarget_phase5_all_tissue_evidence_audit.csv"
)

THREE_WAY_PATH = (
    RESULTS /
    "surfacetarget_phase5_three_scenario_top20.csv"
)

CORRELATION_PATH = (
    RESULTS /
    "surfacetarget_phase5_scenario_rank_correlations.csv"
)

P54_PROVENANCE = (
    META /
    "surfacetarget_phase5_prioritization_provenance.json"
)

P55_PROVENANCE = (
    META /
    "surfacetarget_phase5_robustness_provenance.json"
)


print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 5 — PATCH 5.6A")
print("CORRECTED RESEARCH PANEL AND VALIDATION ROADMAP")
print("=" * 65)


# ------------------------------------------------------------
# 2. VALIDATE SOURCE FILES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING RESEARCH INPUTS")
print("-" * 65)


required_files = {
    "Phase 5.4 ranking": RANKING_PATH,
    "Phase 5.5 tissue audit": ALL_TISSUE_PATH,
    "Three-scenario candidates": THREE_WAY_PATH,
    "Scenario correlations": CORRELATION_PATH,
    "Phase 5.4 provenance": P54_PROVENANCE,
    "Phase 5.5 provenance": P55_PROVENANCE
}


for name, path in required_files.items():

    if not path.exists():
        raise FileNotFoundError(
            f"{name} missing: {path}"
        )


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


with open(P54_PROVENANCE, "r") as handle:
    p54 = json.load(handle)


with open(P55_PROVENANCE, "r") as handle:
    p55 = json.load(handle)


ranking_hash = sha256_file(RANKING_PATH)


if ranking_hash != p54.get("output_sha256"):

    raise RuntimeError(
        "Phase 5.4 ranking checksum mismatch."
    )


if ranking_hash != p55.get(
    "ranking_source_sha256"
):

    raise RuntimeError(
        "Phase 5.5 source checksum mismatch."
    )


print("\nPhase 5.4 ranking checksum: PASSED")
print("Phase 5.5 provenance correspondence: PASSED")


# ------------------------------------------------------------
# 3. LOAD RESEARCH EVIDENCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING COMPLETE RESEARCH EVIDENCE")
print("-" * 65)


ranking = pd.read_csv(
    RANKING_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)


evidence = pd.read_csv(
    ALL_TISSUE_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)


three_way = pd.read_csv(
    THREE_WAY_PATH,
    dtype={"gene_id": str},
    low_memory=False
)


correlations = pd.read_csv(
    CORRELATION_PATH,
    index_col=0
)


for name, table in {
    "Ranking": ranking,
    "Evidence": evidence
}.items():

    if len(table) != 948:
        raise RuntimeError(
            f"{name}: expected 948 genes."
        )

    if table["gene_id"].isna().any():
        raise RuntimeError(
            f"{name}: missing gene identifiers."
        )

    if table["gene_id"].duplicated().any():
        raise RuntimeError(
            f"{name}: duplicate gene identifiers."
        )


if set(ranking["gene_id"]) != set(
    evidence["gene_id"]
):

    raise RuntimeError(
        "Candidate gene inventories disagree."
    )


print("\nPhase 5.4 candidates:", len(ranking))
print("Phase 5.5 candidates:", len(evidence))

print("\nCandidate inventory correspondence: PASSED")


# ------------------------------------------------------------
# 4. CORRECTED DATA-INTEGRITY COMPARISON
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("AUDITING CROSS-CSV EVIDENCE INTEGRITY")
print("-" * 65)


# Pandas DataFrame.equals() requires exact
# numerical and dtype agreement.
#
# Repeated CSV serialization can introduce
# negligible floating-point differences.
#
# Our revised audit:
#
# - Aligns genes by original gene_id
# - Requires exact missing-value patterns
# - Requires exact text and boolean agreement
# - Requires exact integer measurement agreement
# - Allows <= 1e-12 absolute/relative tolerance
#   only for continuous numerical measurements
# - Reports every numerical field with differences
# - Raises on any discrepancy outside tolerance


ranking_index = ranking.set_index(
    "gene_id"
)


evidence_index = evidence.set_index(
    "gene_id"
)


original_fields = [
    column
    for column in ranking.columns
    if column != "gene_id"
]


missing_fields = (
    set(original_fields)
    - set(evidence_index.columns)
)


if missing_fields:

    raise RuntimeError(
        "Phase 5.5 is missing original fields: "
        f"{sorted(missing_fields)}"
    )


evidence_aligned = evidence_index.loc[
    ranking_index.index,
    original_fields
]


FLOAT_ATOL = 1e-12
FLOAT_RTOL = 1e-12


numerical_roundtrip_records = []

mismatch_records = []


def is_numeric_series(series):

    return pd.api.types.is_numeric_dtype(
        series
    )


def integer_valued(values):

    finite = values[
        np.isfinite(values)
    ]

    if len(finite) == 0:
        return True

    return bool(
        np.all(
            finite == np.round(finite)
        )
    )


for column in original_fields:

    left = ranking_index[column]
    right = evidence_aligned[column]


    # ----------------------------------------
    # A. Missing-value correspondence
    # ----------------------------------------

    null_equal = (
        left.isna().to_numpy()
        ==
        right.isna().to_numpy()
    )


    if not null_equal.all():

        mismatch_records.append({

            "column": column,
            "reason": "Missing-value pattern changed",
            "n_mismatches": int((~null_equal).sum())
        })

        continue


    # ----------------------------------------
    # B. Numeric field comparison
    # ----------------------------------------

    if (
        is_numeric_series(left)
        and is_numeric_series(right)
    ):

        a = left.to_numpy(
            dtype=float
        )

        b = right.to_numpy(
            dtype=float
        )


        finite = (
            np.isfinite(a)
            &
            np.isfinite(b)
        )


        # Identifier-like numerical columns
        # and integer-valued measurements must
        # agree exactly.
        #
        # This prevents tolerance from masking
        # changes to gene IDs, counts, or ranks.


        identifier_like = (
            "entrez" in column.lower()
            or column.lower().endswith("_id")
        )


        exact_required = (
            identifier_like
            or (
                integer_valued(a)
                and integer_valued(b)
            )
        )


        if exact_required:

            equal = (
                (a == b)
                |
                (
                    np.isnan(a)
                    &
                    np.isnan(b)
                )
            )

        else:

            equal = np.isclose(
                a,
                b,
                rtol=FLOAT_RTOL,
                atol=FLOAT_ATOL,
                equal_nan=True
            )


        if not equal.all():

            bad_indices = np.where(
                ~equal
            )[0]


            mismatch_records.append({

                "column": column,

                "reason": (
                    "Numerical difference exceeds "
                    "allowed precision"
                    if not exact_required
                    else "Exact numerical agreement failed"
                ),

                "n_mismatches": int(
                    len(bad_indices)
                ),

                "example_gene_id": str(
                    ranking_index.index[
                        bad_indices[0]
                    ]
                ),

                "phase54_value": str(
                    a[bad_indices[0]]
                ),

                "phase55_value": str(
                    b[bad_indices[0]]
                )
            })

            continue


        # Record numerical differences that
        # were small enough to be consistent
        # with CSV floating-point round trips.


        if finite.any():

            differences = np.abs(
                a[finite] - b[finite]
            )

            changed = int(
                np.count_nonzero(
                    differences != 0
                )
            )

            if changed > 0:

                numerical_roundtrip_records.append({

                    "column": column,

                    "changed_values": changed,

                    "max_absolute_difference": float(
                        differences.max()
                    ),

                    "allowed_tolerance": (
                        f"atol={FLOAT_ATOL}, "
                        f"rtol={FLOAT_RTOL}"
                    )

                })


    # ----------------------------------------
    # C. Text and categorical comparison
    # ----------------------------------------

    else:

        a = (
            left.astype("string")
            .fillna("<MISSING>")
        )

        b = (
            right.astype("string")
            .fillna("<MISSING>")
        )


        equal = (
            a.to_numpy()
            ==
            b.to_numpy()
        )


        if not equal.all():

            bad_indices = np.where(
                ~equal
            )[0]


            mismatch_records.append({

                "column": column,

                "reason": (
                    "Categorical or textual value changed"
                ),

                "n_mismatches": int(
                    len(bad_indices)
                ),

                "example_gene_id": str(
                    ranking_index.index[
                        bad_indices[0]
                    ]
                ),

                "phase54_value": str(
                    a.iloc[bad_indices[0]]
                ),

                "phase55_value": str(
                    b.iloc[bad_indices[0]]
                )
            })


# ------------------------------------------------------------
# 5. REPORT INTEGRITY AUDIT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CROSS-CSV INTEGRITY RESULTS")
print("-" * 65)


print("\nOriginal evidence fields checked:",
      len(original_fields))


print(
    "Fields with acceptable numerical round-trip differences:",
    len(numerical_roundtrip_records)
)


print(
    "Fields with substantive discrepancies:",
    len(mismatch_records)
)


if numerical_roundtrip_records:

    roundtrip_df = pd.DataFrame(
        numerical_roundtrip_records
    )


    print("\nAcceptable floating-point differences:")

    print(
        roundtrip_df.sort_values(
            "max_absolute_difference",
            ascending=False
        )
        .head(20)
        .to_string(index=False)
    )

else:

    print(
        "\nNo numerical round-trip differences detected."
    )


if mismatch_records:

    mismatch_df = pd.DataFrame(
        mismatch_records
    )

    print("\nSCIENTIFIC DATA DISCREPANCIES DETECTED:")

    print(
        mismatch_df.head(30).to_string(
            index=False
        )
    )

    raise RuntimeError(
        "Phase 5.4 and 5.5 differ beyond "
        "permitted CSV serialization tolerance. "
        "Inspect the printed discrepancies."
    )


print("\nCandidate identifiers: EXACT MATCH")

print("Missing-value patterns: EXACT MATCH")

print("Categorical evidence: EXACT MATCH")

print("Integer-valued measurements: EXACT MATCH")

print("Continuous numerical evidence: VERIFIED")

print("\nOriginal scientific evidence: PRESERVED")


# ------------------------------------------------------------
# 6. VERIFY THREE-SCENARIO AGREEMENT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING RANKING ROBUSTNESS")
print("-" * 65)


SCENARIO_COLUMNS = [
    "phase5_rank_protein_first",
    "phase5_rank_normal_review_first",
    "phase5_rank_rna_discovery_first"
]


surface = evidence[
    evidence["phase5_followup_lane"].eq(
        "Putative cell-surface antigen review"
    )
].copy()


if len(surface) != 111:

    raise RuntimeError(
        "Expected 111 surface-review candidates."
    )


for column in SCENARIO_COLUMNS:

    values = pd.to_numeric(
        surface[column],
        errors="raise"
    ).astype(int)


    if set(values) != set(
        range(1, 112)
    ):

        raise RuntimeError(
            f"Invalid scenario ranks: {column}"
        )


calculated_three_way = surface[
    surface[SCENARIO_COLUMNS]
    .le(20)
    .all(axis=1)
].copy()


if len(calculated_three_way) != 3:

    raise RuntimeError(
        "Expected three candidates "
        "in all scenario top-20 lists."
    )


if set(
    calculated_three_way["gene_id"]
) != set(
    three_way["gene_id"]
):

    raise RuntimeError(
        "Three-scenario membership changed."
    )


expected_three = {
    "CA9",
    "ADAM12",
    "NPC1L1"
}


if set(
    calculated_three_way["gene_name"]
) != expected_three:

    raise RuntimeError(
        "Unexpected three-scenario candidate identities."
    )


print("\nThree-scenario candidates:")

print(
    calculated_three_way[
        [
            "gene_name",
            *SCENARIO_COLUMNS
        ]
    ].to_string(index=False)
)


print("\nScenario rank correlations:")

print(
    correlations.to_string(
        float_format=lambda x: f"{x:.4f}"
    )
)


print(
    "\nCorrelated rankings do not constitute "
    "independent biological validation."
)


# ------------------------------------------------------------
# 7. DEFINE THE RESEARCH PANEL
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("DEFINING BIOLOGICAL RESEARCH PANEL")
print("-" * 65)


REVIEW_GROUPS = {

    "A | Cross-scenario surface hypotheses": [
        "CA9",
        "ADAM12",
        "NPC1L1"
    ],

    "B | Additional surface-antigen hypotheses": [
        "CDH3",
        "CLDN1",
        "DPEP1",
        "SLC7A5",
        "TDGF1",
        "LGR5"
    ],

    "C | Stromal-targeting hypothesis": [
        "FAP"
    ],

    "D | Membrane-accessibility investigations": [
        "SLC5A6",
        "CEMIP"
    ],

    "E | Extracellular microenvironment hypotheses": [
        "CTHRC1",
        "MXRA5",
        "SFRP4"
    ]
}


panel_records = []


for group, genes in REVIEW_GROUPS.items():

    for position, gene in enumerate(
        genes,
        start=1
    ):

        panel_records.append({

            "gene_name": gene,

            "phase56_review_group": group,

            "phase56_group_display_order": position

        })


panel_definition = pd.DataFrame(
    panel_records
)


if len(panel_definition) != 15:

    raise RuntimeError(
        "Expected 15 panel candidates."
    )


if panel_definition["gene_name"].duplicated().any():

    raise RuntimeError(
        "Duplicate genes in panel definition."
    )


if evidence["gene_name"].duplicated().any():

    raise RuntimeError(
        "Duplicate source gene symbols."
    )


panel = panel_definition.merge(
    evidence,
    on="gene_name",
    how="left",
    validate="one_to_one",
    indicator=True
)


if not panel["_merge"].eq("both").all():

    missing = panel.loc[
        panel["_merge"].ne("both"),
        "gene_name"
    ].tolist()

    raise RuntimeError(
        f"Panel genes missing: {missing}"
    )


panel = panel.drop(
    columns="_merge"
)


print("\nResearch-panel genes:", len(panel))


print("\nGroup distribution:")

print(
    panel["phase56_review_group"]
    .value_counts(sort=False)
    .to_string()
)


# ------------------------------------------------------------
# 8. VALIDATE BIOLOGICAL TARGETING LANES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING TARGETING MECHANISMS")
print("-" * 65)


EXPECTED_LANES = {

    "A": "Putative cell-surface antigen review",

    "B": "Putative cell-surface antigen review",

    "C": "Stromal-associated targeting review",

    "D": (
        "Membrane localization/"
        "accessibility verification"
    ),

    "E": "Extracellular microenvironment review"

}


for _, row in panel.iterrows():

    group_letter = row[
        "phase56_review_group"
    ][0]

    expected_lane = EXPECTED_LANES[
        group_letter
    ]

    actual_lane = row[
        "phase5_followup_lane"
    ]

    if actual_lane != expected_lane:

        raise RuntimeError(
            f"Targeting-lane disagreement "
            f"for {row['gene_name']}: "
            f"{actual_lane}"
        )


print("\nBiological group assignments: PASSED")


# ------------------------------------------------------------
# 9. AUDIT PANEL EVIDENCE COLUMNS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING PANEL EVIDENCE")
print("-" * 65)


required_panel_fields = {

    "gene_id",
    "ensembl_stable_id",
    "gene_name",

    "baseline_log2fc",
    "sensitivity_log2fc",
    "phase5_rna_conservative_log2fc",

    "protein_enriched",
    "phase5_protein_evidence_state",
    "mss_complete_pairs",

    "phase5_topology_class",
    "phase5_followup_lane",

    "phase5_normal_ihc_available",

    "phase55_normal_tissues_medium_high",
    "phase55_normal_tissues_high",
    "phase55_normal_tissues_reliable_medium_high",

    "phase55_normal_medium_high_tissue_names",
    "phase55_normal_high_tissue_names",

    "depmap_crc_models_measured",
    "depmap_crc_median_gene_effect",

    *SCENARIO_COLUMNS

}


missing_fields = (
    required_panel_fields
    - set(panel.columns)
)


if missing_fields:

    raise RuntimeError(
        "Missing panel evidence columns: "
        f"{sorted(missing_fields)}"
    )


print("\nPanel evidence fields: VALIDATED")


# ------------------------------------------------------------
# 10. PRESERVE NORMAL-TISSUE MISSINGNESS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING NORMAL-TISSUE EVIDENCE")
print("-" * 65)


def parse_bool(series):

    values = (
        series.astype("string")
        .str.strip()
        .str.lower()
    )

    accepted = {
        "true", "false", "1", "0",
        "", "nan", "none", "<na>"
    }

    invalid = (
        values.notna()
        & ~values.isin(list(accepted))
    )

    if invalid.any():

        raise RuntimeError(
            "Unexpected boolean annotations: "
            f"{values[invalid].unique().tolist()}"
        )

    return (
        values.isin(["true", "1"])
        .fillna(False)
        .astype(bool)
    )


ihc_available = parse_bool(
    panel["phase5_normal_ihc_available"]
)


normal_count_columns = [
    "phase55_normal_tissues_medium_high",
    "phase55_normal_tissues_high",
    "phase55_normal_tissues_reliable_medium_high"
]


for column in normal_count_columns:

    panel[column] = pd.to_numeric(
        panel[column],
        errors="coerce"
    )


    if panel.loc[
        ~ihc_available,
        column
    ].notna().any():

        raise RuntimeError(
            "Unavailable normal-IHC observations "
            f"were assigned values: {column}"
        )


panel["phase56_normal_ihc_evidence_status"] = (
    np.where(

        ihc_available,

        "HPA normal-IHC observations available",

        "HPA normal-IHC observations unavailable"

    )
)


print(
    "\nPanel genes with normal-IHC observations:",
    int(ihc_available.sum())
)


print(
    "Panel genes without normal-IHC observations:",
    int((~ihc_available).sum())
)


print(
    "\nMissing observations remain missing, "
    "not interpreted as zero expression."
)


# ------------------------------------------------------------
# 11. CREATE EXPERIMENTAL VALIDATION ROADMAP
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING EXPERIMENTAL VALIDATION ROADMAP")
print("-" * 65)


def validation_plan(row):

    gene = row["gene_name"]

    group = row[
        "phase56_review_group"
    ][0]


    if group in ["A", "B"]:

        primary_question = (
            "Is the antigen expressed and "
            "accessible on malignant MSS CRC cells?"
        )

        primary_assay = (
            "Evaluate non-permeabilized flow "
            "cytometry or surface proteomics "
            "in verified MSS CRC models"
        )

        orthogonal_assay = (
            "Use single-cell/spatial expression "
            "analysis and matched normal-colon "
            "organoids to evaluate selectivity"
        )


    elif group == "C":

        primary_question = (
            "Is the protein enriched in "
            "tumor-associated fibroblasts?"
        )

        primary_assay = (
            "Perform multiplex immunostaining "
            "with epithelial and fibroblast markers"
        )

        orthogonal_assay = (
            "Compare tumor-stromal and normal-tissue "
            "localization using spatial protein assays"
        )


    elif group == "D":

        primary_question = (
            "Is an extracellular epitope accessible "
            "on intact cancer cells?"
        )

        primary_assay = (
            "Evaluate live-cell extracellular "
            "antibody binding and membrane localization"
        )

        orthogonal_assay = (
            "Verify surface topology using "
            "orthogonal surface-proteomics methods"
        )


    else:

        primary_question = (
            "Where is the extracellular protein "
            "localized in the tumor environment?"
        )

        primary_assay = (
            "Characterize secreted or extracellular "
            "protein distribution in tumor sections"
        )

        orthogonal_assay = (
            "Compare tumor versus normal-tissue "
            "localization with spatial protein assays"
        )


    additional = []


    protein_state = str(
        row["phase5_protein_evidence_state"]
    )


    if protein_state != (
        "Qualifying MSS paired-protein enrichment"
    ):

        additional.append(
            "Obtain stronger tumor/normal "
            "protein-level evidence"
        )


    if (
        row["phase56_normal_ihc_evidence_status"]
        ==
        "HPA normal-IHC observations unavailable"
    ):

        additional.append(
            "Acquire independent normal-tissue "
            "expression measurements"
        )


    if gene in ["DPEP1", "SLC7A5"]:

        additional.append(
            "Investigate normal colon and kidney "
            "cell-surface expression"
        )


    if gene in ["CDH3", "CLDN1"]:

        additional.append(
            "Improve tumor/normal protein coverage "
            "and investigate normal epithelial expression"
        )


    if gene == "TDGF1":

        additional.append(
            "Validate GPI-associated surface "
            "presentation and potential shedding"
        )


    if gene == "FAP":

        additional.append(
            "Distinguish tumor-associated stromal "
            "expression from malignant-cell expression"
        )


    return {

        "primary_question": primary_question,

        "primary_assay": primary_assay,

        "orthogonal_assay": orthogonal_assay,

        "additional_review": (
            "; ".join(additional)
            if additional
            else "Standard specificity and validation review"
        )

    }


validation_records = []


for _, row in panel.iterrows():

    plan = validation_plan(row)

    validation_records.append({

        "gene_id": row["gene_id"],

        "gene_name": row["gene_name"],

        "review_group":
            row["phase56_review_group"],

        **plan

    })


validation = pd.DataFrame(
    validation_records
)


if len(validation) != 15:

    raise RuntimeError(
        "Validation roadmap lost candidates."
    )


print("\nCandidate-specific validation plans:", len(validation))

print("Experimental validation roadmap: PREPARED")


# ------------------------------------------------------------
# 12. DISPLAY RESEARCH PANEL
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("REVIEWING RESEARCH PANEL EVIDENCE")
print("-" * 65)


panel_display_columns = [

    "gene_name",

    "phase56_review_group",

    "phase5_rna_conservative_log2fc",

    "phase5_protein_evidence_state",

    "phase5_topology_class",

    "phase55_normal_tissues_medium_high",

    "depmap_crc_median_gene_effect"

]


print("\n")

print(
    panel[
        panel_display_columns
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}"
    )
)


print("\nPanel order reflects review groups, "
      "not therapeutic superiority.")


# ------------------------------------------------------------
# 13. PRESERVE COMPLETE CANDIDATE UNIVERSE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PRESERVING COMPLETE 948-GENE UNIVERSE")
print("-" * 65)


panel_lookup = (
    panel.set_index("gene_id")[
        "phase56_review_group"
    ].to_dict()
)


full_audit = evidence.copy(deep=True)


full_audit["phase56_in_review_panel"] = (
    full_audit["gene_id"].isin(
        set(panel["gene_id"])
    )
)


full_audit["phase56_review_group"] = (
    full_audit["gene_id"]
    .map(panel_lookup)
)


if len(full_audit) != 948:

    raise RuntimeError(
        "Original research universe changed."
    )


if int(
    full_audit["phase56_in_review_panel"]
    .sum()
) != 15:

    raise RuntimeError(
        "Panel membership count incorrect."
    )


if not evidence.equals(
    full_audit[evidence.columns]
):

    raise RuntimeError(
        "Original research evidence changed."
    )


print("\nAll candidate genes preserved:", len(full_audit))

print("Detailed research-panel genes:", 15)

print("Other retained candidates:", 933)


# ------------------------------------------------------------
# 14. PREPARE NORMAL-TISSUE REVIEW TABLE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PREPARING NORMAL-TISSUE REVIEW")
print("-" * 65)


normal_review_columns = [

    "gene_id",
    "gene_name",
    "phase56_review_group",

    "phase56_normal_ihc_evidence_status",

    "phase55_normal_tissues_medium_high",
    "phase55_normal_tissues_high",
    "phase55_normal_tissues_reliable_medium_high",

    "phase55_normal_medium_high_tissue_names",
    "phase55_normal_high_tissue_names"

]


normal_review = panel[
    normal_review_columns
].copy()


print("\nNormal-tissue review: PREPARED")


# ------------------------------------------------------------
# 15. SAVE PANEL TABLES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING RESEARCH PANEL RESULTS")
print("-" * 65)


PANEL_PATH = (
    RESULTS /
    "surfacetarget_phase5_research_panel_v1.csv"
)

FULL_PATH = (
    RESULTS /
    "surfacetarget_phase5_full_universe_panel_audit.csv"
)

VALIDATION_PATH = (
    RESULTS /
    "surfacetarget_phase5_experimental_validation_roadmap.csv"
)

NORMAL_PATH = (
    RESULTS /
    "surfacetarget_phase5_panel_normal_tissue_review.csv"
)

INTEGRITY_PATH = (
    META /
    "surfacetarget_phase5_roundtrip_integrity_audit.json"
)


panel.to_csv(
    PANEL_PATH,
    index=False
)


full_audit.to_csv(
    FULL_PATH,
    index=False
)


validation.to_csv(
    VALIDATION_PATH,
    index=False
)


normal_review.to_csv(
    NORMAL_PATH,
    index=False
)


integrity_report = {

    "project": "SurfaceTarget-CRC",

    "patch": "5.6A",

    "comparison": (
        "Phase 5.4 ranking CSV versus "
        "Phase 5.5 all-tissue evidence CSV"
    ),

    "candidate_genes": 948,

    "fields_checked": len(original_fields),

    "numerical_tolerance": {
        "absolute": FLOAT_ATOL,
        "relative": FLOAT_RTOL
    },

    "accepted_numerical_roundtrip_fields":
        numerical_roundtrip_records,

    "substantive_discrepancies":
        mismatch_records,

    "integrity_passed": True,

    "ranking_source_sha256": ranking_hash,

    "evidence_source_sha256":
        sha256_file(ALL_TISSUE_PATH)

}


with open(
    INTEGRITY_PATH,
    "w"
) as handle:

    json.dump(
        integrity_report,
        handle,
        indent=4
    )


saved_panel = pd.read_csv(
    PANEL_PATH,
    dtype={"gene_id": str}
)


saved_full = pd.read_csv(
    FULL_PATH,
    dtype={"gene_id": str}
)


if len(saved_panel) != 15:

    raise RuntimeError(
        "Saved research panel count incorrect."
    )


if len(saved_full) != 948:

    raise RuntimeError(
        "Saved candidate universe count incorrect."
    )


if set(saved_panel["gene_id"]) != set(
    panel["gene_id"]
):

    raise RuntimeError(
        "Saved research panel identifiers changed."
    )


print("\nPanel tables: SAVED")

print("Integrity audit: SAVED")

print("Output validation: PASSED")


# ------------------------------------------------------------
# 16. CREATE HUMAN-READABLE RESEARCH REPORT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING RESEARCH INTERPRETATION REPORT")
print("-" * 65)


REPORT_PATH = (
    REPORTS /
    "SurfaceTarget_CRC_Phase5_Research_Panel.md"
)


report_lines = [

    "# SurfaceTarget-CRC",

    "## Phase 5 — Research Panel and Validation Roadmap",

    "",

    (
        "This report summarizes 15 exploratory research "
        "candidates selected from 948 colorectal "
        "tumor-associated transcriptomic candidates."
    ),

    "",

    (
        "The panel combines cross-scenario ranking "
        "agreement with complementary mechanistic "
        "research hypotheses."
    ),

    "",

    (
        "It does not represent a validated therapeutic "
        "ranking, safety assessment, or clinical recommendation."
    ),

    "",

    "## Research evidence",

    "",

    (
        "- TCGA: tumor-associated RNA evidence, "
        "including computationally low-MSI discovery cases."
    ),

    (
        "- CPTAC: paired MSS colon tumor/normal "
        "proteomics, subject to protein-specific coverage."
    ),

    (
        "- UniProt/HPA: topology and descriptive "
        "normal-tissue expression annotations."
    ),

    (
        "- DepMap: colorectal adenocarcinoma CRISPR "
        "fitness measurements without confirmed MSS "
        "status in this analysis."
    ),

    "",

    "## Ranking limitations",

    "",

    (
        "The protein-first and normal-review-first "
        "rankings are highly correlated."
    ),

    (
        "Only CA9, ADAM12, and NPC1L1 appeared "
        "in the top 20 under all three scenarios."
    ),

    (
        "Cross-scenario agreement does not constitute "
        "independent biological validation."
    ),

    "",

    "## Candidate research panel",

    ""
]


for group, subset in panel.groupby(
    "phase56_review_group",
    sort=False
):

    report_lines.extend([

        f"### {group}",

        ""

    ])


    for _, row in subset.iterrows():

        gene = row["gene_name"]

        rna = row[
            "phase5_rna_conservative_log2fc"
        ]

        protein_state = row[
            "phase5_protein_evidence_state"
        ]

        topology = row[
            "phase5_topology_class"
        ]

        normal_n = row[
            "phase55_normal_tissues_medium_high"
        ]

        effect = row[
            "depmap_crc_median_gene_effect"
        ]


        normal_text = (
            "unavailable"
            if pd.isna(normal_n)
            else str(int(normal_n))
        )


        effect_text = (
            "unavailable"
            if pd.isna(effect)
            else f"{effect:.3f}"
        )


        plan = validation.loc[
            validation["gene_name"].eq(gene)
        ].iloc[0]


        report_lines.extend([

            f"**{gene}**",

            "",

            (
                f"- Conservative RNA log2FC: "
                f"{rna:.3f}"
            ),

            f"- CPTAC evidence: {protein_state}",

            f"- Surface topology: {topology}",

            (
                "- Normal tissues with medium/high "
                f"IHC: {normal_text}"
            ),

            (
                "- Median colorectal CRISPR "
                f"gene effect: {effect_text}"
            ),

            (
                "- Primary validation question: "
                f"{plan['primary_question']}"
            ),

            (
                "- Suggested first assay: "
                f"{plan['primary_assay']}"
            ),

            ""
        ])


report_lines.extend([

    "## Scientific limitations",

    "",

    (
        "Bulk tumor RNA and proteomics do not "
        "establish malignant-cell-specific expression."
    ),

    "",

    (
        "UniProt topology does not directly demonstrate "
        "antibody accessibility or internalization."
    ),

    "",

    (
        "HPA staining does not establish clinical "
        "safety or predict therapeutic toxicity."
    ),

    "",

    (
        "CRISPR knockout effects do not directly "
        "measure antibody-drug conjugate efficacy."
    ),

    "",

    (
        "Missing evidence has not been interpreted "
        "as negative biological evidence."
    ),

    "",

    (
        "All 948 original research candidates "
        "remain preserved in the complete dataset."
    ),

    "",

    (
        "No targets have been experimentally "
        "validated by this project."
    ),

    ""
])


REPORT_PATH.write_text(
    "\n".join(report_lines),
    encoding="utf-8"
)


print("\nResearch report created.")

print("Location:", REPORT_PATH)


# ------------------------------------------------------------
# 17. SAVE RESEARCH PROVENANCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING RESEARCH PANEL PROVENANCE")
print("-" * 65)


PANEL_PROVENANCE_PATH = (
    META /
    "surfacetarget_phase5_research_panel_provenance.json"
)


panel_provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "5.6A",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source_files": {

        name: str(path)

        for name, path in required_files.items()

    },

    "source_sha256": {

        name: sha256_file(path)

        for name, path in required_files.items()

    },

    "integrity_validation": (
        "Exact gene, missingness, categorical "
        "and integer correspondence; strict "
        "floating-point tolerance for continuous "
        "CSV round-trip measurements"
    ),

    "integrity_audit": str(
        INTEGRITY_PATH
    ),

    "candidate_universe": 948,

    "research_panel_size": 15,

    "panel_selection": REVIEW_GROUPS,

    "cross_scenario_consistent_genes": sorted(
        expected_three
    ),

    "panel_method": (
        "Cross-scenario ranking agreement "
        "plus explicitly selected complementary "
        "biological research hypotheses"
    ),

    "ranking_scenario_independence": False,

    "therapeutic_scores_assigned": False,

    "clinical_safety_inferred": False,

    "experimental_validation_completed": False,

    "all_candidate_genes_preserved": True,

    "panel_output": str(PANEL_PATH),

    "panel_sha256": sha256_file(PANEL_PATH),

    "full_audit_output": str(FULL_PATH),

    "full_audit_sha256": sha256_file(FULL_PATH),

    "report_output": str(REPORT_PATH),

    "report_sha256": sha256_file(REPORT_PATH),

    "limitations": [

        "Panel selection is a research review "
        "decision, not a clinically validated ranking.",

        "Two prioritization scenarios are "
        "strongly correlated.",

        "RNA discovery includes computationally "
        "low-MSI rather than exclusively "
        "independently confirmed MSS cases.",

        "DepMap colorectal models do not have "
        "independently verified MSS status here.",

        "Protein-level measurement coverage "
        "varies substantially across genes.",

        "Bulk molecular measurements cannot "
        "establish malignant-cell origin.",

        "Normal-tissue IHC is descriptive "
        "and cannot establish clinical safety.",

        "Cell-surface topology does not prove "
        "antibody binding or accessibility.",

        "No new experimental target "
        "validation was performed."

    ]
}


with open(
    PANEL_PROVENANCE_PATH,
    "w"
) as handle:

    json.dump(
        panel_provenance,
        handle,
        indent=4
    )


print("\nResearch provenance: SAVED")


# ------------------------------------------------------------
# 18. CREATE RESEARCH CHECKPOINT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING PHASE 5 RESEARCH CHECKPOINT")
print("-" * 65)


timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S"
)


checkpoint = (
    CHECKPOINTS /
    f"SurfaceTarget_CRC_Phase5_Patch56A_{timestamp}.zip"
)


backup_folders = [
    "data/metadata",
    "data/processed",
    "data/raw/cptac_colon",
    "data/raw/hpa",
    "data/raw/uniprot",
    "data/raw/hgnc",
    "results/tables",
    "results/figures",
    "src",
    "reports",
    "logs"
]


with ZipFile(
    checkpoint,
    "w",
    ZIP_DEFLATED,
    compresslevel=3
) as archive:

    for folder in backup_folders:

        source_folder = ROOT / folder

        if not source_folder.exists():
            continue

        for file_path in source_folder.rglob("*"):

            if file_path.is_file():

                archive.write(

                    file_path,

                    arcname=str(
                        file_path.relative_to(ROOT)
                    )

                )


checkpoint_size = (
    checkpoint.stat().st_size
    / 1024**2
)


print("\nCheckpoint created.")

print(
    "Checkpoint size:",
    f"{checkpoint_size:.2f} MiB"
)

print("Checkpoint location:")
print(checkpoint)


# ------------------------------------------------------------
# 19. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 5.6A SUMMARY")
print("=" * 65)


print("\nResearch candidates:", len(full_audit))

print("Detailed research panel:", len(panel))

print(
    "Cross-scenario-consistent surface candidates:",
    len(calculated_three_way)
)

print(
    "Biological research groups:",
    len(REVIEW_GROUPS)
)


print("\nScientific integrity audit: PASSED")

print("Source evidence: PRESERVED")

print("Three-scenario agreement: VERIFIED")

print("Normal-tissue missingness: PRESERVED")


print("\nSaved outputs:")

print("1. surfacetarget_phase5_research_panel_v1.csv")

print("2. surfacetarget_phase5_full_universe_panel_audit.csv")

print("3. surfacetarget_phase5_experimental_validation_roadmap.csv")

print("4. surfacetarget_phase5_panel_normal_tissue_review.csv")

print("5. SurfaceTarget_CRC_Phase5_Research_Panel.md")

print("6. surfacetarget_phase5_roundtrip_integrity_audit.json")

print("7. surfacetarget_phase5_research_panel_provenance.json")


print("\nCheckpoint: CREATED")

print(
    "Checkpoint size:",
    f"{checkpoint_size:.2f} MiB"
)


print("\nNo candidates excluded.")

print("No clinical safety scores assigned.")

print("No therapeutic efficacy claims.")

print("No experimental validation claimed.")


print("\nPATCH 5.6A COMPLETE")

print("=" * 65)

In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 6 — PATCH 6.1
#
# RESEARCH FIGURES AND FINAL EVIDENCE VERIFICATION
#
# OBJECTIVES:
# 1. Validate completed Phase 5 research outputs
# 2. Verify the 15-candidate panel and 948-gene universe
# 3. Reconfirm cross-scenario ranking membership
# 4. Generate evidence-coverage figure
# 5. Generate protein-topology distribution figure
# 6. Generate surface-candidate ranking-sensitivity figure
# 7. Prepare figure-source manifest
# 8. Save a preliminary Results evidence audit
# 9. Save provenance
# 10. Create a Phase 6 research checkpoint
#
# IMPORTANT:
# - No new therapeutic rankings
# - No candidate exclusions
# - No clinical safety assessments
# - No therapeutic efficacy claims
# - No source data overwritten
# - All figures are descriptive
# ============================================================


# ------------------------------------------------------------
# 1. IMPORTS
# ------------------------------------------------------------

from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import hashlib
import json


# ------------------------------------------------------------
# 2. PROJECT CONFIGURATION
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

RESULTS = ROOT / "results/tables"
FIGURES = ROOT / "results/figures"
META = ROOT / "data/metadata"
REPORTS = ROOT / "reports"
CHECKPOINTS = ROOT / "checkpoints"

for folder in [
    RESULTS,
    FIGURES,
    META,
    REPORTS,
    CHECKPOINTS
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


PANEL_PATH = (
    RESULTS /
    "surfacetarget_phase5_research_panel_v1.csv"
)

FULL_PATH = (
    RESULTS /
    "surfacetarget_phase5_full_universe_panel_audit.csv"
)

ROADMAP_PATH = (
    RESULTS /
    "surfacetarget_phase5_experimental_validation_roadmap.csv"
)

CORRELATION_PATH = (
    RESULTS /
    "surfacetarget_phase5_scenario_rank_correlations.csv"
)

PANEL_PROVENANCE_PATH = (
    META /
    "surfacetarget_phase5_research_panel_provenance.json"
)

INTEGRITY_PATH = (
    META /
    "surfacetarget_phase5_roundtrip_integrity_audit.json"
)


print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 6 — PATCH 6.1")
print("RESEARCH FIGURES AND EVIDENCE VERIFICATION")
print("=" * 65)


# ------------------------------------------------------------
# 3. VALIDATE INPUT FILES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING PHASE 5 RESEARCH OUTPUTS")
print("-" * 65)


required_files = {
    "Research panel": PANEL_PATH,
    "Complete gene universe": FULL_PATH,
    "Experimental roadmap": ROADMAP_PATH,
    "Ranking correlations": CORRELATION_PATH,
    "Panel provenance": PANEL_PROVENANCE_PATH,
    "Integrity audit": INTEGRITY_PATH
}


for name, path in required_files.items():

    if not path.exists():

        raise FileNotFoundError(
            f"{name} missing: {path}"
        )

    print(f"{name}: FOUND")


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


with open(
    PANEL_PROVENANCE_PATH,
    "r"
) as handle:

    previous_provenance = json.load(handle)


with open(
    INTEGRITY_PATH,
    "r"
) as handle:

    integrity = json.load(handle)


if not integrity.get("integrity_passed", False):

    raise RuntimeError(
        "Previous scientific integrity audit did not pass."
    )


if integrity.get(
    "substantive_discrepancies",
    []
):

    raise RuntimeError(
        "Previous audit contains substantive discrepancies."
    )


if sha256_file(PANEL_PATH) != previous_provenance.get(
    "panel_sha256"
):

    raise RuntimeError(
        "Research panel checksum mismatch."
    )


if sha256_file(FULL_PATH) != previous_provenance.get(
    "full_audit_sha256"
):

    raise RuntimeError(
        "Full candidate audit checksum mismatch."
    )


print("\nPrevious scientific integrity audit: VERIFIED")

print("Research panel checksum: PASSED")

print("Complete universe checksum: PASSED")


# ------------------------------------------------------------
# 4. LOAD RESEARCH TABLES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING RESEARCH EVIDENCE")
print("-" * 65)


panel = pd.read_csv(
    PANEL_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)


full = pd.read_csv(
    FULL_PATH,
    dtype={
        "gene_id": str,
        "ensembl_stable_id": str
    },
    low_memory=False
)


roadmap = pd.read_csv(
    ROADMAP_PATH,
    dtype={"gene_id": str},
    low_memory=False
)


correlations = pd.read_csv(
    CORRELATION_PATH,
    index_col=0
)


if len(full) != 948:

    raise RuntimeError(
        "Expected 948 candidate genes."
    )


if len(panel) != 15:

    raise RuntimeError(
        "Expected 15 research-panel genes."
    )


if len(roadmap) != 15:

    raise RuntimeError(
        "Expected 15 experimental validation plans."
    )


for name, table in {
    "Full": full,
    "Panel": panel,
    "Roadmap": roadmap
}.items():

    if "gene_id" not in table.columns:

        raise RuntimeError(
            f"{name}: missing gene_id."
        )

    if table["gene_id"].isna().any():

        raise RuntimeError(
            f"{name}: missing gene identifiers."
        )

    if table["gene_id"].duplicated().any():

        raise RuntimeError(
            f"{name}: duplicate gene identifiers."
        )


if not set(panel["gene_id"]).issubset(
    set(full["gene_id"])
):

    raise RuntimeError(
        "Research panel contains unexpected genes."
    )


if set(panel["gene_id"]) != set(
    roadmap["gene_id"]
):

    raise RuntimeError(
        "Experimental roadmap does not match panel."
    )


print("\nFull candidate universe:", len(full))

print("Research panel:", len(panel))

print("Experimental validation plans:", len(roadmap))

print("\nGene identity validation: PASSED")


# ------------------------------------------------------------
# 5. STRICT BOOLEAN HELPER
# ------------------------------------------------------------

def as_bool(series, name):

    values = (
        series.astype("string")
        .str.strip()
        .str.lower()
    )

    accepted = {
        "true",
        "false",
        "1",
        "0",
        "",
        "nan",
        "none",
        "<na>"
    }

    invalid = (
        values.notna()
        & ~values.isin(list(accepted))
    )

    if invalid.any():

        raise RuntimeError(
            f"Unexpected boolean values in {name}: "
            f"{values[invalid].unique().tolist()}"
        )

    return (
        values.isin(["true", "1"])
        .fillna(False)
        .astype(bool)
    )


# ------------------------------------------------------------
# 6. VERIFY EVIDENCE COVERAGE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING CROSS-OMICS EVIDENCE COVERAGE")
print("-" * 65)


required_coverage_columns = {
    "protein_enriched",
    "uniprot_reviewed_available",
    "hpa_subcellular_record_available",
    "phase5_normal_ihc_available",
    "depmap_final_mapping_available",
    "depmap_crc_models_measured",
    "phase5_stronger_surface_annotation",
    "phase5_topology_class",
    "phase5_followup_lane",
    "phase56_in_review_panel"
}


missing = required_coverage_columns - set(full.columns)

if missing:

    raise RuntimeError(
        f"Missing evidence fields: {sorted(missing)}"
    )


protein_supported = as_bool(
    full["protein_enriched"],
    "protein_enriched"
)


uniprot_reviewed = as_bool(
    full["uniprot_reviewed_available"],
    "uniprot_reviewed_available"
)


hpa_subcellular = as_bool(
    full["hpa_subcellular_record_available"],
    "hpa_subcellular_record_available"
)


normal_ihc = as_bool(
    full["phase5_normal_ihc_available"],
    "phase5_normal_ihc_available"
)


depmap_mapped = as_bool(
    full["depmap_final_mapping_available"],
    "depmap_final_mapping_available"
)


stronger_surface = as_bool(
    full["phase5_stronger_surface_annotation"],
    "phase5_stronger_surface_annotation"
)


panel_membership = as_bool(
    full["phase56_in_review_panel"],
    "phase56_in_review_panel"
)


depmap_measured = pd.to_numeric(
    full["depmap_crc_models_measured"],
    errors="raise"
)


coverage_data = {

    "Research candidates": len(full),

    "UniProt reviewed mappings":
        int(uniprot_reviewed.sum()),

    "DepMap verified mappings":
        int(depmap_mapped.sum()),

    "DepMap >=80% CRC coverage":
        int(depmap_measured.ge(65).sum()),

    "HPA subcellular annotations":
        int(hpa_subcellular.sum()),

    "HPA normal-tissue IHC":
        int(normal_ihc.sum()),

    "Stronger surface-related topology":
        int(stronger_surface.sum()),

    "Qualifying CPTAC enrichment":
        int(protein_supported.sum())
}


expected_counts = {

    "Research candidates": 948,

    "UniProt reviewed mappings": 944,

    "DepMap verified mappings": 943,

    "DepMap >=80% CRC coverage": 923,

    "HPA subcellular annotations": 755,

    "HPA normal-tissue IHC": 690,

    "Stronger surface-related topology": 112,

    "Qualifying CPTAC enrichment": 20
}


if coverage_data != expected_counts:

    raise RuntimeError(
        "Evidence counts differ from previously "
        "validated results:\n"
        f"Observed: {coverage_data}\n"
        f"Expected: {expected_counts}"
    )


if int(panel_membership.sum()) != 15:

    raise RuntimeError(
        "Expected exactly 15 research-panel members."
    )


print("\nEvidence coverage:")

for name, count in coverage_data.items():

    print(f"{name}: {count}")


print("\nCoverage audit: PASSED")

print(
    "These categories overlap and are not "
    "sequential filtering stages."
)


# ------------------------------------------------------------
# 7. VALIDATE PROTEIN TOPOLOGY CLASSES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING PROTEIN TOPOLOGY DISTRIBUTION")
print("-" * 65)


topology_counts = (
    full["phase5_topology_class"]
    .value_counts()
)


expected_topology = {

    "No qualifying surface-topology evidence "
    "in extracted annotations": 552,

    "Secreted/extracellular; "
    "membrane attachment not established": 128,

    "Membrane with annotated extracellular region": 104,

    "Transmembrane; surface localization unresolved": 88,

    "Membrane-associated; "
    "extracellular topology unresolved": 64,

    "GPI-anchored membrane-associated": 8,

    "Other or insufficient topology annotation": 4
}


if topology_counts.to_dict() != expected_topology:

    raise RuntimeError(
        "Topology distribution differs from Patch 5.2."
    )


print("\n")

print(topology_counts.to_string())

print("\nTopology classification: VERIFIED")


# ------------------------------------------------------------
# 8. VALIDATE CROSS-SCENARIO CANDIDATES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING RANKING SENSITIVITY")
print("-" * 65)


SCENARIO_COLUMNS = {

    "Protein-first":
        "phase5_rank_protein_first",

    "Normal-review-first":
        "phase5_rank_normal_review_first",

    "RNA-discovery-first":
        "phase5_rank_rna_discovery_first"
}


surface = full[
    full["phase5_followup_lane"].eq(
        "Putative cell-surface antigen review"
    )
].copy()


if len(surface) != 111:

    raise RuntimeError(
        "Expected 111 putative surface candidates."
    )


for column in SCENARIO_COLUMNS.values():

    values = pd.to_numeric(
        surface[column],
        errors="raise"
    )

    if values.isna().any():

        raise RuntimeError(
            f"Missing scenario ranks: {column}"
        )

    if set(values.astype(int)) != set(
        range(1, 112)
    ):

        raise RuntimeError(
            f"Unexpected rank distribution: {column}"
        )


three_way_mask = (
    surface[
        list(SCENARIO_COLUMNS.values())
    ]
    .le(20)
    .all(axis=1)
)


three_way = surface[
    three_way_mask
].copy()


expected_three = {
    "CA9",
    "ADAM12",
    "NPC1L1"
}


if set(three_way["gene_name"]) != expected_three:

    raise RuntimeError(
        "Three-scenario candidate identities changed."
    )


print("\nThree-scenario top-20 candidates:")

print(
    three_way[
        [
            "gene_name",
            *SCENARIO_COLUMNS.values()
        ]
    ].to_string(index=False)
)


print(
    "\nThree-scenario agreement: VERIFIED"
)


# ------------------------------------------------------------
# 9. FIGURE 1 — EVIDENCE COVERAGE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENERATING FIGURE 1 — EVIDENCE COVERAGE")
print("-" * 65)


fig1_path = (
    FIGURES /
    "SurfaceTarget_CRC_Fig1_Evidence_Coverage.png"
)


labels = list(coverage_data.keys())

values = list(coverage_data.values())


fig, ax = plt.subplots(
    figsize=(12, 7)
)


positions = np.arange(
    len(labels)
)


bars = ax.barh(
    positions,
    values
)


ax.set_yticks(positions)

ax.set_yticklabels(labels)

ax.invert_yaxis()


ax.set_xlim(
    0,
    1050
)


ax.set_xlabel(
    "Candidate genes"
)


ax.set_title(
    "SurfaceTarget-CRC — Cross-Omics Evidence Coverage",
    fontsize=15,
    pad=18
)


for bar, value in zip(bars, values):

    ax.text(
        value + 12,
        bar.get_y() + bar.get_height() / 2,
        str(value),
        va="center",
        fontsize=10
    )


ax.text(
    0,
    -0.13,
    "",
    transform=ax.transAxes
)


fig.text(
    0.5,
    0.01,
    "Evidence categories overlap; bars do not represent a sequential screening funnel.",
    ha="center",
    fontsize=9
)


fig.tight_layout(
    rect=[0, 0.05, 1, 1]
)


fig.savefig(
    fig1_path,
    dpi=220,
    bbox_inches="tight"
)


plt.show()
plt.close(fig)


print("\nFigure 1 saved:", fig1_path)


# ------------------------------------------------------------
# 10. FIGURE 2 — PROTEIN TOPOLOGY
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENERATING FIGURE 2 — PROTEIN TOPOLOGY")
print("-" * 65)


fig2_path = (
    FIGURES /
    "SurfaceTarget_CRC_Fig2_Topology_Distribution.png"
)


topology_display_labels = {

    "No qualifying surface-topology evidence "
    "in extracted annotations":
        "No qualifying surface topology",

    "Secreted/extracellular; "
    "membrane attachment not established":
        "Secreted / extracellular",

    "Membrane with annotated extracellular region":
        "Membrane + extracellular region",

    "Transmembrane; surface localization unresolved":
        "Transmembrane; residency unresolved",

    "Membrane-associated; "
    "extracellular topology unresolved":
        "Membrane; extracellular topology unresolved",

    "GPI-anchored membrane-associated":
        "GPI-anchored membrane",

    "Other or insufficient topology annotation":
        "Other / insufficient annotation"
}


plot_topology = (
    topology_counts
    .rename(index=topology_display_labels)
)


fig, ax = plt.subplots(
    figsize=(12, 7)
)


bars = ax.barh(
    plot_topology.index,
    plot_topology.values
)


ax.invert_yaxis()


ax.set_xlabel(
    "Candidate genes"
)


ax.set_title(
    "SurfaceTarget-CRC — Protein Topology Evidence",
    fontsize=15,
    pad=18
)


ax.set_xlim(
    0,
    max(plot_topology.values) * 1.15
)


for bar, value in zip(
    bars,
    plot_topology.values
):

    ax.text(
        value + 5,
        bar.get_y() + bar.get_height() / 2,
        str(value),
        va="center",
        fontsize=10
    )


fig.text(
    0.5,
    0.01,
    "Topology annotations do not independently establish antibody accessibility.",
    ha="center",
    fontsize=9
)


fig.tight_layout(
    rect=[0, 0.05, 1, 1]
)


fig.savefig(
    fig2_path,
    dpi=220,
    bbox_inches="tight"
)


plt.show()
plt.close(fig)


print("\nFigure 2 saved:", fig2_path)


# ------------------------------------------------------------
# 11. FIGURE 3 — RANKING SENSITIVITY
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENERATING FIGURE 3 — RANKING SENSITIVITY")
print("-" * 65)


fig3_path = (
    FIGURES /
    "SurfaceTarget_CRC_Fig3_Ranking_Sensitivity.png"
)


# Only surface-antigen research groups A and B
# are comparable in this figure.
#
# FAP, membrane-accessibility and extracellular
# candidates are excluded from this visual
# because they belong to different
# biological follow-up categories.


surface_panel = panel[
    panel["phase56_review_group"]
    .str.startswith(("A", "B"))
].copy()


if len(surface_panel) != 9:

    raise RuntimeError(
        "Expected nine surface-panel genes."
    )


surface_panel = surface_panel.sort_values(
    [
        "phase56_review_group",
        "phase56_group_display_order"
    ]
)


plot_positions = np.arange(
    len(surface_panel)
)


fig, ax = plt.subplots(
    figsize=(12, 7)
)


markers = ["o", "s", "^"]

offsets = [-0.18, 0.0, 0.18]


for (label, column), marker, offset in zip(
    SCENARIO_COLUMNS.items(),
    markers,
    offsets
):

    ax.scatter(
        pd.to_numeric(
            surface_panel[column],
            errors="raise"
        ),
        plot_positions + offset,
        marker=marker,
        s=80,
        label=label
    )


ax.set_yticks(plot_positions)

ax.set_yticklabels(
    surface_panel["gene_name"]
)


ax.invert_yaxis()


ax.set_xlim(
    0,
    115
)


ax.set_xticks(
    [1, 20, 40, 60, 80, 100]
)


ax.axvline(
    20,
    linestyle="--",
    linewidth=1
)


ax.set_xlabel(
    "Rank within the 111-candidate surface-antigen lane"
)


ax.set_title(
    "SurfaceTarget-CRC — Prioritization Scenario Sensitivity",
    fontsize=15,
    pad=18
)


ax.legend(
    loc="lower right"
)


ax.grid(
    axis="x",
    alpha=0.2
)


fig.text(
    0.5,
    0.01,
    "Lower ranks indicate earlier research review, not greater therapeutic efficacy.",
    ha="center",
    fontsize=9
)


fig.tight_layout(
    rect=[0, 0.05, 1, 1]
)


fig.savefig(
    fig3_path,
    dpi=220,
    bbox_inches="tight"
)


plt.show()
plt.close(fig)


print("\nFigure 3 saved:", fig3_path)


# ------------------------------------------------------------
# 12. VERIFY FIGURE FILES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING GENERATED FIGURES")
print("-" * 65)


figure_paths = {
    "Figure 1 — Evidence coverage": fig1_path,
    "Figure 2 — Protein topology": fig2_path,
    "Figure 3 — Ranking sensitivity": fig3_path
}


for name, path in figure_paths.items():

    if not path.exists():

        raise RuntimeError(
            f"Figure was not saved: {name}"
        )

    if path.stat().st_size == 0:

        raise RuntimeError(
            f"Empty figure file: {name}"
        )

    print(
        f"{name}: SAVED "
        f"({path.stat().st_size / 1024:.1f} KiB)"
    )


# ------------------------------------------------------------
# 13. PREPARE FIGURE-SOURCE MANIFEST
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING FIGURE-SOURCE MANIFEST")
print("-" * 65)


manifest_records = [

    {
        "figure": "Figure 1",
        "title": "Cross-Omics Evidence Coverage",
        "path": str(fig1_path),
        "source": str(FULL_PATH),
        "description": (
            "Counts of overlapping evidence "
            "categories within 948 research genes"
        )
    },

    {
        "figure": "Figure 2",
        "title": "Protein Topology Evidence",
        "path": str(fig2_path),
        "source": str(FULL_PATH),
        "description": (
            "Distribution of Phase 5 protein "
            "topology annotations"
        )
    },

    {
        "figure": "Figure 3",
        "title": "Prioritization Scenario Sensitivity",
        "path": str(fig3_path),
        "source": str(PANEL_PATH),
        "description": (
            "Three within-lane ranking scenarios "
            "for nine surface-antigen review genes"
        )
    }

]


manifest = pd.DataFrame(
    manifest_records
)


manifest["figure_sha256"] = (
    manifest["path"]
    .map(
        lambda value: sha256_file(
            Path(value)
        )
    )
)


MANIFEST_PATH = (
    RESULTS /
    "surfacetarget_phase6_figure_manifest.csv"
)


manifest.to_csv(
    MANIFEST_PATH,
    index=False
)


print("\nFigure manifest saved.")


# ------------------------------------------------------------
# 14. CREATE PRELIMINARY RESULTS AUDIT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING PRELIMINARY RESULTS AUDIT")
print("-" * 65)


REPORT_PATH = (
    REPORTS /
    "SurfaceTarget_CRC_Phase6_Evidence_and_Figure_Audit.md"
)


report_lines = [

    "# SurfaceTarget-CRC",

    "## Phase 6.1 — Evidence and Figure Audit",

    "",

    (
        "This document records validated "
        "research evidence used for the "
        "initial SurfaceTarget-CRC figures."
    ),

    "",

    (
        "The analyses concern exploratory "
        "therapeutic targeting hypotheses. "
        "They do not demonstrate therapeutic "
        "efficacy or clinical safety."
    ),

    "",

    "## Candidate inventory",

    "",

    f"- Original candidate universe: {len(full)} genes.",

    f"- Final research-review panel: {len(panel)} genes.",

    f"- Stronger surface-related annotations: "
    f"{int(stronger_surface.sum())} genes.",

    f"- Qualifying CPTAC protein enrichment: "
    f"{int(protein_supported.sum())} genes.",

    f"- Normal-tissue HPA IHC annotations: "
    f"{int(normal_ihc.sum())} genes.",

    f"- Verified DepMap mappings: "
    f"{int(depmap_mapped.sum())} genes.",

    "",

    "## Figure 1 — Evidence coverage",

    "",

    (
        "Evidence availability varies "
        "substantially across molecular and "
        "protein-annotation resources."
    ),

    (
        "These categories overlap and must "
        "not be presented as sequential "
        "candidate elimination stages."
    ),

    "",

    (
        "![Evidence coverage]"
        "(../results/figures/"
        "SurfaceTarget_CRC_Fig1_Evidence_Coverage.png)"
    ),

    "",

    "## Figure 2 — Protein topology",

    "",

    (
        "The topology figure separates "
        "membrane-associated proteins, "
        "secreted proteins, and candidates "
        "without qualifying surface annotations."
    ),

    (
        "These classifications are based "
        "on extracted biological annotations "
        "rather than direct antibody-binding assays."
    ),

    "",

    (
        "![Protein topology]"
        "(../results/figures/"
        "SurfaceTarget_CRC_Fig2_Topology_Distribution.png)"
    ),

    "",

    "## Figure 3 — Ranking sensitivity",

    "",

    (
        "The surface-antigen research panel "
        "was evaluated under three different "
        "research-priority scenarios."
    ),

    (
        "Protein-first and normal-review-first "
        "ordering assumptions were highly "
        "correlated in the original analysis."
    ),

    (
        "Cross-scenario agreement does not "
        "constitute independent biological "
        "validation."
    ),

    "",

    (
        "![Ranking sensitivity]"
        "(../results/figures/"
        "SurfaceTarget_CRC_Fig3_Ranking_Sensitivity.png)"
    ),

    "",

    "## Three-scenario-consistent candidates",

    ""
]


for gene in sorted(expected_three):

    report_lines.append(
        f"- {gene}"
    )


report_lines.extend([

    "",

    (
        "These three genes appeared in "
        "the top 20 of all three scenarios "
        "within the putative surface-antigen "
        "review lane."
    ),

    "",

    (
        "This result reflects ranking "
        "consistency under the selected "
        "rules, not therapeutic superiority."
    ),

    "",

    "## Scope and limitations",

    "",

    (
        "The TCGA discovery workflow "
        "included computationally low-MSI "
        "samples that are not independently "
        "confirmed MSS cases."
    ),

    "",

    (
        "The CPTAC enrichment analysis "
        "used an independent paired MSS "
        "colon cancer proteomic cohort."
    ),

    "",

    (
        "The DepMap colorectal cell-line "
        "cohort has not been established "
        "as entirely MSS."
    ),

    "",

    (
        "Bulk tumor RNA and proteomics "
        "cannot independently determine "
        "whether a signal originates from "
        "malignant epithelial cells or "
        "the tumor microenvironment."
    ),

    "",

    (
        "Normal-tissue IHC findings are "
        "descriptive and do not predict "
        "clinical toxicity."
    ),

    "",

    (
        "Protein topology annotations "
        "do not prove extracellular epitope "
        "accessibility or antibody "
        "internalization."
    ),

    "",

    (
        "CRISPR gene-effect values describe "
        "cell fitness after gene knockout; "
        "they do not directly predict "
        "antibody-drug conjugate efficacy."
    ),

    "",

    (
        "All 948 genes remain retained "
        "in the full research dataset."
    ),

    "",

    (
        "No therapeutic target has been "
        "experimentally validated by this project."
    ),

    ""
])


REPORT_PATH.write_text(
    "\n".join(report_lines),
    encoding="utf-8"
)


print("\nEvidence audit report saved.")

print("Location:", REPORT_PATH)


# ------------------------------------------------------------
# 15. SAVE PHASE 6 PROVENANCE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING PHASE 6 PROVENANCE")
print("-" * 65)


PROVENANCE_OUTPUT = (
    META /
    "surfacetarget_phase6_figure_provenance.json"
)


phase6_provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "6.1",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "candidate_universe": 948,

    "research_panel_size": 15,

    "source_files": {

        name: str(path)

        for name, path in required_files.items()

    },

    "source_sha256": {

        name: sha256_file(path)

        for name, path in required_files.items()

    },

    "evidence_coverage": coverage_data,

    "topology_distribution": (
        topology_counts.to_dict()
    ),

    "three_scenario_consistent_genes":
        sorted(expected_three),

    "figures": {

        name: {

            "path": str(path),

            "sha256": sha256_file(path)

        }

        for name, path in figure_paths.items()

    },

    "figure_manifest": str(
        MANIFEST_PATH
    ),

    "figure_manifest_sha256":
        sha256_file(MANIFEST_PATH),

    "report": str(
        REPORT_PATH
    ),

    "report_sha256":
        sha256_file(REPORT_PATH),

    "source_evidence_modified": False,

    "new_candidate_ranking_performed": False,

    "clinical_safety_inferred": False,

    "experimental_validation_completed": False,

    "limitations": [

        "Evidence coverage categories overlap.",

        "Protein topology is based on "
        "annotations rather than binding assays.",

        "Prioritization scenarios are "
        "not independent experiments.",

        "Normal-tissue expression does "
        "not establish therapeutic safety.",

        "Bulk tumor measurements do "
        "not establish malignant-cell origin.",

        "The TCGA discovery cohort is "
        "not uniformly confirmed MSS.",

        "The DepMap colorectal cohort "
        "is not MSS-confirmed here.",

        "No therapeutic efficacy "
        "has been demonstrated."

    ]
}


with open(
    PROVENANCE_OUTPUT,
    "w"
) as handle:

    json.dump(
        phase6_provenance,
        handle,
        indent=4
    )


print("\nFigure provenance saved.")


# ------------------------------------------------------------
# 16. CREATE PHASE 6 CHECKPOINT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING PHASE 6 RESEARCH CHECKPOINT")
print("-" * 65)


timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S"
)


checkpoint = (
    CHECKPOINTS /
    f"SurfaceTarget_CRC_Phase6_Patch61_{timestamp}.zip"
)


backup_folders = [
    "data/metadata",
    "data/processed",
    "data/raw/cptac_colon",
    "data/raw/hpa",
    "data/raw/uniprot",
    "data/raw/hgnc",
    "results/tables",
    "results/figures",
    "src",
    "reports",
    "logs"
]


with ZipFile(
    checkpoint,
    "w",
    ZIP_DEFLATED,
    compresslevel=3
) as archive:

    for folder in backup_folders:

        folder_path = ROOT / folder

        if not folder_path.exists():
            continue

        for file_path in folder_path.rglob("*"):

            if file_path.is_file():

                archive.write(
                    file_path,
                    arcname=str(
                        file_path.relative_to(ROOT)
                    )
                )


checkpoint_size = (
    checkpoint.stat().st_size / 1024**2
)


print("\nCheckpoint created.")

print(
    "Checkpoint size:",
    f"{checkpoint_size:.2f} MiB"
)

print("Checkpoint location:")
print(checkpoint)


# ------------------------------------------------------------
# 17. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 6.1 SUMMARY")
print("=" * 65)


print("\nResearch candidates:", len(full))

print("Final research panel:", len(panel))

print(
    "Stronger surface-related annotations:",
    int(stronger_surface.sum())
)


print(
    "Qualifying CPTAC-supported candidates:",
    int(protein_supported.sum())
)


print(
    "Three-scenario-consistent surface genes:",
    len(three_way)
)


print("\nFigures created:")

for name in figure_paths:

    print("-", name)


print("\nSaved research outputs:")

print("1. SurfaceTarget_CRC_Fig1_Evidence_Coverage.png")

print("2. SurfaceTarget_CRC_Fig2_Topology_Distribution.png")

print("3. SurfaceTarget_CRC_Fig3_Ranking_Sensitivity.png")

print("4. surfacetarget_phase6_figure_manifest.csv")

print("5. SurfaceTarget_CRC_Phase6_Evidence_and_Figure_Audit.md")

print("6. surfacetarget_phase6_figure_provenance.json")


print("\nOriginal research evidence: PRESERVED")

print("Original candidate rankings: UNCHANGED")

print("No candidate genes excluded.")

print("No therapeutic efficacy scores assigned.")

print("No clinical safety predictions made.")


print("\nPhase 6 checkpoint: CREATED")

print(
    "Checkpoint size:",
    f"{checkpoint_size:.2f} MiB"
)


print("\nPATCH 6.1 COMPLETE")

print("=" * 65)

In [ ]:
# ============================================================
# SURFACETARGET-CRC — PHASE 6, PATCH 6.2
# EVIDENCE-GROUNDED FORMAL RESEARCH REPORT DRAFT
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
import numpy as np
import hashlib
import json


# ============================================================
# 1. PROJECT CONFIGURATION
# ============================================================

ROOT = Path('/content/SurfaceTarget-CRC')

TABLES = ROOT / 'results/tables'
FIGURES = ROOT / 'results/figures'
META = ROOT / 'data/metadata'
REPORTS = ROOT / 'reports'
CHECKPOINTS = ROOT / 'checkpoints'

for directory in [
    TABLES,
    FIGURES,
    META,
    REPORTS,
    CHECKPOINTS
]:
    directory.mkdir(parents=True, exist_ok=True)


PANEL_PATH = (
    TABLES /
    'surfacetarget_phase5_research_panel_v1.csv'
)

FULL_PATH = (
    TABLES /
    'surfacetarget_phase5_full_universe_panel_audit.csv'
)

ROADMAP_PATH = (
    TABLES /
    'surfacetarget_phase5_experimental_validation_roadmap.csv'
)

CORRELATION_PATH = (
    TABLES /
    'surfacetarget_phase5_scenario_rank_correlations.csv'
)

MANIFEST_PATH = (
    TABLES /
    'surfacetarget_phase6_figure_manifest.csv'
)

PANEL_PROVENANCE_PATH = (
    META /
    'surfacetarget_phase5_research_panel_provenance.json'
)

FIGURE_PROVENANCE_PATH = (
    META /
    'surfacetarget_phase6_figure_provenance.json'
)


print('=' * 68)
print('SURFACETARGET-CRC | PHASE 6 — PATCH 6.2')
print('EVIDENCE-GROUNDED FORMAL RESEARCH REPORT DRAFT')
print('=' * 68)


# ============================================================
# 2. VALIDATION HELPERS
# ============================================================

def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, 'rb') as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b''
        ):
            digest.update(chunk)

    return digest.hexdigest()


def require(condition, message):

    if not condition:
        raise RuntimeError(message)


def read_json(path):

    with open(path, encoding='utf-8') as handle:
        return json.load(handle)


def to_bool(series):

    values = (
        series.astype('string')
        .str.strip()
        .str.lower()
    )

    valid = {
        'true', 'false', '1', '0',
        '', 'nan', 'none', '<na>'
    }

    invalid = (
        values.notna()
        & ~values.isin(valid)
    )

    require(
        not invalid.any(),
        f'Invalid boolean tokens: '
        f'{values[invalid].unique().tolist()}'
    )

    return (
        values.isin(['true', '1'])
        .fillna(False)
        .astype(bool)
    )


def md(value):

    if pd.isna(value):
        return 'Not available'

    return (
        str(value)
        .replace('|', r'\|')
        .replace('\n', ' ')
        .strip()
    )


def fmt(value, digits=3):

    num = pd.to_numeric(
        pd.Series([value]),
        errors='coerce'
    ).iloc[0]

    return (
        f'{num:.{digits}f}'
        if pd.notna(num) and np.isfinite(num)
        else 'Not available'
    )


# ============================================================
# 3. VERIFY RESEARCH INPUTS
# ============================================================

print('\nCHECKING INPUTS AND CHECKSUMS')


files = {

    'Panel': PANEL_PATH,

    '948-gene audit': FULL_PATH,

    'Validation roadmap': ROADMAP_PATH,

    'Rank correlations': CORRELATION_PATH,

    'Figure manifest': MANIFEST_PATH,

    'Panel provenance': PANEL_PROVENANCE_PATH,

    'Figure provenance': FIGURE_PROVENANCE_PATH

}


for name, path in files.items():

    require(
        path.is_file(),
        f'Missing {name}: {path}'
    )


p56 = read_json(PANEL_PROVENANCE_PATH)

p61 = read_json(FIGURE_PROVENANCE_PATH)


require(
    sha256_file(PANEL_PATH) == p56.get('panel_sha256'),
    'Panel SHA256 mismatch'
)

require(
    sha256_file(FULL_PATH) == p56.get('full_audit_sha256'),
    'Full-audit SHA256 mismatch'
)

require(
    sha256_file(MANIFEST_PATH)
    == p61.get('figure_manifest_sha256'),
    'Figure-manifest SHA256 mismatch'
)


# Verify saved Figure 6.1 PNGs.

for label, info in p61.get('figures', {}).items():

    path = Path(info['path'])

    require(
        path.is_file(),
        f'Missing figure {label}: {path}'
    )

    require(
        sha256_file(path) == info['sha256'],
        f'Figure checksum mismatch: {label}'
    )


require(
    len(p61.get('figures', {})) == 3,
    'Expected three validated figures'
)


# ============================================================
# 4. LOAD VALIDATED RESEARCH TABLES
# ============================================================

print('\nLOADING VALIDATED RESEARCH TABLES')


panel = pd.read_csv(
    PANEL_PATH,
    dtype={
        'gene_id': str,
        'ensembl_stable_id': str
    },
    low_memory=False
)


full = pd.read_csv(
    FULL_PATH,
    dtype={
        'gene_id': str,
        'ensembl_stable_id': str
    },
    low_memory=False
)


roadmap = pd.read_csv(
    ROADMAP_PATH,
    dtype={'gene_id': str},
    low_memory=False
)


corr = pd.read_csv(
    CORRELATION_PATH,
    index_col=0
)


manifest = pd.read_csv(
    MANIFEST_PATH
)


for name, frame, expected in [

    ('panel', panel, 15),

    ('full', full, 948),

    ('roadmap', roadmap, 15)

]:

    require(
        len(frame) == expected,
        f'Unexpected {name} row count'
    )

    require(
        'gene_id' in frame.columns,
        f'Missing gene_id in {name}'
    )

    require(
        frame['gene_id'].notna().all()
        and frame['gene_id'].is_unique,
        f'Invalid gene IDs in {name}'
    )


require(
    set(panel.gene_id) == set(roadmap.gene_id),
    'Panel/roadmap gene IDs differ'
)

require(
    set(panel.gene_id).issubset(set(full.gene_id)),
    'Panel missing from full universe'
)

require(
    to_bool(full['phase56_in_review_panel']).sum() == 15,
    'Incorrect full-audit panel flags'
)

require(
    set(panel.gene_id)
    ==
    set(
        full.loc[
            to_bool(full['phase56_in_review_panel']),
            'gene_id'
        ]
    ),
    'Panel membership disagreement'
)


# ============================================================
# 5. VALIDATE EVIDENCE FIELDS
# ============================================================

required_fields = [

    'gene_name',

    'phase56_review_group',

    'phase5_followup_lane',

    'phase5_rna_conservative_log2fc',

    'phase5_protein_evidence_state',

    'phase5_topology_class',

    'phase55_normal_tissues_medium_high',

    'depmap_crc_median_gene_effect',

    'mss_complete_pairs',

    'phase5_rank_protein_first',

    'phase5_rank_normal_review_first',

    'phase5_rank_rna_discovery_first'

]


require(
    all(col in panel.columns for col in required_fields),
    'Research panel missing required columns'
)


required_full_fields = [

    'protein_enriched',

    'uniprot_reviewed_available',

    'hpa_subcellular_record_available',

    'phase5_normal_ihc_available',

    'depmap_final_mapping_available',

    'depmap_crc_models_measured',

    'phase5_stronger_surface_annotation',

    'phase5_topology_class',

    'phase5_followup_lane'

]


require(
    all(col in full.columns for col in required_full_fields),
    'Full universe missing expected evidence'
)


# ============================================================
# 6. VALIDATE CROSS-OMICS EVIDENCE COUNTS
# ============================================================

print('\nVALIDATING CROSS-OMICS EVIDENCE COVERAGE')


coverage = {

    'Candidate universe':
        len(full),

    'Reviewed UniProt mappings':
        int(
            to_bool(
                full['uniprot_reviewed_available']
            ).sum()
        ),

    'Verified DepMap mappings':
        int(
            to_bool(
                full['depmap_final_mapping_available']
            ).sum()
        ),

    'High-coverage DepMap genes (>=65/81 models)':
        int(
            pd.to_numeric(
                full['depmap_crc_models_measured'],
                errors='coerce'
            ).ge(65).sum()
        ),

    'HPA subcellular annotations':
        int(
            to_bool(
                full['hpa_subcellular_record_available']
            ).sum()
        ),

    'HPA normal-tissue IHC':
        int(
            to_bool(
                full['phase5_normal_ihc_available']
            ).sum()
        ),

    'Stronger surface-related annotation':
        int(
            to_bool(
                full['phase5_stronger_surface_annotation']
            ).sum()
        ),

    'Qualifying CPTAC enrichment':
        int(
            to_bool(
                full['protein_enriched']
            ).sum()
        )

}


expected = [
    948,
    944,
    943,
    923,
    755,
    690,
    112,
    20
]


require(
    list(coverage.values()) == expected,
    f'Unexpected evidence counts: {coverage}'
)


require(
    len(manifest) == 3,
    'Expected three figure records'
)


print('\nEvidence coverage:')

for name, count in coverage.items():
    print(f'{name}: {count}')


# ============================================================
# 7. VERIFY RANKING ROBUSTNESS
# ============================================================

print('\nVALIDATING RANKING ROBUSTNESS')


surface = full[
    full['phase5_followup_lane'].eq(
        'Putative cell-surface antigen review'
    )
].copy()


require(
    len(surface) == 111,
    'Expected 111 surface-review candidates'
)


ranks = [

    'phase5_rank_protein_first',

    'phase5_rank_normal_review_first',

    'phase5_rank_rna_discovery_first'

]


for rank in ranks:

    values = pd.to_numeric(
        surface[rank],
        errors='raise'
    )

    require(
        values.notna().all()
        and set(values.astype(int)) == set(range(1, 112)),
        f'Rank validity failed: {rank}'
    )


all_three = surface[
    surface[ranks].le(20).all(axis=1)
]


require(
    set(all_three['gene_name'])
    == {'CA9', 'ADAM12', 'NPC1L1'},
    'Three-way membership mismatch'
)


require(
    np.isclose(
        corr.loc[
            'protein_first',
            'normal_review_first'
        ],
        0.9860,
        atol=1e-3
    ),
    'Scenario correlation differs'
)


panel = panel.merge(

    roadmap[
        [
            'gene_id',
            'primary_question',
            'primary_assay',
            'orthogonal_assay',
            'additional_review'
        ]
    ],

    on='gene_id',

    how='left',

    validate='one_to_one'

)


require(
    panel['primary_assay'].notna().all(),
    'Missing validation plans'
)


print('\nChecksums and data identities: PASSED')

print(
    'Gene universe:',
    len(full),
    '| Panel:',
    len(panel),
    '| Figures:',
    len(manifest)
)

print(
    'Three-way top-20:',
    ', '.join(sorted(all_three['gene_name']))
)


# ============================================================
# 8. INITIALIZE SCIENTIFIC REPORT
# ============================================================

print('\nGENERATING STRUCTURED SCIENTIFIC REPORT')


report = []


def add(*lines):
    report.extend(lines)


add(

    '# SurfaceTarget-CRC',

    '## Multi-omics prioritization of colorectal cancer '
    'cell-surface and microenvironment target hypotheses',

    '',

    '**Technical research report — evidence-grounded '
    'draft (Phase 6.2)**',

    '',

    f'Generated: '
    f'{datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")}',

    '',

    '> **Research-use only.** This is a computational '
    'target-hypothesis analysis. No candidate has been '
    'experimentally validated for antibody binding, '
    'therapeutic efficacy, or clinical safety.',

    '',

    '## Abstract',

    '',

    'Identifying potentially accessible tumor-associated '
    'targets requires more than differential RNA expression. '
    'SurfaceTarget-CRC integrates exploratory TCGA colorectal '
    'transcriptomic evidence, paired MSS colon proteomics from '
    'CPTAC, protein topology and normal-tissue observations '
    'from UniProt/HPA, and colorectal cell-line knockout fitness '
    'from DepMap. A total of **948 gene-level candidates** were '
    'retained from the transcriptomic workflow. Among them, '
    '**20** met the project’s qualifying paired-protein enrichment '
    'criteria, **112** had stronger surface-related topology '
    'annotations, and **943** were represented in the validated '
    'DepMap gene-effect mapping. Three deliberately different '
    'within-lane review scenarios were compared for **111 putative '
    'cell-surface candidates**. **CA9, ADAM12, and NPC1L1** occurred '
    'in all three top-20 lists, although the protein-first and '
    'normal-review-first rankings were strongly correlated '
    '(Spearman ρ = 0.986). The output is an explicitly selected, '
    'mechanism-stratified **15-gene research panel**, not an '
    'optimized or clinically validated target ranking.',

    '',

    '## 1. Background and objective',

    '',

    'Tumor/normal differential expression can identify research '
    'leads but does not establish plasma-membrane residency, '
    'extracellular epitope availability, malignant-cell specificity, '
    'favorable normal-tissue distribution, or drug-response effects. '
    'This project asks which colorectal tumor-associated genes '
    'warrant follow-up as potential cell-surface, stromal, or '
    'extracellular-microenvironment targeting hypotheses. '
    'It prioritizes auditability, class-specific interpretation, '
    'source-specific missingness, and explicit experimental '
    'validation needs.',

    '',

    '## 2. Datasets and methods',

    '',

    '### 2.1. Discovery transcriptomics — TCGA/GDC',

    '',

    'An earlier workflow performed colorectal tumor-versus-normal '
    'transcriptomic analyses, QC sensitivity checks, supporting '
    'expression review, and patient-paired directional assessment. '
    'The retained set contains **948** candidate genes. '
    'The discovery workflow included a **computationally low-MSI** '
    'subgroup that must **not** be described as independently '
    'confirmed microsatellite-stable (MSS). The same-study '
    'sensitivity analyses and matched-patient direction checks '
    'are supporting analyses, not independent external validation. '
    'For downstream descriptive ranking, a conservative effect-size '
    'field was defined as the minimum of baseline and QC sensitivity '
    'log2 fold changes.',

    '',

    '### 2.2. Paired protein evidence — CPTAC',

    '',

    'The paired MSS colon tumor/normal proteomics analysis used '
    'a within-patient Wilcoxon signed-rank framework and '
    'multiple-testing adjustment. Candidate-level interpretation '
    'distinguishes qualifying enrichment, adequately covered '
    'nonqualifying measurements, sparse paired observations, '
    'and unavailable coverage. **20/948** candidates met the '
    'project-defined enrichment requirements. Unavailable or '
    'sparse protein measurements were **not** interpreted as '
    'protein absence. The detailed paired-sample inclusion '
    'and significance settings remain in the earlier phase outputs.',

    '',

    '### 2.3. Localization and normal-tissue annotation '
    '— UniProt/HPA',

    '',

    'Reviewed UniProt annotation, membrane/extracellular topology, '
    'GPI anchoring, HPA subcellular localization, and normal-tissue '
    'IHC were preserved as distinct evidence dimensions. '
    'Seven mutually exclusive annotation classes were defined '
    'from extracted topology fields. HPA tissue-level IHC is '
    'descriptive; an unmeasured gene is not assigned zero '
    'normal-tissue staining. Tissue-level maxima do not '
    'resolve all cell types and are not quantitative '
    'clinical toxicity predictions.',

    '',

    '### 2.4. CRISPR functional context — DepMap 26Q3',

    '',

    'Ensembl-to-DepMap gene identity was validated through '
    'HGNC-approved Ensembl/NCBI Gene ID crosswalks. '
    '**943/948** candidates matched DepMap gene-effect columns; '
    '**923** were measured in at least **65 of 81** colorectal '
    'adenocarcinoma models. A **79-model** patient-deduplicated '
    'sensitivity cohort and a **1,260-model** heterogeneous '
    'non-bowel comparator were evaluated in Phase 4. '
    'No MSS status was inferred for the DepMap colorectal models. '
    'CRISPR knockout fitness is not a proxy for antibody binding, '
    'epitope accessibility, or antibody-drug conjugate efficacy.',

    '',

    '### 2.5. Biological lanes and ranking-sensitivity design',

    '',

    'Genes were divided into putative surface-antigen, stromal, '
    'membrane-accessibility, extracellular microenvironment, '
    'and alternative/topology-clarification follow-up lanes. '
    'The **111** putative surface-antigen candidates were ordered '
    'in three deterministic scenarios: protein-evidence-first, '
    'normal-tissue-review-first, and RNA-discovery-first. '
    'Rank comparisons are **within the same lane only**. '
    'These are lexicographic research-review rules, not weighted '
    'therapeutic-efficacy scores. The two scenarios based primarily '
    'on protein and normal-tissue annotation proved highly similar, '
    'so two-of-three top-20 agreement should not be treated '
    'as independent robustness.',

    '',

    '## 3. Results',

    '',

    '### 3.1. Evidence coverage among 948 candidates',

    '',

    '| Evidence category | Number of candidates |',

    '|---|---:|'

)


# ============================================================
# 9. RESULTS — EVIDENCE COVERAGE
# ============================================================

for label, number in coverage.items():

    add(
        f'| {md(label)} | {number} |'
    )


add(

    '',

    'The categories in this table **overlap** and do not '
    'represent successive eliminations from a funnel.',

    '',

    '![Figure 1 — Cross-omics evidence coverage]'
    '(../results/figures/'
    'SurfaceTarget_CRC_Fig1_Evidence_Coverage.png)',

    '',

    '*Figure 1. Source-specific coverage among the same '
    '948 research genes. Categories overlap.*',

    '',

    '### 3.2. Topology classification',

    '',

    '| Extracted topology annotation class | Genes |',

    '|---|---:|'

)


for category, count in (
    full['phase5_topology_class']
    .value_counts()
    .items()
):

    add(
        f'| {md(category)} | {int(count)} |'
    )


add(

    '',

    'The **104** candidates with annotated extracellular '
    'membrane regions and **8** GPI-anchored membrane-associated '
    'candidates constitute the **112** with stronger surface-related '
    'annotations. Topology evidence alone does not demonstrate '
    'antibody recognition or cancer-cell localization.',

    '',

    '![Figure 2 — Protein topology evidence]'
    '(../results/figures/'
    'SurfaceTarget_CRC_Fig2_Topology_Distribution.png)',

    '',

    '*Figure 2. Seven mutually exclusive annotation classes '
    'based on extracted UniProt features.*',

    '',

    '### 3.3. Ranking stability and its limitations',

    '',

    'Across the 111-gene cell-surface review lane, '
    'protein-first and normal-review-first rankings had '
    f'Spearman **ρ = '
    f'{corr.loc["protein_first", "normal_review_first"]:.3f}**, '
    'whereas protein-first versus RNA-first had '
    f'**ρ = '
    f'{corr.loc["protein_first", "rna_discovery_first"]:.3f}**. '
    'Only CA9, ADAM12, and NPC1L1 were ranked in the top 20 '
    'under all three scenarios. This is within-model agreement '
    'and does **not** constitute replicated biological confirmation.',

    '',

    '| Gene | Protein-first | Normal-review-first | RNA-first |',

    '|---|---:|---:|---:|'

)


for _, row in (
    all_three
    .sort_values('gene_name')
    .iterrows()
):

    add(

        f'| {md(row["gene_name"])} | '

        f'{int(row[ranks[0]])} | '

        f'{int(row[ranks[1]])} | '

        f'{int(row[ranks[2]])} |'

    )


add(

    '',

    '![Figure 3 — Scenario ranking sensitivity]'
    '(../results/figures/'
    'SurfaceTarget_CRC_Fig3_Ranking_Sensitivity.png)',

    '',

    '*Figure 3. Within-lane priorities for the nine '
    'surface-antigen hypotheses in the panel; smaller '
    'ranks indicate earlier research review, not '
    'superior clinical efficacy.*',

    '',

    '### 3.4. Mechanism-stratified 15-gene research panel',

    '',

    'The panel consists of **3** cross-scenario surface '
    'hypotheses, **6** complementary surface-antigen '
    'hypotheses, **1** stromal candidate, **2** membrane-'
    'accessibility candidates, and **3** extracellular-'
    'microenvironment candidates. The selection is a '
    'transparent research-review decision; genes outside '
    'this panel remain in the complete 948-gene audit.',

    '',

    '| Group | Gene | Conservative RNA log2FC | CPTAC evidence | '
    'Normal tissues with medium/high IHC | '
    'Median CRC CRISPR effect |',

    '|---|---|---:|---|---:|---:|'

)


for _, row in panel.iterrows():

    group = (
        row['phase56_review_group']
        .split(' | ')[0]
    )

    add(

        f'| {group} | '

        f'{md(row["gene_name"])} | '

        f'{fmt(row["phase5_rna_conservative_log2fc"])} | '

        f'{md(row["phase5_protein_evidence_state"])} | '

        f'{fmt(row["phase55_normal_tissues_medium_high"], 0)} | '

        f'{fmt(row["depmap_crc_median_gene_effect"])} |'

    )


add(

    '',

    'The normal-tissue counts reflect HPA tissue categories '
    'with summarized medium/high IHC, not percentages of cells '
    'stained or a clinical safety margin. A missing value denotes '
    'unavailable IHC evidence. Positive CPTAC evidence must not '
    'be conflated with proven malignant-cell-specific expression.',

    '',

    '## 4. Candidate-specific interpretation and validation',

    ''

)


# ============================================================
# 10. CANDIDATE-SPECIFIC INTERPRETATION
# ============================================================

for group_name, group in panel.groupby(
    'phase56_review_group',
    sort=False
):

    add(
        f'### {md(group_name)}',
        ''
    )


    for _, row in group.iterrows():

        gene = md(
            row['gene_name']
        )

        normal_text = fmt(
            row['phase55_normal_tissues_medium_high'],
            0
        )


        add(

            f'**{gene}.** '

            f'Conservative RNA log2FC '
            f'{fmt(row["phase5_rna_conservative_log2fc"])}; '

            f'CPTAC: '
            f'{md(row["phase5_protein_evidence_state"])}; '

            f'topology: '
            f'{md(row["phase5_topology_class"])}; '

            f'normal medium/high IHC tissues: '
            f'{normal_text}; '

            f'CRC CRISPR median effect: '
            f'{fmt(row["depmap_crc_median_gene_effect"])}. '

            f'Primary question: '
            f'{md(row["primary_question"])} '

            f'Proposed initial assessment: '
            f'{md(row["primary_assay"])} '

            f'Orthogonal follow-up: '
            f'{md(row["orthogonal_assay"])} '

            f'Additional review: '
            f'{md(row["additional_review"])}',

            ''

        )


# ============================================================
# 11. DISCUSSION AND LIMITATIONS
# ============================================================

add(

    '## 5. Interpretation',

    '',

    'The evidence supports different **research pathways** '
    'rather than a single universal therapeutic target order. '
    'CA9, ADAM12, and NPC1L1 are consistent across chosen '
    'prioritization scenarios, yet none has qualifying CPTAC '
    'enrichment evidence under this workflow. CDH3 and CLDN1 '
    'have stronger RNA and extracellular-topology evidence but '
    'sparse paired proteomics and normal epithelial expression '
    'concerns. DPEP1 and SLC7A5 have qualifying protein enrichment '
    'but meaningful normal-tissue IHC observations. FAP is a '
    'distinct stromal targeting hypothesis; its gene knockout '
    'effect in isolated cancer models is not an informative '
    'measure of stromal-antigen utility. TDGF1 has GPI-associated '
    'topology and an appreciable DepMap fitness signal, but both '
    'normal-tissue IHC and paired CPTAC evidence are unavailable '
    'in this workflow. None of these findings identifies a '
    'favorable therapeutic index.',

    '',

    '## 6. Key limitations',

    '',

    '1. **MSS specificity:** The TCGA discovery subset includes '
    'computationally low-MSI cases; the colorectal DepMap cohort '
    'is not independently MSS-confirmed. Only the relevant CPTAC '
    'paired proteomics cohort is described as MSS-confirmed.',

    '2. **Bulk tumor origin:** RNA and protein enrichment can '
    'reflect malignant cells, stroma, vascular compartments, '
    'or inflammatory cells. Cellular origin is unresolved.',

    '3. **Surface accessibility:** Annotated extracellular domains '
    'or GPI anchors do not prove live-cell antibody binding, '
    'adequate antigen density, internalization, or therapeutic '
    'suitability.',

    '4. **Normal-tissue uncertainty:** HPA IHC may be missing or '
    'assay-dependent and is not a quantitative exposure or '
    'clinical toxicity measurement. Tissue-count rankings '
    'are coarse.',

    '5. **Proteomics missingness:** Sparse/unavailable paired '
    'protein coverage is not evidence of protein absence; '
    'adequately measured but nonqualifying genes require '
    'separate interpretation.',

    '6. **Screen model limitations:** DepMap reflects in-vitro '
    'knockout fitness and imperfect representation of primary '
    'tumors and their microenvironment.',

    '7. **Prioritization sensitivity:** Two scenarios are '
    'highly correlated, and the choice of lexicographic '
    'hierarchy can substantially change rank order.',

    '8. **No experimental confirmation:** No candidate received '
    'new wet-lab, binding, tumor-cell selectivity, or in-vivo '
    'efficacy validation in this project.',

    '',

    '## 7. Next experimental steps',

    '',

    'The accompanying '
    '`surfacetarget_phase5_experimental_validation_roadmap.csv` '
    'proposes gene-specific initial and orthogonal assays. '
    'A scientifically useful next study would confirm MSI/MSS '
    'status in selected models; localize expression to malignant '
    'versus nonmalignant compartments using spatial and single-cell '
    'evidence; measure live-cell surface binding with appropriately '
    'validated reagents; quantify antigen abundance relative to '
    'matched normal epithelia and organoids; and assess '
    'internalization or other modality-specific requirements. '
    'These are **proposed studies**, not experiments performed here.',

    '',

    '## 8. Reproducibility and data provenance',

    '',

    'The project retains a **948-gene** full-universe audit, '
    'a **15-gene** research panel, an experimental roadmap, '
    'and provenance JSON files recording source file hashes. '
    'The figures are linked to a three-row figure-source manifest '
    'and independent PNG checksums. The Colab working dataset '
    'also contains prior source-specific methods, model inventories, '
    'mapping audits, and earlier phase outputs. Raw public-source '
    'molecular data are not packaged with the compact research '
    'checkpoints and must be obtained in compliance with their '
    'original access and reuse conditions.',

    '',

    '**Key local research outputs:**',

    '',

    '- `results/tables/surfacetarget_phase5_full_universe_panel_audit.csv`',

    '- `results/tables/surfacetarget_phase5_research_panel_v1.csv`',

    '- `results/tables/surfacetarget_phase5_experimental_validation_roadmap.csv`',

    '- `results/tables/surfacetarget_phase6_figure_manifest.csv`',

    '- `data/metadata/surfacetarget_phase5_research_panel_provenance.json`',

    '- `data/metadata/surfacetarget_phase6_figure_provenance.json`',

    '',

    '## 9. Conclusion',

    '',

    'SurfaceTarget-CRC narrows a 948-gene transcriptomic research '
    'universe into biologically distinct review lanes, audits '
    'orthogonal proteomic, topology, normal-tissue, and functional '
    'evidence, and documents a deliberately selected 15-gene panel. '
    'The main outcome is a **traceable experimental validation '
    'roadmap**, not proof of a clinically actionable target. '
    'Tissue selectivity and live-cell accessibility are the '
    'primary evidence gaps before any treatment-oriented '
    'claims can be made.',

    '',

    '---',

    '',

    '**Publication readiness note:** This draft is grounded in '
    'the validated Phase 5–6 tables. Before external submission, '
    'cross-check the earlier Phase 1–4 sample-selection and '
    'statistical-method details against their original notebooks/'
    'provenance and add formal dataset version citations, '
    'accession identifiers, and literature references.',

    ''

)


# ============================================================
# 12. SAVE FORMAL RESEARCH REPORT
# ============================================================

report_path = (
    REPORTS /
    'SurfaceTarget_CRC_Formal_Research_Report_Draft_v1.md'
)


report_path.write_text(
    '\n'.join(report),
    encoding='utf-8'
)


# ============================================================
# 13. PREPARE SOURCE-TO-CLAIM TRACEABILITY
# ============================================================

print('\nGENERATING RESEARCH CLAIM AUDIT')


claims = [

    (
        'candidate_universe',
        948,
        FULL_PATH,
        'Full candidate-gene table'
    ),

    (
        'panel_size',
        15,
        PANEL_PATH,
        'Research panel'
    ),

    (
        'cptac_enriched_candidates',
        coverage['Qualifying CPTAC enrichment'],
        FULL_PATH,
        'protein_enriched'
    ),

    (
        'stronger_surface_candidates',
        coverage['Stronger surface-related annotation'],
        FULL_PATH,
        'phase5_stronger_surface_annotation'
    ),

    (
        'hpa_normal_ihc_candidates',
        coverage['HPA normal-tissue IHC'],
        FULL_PATH,
        'phase5_normal_ihc_available'
    ),

    (
        'depmap_mapped_candidates',
        coverage['Verified DepMap mappings'],
        FULL_PATH,
        'depmap_final_mapping_available'
    ),

    (
        'depmap_high_coverage_candidates',
        coverage[
            'High-coverage DepMap genes (>=65/81 models)'
        ],
        FULL_PATH,
        'depmap_crc_models_measured'
    ),

    (
        'surface_review_candidates',
        len(surface),
        FULL_PATH,
        'phase5_followup_lane'
    ),

    (
        'three_way_top20_candidates',
        len(all_three),
        FULL_PATH,
        'Three rank fields'
    )

]


claims_frame = pd.DataFrame([

    {
        'claim': name,

        'validated_value': val,

        'source_path': str(source),

        'source_sha256': sha256_file(source),

        'field_or_method': field
    }

    for name, val, source, field in claims

])


claims_path = (
    TABLES /
    'surfacetarget_phase6_report_claim_audit.csv'
)


claims_frame.to_csv(
    claims_path,
    index=False
)


# ============================================================
# 14. CREATE COMPACT REPORT EVIDENCE TABLE
# ============================================================

compact_fields = [

    'gene_id',

    'gene_name',

    'phase56_review_group',

    'phase5_rna_conservative_log2fc',

    'phase5_protein_evidence_state',

    'phase5_topology_class',

    'phase55_normal_tissues_medium_high',

    'depmap_crc_median_gene_effect'

]


panel_table_path = (
    TABLES /
    'surfacetarget_phase6_report_panel_summary.csv'
)


panel[compact_fields].to_csv(
    panel_table_path,
    index=False
)


# ============================================================
# 15. VALIDATE REPORT OUTPUTS
# ============================================================

require(
    report_path.stat().st_size > 5000,
    'Report unexpectedly short'
)


require(
    len(pd.read_csv(claims_path)) == len(claims),
    'Claim audit row mismatch'
)


require(
    len(pd.read_csv(panel_table_path)) == 15,
    'Report panel table row mismatch'
)


# ============================================================
# 16. SAVE REPORT PROVENANCE
# ============================================================

provenance_path = (
    META /
    'surfacetarget_phase6_report_draft_provenance.json'
)


provenance = {

    'project': 'SurfaceTarget-CRC',

    'patch': '6.2',

    'generated_at_utc':
        datetime.now(timezone.utc).isoformat(),

    'report_path':
        str(report_path),

    'report_sha256':
        sha256_file(report_path),

    'report_claim_audit':
        str(claims_path),

    'report_claim_audit_sha256':
        sha256_file(claims_path),

    'report_panel_summary':
        str(panel_table_path),

    'report_panel_summary_sha256':
        sha256_file(panel_table_path),

    'sources_sha256': {

        name: sha256_file(path)
        for name, path in files.items()

    },

    'figure_checksums_verified': True,

    'panel_count': len(panel),

    'full_candidate_count': len(full),

    'scenario_comparison_performed': True,

    'new_target_ranking': False,

    'clinical_safety_inferred': False,

    'experimental_validation_performed': False,

    'outstanding_before_publication': [

        'Confirm detailed Phase 1–4 cohort methods and '
        'thresholds from original notebooks and provenance',

        'Add formal source accessions, version identifiers, '
        'and bibliographic references',

        'Review narrative claims with a domain expert',

        'Inspect report figures at final export resolution',

        'Confirm permissions and licensing before '
        'publishing data or derived files'

    ]

}


provenance_path.write_text(
    json.dumps(provenance, indent=2),
    encoding='utf-8'
)


print('\nReport saved:', report_path)

print(
    'Report words:',
    len(report_path.read_text(encoding='utf-8').split())
)

print('Claim audit:', claims_path)

print('Panel summary:', panel_table_path)

print('Provenance:', provenance_path)


# ============================================================
# 17. CREATE PHASE 6 CHECKPOINT
# ============================================================

print('\nCREATING RESEARCH CHECKPOINT')


stamp = (
    datetime.now(timezone.utc)
    .strftime('%Y%m%d_%H%M%S')
)


checkpoint_path = (
    CHECKPOINTS /
    f'SurfaceTarget_CRC_Phase6_Patch62_{stamp}.zip'
)


backup_folders = [

    'data/metadata',

    'data/processed',

    'data/raw/cptac_colon',

    'data/raw/hpa',

    'data/raw/uniprot',

    'data/raw/hgnc',

    'results/tables',

    'results/figures',

    'src',

    'reports',

    'logs'

]


with ZipFile(
    checkpoint_path,
    'w',
    ZIP_DEFLATED,
    compresslevel=3
) as archive:

    for folder in backup_folders:

        source_folder = ROOT / folder

        if source_folder.is_dir():

            for file_path in source_folder.rglob('*'):

                if file_path.is_file():

                    archive.write(
                        file_path,
                        arcname=str(
                            file_path.relative_to(ROOT)
                        )
                    )


# ============================================================
# 18. FINAL SUMMARY
# ============================================================

print('\n' + '=' * 68)

print('PATCH 6.2 COMPLETE')

print(
    'Report sections: Abstract, Background, Methods, Results, '
    'Candidate interpretation,'
)

print(
    'Limitations, Experimental follow-up, '
    'Reproducibility, Conclusion'
)

print('Source data changed: NO')

print('All 948 genes preserved; 15 panel genes retained')

print('Checkpoint:', checkpoint_path)

print(
    f'Checkpoint size: '
    f'{checkpoint_path.stat().st_size / 1024**2:.2f} MiB'
)

print('=' * 68)

In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 6 — PATCH 6.3
#
# DIRECT LAPTOP BACKUP AND GITHUB PREPARATION
#
# OBJECTIVES:
# 1. Verify essential research results
# 2. Prepare a GitHub-ready research repository
# 3. Generate README and data-access documentation
# 4. Preserve selected scientific outputs
# 5. Create a complete backup of the Colab directory
# 6. Split large backups into browser-friendly parts
# 7. Generate a Windows reconstruction script
# 8. Record checksums for archive verification
#
# IMPORTANT:
# - NO GOOGLE DRIVE
# - NO AUTOMATIC GITHUB UPLOAD
# - NO MODIFICATION OF ORIGINAL RESEARCH DATA
# - COMPLETE BACKUP INCLUDES RAW DATA
# - PUBLIC PACKAGE EXCLUDES RAW DATA
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
import shutil
import hashlib
import json
import re


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

if not ROOT.is_dir():
    raise FileNotFoundError(
        "SurfaceTarget-CRC directory not found."
    )

STAMP = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S")

EXPORT = Path(
    f"/content/SurfaceTarget_CRC_Export_{STAMP}"
)

EXPORT.mkdir(
    parents=True,
    exist_ok=True
)

REPO = EXPORT / "SurfaceTarget-CRC"

REPO.mkdir(
    parents=True,
    exist_ok=True
)

# Browser-friendly download chunk size.
PART_SIZE_MIB = 190

PART_SIZE_BYTES = PART_SIZE_MIB * 1024**2


print("=" * 68)
print("SURFACETARGET-CRC")
print("PHASE 6 — PATCH 6.3")
print("DIRECT SSD BACKUP AND GITHUB PREPARATION")
print("=" * 68)


# ------------------------------------------------------------
# 2. HELPERS
# ------------------------------------------------------------

def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


def check_file(relative):

    path = ROOT / relative

    if not path.is_file():
        raise FileNotFoundError(path)

    return path


def copy_file(relative, required=False):

    source = ROOT / relative

    if not source.is_file():

        if required:
            raise FileNotFoundError(source)

        return False

    if source.is_symlink():
        raise RuntimeError(
            f"Unexpected symlink in public export: {source}"
        )

    destination = REPO / relative

    destination.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    shutil.copy2(source, destination)

    return True


# ------------------------------------------------------------
# 3. VALIDATE RESEARCH
# ------------------------------------------------------------

print("\nVALIDATING COMPLETED RESEARCH")


ESSENTIAL = [
    "reports/SurfaceTarget_CRC_Formal_Research_Report_Draft_v1.md",

    "results/tables/surfacetarget_phase5_research_panel_v1.csv",

    "results/tables/surfacetarget_phase5_full_universe_panel_audit.csv",

    "results/tables/surfacetarget_phase5_experimental_validation_roadmap.csv",

    "results/figures/SurfaceTarget_CRC_Fig1_Evidence_Coverage.png",

    "results/figures/SurfaceTarget_CRC_Fig2_Topology_Distribution.png",

    "results/figures/SurfaceTarget_CRC_Fig3_Ranking_Sensitivity.png"
]


for relative in ESSENTIAL:
    check_file(relative)


panel = pd.read_csv(
    ROOT /
    "results/tables/surfacetarget_phase5_research_panel_v1.csv"
)

universe = pd.read_csv(
    ROOT /
    "results/tables/surfacetarget_phase5_full_universe_panel_audit.csv",
    usecols=["gene_id"]
)


if len(panel) != 15:
    raise RuntimeError("Expected 15 research-panel genes.")

if len(universe) != 948:
    raise RuntimeError("Expected 948 research candidates.")

if universe["gene_id"].duplicated().any():
    raise RuntimeError("Duplicate research gene identifiers.")


print("\nResearch panel:", len(panel))
print("Full candidate universe:", len(universe))
print("Research validation: PASSED")


# ------------------------------------------------------------
# 4. PREPARE GITHUB DIRECTORY
# ------------------------------------------------------------

print("\nPREPARING GITHUB REPOSITORY")


for folder in [
    "reports",
    "results/tables",
    "results/figures",
    "data/metadata",
    "src",
    "notebooks",
    "docs"
]:

    (REPO / folder).mkdir(
        parents=True,
        exist_ok=True
    )


# ------------------------------------------------------------
# 5. COPY SCIENTIFIC REPORTS
# ------------------------------------------------------------

REPORT_FILES = [
    "reports/SurfaceTarget_CRC_Formal_Research_Report_Draft_v1.md",

    "reports/SurfaceTarget_CRC_Phase5_Research_Panel.md",

    "reports/SurfaceTarget_CRC_Phase6_Evidence_and_Figure_Audit.md"
]


for relative in REPORT_FILES:
    copy_file(relative)


# ------------------------------------------------------------
# 6. COPY RESEARCH FIGURES
# ------------------------------------------------------------

FIGURE_FILES = [
    "results/figures/SurfaceTarget_CRC_Fig1_Evidence_Coverage.png",

    "results/figures/SurfaceTarget_CRC_Fig2_Topology_Distribution.png",

    "results/figures/SurfaceTarget_CRC_Fig3_Ranking_Sensitivity.png",

    "results/figures/depmap_26q3_crc_dependency_comparison.png"
]


for relative in FIGURE_FILES:
    copy_file(relative)


# ------------------------------------------------------------
# 7. COPY SELECTED RESEARCH TABLES
# ------------------------------------------------------------

TABLE_FILES = [
    "results/tables/surfacetarget_phase5_research_panel_v1.csv",

    "results/tables/surfacetarget_phase5_experimental_validation_roadmap.csv",

    "results/tables/surfacetarget_phase5_panel_normal_tissue_review.csv",

    "results/tables/surfacetarget_phase5_leading_rank_sensitivity.csv",

    "results/tables/surfacetarget_phase5_scenario_rank_correlations.csv",

    "results/tables/surfacetarget_phase5_ranking_stability.csv",

    "results/tables/surfacetarget_phase5_topology_class_summary.csv",

    "results/tables/surfacetarget_phase5_data_coverage.csv",

    "results/tables/surfacetarget_phase5_followup_lane_summary.csv",

    "results/tables/surfacetarget_phase6_report_claim_audit.csv",

    "results/tables/surfacetarget_phase6_report_panel_summary.csv",

    "results/tables/surfacetarget_phase6_figure_manifest.csv"
]


for relative in TABLE_FILES:
    copy_file(relative)


# ------------------------------------------------------------
# 8. COPY SELECTED PROVENANCE
# ------------------------------------------------------------

METADATA_FILES = [
    "data/metadata/surfacetarget_phase5_research_panel_provenance.json",

    "data/metadata/surfacetarget_phase5_prioritization_provenance.json",

    "data/metadata/surfacetarget_phase5_robustness_provenance.json",

    "data/metadata/surfacetarget_phase6_figure_provenance.json",

    "data/metadata/surfacetarget_phase6_report_draft_provenance.json"
]


for relative in METADATA_FILES:
    copy_file(relative)


# ------------------------------------------------------------
# 9. COPY AVAILABLE SOURCE SCRIPTS
# ------------------------------------------------------------

print("\nCOPYING SOURCE CODE")


SAFE_EXTENSIONS = {
    ".py",
    ".r",
    ".sh",
    ".md",
    ".txt",
    ".yaml",
    ".yml",
    ".toml"
}


copied_scripts = 0

source_folder = ROOT / "src"


if source_folder.is_dir():

    for source in source_folder.rglob("*"):

        if not source.is_file():
            continue

        if source.is_symlink():
            continue

        if source.suffix.lower() not in SAFE_EXTENSIONS:
            continue

        if source.stat().st_size > 5 * 1024**2:
            continue

        relative = source.relative_to(ROOT)

        destination = REPO / relative

        destination.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        shutil.copy2(source, destination)

        copied_scripts += 1


print("Source scripts included:", copied_scripts)


if (ROOT / "requirements.txt").is_file():

    copy_file("requirements.txt")

else:

    (REPO / "requirements-reporting.txt").write_text(
        "pandas\n"
        "numpy\n"
        "matplotlib\n",
        encoding="utf-8"
    )


# ------------------------------------------------------------
# 10. CREATE GITHUB README
# ------------------------------------------------------------

print("\nCREATING README")


README = """# SurfaceTarget-CRC

## Multi-Omics Prioritization of Colorectal Cancer
## Cell-Surface and Microenvironment Target Hypotheses

SurfaceTarget-CRC is an exploratory computational biology
research project integrating transcriptomics, proteomics,
protein localization, normal-tissue annotations, and
CRISPR functional-genomics evidence.

The project develops evidence-supported research hypotheses
for experimental investigation. It does not claim
clinically validated therapeutic targets.

## Research data sources

- TCGA/GDC colorectal transcriptomics
- CPTAC paired MSS colon cancer proteomics
- Human Protein Atlas (HPA)
- UniProt reviewed protein annotations
- DepMap Public 26Q3 CRISPR gene-effect data
- HGNC gene identifier mappings

## Main findings

| Research component | Result |
|---|---:|
| Original research candidates | 948 |
| Qualifying CPTAC-enriched genes | 20 |
| Stronger surface-related topology annotations | 112 |
| Verified DepMap mappings | 943 |
| Putative cell-surface review candidates | 111 |
| Detailed research panel | 15 |

CA9, ADAM12, and NPC1L1 appeared in the
top 20 under all three research-prioritization scenarios.

However, the protein-first and normal-review-first
rankings were highly correlated (Spearman rho = 0.986).

Ranking consistency is not independent
experimental validation.

## Research figures

### Figure 1: Cross-omics evidence coverage

![Evidence coverage](results/figures/SurfaceTarget_CRC_Fig1_Evidence_Coverage.png)

### Figure 2: Protein topology

![Protein topology](results/figures/SurfaceTarget_CRC_Fig2_Topology_Distribution.png)

### Figure 3: Ranking sensitivity

![Ranking sensitivity](results/figures/SurfaceTarget_CRC_Fig3_Ranking_Sensitivity.png)

## Fifteen-gene research panel

**Cross-scenario surface hypotheses**

CA9, ADAM12, NPC1L1

**Additional surface-antigen hypotheses**

CDH3, CLDN1, DPEP1, SLC7A5, TDGF1, LGR5

**Stromal-targeting hypothesis**

FAP

**Membrane-accessibility investigations**

SLC5A6, CEMIP

**Extracellular microenvironment hypotheses**

CTHRC1, MXRA5, SFRP4

The groups describe different biological mechanisms.
They are not a single therapeutic ranking.

## Research documentation

[Formal scientific report](reports/SurfaceTarget_CRC_Formal_Research_Report_Draft_v1.md)

[Experimental validation roadmap](results/tables/surfacetarget_phase5_experimental_validation_roadmap.csv)

[Data access documentation](docs/DATA_ACCESS.md)

## Repository structure

- reports/ : Research interpretation and scientific report
- results/figures/ : Research visualizations
- results/tables/ : Selected derived analysis outputs
- data/metadata/ : Selected provenance files
- src/ : Available analysis scripts
- notebooks/ : Jupyter notebooks after manual review
- docs/ : Data access and reproducibility information

## Reproducibility

This repository contains a documented computational
research analysis and selected derived outputs.

The complete executed Colab notebook should be
added to notebooks/ after reviewing its contents.

Raw TCGA, CPTAC, and DepMap data are not
redistributed in this repository.

See docs/DATA_ACCESS.md for source information.

The report remains a technical draft. Full
independent reproducibility and publication readiness
require verification of earlier cohort methods,
dataset accessions, software versions, and citations.

## Limitations

- TCGA discovery cases are not uniformly MSS-confirmed.
- DepMap colorectal cell lines are not all MSS-confirmed.
- Bulk expression does not establish malignant-cell origin.
- Protein topology does not prove antibody accessibility.
- HPA normal-tissue staining does not establish safety.
- CRISPR dependency does not directly establish ADC efficacy.
- No new experimental validation was performed.

## Status

Exploratory computational research.

The findings represent target hypotheses,
not clinically validated therapeutic recommendations.
"""


(REPO / "README.md").write_text(
    README,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 11. CREATE DATA ACCESS DOCUMENTATION
# ------------------------------------------------------------

DATA_ACCESS = """# Data Access and Reproducibility

The complete original molecular datasets are not
redistributed in this GitHub repository.

## TCGA/GDC

TCGA-COAD and TCGA-READ colorectal
transcriptomic datasets.

## CPTAC

Proteomic Data Commons colon cancer data.

Project accession used:
PDC000116

## DepMap

Public 26Q3:

- Model.csv
- CRISPRGeneEffect.csv

The working CRISPR file was reconstructed from
upload fragments. Its local checksum was validated,
but the original release checksum was not
independently verified.

## Human Protein Atlas

Normal-tissue IHC and subcellular annotations.

## UniProt

Reviewed human protein records and topology features.

## HGNC

Gene symbol and identifier crosswalks.

## Reproducibility considerations

Release versions, original identifiers,
data-access permissions, and source-specific
licenses should be verified before reproduction.

The complete private backup contains the
original research directory, including raw data
and intermediate analyses.

The Colab notebook must be downloaded separately
if it is not already stored inside the project folder.

## Redistribution

Review the reuse and redistribution terms of each
source before publicly distributing derived data.

No third-party datasets are relicensed here.
"""


(
    REPO / "docs/DATA_ACCESS.md"
).write_text(
    DATA_ACCESS,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 12. CREATE GITIGNORE
# ------------------------------------------------------------

GITIGNORE = """# Raw and large datasets
data/raw/
data/processed/
checkpoints/
*.zip
*.tar
*.tar.gz
*.7z
*.part
*.part[0-9]*

# Credentials
.env
.env.*
*.pem
*.key
*credentials*
*secret*
*token*

# Python
__pycache__/
*.pyc
.ipynb_checkpoints/
.pytest_cache/

# Local/runtime files
*.log
.DS_Store
Thumbs.db
"""


(REPO / ".gitignore").write_text(
    GITIGNORE,
    encoding="utf-8"
)


(
    REPO / "notebooks/README.md"
).write_text(

    "# Analysis Notebook\n\n"
    "Add the executed SurfaceTarget-CRC Colab notebook "
    "here after manually checking it for credentials, "
    "personal information and excessive embedded outputs.\n",

    encoding="utf-8"
)


# ------------------------------------------------------------
# 13. SECURITY AND SIZE AUDIT
# ------------------------------------------------------------

print("\nAUDITING GITHUB EXPORT")


SECRET_PATTERNS = [
    re.compile(r"ghp_[A-Za-z0-9]{20,}"),
    re.compile(r"github_pat_[A-Za-z0-9_]{20,}"),
    re.compile(r"sk-[A-Za-z0-9_-]{20,}"),
    re.compile(r"-----BEGIN [A-Z ]*PRIVATE KEY-----")
]


TEXT_EXTENSIONS = {
    ".py",
    ".md",
    ".txt",
    ".csv",
    ".json",
    ".yaml",
    ".yml",
    ".toml",
    ".r",
    ".sh"
}


repo_files = [
    path
    for path in REPO.rglob("*")
    if path.is_file()
]


for path in repo_files:

    if path.stat().st_size > 25 * 1024**2:
        raise RuntimeError(
            f"Large public file requires review: {path}"
        )

    if path.suffix.lower() in TEXT_EXTENSIONS:

        contents = path.read_text(
            encoding="utf-8",
            errors="ignore"
        )

        for pattern in SECRET_PATTERNS:

            if pattern.search(contents):
                raise RuntimeError(
                    f"Potential secret found in: {path}"
                )


print("\nGitHub export files:", len(repo_files))
print("Large-file screening: PASSED")
print("Common secret-pattern screening: PASSED")
print("Manual privacy and licensing review still required.")


# ------------------------------------------------------------
# 14. CREATE GITHUB ZIP
# ------------------------------------------------------------

print("\nCREATING GITHUB ZIP")


GITHUB_ZIP = (
    EXPORT /
    f"SurfaceTarget_CRC_GitHub_{STAMP}.zip"
)


with ZipFile(
    GITHUB_ZIP,
    "w",
    ZIP_DEFLATED,
    compresslevel=3,
    allowZip64=True
) as archive:

    for path in REPO.rglob("*"):

        if path.is_file():

            archive.write(
                path,
                arcname=(
                    "SurfaceTarget-CRC/"
                    + path.relative_to(REPO).as_posix()
                )
            )


print("\nGitHub archive created.")

print(
    "Size:",
    f"{GITHUB_ZIP.stat().st_size / 1024**2:.2f} MiB"
)


# ------------------------------------------------------------
# 15. INVENTORY COMPLETE COLAB DIRECTORY
# ------------------------------------------------------------

print("\nINVENTORYING COMPLETE PROJECT")


all_files = sorted(
    [
        path
        for path in ROOT.rglob("*")
        if path.is_file() and not path.is_symlink()
    ],
    key=lambda p: str(p)
)


symlinks = [
    path
    for path in ROOT.rglob("*")
    if path.is_symlink()
]


total_bytes = sum(
    path.stat().st_size
    for path in all_files
)


print("\nFiles to back up:", len(all_files))

print(
    "Uncompressed project size:",
    f"{total_bytes / 1024**3:.3f} GiB"
)

print(
    "Symlinks not archived:",
    len(symlinks)
)

if symlinks:
    print(
        "WARNING: Symlinks require separate review "
        "if their targets contain additional project data."
    )


# ------------------------------------------------------------
# 16. CREATE COMPLETE PRIVATE BACKUP
# ------------------------------------------------------------

print("\nCREATING COMPLETE PRIVATE BACKUP")


FULL_ZIP = (
    EXPORT /
    f"SurfaceTarget_CRC_COMPLETE_{STAMP}.zip"
)


with ZipFile(
    FULL_ZIP,
    "w",
    ZIP_DEFLATED,
    compresslevel=1,
    allowZip64=True
) as archive:

    for index, path in enumerate(
        all_files,
        start=1
    ):

        archive.write(
            path,
            arcname=(
                ROOT.name
                + "/"
                + path.relative_to(ROOT).as_posix()
            )
        )

        if index % 100 == 0:

            print(
                f"Archived {index}/{len(all_files)} files"
            )


print("\nComplete project archive created.")

print(
    "Compressed size:",
    f"{FULL_ZIP.stat().st_size / 1024**3:.3f} GiB"
)


# ------------------------------------------------------------
# 17. VERIFY ARCHIVES
# ------------------------------------------------------------

print("\nVERIFYING ARCHIVES")


for archive_path in [
    GITHUB_ZIP,
    FULL_ZIP
]:

    with ZipFile(
        archive_path,
        "r"
    ) as archive:

        bad_file = archive.testzip()

        if bad_file is not None:
            raise RuntimeError(
                f"Corrupt archive member: {bad_file}"
            )

    print(
        "Verified:",
        archive_path.name
    )


# ------------------------------------------------------------
# 18. SPLIT LARGE COMPLETE BACKUP
# ------------------------------------------------------------

print("\nPREPARING DIRECT DOWNLOAD PARTS")


full_hash = sha256_file(FULL_ZIP)

download_parts = []


if FULL_ZIP.stat().st_size <= PART_SIZE_BYTES:

    download_parts = [FULL_ZIP]

    print("\nArchive fits in one browser download.")

else:

    with open(FULL_ZIP, "rb") as source:

        part_number = 1

        while True:

            first_chunk = source.read(
                min(PART_SIZE_BYTES, 8 * 1024**2)
            )

            if not first_chunk:
                break

            part_path = Path(
                str(FULL_ZIP)
                + f".part{part_number:03d}"
            )

            written = 0

            with open(part_path, "wb") as output:

                output.write(first_chunk)

                written += len(first_chunk)

                while written < PART_SIZE_BYTES:

                    chunk = source.read(
                        min(
                            8 * 1024**2,
                            PART_SIZE_BYTES - written
                        )
                    )

                    if not chunk:
                        break

                    output.write(chunk)

                    written += len(chunk)

            download_parts.append(part_path)

            print(
                f"Part {part_number}: "
                f"{written / 1024**2:.2f} MiB"
            )

            part_number += 1


# ------------------------------------------------------------
# 19. GENERATE WINDOWS REASSEMBLY SCRIPT
# ------------------------------------------------------------

print("\nPREPARING WINDOWS REASSEMBLY INSTRUCTIONS")


REASSEMBLE_BAT = (
    EXPORT /
    f"Reassemble_SurfaceTarget_CRC_{STAMP}.bat"
)


if len(download_parts) > 1:

    joined_names = "+".join(
        f'"{part.name}"'
        for part in download_parts
    )

    script = (
        "@echo off\n"
        "setlocal\n"
        'cd /d "%~dp0"\n'
        "echo Reassembling SurfaceTarget-CRC backup...\n"
        f'copy /b {joined_names} "{FULL_ZIP.name}"\n'
        "if errorlevel 1 (\n"
        "  echo Reconstruction failed.\n"
        "  pause\n"
        "  exit /b 1\n"
        ")\n"
        "echo.\n"
        "echo Expected SHA256:\n"
        f"echo {full_hash}\n"
        "echo.\n"
        "echo Actual SHA256:\n"
        f'certutil -hashfile "{FULL_ZIP.name}" SHA256\n'
        "echo.\n"
        "echo Confirm the two hashes match.\n"
        "pause\n"
    )

else:

    script = (
        "@echo off\n"
        "echo Your backup is already a complete ZIP.\n"
        "pause\n"
    )


REASSEMBLE_BAT.write_text(
    script,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 20. CREATE DOWNLOAD MANIFEST
# ------------------------------------------------------------

print("\nCREATING DOWNLOAD MANIFEST")


MANIFEST_JSON = EXPORT / "download_manifest.json"


manifest = {
    "project": "SurfaceTarget-CRC",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "complete_archive_name": FULL_ZIP.name,
    "complete_archive_sha256": full_hash,
    "complete_archive_size_bytes": FULL_ZIP.stat().st_size,
    "part_size_mib": PART_SIZE_MIB,
    "download_parts": [
        {
            "name": path.name,
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path)
        }
        for path in download_parts
    ],
    "github_archive": {
        "name": GITHUB_ZIP.name,
        "size_bytes": GITHUB_ZIP.stat().st_size,
        "sha256": sha256_file(GITHUB_ZIP)
    },
    "original_research_directory_modified": False,
    "google_drive_used": False
}


MANIFEST_JSON.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8"
)


# Verify split data reproduces the archive hash.

if len(download_parts) > 1:

    digest = hashlib.sha256()

    for part in download_parts:

        with open(part, "rb") as handle:

            for chunk in iter(
                lambda: handle.read(8 * 1024**2),
                b""
            ):
                digest.update(chunk)

    if digest.hexdigest() != full_hash:

        raise RuntimeError(
            "Backup parts failed reconstruction checksum."
        )

    # The original archive is no longer needed
    # in the temporary export area because the
    # validated parts contain identical bytes.
    # This does NOT delete any original project file.

    FULL_ZIP.unlink()


# ------------------------------------------------------------
# 21. VERIFY DOWNLOAD PACKAGE
# ------------------------------------------------------------

print("\nVERIFYING DOWNLOAD PACKAGE")


download_files = [
    GITHUB_ZIP,
    MANIFEST_JSON,
    REASSEMBLE_BAT,
    *download_parts
]


for path in download_files:

    if not path.is_file():

        raise RuntimeError(
            f"Missing download file: {path}"
        )


print("\nDownload package: VERIFIED")


# ------------------------------------------------------------
# 22. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 68)
print("PATCH 6.3 COMPLETE")
print("=" * 68)


print("\nCOMPLETE PROJECT BACKUP")

print("Original research files:", len(all_files))

print(
    "Total original size:",
    f"{total_bytes / 1024**3:.3f} GiB"
)

print(
    "Complete backup parts:",
    len(download_parts)
)


print("\nGITHUB REPOSITORY")

print(
    "GitHub ZIP:",
    GITHUB_ZIP.name
)

print(
    "GitHub archive size:",
    f"{GITHUB_ZIP.stat().st_size / 1024**2:.2f} MiB"
)


print("\nFILES READY FOR DOWNLOAD:")

for index, path in enumerate(
    download_files,
    start=1
):

    print(
        f"{index}. {path.name} "
        f"({path.stat().st_size / 1024**2:.2f} MiB)"
    )


print("\nEXPORT DIRECTORY:")
print(EXPORT)


print("\nIMPORTANT:")
print("No Google Drive used.")
print("Original project directory unchanged.")
print("No GitHub upload performed.")
print("Save your Colab notebook separately.")
print("Download all parts before disconnecting Colab.")

print("=" * 68)

In [ ]:
# ============================================================
# SURFACETARGET-CRC
# GITHUB PUBLICATION — STEP 1
#
# PREPARE THE COMPLETE ANALYSIS NOTEBOOK
# ============================================================

from pathlib import Path
from google.colab import _message

import json
import re

REPO = Path(
    "/content/SurfaceTarget_CRC_Export_20261121_092801/"
    "SurfaceTarget-CRC"
)

if not REPO.is_dir():
    raise FileNotFoundError(
        "GitHub project folder not found. "
        "Check that Patch 6.3's export still exists."
    )

NOTEBOOKS = REPO / "notebooks"
NOTEBOOKS.mkdir(parents=True, exist_ok=True)

NOTEBOOK_PATH = (
    NOTEBOOKS /
    "SurfaceTarget_CRC_Complete_Analysis.ipynb"
)

print("=" * 65)
print("SURFACETARGET-CRC — NOTEBOOK EXPORT")
print("=" * 65)

print("\nRetrieving current Colab notebook...")

response = _message.blocking_request(
    "get_ipynb",
    timeout_sec=60
)

if not response:
    raise RuntimeError(
        "Could not retrieve the current notebook."
    )

notebook = response.get("ipynb", response)

if isinstance(notebook, str):
    notebook = json.loads(notebook)

if not isinstance(notebook, dict):
    raise RuntimeError(
        "Unexpected notebook response format."
    )

if "cells" not in notebook:
    raise RuntimeError(
        "Notebook response does not contain cells."
    )

# Remove execution outputs and runtime metadata.
# Preserve all Python and Markdown source cells.

clean_cells = []

for cell in notebook["cells"]:

    cell_type = cell.get("cell_type")

    if cell_type not in [
        "code",
        "markdown"
    ]:
        continue

    cleaned = {
        "cell_type": cell_type,
        "metadata": {},
        "source": cell.get("source", [])
    }

    if cell_type == "code":
        cleaned["execution_count"] = None
        cleaned["outputs"] = []

    clean_cells.append(cleaned)

clean_notebook = {
    "cells": clean_cells,
    "metadata": {
        "kernelspec": {
            "display_name": "Python 3",
            "language": "python",
            "name": "python3"
        },
        "language_info": {
            "name": "python"
        }
    },
    "nbformat": 4,
    "nbformat_minor": 4
}

# Check source cells for recognizable credential formats.

source_text = "\n".join(
    "".join(cell["source"])
    if isinstance(cell["source"], list)
    else str(cell["source"])
    for cell in clean_cells
)

secret_patterns = [
    r"ghp_[A-Za-z0-9]{30,}",
    r"github_pat_[A-Za-z0-9_]{30,}",
    r"sk-proj-[A-Za-z0-9_-]{30,}",
    r"-----BEGIN [A-Z ]*PRIVATE KEY-----"
]

for pattern in secret_patterns:
    if re.search(pattern, source_text):
        raise RuntimeError(
            "Potential credential found in notebook source. "
            "Review before uploading."
        )

NOTEBOOK_PATH.write_text(
    json.dumps(
        clean_notebook,
        indent=1,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("\nNotebook saved successfully!")

print("Total cells:", len(clean_cells))

print(
    "Python cells:",
    sum(
        cell["cell_type"] == "code"
        for cell in clean_cells
    )
)

print(
    "Markdown cells:",
    sum(
        cell["cell_type"] == "markdown"
        for cell in clean_cells
    )
)

print(
    "Notebook size:",
    f"{NOTEBOOK_PATH.stat().st_size / 1024**2:.2f} MiB"
)

print("\nLocation:")
print(NOTEBOOK_PATH)

print("\nSaved execution outputs: REMOVED")
print("Source code: PRESERVED")
print("Original notebook: UNCHANGED")

print("\nNOTEBOOK EXPORT COMPLETE")